# Quantum Circuits | 23.17 Local GM: 889-Qubit DQI

**Geometric mean 23.16705** from the official verifiers at their default seeds. That is a local measurement; the leaderboard measures again with its own samples.

For each task this notebook takes the best circuit we have verified. The DQI circuit is our own, explained below. The invsqrt and chemistry circuits are other people's public work, used unchanged: we link them and credit their authors in full. The notebook checks that every member is byte for byte the circuit the official verifiers passed, and writes `/kaggle/working/submission.zip`.

| Task | Circuit by | Q | Mean Toffolis | Cost | Ratio |
|---|---|---:|---:|---:|---:|
| invsqrt | yoyobar and Ahmed Berat Özer, on busyaprime, mrmeows and team selenium | 62 | 1,327.5 | 100,693 | **20.6249** |
| DQI | GPT-6 Astra Pro on our build loop, with Ahmed Berat Özer, Alex Chilton and jday96314 | 889 | 424,128.4 | 4.02702e+11 | **60.6887** |
| Chemistry | Ahmed Berat Özer, Atakan Aldemir, James Barlow, Dmitrii Gluzdov and heliosli, on our team | 430 | 16,266.0 | 3.37798e+08 | **9.9337** |

## How the score works

Each task has an official verifier. It runs the circuit on many sampled inputs, checks every answer exactly, and bills a cost:

| Task | Cost | The letters |
|---|---|---|
| invsqrt | $Q \cdot C$ | $Q$ qubits; $C = N_{\mathrm{Tof}} + N_T / 4$, the mean Toffolis and T gates of one shot |
| DQI | $R \cdot Q^2 \cdot T$ | $T$ the mean Toffolis of one shot; $R \ge 1$ grows with the failed shots |
| Chemistry | $\lambda_{\mathrm{eff}} \cdot T \cdot Q$ | $\lambda_{\mathrm{eff}}$ the Hamiltonian's effective normalization; $T$ the Toffolis of one step |

A task's **ratio** is the organizers' reference cost divided by this cost. The references are 2,076,774, $2.4439 \times 10^{13}$ and 3,355,599,588. The score is the **geometric mean of the three ratios**:

$$\text{score} = \left( \frac{C^{\mathrm{ref}}_{\mathrm{invsqrt}}}{C_{\mathrm{invsqrt}}} \cdot \frac{C^{\mathrm{ref}}_{\mathrm{DQI}}}{C_{\mathrm{DQI}}} \cdot \frac{C^{\mathrm{ref}}_{\mathrm{chem}}}{C_{\mathrm{chem}}} \right)^{1/3}$$

So making any one task 1% cheaper lifts the score by about 0.33%.

The three bills reward different things. In DQI, $Q$ is squared: near 880 qubits one qubit is worth 0.23% of the cost. In invsqrt one qubit near 63 is worth 1.6%. In chemistry each of the three factors counts the same.

## invsqrt · yoyobar's 62-qubit circuit, with Ahmed Berat Özer's phase patch

Used unchanged. Source: the output of yoyobar's notebook [LB 22.99 | invsqrt 101 - 62 qubits](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits), as of 30 October 2026, 18:55 UTC, as invsqrt.kmb inside its submission.zip.

**Credits.** As yoyobar's notebook states: **yoyobar**, with GPT-6 Astra: the 62-qubit circuit, by changes to the lookup, normalization and tail schedule and four nonlinear-cone rewrites. **Ahmed Berat Özer**: a compatible phase patch that saves one Toffoli, in [Quantum AI - DF-THC Group Compression](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression), distributed through **James Barlow (manwithacat)**'s [VQCC Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo). **Akmal Xodarev (busyaprime)**: the 63-qubit starting point, based on **mrmeows / team selenium**.

## DQI · Pro's front half and back half, joined at 889 qubits

**This circuit.**
- $Q = 889$, of which 840 hold the syndromes.
- A shot runs $T = 424{,}128.4$ Toffolis on average, at most 486,571. So $Q^2 T = 3.3520 \times 10^{11}$.
  - That is 1.96% below our 887-qubit decoder, whose $Q^2 T$ was $3.4189 \times 10^{11}$. It has 2 qubits more and about 10,430 Toffolis a shot fewer.
  - When it was built, on 30 October, it was 0.78% below the best public decoder of the time, yoyobar's 889-qubit one, with $Q^2 T = 3.3784 \times 10^{11}$.
- At the default seed, 0 shots decode a wrong value and 2 a wrong phase, so $R = 1.2014$.
  - Its rare failures come from inputs that it shares with the decoders it grew from. At eleven fresh salts of the official verifier, 1,126,400 shots, it failed three times, each with a wrong phase: about 3 in a million. So a draw of 102,400 shots has no failure about three times in four.
  - With no failure, $R = 1.1913$ and the cost is $3.9931 \times 10^{11}$.
- The cost is $4.02702 \times 10^{11}$, 60.6887× below the reference.

**Where it comes from.** On 30 October we gave GPT-6 Astra Pro two questions on one kit, the 889-qubit decoder of our build loop, with $T = 433{,}318.7$. One asked for savings in the front half: extraction, synthesis and their reverses. The other asked for savings in the back half: the conversion and its reverse. Each answer was a whole decoder, and neither knew of the other. Claude joined them in 25 lines: the front half of the first with the conversion pair of the second. The two savings add: 3,815 and 5,375 make the 9,190 this decoder saves on the kit. Then Claude built the circuit twice to the same SHA-256 and ran the official verifier.

**What is new.** These are the changes against the kit's decoder, with the Toffolis a shot that each stage saved.

| Stage | What changed | Toffolis a shot |
|---|---|---:|
| Extraction | Inverses keep raw norm products for their final products, and one joint schedule makes six output coordinates | −449 |
| Synthesis | The tail's common factor is formed once, a telescoping gauge absorbs the prefix's last product, and the same full fold serves both parities | −806 |
| Conversion | The two payloads share their parent correction, quotient heads are measured and rebuilt, and exact middle identities replace whole products | −2,364 |
| Reverse conversion | The same identities, a Hankel schedule that keeps two overlapping old-source copies, and root blocks that pay their edges while the inverse is consumed | −3,011 |
| Reverse synthesis | The reverse works in coordinates modulo the fold, and the known leader of the last fold frees a bank for a retained divider | −190 |
| Reverse extraction | Each inverse keeps the norm coordinates that fit and pays the rest as exact characters, and the trailing node saves one forward scalar replay with a 12-qubit companion | −2,369 |

The back half of the second answer also had a factorized boundary. We left it out: it would save only 137 Toffolis, and it does not fit the first answer's synthesis.

**How it decodes.** One shot, stage by stage, as the kit counts it at three fresh seeds: mean Toffolis and the peak of live qubits.

| Stage | Toffolis | Peak qubits |
|---|---:|---:|
| Extraction | 66,353.5 | 889 |
| Synthesis | 32,267.0 | 889 |
| Conversion | 60,914.0 | 889 |
| Scan | 44,264.2 | 888 |
| Reverse conversion | 62,428.9 | 889 |
| Reverse synthesis | 42,079.0 | 889 |
| Reverse extraction | 115,821.9 | 889 |
| **Total** | **424,128.4** | **889** |


**Credits.** **GPT-6 Astra Pro**, answering our two questions of 30 October: the front half, with its cached extension inverses, the joint six-output inverse, the tail, prefix and fold identities of the synthesis, the retained norm coordinates of the reverse and the trailing node's single scalar replay, and the conversion pair, with its shared parent correction, its measured quotient heads, its middle identities, its Hankel schedule and its root blocks. **Claude** joined the two and built the result. **Our build loop** of AI agents (GPT-6 Astra leading GPT-6.1 Sol builders and a GPT-6.1 Terra reviewer, directed by Claude for Agniv Banerjee, SomeJoint), in seven hours on 30 October: the retained reverse chart and core; the swap-free core and its zero-affine phase character; grouped forward flags with a scoped direct pole phase; adaptive norm coordinates in the inverse characters; streamed 18-Toffoli inverses; the converter route from level 6 with child gap flags 2 and 3; sequential low head restores with a reused inverse flag; high-level gap predicates from the top flag; no quantum work where the classical mask word is zero; and the trailing pole flag erased by a measured phase. **GPT-6 Astra Pro**: five exact substitutions in the decoder, answering our question of 29 October, and an independent observation of the odd full product fused with the CRT remainder, which the loop built. **Our third research team** of AI agents (GPT-6 Astra leading two GPT-6 Astra designers, three GPT-6.1 Sol builders and three GPT-6.1 Terra reviewers): the 889-qubit decoder the loop starts from, with the paired high-component chart of the extraction chains, the inverse19 reverse, the streamed synthesis divide and the retimed head schedule, after its aligned conversion route with measured low heads, the streamed full-head fusion with a monic root, the small truncated products and the replay erasure. **GPT-6 Astra Pro**: the 897-qubit decoder it starts from, with its homogeneous external chains and 658-Toffoli jet. **Ahmed Berat Özer**: the narrowing of our fused decoder in his version 9, from which our 883-qubit decoder came. **Alex Chilton**: the 883-qubit Schur decoder whose design it starts from, based on **jday96314**'s QuantumAI decoder and classical models (that repository states no license). **mrmeows** and **Ahmed Berat Özer**: the public mask projection of round17 and version 8, after **manwithacat**'s classical-mask projection. Earlier links of the design: **Dmitrii Gluzdov**'s round12 decoder, **Agniv Banerjee (SomeJoint)**'s Bézout–Hankel phase solve, **ViberTeam** and **Phuc Thai (DePie)**. Source: our own: the source is embedded below, its data files and the built circuit are in our dataset [qcc-dqi-889-two-halves](https://www.kaggle.com/datasets/somejoint/qcc-dqi-889-two-halves). They are the kit of our questions q8 and q9 to GPT-6 Astra Pro, which is our build loop's 889-qubit decoder of 30 October (commit 3c22a84), Pro's two answers, each a whole decoder on that base, and our 25-line join with our build script.

## Chemistry · Ahmed Berat Özer's weights on Atakan Aldemir's words, as Alex Chilton ships them

Used unchanged. Source: the output of [Alex Chilton's 22.57 LB verified three-task portfolio](https://www.kaggle.com/code/alexchilton/22-57-lb-verified-three-task-portfolio), version 62 by its own count, as of 30 October 2026, 20:34 UTC.

**Credits.** As Alex Chilton's notebook states: **Ahmed Berat Özer**: the new weights, in version 16 of [Quantum AI - DF-THC Group Compression](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression). **Atakan Aldemir**: the word table of [QCC: 888Q Repair and Native Chemistry Co-design](https://www.kaggle.com/code/atakanaldemir/qcc-888q-repair-and-native-chemistry-co-design). **James Barlow (manwithacat)**: the 16,266-Toffoli per-slot loaders, assembled from those of **Dmitrii Gluzdov**, **heliosli** and Atakan Aldemir, and the refit of the weights in [VQCC Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo), who also assembled these three files first, in [The Real $150K Was the Friends We Made](https://www.kaggle.com/code/manwithacat/the-real-150k-was-the-friends-we-made). **Alex Chilton**: this file's portfolio. **Our research team** of AI agents, directed by Claude for Agniv Banerjee (SomeJoint): the full-XOR loaders and factorization this line inherits, from our public dataset [qcc-our-chemistry-b143-c60-xor2](https://www.kaggle.com/datasets/somejoint/qcc-our-chemistry-b143-c60-xor2).

## Run: take the bytes, check, pack

Attach the competition data and the notebook outputs [alexchilton/22-57-lb-verified-three-task-portfolio](https://www.kaggle.com/code/alexchilton/22-57-lb-verified-three-task-portfolio), [yoyobar/lb-22-99-invsqrt-101-62-qubits](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) and the dataset [somejoint/qcc-dqi-889-two-halves](https://www.kaggle.com/datasets/somejoint/qcc-dqi-889-two-halves), and choose the Python 3.13 CPU image. No internet, installs or training are needed.

Our DQI circuit comes ready built from the attached dataset, so a run takes a couple of minutes. Set `REBUILD = True` below to build it here instead, from the source embedded in the setup cell and data files that the setup takes from the attached inputs by SHA-256, in about 6 minutes on Kaggle CPU and 7 GB of memory; the bytes must come out the same. Every member is found by its SHA-256, among the inputs and the built files, so a changed input fails loudly instead of slipping through. The official verifiers ran on these exact bytes before assembly, at their defaults; the tables above quote those runs. Set `VERIFY = True` below to rerun all three here, about ten minutes. The last cell checks the format rules, packs the ZIP and reads it back.

In [ ]:
COMPETITION_FILES = {'kickmix/LICENSE.txt': 'cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30', 'kickmix/kickmix.so': 'c66b3216c4029e0cdfd09054306aa0feadf53ab2f50c1164e37f752874e5005e', 'verifiers/LICENSE.txt': 'cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30', 'verifiers/dfthcsos/FeMoCo76o113e.fcidump': 'e5a48af746fd075026dec9e8f5df2d363802a121e8c53af9a0b984bde695e1bd', 'verifiers/dfthcsos/FeMoCo76o113e.fixed_shifts.npz': '7b7facf63fe35a9d9d3d00787ffdb700c863fa845fe4724d7f7c54778151cd87', 'verifiers/dfthcsos/SCORING_README.md': 'c2228569f96e58e118419bc32be2be9e33d6abffb8117cb0b91e1fc25b2a0230', 'verifiers/dfthcsos/__init__.py': '81915e324cc0cade67f52a54b7a51991fd96abaea94d1f8a4d423dae20e57348', 'verifiers/dfthcsos/scoreinfra/__init__.py': 'be90e0d503d5c20dcac0102af638b9a4ccad0341daa5baa5d49216a259e4516f', 'verifiers/dfthcsos/scoreinfra/circuits.py': 'fc027b52fdcebe14404f403a089122485d097ed65d03e6d989623d620b1ab68c', 'verifiers/dfthcsos/scoreinfra/costs_prepare.py': '7f1e190c42fc5325f3e6689fbbef51f95b0cb12f1c828cda92b6c088e72206da', 'verifiers/dfthcsos/scoreinfra/dfthc_math.py': 'ae82898d47a7a86b96378fb906e0f8a9f46003065cf19c7c09952a123b5b5af8', 'verifiers/dfthcsos/scoreinfra/reference.py': 'a2d5a63191d655729e29f0fde1e85fe4630bf896a91a63bb894d8d7daf71ed39', 'verifiers/dfthcsos/verify.py': '0775be7a528922367caebe9e4d051d6199811f1aefeaa4b793eaaa4621b8ec9d', 'verifiers/dqi/__init__.py': '01c870aa9763f4e4357fc6654bff1d3296cec995bd0943f87b4a48946ada5b40', 'verifiers/dqi/dqi_reference_circuit.py': '2b8e1b80a806157c333a7e308eaf2499ee09de499efbc05875202c327959959e', 'verifiers/dqi/verify_submission_dqi.py': 'f14274bc0943fb9769ddb01131be6063191a15009fe1caf1d4bdd9fd25194e65', 'verifiers/invsqrt/__init__.py': 'e9a6ecf253c343528578be5fa92eb820ccade63333aebb784552fb4ae75dbc0b', 'verifiers/invsqrt/nr_reference_circuit.py': 'bb90330dcf774f7b502de471293facba67f27952cc8808fe4a9970a9b1322e23', 'verifiers/invsqrt/verify_submission_invsqrt.py': 'ddef520dec6b68d1cf0a6f328ddd36d79eb84570b9192cf3f7acaf073f06733a'}
SOURCES = {'notebooks/launch.py': '"""Isolated launcher for the pinned official verifiers and reviewed local code.\n\nSPDX-License-Identifier: Apache-2.0\nOnly authenticated competition directories are added; never input ZIP code.\n"""\nfrom pathlib import Path\nimport runpy\nimport sys\n\nroot = Path(__file__).resolve().parent.parent\nsys.dont_write_bytecode = True\nsys.path[:0] = [str(root / \'data/comp/kickmix\'), str(root / \'data/comp/verifiers\')]\ntarget = Path(sys.argv[1]).resolve()\nsys.argv = [str(target), *sys.argv[2:]]\nrunpy.run_path(str(target), run_name=\'__main__\')\n', 'notebooks/take.py': '"""Copy pinned submission members out of the notebook\'s inputs.\n\nSPDX-License-Identifier: Apache-2.0\nRun with Python -I: take.py INPUT_ROOT OUTPUT_DIR MEMBERS_JSON [MORE_ROOTS...]\nMEMBERS_JSON maps each member name to its sha256 and size. A member is taken from a\nplain file whose path ends with the member\'s name, or else from a ZIP member whose\nname ends the same way; only a copy with the pinned size and SHA-256 is kept. Nothing\nfrom an input is executed, and no archive is extracted whole. MORE_ROOTS, such as the folder of\nmembers built in the notebook, are searched the same way.\n"""\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport sys\nimport zipfile\n\n\ndef digest(stream):\n    h = hashlib.sha256()\n    while chunk := stream.read(1 << 22):\n        h.update(chunk)\n    return h.hexdigest()\n\n\ndef ends_with(path_parts, name_parts):\n    return tuple(path_parts[-len(name_parts):]) == name_parts\n\n\ndef main():\n    root, out, manifest = Path(sys.argv[1]), Path(sys.argv[2]), json.loads(Path(sys.argv[3]).read_text())\n    roots = [root, *map(Path, sys.argv[4:])]\n    files = sorted(p for r in roots if r.is_dir() for p in r.rglob(\'*\') if p.is_file())\n    zips = [p for p in files if p.suffix == \'.zip\']\n    taken = {}\n    for name, pin in manifest.items():\n        parts = PurePosixPath(name).parts\n        assert not PurePosixPath(name).is_absolute() and \'..\' not in parts, name\n        target = out / name\n        target.parent.mkdir(parents=True, exist_ok=True)\n        for path in files:\n            if ends_with(path.parts, parts) and path.stat().st_size == pin[\'size\']:\n                with path.open(\'rb\') as stream:\n                    if digest(stream) == pin[\'sha256\']:\n                        shutil.copyfile(path, target)\n                        taken[name] = str(path)\n                        break\n        for path in zips if name not in taken else []:\n            with zipfile.ZipFile(path) as archive:\n                for info in archive.infolist():\n                    if not ends_with(PurePosixPath(info.filename).parts, parts) or info.file_size != pin[\'size\']:\n                        continue\n                    # Reading to EOF also checks the member\'s ZIP CRC.\n                    with archive.open(info) as source, target.open(\'wb\') as copy:\n                        shutil.copyfileobj(source, copy, length=1 << 20)\n                    with target.open(\'rb\') as stream:\n                        if digest(stream) == pin[\'sha256\']:\n                            taken[name] = f\'{path}:{info.filename}\'\n                            break\n                    target.unlink()\n            if name in taken:\n                break\n        if name not in taken:\n            raise SystemExit(f\'{name}: no input holds SHA-256 {pin["sha256"]}; attach the source its block names\')\n    print(json.dumps({\'status\': \'PASS\', \'taken\': taken}, indent=1))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'notebooks/check-submission': '#!/usr/bin/env python3\n"""Checks a submission folder against the metric\'s extra rules, the ones the local verifiers skip.\n\n    .venv/bin/python -I tools/check-submission SUBMISSION_DIR\n\nThe rules come from the metric notebook, as reported in Kaggle discussion 751951; a submission that breaks one\nis rejected as a whole, even when every verifier passes:\n- every circuit\'s largest qubit or bit id, max(num_qubits, num_bits) - 1, is below 2**30 // 1536 = 699,050;\n- every register has a name;\n- the folder holds exactly invsqrt.kmb, dqi.kmb and dfthcsos/ with angle_k_0 .. angle_k_<n-1> (no gaps) and\n  submission.npz, and nothing else anywhere;\n- submission.npz arrays are little-endian floats, except the index arrays tree_edges and angle_words, which are\n  integer types. The sample submission, which the metric accepted in submission 1, stores angle_words as int64.\n  The 8-bit factorizations of 2026-10-26 also store brot and kept_indices as int64 and one_body_requires_refresh\n  as a bool; the metric scored exactly such a file, version 14 of kirilltym\'s QROM + Tower42 (public 19.57899).\n"""\nimport os\nimport pathlib\nimport re\nimport sys\n\nimport numpy as np\n\nsys.path.insert(0, str(pathlib.Path(__file__).resolve().parent.parent / \'data\' / \'comp\' / \'kickmix\'))\nimport kickmix as km  # noqa: E402\n\nMAX_ID = 2**30 // 1536\nINTEGER_ARRAYS = (\'tree_edges\', \'angle_words\', \'brot\', \'kept_indices\')\nBOOL_ARRAYS = (\'one_body_requires_refresh\',)\n\nroot = sys.argv[1]\nproblems = []\nfiles = sorted(os.path.relpath(os.path.join(d, f), root) for d, _, fs in os.walk(root) for f in fs)\nangles = [f for f in files if re.fullmatch(r\'dfthcsos/angle_k_\\d+\\.kmb\', f)]\nexpected = {\'invsqrt.kmb\', \'dqi.kmb\', \'dfthcsos/submission.npz\'} | {f\'dfthcsos/angle_k_{i}.kmb\' for i in range(len(angles))}\nif set(files) != expected:\n    problems.append(f\'file set differs: extra {sorted(set(files) - expected)}, missing {sorted(expected - set(files))}\')\nfor f in [f for f in files if f.endswith(\'.kmb\')]:\n    c = km.Circuit.from_file(os.path.join(root, f))\n    top = max(c.num_qubits, c.num_bits) - 1\n    unnamed = [k for k in c.register_data if not isinstance(k, str)]\n    print(f\'{f}: qubits {c.num_qubits}, bits {c.num_bits}, largest id {top}, {len(c.register_data)} registers\')\n    if top >= MAX_ID:\n        problems.append(f\'{f}: largest id {top} >= {MAX_ID}\')\n    if unnamed:\n        problems.append(f\'{f}: unnamed registers {unnamed}\')\nnpz = np.load(os.path.join(root, \'dfthcsos\', \'submission.npz\'), allow_pickle=False)\nfor name in npz.files:\n    a = npz[name]\n    kind = np.integer if name in INTEGER_ARRAYS else np.bool_ if name in BOOL_ARRAYS else np.floating\n    kind_ok = np.issubdtype(a.dtype, kind)\n    little = a.dtype.byteorder in (\'<\', \'|\') or (a.dtype.byteorder == \'=\' and sys.byteorder == \'little\')\n    print(f\'npz {name}: dtype {a.dtype.str}, shape {a.shape}\')\n    if not kind_ok or not little:\n        problems.append(f\'npz {name}: dtype {a.dtype.str}\')\nprint(\'PROBLEMS:\' if problems else \'ALL CHECKS PASS\', *problems, sep=\'\\n  \')\nsys.exit(1 if problems else 0)\n', 'notebooks/extra-format.py': '#!/usr/bin/env python3\n"""Supplement check-submission with NPZ header bounds and ANGLE T counts.\n\nFrom runs/submission-8/extra-format.py; booleans are allowed since 2026-10-27, as in check-submission.\n"""\nimport json\nimport math\nfrom pathlib import Path\nimport sys\nimport zipfile\n\nimport kickmix as km\nimport numpy as np\n\nroot = Path(sys.argv[1]) / "dfthcsos"\naudit = {"npz_headers": {}, "angle_circuits": {}}\nwith zipfile.ZipFile(root / "submission.npz") as z:\n    for member in z.infolist():\n        with z.open(member) as f:\n            version = np.lib.format.read_magic(f)\n            assert version == (1, 0), (member.filename, version)\n            shape, _, dtype = np.lib.format.read_array_header_1_0(f)\n        assert dtype.kind in ("f", "i", "u", "b") and dtype.byteorder != ">"\n        assert dtype.names is None and dtype.subdtype is None\n        audit["npz_headers"][member.filename] = {\n            "shape": list(shape), "dtype": dtype.str,\n            "array_bytes": math.prod(shape) * dtype.itemsize,\n            "npy_version": list(version),\n        }\naudit["total_npz_array_bytes"] = sum(x["array_bytes"] for x in audit["npz_headers"].values())\nassert audit["total_npz_array_bytes"] <= 2**26\nfor p in sorted(root.glob("angle_k_*.kmb")):\n    circuit = km.Circuit.from_file(str(p))\n    stats = circuit.operation_statistics()\n    audit["angle_circuits"][p.name] = {\n        "num_qubits": circuit.num_qubits, "num_bits": circuit.num_bits,\n        "registers": list(circuit.register_data), "t_gates": int(stats.num_t_like),\n        "static_toffolis": int(stats.num_toffoli_like),\n    }\n    assert stats.num_t_like == 0 and circuit.num_qubits <= 128\nprint(json.dumps(audit, indent=2))\n', 'LICENSE': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'blocks/dqi/pro-q8q9-889/source/build.py': '"""Builds the decoder that joins Pro q9\'s front half and Pro q8\'s conversion pair: python build.py OUTPUT.kmb.\n\nIt writes OUTPUT.kmb and OUTPUT.build.json with its digest and counts.\nThe entry is decode_q8q9.py, with Kickmix as the kit\'s builder module circ, and\nthe registers S0 to S69 and E0 to E4094 named in the file, as the team\'s own build did.\n"""\nimport hashlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nimport kickmix as km\n\nsys.modules[\'circ\'] = km\n\n\nHERE = Path(__file__).resolve().parent\ntarget = Path(sys.argv[1])\nstart = time.time()\nspec = importlib.util.spec_from_file_location(\'decoder_under_test\', HERE / \'decode_q8q9.py\')\nmodule = importlib.util.module_from_spec(spec)\nsys.modules[\'decoder_under_test\'] = module\nspec.loader.exec_module(module)\nb = km.CircuitBuilder()\nS = [b.alloc_qubits(12) for _ in range(70)]\nE = [b.alloc_bits(12) for _ in range(4095)]\nmodule.decode(b, S, E)\nmetadata = []\nfor i in range(70):\n    metadata.append(f\'REGISTER r{i} "S{i}"\')\n    metadata.extend(f\'APPEND_TO_REGISTER q{12 * i + k} r{i}\' for k in range(12))\nfor j in range(4095):\n    metadata.append(f\'REGISTER r{70 + j} "E{j}"\')\n    metadata.extend(f\'APPEND_TO_REGISTER b{12 * j + k} r{70 + j}\' for k in range(12))\nc = km.Circuit(\'\\n\'.join(metadata)) + b.finish_circuit()\nc.to_file(str(target), format=\'kmb\')\nwith open(target, \'rb\') as f:\n    digest = hashlib.file_digest(f, \'sha256\').hexdigest()\nrecord = dict(sha256=digest, Q=c.num_qubits, T_static=c.operation_statistics().num_toffoli_like, num_ops=len(c),\n              num_bits=c.num_bits, seconds=round(time.time() - start, 1))\ntarget.with_suffix(\'.build.json\').write_text(json.dumps(record, indent=1) + \'\\n\')\nprint(json.dumps(record), flush=True)\n', 'blocks/dqi/pro-q8q9-889/source/decode_frontier.py': '"""Entry point for the complete front-half research decoder."""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0, str(Path(__file__).resolve().parent/\'exp/frontier/root\'))\nfrom decode_combo_v18 import decode\n', 'blocks/dqi/pro-q8q9-889/source/decode_q8q9.py': '"""Pro q9\'s front half with Pro q8\'s converter pair, on the base\'s boundary."""\nfrom pathlib import Path\nimport importlib.util\nimport sys\nimport types\n\nimport circ\n\nROOT = Path(__file__).resolve().parent\n\n\ndef _registers_as_lists(builder):\n    """Pro q8\'s root blocks replace a register in place, so registers are Python lists, not Kickmix\'s arrays."""\n    alloc_qubits, alloc_bits = builder.alloc_qubits, builder.alloc_bits\n    builder.alloc_qubits = lambda self, n: list(alloc_qubits(self, n))\n    builder.alloc_bits = lambda self, n: list(alloc_bits(self, n))\n\n\ndef _load(name, filename):\n    spec = importlib.util.spec_from_file_location(name, filename)\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[name] = module\n    spec.loader.exec_module(module)\n    return module\n\n\n_registers_as_lists(circ.CircuitBuilder)\nfront = _load(\'pro_q9_front\', ROOT/\'decode_frontier.py\').decode\n_load(\'ported\', ROOT/\'research/production/ported.py\')\nback = _load(\'selected_round_two\', ROOT/\'research/production/final_root_v3.py\').decode\ng = dict(zip(back.__code__.co_freevars, back.__closure__))[\'g\'].cell_contents\nscope = dict(front.__globals__, convert_safe=g[\'convert_safe\'], baseline=g[\'baseline\'])\ndecode = types.FunctionType(front.__code__, scope, front.__name__, front.__defaults__, front.__closure__)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a03/inverse_consume17.py': '"""An 18-Toffoli consuming GF64 inverse with seventeen total live wires.\n\nThe fixed schedule was generated by exhaustive truth-table affine algebra.\nEach HMR character is an explicit quadratic in remaining wires plus a linear\ncharacter of the original input, which is returned to the caller.\n"""\nfrom pathlib import Path\nimport sys,json\nHERE=Path(__file__).resolve().parent\nsys.path.insert(0,str(HERE))\nfrom jet_norm6 import replacement_ids,base,xs\nfrom exp.jet24 import pro_jet as jet\nfrom exp.inverse_search import inverse20 as ref\nfrom exp.noscratch import schedules\nfrom exp.arithmetic import field as ar\nPLAN=json.loads((HERE/\'consuming_plan.json\').read_text())\n\n\ndef change_basis(b,live,rows):\n    """Apply an invertible affine map, with rows in constant-first format."""\n    n=len(live)\n    columns=[sum(((r>>(j+1))&1)<<i for i,r in enumerate(rows)) for j in range(n)]\n    ar.linear_inplace(b,live,columns)\n    for q,r in zip(live,rows):\n        if r&1:b.x(q)\n\n\ndef retire(b,live,pool,transcript,record):\n    change_basis(b,live,record[\'rows\'])\n    for action in record[\'retire\']:\n        p=action[\'index\']\n        for j in action[\'mix\']:b.cx(live[j],live[p])\n        q=live.pop(p)\n        m=b.alloc_bits(1)[0];b.hmr(q,m)\n        for term in action[\'phase\']:\n            if term[0]==\'constant\':b.z(m)\n            elif term[0]==\'linear\':b.cz(live[term[1]],m)\n            elif term[0]==\'pair\':b.ccz(live[term[1]],live[term[2]],m)\n            else:b.cx(m,transcript[term[1]])\n        b.free(m);pool.append(q)\n\n\ndef inverse6_consume17(b,x):\n    """x <- inverse0(x); return the exact original-x linear phase mask."""\n    assert len(x)==6\n    ar.linear_inplace(b,x,jet.TO6)\n    raw=b.alloc_qubits(6)\n    for j,(left,right) in enumerate(jet.RAW):\n        with ref.expose_pair(b,x,left,right) as (p,q):b.init_and(p,q,raw[j])\n    w=ref.inverse3_virtual(b,list(x)+list(raw),ref.norm_forms(1))\n    pool=list(b.alloc_qubits(2))\n    all_wires=list(x)+list(raw)+list(w)+pool[:]\n    live=list(x)+list(raw)+list(w)\n    transcript=b.alloc_bits(6);b.bit_store(transcript,False)\n    for step in PLAN[\'steps\']:\n        out=pool.pop()\n        with ref.expose_pair(b,live,step[\'left\'],step[\'right\']) as (p,q):b.init_and(p,q,out)\n        live.append(out)\n        retire(b,live,pool,transcript,step)\n    retire(b,live,pool,transcript,PLAN[\'final\'])\n    assert len(live)==6\n    for i,target in enumerate(x):\n        source=live[i]\n        if str(source)==str(target):continue\n        other=next((j for j,q in enumerate(live) if str(q)==str(target)),None)\n        b.swap(source,target)\n        live[i]=target\n        if other is not None:live[other]=source\n    retained=set(map(str,x))\n    b.free([q for q in all_wires if str(q) not in retained])\n    return transcript\n\n\ndef inverse_stream_narrow(b,x):\n    live=b.num_allocated_qubits\n    if len(x)==6 and live+12>889 and live+11<=889:\n        return inverse6_consume17(b,x)\n    return fallback(b,x)\n\nfallback=xs.clone(schedules.inverse_stream_noscratch,dict(schedules.inverse_stream_noscratch.__globals__,inverse_stream_noscratch=inverse_stream_narrow))\ndecode=xs.private_rebind(base.decode,replacement_ids(base.decode,schedules.inverse_stream_noscratch,inverse_stream_narrow))\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a03/inverse_consume17_front.py': '"""Install the consuming inverse only in extraction and synthesis closures."""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nfrom inverse_consume17 import inverse6_consume17,inverse_stream_narrow,fallback\nfrom jet_norm6 import base,xs,replacement_ids\nfrom exp.noscratch import schedules\n\n\ndef install(entry):\n    updates={}\n    for name in (\'extraction\',\'synthesize\'):\n        original=entry.__globals__[name]\n        replacements=replacement_ids(original,schedules.inverse_stream_noscratch,inverse_stream_narrow)\n        updates[name]=xs.private_rebind(original,replacements)\n    return xs.clone(entry,dict(entry.__globals__,**updates))\n\n\ndecode=install(base.decode)\nassert decode.__globals__[\'convert_safe\'] is base.decode.__globals__[\'convert_safe\']\nassert decode.__globals__[\'baseline\'] is base.decode.__globals__[\'baseline\']\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a03/jet_norm6.py': '"""Keep the six-bit inverse-norm source through each local jet reverse."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[3]\nsys.path.insert(0,str(ROOT/\'exp/r-main/narrow240\'))\nimport decode_combo as base\nimport exact_subs as xs\nimport inspect\nfrom types import FunctionType, SimpleNamespace, ModuleType\nfrom exp.jet24 import pro_jet as jet\nfrom exp.jet24.pro_jet import *\n\n\ndef start_inverse_kept(b,N):\n    """Return k=inv0(D), F=[N=0], D=Norm12(N), preserving all three."""\n    to,frm,K=tower(12)\n    linear_inplace(b,N,to)\n    D=b.alloc_qubits(6)\n    norm_xor_noscratch(b,N,D,K)\n    k,F=inverse6_zero18(b,D)\n    linear_inplace(b,N,frm)\n    return k,F,D\n\n\ndef erase_inverse_kept(b,t,N,k,F,fz,D):\n    """Erase N=inv0(t) using retained D=inv0(k); close all phases."""\n    to,frm,K=tower(12)\n    linear_inplace(b,t,to)\n    linear_inplace(b,N,to)\n    m=b.alloc_bits(12);b.hmr(N,m);b.free(N)\n    L=b.alloc_qubits(6);f=field(6)\n    def coeff():\n        for j in range(6):\n            for i in range(6):\n                for r in bits(f.mul(1<<i,1<<j)):\n                    b.ccx(m[r],t[i],L[j])\n                    b.ccx(m[r],t[i+6],L[j])\n                    b.ccx(m[r+6],t[i+6],L[j])\n    coeff();b.cz(L,D);coeff();b.free(L);b.free(m)\n    b.cz(F,fz[0]);b.free(fz)\n    fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n    b.z(fm[0])\n    mask=b.alloc_bits(6);b.bit_store(mask,False);b.cx(fm[0],mask[0])\n    ar.phase_product(b,D,k,mask);b.free(mask);b.free(fm)\n    dm=b.alloc_bits(6);b.hmr(D,dm);b.free(D)\n    xs.phase_inverse6_quad4(b,k,dm);b.free(dm)\n    ar._erase_norm(b,t,k,K);b.free(k)\n    linear_inplace(b,t,frm)\n\n_source=inspect.getsource(jet.reverse_canonical)\n_source=_source.replace(\'k, F = start_inverse(b, N)\',\'k, F, D = start_inverse_kept(b, N)\')\n_source=_source.replace(\'erase_inverse_cached(b, t, N, k, F, fz)\',\'erase_inverse_kept(b, t, N, k, F, fz, D)\')\n_env=dict(jet.reverse_canonical.__globals__,start_inverse_kept=start_inverse_kept,erase_inverse_kept=erase_inverse_kept)\nexec(compile(_source,__file__,\'exec\'),_env)\nreverse_canonical_kept=_env[\'reverse_canonical\']\n\n\ndef replacement_ids(entry, original, replacement):\n    seen=set();matches={}\n    def visit(v):\n        if id(v) in seen:return\n        seen.add(id(v))\n        if isinstance(v,FunctionType):\n            if v.__code__ is original.__code__:matches[id(v)]=replacement\n            for name in v.__code__.co_names:\n                if name in v.__globals__:visit(v.__globals__[name])\n            if v.__closure__:\n                for c in v.__closure__:visit(c.cell_contents)\n            for c in v.__code__.co_consts:\n                if hasattr(c,\'co_names\'):\n                    for name in c.co_names:\n                        if name in v.__globals__:visit(v.__globals__[name])\n        elif isinstance(v,(ModuleType,SimpleNamespace)):\n            path=getattr(v,\'__file__\',None)\n            if path is None or str(ROOT) in str(path):\n                for x in vars(v).values():visit(x)\n        elif isinstance(v,dict):\n            for x in v.values():visit(x)\n        elif isinstance(v,(tuple,list)):\n            for x in v:visit(x)\n    visit(entry);return matches\n\nreplacements=replacement_ids(base.decode,jet.reverse_canonical,reverse_canonical_kept)\nassert replacements\nbody=xs.private_rebind(base.decode,replacements)\ndef decode(b,S,E):\n    return body(b,S,E)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a03/jet_norm6_front.py': '"""Install the complete retained inverse-norm jet only in extraction."""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nfrom jet_norm6 import base,xs,replacement_ids,jet,reverse_canonical_kept\n\n\ndef install(entry):\n    original=entry.__globals__[\'extraction\']\n    replacements=replacement_ids(original,jet.reverse_canonical,reverse_canonical_kept)\n    extraction=xs.private_rebind(original,replacements)\n    return xs.clone(entry,dict(entry.__globals__,extraction=extraction))\n\n\ndecode=install(base.decode)\nassert decode.__globals__[\'convert_safe\'] is base.decode.__globals__[\'convert_safe\']\nassert decode.__globals__[\'baseline\'] is base.decode.__globals__[\'baseline\']\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a04/nonzero_consume16.py': '"""Consume a promised-nonzero GF64 input with an exact phase transcript."""\nfrom pathlib import Path\nfrom types import FunctionType,SimpleNamespace,ModuleType,CodeType\nimport sys,json\nHERE=Path(__file__).resolve().parent\nROOT=HERE.parents[2]\nsys.path[:0]=[str(ROOT/\'exp/r-main/narrow240\')]\nimport decode_combo as combo\nimport frozen_chart\nimport retained_core_safe27 as retained\nimport norm_cache\nfrom exp.jet24 import pro_jet as jet\nfrom exp.inverse_search import inverse20 as ref\nfrom exp.noscratch import schedules\nfrom exp.arithmetic import field as ar\nPLAN=json.loads((HERE/\'nonzero_plan16.json\').read_text())\nTRACE=[]\n\ndef change_basis(b,live,rows):\n    columns=[sum(((r>>(j+1))&1)<<i for i,r in enumerate(rows)) for j in range(len(live))]\n    ar.linear_inplace(b,live,columns)\n    for q,r in zip(live,rows):\n        if r&1:b.x(q)\n\ndef retire(b,live,pool,transcript,record):\n    change_basis(b,live,record[\'rows\'])\n    for action in record[\'retire\']:\n        p=action[\'index\']\n        for j in action[\'mix\']:b.cx(live[j],live[p])\n        q=live.pop(p);m=b.alloc_bits(1)[0];b.hmr(q,m)\n        for term in action[\'phase\']:\n            if term[0]==\'constant\':b.z(m)\n            elif term[0]==\'linear\':b.cz(live[term[1]],m)\n            elif term[0]==\'pair\':b.ccz(live[term[1]],live[term[2]],m)\n            else:b.cx(m,transcript[term[1]])\n        b.free(m);pool.append(q)\n\ndef inverse6_nonzero_consume16(b,x):\n    """x <- 1/x for x!=0; return original-x HMR character in canonical basis."""\n    assert len(x)==6\n    ar.linear_inplace(b,x,jet.TO6)\n    raw=b.alloc_qubits(6)\n    for j,(left,right) in enumerate(jet.RAW):\n        with ref.expose_pair(b,x,left,right) as (p,q):b.init_and(p,q,raw[j])\n    w=ref.inverse3_virtual(b,list(x)+list(raw),ref.norm_forms(1))\n    pool=list(b.alloc_qubits(1))\n    all_wires=list(x)+list(raw)+list(w)+pool[:]\n    live=list(x)+list(raw)+list(w)\n    transcript=b.alloc_bits(6);b.bit_store(transcript,False)\n    retire(b,live,pool,transcript,PLAN[\'begin\'])\n    for step in PLAN[\'steps\']:\n        out=pool.pop()\n        with ref.expose_pair(b,live,step[\'left\'],step[\'right\']) as (p,q):b.init_and(p,q,out)\n        live.append(out);retire(b,live,pool,transcript,step)\n    retire(b,live,pool,transcript,PLAN[\'final\'])\n    assert len(live)==6\n    for i,target in enumerate(x):\n        source=live[i]\n        if str(source)==str(target):continue\n        other=next((j for j,q in enumerate(live) if str(q)==str(target)),None)\n        b.swap(source,target);live[i]=target\n        if other is not None:live[other]=source\n    retained=set(map(str,x));b.free([q for q in all_wires if str(q) not in retained])\n    return transcript\n\ndef stream_nonzero(b,x):\n    live=b.num_allocated_qubits\n    if len(x)==6:\n        selected=live+10<=889\n        TRACE.append(dict(live=live,selected=selected))\n        if selected:return inverse6_nonzero_consume16(b,x)\n    return fallback(b,x)\n\nfallback=combo.xsub.clone(schedules.inverse_stream_noscratch,dict(schedules.inverse_stream_noscratch.__globals__,inverse_stream_noscratch=stream_nonzero))\n\ndef functions(entry):\n    seen=set();found=[]\n    def code_names(code):\n        names=set(code.co_names)\n        for item in code.co_consts:\n            if isinstance(item,CodeType):names.update(code_names(item))\n        return names\n    def visit(value):\n        if id(value) in seen:return\n        seen.add(id(value))\n        if isinstance(value,FunctionType):\n            found.append(value)\n            for name in code_names(value.__code__):\n                if name in value.__globals__:visit(value.__globals__[name])\n            for cell in value.__closure__ or ():visit(cell.cell_contents)\n        elif isinstance(value,(SimpleNamespace,ModuleType)):\n            path=getattr(value,\'__file__\',None)\n            if path is None or str(ROOT) in str(path):\n                for item in vars(value).values():visit(item)\n        elif isinstance(value,dict):\n            for item in value.values():visit(item)\n        elif isinstance(value,(tuple,list)):\n            for item in value:visit(item)\n    visit(entry);return found\n\ndef make_stream(ceiling):\n    def chosen(b,x):\n        if len(x)==6 and b.num_allocated_qubits+10<=ceiling:\n            return inverse6_nonzero_consume16(b,x)\n        return recursive(b,x)\n    recursive=combo.xsub.clone(schedules.inverse_stream_noscratch,dict(schedules.inverse_stream_noscratch.__globals__,inverse_stream_noscratch=chosen))\n    return chosen\n\ndef keep_for_fast_inverse(b,D):\n    k=min(6,max(0,889-b.num_allocated_qubits-10))\n    to,back=norm_cache.layout(k)\n    saved=b.alloc_qubits(k)\n    ar.linear_xor(b,D,saved,[v&((1<<k)-1) for v in to])\n    return saved\n\ndef install(entry,forward=True,reverse=True,character=True,ceiling=889,retune_cache=False):\n    """Patch only chart routines whose input/scalar norm is nonzero by invariant."""\n    codes=set()\n    if forward:codes.add(frozen_chart.normalize.__code__)\n    if reverse:codes.add(retained.normalize_early.__code__)\n    if character:codes.add(retained.linear_inverse_character.__code__)\n    stream=stream_nonzero if ceiling==889 else make_stream(ceiling)\n    extraction=entry.__globals__[\'extraction\'];replacements={}\n    for fn in functions(extraction):\n        if fn.__code__ in codes:\n            scope=dict(fn.__globals__,inverse_stream_noscratch=stream)\n            if retune_cache and fn.__code__ is retained.linear_inverse_character.__code__:\n                old_core=scope[\'core\']\n                cached=SimpleNamespace(**dict(vars(old_core.norm_cache),keep=keep_for_fast_inverse))\n                scope[\'core\']=SimpleNamespace(**dict(vars(old_core),norm_cache=cached))\n            replacements[id(fn)]=combo.xsub.clone(fn,scope)\n    assert replacements\n    scope=dict(entry.__globals__,extraction=combo.xsub.private_rebind(extraction,replacements))\n    return combo.xsub.clone(entry,scope)\n\ndecode=install(combo.decode)\nassert decode.__globals__[\'convert_safe\'] is combo.decode.__globals__[\'convert_safe\']\nassert decode.__globals__[\'baseline\'] is combo.decode.__globals__[\'baseline\']\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a05/scalar_projection.py': '"""Retain a small first-state projection to erase the third scalar product.\n\nThis adds no unproved value promise. Every measured output character is\npaid from the cached linear state and the product\'s still-live inputs.\n"""\nfrom pathlib import Path\nfrom functools import lru_cache\nimport sys\n\nHERE = Path(__file__).resolve().parent\nKIT_ROOT = HERE.parents[2]\nsys.path[:0] = [str(KIT_ROOT / \'exp/r-main/narrow240\'), str(HERE.parent / \'a03\')]\nimport decode_combo as combo\nimport retained_core_safe27 as ret\nfrom jet_norm6 import replacement_ids\nfrom retained_core_safe27 import *\nfrom exp.mul99.kernels import tensor99_extension\nfrom exp.arithmetic.division import phase_product24\nfrom replay_erase import scalar_erase_second24\n\n\n@lru_cache(None)\ndef projection(k):\n    terms = tensor99_extension(K)\n    first = [w for _, w in terms[:9]]\n    states = {frozenset((0,)): ()}\n    for _ in range(k):\n        following = {}\n        for span, basis in states.items():\n            for value in first:\n                if value in span:\n                    continue\n                grown = span | {v ^ value for v in span}\n                following.setdefault(grown, basis + (value,))\n        states = following\n    span, columns = max(states.items(), key=lambda s: (sum(w in s[0] for _, w in terms), s[1]))\n    columns = list(columns)\n    pivots = {}\n    def add(value):\n        while value:\n            p = value.bit_length() - 1\n            if p in pivots:\n                value ^= pivots[p]\n            else:\n                pivots[p] = value\n                return True\n        return False\n    for value in columns:\n        assert add(value)\n    for value in (1 << i for i in range(24)):\n        if add(value):\n            columns.append(value)\n    to = tuple(ar.inverse_columns(columns))\n    projected = tuple((m, m, ar.apply(to, w) & ~((1 << k) - 1)) for m, w in terms)\n    return to, tuple(columns), tuple(t for t in projected if t[2])\n\n\ndef copy_first_projection(b, first_state, q1, k):\n    to, _, _ = projection(k)\n    cache = b.alloc_qubits(k)\n    columns = [ar.apply(to, emul(q1, 1 << i)) & ((1 << k) - 1) for i in range(24)]\n    ar.linear_xor(b, first_state, cache, columns)\n    return cache\n\n\ndef erase_third_step(b, head, N, D, q, ct, cache):\n    # The standard reverse step starts by exposing D3 as N and D2 as D.\n    scalepair(b, N, inv(q))\n    b.swap(N, D)\n    k = len(cache)\n    to, back, terms = projection(k)\n    ar.linear_inplace(b, N, to)\n    a, h = head\n    cx(b, h, a, ct)\n    xsub.accumulator(b, combine(head), D, N, terms)\n    m = b.alloc_bits(k)\n    b.hmr(N[:k], m)\n    b.cz(cache, m)\n    original_mask = b.alloc_bits(24)\n    b.bit_store(original_mask, False)\n    for j, column in enumerate(to):\n        for i in ar.bits(column & ((1 << k) - 1)):\n            b.cx(m[i], original_mask[j])\n    phase_product24(b, combine(head), D, original_mask, K)\n    b.free(original_mask)\n    b.free(m)\n    b.cx(cache, N[:k])\n    cx(b, h, a, ct)\n    ar.linear_inplace(b, N, back)\n    ar.linear_xor(b, N, cache, [column & ((1 << k) - 1) for column in to])\n    b.free(cache)\n\n\ndef full_core_phase(b, N, heads, params, ct, masks, pole, dirty):\n    selected = CACHE_BY_HEADS.get(len(heads), 0)\n    D = b.alloc_qubits(24)\n    b.x(D[0])\n    cache = None\n    for i, (head, (q, c)) in enumerate(zip(heads, params)):\n        ref.external_scalar_step(b, head, N, D, q, ct, first=i == 0)\n        if i == 0 and selected:\n            cache = copy_first_projection(b, D, params[1][0], selected)\n    b.cswap(pole, N, D)\n    A, g = D[:12], D[12:]\n    zero_affine_character(b, g, A, masks[\'g\'], masks[\'f\'][0])\n    zero = b.alloc_bits(12)\n    b.bit_store(zero, False)\n    masks[\'i\'] = zero\n    b.cz(g, masks[\'g\'])\n    b.cz(A, masks[\'h\'])\n    f = ar.field(12)\n    for j in range(12):\n        for r in ar.bits(f.mul(K, 1 << j)):\n            b.cz(g[j], masks[\'h\'][r])\n    core.affine_inverse_g_character(b, A, g, masks)\n    for m in masks.values():\n        b.free(m)\n    b.cswap(pole, N, D)\n    fm = b.alloc_bits(1)\n    b.hmr(pole, fm[0])\n    b.free(pole)\n    b.push_condition(fm[0])\n    clean_zero_phase24(b, D)\n    b.pop_condition()\n    b.free(fm)\n    for i in reversed(range(len(heads))):\n        if i == 2 and cache is not None:\n            erase_third_step(b, heads[i], N, D, params[i][0], ct, cache)\n        elif i == 1:\n            scalar_erase_second24(b, heads[i], N, D, params[i][0], ct, params[0][0])\n        else:\n            ref.external_scalar_step(b, heads[i], N, D, params[i][0], ct, True, first=i == 0)\n    b.x(D[0])\n    b.free(D)\n\n\ndef install(entry, cache_by_gi):\n    env = dict(full_core_phase.__globals__, CACHE_BY_HEADS={2 * gi: k for gi, k in cache_by_gi.items()})\n    candidate = combo.xsub.clone(full_core_phase, env)\n    candidate = combo.xsub.private_rebind(candidate, combo.synth.replacements)\n    changes = replacement_ids(entry, ret.full_core_phase, candidate)\n    assert changes\n    scope = dict(entry.__globals__)\n    scope[\'extraction\'] = combo.xsub.private_rebind(scope[\'extraction\'], changes)\n    return combo.xsub.clone(entry, scope)\n\n\nCACHE_BY_HEADS = {}\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a05/scalar_projection12.py': '"""Cache the complete linear first state of the trailing subfield replay."""\nfrom types import SimpleNamespace\nimport scalar_projection as shared\nfrom scalar_projection import *\nimport replay_erase\n\n\ndef erase_third12(b, head, N, D, q, cache):\n    scale(b, N, inv(q))\n    b.swap(N, D)\n    mask = b.alloc_bits(12)\n    b.hmr(N, mask)\n    b.cz(cache, mask)\n    ar.phase_product(b, head[0], D, mask)\n    b.free(mask)\n    b.cx(cache, N)\n    b.cx(N, cache)\n    b.free(cache)\n\n\ndef install(entry):\n    original = entry.__globals__[\'extraction\']\n    ref = original.__globals__[\'ref\']\n    original_step = ref.scalar_step\n    state = {}\n    def scalar_step(b, head, N, D, q, reverse=False, first=False):\n        if not reverse:\n            original_step(b, head, N, D, q, reverse, first)\n            if first:\n                state.clear()\n                cache = b.alloc_qubits(12)\n                b.cx(D, cache)\n                scale(b, cache, ref.ks[1])\n                state.update(cache=cache, index=0)\n            else:\n                state[\'index\'] += 1\n                if state[\'index\'] == 2:\n                    state[\'third_head\'] = head\n        elif head is state.get(\'third_head\'):\n            erase_third12(b, head, N, D, q, state.pop(\'cache\'))\n        else:\n            original_step(b, head, N, D, q, reverse, first)\n    patched_ref = SimpleNamespace(**dict(vars(ref), scalar_step=scalar_step))\n    extraction = combo.xsub.clone(original, dict(original.__globals__, ref=patched_ref))\n    return combo.xsub.clone(entry, dict(entry.__globals__, extraction=extraction))\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a05/scalar_projection_multi.py': '"""Whole-block projection variants and composition-preserving installer."""\nfrom functools import lru_cache\nfrom itertools import combinations\nfrom pathlib import Path\nimport sys\nfrom types import FunctionType, ModuleType, SimpleNamespace, CodeType\nimport scalar_projection as single\nfrom scalar_projection import *\n\n\ndef independent(values):\n    pivots = {}\n    result = []\n    for original in values:\n        value = original\n        while value:\n            p = value.bit_length() - 1\n            if p in pivots:\n                value ^= pivots[p]\n            else:\n                pivots[p] = value\n                result.append(original)\n                break\n    return tuple(result)\n\n\ndef span_of(basis):\n    span = {0}\n    for value in basis:\n        span |= {v ^ value for v in span}\n    return frozenset(span)\n\n\n@lru_cache(None)\ndef projection(k):\n    if k <= 4:\n        return single.projection(k)\n    terms = tensor99_extension(K)\n    blocks = [independent([w for _, w in terms[i:i+9]]) for i in range(0, 99, 9)]\n    # Complete GF16 output blocks first, then choose every subspace of one\n    # further block. This is a concrete finite family, not a rank lower bound.\n    whole, rest = divmod(k, 4)\n    options = {}\n    for indices in combinations(range(len(blocks)), whole):\n        basis = independent([v for i in indices for v in blocks[i]])\n        if len(basis) != whole * 4:\n            continue\n        if not rest:\n            options.setdefault(span_of(basis), basis)\n            continue\n        for j, block in enumerate(blocks):\n            if j in indices:\n                continue\n            states = {span_of(basis): basis}\n            for _ in range(rest):\n                grown = {}\n                for span, old in states.items():\n                    for v in span_of(block):\n                        if v in span:\n                            continue\n                        bigger = span | {x ^ v for x in span}\n                        grown.setdefault(bigger, old + (v,))\n                states = grown\n            options.update(states)\n    span, basis = max(options.items(), key=lambda x: (sum(v in x[0] for _, v in terms), x[1]))\n    columns = independent(list(basis) + [1 << i for i in range(24)])\n    to = tuple(ar.inverse_columns(columns))\n    projected = tuple((m, m, ar.apply(to, w) & ~((1 << k) - 1)) for m, w in terms)\n    return to, columns, tuple(t for t in projected if t[2])\n\n\ndef functions(entry):\n    seen = set()\n    out = []\n    def names(code):\n        result = set(code.co_names)\n        for c in code.co_consts:\n            if isinstance(c, CodeType):\n                result.update(names(c))\n        return result\n    def visit(value):\n        if id(value) in seen:\n            return\n        seen.add(id(value))\n        if isinstance(value, FunctionType):\n            out.append(value)\n            for name in names(value.__code__):\n                if name in value.__globals__:\n                    visit(value.__globals__[name])\n            for cell in value.__closure__ or ():\n                visit(cell.cell_contents)\n        elif isinstance(value, (ModuleType, SimpleNamespace)):\n            path = getattr(value, \'__file__\', None)\n            if path is None or str(KIT_ROOT) in str(path):\n                for item in vars(value).values():\n                    visit(item)\n        elif isinstance(value, dict):\n            for item in value.values():\n                visit(item)\n        elif isinstance(value, (tuple, list)):\n            for item in value:\n                visit(item)\n    visit(entry)\n    return out\n\n\ndef install(entry, cache_by_gi):\n    copy = combo.xsub.clone(copy_first_projection, dict(copy_first_projection.__globals__, projection=projection))\n    erase = combo.xsub.clone(erase_third_step, dict(erase_third_step.__globals__, projection=projection))\n    # Preserve every already-installed arithmetic descendant of each matched\n    # full_core_phase. Only these two new routines and the schedule are new.\n    replacements = {}\n    original_extraction = entry.__globals__[\'extraction\']\n    for function in functions(original_extraction):\n        if function.__code__ is not ret.full_core_phase.__code__:\n            continue\n        scope = dict(function.__globals__,\n                     CACHE_BY_HEADS={2 * gi: k for gi, k in cache_by_gi.items()},\n                     copy_first_projection=copy, erase_third_step=erase,\n                     scalar_erase_second24=scalar_erase_second24)\n        replacements[id(function)] = combo.xsub.clone(full_core_phase, scope)\n    assert replacements\n    scope = dict(entry.__globals__, extraction=combo.xsub.private_rebind(original_extraction, replacements))\n    return combo.xsub.clone(entry, scope)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a07/decode_correction_and.py': '"""Use a measured AND for the synthesis correction predicate.\n\nThe shared-factor kernel is the root\'s existing exact correction.  This\nvariant changes only its F=[d=0] AND NOT pole predicate cleanup.\n"""\nfrom pathlib import Path\nimport sys\nHERE = Path(__file__).resolve().parent\nROOT = HERE.parents[2]\nsys.path.insert(0, str(ROOT / \'exp/r-main/narrow240\'))\nimport decode_combo as combo\n\n\ndef term_correction_with_inverse(b, p, q, d, u, v, pole, qi,\n                                 reverse=False, token=None, Z=None):\n    if reverse:\n        Z = guarded.fresh_zero_flag(b, d)\n        b.cz(Z, token[0])\n        b.free(token)\n    F = b.alloc_qubits(1)[0]\n    b.x(pole)\n    b.init_and(Z, pole, F)\n    b.x(pole)\n    factor = b.alloc_qubits(12)\n    b.init_and(F, qi, factor)\n    if reverse:\n        ar.mul_xor0(b, v, factor, u)\n        ar.mul_xor0(b, p, factor, v)\n    else:\n        ar.mul_xor0(b, p, factor, v)\n        ar.mul_xor0(b, v, factor, u)\n    b.del_and(F, qi, factor)\n    b.free(factor)\n    b.x(pole)\n    b.del_and(Z, pole, F)\n    b.x(pole)\n    b.free(F)\n    if reverse:\n        return Z\n    zm = b.alloc_bits(1)\n    b.hmr(Z, zm[0])\n    b.free(Z)\n    return zm\n\n\ndef install(entry):\n    synth = entry.__globals__[\'synthesize\']\n    tail = synth.__globals__[\'tail_inplace\']\n    finish = tail.__globals__[\'tail_finish\']\n    old = finish.__globals__[\'term_correction_with_inverse\']\n    new = combo.xsub.clone(term_correction_with_inverse, dict(old.__globals__))\n    scope = dict(entry.__globals__)\n    scope[\'synthesize\'] = combo.xsub.private_rebind(synth, {id(old): new})\n    return combo.xsub.clone(entry, scope)\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a07/decode_prefix_gauge.py': '"""Absorb the terminal continuant product into the prefix\'s linear head gauge.\n\nThe old prefix accumulator, plus its final r*border product, equals the new\naccumulator identically.  No additional qubit or measurement is introduced.\n"""\nfrom pathlib import Path\nimport sys\nimport inspect\n\nHERE = Path(__file__).resolve().parent\nROOT = HERE.parents[2]\nsys.path.insert(0, str(ROOT / \'exp/r-main/narrow240\'))\nimport decode_combo as combo\n\n\ndef install(entry):\n    original = entry.__globals__[\'synthesize\']\n    source = inspect.getsource(original)\n    edits = (\n        (\'b.cx(border,r);b.cx(border,d);b.cx(r,border);b.x(border[0])\',\n         \'b.cx(border,r);b.cx(r,border);b.x(border[0])\'),\n        (\'folds.add_tail(b,heads[i],qi,d,bool(i%2))\',\n         \'folds.add_tail(b,heads[i],qi,d,not bool(i%2))\'),\n        (\'ar.mul_xor0(b,r,border,d);tt=tail_inplace\',\n         \'tt=tail_inplace\'),\n        (\'tail_inplace(b,r,border,d,u,v,pole,True,tt);ar.mul_xor0(b,r,border,d)\',\n         \'tail_inplace(b,r,border,d,u,v,pole,True,tt)\'),\n        (\'b.cx(r,d);b.x(border[0]);b.cx(r,border);b.cx(border,r);b.free(r)\',\n         \'b.x(border[0]);b.cx(r,border);b.cx(border,r);b.free(r)\'),\n    )\n    for before, after in edits:\n        expected = 2 if before.startswith(\'folds.add_tail\') else 1\n        assert source.count(before) == expected, (before, source.count(before))\n        source = source.replace(before, after)\n    env = dict(original.__globals__)\n    exec(compile(source, __file__, \'exec\'), env)\n    out = combo.xsub.clone(entry, dict(entry.__globals__, synthesize=env[\'synthesize\']))\n    assert out.__globals__[\'convert_safe\'] is entry.__globals__[\'convert_safe\']\n    assert out.__globals__[\'baseline\'] is entry.__globals__[\'baseline\']\n    return out\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a08/decode_unitary3.py': '"""Whole-decoder control: exact in-place GF8 inversion in scalar streams."""\nfrom pathlib import Path\nimport sys\nHERE=Path(__file__).resolve().parent\nROOT=HERE.parents[2]\nsys.path[:0]=[str(ROOT/\'exp/r-main/narrow240\'),str(HERE.parent/\'a03\')]\nimport decode_combo as combo\nfrom jet_norm6 import replacement_ids\nfrom exp.noscratch import schedules\nfrom exp.arithmetic import field as ar\n\n\ndef transposition_layout(a,c):\n    difference=a^c\n    orthogonal=[v for v in range(1,8) if not (v&difference).bit_count()%2]\n    rows=[orthogonal[0],next(v for v in orthogonal if v!=orthogonal[0]),\n          next(v for v in range(1,8) if (v&difference).bit_count()%2)]\n    columns=[sum(((row>>j)&1)<<i for i,row in enumerate(rows)) for j in range(3)]\n    image=ar.apply(columns,a)\n    return columns,ar.inverse_columns(columns),[i for i in (0,1) if not (image>>i)&1]\n\n\nTRANSPOSE=[transposition_layout(a,ar.field(3).invert(a)) for a in range(1,8)\n           if a<ar.field(3).invert(a)]\n\n\ndef inverse3_inplace(b,x):\n    """The three disjoint reciprocal transpositions, with no fresh qubit."""\n    for columns,back,negated in TRANSPOSE:\n        ar.linear_inplace(b,x,columns)\n        for i in negated:b.x(x[i])\n        b.ccx(x[0],x[1],x[2])\n        for i in reversed(negated):b.x(x[i])\n        ar.linear_inplace(b,x,back)\n\n\ndef stream(b,x):\n    if len(x)==3:\n        inverse3_inplace(b,x)\n        mask=b.alloc_bits(3);b.bit_store(mask,False)\n        return mask\n    return recursive(b,x)\n\n\nrecursive=combo.xsub.clone(schedules.inverse_stream_noscratch,\n                         dict(schedules.inverse_stream_noscratch.__globals__,inverse_stream_noscratch=stream))\n\n\ndef install(entry):\n    replacements=replacement_ids(entry,schedules.inverse_stream_noscratch,stream)\n    assert replacements\n    scope=dict(entry.__globals__)\n    for name in (\'extraction\',\'synthesize\'):\n        scope[name]=combo.xsub.private_rebind(scope[name],replacements)\n    return combo.xsub.clone(entry,scope)\n\n\ndecode=install(combo.decode)\nassert decode.__globals__[\'convert_safe\'] is combo.decode.__globals__[\'convert_safe\']\nassert decode.__globals__[\'baseline\'] is combo.decode.__globals__[\'baseline\']\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a08/inverse19_consume16.py': '"""All-input consuming GF64 inverse using sixteen total live wires."""\nfrom pathlib import Path\nimport sys,json\nHERE=Path(__file__).resolve().parent\nROOT=HERE.parents[2]\nsys.path[:0]=[str(ROOT/\'exp/r-main/narrow240\'),str(HERE.parent/\'a03\')]\nimport decode_combo as combo\nfrom jet_norm6 import replacement_ids\nfrom inverse_consume17 import inverse6_consume17\nfrom exp.inverse_width_subspaces import kernel19 as basic\nfrom exp.inverse_search import inverse20 as ref\nfrom exp.noscratch import schedules\nfrom exp.arithmetic import field as ar\nPLAN=json.loads((HERE/\'generic19_plan16.json\').read_text())\nassert PLAN[\'domain\']==\'all\' and PLAN[\'max_live\']==16\n\n\ndef change_basis(b,live,rows):\n    columns=[sum(((r>>(j+1))&1)<<i for i,r in enumerate(rows)) for j in range(len(live))]\n    ar.linear_inplace(b,live,columns)\n    for q,row in zip(live,rows):\n        if row&1:b.x(q)\n\n\ndef retire(b,live,pool,transcript,record):\n    change_basis(b,live,record[\'rows\'])\n    for action in record[\'retire\']:\n        p=action[\'index\']\n        for j in action[\'mix\']:b.cx(live[j],live[p])\n        q=live.pop(p);m=b.alloc_bits(1)[0];b.hmr(q,m)\n        for term in action[\'phase\']:\n            if term[0]==\'constant\':b.z(m)\n            elif term[0]==\'linear\':b.cz(live[term[1]],m)\n            elif term[0]==\'pair\':b.ccz(live[term[1]],live[term[2]],m)\n            else:b.cx(m,transcript[term[1]])\n        b.free(m);pool.append(q)\n\n\ndef inverse6_consume19(b,x):\n    """Consume x, return inv0(x) and the original-input linear HMR mask."""\n    to,back,K=ar.tower(6);ar.linear_inplace(b,x,to)\n    raw=b.alloc_qubits(6)\n    for j,(left,right) in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,x,left,right) as (a,c):b.init_and(a,c,raw[j])\n    inner=ref.inverse3_virtual(b,list(x)+list(raw),ref.norm_forms(K))\n    pool=list(b.alloc_qubits(1))\n    all_wires=list(x)+list(raw)+list(inner)+pool[:]\n    live=list(x)+list(raw)+list(inner)\n    transcript=b.alloc_bits(6);b.bit_store(transcript,False)\n    retire(b,live,pool,transcript,PLAN[\'begin\'])\n    for step in PLAN[\'steps\']:\n        if step[\'kind\']==\'new\':\n            target=pool.pop()\n            with ref.expose_pair(b,live,step[\'left\'],step[\'right\']) as (a,c):\n                b.init_and(a,c,target)\n            live.append(target)\n        else:\n            change_basis(b,live,step[\'pre_rows\'])\n            with ref.expose_pair(b,live,step[\'left\'],step[\'right\']) as (a,c):\n                b.ccx(a,c,live[step[\'target\']])\n        retire(b,live,pool,transcript,step)\n    retire(b,live,pool,transcript,PLAN[\'final\'])\n    assert len(live)==6\n    for i,target in enumerate(x):\n        source=live[i]\n        if str(source)==str(target):continue\n        other=next((j for j,q in enumerate(live) if str(q)==str(target)),None)\n        b.swap(source,target);live[i]=target\n        if other is not None:live[other]=source\n    retained=set(map(str,x))\n    b.free([q for q in all_wires if str(q) not in retained])\n    return transcript\n\n\ndef stream(b,x):\n    if len(x)==6:\n        live=b.num_allocated_qubits\n        if live+12>889 and live+11<=889:return inverse6_consume17(b,x)\n        if live+11>889 and live+10<=889:return inverse6_consume19(b,x)\n    return recursive(b,x)\n\n\nrecursive=combo.xsub.clone(schedules.inverse_stream_noscratch,\n                         dict(schedules.inverse_stream_noscratch.__globals__,inverse_stream_noscratch=stream))\n\n\ndef install(entry):\n    replacements=replacement_ids(entry,schedules.inverse_stream_noscratch,stream)\n    assert replacements\n    scope=dict(entry.__globals__)\n    for name in (\'extraction\',\'synthesize\'):\n        scope[name]=combo.xsub.private_rebind(scope[name],replacements)\n    return combo.xsub.clone(entry,scope)\n\n\ndecode=install(combo.decode)\nassert decode.__globals__[\'convert_safe\'] is combo.decode.__globals__[\'convert_safe\']\nassert decode.__globals__[\'baseline\'] is combo.decode.__globals__[\'baseline\']\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a08/inverse20_consume14.py': '"""Consume a GF64 input with the isotopy20 circuit and fourteen wires."""\nfrom pathlib import Path\nimport sys,json\nHERE=Path(__file__).resolve().parent\nsys.path.insert(0,str(HERE))\nfrom inverse19_consume16 import (ar,ref,combo,schedules,replacement_ids,\n                               change_basis,retire,inverse6_consume17,inverse6_consume19)\nfrom decode_unitary3 import inverse3_inplace\nfrom exp.inverse_search import inverse20_iso as iso\nPLAN=json.loads((HERE/\'generic20_plan.json\').read_text())\nassert PLAN[\'domain\']==\'all\' and PLAN[\'max_live\']==14\n\n\ndef inverse6_consume20(b,x):\n    """Exact inverse0; consume the source and return its linear character."""\n    to,back,K=ar.tower(6);ar.linear_inplace(b,x,to)\n    raw=b.alloc_qubits(6)\n    for j,(left,right) in enumerate(iso.raw_forms()):\n        with ref.expose_pair(b,x,left,right) as (a,c):b.init_and(a,c,raw[j])\n    rows=iso.NORM_ROWS+iso.EXTRA_ROWS\n    columns=[sum(((row>>j)&1)<<i for i,row in enumerate(rows)) for j in range(6)]\n    ar.linear_inplace(b,raw,columns)\n    for j,form in enumerate(ref.norm_forms(K)):\n        ref.form_xor(b,x,form&127,raw[j])\n    inverse3_inplace(b,raw[:3])\n    pool=list(b.alloc_qubits(2))\n    all_wires=list(x)+list(raw)+pool[:]\n    live=list(x)+list(raw)\n    transcript=b.alloc_bits(6);b.bit_store(transcript,False)\n    retire(b,live,pool,transcript,PLAN[\'begin\'])\n    for step in PLAN[\'steps\']:\n        if step[\'kind\']==\'new\':\n            target=pool.pop()\n            with ref.expose_pair(b,live,step[\'left\'],step[\'right\']) as (a,c):\n                b.init_and(a,c,target)\n            live.append(target)\n        else:\n            change_basis(b,live,step[\'pre_rows\'])\n            with ref.expose_pair(b,live,step[\'left\'],step[\'right\']) as (a,c):\n                b.ccx(a,c,live[step[\'target\']])\n        retire(b,live,pool,transcript,step)\n    retire(b,live,pool,transcript,PLAN[\'final\'])\n    assert len(live)==6\n    for i,target in enumerate(x):\n        source=live[i]\n        if str(source)==str(target):continue\n        other=next((j for j,q in enumerate(live) if str(q)==str(target)),None)\n        b.swap(source,target);live[i]=target\n        if other is not None:live[other]=source\n    retained=set(map(str,x))\n    b.free([q for q in all_wires if str(q) not in retained])\n    return transcript\n\n\ndef stream(b,x):\n    if len(x)==6:\n        live=b.num_allocated_qubits\n        if live+12>889 and live+11<=889:return inverse6_consume17(b,x)\n        if live+11>889 and live+10<=889:return inverse6_consume19(b,x)\n        if live+10>889 and live+8<=889:return inverse6_consume20(b,x)\n    return recursive(b,x)\n\n\nrecursive=combo.xsub.clone(schedules.inverse_stream_noscratch,\n                         dict(schedules.inverse_stream_noscratch.__globals__,inverse_stream_noscratch=stream))\n\n\ndef install(entry):\n    replacements=replacement_ids(entry,schedules.inverse_stream_noscratch,stream)\n    scope=dict(entry.__globals__)\n    for name in (\'extraction\',\'synthesize\'):\n        scope[name]=combo.xsub.private_rebind(scope[name],replacements)\n    return combo.xsub.clone(entry,scope)\n\n\ndecode=install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a08/preserve_streams.py': '"""Add narrow generic leaves without replacing any existing recursive body."""\nfrom pathlib import Path\nimport sys\nHERE=Path(__file__).resolve().parent\nsys.path[:0]=[str(HERE),str(HERE.parent/\'a04\')]\nfrom inverse20_consume14 import inverse6_consume20,inverse6_consume19,combo,schedules\nfrom nonzero_consume16 import functions\n\n\ndef wrap(original,ceiling):\n    def chosen(b,x):\n        if len(x)==6:\n            space=ceiling-b.num_allocated_qubits\n            if space==10:return inverse6_consume19(b,x)\n            if 8<=space<10:return inverse6_consume20(b,x)\n        return original(b,x)\n    return chosen\n\n\ndef install(entry,ceiling=889):\n    scope=dict(entry.__globals__)\n    for name in (\'extraction\',\'synthesize\'):\n        original=scope[name]\n        replacements={id(fn):wrap(fn,ceiling) for fn in functions(original)\n                      if fn.__code__ is schedules.inverse_stream_noscratch.__code__}\n        scope[name]=combo.xsub.private_rebind(original,replacements)\n    result=combo.xsub.clone(entry,scope)\n    for name in (\'convert_safe\',\'baseline\'):\n        assert result.__globals__[name] is entry.__globals__[name]\n    return result\n\n\ndecode=install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a11/decode_low_crt.py': '"""Full-fold reverse in remainder/low-Taylor coordinates.\n\nThis changes only coordinate transforms and bilinear coefficient updates.\nAll measurements are performed by the supplied exact fused divider.\n"""\nfrom functools import lru_cache\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport sys\n\nHERE=Path(__file__).resolve().parent\nROOT=HERE.parents[2]\nsys.path[:0]=[str(ROOT/\'exp/r-main/narrow240\'),str(ROOT/\'exp/frontier/root\')]\nimport decode_combo as combo\nimport decode_fullfold\n\n@lru_cache(None)\ndef crt_matrix(k,c,kap):\n    """coeffs -> [remainder0,remainder1,low0,...,low(k-1),leader]."""\n    from exp.decoder.common import divmodp\n    n=k+3\n    rows=[[0]*n for _ in range(n)]\n    for j in range(n):\n        poly=[0]*j+[1]\n        _,rem=divmodp(poly,[kap,1,1])\n        rem=rem+[0]*(2-len(rem))\n        rows[0][j],rows[1][j]=rem[:2]\n    for j in range(k):rows[2+j][j]=1\n    rows[-1][-1]=1\n    return tuple(tuple(row) for row in rows)\n\n\ndef absorb_even_low_crt(b,head,A,B,Q,r,c,kap,reverse=False,transcript=None):\n    assert reverse\n    k=len(B)\n    out=head+B\n    value=old.eval_implicit(b,A,Q,c)\n    F=dv.zero_flag(b,value)\n    old.eval_implicit(b,A,Q,c,True)\n    linear_words(b,out+[r],crt_matrix(k,c,kap))\n    v=epow(c,k+2)\n    co=[(v&4095)^mul(c,v>>12),v>>12]\n    old.controlled_linear_xor(b,F,r,head,co)\n    value=old.eval_implicit(b,A,Q,c)\n    cx(b,head[1],head[0],c)\n    hm,fm=transcript\n    def after():\n        cx(b,head[1],head[0],c)\n        old.eval_implicit(b,A,Q,c,True)\n        b.cx(F,A[0][0])\n        low_product(b,head,A,B)\n        b.cx(F,A[0][0])\n        q_times_low(b,B,kap,True)\n        ar.mul_xor0(b,head[1],Q,r)\n        old.eval_implicit(b,A,Q,c)\n    divide_fused(b,combine(head),value,F,fm,after,K)\n    b.cz(combine(head),hm)\n    b.free(hm)\n    old.eval_implicit(b,A,Q,c,True)\n    return out,A,transcript\n\n\ndef install(entry,select=None):\n    synth=entry.__globals__[\'synthesize\']\n    folds=synth.__globals__[\'folds\']\n    original=folds.ug[\'absorb_even\']\n    patched=combo.xsub.clone(absorb_even_low_crt,dict(original.__globals__,crt_matrix=crt_matrix,\n        low_product=folds.ug[\'absorb\'].__globals__[\'low_product\']))\n    if select is not None:\n        def choose(b,head,A,B,Q,r,c,kap,reverse=False,transcript=None):\n            fn=patched if len(B) in select else original\n            return fn(b,head,A,B,Q,r,c,kap,reverse,transcript)\n        replacement=choose\n    else:replacement=patched\n    ug=dict(folds.ug,absorb_even=replacement)\n    proxy=SimpleNamespace(**dict(vars(folds),ug=ug))\n    new_synth=combo.xsub.clone(synth,dict(synth.__globals__,folds=proxy))\n    result=combo.xsub.clone(entry,dict(entry.__globals__,synthesize=new_synth))\n    assert result.__globals__[\'convert_safe\'] is entry.__globals__[\'convert_safe\']\n    assert result.__globals__[\'baseline\'] is entry.__globals__[\'baseline\']\n    return result\n\n\ndecode=install(decode_fullfold.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a11/decode_outer35.py': '"""Recover the last full product\'s known highest coefficient linearly.\n\nAt this boundary r_before=1+h*Q. Interpolation at35 points gives the full\nproduct modulo their monic polynomial. Its missing coefficient is h*Q,\nwhich is already present as r_before+1. The redundant leader is measured\nand its exact bilinear phase is paid before the head is consumed.\n"""\nfrom functools import lru_cache\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport sys\nHERE=Path(__file__).resolve().parent\nsys.path.insert(0,str(HERE))\nimport decode_virtual_leader as virtual\nfrom decode_low_crt import combo\n\n@lru_cache(None)\ndef points_polynomial():\n    from exp.decoder.common import pmul\n    p=[1]\n    for t in range(35):p=pmul(p,[t,1])\n    assert len(p)==36 and p[-1]==1\n    return tuple(p)\n\n\ndef install(entry):\n    synth=entry.__globals__[\'synthesize\'];folds=synth.__globals__[\'folds\'];sg=dict(folds.sg)\n    fn=sg[\'absorb_even\'];original=fn.__globals__[\'full_product\'];scope=fn.__globals__\n    base,old,ar,cx=scope[\'base\'],scope[\'old\'],scope[\'ar\'],scope[\'cx\']\n    def full_product(b,head,A,Q,target):\n        if len(target)!=36:return original(b,head,A,Q,target)\n        out=target[:-1];r=target[-1];V,W=base.vandermonde(35)\n        for t in range(35):\n            old.rank_affine(b,head,[1,t],A+[Q],V[t][:len(A)+1],out,[W[j][t]for j in range(35)])\n        for word,co in zip(out,points_polynomial()[:-1]):\n            if not co:continue\n            cx(b,r,word,co)\n            for j in ar.bits(co):b.x(word[j])\n        mask=b.alloc_bits(12);b.hmr(r,mask)\n        b.z(mask[0]);ar.phase_product(b,head[1],Q,mask);b.free(mask)\n        b.x(r[0])\n    sg[\'absorb_even\']=combo.xsub.clone(fn,dict(scope,full_product=full_product))\n    fs=SimpleNamespace(**dict(vars(folds),sg=sg))\n    sy=combo.xsub.clone(synth,dict(synth.__globals__,folds=fs))\n    result=combo.xsub.clone(entry,dict(entry.__globals__,synthesize=sy))\n    assert result.__globals__[\'convert_safe\'] is entry.__globals__[\'convert_safe\']\n    assert result.__globals__[\'baseline\'] is entry.__globals__[\'baseline\']\n    return result\n\n\ndecode=install(virtual.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a11/decode_outer_alias_constant.py': '"""Use the structurally constant outer implicit leader in its zero alias.\n\nAfter the last synthesis fold, border is identically 1 by the prefix\nrecurrence; before the first reverse fold it is reclaimed as that constant.\n"""\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport inspect,sys\nHERE=Path(__file__).resolve().parent\nsys.path[:0]=[str(HERE),str(HERE.parent/\'root\')]\nimport decode_root_v8\nfrom decode_low_crt import combo\n\n\ndef constant_alias(b,F,targets,coeffs):\n    for word,co in zip(targets,coeffs):\n        for j in range(12):\n            if co>>j&1:b.cx(F,word[j])\n\n\ndef install(entry):\n    synth=entry.__globals__[\'synthesize\'];folds=synth.__globals__[\'folds\']\n    sg=dict(folds.sg);ug=dict(folds.ug)\n    for table,key in((sg,\'absorb_even\'),(ug,\'absorb_even\')):\n        fn=table[key];source=inspect.getsource(fn)\n        if table is sg:\n            before=\'  old.controlled_linear_xor(b,F,r,full[-3:-1],[kap,1])\'\n            after=\'\'\'  if k==33 and c==0:constant_alias(b,F,full[-3:-1],[kap,1])\n  else:old.controlled_linear_xor(b,F,r,full[-3:-1],[kap,1])\'\'\'\n        else:\n            before=\'    old.controlled_linear_xor(b,F,r,head,co)\'\n            after=\'\'\'    if k==33 and c==0:constant_alias(b,F,head,co)\n    else:old.controlled_linear_xor(b,F,r,head,co)\'\'\'\n        assert source.count(before)==1,(key,source)\n        source=source.replace(before,after)\n        scope=dict(fn.__globals__,constant_alias=constant_alias)\n        exec(compile(source,__file__,\'exec\'),scope)\n        table[key]=scope[fn.__name__]\n    fs=SimpleNamespace(**dict(vars(folds),sg=sg,ug=ug))\n    sy=combo.xsub.clone(synth,dict(synth.__globals__,folds=fs))\n    result=combo.xsub.clone(entry,dict(entry.__globals__,synthesize=sy))\n    assert result.__globals__[\'convert_safe\'] is entry.__globals__[\'convert_safe\']\n    assert result.__globals__[\'baseline\'] is entry.__globals__[\'baseline\']\n    return result\n\n\ndecode=install(decode_root_v8.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a11/decode_root_v8.py': '"""A11 CRT reverse on root v8, preserving all previous exact front kernels."""\nfrom pathlib import Path\nimport sys\nHERE=Path(__file__).resolve().parent\nsys.path[:0]=[str(HERE),str(HERE.parent/\'root\')]\nimport decode_combo_v8\nfrom decode_low_crt import install\ndecode=install(decode_combo_v8.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a11/decode_virtual_leader.py': '"""Defer the known-one implicit leader during the first reverse fold.\n\nThe released twelve wires permit a retained twelve-bit inverse in that one\nfold. The leader is allocated only after divider scratch is completely gone.\n"""\nfrom functools import lru_cache\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport inspect,sys\nHERE=Path(__file__).resolve().parent\nsys.path[:0]=[str(HERE),str(HERE.parent/\'root\')]\nimport decode_outer_alias_constant as outer\nfrom decode_low_crt import combo,crt_matrix\n\n@lru_cache(None)\ndef virtual_matrix(k,c,kap):\n    return tuple(tuple(row[:-1])for row in crt_matrix(k,c,kap)[:-1])\n\n\ndef divide_retained_u(b,x,y,F,fm,after,K=None):\n    b.cx(F,y[0]);b.cx(y[12:],y[:12])\n    oldmask=multiply_forward99(b,x,y,K)\n    b.cx(y[12:],y[:12])\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,y,N,K)\n    u=inverse_initialized(b,N)\n    for off in(0,12):\n        half=x[off:off+12];out=b.alloc_qubits(12)\n        ar.mul_xor0(b,half,u,out)\n        ar.erase_product(b,out,N,half)\n        b.swap(half,out);b.free(out)\n    phase_product24(b,x,y,oldmask,K);b.free(oldmask)\n    b.cx(F,y[0]);b.cx(F,N[0])\n    after()\n    mask=b.alloc_bits(1);b.hmr(F,mask[0]);b.free(F)\n    b.cx(fm[0],mask[0]);b.free(fm)\n    b.z(mask[0]);pm=b.alloc_bits(12);b.bit_store(pm,False)\n    b.cx(mask[0],pm[0]);ar.phase_product(b,N,u,pm)\n    b.free(pm);b.free(mask)\n    um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    phase_inverse34(b,N,um,zero_mask=um[0]);b.free(um)\n    ar._erase_norm(b,y,N,K);b.free(N)\n\n\ndef absorb_virtual_one(b,head,A,B,Q,r,c,kap,reverse=False,transcript=None):\n    assert reverse and len(B)==33 and c==0\n    k=len(B);out=head+B\n    value=old.eval_implicit(b,A,Q,c);F=dv.zero_flag(b,value)\n    old.eval_implicit(b,A,Q,c,True)\n    linear_words(b,out,virtual_matrix(k,c,kap))\n    v=epow(c,k+2);co=[(v&4095)^mul(c,v>>12),v>>12]\n    for word,a in zip(head,co):\n        for j in ar.bits(a):b.x(word[j])\n    constant_alias(b,F,head,co)\n    value=old.eval_implicit(b,A,Q,c);cx(b,head[1],head[0],c)\n    hm,fm=transcript\n    def after():\n        cx(b,head[1],head[0],c);old.eval_implicit(b,A,Q,c,True)\n        b.cx(F,A[0][0]);low_product(b,head,A,B);b.cx(F,A[0][0])\n        q_times_low(b,B,kap,True)\n        old.eval_implicit(b,A,Q,c)\n    divide_retained_u(b,combine(head),value,F,fm,after,K)\n    b.cz(combine(head),hm);b.free(hm)\n    old.eval_implicit(b,A,Q,c,True)\n    reclaim(b,[r]);b.x(r[0]);ar.mul_xor0(b,head[1],Q,r)\n    return out,A,transcript\n\n\ndef install(entry):\n    synth=entry.__globals__[\'synthesize\'];folds=synth.__globals__[\'folds\'];ug=dict(folds.ug)\n    original=ug[\'absorb_even\'];div=original.__globals__[\'divide_fused\']\n    divscope=dict(div.__globals__,inverse_initialized=synth.__globals__[\'inverse_initialized\'],\n                  phase_inverse34=combo.xsub.phase_inverse12_virtual)\n    newdiv=combo.xsub.clone(divide_retained_u,divscope)\n    scope=dict(original.__globals__,virtual_matrix=virtual_matrix,constant_alias=outer.constant_alias,\n               divide_retained_u=newdiv)\n    special=combo.xsub.clone(absorb_virtual_one,scope)\n    def choose(b,head,A,B,Q,r,c,kap,reverse=False,transcript=None):\n        fn=special if len(B)==33 and c==0 else original\n        return fn(b,head,A,B,Q,r,c,kap,reverse,transcript)\n    ug[\'absorb_even\']=choose\n    fs=SimpleNamespace(**dict(vars(folds),ug=ug))\n    # Build from the supplied canonical driver because the current driver\n    # was compiled dynamically by the full-fold source transform.\n    source=inspect.getsource(combo.decode.__globals__[\'synthesize\'])\n    old=\'\'\'        if i%2:\n            ar.mul_xor0(b,heads[i][1],r,border);qi,p=r,border\n            A,B,t=folds.sg[\'absorb\'](b,heads[i],A,B,qi,p,None,0,cs[i],ks[i],wide=False)\n        else:\n            qi,p=border,r\n            A,B,t=folds.sg[\'absorb_even\'](b,heads[i],A,B,qi,r,cs[i],ks[i])\'\'\'\n    new=\'\'\'        qi,p=(r,border) if i%2 else (border,r)\n        A,B,t=folds.sg[\'absorb_even\'](b,heads[i],A,B,qi,p,cs[i],ks[i])\'\'\'\n    assert old in source;source=source.replace(old,new)\n    old=\'\'\'            if i%2:\n                folds.ug[\'absorb\'](b,heads[i],aa,bb,qi,p,None,0,cs[i],ks[i],True,t,False)\n                ar.mul_xor0(b,heads[i][1],r,border)\n            else:folds.ug[\'absorb_even\'](b,heads[i],aa,bb,qi,r,cs[i],ks[i],True,t)\'\'\'\n    new="            folds.ug[\'absorb_even\'](b,heads[i],aa,bb,qi,p,cs[i],ks[i],True,t)"\n    assert old in source;source=source.replace(old,new)\n    edits=(\n        (\'b.cx(border,r);b.cx(border,d);b.cx(r,border);b.x(border[0])\',\'b.cx(border,r);b.cx(r,border);b.x(border[0])\'),\n        (\'folds.add_tail(b,heads[i],qi,d,bool(i%2))\',\'folds.add_tail(b,heads[i],qi,d,not bool(i%2))\'),\n        (\'ar.mul_xor0(b,r,border,d);tt=tail_inplace\',\'tt=tail_inplace\'),\n        (\'tail_inplace(b,r,border,d,u,v,pole,True,tt);ar.mul_xor0(b,r,border,d)\',\'tail_inplace(b,r,border,d,u,v,pole,True,tt)\'),\n        (\'b.cx(r,d);b.x(border[0]);b.cx(r,border);b.cx(border,r);b.free(r)\',\'b.x(border[0]);b.cx(r,border);b.cx(border,r);b.free(r)\'),\n        ("        folds.ug[\'reclaim\'](b,[border]);b.x(border[0])\\n",\'\'),\n    )\n    for before,after in edits:\n        assert before in source;source=source.replace(before,after)\n    env=dict(synth.__globals__,folds=fs)\n    exec(compile(source,__file__,\'exec\'),env)\n    result=combo.xsub.clone(entry,dict(entry.__globals__,synthesize=env[\'synthesize\']))\n    assert result.__globals__[\'convert_safe\'] is entry.__globals__[\'convert_safe\']\n    assert result.__globals__[\'baseline\'] is entry.__globals__[\'baseline\']\n    return result\n\n\ndecode=install(outer.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a12/decode_companion_preserve.py': '"""Preserve the input decoder\'s actual scalar-stream implementation."""\nfrom pathlib import Path\nimport sys\nHERE=Path(__file__).resolve().parent\nsys.path[:0]=[str(HERE),str(HERE.parent/\'root\')]\nimport decode_trailing_companion as companion\nimport decode_trailing_normcache as cached\n\n\ndef install(entry,with_norm=True,ceiling=889):\n    original=entry.__globals__[\'extraction\']\n    ref=original.__globals__[\'ref\']\n    current_stream=ref.external_normalize_paired.__globals__[\'inverse_stream_noscratch\']\n    result=cached.install(entry,ceiling) if with_norm else companion.install(entry)\n    extraction=result.__globals__[\'extraction\']\n    scope=dict(extraction.__globals__)\n    scope[\'inverse_stream_noscratch\']=current_stream\n    character=scope[\'linear_inverse_character\']\n    scope[\'linear_inverse_character\']=companion.combo.xsub.clone(character,\n        dict(character.__globals__,inverse_stream_noscratch=current_stream))\n    if with_norm:\n        replacements=dict(companion.combo.synth.replacements)\n        replacements[id(cached.ns.inverse_stream_noscratch)]=current_stream\n        inverse=companion.combo.xsub.private_rebind(cached.inverse12_keep_norm,replacements)\n        def consume(b,x):return inverse(b,x,ceiling)\n        scope[\'inverse12_keep_norm\']=consume\n    extraction=companion.combo.xsub.clone(extraction,scope)\n    return companion.combo.xsub.clone(result,dict(result.__globals__,extraction=extraction))\n\n\ndecode=install(companion.combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a12/decode_trailing_companion.py': '"""Retain the trailing scalar ratio and remove its forward phase replay.\n\nThe denominator is inverted in place. All normalization masks are combined\ninto one inverse character before either original input word is retired.\n"""\nfrom pathlib import Path\nimport sys\nHERE = Path(__file__).resolve().parent\nROOT = HERE.parents[2]\nsys.path[:0] = [str(ROOT/\'exp/r-main/narrow240\')]\nimport decode_combo as combo\nimport retained_core_safe27 as retained\nfrom exp.noscratch.schedules import inverse_stream_noscratch\n\n\ndef transpose_square_mask(b, mask):\n    columns = [ar.field(12).square(1 << j) for j in range(12)]\n    out = b.alloc_bits(12)\n    b.bit_store(out, False)\n    for j, column in enumerate(columns):\n        for i in ar.bits(column):\n            b.cx(mask[i], out[j])\n    return out\n\n\ndef scale_defer(b, x, factor):\n    out = b.alloc_qubits(12)\n    ar.mul_xor0(b, x, factor, out)\n    mask = b.alloc_bits(12)\n    b.hmr(x, mask)\n    b.swap(x, out)\n    b.free(out)\n    return mask\n\n\ndef extraction_companion(b, words, reverse=False, token=None):\n    if not reverse:\n        return forward_original(b, words, False, token)\n    heads = [words[2*i:2*i+2] for i in range(34)]\n    N, L = words[-2:]\n    pole = token[\'pole\']\n    controlled_inverse(b, L, pole, True, token[\'chart_inverse\'])\n    ratio = b.alloc_qubits(12)\n    b.cx(N, ratio)\n    D = b.alloc_qubits(12)\n    b.x(D[0])\n    b.cx(pole, N[0])\n    b.cx(pole, D[0])\n    for head, kap, c in reversed(list(zip(heads, ks, corrections))[1:]):\n        step(b, head, N, D, L, kap, c, True)\n    scale(b, N, inv(ks[0]))\n    b.swap(N, D)\n    scale(b, L, inv(ks[0]))\n    # D=1/C, N=(N0+a0)/C, L=(L0+h0)/C², ratio=selected Nf/C.\n    im = inverse_stream_noscratch(b, D)\n    nm = scale_defer(b, N, D)\n    rm = scale_defer(b, ratio, D)\n    square12(b, D)\n    lm = scale_defer(b, L, D)\n    square12(b, D, True)\n    constant = token[\'normalization\']\n    b.cx(im, constant)\n    b.free(im)\n    square12(b, L, True)\n    sm = transpose_square_mask(b, lm)\n    linear_inverse_character(b, D, constant, [(N, nm), (ratio, rm), (L, sm)])\n    square12(b, L)\n    for mask in (constant, nm, rm, lm, sm):\n        b.free(mask)\n    b.cz(D, token[\'gauge_mask\'])\n    b.free(token[\'gauge_mask\'])\n    b.cx(heads[0][0], N)\n    b.cx(heads[0][1], L)\n    original_mask = b.alloc_bits(12)\n    b.hmr(N, original_mask)\n    b.swap(N, ratio)\n    b.free(ratio)\n    b.cswap(pole, N, D)\n    # Deliberately retain the supplied trailing pole toggle, not the user\'s\n    # excluded newer measured-pole improvement.\n    guarded.zero_toggle(b, D, pole)\n    for i in reversed(range(1, len(heads))):\n        if i == 1:\n            scalar_erase_second12(b, heads[i], N, D, ks[i], ks[0])\n        else:\n            scalar_step(b, heads[i], N, D, ks[i], True)\n    scalar_step(b, heads[0], N, D, ks[0], True, first=True)\n    b.x(D[0])\n    b.free(D)\n    b.free(pole)\n    b.cz(N, original_mask)\n    b.free(original_mask)\n    external_extraction(b, words, True, token[\'external\'])\n\n\ndef install(entry):\n    original = entry.__globals__[\'extraction\']\n    scope = dict(original.__globals__)\n    ref = scope[\'ref\']\n    for name in (\'controlled_inverse\', \'ks\', \'corrections\', \'step\',\n                 \'scalar_step\', \'guarded\', \'square12\'):\n        scope[name] = getattr(ref, name)\n    scope[\'scalar_erase_second12\'] = retained.__dict__.get(\'scalar_erase_second12\')\n    from replay_erase import scalar_erase_second12\n    scope[\'scalar_erase_second12\'] = scalar_erase_second12\n    scope.update(forward_original=original, inverse_stream_noscratch=inverse_stream_noscratch,\n                 linear_inverse_character=retained.linear_inverse_character)\n    scope[\'transpose_square_mask\'] = combo.xsub.clone(transpose_square_mask, scope)\n    scope[\'scale_defer\'] = combo.xsub.clone(scale_defer, scope)\n    candidate = combo.xsub.clone(extraction_companion, scope)\n    candidate = combo.xsub.private_rebind(candidate, combo.synth.replacements)\n    result = combo.xsub.clone(entry, dict(entry.__globals__, extraction=candidate))\n    assert result.__globals__[\'convert_safe\'] is entry.__globals__[\'convert_safe\']\n    assert result.__globals__[\'baseline\'] is entry.__globals__[\'baseline\']\n    return result\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a12/decode_trailing_normcache.py': '"""Carry three norm coordinates from the consumed denominator to its phase."""\nfrom pathlib import Path\nimport inspect\nimport sys\nfrom types import SimpleNamespace\nHERE=Path(__file__).resolve().parent\nsys.path.insert(0,str(HERE))\nimport decode_trailing_companion as companion\nfrom exp.noscratch import schedules as ns\nfrom exp.arithmetic.lowwidth import classical_transpose\nimport norm_cache\n\n\ndef inverse12_keep_norm(b,x,ceiling=889):\n    to,frm,K=ns.tower(12)\n    ns.linear_inplace(b,x,to)\n    N=b.alloc_qubits(6)\n    ns.norm_xor_noscratch(b,x,N,K)\n    nm=ns.inverse_stream_noscratch(b,N)\n    ns.ar._phase_norm(b,x,nm,K)\n    b.free(nm)\n    out=b.alloc_qubits(6)\n    ns.mul_xor_noscratch(b,x[6:],N,out)\n    raw=b.alloc_bits(12)\n    b.hmr(x[6:],raw[6:])\n    ns.mul_xor_noscratch(b,x[:6],N,x[6:])\n    b.cx(out,x[6:])\n    b.hmr(x[:6],raw[:6])\n    b.swap(x[:6],x[6:])\n    b.swap(x[6:],out)\n    b.free(out)\n    keep=min(6,max(0,ceiling-(b.num_allocated_qubits-6)-12))\n    columns,back=norm_cache.layout(keep)\n    ns.linear_inplace(b,N,columns)\n    mask=b.alloc_bits(6)\n    b.bit_store(mask,False)\n    b.hmr(N[keep:],mask[keep:])\n    b.free(N[keep:])\n    original=classical_transpose(b,mask,columns)\n    b.free(mask)\n    ns.ar._phase_norm(b,x,original,K)\n    b.free(original)\n    ns.linear_inplace(b,x,frm)\n    result=classical_transpose(b,raw,to)\n    b.free(raw)\n    return result,N[:keep]\n\n\ndef install(entry,ceiling=889):\n    entry=companion.install(entry)\n    extraction=entry.__globals__[\'extraction\']\n    scope=dict(extraction.__globals__)\n    inverse=companion.combo.xsub.private_rebind(inverse12_keep_norm,\n                                               companion.combo.synth.replacements)\n    def consume(b,x):return inverse(b,x,ceiling)\n    scope[\'inverse12_keep_norm\']=consume\n    original_character=scope[\'linear_inverse_character\']\n    source=inspect.getsource(companion.retained.linear_inverse_character)\n    source=source.replace(\'def linear_inverse_character(b,g,constant,terms):\',\n                          \'def linear_inverse_character(b,g,constant,terms,saved_norm):\')\n    before=\'D=b.alloc_qubits(6);core.norm_xor_noscratch(b,g,D,kval)\'\n    assert before in source\n    source=source.replace(before,\'D=core.norm_cache.rebuild(b,g,saved_norm,kval)\')\n    character_scope=dict(original_character.__globals__)\n    def keep_for_ceiling(b,D):\n        count=min(6,max(0,ceiling-b.num_allocated_qubits-6))\n        columns,back=norm_cache.layout(count)\n        saved=b.alloc_qubits(count)\n        ns.ar.linear_xor(b,D,saved,[v&((1<<count)-1)for v in columns])\n        return saved\n    original_core=character_scope[\'core\']\n    cache=SimpleNamespace(**dict(vars(original_core.norm_cache),keep=keep_for_ceiling))\n    character_scope[\'core\']=SimpleNamespace(**dict(vars(original_core),norm_cache=cache))\n    exec(compile(source,__file__,\'exec\'),character_scope)\n    scope[\'linear_inverse_character\']=character_scope[\'linear_inverse_character\']\n    source=inspect.getsource(companion.extraction_companion)\n    source=source.replace(\'im = inverse_stream_noscratch(b, D)\',\n                          \'im, saved_norm = inverse12_keep_norm(b, D)\')\n    source=source.replace(\'[(N, nm), (ratio, rm), (L, sm)])\',\n                          \'[(N, nm), (ratio, rm), (L, sm)], saved_norm)\')\n    exec(compile(source,__file__,\'exec\'),scope)\n    return companion.combo.xsub.clone(entry,dict(entry.__globals__,extraction=scope[\'extraction_companion\']))\n\n\ndecode=install(companion.combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a14/decode_inverse20_12.py': '"""All-input consuming GF64 inverse with twelve total live wires."""\nfrom pathlib import Path\nimport sys,json\nHERE=Path(__file__).resolve().parent\nROOT=HERE.parents[2]\nsys.path[:0]=[str(HERE.parent/\'a08\'),str(HERE.parent/\'a04\')]\nfrom inverse19_consume16 import ar,ref,basic,combo,schedules,change_basis,retire\nfrom decode_unitary3 import inverse3_inplace\nfrom nonzero_consume16 import functions\nPLAN=json.loads((HERE/\'new_consuming12_plan.json\').read_text())\n\n\ndef solve_columns(value,columns):\n    basis={}\n    for i,column in enumerate(columns):\n        coefficient=1<<i\n        while column:\n            pivot=column.bit_length()-1\n            if pivot not in basis:\n                basis[pivot]=(column,coefficient)\n                break\n            old,tag=basis[pivot]\n            column^=old\n            coefficient^=tag\n    answer=0\n    while value:\n        pivot=value.bit_length()-1\n        if pivot not in basis:return None\n        old,tag=basis[pivot]\n        value^=old\n        answer^=tag\n    return answer\n\n\ndef retained_prefix(b,x,rows):\n    """Consume the norm, retaining a possibly intersecting raw subspace."""\n    to,back,K=ar.tower(6)\n    ar.linear_inplace(b,x,to)\n    raw=b.alloc_qubits(6)\n    forms=basic.raw_forms19()\n    for j,(left,right) in enumerate(forms):\n        with ref.expose_pair(b,x,left,right) as(a,c):b.init_and(a,c,raw[j])\n    norm_forms=ref.norm_forms(K)\n    norm_rows=[(value>>7)&63 for value in norm_forms]\n    active=norm_rows[:]\n    for row in rows:\n        if solve_columns(row,active) is None:active.append(row)\n    full=active[:]\n    for i in range(6):\n        if solve_columns(1<<i,full) is None:full.append(1<<i)\n    columns=[sum(((row>>j)&1)<<i for i,row in enumerate(full)) for j in range(6)]\n    ar.linear_inplace(b,raw,columns)\n    for j,form in enumerate(norm_forms):ref.form_xor(b,x,form&127,raw[j])\n    for q,row in zip(raw[len(active):],full[len(active):]):\n        mask=b.alloc_bits(1)[0]\n        b.hmr(q,mask)\n        for j in ar.bits(row):\n            with ref.expose_pair(b,x,*forms[j]) as(a,c):b.ccz(a,c,mask)\n        b.free(mask)\n    transformed=[1,2,4]\n    zero_positions=list(range(len(active),6))\n    coefficients=[]\n    for row in rows:\n        coefficient=solve_columns(row,active)\n        coefficients.append(coefficient)\n        physical=coefficient\n        if solve_columns(physical,transformed) is not None:\n            physical^=1<<zero_positions.pop()\n        transformed.append(physical)\n    columns=[sum(((row>>j)&1)<<i for i,row in enumerate(transformed)) for j in range(6)]\n    ar.linear_inplace(b,raw,columns)\n    for q,coefficient in zip(raw[3:],coefficients):\n        affine=0\n        for j in ar.bits(coefficient&7):affine^=norm_forms[j]&127\n        ref.form_xor(b,x,affine,q)\n    inverse3_inplace(b,raw[:3])\n    return raw\n\n\ndef inverse6_consume20_12(b,x):\n    raw=retained_prefix(b,x,PLAN[\'raw_rows\'])\n    pool=list(b.alloc_qubits(PLAN[\'max_live\']-12))\n    all_wires=list(x)+list(raw)+pool[:]\n    live=list(x)+list(raw)\n    transcript=b.alloc_bits(6)\n    b.bit_store(transcript,False)\n    retire(b,live,pool,transcript,PLAN[\'begin\'])\n    for step in PLAN[\'steps\']:\n        if step[\'kind\']==\'new\':\n            target=pool.pop()\n            with ref.expose_pair(b,live,step[\'left\'],step[\'right\']) as(a,c):\n                b.init_and(a,c,target)\n            live.append(target)\n        else:\n            change_basis(b,live,step[\'pre_rows\'])\n            with ref.expose_pair(b,live,step[\'left\'],step[\'right\']) as(a,c):\n                b.ccx(a,c,live[step[\'target\']])\n        retire(b,live,pool,transcript,step)\n    retire(b,live,pool,transcript,PLAN[\'final\'])\n    assert len(live)==6\n    for i,target in enumerate(x):\n        source=live[i]\n        if str(source)==str(target):continue\n        other=next((j for j,q in enumerate(live) if str(q)==str(target)),None)\n        b.swap(source,target)\n        live[i]=target\n        if other is not None:live[other]=source\n    kept=set(map(str,x))\n    b.free([q for q in all_wires if str(q)not in kept])\n    return transcript\n\n\ndef install(entry,ceiling=889):\n    def wrap(original):\n        def chosen(b,x):\n            if len(x)==6 and ceiling-b.num_allocated_qubits in (6,7):\n                return inverse6_consume20_12(b,x)\n            return original(b,x)\n        return chosen\n    replacements={id(fn):wrap(fn)for fn in functions(entry)\n                  if fn.__code__ is schedules.inverse_stream_noscratch.__code__}\n    assert replacements\n    scope=dict(entry.__globals__)\n    for name in(\'extraction\',\'synthesize\'):\n        scope[name]=combo.xsub.private_rebind(scope[name],replacements)\n    result=combo.xsub.clone(entry,scope)\n    for name in(\'convert_safe\',\'baseline\'):\n        assert result.__globals__[name] is entry.__globals__[name]\n    return result\n\n\ndecode=install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a14/decode_retained19.py': '"""Strictly nonzero 19-product scalar producer retaining two source coordinates."""\nfrom pathlib import Path\nimport sys,json\nHERE=Path(__file__).resolve().parent\nsys.path[:0]=[str(HERE),str(HERE.parent/\'root\')]\nfrom decode_inverse20_12 import (ar,ref,combo,change_basis,retire,retained_prefix,functions)\nPLAN=json.loads((HERE/\'new19_retained2_b14_plan.json\').read_text())\nassert PLAN[\'found\'] and PLAN[\'retained\']==2 and PLAN[\'domain\']==\'nonzero\'\n\n\ndef inverse6_retained19(b, x):\n    plan = PLAN\n    """Return an original-input phase mask and k linear source coordinates."""\n    assert plan[\'domain\'] == \'nonzero\'\n    raw = retained_prefix(b, x, plan[\'raw_rows\'])\n    pool = list(b.alloc_qubits(plan[\'max_live\']-12))\n    all_wires = list(x) + list(raw) + pool[:]\n    live = list(x) + list(raw)\n    transcript = b.alloc_bits(6)\n    b.bit_store(transcript, False)\n    retire(b, live, pool, transcript, plan[\'begin\'])\n    for step in plan[\'steps\']:\n        if step[\'kind\'] == \'new\':\n            target = pool.pop()\n            with ref.expose_pair(b, live, step[\'left\'], step[\'right\']) as (a, c):\n                b.init_and(a, c, target)\n            live.append(target)\n        else:\n            change_basis(b, live, step[\'pre_rows\'])\n            with ref.expose_pair(b, live, step[\'left\'], step[\'right\']) as (a, c):\n                b.ccx(a, c, live[step[\'target\']])\n        retire(b, live, pool, transcript, step)\n    retire(b, live, pool, transcript, plan[\'final\'])\n    assert len(live) == 6 + plan[\'retained\']\n    for i, target in enumerate(x):\n        source = live[i]\n        if str(source) == str(target):\n            continue\n        other = next((j for j, q in enumerate(live) if str(q) == str(target)), None)\n        b.swap(source, target)\n        live[i] = target\n        if other is not None:\n            live[other] = source\n    cache = live[6:]\n    held = set(map(str, list(x)+cache))\n    b.free([q for q in all_wires if str(q) not in held])\n    return transcript, cache\n\n\ndef install(entry):\n    # This exact closure exists only in root\'s clipped, nonzero fold divider.\n    targets=[fn for fn in functions(entry)\n             if fn.__name__==\'produce_retained\'\n             and Path(fn.__code__.co_filename).name==\'decode_inverse_retained.py\']\n    assert targets\n    replacements={id(fn):inverse6_retained19 for fn in targets}\n    scope=dict(entry.__globals__)\n    scope[\'synthesize\']=combo.xsub.private_rebind(scope[\'synthesize\'],replacements)\n    result=combo.xsub.clone(entry,scope)\n    for name in (\'extraction\',\'convert_safe\',\'baseline\'):\n        assert result.__globals__[name] is entry.__globals__[name]\n    return result\n\n\nimport decode_root_v11_six\ndecode=install(decode_root_v11_six.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a14/decode_root_v11_six.py': '"""Conservative six-fresh scalar hook on the parent\'s v11 decoder."""\nfrom pathlib import Path\nimport sys\nHERE=Path(__file__).resolve().parent\nsys.path[:0]=[str(HERE),str(HERE.parent/\'root\')]\nimport decode_combo_v11\nfrom decode_inverse20_12 import install\ndecode=install(decode_combo_v11.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/decode_pair.py': '"""Use a jointly synthesized inverse-character pair at four-coordinate norm sites."""\nfrom pathlib import Path\nimport sys\nHERE=Path(__file__).resolve().parent\nsys.path[:0]=[str(HERE),str(HERE.parent/\'root\'),str(HERE.parent/\'a04\')]\nimport decode_local_norm_partial as partial\nfrom nonzero_consume16 import functions\nfrom phase_pair import phase_pair,MASKS,xs\n\n\ndef _layout():\n    rows=list(MASKS)\n    def rank(values):\n        bs={}\n        for v in values:\n            while v:\n                p=v.bit_length()-1\n                if p in bs:v^=bs[p]\n                else:bs[p]=v;break\n        return len(bs)\n    for i in range(6):\n        if rank(rows+[1<<i])>len(rows):rows.append(1<<i)\n    rows=rows[2:]+rows[:2]\n    assert len(rows)==6\n    columns=[sum(((r>>i)&1)<<j for j,r in enumerate(rows)) for i in range(6)]\n    return tuple(rows),tuple(columns),tuple(xs.inverse_columns(columns))\n\n\nROWS,COLUMNS,INVERSE=_layout()\n\n\ndef transpose_mask(b,m):\n    out=b.alloc_bits(6)\n    b.bit_store(out,False)\n    for j,row in enumerate(ROWS):\n        for i in bits(row):b.cx(m[j],out[i])\n    return out\n\n\ndef inverse12_keep_partial_norm(b,x,keep):\n    if keep!=4:return original_produce(b,x,keep)\n    to,frm,K=tower(12)\n    linear_inplace(b,x,to)\n    D=b.alloc_qubits(6)\n    norm_xor_noscratch(b,x,D,K)\n    u=inverse6_init19(b,D)\n    linear_inplace(b,D,COLUMNS)\n    dm=b.alloc_bits(6);b.bit_store(dm,False)\n    b.hmr(D[keep:],dm[keep:]);b.free(D[keep:])\n    original_mask=transpose_mask(b,dm)\n    ar._phase_norm(b,x,original_mask,K)\n    b.free(original_mask);b.free(dm)\n    t=b.alloc_qubits(12)\n    b.cx(x[6:],x[:6])\n    mul_xor_noscratch(b,x[:6],u,t[:6])\n    mul_xor_noscratch(b,x[6:],u,t[6:])\n    b.cx(x[6:],x[:6])\n    linear_inplace(b,x,frm);linear_inplace(b,t,frm)\n    return t,u,D[:keep]\n\n\ndef erase_norm_with_partial(b,t,N,u,D):\n    if len(D)!=4:return original_erase(b,t,N,u,D)\n    to,frm,K=tower(12)\n    linear_inplace(b,t,to);linear_inplace(b,N,to)\n    m=b.alloc_bits(12);b.hmr(N,m);b.free(N)\n    L=b.alloc_qubits(2)\n    f=field(6)\n    def coefficient(j,target):\n        k=INVERSE[j]\n        for i in range(6):\n            for r in bits(f.mul(1<<i,k)):\n                b.ccx(m[r],t[i],target)\n                b.ccx(m[r],t[i+6],target)\n                b.ccx(m[r+6],t[i+6],target)\n    for j in range(4):\n        coefficient(j,L[0]);b.cz(L[0],D[j]);coefficient(j,L[0])\n    coefficient(4,L[0]);coefficient(5,L[1])\n    phase_pair(b,u,L)\n    coefficient(5,L[1]);coefficient(4,L[0])\n    b.free(L);b.free(m)\n    dm=b.alloc_bits(6);b.bit_store(dm,False)\n    b.hmr(D,dm[:4]);b.free(D)\n    original_mask=transpose_mask(b,dm)\n    xs.phase_inverse6_quad4(b,u,original_mask)\n    b.free(original_mask);b.free(dm)\n    ar._erase_norm(b,t,u,K);b.free(u)\n    linear_inplace(b,t,frm)\n\n\ndef install(entry):\n    replacements={}\n    for fn in functions(entry):\n        if fn.__code__ is partial.inverse12_keep_partial_norm.__code__:\n            extra=dict(original_produce=fn,COLUMNS=COLUMNS,transpose_mask=transpose_mask)\n            replacements[id(fn)]=xs.clone(inverse12_keep_partial_norm,dict(fn.__globals__,**extra))\n        elif fn.__code__ is partial.erase_norm_with_partial.__code__:\n            extra=dict(original_erase=fn,INVERSE=INVERSE,phase_pair=phase_pair,transpose_mask=transpose_mask)\n            replacements[id(fn)]=xs.clone(erase_norm_with_partial,dict(fn.__globals__,**extra))\n    assert len(replacements)>=2\n    scope=dict(entry.__globals__)\n    for name in (\'extraction\',\'synthesize\'):\n        scope[name]=xs.private_rebind(scope[name],replacements)\n    result=xs.clone(entry,scope)\n    for name in (\'convert_safe\',\'baseline\'):\n        assert result.__globals__[name] is entry.__globals__[name]\n    return result\n\n\n# This helper is called from both privately rebound runtime functions.\nbits=xs.bits\nimport decode_combo_v11 as current\ndecode=install(current.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/decode_partial_many.py': '"""Use two through five retained norm coordinates, with all inverse phases fused."""\nfrom pathlib import Path\nimport sys\nHERE=Path(__file__).resolve().parent\nsys.path.insert(0,str(HERE))\nimport decode_pair as pair\nfrom phase_many_fused import phase_many_fused\n\n\ndef inverse12_keep_partial_norm(b,x,keep):\n    if keep==6:return original_produce(b,x,keep)\n    to,frm,K=tower(12)\n    linear_inplace(b,x,to)\n    D=b.alloc_qubits(6);norm_xor_noscratch(b,x,D,K)\n    u=inverse6_init19(b,D)\n    linear_inplace(b,D,COLUMNS)\n    dm=b.alloc_bits(6);b.bit_store(dm,False)\n    b.hmr(D[keep:],dm[keep:]);b.free(D[keep:])\n    original_mask=transpose_mask(b,dm)\n    ar._phase_norm(b,x,original_mask,K)\n    b.free(original_mask);b.free(dm)\n    t=b.alloc_qubits(12)\n    b.cx(x[6:],x[:6])\n    mul_xor_noscratch(b,x[:6],u,t[:6])\n    mul_xor_noscratch(b,x[6:],u,t[6:])\n    b.cx(x[6:],x[:6])\n    linear_inplace(b,x,frm);linear_inplace(b,t,frm)\n    return t,u,D[:keep]\n\n\ndef erase_norm_with_partial(b,t,N,u,D):\n    keep=len(D)\n    if keep==6:return original_erase(b,t,N,u,D)\n    to,frm,K=tower(12)\n    linear_inplace(b,t,to);linear_inplace(b,N,to)\n    m=b.alloc_bits(12);b.hmr(N,m);b.free(N)\n    L=b.alloc_qubits(6-keep)\n    f=field(6)\n    def coefficient(j,target):\n        k=INVERSE[j]\n        for i in range(6):\n            for r in bits(f.mul(1<<i,k)):\n                b.ccx(m[r],t[i],target)\n                b.ccx(m[r],t[i+6],target)\n                b.ccx(m[r+6],t[i+6],target)\n    for j in range(keep):\n        coefficient(j,L[0]);b.cz(L[0],D[j]);coefficient(j,L[0])\n    dm=b.alloc_bits(6);b.bit_store(dm,False)\n    b.hmr(D,dm[:keep]);b.free(D)\n    original_mask=transpose_mask(b,dm)\n    for j in range(keep,6):coefficient(j,L[j-keep])\n    phase_many_fused(b,u,{ROWS[j]:L[j-keep] for j in range(keep,6)},original_mask)\n    for j in reversed(range(keep,6)):coefficient(j,L[j-keep])\n    b.free(L);b.free(m);b.free(original_mask);b.free(dm)\n    ar._erase_norm(b,t,u,K);b.free(u)\n    linear_inplace(b,t,frm)\n\n\ndef inverse24_reverse_partial(b,x,mask,K=None):\n    keep=min(6,889-b.num_allocated_qubits-42)\n    if keep<2:return fallback(b,x,mask,K)\n    cols,back,_=extension_columns(K)\n    linear_inplace(b,x,cols)\n    to,frm,K24=tower(24);linear_inplace(b,x,to)\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,x,N,K24)\n    t,u,D=inverse12_keep_partial_norm(b,N,keep)\n    out=b.alloc_qubits(12)\n    mul_xor_noscratch(b,x[12:],t,out)\n    erase_product(b,out,N,x[12:])\n    mul_xor_noscratch(b,x[:12],t,x[12:])\n    erase_product(b,x[12:],N,x[:12])\n    b.cx(out,x[12:]);b.swap(x[:12],x[12:]);b.swap(x[12:],out);b.free(out)\n    erase_norm_with_partial(b,t,N,u,D)\n    ar._erase_norm(b,x,t,K24);b.free(t)\n    linear_inplace(b,x,frm);b.cz(x,mask);b.free(mask);linear_inplace(b,x,back)\n\n\ndef install(entry):\n    replacements={}\n    for fn in pair.functions(entry.__globals__[\'extraction\']):\n        if fn.__code__ is pair.partial.inverse24_reverse_partial.__code__:\n            scope=dict(fn.__globals__)\n            produce=scope[\'inverse12_keep_partial_norm\'];erase=scope[\'erase_norm_with_partial\']\n            pscope=dict(produce.__globals__,original_produce=produce,COLUMNS=pair.COLUMNS,transpose_mask=pair.transpose_mask)\n            escope=dict(erase.__globals__,original_erase=erase,ROWS=pair.ROWS,INVERSE=pair.INVERSE,transpose_mask=pair.transpose_mask,phase_many_fused=phase_many_fused)\n            scope[\'inverse12_keep_partial_norm\']=pair.xs.clone(inverse12_keep_partial_norm,pscope)\n            scope[\'erase_norm_with_partial\']=pair.xs.clone(erase_norm_with_partial,escope)\n            replacements[id(fn)]=pair.xs.clone(inverse24_reverse_partial,scope)\n    assert replacements\n    scope=dict(entry.__globals__)\n    scope[\'extraction\']=pair.xs.private_rebind(scope[\'extraction\'],replacements)\n    result=pair.xs.clone(entry,scope)\n    for name in (\'convert_safe\',\'baseline\'):\n        assert result.__globals__[name] is entry.__globals__[name]\n    return result\n\n\ndecode=install(pair.current.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/phase_many_fused.py': '"""Several quantum inverse characters and arbitrary classical inverse character."""\nfrom pathlib import Path\nimport json,sys\nHERE=Path(__file__).resolve().parent\nsys.path.insert(0,str(HERE))\nfrom phase_pair_fused import FORMS,RECORD,xs\n\n\ndef decompose(terms):\n    rows=[0]*13;linear=0;constant=0\n    for term in terms:\n        if term[0]==\'constant\':constant^=1\n        elif term[0]==\'linear\':linear^=1<<term[1]\n        else:\n            i,j=term[1:];rows[i]^=1<<j;rows[j]^=1<<i\n    pairs=[]\n    while any(rows):\n        i=next(i for i,r in enumerate(rows) if r)\n        j=next(xs.bits(rows[i]))\n        left,right=rows[i],rows[j]\n        pairs.append((left,right));linear^=left&right\n        for k in range(13):\n            if left>>k&1:rows[k]^=right\n            if right>>k&1:rows[k]^=left\n    return constant,linear,tuple(pairs)\n\n\nCHARACTERS={int(k):decompose(v[\'terms\']) for k,v in json.loads((HERE/\'fused_characters.json\').read_text()).items()}\n\n\ndef phase_many_fused(b,x,coefficients,mask):\n    """coefficients maps constant field masks to quantum coefficients; inv0 is total."""\n    raw=b.alloc_qubits(7);source=list(x)+list(raw)\n    for i,(left,right) in enumerate(FORMS):\n        with xs.expose_pair(b,source[:6+i],left,right) as (a,c):\n            b.init_and(a,c,raw[i])\n    for char,coefficient in coefficients.items():\n        constant,linear,pairs=CHARACTERS[char]\n        if constant:b.z(coefficient)\n        for i in xs.bits(linear):b.cz(source[i],coefficient)\n        for left,right in pairs:\n            with xs.expose_pair(b,source,left<<1,right<<1) as (a,c):\n                b.ccz(a,c,coefficient)\n    for coefficient,terms in zip(mask,RECORD[\'classical\']):\n        for term in terms:\n            if term[0]==\'constant\':b.z(coefficient)\n            elif term[0]==\'linear\':b.cz(source[term[1]],coefficient)\n            else:b.ccz(source[term[1]],source[term[2]],coefficient)\n    for i in reversed(range(7)):\n        with xs.expose_pair(b,source[:6+i],*FORMS[i]) as (a,c):\n            b.del_and(a,c,raw[i])\n    b.free(raw)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/phase_pair.py': '"""Two exact GF(64) inverse characters sharing five nonlinear intermediates."""\nfrom pathlib import Path\nimport json, sys\nHERE=Path(__file__).resolve().parent\nROOT=HERE.parents[2]\nsys.path.insert(0,str(ROOT/\'exp/r-main/narrow240\'))\nimport exact_subs as xs\n\nRECORD=json.loads((HERE/\'best_pair.json\').read_text())\nMASKS=tuple(RECORD[\'masks\'])\nFORMS=tuple(xs.FORMS)+((RECORD[\'left\'],RECORD[\'right\']),)\n\n\ndef decompose(terms):\n    rows=[0]*11;linear=0;constant=0\n    for term in terms:\n        if term[0]==\'constant\':constant^=1\n        elif term[0]==\'linear\':linear^=1<<term[1]\n        else:\n            i,j=term[1:];rows[i]^=1<<j;rows[j]^=1<<i\n    pairs=[]\n    while any(rows):\n        i=next(i for i,r in enumerate(rows) if r)\n        j=next(xs.bits(rows[i]))\n        left,right=rows[i],rows[j]\n        pairs.append((left,right));linear^=left&right\n        for k in range(11):\n            if left>>k&1:rows[k]^=right\n            if right>>k&1:rows[k]^=left\n    return constant,linear,tuple(pairs)\n\n\nEXPRESSIONS=tuple(decompose(t) for t in RECORD[\'terms\'])\nassert [len(t[2]) for t in EXPRESSIONS]==[2,2]\n\n\ndef phase_pair(b,x,coefficients):\n    """Pay sum_i coefficients[i] * dot(MASKS[i],inv0(x)); preserve inputs."""\n    raw=b.alloc_qubits(5)\n    source=list(x)+list(raw)\n    for i,(left,right) in enumerate(FORMS):\n        with xs.expose_pair(b,source[:6+i],left,right) as (a,c):\n            b.init_and(a,c,raw[i])\n    for coefficient,(constant,linear,pairs) in zip(coefficients,EXPRESSIONS):\n        if constant:b.z(coefficient)\n        for i in xs.bits(linear):b.cz(source[i],coefficient)\n        for left,right in pairs:\n            with xs.expose_pair(b,source,left<<1,right<<1) as (a,c):\n                b.ccz(a,c,coefficient)\n    for i in reversed(range(5)):\n        with xs.expose_pair(b,source[:6+i],*FORMS[i]) as (a,c):\n            b.del_and(a,c,raw[i])\n    b.free(raw)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/phase_pair_fused.py': '"""Pair of quantum inverse characters plus any classical inverse character."""\nfrom pathlib import Path\nimport json,sys\nHERE=Path(__file__).resolve().parent\nROOT=HERE.parents[2]\nsys.path.insert(0,str(ROOT/\'exp/r-main/narrow240\'))\nimport exact_subs as xs\nRECORD=json.loads((HERE/\'fused_pair.json\').read_text())\nFORMS=tuple(xs.FORMS)+tuple(tuple(t) for t in RECORD[\'forms_extra\'])\n\n\ndef phase_pair_fused(b,x,coefficients,mask):\n    """Pay L0·inv(x)[1] + L1·inv(x)[56] + dot(mask,inv(x)), including x=0."""\n    raw=b.alloc_qubits(7);source=list(x)+list(raw)\n    for i,(left,right) in enumerate(FORMS):\n        with xs.expose_pair(b,source[:6+i],left,right) as (a,c):\n            b.init_and(a,c,raw[i])\n    for coefficient,expression in zip(coefficients,RECORD[\'quantum\']):\n        if expression[\'constant\']:b.z(coefficient)\n        for i in xs.bits(expression[\'linear\']):b.cz(source[i],coefficient)\n        with xs.expose_pair(b,source,*expression[\'pair\']) as (a,c):\n            b.ccz(a,c,coefficient)\n    for coefficient,terms in zip(mask,RECORD[\'classical\']):\n        for term in terms:\n            if term[0]==\'constant\':b.z(coefficient)\n            elif term[0]==\'linear\':b.cz(source[term[1]],coefficient)\n            else:b.ccz(source[term[1]],source[term[2]],coefficient)\n    for i in reversed(range(7)):\n        with xs.expose_pair(b,source[:6+i],*FORMS[i]) as (a,c):\n            b.del_and(a,c,raw[i])\n    b.free(raw)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a17/decode_initialized19.py': '"""All-input initialized GF64 inverse with ten fresh qubits."""\nfrom pathlib import Path\nimport json\nimport sys\n\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE.parent/\'a08\'), str(HERE.parent/\'a04\')]\nfrom inverse19_consume16 import ar, ref, basic, combo, change_basis, retire\nfrom nonzero_consume16 import functions\nfrom exp.inverse_search import inverse20_iso as iso\n\nPLAN = json.loads((HERE/\'source_preserving19_b16_result.json\').read_text())\nassert PLAN[\'found\'] and PLAN[\'domain\'] == \'all\' and PLAN[\'max_live\'] == 16\n\n\ndef inverse6_initialized19(b, x):\n    """Preserve x, return inverse0(x), and pay every internal HMR phase."""\n    to, back, K = ar.tower(6)\n    ar.linear_inplace(b, x, to)\n    raw = b.alloc_qubits(6)\n    for j, (left, right) in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b, x, left, right) as (a, c):\n            b.init_and(a, c, raw[j])\n    inner = ref.inverse3_virtual(b, list(x)+list(raw), ref.norm_forms(K))\n    pool = list(b.alloc_qubits(1))\n    all_wires = list(x)+list(raw)+list(inner)+pool[:]\n    live = list(x)+list(raw)+list(inner)\n    transcript = b.alloc_bits(6)\n    b.bit_store(transcript, False)\n    retire(b, live, pool, transcript, PLAN[\'begin\'])\n    for step in PLAN[\'steps\']:\n        if step[\'kind\'] == \'new\':\n            target = pool.pop()\n            with ref.expose_pair(b, live, step[\'left\'], step[\'right\']) as (a, c):\n                b.init_and(a, c, target)\n            live.append(target)\n        else:\n            change_basis(b, live, step[\'pre_rows\'])\n            with ref.expose_pair(b, live, step[\'left\'], step[\'right\']) as (a, c):\n                b.ccx(a, c, live[step[\'target\']])\n        retire(b, live, pool, transcript, step)\n    retire(b, live, pool, transcript, PLAN[\'final\'])\n    assert len(live) == 12\n    # The final basis is (canonical reciprocal, canonical original source).\n    for i, target in enumerate(x, 6):\n        source = live[i]\n        if str(source) == str(target):\n            continue\n        other = next((j for j, q in enumerate(live) if str(q) == str(target)), None)\n        b.swap(source, target)\n        live[i] = target\n        if other is not None:\n            live[other] = source\n    out = live[:6]\n    kept = set(map(str, list(x)+out))\n    b.free([q for q in all_wires if str(q) not in kept])\n    b.cz(x, transcript)\n    b.free(transcript)\n    return out\n\n\ndef install(entry, ceiling=889):\n    def wrap(original):\n        def chosen(b, x, z=None, keep_norm=False):\n            if z is None and not keep_norm and b.num_allocated_qubits+10 <= ceiling:\n                return inverse6_initialized19(b, x)\n            return original(b, x, z, keep_norm)\n        return chosen\n    replacements = {id(fn):wrap(fn) for fn in functions(entry)\n                    if fn.__code__ is iso.inverse6_xor20.__code__}\n    assert replacements\n    scope = dict(entry.__globals__)\n    for name in (\'extraction\', \'synthesize\'):\n        scope[name] = combo.xsub.private_rebind(scope[name], replacements)\n    result = combo.xsub.clone(entry, scope)\n    for name in (\'convert_safe\', \'baseline\'):\n        assert result.__globals__[name] is entry.__globals__[name]\n    return result\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a17/decode_nonzero19_stream.py': '"""Use the seven-fresh19 inverse only inside proved-nonzero streams."""\nfrom pathlib import Path\nimport json\nimport sys\n\nHERE=Path(__file__).resolve().parent\nsys.path[:0]=[str(HERE.parent/\'a14\'),str(HERE.parent/\'a04\')]\nimport decode_inverse20_12 as generic\nimport nonzero_consume16 as nonzero\n\nPLAN=json.loads((HERE/\'nonzero19_stream_plan.json\').read_text())\nassert PLAN[\'domain\']==\'nonzero\' and PLAN[\'max_live\']==13\ninverse6_nonzero19=generic.combo.xsub.clone(generic.inverse6_consume20_12,\n    dict(generic.inverse6_consume20_12.__globals__,PLAN=PLAN))\n\n\ndef install(entry,ceiling=889,minimum_space=7):\n    def wrap(original):\n        def chosen(b,x):\n            space=ceiling-b.num_allocated_qubits\n            if len(x)==6 and minimum_space<=space<10:\n                return inverse6_nonzero19(b,x)\n            return original(b,x)\n        return chosen\n    replacements={id(fn):wrap(fn) for fn in nonzero.functions(entry)\n                  if fn.__code__ is nonzero.stream_nonzero.__code__}\n    assert replacements\n    scope=dict(entry.__globals__)\n    scope[\'extraction\']=generic.combo.xsub.private_rebind(scope[\'extraction\'],replacements)\n    result=generic.combo.xsub.clone(entry,scope)\n    for name in (\'synthesize\',\'convert_safe\',\'baseline\'):\n        assert result.__globals__[name] is entry.__globals__[name]\n    return result\n\n\ndecode=install(nonzero.install(generic.combo.decode))\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a17/initialized17_factored.py': '"""Initialized A18 inverse with explicit conditional cubic phase repairs."""\nfrom pathlib import Path\nimport json,sys\nHERE=Path(__file__).resolve().parent\nsys.path[:0]=[str(HERE),str(HERE.parent/\'a08\'),str(HERE.parent/\'a04\'),str(HERE.parent/\'root\')]\nfrom inverse19_consume16 import ar,ref,combo,change_basis\nfrom nonzero_consume16 import functions\nimport decode_initialized19 as old\nDAG=json.loads((HERE/\'a18_dag17_frozen.json\').read_text())\nPLAN=json.loads((HERE/\'a18_17_initialized_b16_cubic_plan.json\').read_text())\nassert PLAN[\'found\'] and PLAN[\'preserve\'] and PLAN[\'max_live\']==16\n\n\ndef retire_cubic(b,live,pool,transcript,debts,record):\n    change_basis(b,live,record[\'rows\'])\n    for action in record[\'retire\']:\n        p=action[\'index\']\n        for j in action[\'mix\']:b.cx(live[j],live[p])\n        q=live.pop(p);m=b.alloc_bits(1)[0];b.hmr(q,m)\n        for term in action[\'phase\']:\n            kind=term[0]\n            if kind==\'constant\':b.z(m)\n            elif kind==\'linear\':b.cz(live[term[1]],m)\n            elif kind==\'pair\':b.ccz(live[term[1]],live[term[2]],m)\n            elif kind==\'original\':b.cx(m,transcript[term[1]])\n            elif kind==\'triple\':\n                b.push_condition(m)\n                b.ccz(*[live[i] for i in term[1:]])\n                b.pop_condition()\n            else:b.cx(m,debts[tuple(term)])\n        b.free(m);pool.append(q)\n\n\ndef pay_final_cubics_factored(b,live,debts):\n    """Exact factorization of the two late HMR characters in this plan."""\n    expected={("final_triple",0,1,k) for k in (2,3,4,5,8,9,10)}\n    expected.update(("final_triple",0,2,k) for k in (3,4,5))\n    assert set(debts)==expected\n    late=debts[("final_triple",0,1,2)]\n    earlier=debts[("final_triple",0,1,5)]\n    b.cx(late,earlier)\n    b.push_condition(late)\n    for j in (3,4,5,8,9,10):b.cx(live[j],live[2])\n    b.ccz(live[0],live[1],live[2])\n    for j in reversed((3,4,5,8,9,10)):b.cx(live[j],live[2])\n    for j in (4,5):b.cx(live[j],live[3])\n    b.ccz(live[0],live[2],live[3])\n    for j in (5,4):b.cx(live[j],live[3])\n    b.pop_condition()\n    b.push_condition(earlier)\n    b.ccz(live[0],live[1],live[5])\n    b.pop_condition()\n\n\ndef inverse6_initialized17_factored(b,x):\n    raw=b.alloc_qubits(7)\n    source=list(x)+list(raw)\n    for i,(left,right) in enumerate(DAG[\'source_forms\']):\n        with ref.expose_pair(b,source[:6+i],left,right) as(a,c):\n            b.init_and(a,c,raw[i])\n    pool=list(b.alloc_qubits(3));all_wires=source+pool[:];live=source[:]\n    transcript=b.alloc_bits(6);b.bit_store(transcript,False)\n    phase_bits=b.alloc_bits(len(PLAN[\'final_phases\']));b.bit_store(phase_bits,False)\n    debts={tuple(term):bit for term,bit in zip(PLAN[\'final_phases\'],phase_bits)}\n    retire_cubic(b,live,pool,transcript,debts,PLAN[\'begin\'])\n    for step in PLAN[\'steps\']:\n        if step[\'kind\']==\'new\':\n            target=pool.pop()\n            with ref.expose_pair(b,live,step[\'left\'],step[\'right\']) as(a,c):\n                b.init_and(a,c,target)\n            live.append(target)\n        else:\n            change_basis(b,live,step[\'pre_rows\'])\n            with ref.expose_pair(b,live,step[\'left\'],step[\'right\']) as(a,c):\n                b.ccx(a,c,live[step[\'target\']])\n        retire_cubic(b,live,pool,transcript,debts,step)\n    retire_cubic(b,live,pool,transcript,debts,PLAN[\'final\'])\n    assert len(live)==12\n    pay_final_cubics_factored(b,live,debts)\n    b.free(phase_bits)\n    for i,target in enumerate(x,6):\n        source=live[i]\n        if str(source)==str(target):continue\n        other=next((j for j,q in enumerate(live) if str(q)==str(target)),None)\n        b.swap(source,target);live[i]=target\n        if other is not None:live[other]=source\n    out=live[:6];kept=set(map(str,list(x)+out))\n    b.free([q for q in all_wires if str(q) not in kept])\n    b.cz(x,transcript);b.free(transcript)\n    return out\n\n\ndef install(entry):\n    replacements={id(fn):inverse6_initialized17_factored for fn in functions(entry)\n                  if fn.__code__ is old.inverse6_initialized19.__code__}\n    assert replacements\n    scope=dict(entry.__globals__)\n    for name in (\'extraction\',\'synthesize\'):\n        scope[name]=combo.xsub.private_rebind(scope[name],replacements)\n    result=combo.xsub.clone(entry,scope)\n    for name in (\'convert_safe\',\'baseline\'):\n        assert result.__globals__[name] is entry.__globals__[name]\n    return result\n\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a18/decode_initialized17_retire_last.py': '"""Exact initialized seventeen-product inverse with twelve fresh qubits."""\nfrom pathlib import Path\nimport json,sys\nHERE=Path(__file__).resolve().parent\nsys.path[:0]=[str(HERE),str(HERE.parent/\'root\')]\nimport decode_combo_v16 as parent\nfrom decode_vector_control import xs,functions\nfrom exp.inverse_search import inverse20 as ref\nPLAN=json.loads((HERE/\'retire_last_plan.json\').read_text())\n\n\ndef inverse6_initialized17_retire_last(b,x):\n    """Retire the last raw AND before allocating the sixth output coordinate."""\n    raw=list(b.alloc_qubits(7))\n    source=list(x)+raw\n    for i,(left,right)in enumerate(PLAN[\'source_forms\']):\n        with xs.expose_pair(b,source[:6+i],left,right)as(a,c):\n            b.init_and(a,c,raw[i])\n    out=list(b.alloc_qubits(5))\n    for j,form in enumerate(PLAN[\'linear\'][:5]):\n        if form&1:b.x(out[j])\n        for i in xs.bits(form>>1):b.cx(source[i],out[j])\n    for gate in PLAN[\'products\']:\n        if not((gate[\'left\']|gate[\'right\'])>>12&1):continue\n        with xs.expose_pair(b,source,gate[\'left\']<<1,gate[\'right\']<<1)as(a,c):\n            ref.rankone_xor(b,a,c,out,gate[\'vector\'])\n    with xs.expose_pair(b,source[:12],*PLAN[\'source_forms\'][6])as(a,c):\n        b.del_and(a,c,raw[-1])\n    b.free(raw[-1]);raw.pop();source=list(x)+raw\n    out+=list(b.alloc_qubits(1))\n    form=PLAN[\'linear\'][5]\n    if form&1:b.x(out[5])\n    for i in xs.bits(form>>1):b.cx(source[i],out[5])\n    for gate in PLAN[\'products\']:\n        if(gate[\'left\']|gate[\'right\'])>>12&1:continue\n        with xs.expose_pair(b,source,gate[\'left\']<<1,gate[\'right\']<<1)as(a,c):\n            ref.rankone_xor(b,a,c,out,gate[\'vector\'])\n    for i in reversed(range(6)):\n        with xs.expose_pair(b,source[:6+i],*PLAN[\'source_forms\'][i])as(a,c):\n            b.del_and(a,c,raw[i])\n    b.free(raw)\n    xs.linear_inplace(b,out,xs.inverse_columns(PLAN[\'output_columns\']))\n    return out\n\n\ndef install(entry):\n    replacements={id(fn):inverse6_initialized17_retire_last for fn in functions(entry)\n                  if fn.__code__ is xs.inverse6_init18.__code__}\n    assert replacements\n    scope=dict(entry.__globals__)\n    for name in(\'extraction\',\'synthesize\'):\n        scope[name]=xs.private_rebind(scope[name],replacements)\n    result=xs.clone(entry,scope)\n    for name in(\'convert_safe\',\'baseline\'):\n        assert result.__globals__[name]is entry.__globals__[name]\n    return result\n\n\ndecode=install(parent.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a18/decode_vector_control.py': '"""Whole-decoder control for seven-source vector inverse characters."""\nfrom pathlib import Path\nimport json,sys\nHERE=Path(__file__).resolve().parent\nROOT=HERE.parents[2]\nsys.path[:0]=[str(HERE.parent/\'root\'),str(HERE.parent/\'a04\')]\nimport decode_combo_v14 as current\nimport decode_quantum_inverse_phase as prior\nfrom nonzero_consume16 import functions\nfrom exp.inverse_search import inverse20 as ref\nxs=prior.xs\nPLAN=json.loads((HERE/\'control_plan.json\').read_text())\n\n\ndef phase_inverse6_vector(b,x,coefficient,annihilates_zero=False):\n    """Pay coefficient dot inverse0(x), preserving both quantum words."""\n    raw=b.alloc_qubits(7)\n    source=list(x)+list(raw)\n    for i,(left,right) in enumerate(PLAN[\'source_forms\']):\n        with xs.expose_pair(b,source[:6+i],left,right)as(a,c):\n            b.init_and(a,c,raw[i])\n    for j,form in enumerate(PLAN[\'linear\']):\n        if form&1:b.z(coefficient[j])\n        for i in xs.bits(form>>1):b.cz(source[i],coefficient[j])\n    for gate in PLAN[\'products\']:\n        positions=list(xs.bits(gate[\'vector\']))\n        pivot=positions[0]\n        for j in positions[1:]:b.cx(coefficient[j],coefficient[pivot])\n        with xs.expose_pair(b,source,gate[\'left\']<<1,gate[\'right\']<<1)as(a,c):\n            b.ccz(a,c,coefficient[pivot])\n        for j in reversed(positions[1:]):b.cx(coefficient[j],coefficient[pivot])\n    for i in reversed(range(7)):\n        with xs.expose_pair(b,source[:6+i],*PLAN[\'source_forms\'][i])as(a,c):\n            b.del_and(a,c,raw[i])\n    b.free(raw)\n\n\ndef install(entry,plan=PLAN):\n    replacement=xs.clone(phase_inverse6_vector,dict(phase_inverse6_vector.__globals__,PLAN=plan))\n    targets={id(fn):replacement for fn in functions(entry)\n             if fn.__code__ is prior.phase_inverse6_quantum.__code__}\n    assert targets\n    scope=dict(entry.__globals__)\n    for name in (\'extraction\',\'synthesize\'):\n        scope[name]=xs.private_rebind(scope[name],targets)\n    result=xs.clone(entry,scope)\n    for name in (\'convert_safe\',\'baseline\'):\n        assert result.__globals__[name] is entry.__globals__[name]\n    return result\n\n\ndecode=install(current.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a20/initialized17_linked.py': '"""Initialized inverse with exact cubic repairs on linked (inverse(x),x)."""\nfrom pathlib import Path\nimport json\nimport sys\n\nHERE=Path(__file__).resolve().parent\nsys.path[:0]=[str(HERE.parent/\'a17\')]\nimport initialized17_factored as original\n\nDATA=json.loads((HERE/\'final_cubic_search.json\').read_text())[\'found\']\n\n\ndef rank(rows):\n    basis={}\n    for row in rows:\n        while row:\n            p=row.bit_length()-1\n            if p in basis:row^=basis[p]\n            else:basis[p]=row;break\n    return len(basis)\n\n\ndef complete(rows):\n    rows=list(rows)\n    assert rank(rows)==len(rows)\n    for i in range(12):\n        if rank(rows+[1<<i])>len(rows):rows.append(1<<i)\n    assert len(rows)==12\n    return rows\n\n\ndef inverse_rows(rows):\n    work=[row|(1<<(12+i)) for i,row in enumerate(rows)]\n    for i in range(12):\n        j=next(j for j in range(i,12) if work[j]>>i&1)\n        work[i],work[j]=work[j],work[i]\n        for j in range(12):\n            if j!=i and work[j]>>i&1:work[j]^=work[i]\n    assert all((row&4095)==1<<i for i,row in enumerate(work))\n    return [row>>12 for row in work]\n\n\ndef columns(rows):\n    return [sum(((row>>j)&1)<<i for i,row in enumerate(rows)) for j in range(12)]\n\n\nMAPS={name:(columns(complete(plan[\'triple\'])),\n            columns(inverse_rows(complete(plan[\'triple\']))))\n      for name,plan in DATA.items()}\n\n\ndef pay_final_linked(b,live,debts):\n    """Pay both late characters on all64 linked inputs and all four masks."""\n    expected={("final_triple",0,1,k) for k in (2,3,4,5,8,9,10)}\n    expected.update(("final_triple",0,2,k) for k in (3,4,5))\n    assert set(debts)==expected\n    late=debts[("final_triple",0,1,2)]\n    earlier=debts[("final_triple",0,1,5)]\n    b.cx(late,earlier)\n    choices=b.alloc_bits(3);b.bit_store(choices,False)\n    late_only,early_only,both=choices\n    b.push_condition(late)\n    b.cx(earlier,both)\n    b.pop_condition()\n    b.cx(late,late_only);b.cx(both,late_only)\n    b.cx(earlier,early_only);b.cx(both,early_only)\n    for name,mask in [(\'late\',late_only),(\'early\',early_only),(\'xor\',both)]:\n        forward,back=MAPS[name]\n        b.push_condition(mask)\n        original.ar.linear_inplace(b,live,forward)\n        b.ccz(live[0],live[1],live[2])\n        original.ar.linear_inplace(b,live,back)\n        b.pop_condition()\n        for term in DATA[name][\'quadratic\']:\n            if term[0]==\'constant\':b.z(mask)\n            elif term[0]==\'linear\':b.cz(live[term[1]],mask)\n            else:\n                assert term[0]==\'pair\'\n                b.ccz(live[term[1]],live[term[2]],mask)\n    b.free(choices)\n\n\n_run=original.combo.xsub.clone(original.inverse6_initialized17_factored,\n    dict(original.inverse6_initialized17_factored.__globals__,\n         pay_final_cubics_factored=pay_final_linked))\n\n\ndef inverse6_initialized17_linked(b,x):\n    return _run(b,x)\n\n\ndef install(entry):\n    replacements={id(fn):inverse6_initialized17_linked\n                  for fn in original.functions(entry)\n                  if fn.__code__ is original.old.inverse6_initialized19.__code__}\n    assert replacements\n    scope=dict(entry.__globals__)\n    for name in (\'extraction\',\'synthesize\'):\n        scope[name]=original.combo.xsub.private_rebind(scope[name],replacements)\n    result=original.combo.xsub.clone(entry,scope)\n    for name in (\'convert_safe\',\'baseline\'):\n        assert result.__globals__[name] is entry.__globals__[name]\n    return result\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_basic18.py': '"""Use the exact initialized inverse at remaining basic GF64 producers."""\nfrom pathlib import Path\nimport sys\nHERE = Path(__file__).resolve().parent\nKIT_ROOT = HERE.parents[2]\nsys.path[:0] = [str(KIT_ROOT / \'exp/r-main/narrow240\'), str(HERE.parent / \'a03\')]\nimport decode_combo as combo\nimport exact_subs as xs\nfrom jet_norm6 import replacement_ids\nfrom exp.inverse_width_subspaces import kernel19 as basic\n\n\ndef inverse6_initialized18(b, x, z=None, keep_norm=False):\n    if z is None and not keep_norm:\n        return xs.inverse6_init18(b, x)\n    return fallback(b, x, z, keep_norm)\n\n\nfallback = basic.inverse6_xor19\nreplacement = xs.private_rebind(inverse6_initialized18, combo.synth.replacements)\n\n\ndef install(entry):\n    replacements = replacement_ids(entry, basic.inverse6_xor19, replacement)\n    assert replacements\n    scope = dict(entry.__globals__)\n    for name in (\'extraction\', \'synthesize\'):\n        scope[name] = xs.private_rebind(scope[name], replacements)\n    return xs.clone(entry, scope)\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_cached12.py': '"""Reuse the cached GF4096 inverse in front-half initialized producers."""\nfrom pathlib import Path\nimport sys\nHERE = Path(__file__).resolve().parent\nKIT_ROOT = HERE.parents[2]\nsys.path[:0] = [str(KIT_ROOT / \'exp/r-main/narrow240\'), str(HERE.parent / \'a03\')]\nimport decode_combo as combo\nimport exact_subs as xs\nfrom jet_norm6 import replacement_ids\nfrom exp.jet24 import pro_jet as jet\nfrom exp.inverse_search import inverse20 as invref\n\n\ndef inverse_initialized_cached12(b, x, target=None):\n    if len(x) != 12 or target is not None or b.num_allocated_qubits + 22 > 889:\n        return initialized_fallback(b, x, target)\n    out, F = jet.cached_inverse(b, x, jet.PLAN12, True)\n    fm = b.alloc_bits(1)\n    b.hmr(F, fm[0])\n    b.free(F)\n    b.z(fm[0])\n    mask = b.alloc_bits(12)\n    b.bit_store(mask, False)\n    b.cx(fm[0], mask[0])\n    jet.ar.phase_product(b, x, out, mask)\n    b.free(mask)\n    b.free(fm)\n    return out\n\n\ndef inverse_zero_cached12(b, x):\n    if b.num_allocated_qubits + 22 > 889:\n        return zero_fallback(b, x)\n    return jet.cached_inverse(b, x, jet.PLAN12, True)\n\n\ninitialized_fallback = invref.inverse_initialized\nzero_fallback = xs.inverse_zero\ninitialized_replacement = xs.private_rebind(inverse_initialized_cached12, combo.synth.replacements)\nzero_replacement = xs.private_rebind(inverse_zero_cached12, combo.synth.replacements)\n\n\ndef install(entry):\n    replacements = replacement_ids(entry, invref.inverse_initialized, initialized_replacement)\n    replacements.update(replacement_ids(entry, xs.inverse_zero, zero_replacement))\n    assert replacements\n    scope = dict(entry.__globals__)\n    for name in (\'extraction\', \'synthesize\'):\n        scope[name] = xs.private_rebind(scope[name], replacements)\n    return xs.clone(entry, scope)\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_cached_leaf20.py': '"""Use the supplied narrow isotopy inverse in the new flag-free caches."""\nfrom pathlib import Path\nimport sys\nHERE = Path(__file__).resolve().parent\nsys.path.insert(0, str(HERE))\nimport decode_flagfree_cached as cache\nfrom exp.inverse_search import inverse20_iso as iso\n\n\ndef install(entry):\n    replacements = cache.replacement_ids(entry, cache.inverse6_initialized_nine,\n                                         iso.inverse6_xor20)\n    assert replacements\n    scope = dict(entry.__globals__)\n    for name in (\'extraction\', \'synthesize\'):\n        scope[name] = cache.xs.private_rebind(scope[name], replacements)\n    return cache.xs.clone(entry, scope)\n\n\ndecode = install(cache.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v11.py': '"""Exact front half with all current accepted arithmetic and replay changes."""\nfrom pathlib import Path\nimport sys\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent/\'a11\'), str(HERE.parent/\'a12\')]\nimport decode_combo_v9\nimport decode_inverse_retained\nimport decode_outer_alias_constant\nimport decode_virtual_leader\nimport decode_outer35\nimport decode_companion_preserve\n\ndecode = decode_inverse_retained.install(decode_combo_v9.decode)\ndecode = decode_outer_alias_constant.install(decode)\ndecode = decode_virtual_leader.install(decode)\ndecode = decode_outer35.install(decode)\ndecode = decode_companion_preserve.install(decode, with_norm=True, ceiling=889)\nfor name in (\'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is decode_combo_v9.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v12.py': '"""Exact front half with six-fresh inverse and paired inverse characters."""\nfrom pathlib import Path\nimport sys\n\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent/\'a14\'), str(HERE.parent/\'a16\')]\nimport decode_combo_v11\nimport decode_inverse20_12\nimport decode_pair\n\ndecode = decode_inverse20_12.install(decode_combo_v11.decode)\ndecode = decode_pair.install(decode)\nfor name in (\'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is decode_combo_v11.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v13.py': '"""Exact front half with the clipped-divider nonzero retained inverse."""\nfrom pathlib import Path\nimport sys\n\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent/\'a14\')]\nimport decode_combo_v12\nimport decode_retained19\n\ndecode = decode_retained19.install(decode_combo_v12.decode)\nfor name in (\'extraction\', \'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is decode_combo_v12.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v14.py': '"""Exact front half with fused quantum/classical retained-norm characters."""\nfrom pathlib import Path\nimport sys\n\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent/\'a16\')]\nimport decode_combo_v13\nimport decode_partial_many\n\ndecode = decode_partial_many.install(decode_combo_v13.decode)\nfor name in (\'synthesize\', \'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is decode_combo_v13.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v15.py': '"""Exact front half with an additional initialized scalar inverse tier."""\nfrom pathlib import Path\nimport sys\n\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent/\'a17\')]\nimport decode_combo_v14\nimport decode_initialized19\n\ndecode = decode_initialized19.install(decode_combo_v14.decode)\nfor name in (\'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is decode_combo_v14.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v16.py': '"""Exact front half with the additional scoped nonzero scalar stream."""\nfrom pathlib import Path\nimport sys\n\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent/\'a17\')]\nimport decode_combo_v15\nimport decode_nonzero19_stream\n\ndecode = decode_nonzero19_stream.install(decode_combo_v15.decode)\nfor name in (\'synthesize\', \'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is decode_combo_v15.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v18.py': '"""Exact front half with same-width initialized17 and linked-phase inverse tiers."""\nfrom pathlib import Path\nimport sys\n\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent/\'a18\'), str(HERE.parent/\'a20\')]\nimport decode_combo_v16\nimport decode_initialized17_retire_last\nimport initialized17_linked\n\nwide = decode_initialized17_retire_last.install(decode_combo_v16.decode)\ndecode = initialized17_linked.install(wide)\nfor name in (\'convert_safe\', \'baseline\', \'linear_words\', \'matrix\'):\n    assert decode.__globals__[name] is decode_combo_v16.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v3.py': '"""Combined exact front half, including proved-nonzero consuming inverses."""\nfrom pathlib import Path\nimport sys\nHERE = Path(__file__).resolve().parent\nKIT_ROOT = HERE.parents[2]\nsys.path[:0] = [str(HERE), str(HERE.parent / \'a03\'), str(HERE.parent / \'a04\'),\n               str(KIT_ROOT / \'exp/r-main/narrow240\')]\nimport decode_combo as supplied\nimport jet_norm6_front\nimport inverse_consume17_front\nimport nonzero_consume16\nimport decode_shared_correction_front\nimport decode_local_norm_partial\nimport decode_local_forward\n\ndecode = supplied.decode\nfor component in (jet_norm6_front, decode_shared_correction_front,\n                  decode_local_norm_partial, decode_local_forward,\n                  inverse_consume17_front, nonzero_consume16):\n    decode = component.install(decode)\nfor name in (\'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is supplied.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v4.py': '"""Combined exact front half with cached initialized GF4096 producers."""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport decode_combo_v3\nimport decode_cached12\nimport decode_basic18\ndecode = decode_cached12.install(decode_combo_v3.decode)\ndecode = decode_basic18.install(decode)\nfor name in (\'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is decode_combo_v3.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v5.py': '"""Combined exact front-half improvements after the second agent wave."""\nfrom pathlib import Path\nimport sys\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent/\'a05\'), str(HERE.parent/\'a07\')]\nimport decode_combo_v4\nimport decode_local_forward_sheared\nimport decode_quantum_inverse_phase\nimport decode_correction_and\nimport decode_prefix_gauge\nimport scalar_projection_multi\nimport scalar_projection12\n\ndecode = decode_local_forward_sheared.install(decode_combo_v4.decode)\ndecode = decode_quantum_inverse_phase.install(decode)\ndecode = decode_correction_and.install(decode)\ndecode = decode_prefix_gauge.install(decode)\ndecode = scalar_projection_multi.install(decode, {9:5, 10:4, 11:3, 12:2, 13:1})\ndecode = scalar_projection12.install(decode)\nfor name in (\'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is decode_combo_v4.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v6.py': '"""Exact front-half combination including flag-free scalar caches."""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport decode_combo_v5\nimport decode_flagfree_cached\nimport decode_stream12_cached\n\ndecode = decode_flagfree_cached.install(decode_combo_v5.decode)\ndecode = decode_stream12_cached.install(decode)\nfor name in (\'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is decode_combo_v5.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v7.py': '"""Exact front half with the narrow isotopy scalar producer."""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport decode_combo_v6\nimport decode_cached_leaf20\ndecode = decode_cached_leaf20.install(decode_combo_v6.decode)\nfor name in (\'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is decode_combo_v6.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v8.py': '"""Exact front half with full polynomial folds and narrow scalar leaves."""\nfrom pathlib import Path\nimport sys\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent / \'a08\')]\nimport decode_combo_v7\nimport decode_fullfold\nimport preserve_streams\n\ndecode = decode_fullfold.install(decode_combo_v7.decode, prefix_gauge=True)\ndecode = preserve_streams.install(decode)\nfor name in (\'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is decode_combo_v7.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v9.py': '"""Exact front half with low-Taylor full-fold reverses."""\nfrom pathlib import Path\nimport sys\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent/\'a11\')]\nimport decode_combo_v8\nimport decode_low_crt\n\ndecode = decode_low_crt.install(decode_combo_v8.decode)\nfor name in (\'convert_safe\', \'baseline\'):\n    assert decode.__globals__[name] is decode_combo_v8.decode.__globals__[name]\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_flagfree_cached.py': '"""Flag-free cached inverses at scalar sites that do not need a zero flag."""\nfrom pathlib import Path\nfrom functools import lru_cache\nimport sys\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent/\'a03\')]\nimport decode_local_forward_sheared as sheared\nimport decode_local_forward_partial as partial\nimport decode_local_forward as old_forward\nimport decode_cached12 as old_cached12\nfrom decode_local_forward import combo, jet, joint, replacement_ids\nfrom exp.noscratch import schedules, kernels\nfrom exp.inverse_search import inverse20 as invref\nxs = combo.xsub\n\n\ndef inverse6_initialized_nine(b, x):\n    """Preserve x and return inverse0(x), using the quadratic tower."""\n    to, back, constant = jet.tower(6)\n    jet.linear_inplace(b, x, to)\n    norm = b.alloc_qubits(3)\n    schedules.norm_xor_noscratch(b, x, norm, constant)\n    reciprocal = kernels.inverse3_noscratch(b, norm)\n    jet.ar._erase_norm(b, x, norm, constant)\n    b.free(norm)\n    out = b.alloc_qubits(6)\n    b.cx(x[3:], x[:3])\n    kernels.mul_xor_noscratch(b, x[:3], reciprocal, out[:3])\n    kernels.mul_xor_noscratch(b, x[3:], reciprocal, out[3:])\n    b.cx(x[3:], x[:3])\n    jet.ar._erase_norm(b, out, reciprocal, constant)\n    b.free(reciprocal)\n    jet.linear_inplace(b, x, back)\n    jet.linear_inplace(b, out, back)\n    return out\n\n\ndef leaf_fast(b, x, with_zero):\n    assert not with_zero\n    return xs.inverse6_init18(b, x), None\n\n\ndef leaf_narrow(b, x, with_zero):\n    assert not with_zero\n    return inverse6_initialized_nine(b, x), None\n\n\ndef make_cached12(leaf):\n    return xs.clone(jet.cached_inverse,\n                    dict(jet.cached_inverse.__globals__, inverse6_fixed18=leaf))\n\n\nCACHED12 = (make_cached12(leaf_fast), make_cached12(leaf_narrow))\nCACHED24 = tuple(xs.clone(sheared.cached_inverse_partial,\n    dict(sheared.cached_inverse_partial.__globals__, cached_inverse=inner))\n    for inner in CACHED12)\n\n\n@lru_cache(None)\ndef choose_outer(space):\n    options = []\n    for narrow, inner_width in enumerate((22, 19)):\n        for count in range(7):\n            if max(21-count+inner_width, 36) > space:\n                continue\n            columns, _ = sheared.partial_layout(count)\n            rebuild = sum(bool(jet.apply(columns, v) >> (21-count))\n                          for _, _, v in jet.PLAN24[\'terms\'])\n            options.append((3*narrow + rebuild, count, narrow))\n    return min(options)[1:] if options else None\n\n\ndef inverse24_forward_flagfree(b, x, K=None):\n    selection = choose_outer(889-b.num_allocated_qubits)\n    if selection is None:\n        return joint.inverse24_forward190(b, x, K)\n    count, narrow = selection\n    columns, back, _ = jet.extension_columns(K)\n    jet.linear_inplace(b, x, columns)\n    out, flag = CACHED24[narrow](b, x, jet.PLAN24, count, False)\n    assert flag is None\n    transcript = b.alloc_bits(24)\n    b.hmr(x, transcript)\n    b.swap(x, out)\n    b.free(out)\n    jet.linear_inplace(b, x, back)\n    return transcript\n\n\ndef inverse_initialized_flagfree12(b, x, target=None):\n    space = 889-b.num_allocated_qubits\n    if len(x) != 12 or target is not None or space < 19:\n        return invref.inverse_initialized(b, x, target)\n    out, flag = CACHED12[space < 22](b, x, jet.PLAN12, False)\n    assert flag is None\n    return out\n\n\ndef install(entry):\n    replacements = {}\n    forward = xs.private_rebind(inverse24_forward_flagfree, combo.synth.replacements)\n    initialized = xs.private_rebind(inverse_initialized_flagfree12, combo.synth.replacements)\n    for old in (joint.inverse24_forward190, old_forward.inverse24_forward_cached,\n                partial.inverse24_forward_partial):\n        replacements.update(replacement_ids(entry, old, forward))\n    for old in (invref.inverse_initialized, old_cached12.inverse_initialized_cached12):\n        replacements.update(replacement_ids(entry, old, initialized))\n    assert replacements\n    scope = dict(entry.__globals__)\n    for name in (\'extraction\', \'synthesize\'):\n        scope[name] = xs.private_rebind(scope[name], replacements)\n    return xs.clone(entry, scope)\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_fullfold.py': '"""Absorb the odd prefix update into the full polynomial fold.\n\nAt an odd fold, p_after=p_before+h*Q.  The old projected product adds\nh*Q to the two top stored coordinates, and its alias then adds p_after\nthere.  These cancel to p_before.  A full fold performs that update and\nthe p update in one polynomial product, retaining the same zero alias.\n"""\nfrom pathlib import Path\nimport inspect\nimport sys\nHERE = Path(__file__).resolve().parent\nROOT = HERE.parents[2]\nsys.path.insert(0, str(ROOT/\'exp/r-main/narrow240\'))\nimport decode_combo as combo\n\n\ndef install(entry, prefix_gauge=False):\n    original = entry.__globals__[\'synthesize\']\n    source = inspect.getsource(combo.decode.__globals__[\'synthesize\'])\n    old = \'\'\'        if i%2:\n            ar.mul_xor0(b,heads[i][1],r,border);qi,p=r,border\n            A,B,t=folds.sg[\'absorb\'](b,heads[i],A,B,qi,p,None,0,cs[i],ks[i],wide=False)\n        else:\n            qi,p=border,r\n            A,B,t=folds.sg[\'absorb_even\'](b,heads[i],A,B,qi,r,cs[i],ks[i])\'\'\'\n    new = \'\'\'        qi,p=(r,border) if i%2 else (border,r)\n        A,B,t=folds.sg[\'absorb_even\'](b,heads[i],A,B,qi,p,cs[i],ks[i])\'\'\'\n    assert source.count(old) == 1\n    source = source.replace(old, new)\n    old = \'\'\'            if i%2:\n                folds.ug[\'absorb\'](b,heads[i],aa,bb,qi,p,None,0,cs[i],ks[i],True,t,False)\n                ar.mul_xor0(b,heads[i][1],r,border)\n            else:folds.ug[\'absorb_even\'](b,heads[i],aa,bb,qi,r,cs[i],ks[i],True,t)\'\'\'\n    new = \'\'\'            folds.ug[\'absorb_even\'](b,heads[i],aa,bb,qi,p,cs[i],ks[i],True,t)\'\'\'\n    assert source.count(old) == 1\n    source = source.replace(old, new)\n    if prefix_gauge:\n        edits = (\n            (\'b.cx(border,r);b.cx(border,d);b.cx(r,border);b.x(border[0])\',\n             \'b.cx(border,r);b.cx(r,border);b.x(border[0])\'),\n            (\'folds.add_tail(b,heads[i],qi,d,bool(i%2))\',\n             \'folds.add_tail(b,heads[i],qi,d,not bool(i%2))\'),\n            (\'ar.mul_xor0(b,r,border,d);tt=tail_inplace\', \'tt=tail_inplace\'),\n            (\'tail_inplace(b,r,border,d,u,v,pole,True,tt);ar.mul_xor0(b,r,border,d)\',\n             \'tail_inplace(b,r,border,d,u,v,pole,True,tt)\'),\n            (\'b.cx(r,d);b.x(border[0]);b.cx(r,border);b.cx(border,r);b.free(r)\',\n             \'b.x(border[0]);b.cx(r,border);b.cx(border,r);b.free(r)\'),\n        )\n        for before, after in edits:\n            assert before in source\n            source = source.replace(before, after)\n    environment = dict(original.__globals__)\n    exec(compile(source, __file__, \'exec\'), environment)\n    return combo.xsub.clone(entry, dict(entry.__globals__, synthesize=environment[\'synthesize\']))\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_inverse_retained.py': '"""Joint consuming inversion and retained norm coordinates in fold division."""\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport inspect\nimport json\nimport sys\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent/\'a08\')]\nfrom inverse20_consume14 import ar, ref, iso, change_basis, retire, inverse3_inplace, combo\nfrom final_decoder import synthesis as original_folds\nimport norm_cache\nimport decode_combo_v9\n\n\ndef inverse6_retained(b, x, plan):\n    """Return an original-input phase mask and k linear source coordinates."""\n    assert plan[\'domain\'] == \'all\'\n    to, back, K = ar.tower(6)\n    ar.linear_inplace(b, x, to)\n    raw = b.alloc_qubits(6)\n    for j, (left, right) in enumerate(iso.raw_forms()):\n        with ref.expose_pair(b, x, left, right) as (a, c):\n            b.init_and(a, c, raw[j])\n    rows = iso.NORM_ROWS + iso.EXTRA_ROWS\n    columns = [sum(((row >> j) & 1) << i for i, row in enumerate(rows)) for j in range(6)]\n    ar.linear_inplace(b, raw, columns)\n    for j, form in enumerate(ref.norm_forms(K)):\n        ref.form_xor(b, x, form & 127, raw[j])\n    inverse3_inplace(b, raw[:3])\n    pool = list(b.alloc_qubits(plan[\'max_live\']-12))\n    all_wires = list(x) + list(raw) + pool[:]\n    live = list(x) + list(raw)\n    transcript = b.alloc_bits(6)\n    b.bit_store(transcript, False)\n    retire(b, live, pool, transcript, plan[\'begin\'])\n    for step in plan[\'steps\']:\n        if step[\'kind\'] == \'new\':\n            target = pool.pop()\n            with ref.expose_pair(b, live, step[\'left\'], step[\'right\']) as (a, c):\n                b.init_and(a, c, target)\n            live.append(target)\n        else:\n            change_basis(b, live, step[\'pre_rows\'])\n            with ref.expose_pair(b, live, step[\'left\'], step[\'right\']) as (a, c):\n                b.ccx(a, c, live[step[\'target\']])\n        retire(b, live, pool, transcript, step)\n    retire(b, live, pool, transcript, plan[\'final\'])\n    assert len(live) == 6 + plan[\'retained\']\n    for i, target in enumerate(x):\n        source = live[i]\n        if str(source) == str(target):\n            continue\n        other = next((j for j, q in enumerate(live) if str(q) == str(target)), None)\n        b.swap(source, target)\n        live[i] = target\n        if other is not None:\n            live[other] = source\n    cache = live[6:]\n    held = set(map(str, list(x)+cache))\n    b.free([q for q in all_wires if str(q) not in held])\n    return transcript, cache\n\n\ndef install(entry, keep=2):\n    plan = json.loads((HERE/f\'retained_inverse_k{keep}_b14_all.json\').read_text())\n    assert plan[\'found\']\n    synthesize = entry.__globals__[\'synthesize\']\n    folds = synthesize.__globals__[\'folds\']\n    old_divide = folds.ug[\'absorb_even\'].__globals__[\'divide_fused\']\n    source = inspect.getsource(original_folds.divide_fused)\n    before = \'    t = inverse6_xor20(b, M)\\n\'\n    after = \'\'\'    inverse_mask, kept_M = produce_retained(b, M)\n    t = M\n    part_masks = []\n\'\'\'\n    assert source.count(before) == 1\n    source = source.replace(before, after)\n    before = \'            ar.erase_product(b, out, M, part)\\n\'\n    after = \'\'\'            pm = b.alloc_bits(6)\n            b.hmr(part, pm)\n            part_masks.append(pm)\n\'\'\'\n    assert source.count(before) == 1\n    source = source.replace(before, after)\n    before = \'    for off, m in zip((0, 12), masks):\\n\'\n    after = \'\'\'    ar.linear_inplace(b, N, to)\n    M = rebuild_retained(b, N, kept_M, k6)\n    ar.linear_inplace(b, N, frm)\n    b.cz(M, inverse_mask)\n    b.free(inverse_mask)\n    for off in (0, 12):\n        half = x[off:off+12]\n        ar.linear_inplace(b, half, to)\n        for j in (0, 6):\n            pm = part_masks.pop(0)\n            ar.phase_product(b, half[j:j+6], M, pm)\n            b.free(pm)\n        ar.linear_inplace(b, half, frm)\n    assert not part_masks\n    for off, m in zip((0, 12), masks):\n\'\'\'\n    assert source.count(before) == 1\n    source = source.replace(before, after)\n    def produce_retained(b, x):\n        return inverse6_retained(b, x, plan)\n    environment = dict(old_divide.__globals__, produce_retained=produce_retained,\n                       rebuild_retained=norm_cache.rebuild)\n    exec(compile(source, __file__, \'exec\'), environment)\n    ug = dict(folds.ug)\n    for name in (\'absorb\', \'absorb_even\'):\n        fn = ug[name]\n        ug[name] = combo.xsub.clone(fn, dict(fn.__globals__, divide_fused=environment[\'divide_fused\']))\n    proxy = SimpleNamespace(**dict(vars(folds), ug=ug))\n    replacement = combo.xsub.clone(synthesize, dict(synthesize.__globals__, folds=proxy))\n    result = combo.xsub.clone(entry, dict(entry.__globals__, synthesize=replacement))\n    for name in (\'convert_safe\', \'baseline\'):\n        assert result.__globals__[name] is entry.__globals__[name]\n    return result\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_local_forward.py': '"""Reuse the jet\'s cached inverse for local scalar heads where it fits."""\nfrom pathlib import Path\nimport sys\n\nHERE = Path(__file__).resolve().parent\nKIT_ROOT = HERE.parents[2]\nsys.path[:0] = [str(KIT_ROOT / \'exp/r-main/narrow240\'), str(HERE.parent / \'a03\')]\nimport decode_combo as combo\nfrom jet_norm6 import replacement_ids\nfrom exp.jet24 import pro_jet as jet\nfrom exp.inverse12_escape import joint\n\n\ndef inverse24_forward_cached(b, x, K=None):\n    if b.num_allocated_qubits + 43 > 889:\n        return fallback(b, x, K)\n    cols, back, _ = jet.extension_columns(K)\n    jet.linear_inplace(b, x, cols)\n    out, F = jet.cached_inverse(b, x, jet.PLAN24, True)\n    fm = b.alloc_bits(1)\n    b.hmr(F, fm[0])\n    b.free(F)\n    b.z(fm[0])\n    mask = b.alloc_bits(24)\n    b.bit_store(mask, False)\n    b.cx(fm[0], mask[0])\n    jet.phase_product_quantum99(b, x, out, mask)\n    b.free(mask)\n    b.free(fm)\n    mu = b.alloc_bits(24)\n    b.hmr(x, mu)\n    b.swap(x, out)\n    b.free(out)\n    jet.linear_inplace(b, x, back)\n    return mu\n\n\nfallback = joint.inverse24_forward190\nreplacement = combo.xsub.private_rebind(inverse24_forward_cached, combo.synth.replacements)\n\n\ndef install(entry):\n    replacements = replacement_ids(entry, joint.inverse24_forward190, replacement)\n    assert replacements\n    scope = dict(entry.__globals__)\n    scope[\'extraction\'] = combo.xsub.private_rebind(scope[\'extraction\'], replacements)\n    return combo.xsub.clone(entry, scope)\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_local_forward_partial.py': '"""Retire selected raw norm directions during the nested cached inverse."""\nfrom pathlib import Path\nfrom functools import lru_cache\nimport inspect\nimport sys\nHERE = Path(__file__).resolve().parent\nsys.path.insert(0, str(HERE))\nfrom decode_local_forward import *\n\nOMITTED = (212, 324, 32, 75, 2)\n\n\n@lru_cache(None)\ndef partial_layout(count):\n    omitted = list(OMITTED[:count])\n    span = {0}\n    for row in omitted:\n        assert row not in span\n        span |= {v ^ row for v in tuple(span)}\n    kept = []\n    for row in (1 << j for j in range(9)):\n        if row not in span:\n            kept.append(row)\n            span |= {v ^ row for v in tuple(span)}\n    rows = [1 << j for j in range(12)] + [v << 12 for v in kept + omitted]\n    columns = [sum(((row >> j) & 1) << i for i, row in enumerate(rows))\n               for j in range(21)]\n    return tuple(columns), tuple(jet.inverse_columns(columns))\n\n\ndef retire_raw(b, x, raw, count):\n    columns, back = partial_layout(count)\n    jet.linear_inplace(b, raw, columns)\n    mask = b.alloc_bits(count)\n    b.hmr(raw[21-count:], mask)\n    b.free(raw[21-count:])\n    for left, right, vector in jet.PLAN24[\'terms\']:\n        removed = jet.apply(columns, vector) >> (21-count)\n        if not removed:\n            continue\n        with jet.ref.expose_pair(b, x, left << 1, right << 1) as (a, c):\n            for i in jet.bits(removed):\n                b.ccz(a, c, mask[i])\n    b.free(mask)\n    return list(raw[:21-count])\n\n\ndef rebuild_raw(b, x, raw, count):\n    columns, back = partial_layout(count)\n    removed = b.alloc_qubits(count)\n    for left, right, vector in jet.PLAN24[\'terms\']:\n        projected = jet.apply(columns, vector) >> (21-count)\n        if not projected:\n            continue\n        with jet.ref.expose_pair(b, x, left << 1, right << 1) as (a, c):\n            jet.rankone_xor(b, a, c, removed, projected)\n    raw = list(raw) + list(removed)\n    jet.linear_inplace(b, raw, back)\n    return raw\n\n\nsource = inspect.getsource(jet.cached_inverse)\nsource = source.replace(\'def cached_inverse(b, x, plan, with_zero=True):\',\n                        \'def cached_inverse_partial(b, x, plan, count, with_zero=True):\')\nsource = source.replace(\'    D = raw[:h]\',\n                        \'    raw = retire_raw(b, x, raw, count)\\n    D = raw[:h]\')\nsource = source.replace("    linear_inplace(b, raw, plan[\'reduce\'])",\n                        "    raw = rebuild_raw(b, x, raw, count)\\n    linear_inplace(b, raw, plan[\'reduce\'])")\nenvironment = dict(jet.cached_inverse.__globals__, retire_raw=retire_raw,\n                   rebuild_raw=rebuild_raw)\nexec(compile(source, __file__, \'exec\'), environment)\ncached_inverse_partial = environment[\'cached_inverse_partial\']\n\n\ndef inverse24_forward_partial(b, x, K=None):\n    omitted = max(0, b.num_allocated_qubits + 43 - 889)\n    if omitted > len(OMITTED):\n        return fallback(b, x, K)\n    cols, back, _ = jet.extension_columns(K)\n    jet.linear_inplace(b, x, cols)\n    if omitted:\n        out, F = cached_inverse_partial(b, x, jet.PLAN24, omitted, True)\n    else:\n        out, F = jet.cached_inverse(b, x, jet.PLAN24, True)\n    fm = b.alloc_bits(1)\n    b.hmr(F, fm[0])\n    b.free(F)\n    b.z(fm[0])\n    mask = b.alloc_bits(24)\n    b.bit_store(mask, False)\n    b.cx(fm[0], mask[0])\n    jet.phase_product_quantum99(b, x, out, mask)\n    b.free(mask)\n    b.free(fm)\n    mu = b.alloc_bits(24)\n    b.hmr(x, mu)\n    b.swap(x, out)\n    b.free(out)\n    jet.linear_inplace(b, x, back)\n    return mu\n\n\nreplacement = combo.xsub.private_rebind(inverse24_forward_partial, combo.synth.replacements)\n\n\ndef install(entry):\n    replacements = replacement_ids(entry, joint.inverse24_forward190, replacement)\n    replacements.update(replacement_ids(entry, inverse24_forward_cached, replacement))\n    assert replacements\n    scope = dict(entry.__globals__)\n    scope[\'extraction\'] = combo.xsub.private_rebind(scope[\'extraction\'], replacements)\n    return combo.xsub.clone(entry, scope)\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_local_forward_sheared.py': '"""Retire raw cache coordinates after exact norm-coordinate shears."""\nfrom functools import lru_cache\nfrom pathlib import Path\nimport inspect\nimport sys\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport decode_local_forward_partial as partial\nfrom decode_local_forward_partial import combo, jet, replacement_ids, joint\n\nROWS = (446783, 868352, 1573003, 24895, 90548, 196747)\n\n\n@lru_cache(None)\ndef partial_layout(count):\n    omitted = list(ROWS[:count])\n    span = {0}\n    for row in omitted:\n        quotient = row >> 12\n        assert quotient not in span\n        span |= {v ^ quotient for v in tuple(span)}\n    kept = []\n    for quotient in (1 << j for j in range(9)):\n        if quotient not in span:\n            kept.append(quotient << 12)\n            span |= {v ^ quotient for v in tuple(span)}\n    rows = [1 << j for j in range(12)] + kept + omitted\n    columns = [sum(((row >> j) & 1) << i for i, row in enumerate(rows))\n               for j in range(21)]\n    return tuple(columns), tuple(jet.inverse_columns(columns))\n\n\nretire_raw = combo.xsub.clone(partial.retire_raw,\n                              dict(partial.retire_raw.__globals__, partial_layout=partial_layout))\nrebuild_raw = combo.xsub.clone(partial.rebuild_raw,\n                               dict(partial.rebuild_raw.__globals__, partial_layout=partial_layout))\ncached_inverse_partial = combo.xsub.clone(partial.cached_inverse_partial,\n    dict(partial.cached_inverse_partial.__globals__, retire_raw=retire_raw, rebuild_raw=rebuild_raw))\nsource = inspect.getsource(partial.inverse24_forward_partial)\nsource = source.replace(\'if omitted > len(OMITTED):\',\n                        \'if omitted > len(OMITTED) or b.num_allocated_qubits + 37 > 889:\')\nenvironment = dict(partial.inverse24_forward_partial.__globals__,\n                   OMITTED=ROWS, cached_inverse_partial=cached_inverse_partial)\nexec(compile(source, __file__, \'exec\'), environment)\nreplacement = combo.xsub.private_rebind(environment[\'inverse24_forward_partial\'], combo.synth.replacements)\ninstall = combo.xsub.clone(partial.install, dict(partial.install.__globals__, replacement=replacement))\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_local_norm.py': '"""Retain a nested norm in local GF(2^24) inverse reverses when width allows."""\nfrom pathlib import Path\nimport sys\n\nHERE = Path(__file__).resolve().parent\nROOT = HERE.parents[2]\nsys.path[:0] = [str(ROOT / \'exp/r-main/narrow240\'), str(HERE.parent / \'a03\')]\nimport decode_combo as combo\nimport exact_subs as xs\nfrom jet_norm6 import replacement_ids\nfrom exp.inverse12_escape import joint\nfrom exp.inverse12_escape.joint import *\n\n\ndef inverse12_keep_both_norms(b, x):\n    to, frm, K = tower(12)\n    linear_inplace(b, x, to)\n    D = b.alloc_qubits(6)\n    norm_xor_noscratch(b, x, D, K)\n    u = inverse6_init19(b, D)\n    t = b.alloc_qubits(12)\n    b.cx(x[6:], x[:6])\n    mul_xor_noscratch(b, x[:6], u, t[:6])\n    mul_xor_noscratch(b, x[6:], u, t[6:])\n    b.cx(x[6:], x[:6])\n    linear_inplace(b, x, frm)\n    linear_inplace(b, t, frm)\n    return t, u, D\n\n\ndef erase_norm_with_both(b, t, N, u, D):\n    to, frm, K = tower(12)\n    linear_inplace(b, t, to)\n    linear_inplace(b, N, to)\n    m = b.alloc_bits(12)\n    b.hmr(N, m)\n    b.free(N)\n    L = b.alloc_qubits(6)\n    f = field(6)\n    def coefficient():\n        for j in range(6):\n            for i in range(6):\n                for r in bits(f.mul(1 << i, 1 << j)):\n                    b.ccx(m[r], t[i], L[j])\n                    b.ccx(m[r], t[i+6], L[j])\n                    b.ccx(m[r+6], t[i+6], L[j])\n    coefficient()\n    b.cz(L, D)\n    coefficient()\n    b.free(L)\n    b.free(m)\n    dm = b.alloc_bits(6)\n    b.hmr(D, dm)\n    b.free(D)\n    xs.phase_inverse6_quad4(b, u, dm)\n    b.free(dm)\n    ar._erase_norm(b, t, u, K)\n    b.free(u)\n    linear_inplace(b, t, frm)\n\n\ndef inverse24_reverse_both_norms(b, x, mask, K=None):\n    if b.num_allocated_qubits + 48 > 889:\n        return fallback(b, x, mask, K)\n    cols, back, _ = extension_columns(K)\n    linear_inplace(b, x, cols)\n    to, frm, K24 = tower(24)\n    linear_inplace(b, x, to)\n    N = b.alloc_qubits(12)\n    norm_xor_noscratch(b, x, N, K24)\n    t, u, D = inverse12_keep_both_norms(b, N)\n    out = b.alloc_qubits(12)\n    mul_xor_noscratch(b, x[12:], t, out)\n    erase_product(b, out, N, x[12:])\n    mul_xor_noscratch(b, x[:12], t, x[12:])\n    erase_product(b, x[12:], N, x[:12])\n    b.cx(out, x[12:])\n    b.swap(x[:12], x[12:])\n    b.swap(x[12:], out)\n    b.free(out)\n    erase_norm_with_both(b, t, N, u, D)\n    ar._erase_norm(b, x, t, K24)\n    b.free(t)\n    linear_inplace(b, x, frm)\n    b.cz(x, mask)\n    b.free(mask)\n    linear_inplace(b, x, back)\n\n\nfallback = joint.inverse24_reverse209\nreplacement = xs.private_rebind(inverse24_reverse_both_norms, combo.synth.replacements)\n\n\ndef install(entry):\n    replacements = replacement_ids(entry, joint.inverse24_reverse209, replacement)\n    assert replacements\n    scope = dict(entry.__globals__)\n    for name in (\'extraction\', \'synthesize\'):\n        scope[name] = xs.private_rebind(scope[name], replacements)\n    return xs.clone(entry, scope)\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_local_norm_partial.py': '"""Use partial nested norms at additional local inverse-reverse sites."""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom decode_local_norm import *\nfrom quantum_phase6 import phase_quantum_character\n\n\ndef inverse12_keep_partial_norm(b, x, keep):\n    to, frm, K = tower(12)\n    linear_inplace(b, x, to)\n    D = b.alloc_qubits(6)\n    norm_xor_noscratch(b, x, D, K)\n    u = inverse6_init19(b, D)\n    if keep < 6:\n        dm = b.alloc_bits(6)\n        b.bit_store(dm, False)\n        b.hmr(D[keep:], dm[keep:])\n        b.free(D[keep:])\n        ar._phase_norm(b, x, dm, K)\n        b.free(dm)\n    t = b.alloc_qubits(12)\n    b.cx(x[6:], x[:6])\n    mul_xor_noscratch(b, x[:6], u, t[:6])\n    mul_xor_noscratch(b, x[6:], u, t[6:])\n    b.cx(x[6:], x[:6])\n    linear_inplace(b, x, frm)\n    linear_inplace(b, t, frm)\n    return t, u, D[:keep]\n\n\ndef erase_norm_with_partial(b, t, N, u, D):\n    to, frm, K = tower(12)\n    linear_inplace(b, t, to)\n    linear_inplace(b, N, to)\n    m = b.alloc_bits(12)\n    b.hmr(N, m)\n    b.free(N)\n    L = b.alloc_qubits(1)[0]\n    f = field(6)\n    def coefficient(j):\n        for i in range(6):\n            for r in bits(f.mul(1 << i, 1 << j)):\n                b.ccx(m[r], t[i], L)\n                b.ccx(m[r], t[i+6], L)\n                b.ccx(m[r+6], t[i+6], L)\n    for j in range(6):\n        coefficient(j)\n        if j < len(D):\n            b.cz(L, D[j])\n        else:\n            phase_quantum_character(b, u, L, 1 << j)\n        coefficient(j)\n    b.free(L)\n    b.free(m)\n    dm = b.alloc_bits(6)\n    b.bit_store(dm, False)\n    b.hmr(D, dm[:len(D)])\n    b.free(D)\n    xs.phase_inverse6_quad4(b, u, dm)\n    b.free(dm)\n    ar._erase_norm(b, t, u, K)\n    b.free(u)\n    linear_inplace(b, t, frm)\n\n\ndef inverse24_reverse_partial(b, x, mask, K=None):\n    keep = min(6, 889 - b.num_allocated_qubits - 42)\n    if keep < 4:\n        return fallback(b, x, mask, K)\n    cols, back, _ = extension_columns(K)\n    linear_inplace(b, x, cols)\n    to, frm, K24 = tower(24)\n    linear_inplace(b, x, to)\n    N = b.alloc_qubits(12)\n    norm_xor_noscratch(b, x, N, K24)\n    t, u, D = inverse12_keep_partial_norm(b, N, keep)\n    out = b.alloc_qubits(12)\n    mul_xor_noscratch(b, x[12:], t, out)\n    erase_product(b, out, N, x[12:])\n    mul_xor_noscratch(b, x[:12], t, x[12:])\n    erase_product(b, x[12:], N, x[:12])\n    b.cx(out, x[12:])\n    b.swap(x[:12], x[12:])\n    b.swap(x[12:], out)\n    b.free(out)\n    erase_norm_with_partial(b, t, N, u, D)\n    ar._erase_norm(b, x, t, K24)\n    b.free(t)\n    linear_inplace(b, x, frm)\n    b.cz(x, mask)\n    b.free(mask)\n    linear_inplace(b, x, back)\n\n\nreplacement = xs.private_rebind(inverse24_reverse_partial, combo.synth.replacements)\n\n\ndef install(entry):\n    replacements = replacement_ids(entry, joint.inverse24_reverse209, replacement)\n    assert replacements\n    scope = dict(entry.__globals__)\n    scope[\'extraction\'] = xs.private_rebind(scope[\'extraction\'], replacements)\n    return xs.clone(entry, scope)\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_quantum_inverse_phase.py': '"""Exact W14 inverse characters with a quantum coefficient.\n\nThe specialized caller has L=0 whenever u=0, since u=Norm(t) and L is\nlinear in t.  That identity removes the zero-output product without\nassuming that an independently supplied field element is nonzero.\n"""\nfrom pathlib import Path\nimport sys\nHERE = Path(__file__).resolve().parent\nROOT = HERE.parents[2]\nsys.path[:0] = [str(ROOT/\'exp/r-main/narrow240\'), str(HERE.parent/\'a03\')]\nimport decode_combo as combo\nimport exact_subs as xs\nfrom jet_norm6 import replacement_ids\nfrom exp.jet24 import pro_jet as jet\nfrom exp.inverse12_escape import joint\n\n\ndef phase_inverse6_quantum(b, x, coefficient, annihilates_zero=False):\n    """Pay coefficient dot inv0(x); optionally use coefficient(0)=0."""\n    dag = jet.W14_JOINT18\n    xs.linear_inplace(b, x, jet.TO6)\n    raw = b.alloc_qubits(6)\n    for j, (left, right) in enumerate(dag[\'raw_gates\']):\n        with xs.expose_pair(b, x, left, right) as (a, c):\n            b.init_and(a, c, raw[j])\n    prefix = list(x) + list(raw)\n    forms = xs.ref.norm_forms(1)\n    inner = xs.ref.inverse3_virtual(b, prefix, forms)\n    source = prefix + list(inner)\n    transpose = [sum(((jet.FROM6[i] >> j) & 1) << i for i in range(6))\n                 for j in range(6)]\n    xs.linear_inplace(b, coefficient, transpose)\n    for j, expression in enumerate(dag[\'outputs\'][:6]):\n        linear = expression & 65535\n        if annihilates_zero and (expression >> 16) & 1:\n            linear ^= 1555\n        if linear & 1:\n            b.z(coefficient[j])\n        for i in xs.bits(linear >> 1):\n            b.cz(source[i], coefficient[j])\n    start = 1 if annihilates_zero else 0\n    for i in range(start, len(dag[\'gates\'])):\n        vector = sum(((expression >> (16+i)) & 1) << j\n                     for j, expression in enumerate(dag[\'outputs\'][:6]))\n        indices = list(xs.bits(vector))\n        if not indices:\n            continue\n        pivot = indices[0]\n        for j in indices[1:]:\n            b.cx(coefficient[j], coefficient[pivot])\n        with xs.expose_pair(b, source, *dag[\'gates\'][i]) as (a, c):\n            b.ccz(a, c, coefficient[pivot])\n        for j in reversed(indices[1:]):\n            b.cx(coefficient[j], coefficient[pivot])\n    mask = b.alloc_bits(3)\n    b.hmr(inner, mask)\n    b.free(inner)\n    xs.ref.phase_inverse3_virtual(b, prefix, forms, mask)\n    b.free(mask)\n    for j, (left, right) in reversed(list(enumerate(dag[\'raw_gates\']))):\n        with xs.expose_pair(b, x, left, right) as (a, c):\n            b.del_and(a, c, raw[j])\n    b.free(raw)\n    xs.linear_inplace(b, coefficient, xs.inverse_columns(transpose))\n    xs.linear_inplace(b, x, jet.FROM6)\n\n\ndef phase_inverse6_quantum17(b, x, coefficient):\n    return phase_inverse6_quantum(b, x, coefficient, True)\n\n\ndef install(entry):\n    old = joint.erase_norm_via_inverse_cache19\n    specialized = xs.clone(old, dict(old.__globals__,\n                                     phase_inverse6_quantum19=phase_inverse6_quantum17))\n    specialized = xs.private_rebind(specialized, combo.synth.replacements)\n    replacements = replacement_ids(entry, old, specialized)\n    assert replacements\n    scope = dict(entry.__globals__)\n    for name in (\'extraction\', \'synthesize\'):\n        scope[name] = xs.private_rebind(scope[name], replacements)\n    return xs.clone(entry, scope)\n\n\ndecode = install(combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_shared_correction.py': '"""Share the masked inverse across the two synthesis tail corrections."""\nfrom pathlib import Path\nimport sys\n\nROOT = Path(__file__).resolve().parents[3]\nsys.path.insert(0, str(ROOT / \'exp/r-main/narrow240\'))\nimport decode_combo as base\n\n\ndef term_correction_with_inverse(b, p, q, d, u, v, pole, qi,\n                                 reverse=False, token=None, Z=None):\n    if reverse:\n        Z = guarded.fresh_zero_flag(b, d)\n        b.cz(Z, token[0])\n        b.free(token)\n    F = b.alloc_qubits(1)[0]\n    b.cx(Z, F)\n    b.ccx(Z, pole, F)\n    factor = b.alloc_qubits(12)\n    b.init_and(F, qi, factor)\n    if reverse:\n        ar.mul_xor0(b, v, factor, u)\n        ar.mul_xor0(b, p, factor, v)\n    else:\n        ar.mul_xor0(b, p, factor, v)\n        ar.mul_xor0(b, v, factor, u)\n    b.del_and(F, qi, factor)\n    b.free(factor)\n    b.ccx(Z, pole, F)\n    b.cx(Z, F)\n    b.free(F)\n    if reverse:\n        return Z\n    zm = b.alloc_bits(1)\n    b.hmr(Z, zm[0])\n    b.free(Z)\n    return zm\n\n\ndef install(entry):\n    synth = entry.__globals__[\'synthesize\']\n    tail = synth.__globals__[\'tail_inplace\']\n    finish = tail.__globals__[\'tail_finish\']\n    old = finish.__globals__[\'term_correction_with_inverse\']\n    new = base.xsub.clone(term_correction_with_inverse, dict(old.__globals__))\n    return base.xsub.private_rebind(entry, {id(old): new})\n\n\ndecode = install(base.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_shared_correction_front.py': '"""Install the shared tail factor only in the synthesis graph."""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom decode_shared_correction import term_correction_with_inverse, base\n\n\ndef install(entry):\n    synth = entry.__globals__[\'synthesize\']\n    tail = synth.__globals__[\'tail_inplace\']\n    finish = tail.__globals__[\'tail_finish\']\n    old = finish.__globals__[\'term_correction_with_inverse\']\n    new = base.xsub.clone(term_correction_with_inverse, dict(old.__globals__))\n    scope = dict(entry.__globals__)\n    scope[\'synthesize\'] = base.xsub.private_rebind(synth, {id(old): new})\n    return base.xsub.clone(entry, scope)\n\n\ndecode = install(base.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_stream12_cached.py': '"""Use flag-free cached twelve-bit inverses in streamed scalar calls."""\nfrom pathlib import Path\nimport sys\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE), str(HERE.parent/\'a04\')]\nimport decode_flagfree_cached as cache\nfrom nonzero_consume16 import functions\n\n\ndef stream_wrapper(original):\n    def stream(b, x):\n        space = 889-b.num_allocated_qubits\n        if len(x) != 12 or space < 19:\n            return original(b, x)\n        out, flag = cache.CACHED12[space < 22](b, x, cache.jet.PLAN12, False)\n        assert flag is None\n        transcript = b.alloc_bits(12)\n        b.hmr(x, transcript)\n        b.swap(x, out)\n        b.free(out)\n        return transcript\n    return stream\n\n\ndef install(entry):\n    names = {\'inverse_stream_noscratch\', \'inverse_stream_narrow\', \'stream_nonzero\', \'chosen\'}\n    replacements = {id(fn): stream_wrapper(fn) for fn in functions(entry)\n                    if fn.__name__ in names}\n    assert replacements\n    scope = dict(entry.__globals__)\n    for name in (\'extraction\', \'synthesize\'):\n        scope[name] = cache.xs.private_rebind(scope[name], replacements)\n    return cache.xs.clone(entry, scope)\n\n\ndecode = install(cache.combo.decode)\n', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/quantum_phase6.py': '"""One fixed inverse-field character with a quantum coefficient."""\nfrom pathlib import Path\nimport json\nimport sys\nHERE = Path(__file__).resolve().parent\nKIT_ROOT = HERE.parents[2]\nsys.path.insert(0, str(KIT_ROOT / \'exp/r-main/narrow240\'))\nimport exact_subs as xs\n\n\ndef quadratic_decomposition(terms):\n    rows = [0] * 10\n    linear = 0\n    constant = 0\n    for term in terms:\n        if term[0] == \'constant\':\n            constant ^= 1\n        elif term[0] == \'linear\':\n            linear ^= 1 << term[1]\n        else:\n            a, c = term[1:]\n            rows[a] ^= 1 << c\n            rows[c] ^= 1 << a\n    pairs = []\n    while any(rows):\n        a = next(i for i, row in enumerate(rows) if row)\n        c = next(xs.bits(rows[a]))\n        left, right = rows[a], rows[c]\n        pairs.append((left, right))\n        linear ^= left & right\n        for i in range(10):\n            if left >> i & 1:\n                rows[i] ^= right\n            if right >> i & 1:\n                rows[i] ^= left\n    return constant, linear, tuple(pairs)\n\n\n_record = json.loads((HERE / \'search_quantum_phase6.json\').read_text())\nCONSTANT, LINEAR, PAIRS = quadratic_decomposition(_record[\'terms\'])\nassert len(PAIRS) == 2\n\n\ndef phase_quantum_character(b, x, coefficient, mask):\n    """Pay coefficient * dot(mask, inverse0(x)); preserve both inputs."""\n    if not mask:\n        return\n    xs.linear_inplace(b, x, xs.SCALES[mask])\n    raw = b.alloc_qubits(4)\n    source = list(x) + list(raw)\n    for i, (left, right) in enumerate(xs.FORMS):\n        with xs.expose_pair(b, source[:6+i], left, right) as (a, c):\n            b.init_and(a, c, raw[i])\n    if CONSTANT:\n        b.z(coefficient)\n    for i in xs.bits(LINEAR):\n        b.cz(source[i], coefficient)\n    for left, right in PAIRS:\n        with xs.expose_pair(b, source, left << 1, right << 1) as (a, c):\n            b.ccz(a, c, coefficient)\n    for i in reversed(range(4)):\n        with xs.expose_pair(b, source[:6+i], *xs.FORMS[i]) as (a, c):\n            b.del_and(a, c, raw[i])\n    b.free(raw)\n    xs.linear_inplace(b, x, xs.UNSCALES[mask])\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/adaptive_zero.py': '"""Exact partitioned zero predicates chosen at the current live width."""\nfrom functools import lru_cache\nimport frozen_chart as frozen\n\nTRACE=[]\n\ndef partitions(n,lo=1):\n    if not n:\n        yield ()\n    for a in range(lo,n+1):\n        for rest in partitions(n-a,a):yield (a,)+rest\n\ndef metrics(parts,phase):\n    groups=sorted((n for n in parts if n>1),reverse=True)\n    k=len(groups);m=len(parts)\n    if m<2:return None\n    peak=0\n    for i,n in enumerate(groups):peak=max(peak,i+n-1)\n    peak=max(peak,k+m-(2 if phase else 1))\n    # Aggregate ladder has gone. Erase small groups first, big groups last.\n    for i,n in enumerate(reversed(groups)):\n        if n>2:peak=max(peak,k-i-1+(0 if phase else 1)+n-2)\n    mean=sum(n-1 for n in groups)+m-(2 if phase else 1)+sum(n-2 for n in groups)/2\n    return mean,peak\n\n@lru_cache(None)\ndef plan(n,space,phase):\n    options=[]\n    for p in partitions(n):\n        z=metrics(p,phase)\n        if z is not None and z[1]<=space:options.append((z[0],z[1],p))\n    if not options:raise ValueError((\'no zero partition\',n,space,phase))\n    return min(options)\n\ndef phase_and(b,x):\n    if len(x)==2:b.cz(x[0],x[1]);return\n    nodes=b.alloc_qubits(len(x)-2)\n    b.init_and(x[0],x[1],nodes[0])\n    for j in range(1,len(nodes)):b.init_and(nodes[j-1],x[j+1],nodes[j])\n    b.cz(nodes[-1],x[-1])\n    for j in reversed(range(len(nodes))):\n        if j:b.del_and(nodes[j-1],x[j+1],nodes[j])\n        else:b.del_and(x[0],x[1],nodes[0])\n    b.free(nodes)\n\ndef predicate(b,x,phase):\n    live=b.num_allocated_qubits\n    mean,peak,parts=plan(len(x),889-live,phase)\n    TRACE.append(dict(n=len(x),phase=phase,live=live,parts=parts,mean=mean,scratch=peak))\n    b.x(x);groups=[];controls=[];offset=0\n    for n in sorted(parts,reverse=True):\n        group=x[offset:offset+n];offset+=n\n        if n==1:controls.append(group[0])\n        else:\n            flag=frozen.and_flag(b,group)\n            groups.append((group,flag));controls.append(flag)\n    if phase:phase_and(b,controls);out=None\n    else:out=frozen.and_flag(b,controls)\n    for group,flag in reversed(groups):\n        if len(group)==2:b.del_and(group[0],group[1],flag);b.free(flag)\n        else:\n            mask=b.alloc_bits(1);b.hmr(flag,mask[0]);b.free(flag)\n            b.push_condition(mask[0]);phase_and(b,group);b.pop_condition();b.free(mask)\n    b.x(x)\n    return out\n\ndef zero_flag24(b,x):return predicate(b,x,False)\ndef clean_zero_phase24(b,x):predicate(b,x,True)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/chart_retimed.py': '"""Frozen b-width chart with its complete paired character replaced only."""\nimport frozen_chart as frozen\nfrom frozen_chart import *\nfrom core90 import phase_core90\nfrom retimed import phase_rational_retimed240\n\n\ndef pay_charts(b, D, forward, reverse):\n    A, g = D[:12], D[12:]\n    F = ref.guarded.fresh_zero_flag(b, g)\n    b.cz(F, forward[\'f\'][0])\n    b.cz(F, reverse[\'f\'][0])\n    b.free(forward[\'f\'])\n    b.free(reverse[\'f\'])\n    b.z(reverse[\'r\'][0])\n    b.cz(F, reverse[\'r\'][0])\n    masks = dict(g=add_masks(b, forward[\'g\'], reverse[\'t\']),\n                 i=add_masks(b, forward[\'i\'], reverse[\'h\']),\n                 h=add_masks(b, forward[\'h\'], reverse[\'i\']),\n                 t=add_masks(b, forward[\'t\'], reverse[\'g\']),\n                 r=add_masks(b, forward[\'r\'], reverse[\'r\']))\n    b.cswap(F, A, g)\n    with b.mark(\'direct_core90\'):\n        phase_core90(b, A, g, F, masks, extra_F_mask=masks[\'t\'])\n    for key in (\'g\', \'i\', \'h\', \'r\'):\n        b.free(masks[key])\n    b.cswap(F, A, g)\n    ref.guarded.zero_toggle(b, g, F)\n    b.free(F)\n    with b.mark(\'direct_residual240\'):\n        phase_rational_retimed240(b, A, g, masks[\'t\'])\n    b.free(masks[\'t\'])\n\n\nexternal_node_chain = clone(frozen.external_node_chain,\n                            dict(frozen.external_node_chain.__globals__, pay_charts=pay_charts))\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/converter.py': '"""Private converter graph, with auditable cost categories.\n\n`route=True` is the baseline\'s complete gap1/2 converter. `route=False`\nremoves all gap movement and parent correction for a generic-chart control;\nit is NOT a candidate. No stable module globals are mutated.\n"""\nfrom pathlib import Path\nimport sys\nfrom types import FunctionType,SimpleNamespace\nfrom contextlib import contextmanager\nsys.path.insert(0,str(Path(__file__).resolve().parents[3]/\'playbook/kit/decoder\'))\nfrom final_decoder import conversion as stable\n\n\ndef clone(fn,scope):\n out=FunctionType(fn.__code__,scope,fn.__name__,fn.__defaults__,fn.__closure__)\n out.__kwdefaults__=fn.__kwdefaults__;return out\n\ndef marked(fn,category):\n def call(b,*args,**kw):\n  with b.mark(category):return fn(b,*args,**kw)\n return call\n\ndef plain_q_post(b,q,payloads,H,m,parent,reverse=False,phase=None):\n with q_inverse(b,q,m,reverse,phase) as qi:\n  for v in payloads:scale(b,v[-1],q,qi,not reverse)\n\ndef routed_q_post(b,q,payloads,H,m,parent,reverse=False,phase=None):\n if parent is None:\n  with q_inverse(b,q,m,reverse,phase) as qi:\n   for v in payloads:scale(b,v[-1],q,qi,not reverse)\n  return\n tops,p,c=parent\n if reverse or m+1 in KEEP_CHILD:F=phase[\'given_gap\'].pop(m+1)\n else:\n  F=fresh_zero_flag(b,q);consume_flag(b,F,phase[\'incoming_gap\'].pop(m+1))\n with b.mark(\'gap_routing\'):b.cswap(F,q,p)\n if not reverse:\n  with b.mark(\'gap_routing\'):\n   for v,t in zip(payloads,tops):b.cswap(F,v[-1],t)\n with q_inverse(b,q,m,reverse,phase) as qi:\n  if not reverse:\n   for v in payloads:scale(b,v[-1],q,qi,True)\n   for v,t in zip(payloads,tops):masked_product(b,F,v[-1],c,t)\n  else:\n   for v,t in zip(payloads,tops):masked_product(b,F,v[-1],c,t)\n   for v in payloads:scale(b,v[-1],q,qi,False)\n if reverse:\n  with b.mark(\'gap_routing\'):\n   for v,t in zip(payloads,tops):b.cswap(F,v[-1],t)\n with b.mark(\'gap_routing\'):b.cswap(F,q,p)\n if reverse and m+1 not in KEEP_CHILD:phase[\'return_mask\'][m+1]=hold_flag(b,F)\n else:phase[\'returned_gap\'][m+1]=F\n\ndef plain_look_solve(b,f,g,payloads,H=None,parent=None,reverse=False,phase=None):\n m=len(f)-1\n if m==1:return base_case(b,f,g,payloads,H,parent,reverse,phase)\n if not reverse:carry=pre(b,f,g,payloads,defer=True)\n else:carry=post_look(b,f,g,payloads,H,parent,True,incoming=phase[\'a\'].pop(m),defer=True,phase=phase)\n childparent=([v[-1]for v in payloads],f[-1],f[-2])\n look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,childparent,reverse,phase)\n if not reverse:phase[\'a\'][m]=post_look(b,f,g,payloads,H,parent,incoming=carry,defer=True,phase=phase)\n else:pre(b,f,g,payloads,True,incoming=carry)\n\ndef plain_root(b,O,g,payloads,reverse,phase):\n a=g[-1];ai,Z=old.inverse_zero(b,a);b.cx(Z,ai[0])\n if reverse:\n  mask=phase.pop(\'_root_inverse_mask\');b.cz(ai,mask);b.free(mask)\n f=list(O)+[ai]\n cached_root_action(b,f,g,payloads,ai,Z,post=reverse,undo=reverse)\n erase_product_zero(b,a,ai,Z)\n parent=([v[-1]for v in payloads],f[-1],f[-2])\n old.look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,parent,reverse,phase)\n Z=old.fresh_zero_flag(b,a)\n cached_root_action(b,f,g,payloads,ai,Z,post=not reverse,undo=reverse)\n erase_product_zero(b,a,ai,Z)\n mask=b.alloc_bits(12);b.hmr(ai,mask);b.free(ai)\n if reverse:phase_inverse34(b,a,mask,zero_mask=mask[0]);b.free(mask)\n else:phase[\'_root_inverse_mask\']=mask\n\n\ndef make_converter(route=True):\n scope=dict(stable.scope)\n targets=[]\n def head_wrapper(fn,post=False):\n  def call(b,f,g,payloads,*args,**kw):\n   if post:rev=kw.get(\'reverse\',args[2] if len(args)>2 else False)\n   else:rev=kw.get(\'undo\',args[0] if args else False)\n   targets.append((set(str(q) for w in f[-2:] for q in w),\n                   \'head_restore\' if post!=bool(rev) else \'head_prepare\'))\n   try:return fn(b,f,g,payloads,*args,**kw)\n   finally:targets.pop()\n  return call\n def categorized_ops(b,ops,a,ai,undo=False):\n  for op in (reversed(ops) if undo else ops):\n   dest=op[1] if op[0]==\'s\' else op[3]\n   if op[0]==\'c\':dest=[q for w in dest for q in w]\n   head=bool(targets and any(str(q) in targets[-1][0] for q in dest))\n   category=targets[-1][1] if head else \'raw_arithmetic\'\n   if head and op[0]==\'s\':category=\'head_prepare\' if op[2]^undo else \'head_restore\'\n   with b.mark(category):\n    if op[0]==\'s\':scope[\'scale\'](b,op[1],a,ai,op[2]^undo)\n    elif op[0]==\'m\':scope[\'ar\'].mul_xor0(b,op[1],op[2],op[3])\n    elif op[0]==\'c\':scope[\'convolution_xor\'](b,op[1],op[2],op[3],undo)\n    else:raise ValueError(op[0])\n invscope=dict(scope[\'inverse_apply_safe\'].__globals__)\n erasescope=dict(invscope[\'erase\'].__globals__)\n erasescope[\'phase_inverse36_zero\']=marked(stable.phase_inverse34,\'inverse_phase\')\n invscope[\'erase\']=clone(invscope[\'erase\'],erasescope)\n invscope[\'inverse_zero\']=marked(invscope[\'inverse_zero\'],\'inverse_compute\')\n scope[\'inverse_apply_safe\']=clone(scope[\'inverse_apply_safe\'],invscope)\n scope[\'inverse_zero\']=marked(scope[\'inverse_zero\'],\'inverse_compute\')\n scope[\'erase\']=invscope[\'erase\']\n # Rebuild call graph so categories are assigned at the innermost operation.\n for name in [\'scale\',\'runops\',\'zero_toggle\',\'inverse_apply\',\'pre\',\'canonical_permute\',\n              \'masked_product\',\'fresh_zero_flag\',\'post_look\',\'base\',\'bez_lookahead\']:\n  scope[name]=clone(scope[name],scope)\n scope[\'runops\']=categorized_ops\n scope[\'pre\']=head_wrapper(scope[\'pre\'])\n scope[\'post_look\']=head_wrapper(scope[\'post_look\'],True)\n scope[\'base\']=head_wrapper(scope[\'base\'],True)\n scope[\'base_case\']=scope[\'base\']\n for name,category in [(\'scale\',\'raw_scale\'),(\'canonical_permute\',\'gap_routing\'),\n                       (\'rotate\',\'gap_routing\'),(\'masked_product\',\'parent_correction\'),\n                       (\'zero_toggle\',\'flags\'),(\'clear_flag\',\'flags\'),\n                       (\'hold_flag\',\'flag_measure\'),(\'consume_flag\',\'flag_phase\')]:\n  fn=scope[name]\n  if hasattr(fn,\'__code__\') and fn.__module__==stable.__name__:fn=clone(fn,scope)\n  scope[name]=marked(fn,category)\n scope[\'q_inverse\']=contextmanager(clone(stable.scope[\'q_inverse\'].__wrapped__,scope))\n scope[\'q_post\']=clone(routed_q_post if route else plain_q_post,scope)\n scope[\'look_solve\']=clone(stable.look_solve if route else plain_look_solve,scope)\n short=dict(stable.short,old=SimpleNamespace(**scope),\n            clear_flag=scope[\'clear_flag\'],hold_flag=scope[\'hold_flag\'],\n            consume_flag=scope[\'consume_flag\'],\n            phase_inverse34=marked(stable.phase_inverse34,\'inverse_phase\'))\n for name,fn in [(\'cached_root_action\',stable.cached_root_action),\n                 (\'look_root_cached\',stable.look_root_cached if route else plain_root),\n                 (\'_bez_monic\',stable._bez_monic)]:\n  short[name]=clone(fn,short)\n rootaction=short[\'cached_root_action\']\n def root_heads(b,f,g,payloads,ai,Z,post,undo):\n  targets.append((set(str(q) for w in f[-2:] for q in w),\n                  \'head_restore\' if post!=bool(undo) else \'head_prepare\'))\n  try:return rootaction(b,f,g,payloads,ai,Z,post,undo)\n  finally:targets.pop()\n short[\'cached_root_action\']=root_heads\n core=clone(stable.base._short_scope[\'convert_safe\'],short)\n def convert_safe(b,A,B,reverse=False,history_scratch12=None,phase_token=None,*,monic_root=True):\n  if phase_token is None:\n   phase_token={\'a\':{},\'q\':{},\'_short_monic\':bool(monic_root)}\n   for key in [\'gap_flags\',\'given_gap\',\'incoming_gap\',\'return_mask\',\'returned_gap\',\'held_gap\']:phase_token[key]={}\n  result=core(b,A,B,reverse,history_scratch12,phase_token,monic_root=monic_root)\n  return result\n return convert_safe\n\nconvert_safe=make_converter(True)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/core90.py': '"""Four chart characters as one inverse character with a virtual affine mask.\n\nUnlike materializing the mask bank, a classical first-nonzero pivot exposes\neach requested affine parity directly in A. Preserves A, g, F, and masks.\n"""\nfrom rational import *\nfrom phase_quantum import phase_inverse6_9\nfrom exp.noscratch.schedules import norm_xor_noscratch, inverse_stream_noscratch\nfrom exp.mul99.phase36 import phase_classical_parity\nimport norm_cache\n\n\ndef affine_inverse_g_character(b, A, g, masks, extra_F_mask=None, F=None):\n    """Pay (I + R*A + H*A^2)/g as binary-dot characters, 90T/12fresh."""\n    to, frm, kval = ar.tower(12)\n    ar.linear_inplace(b, g, to)\n    D = b.alloc_qubits(6)\n    norm_xor_noscratch(b, g, D, kval)\n    saved = norm_cache.keep(b, D)\n    dm = inverse_stream_noscratch(b, D)\n    ar._phase_norm(b, g, dm, kval)\n    b.free(dm)\n    f = ar.field(12)\n    b.cx(g[6:], g[:6])\n    for seg in (0, 1):\n        x = g[:6] if seg == 0 else g[6:]\n        for sm, vv in ar.tensor(6):\n            h = ar.apply(frm, vv << (6 * seg))\n            co = b.alloc_bits(12 + (extra_F_mask is not None))\n            offset = b.alloc_bits(1)[0]\n            b.bit_store(co, False)\n            b.bit_store(offset, False)\n            for j in ar.bits(h):\n                b.cx(masks[\'i\'][j], offset)\n            for i in range(12):\n                for j in ar.bits(f.mul(1 << i, h)):\n                    b.cx(masks[\'r\'][j], co[i])\n                for j in ar.bits(f.mul(f.square(1 << i), h)):\n                    b.cx(masks[\'h\'][j], co[i])\n            if extra_F_mask is not None:\n                for j in ar.bits(h):\n                    b.cx(extra_F_mask[j], co[12])\n            ix = list(ar.bits(sm))\n            p = ix[0]\n            for i in ix[1:]:\n                b.cx(x[i], x[p])\n                b.cx(D[i], D[p])\n            b.ccz(x[p], D[p], offset)\n            source = list(A) + ([F] if extra_F_mask is not None else [])\n            phase_classical_parity(b, x[p], D[p], source, co)\n            for i in reversed(ix[1:]):\n                b.cx(D[i], D[p])\n                b.cx(x[i], x[p])\n            b.free([offset])\n            b.free(co)\n    b.cx(g[6:], g[:6])\n    im = b.alloc_bits(6)\n    b.hmr(D, im)\n    b.free(D)\n    D = norm_cache.rebuild(b, g, saved, kval)\n    phase_inverse6_9(b, D, im)\n    b.free(im)\n    ar._erase_norm(b, g, D, kval)\n    b.free(D)\n    ar.linear_inplace(b, g, frm)\n\n\ndef phase_core90(b, A, g, F, masks, extra_F_mask=None):\n    b.cz(g, masks[\'g\'])\n    b.cz(A, masks[\'h\'])\n    f = ar.field(12)\n    for j in range(12):\n        for k in ar.bits(f.mul(K, 1 << j)):\n            b.cz(g[j], masks[\'h\'][k])\n        for k in ar.bits(f.mul(K ^ 1, 1 << j)):\n            b.ccz(F, g[j], masks[\'h\'][k])\n    affine_inverse_g_character(b, A, g, masks, extra_F_mask, F)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/decode_combo.py': '"""Exact synthesis retention plus width-safe retained reverse chart v5."""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport decode_reverse as synth\nimport decode_subs_combined as kept\nimport retained_core_safe27 as retained\n\nbase, xsub = kept.base, kept.xsub\nnode = xsub.clone(retained.external_node_chain,\n                 dict(retained.external_node_chain.__globals__,\n                      terminal_two_phase=retained.full_core_phase))\nnode = kept.edited(node, \'D=b.alloc_qubits(24);b.x(D[0]);b.cswap(pole,N,D)\',\n                   \'D=b.alloc_qubits(24);b.x(D[0]);b.cx(pole,N[0]);b.cx(pole,D[0])\')\nreplacements = dict(synth.replacements)\nreplacements[id(base.chart_chain)] = xsub.private_rebind(node, replacements)\ndecode = xsub.private_rebind(base.decode, replacements)\ntrailing = kept.edited(decode.__globals__[\'extraction\'],\n                      \'D=b.alloc_qubits(12);b.x(D[0]);b.cswap(pole,N,D)\',\n                      \'D=b.alloc_qubits(12);b.x(D[0]);b.cx(pole,N[0]);b.cx(pole,D[0])\')\ndecode = xsub.clone(decode, dict(decode.__globals__, extraction=trailing))\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/decode_retimed.py': "from pathlib import Path\nimport sys\nfrom types import SimpleNamespace\nimport inspect\nhere=Path(__file__).resolve().parent;root=here.parents[2]\nsys.path[:0]=[str(here),str(here),str(root/'playbook/kit/decoder'),str(root/'playbook/kit')]\nfrom exp.inverse19_joint.kernel import inverse24_reverse19\nfrom final_decoder import decode as ref\nfrom high_stream import convert_safe\nimport replay_erase\nimport chart_retimed as chart\nfrom synth_stream import make_synthesize\nclone=chart.clone\nsynthesize,_divide=make_synthesize(ref)\nchart_chain=clone(chart.external_node_chain,dict(chart.external_node_chain.__globals__,scalar_erase_second24=replay_erase.scalar_erase_second24))\ndef external_node_chain(b,words,gi,reverse=False,token=None,normalizer=None):\n if gi==8:return replay_erase.external_node_chain(b,words,gi,reverse,token,normalizer)\n return chart_chain(b,words,gi,reverse,token,normalizer)\next_source=inspect.getsource(ref.external_extraction)\nold='inverse24_reverse209(b,v,local,K)'\nassert ext_source.count(old)==1\next_source=ext_source.replace(old,'(inverse24_reverse19 if gi>=15 else inverse24_reverse209)(b,v,local,K)')\next_env=dict(ref.external_extraction.__globals__,external_node_chain=external_node_chain,inverse24_reverse19=inverse24_reverse19)\nexec(compile(ext_source,__file__,'exec'),ext_env)\next=ext_env['external_extraction']\nfwd=clone(ref.extraction,dict(ref.extraction.__globals__,external_extraction=ext))\nproxy=SimpleNamespace(**dict(vars(ref),extraction=fwd))\nextraction=clone(replay_erase.extraction,dict(replay_erase.extraction.__globals__,ref=proxy,external_extraction=ext))\ndecode=clone(ref.decode,dict(ref.decode.__globals__,convert_safe=convert_safe,extraction=extraction,synthesize=synthesize))\n", 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/decode_reverse.py': '"""Accepted port of the exact substitutions plus exact synthesis low products."""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport decode_subs_combined as base\nimport reverse_exact as exact\nfrom exp.fold_joint import kernel as fold\nreplacements=dict(base.replacements)\nreplacements[id(fold.low_product)]=exact.low_product\ndecode=base.xsub.private_rebind(base.base.decode,replacements)\ntrailing=base.edited(decode.__globals__[\'extraction\'],\n    \'D=b.alloc_qubits(12);b.x(D[0]);b.cswap(pole,N,D)\',\n    \'D=b.alloc_qubits(12);b.x(D[0]);b.cx(pole,N[0]);b.cx(pole,D[0])\')\ndecode=base.xsub.clone(decode,dict(decode.__globals__,extraction=trailing))\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/decode_subs.py': '"""Exact substitutions on the frozen narrow240 decoder (isolated port)."""\nfrom pathlib import Path\nimport sys\nhere = Path(__file__).resolve().parent\nsys.path.insert(0, str(here))\nimport decode_retimed as base\nimport exact_subs as xsub\nfrom exp.arithmetic.lowwidth import phase_inverse_generic6\nfrom phase_quantum import phase_inverse6_9\n\nreplacements = {\n    id(xsub.core.external_chain_step): xsub.chain_step,\n    id(xsub.old6): xsub.phase_inverse6_quad4,\n    id(xsub.old_generic6): xsub.phase_inverse6_quad4,\n    id(phase_inverse6_9): xsub.phase_inverse6_quad4,\n    id(xsub.old12): xsub.phase_inverse12_virtual,\n    id(xsub.zero1.inverse_zero): xsub.inverse_zero,\n    id(xsub.joint.inverse_zero): xsub.inverse_zero,\n    id(xsub.joint.inverse6_init19): xsub.initialized19,\n    id(xsub.iso.inverse6_xor20): xsub.initialized_only(xsub.iso.inverse6_xor20),\n    id(xsub.ref.inverse6_xor20): xsub.initialized_only(xsub.ref.inverse6_xor20),\n}\ndecode = xsub.private_rebind(base.decode, replacements)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/decode_subs_combined.py': '"""Exact substitutions plus retained zero-cost early replay erasure and entry CNOTs."""\nfrom pathlib import Path\nimport sys, inspect\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom decode_subs import *\n\ndef edited(fn, old, new, extra=None):\n    source=inspect.getsource(fn)\n    assert source.count(old)==1, (fn.__name__, source.count(old))\n    env=dict(fn.__globals__)\n    if extra: env.update(extra)\n    exec(compile(source.replace(old,new),fn.__code__.co_filename,\'exec\'),env)\n    return env[fn.__name__]\n\nterminal=edited(xsub.terminal_two_phase,\n    "        external_scalar_step(b, heads[i], N, D, params[i][0], ct, True, first=i == 0)",\n    "        if i == 1: scalar_erase_second24(b, heads[i], N, D, params[i][0], ct, params[0][0])\\n        else: external_scalar_step(b, heads[i], N, D, params[i][0], ct, True, first=i == 0)",\n    dict(scalar_erase_second24=base.replay_erase.scalar_erase_second24))\noriginal_node=xsub.clone(xsub.external_node_chain,\n    dict(xsub.external_node_chain.__globals__,terminal_two_phase=terminal))\nchart_node=edited(base.chart_chain,\n    \'D=b.alloc_qubits(24);b.x(D[0]);b.cswap(pole,N,D)\',\n    \'D=b.alloc_qubits(24);b.x(D[0]);b.cx(pole,N[0]);b.cx(pole,D[0])\')\nreplacements=dict(replacements)\nreplacements.update({\n    id(base.replay_erase.external_node_chain):xsub.private_rebind(original_node,replacements),\n    id(base.chart_chain):xsub.private_rebind(chart_node,replacements),\n})\ndecode=xsub.private_rebind(base.decode,replacements)\ntrailing=edited(decode.__globals__[\'extraction\'],\n    \'D=b.alloc_qubits(12);b.x(D[0]);b.cswap(pole,N,D)\',\n    \'D=b.alloc_qubits(12);b.x(D[0]);b.cx(pole,N[0]);b.cx(pole,D[0])\')\ndecode=xsub.clone(decode,dict(decode.__globals__,extraction=trailing))\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/direct_converter.py': '"""Aligned gap route, with full-product heads on the small remaining levels."""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nimport direct_kernels as dk\nimport measured_heads as mh\nfrom converter import clone\nfrom types import SimpleNamespace\nfrom final_decoder import conversion as stable\nTHRESHOLD=6\n\n\ndef pivot(b,g,F):\n with b.mark(\'head_pivot\'):b.cswap(F,g[-1],g[-2])\n\ndef aligned(b,f,g,F):\n with b.mark(\'aligned_route\'):\n  for x,y in zip(f[:-2],g[:-1]):b.cswap(F,x,y)\n\ndef cross(b,g,q1,q0,F):\n with dk.selected(b,g[-2],g[-3],F) as c:ar.mul_xor0(b,q1,c,q0)\n\n\ndef direct_pre(b,f,g,payloads,F,undo=False,incoming=None,defer=False):\n pivot(b,g,F);a=g[-1];q0,q1=f[-2:]\n def action(ai):\n  if not undo:\n   for v in payloads:scale(b,v[-1],a,ai,True)\n   for v in payloads:dk.shifted_product(b,g[:-1],v[-1],v[:-1],F)\n   scale(b,q1,a,ai,True);cross(b,g,q1,q0,F);scale(b,q0,a,ai,True)\n   dk.shifted_conv(b,g[:-1],[q0,q1],f[:-2],F)\n  else:\n   dk.shifted_conv(b,g[:-1],[q0,q1],f[:-2],F,True)\n   scale(b,q0,a,ai,False);cross(b,g,q1,q0,F);scale(b,q1,a,ai,False)\n   for v in payloads:dk.shifted_product(b,g[:-1],v[-1],v[:-1],F)\n   for v in payloads:scale(b,v[-1],a,ai,False)\n out=inverse_apply(b,a,action,incoming,defer);pivot(b,g,F);return out\n\n\ndef direct_post(b,f,g,payloads,F,H,parent,reverse=False,incoming=None,defer=False,phase=None):\n m=len(f)-1;q0,q1=f[-2:]\n if not reverse:q_post(b,q1,payloads,H,m,parent,phase=phase)\n pivot(b,g,F);a=g[-1]\n def dots():\n  for v in payloads:dk.selected_dot(b,g[:-1],v[:-1],v[-1],F)\n def action(ai):\n  if not reverse:\n   for v in payloads:scale(b,v[-1],a,ai,True)\n   dk.shifted_conv(b,g[:-1],[q0,q1],f[:-2],F)\n   scale(b,q0,a,ai,False);cross(b,g,q1,q0,F);scale(b,q1,a,ai,False)\n  else:\n   scale(b,q1,a,ai,True);cross(b,g,q1,q0,F);scale(b,q0,a,ai,True)\n   dk.shifted_conv(b,g[:-1],[q0,q1],f[:-2],F,True)\n   for v in payloads:scale(b,v[-1],a,ai,False)\n if not reverse:dots()\n out=inverse_apply(b,a,action,incoming,defer)\n if reverse:dots()\n pivot(b,g,F)\n if reverse:q_post(b,q1,payloads,H,m,parent,True,phase)\n return out\n\n\ndef look_solve(b,f,g,payloads,H=None,parent=None,reverse=False,phase=None):\n m=len(f)-1\n if m<THRESHOLD:return old_look_solve(b,f,g,payloads,H,parent,reverse,phase)\n keep=m in KEEP_CHILD;full=m in KEEP_FULL\n F=phase[\'held_gap\'].pop(m) if reverse and full else fresh_zero_flag(b,g[-1])\n if reverse and not full:consume_flag(b,F,phase[\'gap_flags\'].pop(m))\n if not reverse:carry=direct_pre(b,f,g,payloads,F,defer=True)\n else:carry=direct_post(b,f,g,payloads,F,H,parent,True,phase[\'a\'].pop(m),True,phase)\n aligned(b,f,g,F)\n if reverse or keep:phase[\'given_gap\'][m]=F\n else:phase[\'incoming_gap\'][m]=hold_flag(b,F)\n childparent=([v[-1]for v in payloads],f[-1],f[-2])\n look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,childparent,reverse,phase)\n if reverse and not keep:\n  F=fresh_zero_flag(b,g[-1]);consume_flag(b,F,phase[\'return_mask\'].pop(m))\n else:F=phase[\'returned_gap\'].pop(m)\n aligned(b,f,g,F)\n if not reverse:phase[\'a\'][m]=direct_post(b,f,g,payloads,F,H,parent,incoming=carry,defer=True,phase=phase)\n else:direct_pre(b,f,g,payloads,F,True,incoming=carry)\n if reverse:clear_flag(b,g[-1],F)\n elif full:phase[\'held_gap\'][m]=F\n else:phase[\'gap_flags\'][m]=hold_flag(b,F)\n\n\ndef root_action(b,f,g,payloads,ai,Z,F,post,undo):\n a=g[-1];q0,q1=f[-2:]\n if post:\n  ops=[(\'s\',v[-1],False) for v in payloads]+[(\'d\',v) for v in payloads]\n  ops +=[(\'s\',v[-1],True) for v in payloads]+[(\'c\',),(\'s\',q0,False),(\'x\',)]\n else:\n  ops=[(\'s\',v[-1],True) for v in payloads]+[(\'u\',v) for v in payloads]\n  ops +=[(\'x\',),(\'s\',q0,True),(\'c\',)]\n b.cx(Z,a[0])\n for op in (reversed(ops) if undo else ops):\n  if op[0]==\'s\':old.scale(b,op[1],a,ai,op[2]^undo)\n  elif op[0]==\'u\':dk.shifted_product(b,g[:-1],op[1][-1],op[1][:-1],F)\n  elif op[0]==\'d\':dk.selected_dot(b,g[:-1],op[1][:-1],op[1][-1],F)\n  elif op[0]==\'c\':dk.shifted_conv(b,g[:-1],[q0,q1],f[:-2],F,undo)\n  elif op[0]==\'x\':\n   with dk.selected(b,g[-2],g[-3],F) as c:ar.mul_xor0(b,q1,c,q0)\n b.cx(Z,a[0])\n\n\ndef look_root_cached(b,O,g,payloads,reverse,phase):\n m=len(O);keep=m in KEEP_CHILD;full=m in KEEP_FULL\n F=phase[\'held_gap\'].pop(m) if reverse and full else old.fresh_zero_flag(b,g[-1])\n if reverse and not full:consume_flag(b,F,phase[\'gap_flags\'].pop(m))\n pivot(b,g,F);a=g[-1];ai,Z=old.inverse_zero(b,a);b.cx(Z,ai[0])\n if reverse:\n  mask=phase.pop(\'_root_inverse_mask\');b.cz(ai,mask);b.free(mask)\n f=list(O)+[ai]\n root_action(b,f,g,payloads,ai,Z,F,post=reverse,undo=reverse)\n erase_product_zero(b,a,ai,Z);pivot(b,g,F);aligned(b,f,g,F)\n if reverse or keep:phase[\'given_gap\'][m]=F\n else:phase[\'incoming_gap\'][m]=hold_flag(b,F)\n parent=([v[-1]for v in payloads],f[-1],f[-2])\n old.look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,parent,reverse,phase)\n if reverse and not keep:\n  F=old.fresh_zero_flag(b,g[-1]);consume_flag(b,F,phase[\'return_mask\'].pop(m))\n else:F=phase[\'returned_gap\'].pop(m)\n aligned(b,f,g,F);pivot(b,g,F);Z=old.fresh_zero_flag(b,a)\n root_action(b,f,g,payloads,ai,Z,F,post=not reverse,undo=reverse)\n erase_product_zero(b,a,ai,Z)\n mask=b.alloc_bits(12);b.hmr(ai,mask);b.free(ai)\n if reverse:phase_inverse34(b,a,mask,zero_mask=mask[0]);b.free(mask)\n else:phase[\'_root_inverse_mask\']=mask\n pivot(b,g,F)\n if reverse:clear_flag(b,g[-1],F)\n elif full:phase[\'held_gap\'][m]=F\n else:phase[\'gap_flags\'][m]=hold_flag(b,F)\n\n\ndef make_converter():\n _,scope,short=mh.make_converter(True)\n scope.update(dk=dk,THRESHOLD=THRESHOLD,old_look_solve=scope[\'look_solve\'])\n for name in [\'pivot\',\'aligned\',\'cross\',\'direct_pre\',\'direct_post\',\'look_solve\']:\n  scope[name]=clone(globals()[name],scope)\n short.update(old=SimpleNamespace(**scope),dk=dk,pivot=scope[\'pivot\'],aligned=scope[\'aligned\'])\n for name in [\'root_action\',\'look_root_cached\']:\n  short[name]=clone(globals()[name],short)\n short[\'_bez_monic\']=clone(stable._bez_monic,short)\n core=clone(stable.base._short_scope[\'convert_safe\'],short)\n def convert_safe(b,A,B,reverse=False,history_scratch12=None,phase_token=None,*,monic_root=True):\n  if phase_token is None:\n   phase_token={\'a\':{},\'q\':{},\'heads\':{},\'_short_monic\':bool(monic_root)}\n   for key in [\'gap_flags\',\'given_gap\',\'incoming_gap\',\'return_mask\',\'returned_gap\',\'held_gap\']:phase_token[key]={}\n  out=core(b,A,B,reverse,history_scratch12,phase_token,monic_root=monic_root)\n  if reverse:assert not phase_token[\'heads\']\n  return out\n return convert_safe\nconvert_safe=make_converter()\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/direct_kernels.py': '"""Direct gap-chart arithmetic kernels; no stable module mutations.\n\nAfter the pivot head swap, F*g[-2]=0. Therefore the shifted low-g times\nshifted coefficient products have one identically zero highest coefficient.\nThis algebraic constraint, not an approximation, permits one fewer evaluation.\n"""\nfrom pathlib import Path\nimport sys\nfrom contextlib import contextmanager\nfrom types import FunctionType,SimpleNamespace\nsys.path.insert(0,str(Path(__file__).resolve().parents[3]/\'playbook/kit/decoder\'))\nsys.path.insert(0,str(Path(__file__).resolve().parents[3]/\'playbook/kit/decoder/exp/convert\'))\nfrom exp.convert import kernels as k\nfrom exp.decoder.common import ar\n\n_scope=dict(vars(k),ar=SimpleNamespace(**dict(vars(ar),mul_xor=ar.mul_xor0)))\nrank_update=FunctionType(k.rank_update.__code__,_scope,\'rank_update\',k.rank_update.__defaults__,k.rank_update.__closure__)\n\ndef conv(b,g,q,out,N,reverse=False):\n """Rank-N product with known degree<N; target truncation permitted."""\n V,W=k.vandermonde(N)\n for t in (reversed(range(N)) if reverse else range(N)):\n  tc=[W[j][t] for j in range(len(out))]\n  if any(tc):rank_update(b,g,V[t][:len(g)],q,V[t][:len(q)],out,tc)\n\n@contextmanager\ndef shifted_pair(b,q,F):\n """Embed(q0,q1) as q or x*q. Restore heads with one measured AND copy."""\n z=b.alloc_qubits(12)\n b.cswap(F,q[1],z);b.cswap(F,q[0],q[1])\n yield [q[0],q[1],z]\n b.cswap(F,q[0],q[1]);b.cx(z,q[1])\n mask=b.alloc_bits(12);b.hmr(z,mask);b.free(z)\n for j in range(12):b.ccz(F,q[1][j],mask[j])\n b.free(mask)\n\ndef shifted_conv(b,g,q,out,F,reverse=False):\n with shifted_pair(b,q,F) as coeff:conv(b,g,coeff,out,len(g)+1,reverse)\n\n\ndef shifted_product(b,g,x,out,F):\n z=b.alloc_qubits(12);b.cswap(F,x,z)\n conv(b,g,[x,z],out,len(g))\n b.cx(z,x)\n mask=b.alloc_bits(12);b.hmr(z,mask);b.free(z)\n for j in range(12):b.ccz(F,x[j],mask[j])\n b.free(mask)\n\n@contextmanager\ndef selected(b,x,y,F):\n """Select x if F0, y if F1, with12T and exact zero-T phase closure."""\n z=b.alloc_qubits(12);b.cx(y,x)\n for j in range(12):b.init_and(F,x[j],z[j])\n b.cx(y,x);b.cx(x,z)\n yield z\n mask=b.alloc_bits(12);b.hmr(z,mask);b.free(z)\n b.cz(x,mask);b.cx(y,x)\n for j in range(12):b.ccz(F,x[j],mask[j])\n b.cx(y,x);b.free(mask)\n\n\ndef selected_dot(b,g,v,target,F):\n """Transposed shifted scalar product;24 fresh wires,12 extraT.\n\nCompute both transpose outputs, select into target, then erase the unselected\nbilinear outputs in the unshifted frame. Both HMR repairs are free CZ/CCZ\nwith a classical control; no deferred cubic phase debt is introduced.\n"""\n d=[b.alloc_qubits(12),b.alloc_qubits(12)]\n N=len(g);V,W=k.vandermonde(N)\n for t in range(N):\n  rank_update(b,g,V[t],v,[W[i][t] for i in range(N)],d,[1,t])\n b.cx(d[0],target);b.cx(d[1],d[0])\n for j in range(12):b.ccx(F,d[0][j],target[j])\n b.cx(d[1],d[0])\n masks=[b.alloc_bits(12),b.alloc_bits(12)]\n for x,m in zip(d,masks):b.hmr(x,m);b.free(x)\n # Transpose of output map(1,t) into a classical product mask.\n for t in range(N):\n  mask=b.alloc_bits(12);b.bit_store(mask,False);b.cx(masks[0],mask)\n  # mask dot(t*product) = transpose(multiply t)*mask.\n  f=ar.field(12)\n  for j in range(12):\n   value=f.mul(t,1<<j)\n   for i in ar.bits(value):b.cx(masks[1][i],mask[j])\n  x,p=k.expose(b,g,V[t]);y,r=k.expose(b,v,[W[i][t] for i in range(N)])\n  ar.phase_product(b,x,y,mask)\n  k.unexpose(b,v,[W[i][t] for i in range(N)],r);k.unexpose(b,g,V[t],p)\n  b.free(mask)\n for m in masks:b.free(m)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/exact_subs.py': '"""DQI survey: exact arithmetic and phase composition on the supplied decoder.\n\nKeeps the original extraction threshold, every original chart flag, and the\noriginal conversion gap schedule.  All replacements are exact on their\nstated APIs, including zero inputs and every measurement outcome.  The\nwhole decoder therefore inherits the base\'s exceptional-input domain.\n\nNew components: a shared Frobenius-product tensor, exact terminal gauge\nphase, 4-AND GF64 scalar phases, 32-T GF4096 scalar phase, and initialized\n18-T GF64 inversion with quadratic zero-flag erasure.\n\nSources: Bartoli--Zini--Zullo arXiv:2201.08242; Calik--Sonmez Turan--Peralta\nDOI10.1007/s12095-018-0297-2; Soeken arXiv:2005.01778. Finite-size schedules\nand the SAT quadratic-output phase are derived and checked in this survey.\nThis file depends only on unchanged files supplied in decoderkit-2.\n"""\nfrom pathlib import Path\nfrom functools import lru_cache\nfrom contextlib import contextmanager\nfrom types import CodeType, FunctionType, ModuleType, SimpleNamespace, CellType\nimport sys\n\nROOT = Path(__file__).resolve().parents[3] / "playbook/kit/decoder"\nDECODER = Path(__file__).resolve().parents[3]\nsys.path[:0] = [str(ROOT), str(ROOT.parent)]\nfrom final_decoder import decode as core\nfrom final_decoder.decode import *\nfrom exp.mul99.kernels import tensor99_extension\nfrom exp.arithmetic.field import (bits, field, tower, linear_inplace,\n                                  inverse_columns, apply)\nfrom exp.arithmetic.lowwidth import classical_transpose\nfrom exp.phase24_tradeoff.phase6small import classical_equal\nfrom exp.inverse_search.inverse20 import expose_pair\nfrom exp.inverse_search import inverse20 as ref\nfrom exp.inverse_search import inverse20_iso as iso\nfrom exp.inverse19_joint import kernel as joint\nfrom exp.inverse12_escape import zero1\nfrom exp.jet24.pro_jet import (W14_JOINT18, TO6, FROM6,\n                               inverse6_joint18_12extra)\nfrom exp.mul99.phase36 import phase_classical_parity\nfrom exp.noscratch.schedules import norm_xor_noscratch\nfrom exp.noscratch.kernels import mul_xor_noscratch\nfrom exp.fold_cached_norm.phase34 import phase_inverse34 as old12\nfrom exp.phase24_tradeoff.phase6small import phase_inverse6_classical6 as old6\nfrom exp.arithmetic.lowwidth import phase_inverse_generic6 as old_generic6\n\nar, emul, einv = core.ar, core.emul, core.einv\n\n\nPAIR4 = ((1, 7, 140), (13, 1, 191), (12, 8, 45), (5, 6, 82),\n         (4, 3, 17), (6, 14, 123), (11, 9, 55), (2, 4, 249),\n         (7, 5, 234), (9, 2, 99), (8, 12, 182), (3, 11, 255))\n\n\nEVAL = (1, 8, 7, 6, 7, 0, 9, 4, 0, 0, 8, 13,\n        4, 11, 9, 4, 5, 8, 9, 2, 2, 6, 12, 14)\n\n\nOUT = (5733313, 3867560, 5497562, 4112984)\n\n\ndef pull(mask, columns):\n    return sum(((mask & col).bit_count() & 1) << i\n               for i, col in enumerate(columns))\n\n\n@lru_cache(None)\ndef joint_tensor(c):\n    sq = [emul(1 << i, 1 << i) for i in range(24)]\n    hc = [emul(c, 1 << i) ^ ((1 << i) >> 12) for i in range(24)]\n    m99 = tensor99_extension(2048)\n    if c:\n        # EVAL(lam*H)=EVAL(c*H+h), EVAL(mu*D)^2=EVAL(D^2).\n        lam, mu = c ^ 10905896, 7455554\n        lc = [emul(lam, 1 << i) for i in range(24)]\n        mc = [emul(mu, 1 << i) for i in range(24)]\n        xc = [ar.apply(EVAL, z) for z in hc]\n        yc = [ar.apply(EVAL, z) for z in mc]\n        assert [ar.apply(EVAL, z) for z in lc] == xc\n        assert [ar.field(4).square(z) for z in yc] == [ar.apply(EVAL, z) for z in sq]\n        scale = einv(emul(lam, mu))\n        out1 = [emul(scale, z) for z in OUT]\n        terms = [(pull(u, xc), pull(v, yc),\n                  ar.apply(out1, w & 15) | (ar.apply(OUT, w >> 4) << 24))\n                 for u, v, w in PAIR4]\n        for mask, value in m99[9:]:\n            terms.append((pull(mask, lc), pull(mask, mc), emul(scale, value)))\n            terms.append((pull(mask, hc), pull(mask, sq), value << 24))\n    else:\n        # Align one product term with the GF12-by-GF24 scalar product.\n        lam, mu = 6520244, 6404392\n        lc = [emul(lam, 1 << i) for i in range(24)]\n        mc = [emul(mu, 1 << i) for i in range(24)]\n        scale = einv(emul(lam, mu))\n        terms = [(pull(m, lc), pull(m, mc), emul(scale, w)) for m, w in m99]\n        for off in (0, 12):\n            ds = [(v >> off) & 4095 for v in sq]\n            terms.extend((m << 12, pull(m, ds), w << (24 + off))\n                         for m, w in ar.tensor(12))\n    combined = {}\n    for u, v, w in terms:\n        combined[u, v] = combined.get((u, v), 0) ^ w\n    result = tuple((u, v, w) for (u, v), w in combined.items() if w)\n    assert len(result) == (192 if c else 182)\n    return result\n\n\ndef accumulator(b, x, y, z, terms):\n    """Accumulate a binary tensor with no scratch, restoring both inputs."""\n    for u, v, w in terms:\n        ix, iy, iz = [list(ar.bits(mask)) for mask in (u, v, w)]\n        a, c, t = ix[0], iy[0], iz[0]\n        for i in ix[1:]: b.cx(x[i], x[a])\n        for i in iy[1:]: b.cx(y[i], y[c])\n        for i in iz[1:]: b.cx(z[t], z[i])\n        b.ccx(x[a], y[c], z[t])\n        for i in reversed(iz[1:]): b.cx(z[t], z[i])\n        for i in reversed(iy[1:]): b.cx(y[i], y[c])\n        for i in reversed(ix[1:]): b.cx(x[i], x[a])\n\n\ndef chain_step(b, head, N, D, L, q, ct, c, reverse=False, first=False):\n    if first:\n        return core.external_chain_step(b, head, N, D, L, q, ct, c, reverse, first)\n    if reverse:\n        core.scalepair(b, N, core.inv(q))\n        b.swap(N, D)\n        core.scalepair(b, L, core.inv(q))\n    a, h = head\n    core.cx(b, h, a, ct)\n    accumulator(b, core.combine(head), D, list(N) + list(L), joint_tensor(c))\n    core.cx(b, h, a, ct)\n    if not reverse:\n        core.scalepair(b, L, q)\n        b.swap(N, D)\n        core.scalepair(b, N, q)\n\ndef clone(fn,scope):\n    out=FunctionType(fn.__code__,scope,fn.__name__,fn.__defaults__,fn.__closure__)\n    out.__kwdefaults__=fn.__kwdefaults__\n    return out\n\n\ndef dirty_zero_phase(b, x, dirty):\n    """Exact zero phase, one clean bit and len(x)-3 borrowed dirty bits."""\n    work = list(b.alloc_qubits(1)) + list(dirty[:len(x)-3])\n    b.x(x)\n    for first in (True, False):\n        if first: b.init_and(x[0], x[1], work[0])\n        start = 1 if first else 2\n        for j in range(start, len(work)):\n            b.ccx(work[j-1], x[j+1], work[j])\n        b.cz(work[-1], x[-1])\n        for j in reversed(range(start, len(work))):\n            b.ccx(work[j-1], x[j+1], work[j])\n        if first: b.del_and(x[0], x[1], work[0])\n    b.x(x)\n    b.free(work[0])\n\n\ndef direct_gauge_narrow(b, D, norm_pair, gauge_mask):\n    """Recover the gauge and close both norm histories with 48 private bits."""\n    reciprocal, direct = norm_pair\n    ar._erase_norm(b, D, reciprocal, K)\n    b.free(reciprocal)\n    b.cx(D[12:], D[:12])\n    oldmask = b.alloc_bits(24)\n    for off in (0, 12):\n        half = D[off:off+12]\n        out = b.alloc_qubits(12)\n        mul_xor_noscratch(b, half, direct, out)\n        b.hmr(half, oldmask[off:off+12])\n        b.swap(half, out)\n        b.free(out)\n    b.cz(D, gauge_mask)\n    b.free(gauge_mask)\n    inverse_mask = inverse_stream_noscratch(b, direct)\n    ar._phase_norm(b, D, inverse_mask, K)\n    b.free(inverse_mask)\n    ar.phase_product(b, D[:12], direct, oldmask[:12])\n    ar.phase_product(b, D[12:], direct, oldmask[12:])\n    b.free(oldmask)\n    norm = b.alloc_qubits(12)\n    norm_xor_noscratch(b, D, norm, K)\n    m = b.alloc_bits(12)\n    b.hmr(direct, m)\n    b.free(direct)\n    phase_inverse36_pivot(b, norm, m)\n    b.free(m)\n    ar._erase_norm(b, D, norm, K)\n    b.free(norm)\n    mask = b.alloc_bits(24)\n    b.hmr(D, mask)\n    b.free(D)\n    return mask\n\n\ndef head_phase(b, head, factor, mask, ct):\n    a, h = head\n    cx(b, h, a, ct)\n    phase_product24(b, combine(head), factor, mask, K)\n    cx(b, h, a, ct)\n\n\ndef terminal_two_phase(b, N, heads, params, ct, mask, pole, dirty):\n    """Close the last two scalar replay steps and the pole flag exactly."""\n    stop = len(heads)-2\n    D = b.alloc_qubits(24)\n    b.x(D[0])\n    for i in range(stop):\n        external_scalar_step(b, heads[i], N, D, params[i][0], ct, first=i == 0)\n    W = b.alloc_qubits(24)\n    b.cx(N, W)\n    external_mul_head(b, heads[-2], D, W, ct)\n    scalepair(b, D, params[-2][0])\n    b.cz(D, mask)\n    scalepair(b, D, inv(params[-2][0]))\n    head_phase(b, heads[-1], W, mask, ct)\n    scalepair(b, W, params[-1][0])\n    b.ccz(pole, W, mask)\n    scalepair(b, W, inv(params[-1][0]))\n    fm = b.alloc_bits(1)\n    b.hmr(pole, fm[0])\n    b.free(pole)\n    b.push_condition(fm[0])\n    scalepair(b, D, params[-2][0])\n    external_mul_head(b, heads[-1], W, D, ct)\n    dirty_zero_phase(b, D, dirty)\n    external_mul_head(b, heads[-1], W, D, ct)\n    scalepair(b, D, inv(params[-2][0]))\n    b.pop_condition()\n    b.free(fm)\n    wm = b.alloc_bits(24)\n    b.hmr(W, wm)\n    b.free(W)\n    b.cz(N, wm)\n    head_phase(b, heads[-2], D, wm, ct)\n    b.free(wm)\n    b.free(mask)\n    for i in reversed(range(stop)):\n        external_scalar_step(b, heads[i], N, D, params[i][0], ct, True, first=i == 0)\n    b.x(D[0])\n    b.free(D)\n\n\ndef external_node_chain(b,words,gi,reverse=False,token=None,normalizer=None):\n    g=groups[gi];N=combine(words[g[\'word\']:g[\'word\']+2]);L=combine(words[g[\'word\']+2:g[\'word\']+4])\n    heads=[words[2*i:2*i+2]for i in range(2*gi)]\n    params=[];c=0\n    for i in range(2*gi):\n        q=ks[i]^g[\'kap\'];assert q\n        params.append((q,c));c^=inv(q)\n    assert c==0\n    if not reverse:\n        D=b.alloc_qubits(24);b.x(D[0])\n        for i,(head,(q,c))in enumerate(zip(heads,params)):\n            external_chain_step(b,head,N,D,L,q,g[\'c\'],c,first=i==0)\n        pole=zero_flag(b,D);b.cswap(pole,N,D)\n        norm=external_normalize_paired(b,N,D,L)\n        gauge=b.alloc_bits(24);b.hmr(D,gauge);b.free(D)\n        return dict(pole=pole,gauge=gauge,norm=norm)\n    pole=token[\'pole\'];D=b.alloc_qubits(24);b.x(D[0]);b.cx(pole,N[0]);b.cx(pole,D[0])\n    for head,(q,c)in reversed(list(zip(heads,params))[1:]):external_chain_step(b,head,N,D,L,q,g[\'c\'],c,True)\n    q0=params[0][0];scalepair(b,N,inv(q0));b.swap(N,D);scalepair(b,L,inv(q0))\n    norm=external_normalize_paired(b,N,D,L,True,token[\'norm\'])\n    a,h=heads[0];cx(b,h,a,g[\'c\']);b.cx(combine(heads[0]),N);cx(b,h,a,g[\'c\']);b.cx(h,L[:12])\n    mask=direct_gauge_narrow(b,D,norm,token[\'gauge\'])\n    terminal_two_phase(b,N,heads,params,g[\'c\'],mask,pole,L)\n\n\ndef extraction(b,words,reverse=False,token=None):\n    heads=[words[2*i:2*i+2]for i in range(34)];N,L=words[-2:]\n    if not reverse:\n        ext=external_extraction(b,words);D=b.alloc_qubits(12);b.x(D[0])\n        first_step(b,heads[0],N,D,L,ks[0])\n        for head,kap,c in zip(heads[1:],ks[1:],corrections[1:]):step(b,head,N,D,L,kap,c)\n        pole=b.alloc_qubits(1)[0];guarded.zero_toggle(b,D,pole);b.cswap(pole,N,D)\n        norm=normal_form(b,N,D,L);ci=controlled_inverse(b,L,pole)\n        gauge=b.alloc_bits(12);b.hmr(D,gauge);b.free(D)\n        return dict(external=ext,pole=pole,normalization=norm,gauge_mask=gauge,chart_inverse=ci)\n    pole=token[\'pole\'];controlled_inverse(b,L,pole,True,token[\'chart_inverse\'])\n    D=b.alloc_qubits(12);b.x(D[0]);b.cx(pole,N[0]);b.cx(pole,D[0])\n    for head,kap,c in reversed(list(zip(heads,ks,corrections))[1:]):step(b,head,N,D,L,kap,c,True)\n    scale(b,N,inv(ks[0]));b.swap(N,D);scale(b,L,inv(ks[0]))\n    extra_z_mask=normal_form(b,N,D,L)\n    b.cx(heads[0][0],N);b.cx(heads[0][1],L)\n    b.cz(D,token[\'normalization\']);b.free(token[\'normalization\'])\n    inverse_mask=b.alloc_bits(12);b.hmr(D,inverse_mask);b.free(D)\n    z_mask=token[\'gauge_mask\'];b.cx(extra_z_mask,z_mask);b.free(extra_z_mask)\n    D=b.alloc_qubits(12);b.x(D[0]);scalar_step(b,heads[0],N,D,ks[0],first=True)\n    for head,kap in zip(heads[1:],ks[1:]):scalar_step(b,head,N,D,kap)\n    b.cswap(pole,N,D);b.cz(D,z_mask);b.free(z_mask)\n    phase_inverse34(b,D,inverse_mask);b.free(inverse_mask);b.cswap(pole,N,D)\n    guarded.zero_toggle(b,D,pole)\n    for head,kap in reversed(list(zip(heads[1:],ks[1:]))):scalar_step(b,head,N,D,kap,True)\n    scalar_step(b,heads[0],N,D,ks[0],True,first=True)\n    b.x(D[0]);b.free(D);b.free(pole);external_extraction(b,words,True,token[\'external\'])\n\ndef _mask_scalars():\n    out = {}\n    f = field(6)\n    for s in range(1, 64):\n        si = f.invert(s)\n        mask = sum((f.mul(si, 1 << i) & 1) << i for i in range(6))\n        assert mask and mask not in out\n        out[mask] = s\n    return out\n\n\ndef _fixed_phase(b, x, gates, output):\n    """Phase one affine-XOR-AND circuit; final AND is a free CZ."""\n    count = len(gates) - 1\n    a = b.alloc_qubits(count)\n    source = list(x) + list(a)\n    for i, (left, right) in enumerate(gates[:-1]):\n        with expose_pair(b, source[:6+i], left, right) as (p, q):\n            b.init_and(p, q, a[i])\n    assert output >> (7 + count) == 1\n    if output & 1:\n        b.z(True)\n    for i in bits((output >> 1) & ((1 << (6 + count))-1)):\n        b.z(source[i])\n    with expose_pair(b, source, *gates[-1]) as (p, q):\n        b.cz(p, q)\n    for i in reversed(range(count)):\n        with expose_pair(b, source[:6+i], *gates[i]) as (p, q):\n            b.del_and(p, q, a[i])\n    b.free(a)\n\n\ndef _zero_phase(b, x):\n    """Phase [x=0], exactly 4 T / 4 scratch for six input bits."""\n    b.x(x)\n    a = b.alloc_qubits(4)\n    b.init_and(x[0], x[1], a[0])\n    for i in range(1, 4):\n        b.init_and(a[i-1], x[i+1], a[i])\n    b.cz(a[3], x[5])\n    for i in reversed(range(1, 4)):\n        b.del_and(a[i-1], x[i+1], a[i])\n    b.del_and(x[0], x[1], a[0])\n    b.free(a)\n    b.x(x)\n\nSCALARS = _mask_scalars()\nGATES = {1: ((23, 78), (60, 234), (15, 182), (236, 310), (552, 1137))}\nOUTPUT = {1: 3373}\n\n\nFORMS = ((8, 20), (38, 92), (32, 280), (414, 538))\n\n\nTERMS = ((0,), (2,), (3,), (4,), (5,), (9,), (0, 1), (0, 2),\n         (0, 8), (0, 9), (1, 2), (1, 3), (1, 6), (1, 7),\n         (2, 5), (2, 9), (3, 7), (3, 8), (3, 9), (5, 6),\n         (5, 9), (6, 7))\n\n\nSCALES = {m: [field(6).mul(s, 1 << i) for i in range(6)]\n          for m, s in SCALARS.items()}\n\n\nUNSCALES = {m: inverse_columns(c) for m, c in SCALES.items()}\n\n\ndef _ordinary(b, x):\n    g = b.alloc_qubits(4)\n    source = list(x) + list(g)\n    for i, (left, right) in enumerate(FORMS):\n        with expose_pair(b, source[:6+i], left, right) as (p, q):\n            b.init_and(p, q, g[i])\n    for term in TERMS:\n        if len(term) == 1:\n            b.z(source[term[0]])\n        else:\n            b.cz(source[term[0]], source[term[1]])\n    for i in reversed(range(4)):\n        with expose_pair(b, source[:6+i], *FORMS[i]) as (p, q):\n            b.del_and(p, q, g[i])\n    b.free(g)\n\n\ndef phase_inverse6_quad4(b, x, mask, zero_mask=None):\n    """Phase dot(mask,inv0(x)) + zero_mask*[x=0], <=4 T /4 scratch.\n\n    All controls are classical.  No input is exceptional; masks and x are\n    restored exactly, with del_and closing every measurement phase.\n    """\n    assert len(x) == len(mask) == 6\n    for m in range(1, 64):\n        eq = classical_equal(b, mask, m)\n        b.push_condition(eq)\n        linear_inplace(b, x, SCALES[m])\n        if zero_mask is None:\n            _ordinary(b, x)\n        else:\n            b.push_condition(zero_mask)\n            _fixed_phase(b, x, GATES[1], OUTPUT[1])\n            b.pop_condition()\n            b.x(zero_mask)\n            b.push_condition(zero_mask)\n            _ordinary(b, x)\n            b.pop_condition()\n            b.x(zero_mask)\n        linear_inplace(b, x, UNSCALES[m])\n        b.pop_condition()\n        b.free(eq)\n    if zero_mask is not None:\n        eq = classical_equal(b, mask, 0)\n        b.push_condition(eq)\n        b.push_condition(zero_mask)\n        _zero_phase(b, x)\n        b.pop_condition()\n        b.pop_condition()\n        b.free(eq)\n\ndef phase_inverse12_virtual(b, x, mask, zero_mask=None):\n    # Handle types are build-time metadata, not measured quantum values.\n    # Preserve the unguarded API for a quantum zero-phase coefficient.\n    if zero_mask is not None and type(zero_mask) is type(x[0]):\n        return phase_inverse12_virtual_nonzero(b, x, mask, zero_mask)\n    active = b.alloc_bits(1)[0]\n    b.bit_store(active, False)\n    for bit in mask:\n        b.bit_store(active, True, control=bit)\n    if zero_mask is not None:\n        b.bit_store(active, True, control=zero_mask)\n    b.push_condition(active)\n    phase_inverse12_virtual_nonzero(b, x, mask, zero_mask)\n    b.pop_condition()\n    b.free(active)\n\ndef phase_inverse12_virtual_nonzero(b, x, mask, zero_mask=None):\n    """Same exact phase, <=32 executed T / 15 scratch.\n\n    The quantum coefficient is exposed as a parity of the original x,\n    chosen solely by classical mask bits.  No quantum mask is treated as\n    classical, and no six-qubit coefficient register is allocated.\n    """\n    assert len(x) == len(mask) == 12\n    to, frm, K = tower(12)\n    linear_inplace(b, x, to)\n    mt = classical_transpose(b, mask, frm)\n    D = b.alloc_qubits(6)\n    norm_xor_noscratch(b, x, D, K)\n    linear_inplace(b, D, TO6)\n    dag = W14_JOINT18\n    g = b.alloc_qubits(6)\n    for i, (left, right) in enumerate(dag[\'raw_gates\']):\n        with ref.expose_pair(b, D, left, right) as (p, q):\n            b.init_and(p, q, g[i])\n    prefix = list(D) + list(g)\n    forms = ref.norm_forms(1)\n    w = ref.inverse3_virtual(b, prefix, forms)\n    source = prefix + list(w)\n    def coefficient(vector):\n        h = apply(FROM6, vector)\n        co = b.alloc_bits(12)\n        b.bit_store(co, False)\n        for j in range(6):\n            for r in bits(field(6).mul(h, 1 << j)):\n                b.cx(mt[r], co[j])\n                b.cx(mt[r], co[j+6])\n                b.cx(mt[r+6], co[j+6])\n        return co\n    linear = [(e & 65535) ^ (1555 if e >> 16 & 1 else 0)\n              for e in dag[\'outputs\'][:6]]\n    # Linear and constant source terms are bilinear in x and classical mask.\n    for r in range(16):\n        vector = sum(((e >> r) & 1) << j for j, e in enumerate(linear))\n        if not vector:\n            continue\n        co = coefficient(vector)\n        for j in range(12):\n            if r == 0:\n                b.cz(x[j], co[j])\n            else:\n                b.ccz(source[r-1], x[j], co[j])\n        b.free(co)\n    for i, (left, right) in enumerate(dag[\'gates\'][1:], 1):\n        vector = sum(((e >> (16+i)) & 1) << j for j, e in enumerate(dag[\'outputs\'][:6]))\n        if not vector:\n            continue\n        co = coefficient(vector)\n        with ref.expose_pair(b, source, left, right) as (p, q):\n            phase_classical_parity(b, p, q, x, co)\n        b.free(co)\n    if zero_mask is not None:\n        b.z(zero_mask)\n        for r in bits(1555 >> 1):\n            b.cz(source[r], zero_mask)\n        with ref.expose_pair(b, source, *dag[\'gates\'][0]) as (p, q):\n            b.ccz(p, q, zero_mask)\n    wm = b.alloc_bits(3)\n    b.hmr(w, wm)\n    b.free(w)\n    ref.phase_inverse3_virtual(b, prefix, forms, wm)\n    b.free(wm)\n    for i in reversed(range(6)):\n        with ref.expose_pair(b, D, *dag[\'raw_gates\'][i]) as (p, q):\n            b.del_and(p, q, g[i])\n    b.free(g)\n    linear_inplace(b, D, FROM6)\n    ar._erase_norm(b, x, D, K)\n    b.free(D)\n    b.free(mt)\n    linear_inplace(b, x, frm)\n\ndef inverse6_init18(b,x):\n    """Initialized inverse0; the zero output is erased quadratically."""\n    t,F=inverse6_joint18_12extra(b,x)\n    m=b.alloc_bits(1)\n    b.hmr(F,m[0]);b.free(F)\n    b.z(m[0])\n    mask=b.alloc_bits(6);b.bit_store(mask,False)\n    b.cx(m[0],mask[0])\n    ar.phase_product(b,x,t,mask)\n    b.free(mask);b.free(m)\n    return t\n\n\ndef inverse_zero(b,x):\n    to,frm,K=tower(12)\n    linear_inplace(b,x,to)\n    N=b.alloc_qubits(6)\n    norm_xor_noscratch(b,x,N,K)\n    t,F=inverse6_joint18_12extra(b,N)\n    ar._erase_norm(b,x,N,K)\n    b.free(N)\n    out=b.alloc_qubits(12)\n    b.cx(x[6:],x[:6])\n    mul_xor_noscratch(b,x[:6],t,out[:6])\n    mul_xor_noscratch(b,x[6:],t,out[6:])\n    b.cx(x[6:],x[:6])\n    ar._erase_norm(b,out,t,K)\n    b.free(t)\n    linear_inplace(b,x,frm)\n    linear_inplace(b,out,frm)\n    return out,F\n\ndef private_rebind(entry, replacements):\n    memo = {}\n    scopes = {}\n    processed = set()\n    def owned(filename):\n        try:\n            return Path(filename).resolve().is_relative_to(DECODER)\n        except (TypeError, ValueError):\n            return False\n    def names(code):\n        result = set(code.co_names)\n        for value in code.co_consts:\n            if isinstance(value, CodeType):\n                result.update(names(value))\n        return result\n    def copy(value):\n        key = id(value)\n        if key in replacements:\n            return replacements[key]\n        if key in memo:\n            return memo[key]\n        if isinstance(value, FunctionType) and owned(value.__code__.co_filename):\n            old = value.__globals__\n            scope = scopes.setdefault(id(old), dict(old))\n            closure = value.__closure__\n            if closure:\n                closure = tuple(CellType(copy(c.cell_contents)) for c in closure)\n            out = FunctionType(value.__code__, scope, value.__name__, value.__defaults__, closure)\n            memo[key] = out\n            out.__kwdefaults__ = value.__kwdefaults__\n            for name in names(value.__code__):\n                token = (id(old), name)\n                if token not in processed and name in old:\n                    processed.add(token)\n                    scope[name] = copy(old[name])\n            return out\n        # contextmanager decorators store the original generator in a cell.\n        if isinstance(value, FunctionType) and getattr(value, \'__wrapped__\', None) is not None:\n            wrapped = value.__wrapped__\n            if isinstance(wrapped, FunctionType) and owned(wrapped.__code__.co_filename):\n                out = contextmanager(copy(wrapped))\n                memo[key] = out\n                return out\n        if isinstance(value, SimpleNamespace) or (isinstance(value, ModuleType) and owned(getattr(value, \'__file__\', None))):\n            out = SimpleNamespace()\n            memo[key] = out\n            for name, item in vars(value).items():\n                if name != \'__builtins__\':\n                    setattr(out, name, copy(item))\n            return out\n        if isinstance(value, dict):\n            out = {}\n            memo[key] = out\n            out.update((k, copy(v)) for k, v in value.items())\n            return out\n        if isinstance(value, (list, tuple)) and any(isinstance(v, (FunctionType, SimpleNamespace, dict)) for v in value):\n            out = [copy(v) for v in value]\n            out = tuple(out) if isinstance(value, tuple) else out\n            memo[key] = out\n            return out\n        return value\n    return copy(entry)\n\n\ndef initialized19(b, x, with_zero=False):\n    if with_zero:\n        return inverse6_joint18_12extra(b, x)\n    return inverse6_init18(b, x)\n\n\ndef initialized_only(old):\n    def wrapped(b, x, z=None, keep_norm=False):\n        if z is None and not keep_norm:\n            return inverse6_init18(b, x)\n        return old(b, x, z, keep_norm)\n    return wrapped\n\n\nexternal_chain_step = chain_step\nexternal_extraction = clone(core.external_extraction, globals())\n_base_body = clone(core.decode, globals())\n_body = private_rebind(_base_body, {\n    id(old6): phase_inverse6_quad4,\n    id(old_generic6): phase_inverse6_quad4,\n    id(old12): phase_inverse12_virtual,\n    id(zero1.inverse_zero): inverse_zero,\n    id(joint.inverse_zero): inverse_zero,\n    id(joint.inverse6_init19): initialized19,\n    id(iso.inverse6_xor20): initialized_only(iso.inverse6_xor20),\n    id(ref.inverse6_xor20): initialized_only(ref.inverse6_xor20),\n})\n\n\ndef decode(b, S, E):\n    _body(b, S, E)\n\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/frozen_chart.py': '"""Width-safe paired normalization in high-component denominator chart.\n\nAll scalar-inverse and gauge debts are explicit; forward chain unchanged.\nThis is an intentionally conservative combined-width baseline, not a cost win.\n"""\nfrom pathlib import Path\nimport sys\nfrom functools import lru_cache\nfrom types import FunctionType\nroot=Path(__file__).resolve().parents[3]\nsys.path[:0]=[str(root/\'playbook/kit/decoder\'),str(root/\'playbook/kit\')]\nfrom final_decoder import decode as ref\nfrom exp.decoder.common import *\nfrom exp.noscratch.schedules import inverse_stream_noscratch\nfrom exp.mul99.phase36 import phase_inverse36_pivot\nfrom final_decoder.conversion import clear_flag\n\n\ndef clone(fn,scope):\n out=FunctionType(fn.__code__,scope,fn.__name__,fn.__defaults__,fn.__closure__)\n out.__kwdefaults__=fn.__kwdefaults__;return out\n\n\ndef scale_defer(b,x,f):\n out=b.alloc_qubits(12);ar.mul_xor0(b,x,f,out)\n m=b.alloc_bits(12);b.hmr(x,m);b.swap(x,out);b.free(out);return m\n\n\ndef add_masks(b,*ms):\n out=ms[0]\n for m in ms[1:]:b.cx(m,out);b.free(m)\n return out\n\n\n@lru_cache(None)\ndef norm_chart_columns():\n # r -> r^2+r, with one kernel bit to make the map invertible.\n f=ar.field(12);base=[f.mul(1<<i,1<<i)^(1<<i)for i in range(12)]\n # kernel={0,1}; keep r\'s low bit, and inject a missing image direction.\n for extra in range(1,4096):\n  cols=[v|((1 if i==0 else 0)<<12)for i,v in enumerate(base)]+[extra]\n  try:back=ar.inverse_columns(cols)\n  except (AssertionError,ValueError,StopIteration):continue\n  return tuple(cols),tuple(back)\n raise AssertionError(\'no rank13 norm chart\')\n\n\ndef chart_norm(b,r,F,h,reverse=False):\n cols,back=norm_chart_columns();w=list(r)+[h]\n # q=r²+r+K except subfield chart F=1,r=0, where q=1.\n if not reverse:\n  ar.linear_inplace(b,w,cols)\n  for i in ar.bits(K):b.x(r[i])\n  for i in ar.bits(K^1):b.cx(F,r[i])\n else:\n  for i in ar.bits(K^1):b.cx(F,r[i])\n  for i in ar.bits(K):b.x(r[i])\n  ar.linear_inplace(b,w,back)\n\n\ndef correct_subfield(b,x,F):\n # In the subfield chart, swap input halves and use final shear r+1-F.\n # This costs one controlled swap rather than two controlled shears.\n b.cswap(F,x[:12],x[12:])\n\n\ndef normalize(b,N,D,L):\n # sqrt(L)/D, then square, equals L/D² with no nonlinear cost.\n ref.external_square24(b,L,True)\n r,g=D[:12],D[12:]\n F=ref.guarded.fresh_zero_flag(b,g);b.cswap(F,r,g)\n oldg=inverse_stream_noscratch(b,g)\n olda=scale_defer(b,r,g)\n for x in (N,L):correct_subfield(b,x,F)\n for x in (N,L):ar.mul_xor0(b,r,x[12:],x[:12])\n half_masks=[scale_defer(b,x[12:],g)for x in (N,L)]\n im=inverse_stream_noscratch(b,g) # g restored\n b.cz(g,oldg);b.free(oldg)\n ar.phase_product(b,r,g,olda);b.free(olda)\n for x,m in zip((N,L),half_masks):ar.phase_product(b,x[12:],g,m);b.free(m)\n h=b.alloc_qubits(1)[0];chart_norm(b,r,F,h)\n gm=scale_defer(b,g,r) # g*q\n chart_norm(b,r,F,h,True);b.free(h)\n hm=inverse_stream_noscratch(b,g) # t=1/(g*q)\n half_masks=[scale_defer(b,x[:12],g)for x in (N,L)]\n tm=inverse_stream_noscratch(b,g) # g*q restored\n for x,m in zip((N,L),half_masks):ar.phase_product(b,x[:12],g,m);b.free(m)\n b.x(r[0]);b.cx(F,r[0])\n for x in (N,L):ar.mul_xor0(b,r,x[:12],x[12:]);b.swap(x[:12],x[12:])\n b.cx(F,r[0]);b.x(r[0]);ref.external_square24(b,L)\n rm=b.alloc_bits(12);gg=b.alloc_bits(12);fm=b.alloc_bits(1)\n b.hmr(r,rm);b.hmr(g,gg);b.free(D);b.hmr(F,fm[0]);b.free(F)\n return dict(g=gm,i=im,h=add_masks(b,hm,gg),t=tm,r=rm,f=fm)\n\n\ndef pay_charts(b,D,forward,reverse):\n # Reciprocity preserves F and q, sends r->r+1 (unless F), and\n # (g,1/g,g*q,1/(g*q))->(t,h,1/g,g).\n r,g=D[:12],D[12:]\n F=ref.guarded.fresh_zero_flag(b,g)\n b.cz(F,forward[\'f\'][0]);b.cz(F,reverse[\'f\'][0])\n b.free(forward[\'f\']);b.free(reverse[\'f\'])\n b.z(reverse[\'r\'][0]);b.cz(F,reverse[\'r\'][0])\n masks=dict(g=add_masks(b,forward[\'g\'],reverse[\'t\']),\n            i=add_masks(b,forward[\'i\'],reverse[\'h\']),\n            h=add_masks(b,forward[\'h\'],reverse[\'i\']),\n            t=add_masks(b,forward[\'t\'],reverse[\'g\']),\n            r=add_masks(b,forward[\'r\'],reverse[\'r\']))\n b.cswap(F,r,g)\n oldg=inverse_stream_noscratch(b,g)\n olda=scale_defer(b,r,g)\n im=inverse_stream_noscratch(b,g)\n b.cz(g,oldg);b.free(oldg)\n ar.phase_product(b,r,g,olda);b.free(olda)\n masks[\'i\']=add_masks(b,masks[\'i\'],im)\n b.cz(g,masks[\'g\']);b.free(masks[\'g\'])\n b.cz(r,masks[\'r\']);b.free(masks[\'r\'])\n h=b.alloc_qubits(1)[0];chart_norm(b,r,F,h)\n gm=scale_defer(b,g,r)\n chart_norm(b,r,F,h,True);b.free(h)\n b.cz(g,masks[\'h\']);b.free(masks[\'h\'])\n phase_inverse36_pivot(b,g,masks[\'t\']);b.free(masks[\'t\'])\n # Restore g from g*q, retaining r through the rank11 affine norm chart.\n h=b.alloc_qubits(1)[0];chart_norm(b,r,F,h)\n qm=inverse_stream_noscratch(b,r)\n hm=scale_defer(b,g,r)\n qi=inverse_stream_noscratch(b,r)\n b.cz(r,qm);b.free(qm)\n ar.phase_product(b,g,r,hm);b.free(hm)\n phase_inverse36_pivot(b,r,qi);b.free(qi)\n chart_norm(b,r,F,h,True);b.free(h)\n am=scale_defer(b,r,g)\n b.cz(g,gm);b.free(gm)\n # Fresh old-r mask and both normalizers\' reciprocal-g debts.\n z=inverse_stream_noscratch(b,g)\n b.cz(g,masks[\'i\']);b.free(masks[\'i\'])\n ar.phase_product(b,r,g,am);b.free(am)\n zi=inverse_stream_noscratch(b,g)\n b.cz(g,z);b.free(z)\n phase_inverse36_pivot(b,g,zi);b.free(zi)\n b.cswap(F,r,g);ref.guarded.zero_toggle(b,g,F);b.free(F)\n\n\ndef external_node_chain(b,words,gi,reverse=False,token=None,normalizer=None):\n gnode=ref.groups[gi];N=combine(words[gnode[\'word\']:gnode[\'word\']+2]);L=combine(words[gnode[\'word\']+2:gnode[\'word\']+4])\n heads=[words[2*i:2*i+2]for i in range(2*gi)]\n params=[];c=0\n for i in range(2*gi):\n  q=ref.ks[i]^gnode[\'kap\'];params.append((q,c));c^=inv(q)\n assert c==0\n if not reverse:\n  D=b.alloc_qubits(24);b.x(D[0])\n  for i,(head,(q,c))in enumerate(zip(heads,params)):\n   ref.external_chain_step(b,head,N,D,L,q,gnode[\'c\'],c,first=i==0)\n  pole=zero_flag24(b,D);b.cswap(pole,N,D)\n  norm=normalize(b,N,D,L)\n  return dict(pole=pole,norm=norm)\n pole=token[\'pole\'];D=b.alloc_qubits(24);b.x(D[0]);b.cswap(pole,N,D)\n for head,(q,c)in reversed(list(zip(heads,params))[1:]):ref.external_chain_step(b,head,N,D,L,q,gnode[\'c\'],c,True)\n q0=params[0][0];scalepair(b,N,inv(q0));b.swap(N,D);scalepair(b,L,inv(q0))\n norm=normalize(b,N,D,L)\n a,h=heads[0];cx(b,h,a,gnode[\'c\']);b.cx(combine(heads[0]),N);cx(b,h,a,gnode[\'c\']);b.cx(h,L[:12])\n D=b.alloc_qubits(24);b.x(D[0])\n for i,(head,(q,c))in enumerate(zip(heads,params)):ref.external_scalar_step(b,head,N,D,q,gnode[\'c\'],first=i==0)\n b.cswap(pole,N,D);pay_charts(b,D,token[\'norm\'],norm);b.cswap(pole,N,D)\n clear_flag24(b,D,pole)\n for i in reversed(range(len(heads))):\n  if i==1:scalar_erase_second24(b,heads[i],N,D,params[i][0],gnode[\'c\'],q0)\n  else:ref.external_scalar_step(b,heads[i],N,D,params[i][0],gnode[\'c\'],True,first=i==0)\n b.x(D[0]);b.free(D)\n\n\ndef and_flag(b,x):\n ladder=b.alloc_qubits(len(x)-1)\n b.init_and(x[0],x[1],ladder[0])\n for i in range(1,len(ladder)):b.init_and(ladder[i-1],x[i+1],ladder[i])\n for i in reversed(range(len(ladder)-1)):\n  if i:b.del_and(ladder[i-1],x[i+1],ladder[i])\n  else:b.del_and(x[0],x[1],ladder[0])\n b.free(ladder[:-1]);return ladder[-1]\n\n\ndef phase_and(b,x,m):\n b.push_condition(m)\n nodes=b.alloc_qubits(len(x)-2)\n b.init_and(x[0],x[1],nodes[0])\n for i in range(1,len(nodes)):b.init_and(nodes[i-1],x[i+1],nodes[i])\n b.cz(nodes[-1],x[-1])\n for i in reversed(range(len(nodes))):\n  if i:b.del_and(nodes[i-1],x[i+1],nodes[i])\n  else:b.del_and(x[0],x[1],nodes[0])\n b.free(nodes);b.pop_condition()\n\n\ndef erase_and(b,x,F):\n m=b.alloc_bits(1);b.hmr(F,m[0]);b.free(F);phase_and(b,x,m[0]);b.free(m)\n\n\ndef zero_flag24(b,x):\n b.x(x);a=and_flag(b,x[:12]);c=and_flag(b,x[12:]);F=b.alloc_qubits(1)[0]\n b.init_and(a,c,F);erase_and(b,x[:12],a);erase_and(b,x[12:],c);b.x(x)\n return F\n\n\ndef clear_flag24(b,x,F):\n m=b.alloc_bits(1);b.hmr(F,m[0]);b.free(F)\n b.push_condition(m[0]);b.x(x)\n a=and_flag(b,x[:12]);c=and_flag(b,x[12:]);b.cz(a,c)\n erase_and(b,x[:12],a);erase_and(b,x[12:],c)\n b.x(x);b.pop_condition();b.free(m)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/head_stream.py': '"""Exact low head restore, 12 rather than 24 simultaneous output wires."""\nfrom exp.decoder.common import ar\nfrom direct_kernels import conv,k\n\ndef full_restore12(b,f,g):\n m=len(g);lo=b.alloc_qubits(12)\n with b.mark(\'full_head_product_stream\'):conv(b,g,f[-2:],f[:-2]+[lo],m)\n masks=[b.alloc_bits(12),b.alloc_bits(12)]\n b.hmr(f[-2],masks[0]);b.swap(f[-2],lo);b.free(lo)\n hi=b.alloc_qubits(12);ar.mul_xor0(b,g[-1],f[-1],hi)\n b.hmr(f[-1],masks[1]);b.swap(f[-1],hi);b.free(hi)\n V,W=k.vandermonde(m);field=ar.field(12)\n for j,word in enumerate(f[:-1]):\n  correction=0\n  for t in range(m):correction^=field.mul(W[j][t],field.pow(t,m))\n  k.cx(b,f[-1],word,correction)\n return masks\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/high_stream.py': '"""Separate experiment: full-head product in original divisor coordinates.\n\nFrozen decode_truncated sources are not changed. Nonroot direct levels only.\n"""\nfrom pathlib import Path\nimport sys\nfrom types import SimpleNamespace\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nimport direct_converter as direct\nimport truncated_converter as trunc\nimport root_fusion as rf\nfrom converter import clone\n\n\ndef high_full_restore(b,f,g,F):\n m=len(g);lo=b.alloc_qubits(12)\n with dk.shifted_pair(b,f[-2:],F) as q:\n  dk.conv(b,g,q,f[:-2]+[lo],m)\n masks=[b.alloc_bits(12),b.alloc_bits(12)]\n b.hmr(f[-2],masks[0]);b.swap(f[-2],lo);b.free(lo)\n hi=b.alloc_qubits(12)\n with dk.selected(b,g[-1],g[-2],F) as a:ar.mul_xor0(b,a,f[-1],hi)\n b.hmr(f[-1],masks[1]);b.swap(f[-1],hi);b.free(hi)\n V,W=dk.k.vandermonde(m);field=ar.field(12)\n for j,word in enumerate(f[:-1]):\n  correction=0\n  for t in range(m):correction^=field.mul(W[j][t],field.pow(t,m))\n  dk.k.cx(b,f[-1],word,correction)\n return masks\n\n\ndef high_close_heads(b,f,g,ai,F,masks):\n lo,hi=f[-2:]\n ar.phase_product(b,hi,ai,masks[1]);ar.phase_product(b,lo,ai,masks[0])\n b.cswap(F,g[-2],g[-3]);c=g[-2]\n if True:\n  product=b.alloc_qubits(12);ar.mul_xor0(b,hi,c,product)\n  ar.linear_inplace(b,ai,ar.square_columns(12))\n  ar.phase_product(b,product,ai,masks[0])\n  ar.linear_inplace(b,ai,[ar.field(12).pow(1<<i,2048)for i in range(12)])\n  ar.erase_product(b,hi,c,product);b.free(product)\n b.cswap(F,g[-2],g[-3])\n for mask in masks:b.free(mask)\n\n\ndef direct_pre(b,f,g,payloads,F,undo=False,incoming=None,defer=False):\n if not undo:return ordinary_pre(b,f,g,payloads,F,False,incoming,defer)\n from gap_zero import gap_zero\n Z0=gap_zero(b,g,F);b.cx(Z0,g[-2][0])\n masks=high_full_restore(b,f,g,F)\n # Z0=>F: if F=0 the original top was nonzero. Thus the clipped selected\n # pivot is at the second-top slot whenever the clipping bit is actually1.\n b.cx(Z0,g[-2][0]);pivot(b,g,F);a=g[-1]\n ai,Z=inverse_zero(b,a);b.cx(Z,Z0);b.free(Z0);consume(b,ai,Z,incoming)\n b.cx(Z,a[0]);b.cx(Z,ai[0]);high_close_heads(b,f,g,ai,F,masks)\n for v in payloads:dk.shifted_product(b,g[:-1],v[-1],v[:-1],F)\n for v in payloads:scale(b,v[-1],a,ai,False)\n b.cx(Z,ai[0]);b.cx(Z,a[0]);out=erase(b,a,ai,Z,defer)\n pivot(b,g,F);return out\n\n\ndef direct_post(b,f,g,payloads,F,H,parent,reverse=False,incoming=None,defer=False,phase=None):\n m=len(f)-1;q0,q1=f[-2:]\n if not reverse:\n  q_post(b,q1,payloads,H,m,parent,phase=phase)\n  pivot(b,g,F);a=g[-1]\n  for v in payloads:dk.selected_dot(b,g[:-1],v[:-1],v[-1],F)\n  ai,Z=inverse_zero(b,a);consume(b,ai,Z,incoming)\n  b.cx(Z,a[0]);b.cx(Z,ai[0])\n  for v in payloads:scale(b,v[-1],a,ai,True)\n  b.cx(Z,ai[0]);mask=b.alloc_bits(12);b.hmr(ai,mask);b.free(ai)\n  pivot(b,g,F);phase[\'heads\'][m]=high_full_restore(b,f,g,F)\n  b.cx(Z,g[-2][0]);zm=b.alloc_bits(1);b.hmr(Z,zm[0]);b.free(Z)\n  assert defer\n  return mask,zm\n pivot(b,g,F);a=g[-1]\n def action(ai):\n  scale(b,q1,a,ai,True);cross(b,g,q1,q0,F);scale(b,q0,a,ai,True)\n  for word,mask in zip([q0,q1],phase[\'heads\'].pop(m)):\n   b.cz(word,mask);b.free(mask)\n  dk.shifted_conv(b,g[:-1],[q0,q1],f[:-2],F,True)\n  for v in payloads:scale(b,v[-1],a,ai,False)\n carry=inverse_apply(b,a,action,incoming,defer)\n for v in payloads:dk.selected_dot(b,g[:-1],v[:-1],v[-1],F)\n pivot(b,g,F);q_post(b,q1,payloads,H,m,parent,True,phase)\n return carry\n\n\ndef make_converter():\n original_factory=direct.mh.make_converter\n def factory(return_scope=False):\n  cv,scope,short=original_factory(True);oldconv=scope[\'convolution_xor\']\n  def convolution(b,x,y,z,reverse=False):\n   k=len(z)\n   if k in (2,3)and len(x)==k and len(y)==2:\n    trunc.small.accumulate(b,[q for w in x for q in w],[q for w in y for q in w],[q for w in z for q in w],k)\n   else:oldconv(b,x,y,z,reverse)\n  scope[\'convolution_xor\']=convolution\n  for name in [\'high_full_restore\',\'high_close_heads\']:\n   scope[name]=clone(globals()[name],scope)\n  scope[\'ordinary_pre\']=clone(direct.direct_pre,scope)\n  # These custom functions intentionally replace only high direct levels;\n  # ordinary measured-head low pre/post retain their own high_full_restore globals.\n  short[\'old\']=SimpleNamespace(**scope)\n  short[\'root_restore\']=clone(rf.root_restore,short)\n  short[\'ordinary_root_action\']=clone(direct.root_action,short)\n  return (cv,scope,short)if return_scope else cv\n env=dict(vars(direct),mh=SimpleNamespace(make_converter=factory),\n  direct_pre=direct_pre,direct_post=direct_post,root_action=rf.root_action)\n return clone(direct.make_converter,env)()\nconvert_safe=make_converter()\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/measured_heads.py': '"""NOTE1 full-product quotient-head restoration with exact clipped completion.\n\nNonroot levels only. Forward a inverse is retired before the24 product\noutputs while its zero flag stays live. Reverse restoration gets an11T\nzero flag, absorbed into the next inverse\'s correlated flag. This retiming\npays11T/level to remain within the routed workspace.\n"""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nfrom converter import clone\nfrom direct_kernels import conv\nfrom final_decoder import conversion as stable\nfrom types import SimpleNamespace\n\n\ndef full_restore(b,f,g):\n from head_stream import full_restore12\n return full_restore12(b,f,g)\n\n\ndef close_heads(b,f,g,ai,masks):\n lo,hi=f[-2:];c=g[-2]\n with b.mark(\'head_phase\'):\n  ar.phase_product(b,hi,ai,masks[1]);ar.phase_product(b,lo,ai,masks[0])\n  product=b.alloc_qubits(12);ar.mul_xor0(b,hi,c,product)\n  ar.linear_inplace(b,ai,ar.square_columns(12))\n  ar.phase_product(b,product,ai,masks[0])\n  ar.linear_inplace(b,ai,[ar.field(12).pow(1<<i,2048) for i in range(12)])\n  ar.erase_product(b,hi,c,product);b.free(product)\n for mask in masks:b.free(mask)\n\n\ndef pre(b,f,g,payloads,undo=False,incoming=None,defer=False):\n m=len(f)-1;n=m-1;a=g[-1];q0=f[-2];q1=f[-1]\n if not undo:\n  def action(ai):\n   ops=[(\'s\',v[-1],True)for v in payloads]\n   ops +=[(\'m\',g[i],v[-1],v[i])for v in payloads for i in range(n)]\n   ops +=[(\'s\',q1,True),(\'m\',q1,g[-2],q0),(\'s\',q0,True),(\'c\',g[:-1],[q0,q1],f[:-2])]\n   runops(b,ops,a,ai)\n  return inverse_apply(b,a,action,incoming,defer)\n # The streamed product needs12 outputs, so inverse12 +zero fits here.\n ai,Z=inverse_zero(b,a);consume(b,ai,Z,incoming)\n b.cx(Z,a[0]);b.cx(Z,ai[0]);masks=full_restore(b,f,g)\n close_heads(b,f,g,ai,masks)\n for v in payloads:\n  for i in range(n):ar.mul_xor0(b,g[i],v[-1],v[i])\n for v in payloads:scale(b,v[-1],a,ai,False)\n b.cx(Z,ai[0]);b.cx(Z,a[0])\n return erase(b,a,ai,Z,defer)\n\n\ndef post_look(b,f,g,payloads,H,parent,reverse=False,incoming=None,defer=False,phase=None):\n m=len(f)-1;n=m-1;a=g[-1];q0=f[-2];q1=f[-1]\n def dots():\n  for v in payloads:\n   for i in range(n):ar.mul_xor0(b,g[i],v[i],v[-1])\n if not reverse:\n  q_post(b,q1,payloads,H,m,parent,phase=phase);dots()\n  ai,Z=inverse_zero(b,a);consume(b,ai,Z,incoming)\n  b.cx(Z,a[0]);b.cx(Z,ai[0])\n  for v in payloads:scale(b,v[-1],a,ai,True)\n  b.cx(Z,ai[0]);mask=b.alloc_bits(12);b.hmr(ai,mask);b.free(ai)\n  phase[\'heads\'][m]=full_restore(b,f,g)\n  b.cx(Z,a[0]);zm=b.alloc_bits(1);b.hmr(Z,zm[0]);b.free(Z)\n  assert defer\n  return mask,zm\n def action(ai):\n  runops(b,[(\'s\',q1,True),(\'m\',q1,g[-2],q0),(\'s\',q0,True)],a,ai)\n  for word,mask in zip([q0,q1],phase[\'heads\'].pop(m)):\n   b.cz(word,mask);b.free(mask)\n  convolution_xor(b,g[:-1],[q0,q1],f[:-2],True)\n  for v in payloads:scale(b,v[-1],a,ai,False)\n carry=inverse_apply(b,a,action,incoming,defer);dots()\n q_post(b,q1,payloads,H,m,parent,True,phase)\n return carry\n\n\ndef make_converter(return_scope=False):\n scope=dict(stable.scope)\n scope.update(KEEP_FULL=frozenset(),KEEP_CHILD=frozenset({2,3}))\n for name in [\'scale\',\'runops\',\'zero_toggle\',\'inverse_apply\',\'canonical_permute\',\n              \'masked_product\',\'fresh_zero_flag\',\'base\',\'bez_lookahead\']:\n  scope[name]=clone(scope[name],scope)\n # consume is exported through inverse helper globals, not converter globals.\n scope[\'consume\']=scope[\'inverse_apply_safe\'].__globals__[\'consume\']\n for name,fn in [(\'full_restore\',full_restore),(\'close_heads\',close_heads),(\'pre\',pre),(\'post_look\',post_look)]:\n  scope[name]=clone(fn,dict(scope,conv=conv)) if name==\'full_restore\' else clone(fn,scope)\n scope[\'base_case\']=scope[\'base\']\n from contextlib import contextmanager\n scope[\'q_inverse\']=contextmanager(clone(stable.scope[\'q_inverse\'].__wrapped__,scope))\n scope[\'q_post\']=clone(stable.q_post,scope)\n scope[\'look_solve\']=clone(stable.look_solve,scope)\n short=dict(stable.short,old=SimpleNamespace(**scope),KEEP_FULL=frozenset(),KEEP_CHILD=frozenset({2,3}))\n for name,fn in [(\'cached_root_action\',stable.cached_root_action),(\'look_root_cached\',stable.look_root_cached),(\'_bez_monic\',stable._bez_monic)]:\n  short[name]=clone(fn,short)\n core=clone(stable.base._short_scope[\'convert_safe\'],short)\n def convert_safe(b,A,B,reverse=False,history_scratch12=None,phase_token=None,*,monic_root=True):\n  if phase_token is None:\n   phase_token={\'a\':{},\'q\':{},\'heads\':{},\'_short_monic\':bool(monic_root)}\n   for key in [\'gap_flags\',\'given_gap\',\'incoming_gap\',\'return_mask\',\'returned_gap\',\'held_gap\']:phase_token[key]={}\n  out=core(b,A,B,reverse,history_scratch12,phase_token,monic_root=monic_root)\n  if reverse:assert not phase_token[\'heads\']\n  return out\n return (convert_safe,scope,short) if return_scope else convert_safe\nconvert_safe=make_converter()\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/norm_cache.py': '"""Retain the most profitable norm coordinates within the live width ceiling."""\nfrom functools import lru_cache\nfrom exp.decoder.common import ar\nfrom exp.noscratch.kernels import rankone_xor\n\nTRACE=[]\nFORCE=None\n\n@lru_cache(None)\ndef layout(k):\n    # Enumerate subspaces, not ordered bases. This is only six dimensions.\n    tensor=ar.tensor(6)\n    spaces={frozenset({0}):()}\n    for dim in range(k):\n        nxt={}\n        for span,basis in spaces.items():\n            for w in range(1,64):\n                if w in span:continue\n                full=span|{a^w for a in span}\n                if full not in nxt:nxt[full]=basis+(w,)\n        spaces=nxt\n    span,columns=max(spaces.items(),key=lambda item:(sum(v in item[0] for _,v in tensor),item[1]))\n    columns=list(columns)\n    for w in (1,2,4,8,16,32):\n        if w not in span:\n            columns.append(w);span=span|{a^w for a in span}\n    return tuple(ar.inverse_columns(columns)),tuple(columns)\n\ndef keep(b,D):\n    # D is already live; its streamed inverse needs six further wires.\n    live=b.num_allocated_qubits\n    k=min(6,max(0,889-live-6)) if FORCE is None else FORCE\n    to,frm=layout(k)\n    saved=b.alloc_qubits(k)\n    ar.linear_xor(b,D,saved,[v&((1<<k)-1) for v in to])\n    skipped=sum(ar.apply(to,v)>>k==0 for _,v in ar.tensor(6))\n    TRACE.append(dict(live_with_D=live,kept=k,skipped=skipped,expected_peak=live+6+k))\n    return saved\n\ndef rebuild(b,g,saved,K):\n    k=len(saved);to,frm=layout(k)\n    rest=b.alloc_qubits(6-k)\n    square=ar.square_columns(6)\n    ar.linear_xor(b,g[:6],rest,[ar.apply(to,v)>>k for v in square])\n    ar.linear_xor(b,g[6:],rest,[ar.apply(to,ar.field(6).mul(K,v))>>k for v in square])\n    for mask,vector in ar.tensor(6):\n        vector=ar.apply(to,vector)>>k\n        if not vector:continue\n        ix=list(ar.bits(mask));p=ix[0]\n        for j in ix[1:]:b.cx(g[j],g[p]);b.cx(g[j+6],g[p+6])\n        rankone_xor(b,g[p],g[p+6],rest,vector)\n        for j in reversed(ix[1:]):b.cx(g[j+6],g[p+6]);b.cx(g[j],g[p])\n    D=list(saved)+list(rest)\n    ar.linear_inplace(b,D,frm)\n    return D\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/norm_keep2.py': '"""Keep two norm coordinates through streamed inversion.\n\nOnly terms with output outside their span are recomputed.  The old M6\nmeasurement debt is still paid in its original polynomial coordinates.\n"""\nfrom functools import lru_cache\nfrom exp.decoder.common import ar\nfrom exp.noscratch.kernels import rankone_xor\n\n@lru_cache(None)\ndef layout():\n    tensor=ar.tensor(6)\n    _,u,v=max((sum(w in (u,v,u^v) for _,w in tensor),u,v)\n              for u in range(1,64) for v in range(u+1,64))\n    columns=[u,v]\n    span={0,u,v,u^v}\n    for w in (1,2,4,8,16,32):\n        if w not in span:\n            columns.append(w);span|={a^w for a in tuple(span)}\n    return tuple(ar.inverse_columns(columns)),tuple(columns)\n\ndef keep(b,M):\n    to,_=layout()\n    kept=b.alloc_qubits(2)\n    ar.linear_xor(b,M,kept,[v&3 for v in to])\n    return kept\n\ndef rebuild(b,N,kept,K):\n    to,frm=layout()\n    rest=b.alloc_qubits(4)\n    square=ar.square_columns(6)\n    ar.linear_xor(b,N[:6],rest,[ar.apply(to,v)>>2 for v in square])\n    ar.linear_xor(b,N[6:],rest,[ar.apply(to,ar.field(6).mul(K,v))>>2 for v in square])\n    for mask,vector in ar.tensor(6):\n        vector=ar.apply(to,vector)>>2\n        if not vector:continue\n        ix=list(ar.bits(mask));p=ix[0]\n        for j in ix[1:]:b.cx(N[j],N[p]);b.cx(N[j+6],N[p+6])\n        rankone_xor(b,N[p],N[p+6],rest,vector)\n        for j in reversed(ix[1:]):b.cx(N[j+6],N[p+6]);b.cx(N[j],N[p])\n    M=list(kept)+list(rest)\n    ar.linear_inplace(b,M,frm)\n    return M\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/phase_core.py': '"""Direct G/I/H/R chart characters; T=1/(gq) is deliberately NOT paid."""\nfrom pathlib import Path\nimport sys\nroot=Path(__file__).resolve().parents[3]\nsys.path[:0]=[str(root/\'playbook/kit/decoder\'),str(root/\'playbook/kit\')]\nfrom exp.decoder.common import *\nfrom exp.noscratch.schedules import inverse_stream_noscratch\nfrom exp.mul99.phase36 import phase_inverse36_pivot\n\ndef phase_core(b,A,g,F,masks):\n """Pay G.g+I/g+H.(gq)+R.(A/g), q=(A/g)^2+A/g+K+F(K+1).\n\n Promise g!=0, and F=>A=0. Preserve A,g,F and every classical mask.\n 168T/12 private, no ratio materialization. Does not pay T/(gq).\n """\n b.cz(g,masks[\'g\']);b.cz(A,masks[\'h\'])\n f=ar.field(12)\n for j in range(12):\n  for k in ar.bits(f.mul(K,1<<j)):b.cz(g[j],masks[\'h\'][k])\n  for k in ar.bits(f.mul(K^1,1<<j)):b.ccz(F,g[j],masks[\'h\'][k])\n z=inverse_stream_noscratch(b,g)\n b.cz(g,masks[\'i\']);ar.phase_product(b,A,g,masks[\'r\'])\n ar.linear_inplace(b,A,ar.square_columns(12));ar.phase_product(b,A,g,masks[\'h\']);ar.linear_inplace(b,A,[ar.field(12).pow(1<<j,2048)for j in range(12)])\n zi=inverse_stream_noscratch(b,g)\n b.cz(g,z);b.free(z);phase_inverse36_pivot(b,g,zi);b.free(zi)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/phase_quantum.py': '"""Direct quantum-coefficient inverse12 character:90T/12private."""\nfrom phase_core import *\nfrom types import FunctionType\nfrom exp.noscratch.schedules import norm_xor_noscratch\nfrom exp.arithmetic.lowwidth import phase_product_quantum,phase_inverse_generic6\nfrom exp.noscratch.kernels import inverse3_noscratch\nphase_inverse6_9=FunctionType(phase_inverse_generic6.__code__,dict(phase_inverse_generic6.__globals__,inverse=inverse3_noscratch),\'phase_inverse6_9\')\n\ndef phase_inverse12_quantum90(b,x,co):\n to,frm,k=ar.tower(12);ar.linear_inplace(b,x,to)\n tr=[sum(((frm[j]>>i)&1)<<j for j in range(12))for i in range(12)]\n ar.linear_inplace(b,co,tr)\n N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,k)\n nm=inverse_stream_noscratch(b,N);ar._phase_norm(b,x,nm,k);b.free(nm)\n b.cx(x[6:],x[:6]);phase_product_quantum(b,x[:6],N,co[:6]);phase_product_quantum(b,x[6:],N,co[6:]);b.cx(x[6:],x[:6])\n im=b.alloc_bits(6);b.hmr(N,im);b.free(N)\n N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,k);phase_inverse6_9(b,N,im);b.free(im)\n ar._erase_norm(b,x,N,k);b.free(N);ar.linear_inplace(b,x,frm);ar.linear_inplace(b,co,ar.inverse_columns(tr))\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/rational.py': '"""Exact rational character baselines; all physical banks are counted.\n\nNo circuit in this module is claimed to meet the <=15-fresh target yet.\nThe source-preserving composition is 132 executed T / 24 fresh qubits.\n"""\nfrom pathlib import Path\nimport sys\n\nROOT = Path(__file__).resolve().parents[3]\nsys.path[:0] = [str(ROOT / \'exp/r-main/narrow240\'), str(ROOT / \'playbook/kit/decoder\'),\n                str(ROOT / \'playbook/kit\')]\nfrom phase_quantum import phase_inverse12_quantum90\nfrom phase_core import ar, K\nfrom exp.noscratch.kernels import mul_xor_noscratch\n\n\ndef norm_source(b, A, g, out, F=None):\n    """out ^= A^2 + A*g + (K+F*(K+1))*g^2, 42T/0 scratch."""\n    f = ar.field(12)\n    ar.linear_xor(b, A, out, ar.square_columns(12))\n    ar.linear_xor(b, g, out, [f.mul(K, v) for v in ar.square_columns(12)])\n    if F is not None:\n        for j, v in enumerate(ar.square_columns(12)):\n            for k in ar.bits(f.mul(K ^ 1, v)):\n                b.ccx(F, g[j], out[k])\n    mul_xor_noscratch(b, A, g, out)\n\n\ndef erase_norm_source(b, A, g, out, F=None):\n    """Every source measurement is closed, including the F*g^2 term."""\n    m = b.alloc_bits(12)\n    b.hmr(out, m)\n    f = ar.field(12)\n    ar.phase_linear(b, A, m, ar.square_columns(12))\n    ar.phase_linear(b, g, m, [f.mul(K, v) for v in ar.square_columns(12)])\n    if F is not None:\n        for j, v in enumerate(ar.square_columns(12)):\n            for k in ar.bits(f.mul(K ^ 1, v)):\n                b.ccz(F, g[j], m[k])\n    ar.phase_product(b, A, g, m)\n    b.free(m)\n\n\ndef coefficient_ops(b, mask):\n    """Classical elimination transcript for L_m(g)[j] = m dot (g*2^j).\n\n    For m!=0, L_m is invertible by nondegeneracy of the field product.\n    The transcript contains only classical controls, never quantum data.\n    Return its row-reduction operations and allocated classical registers.\n    Reverse order applies L_m; forward order restores g.\n    """\n    n = len(mask)\n    f = ar.field(n)\n    rows = [b.alloc_bits(n) for _ in range(n)]\n    for j in range(n):\n        b.bit_store(rows[j], False)\n        for i in range(n):\n            for k in ar.bits(f.mul(1 << i, 1 << j)):\n                b.cx(mask[k], rows[j][i])\n    ops = []\n    allocated = list(rows)\n    for j in range(n):\n        seen = b.alloc_bits(1)[0]\n        b.bit_store(seen, False)\n        allocated.append([seen])\n        for i in range(j, n):\n            p = b.alloc_bits(1)[0]\n            allocated.append([p])\n            b.bit_store(p, False)\n            b.x(seen)\n            b.push_condition(seen)\n            b.cx(rows[i][j], p)\n            b.pop_condition()\n            b.x(seen)\n            b.bit_store(seen, True, control=p)\n            if i != j:\n                b.push_condition(p)\n                b.swap(rows[i], rows[j])\n                b.pop_condition()\n                ops.append((\'swap\', i, j, p))\n        for i in range(n):\n            if i == j:\n                continue\n            c = b.alloc_bits(1)[0]\n            allocated.append([c])\n            b.bit_store(c, False)\n            b.cx(rows[i][j], c)\n            b.push_condition(c)\n            b.cx(rows[j], rows[i])\n            b.pop_condition()\n            ops.append((\'cx\', j, i, c))\n    return ops, allocated\n\n\ndef apply_coefficient_ops(b, g, ops, reverse=False):\n    for op, i, j, cond in (reversed(ops) if reverse else ops):\n        b.push_condition(cond)\n        getattr(b, op)(g[i], g[j])\n        b.pop_condition()\n\n\ndef nonzero_classical(b, mask):\n    nz = b.alloc_bits(1)[0]\n    b.bit_store(nz, False)\n    for q in mask:\n        b.bit_store(nz, True, control=q)\n    return nz\n\n\ndef phase_rational_baseline(b, A, g, mask, F=None):\n    """Phase mask dot g/Norm(A,g), exact; 24 fresh, 132T at F absent.\n\n    With F present the source F*g^2 CCXs add 53 static/mean T for the\n    supplied K and tower basis. The F-free lead split avoids those gates.\n    """\n    nz = nonzero_classical(b, mask)\n    b.push_condition(nz)\n    N = b.alloc_qubits(12)\n    with b.mark(\'source\'):\n        norm_source(b, A, g, N, F)\n    ops, allocated = coefficient_ops(b, mask)\n    apply_coefficient_ops(b, g, ops, reverse=True)\n    with b.mark(\'inverse_character\'):\n        phase_inverse12_quantum90(b, N, g)\n    apply_coefficient_ops(b, g, ops)\n    for reg in allocated:\n        b.free(reg)\n    with b.mark(\'source_cleanup\'):\n        erase_norm_source(b, A, g, N, F)\n    b.free(N)\n    b.pop_condition()\n    b.free(nz)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/replay_erase.py': '"""Erase the first nonlinear replay step through its still-live operands.\n\nThis changes neither homogeneous forward values nor any completion/failure set.\nThe scalar replay\'s second step adds a product to a known constant, so its last\nnonlinear reverse operation can use HMR plus a bilinear phase (0 T).\n"""\nfrom pathlib import Path\nimport sys\nfrom types import FunctionType\nROOT=Path(__file__).resolve().parents[3]\nsys.path[:0]=[str(ROOT/\'playbook/kit/decoder\'),str(ROOT/\'playbook/kit\')]\nfrom final_decoder import decode as ref\nfrom exp.decoder.common import *\nfrom exp.arithmetic.division import phase_product24\nfrom final_decoder.conversion import clear_flag\n\n\ndef scalar_erase_second24(b,head,N,D,q,ct,q0):\n    """Reverse second scalar step; its pre-step N is known q0, D arbitrary."""\n    scalepair(b,N,inv(q));b.swap(N,D)\n    for j in ar.bits(q0):b.x(N[j])\n    m=b.alloc_bits(24);b.hmr(N,m)\n    a,h=head;cx(b,h,a,ct);phase_product24(b,combine(head),D,m,K);cx(b,h,a,ct)\n    b.free(m)\n    for j in ar.bits(q0):b.x(N[j])\n\n\ndef external_node_chain(b,words,gi,reverse=False,token=None,normalizer=None):\n    if not reverse:return ref.external_node_chain(b,words,gi,False,None,normalizer)\n    g=ref.groups[gi];N=combine(words[g[\'word\']:g[\'word\']+2]);L=combine(words[g[\'word\']+2:g[\'word\']+4])\n    heads=[words[2*i:2*i+2]for i in range(2*gi)]\n    params=[];c=0\n    for i in range(2*gi):\n        q=ref.ks[i]^g[\'kap\'];assert q\n        params.append((q,c));c^=inv(q)\n    assert c==0\n    pole=token[\'pole\'];D=b.alloc_qubits(24);b.x(D[0]);b.cswap(pole,N,D)\n    for head,(q,c)in reversed(list(zip(heads,params))[1:]):ref.external_chain_step(b,head,N,D,L,q,g[\'c\'],c,True)\n    q0=params[0][0];scalepair(b,N,inv(q0));b.swap(N,D);scalepair(b,L,inv(q0))\n    norm=ref.external_normalize_paired(b,N,D,L,True,token[\'norm\'])\n    a,h=heads[0];cx(b,h,a,g[\'c\']);b.cx(combine(heads[0]),N);cx(b,h,a,g[\'c\']);b.cx(h,L[:12])\n    invmask=b.alloc_bits(24);b.hmr(D,invmask);b.free(D)\n    D=b.alloc_qubits(24);b.x(D[0])\n    for i,(head,(q,c))in enumerate(zip(heads,params)):ref.external_scalar_step(b,head,N,D,q,g[\'c\'],first=i==0)\n    b.cswap(pole,N,D);b.cz(D,token[\'gauge\']);b.free(token[\'gauge\'])\n    reciprocal,direct=norm\n    ref.external_inverse_phase_norm_pair(b,D,invmask,reciprocal,direct);b.free(invmask);b.cswap(pole,N,D)\n    clear_flag(b,D,pole)\n    for i in reversed(range(len(heads))):\n        if i==1:scalar_erase_second24(b,heads[i],N,D,params[i][0],g[\'c\'],q0)\n        else:ref.external_scalar_step(b,heads[i],N,D,params[i][0],g[\'c\'],True,first=i==0)\n    b.x(D[0]);b.free(D)\n\nexternal_extraction=FunctionType(ref.external_extraction.__code__,dict(ref.external_extraction.__globals__,external_node_chain=external_node_chain),\'external_extraction\',ref.external_extraction.__defaults__,ref.external_extraction.__closure__)\nextraction=FunctionType(ref.extraction.__code__,dict(ref.extraction.__globals__,external_extraction=external_extraction),\'extraction\',ref.extraction.__defaults__,ref.extraction.__closure__)\ndecode=FunctionType(ref.decode.__code__,dict(ref.decode.__globals__,extraction=extraction),\'decode\',ref.decode.__defaults__,ref.decode.__closure__)\n\n\ndef scalar_erase_second12(b,head,N,D,q,q0):\n    """Subfield version for the trailing zero-node scalar replay."""\n    scale(b,N,inv(q));b.swap(N,D)\n    for j in ar.bits(q0):b.x(N[j])\n    ar.erase_product(b,head[0],D,N)\n    for j in ar.bits(q0):b.x(N[j])\n\n\ndef extraction(b,words,reverse=False,token=None):\n    if not reverse:return ref.extraction(b,words)\n    heads=[words[2*i:2*i+2]for i in range(34)];N,L=words[-2:]\n    pole=token[\'pole\'];ref.controlled_inverse(b,L,pole,True,token[\'chart_inverse\'])\n    D=b.alloc_qubits(12);b.x(D[0]);b.cswap(pole,N,D)\n    for head,kap,c in reversed(list(zip(heads,ref.ks,ref.corrections))[1:]):ref.step(b,head,N,D,L,kap,c,True)\n    scale(b,N,inv(ref.ks[0]));b.swap(N,D);scale(b,L,inv(ref.ks[0]))\n    extra_z_mask=ref.normal_form(b,N,D,L)\n    b.cx(heads[0][0],N);b.cx(heads[0][1],L)\n    b.cz(D,token[\'normalization\']);b.free(token[\'normalization\'])\n    inverse_mask=b.alloc_bits(12);b.hmr(D,inverse_mask);b.free(D)\n    z_mask=token[\'gauge_mask\'];b.cx(extra_z_mask,z_mask);b.free(extra_z_mask)\n    D=b.alloc_qubits(12);b.x(D[0]);ref.scalar_step(b,heads[0],N,D,ref.ks[0],first=True)\n    for head,kap in zip(heads[1:],ref.ks[1:]):ref.scalar_step(b,head,N,D,kap)\n    b.cswap(pole,N,D);b.cz(D,z_mask);b.free(z_mask)\n    ref.phase_inverse34(b,D,inverse_mask);b.free(inverse_mask);b.cswap(pole,N,D)\n    ref.guarded.zero_toggle(b,D,pole)\n    for i in reversed(range(1,len(heads))):\n        if i==1:scalar_erase_second12(b,heads[i],N,D,ref.ks[i],ref.ks[0])\n        else:ref.scalar_step(b,heads[i],N,D,ref.ks[i],True)\n    ref.scalar_step(b,heads[0],N,D,ref.ks[0],True,first=True)\n    b.x(D[0]);b.free(D);b.free(pole);external_extraction(b,words,True,token[\'external\'])\n\n\ndecode=FunctionType(ref.decode.__code__,dict(ref.decode.__globals__,extraction=extraction),\'decode\',ref.decode.__defaults__,ref.decode.__closure__)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/retained_core_safe27.py': '"""Retain the reverse chart, close its ledger, HMR a linear direct gauge."""\nimport frozen_chart as frozen\nfrom frozen_chart import *\nfrom core90 import affine_inverse_g_character\nfrom exact_subs import terminal_two_phase\nimport core90 as core\nimport exact_subs as xsub\nimport inspect\n\n_source=inspect.getsource(frozen.normalize)\n_source=_source.replace(\'def normalize(b,N,D,L):\',\'def normalize_retained(b,N,D,L):\')\n_start=_source.index(\' rm=b.alloc_bits(12);gg=\')\n_source=_source[:_start]+\'\'\' return dict(g=gm,i=im,h=hm,t=tm),F\n\'\'\'\nexec(compile(_source,__file__,\'exec\'))\n\n_source2=_source.replace(\'def normalize_retained\',\'def normalize_early\')\n_lo=_source2.index(\' tm=inverse_stream_noscratch\')\n_hi=_source2.index(\' b.x(r[0]);b.cx(F,r[0])\',_lo)\n_source2=_source2[:_lo]+_source2[_hi:]\n_source2=_source2.replace(\'return dict(g=gm,i=im,h=hm,t=tm),F\',\'return dict(g=gm,i=im,h=hm,half=half_masks),F\')\nexec(compile(_source2,__file__,\'exec\'))\n\ndef linear_inverse_character(b,g,constant,terms):\n to,frm,kval=ar.tower(12);ar.linear_inplace(b,g,to)\n D=b.alloc_qubits(6);core.norm_xor_noscratch(b,g,D,kval)\n saved=core.norm_cache.keep(b,D)\n dm=inverse_stream_noscratch(b,D);ar._phase_norm(b,g,dm,kval);b.free(dm)\n f=ar.field(12);b.cx(g[6:],g[:6])\n source=[q for x,m in terms for q in x]\n for seg in (0,1):\n  x=g[:6]if seg==0 else g[6:]\n  for sm,vv in ar.tensor(6):\n   h=ar.apply(frm,vv<<(6*seg));co=b.alloc_bits(len(source));offset=b.alloc_bits(1)[0]\n   b.bit_store(co,False);b.bit_store(offset,False)\n   for j in ar.bits(h):b.cx(constant[j],offset)\n   for off,(xx,mm)in enumerate(terms):\n    for i in range(12):\n     for j in ar.bits(f.mul(1<<i,h)):b.cx(mm[j],co[off*12+i])\n   ix=list(ar.bits(sm));p=ix[0]\n   for i in ix[1:]:b.cx(x[i],x[p]);b.cx(D[i],D[p])\n   b.ccz(x[p],D[p],offset);core.phase_classical_parity(b,x[p],D[p],source,co)\n   for i in reversed(ix[1:]):b.cx(D[i],D[p]);b.cx(x[i],x[p])\n   b.free(offset);b.free(co)\n b.cx(g[6:],g[:6]);im=b.alloc_bits(6);b.hmr(D,im);b.free(D)\n D=core.norm_cache.rebuild(b,g,saved,kval)\n core.phase_inverse6_9(b,D,im);b.free(im);ar._erase_norm(b,g,D,kval);b.free(D);ar.linear_inplace(b,g,frm)\n\n@lru_cache(None)\ndef merged_chart_columns():\n f=ar.field(12)\n cols=[(f.square(1<<i)^(1<<i))|((1 if i==0 else 0)<<12)for i in range(12)]+[K^1]\n return tuple(cols),tuple(ar.inverse_columns(cols))\n\ndef merged_chart(b,r,F,reverse=False):\n cols,back=merged_chart_columns();w=list(r)+[F]\n if not reverse:\n  ar.linear_inplace(b,w,cols)\n  for i in ar.bits(K):b.x(r[i])\n else:\n  for i in ar.bits(K):b.x(r[i])\n  ar.linear_inplace(b,w,back)\n\ndef direct_chart(b,D,F,forward,reverse,N=None,L=None):\n r,g=D[:12],D[12:]\n b.cz(F,forward[\'f\'][0]);b.free(forward[\'f\'])\n # r_reverse=r_forward+1-F, g_reverse*q_reverse=1/g_forward.\n b.x(r[0]);b.cx(F,r[0])\n masks=dict(g=forward[\'g\'] if N is not None else add_masks(b,forward[\'g\'],reverse[\'t\']),\n            i=add_masks(b,forward[\'i\'],reverse[\'h\']),\n            h=add_masks(b,forward[\'h\'],reverse[\'i\']),\n            t=add_masks(b,forward[\'t\'],reverse[\'g\']),r=forward[\'r\'])\n if N is None:\n  b.cz(g,masks[\'i\']);b.free(masks[\'i\'])\n  im=inverse_stream_noscratch(b,g)\n  phase_inverse36_pivot(b,g,im);b.free(im)\n else:\n  ref.external_square24(b,L,True)\n  linear_inverse_character(b,g,masks[\'i\'],[(N[12:],reverse[\'half\'][0]),(L[12:],reverse[\'half\'][1])])\n  ref.external_square24(b,L)\n  b.free(masks[\'i\'])\n  for m in reverse[\'half\']:b.free(m)\n b.cz(g,masks[\'g\']);b.free(masks[\'g\'])\n b.cz(r,masks[\'r\']);b.free(masks[\'r\'])\n # Include F in the invertible rank13 chart: no extra kernel bit.\n merged_chart(b,r,F)\n gm=scale_defer(b,g,r)\n b.cz(g,masks[\'h\']);b.free(masks[\'h\'])\n phase_inverse36_pivot(b,g,masks[\'t\']);b.free(masks[\'t\'])\n merged_chart(b,r,F,True)\n rm=b.alloc_bits(12);hm=b.alloc_bits(12);fm=b.alloc_bits(1)\n b.hmr(r,rm);b.hmr(g,hm);b.hmr(F,fm[0]);b.free(D);b.free(F)\n return dict(g=gm,r=rm,h=hm,f=fm)\n\ndef zero_affine_character(b,g,A,gm,fm):\n # [g=0] * (fm XOR dot(A,gm)), 11T/12fresh, exact at g=0.\n C=b.alloc_qubits(1)[0];b.cx(fm,C)\n for a,m in zip(A,gm):b.ccx(a,m,C)\n b.x(g);ladder=b.alloc_qubits(len(g)-1)\n b.init_and(g[0],g[1],ladder[0])\n for j in range(1,len(ladder)):b.init_and(ladder[j-1],g[j+1],ladder[j])\n b.cz(ladder[-1],C)\n for j in reversed(range(len(ladder))):\n  if j:b.del_and(ladder[j-1],g[j+1],ladder[j])\n  else:b.del_and(g[0],g[1],ladder[0])\n b.free(ladder);b.x(g)\n for a,m in zip(A,gm):b.ccx(a,m,C)\n b.cx(fm,C);b.free(C)\n\ndef full_core_phase(b,N,heads,params,ct,masks,pole,dirty):\n D=b.alloc_qubits(24);b.x(D[0])\n for i,(head,(q,c))in enumerate(zip(heads,params)):\n  ref.external_scalar_step(b,head,N,D,q,ct,first=i==0)\n b.cswap(pole,N,D)\n A,g=D[:12],D[12:]\n zero_affine_character(b,g,A,masks[\'g\'],masks[\'f\'][0])\n zero=b.alloc_bits(12);b.bit_store(zero,False)\n masks[\'i\']=zero\n # Unswapped chart: inv(0)=0 and F implies g=0.\n b.cz(g,masks[\'g\']);b.cz(A,masks[\'h\'])\n f=ar.field(12)\n for j in range(12):\n  for k in ar.bits(f.mul(K,1<<j)):b.cz(g[j],masks[\'h\'][k])\n core.affine_inverse_g_character(b,A,g,masks)\n for m in masks.values():b.free(m)\n b.cswap(pole,N,D)\n fm=b.alloc_bits(1);b.hmr(pole,fm[0]);b.free(pole)\n b.push_condition(fm[0]);clean_zero_phase24(b,D);b.pop_condition();b.free(fm)\n from replay_erase import scalar_erase_second24\n for i in reversed(range(len(heads))):\n  if i==1:scalar_erase_second24(b,heads[i],N,D,params[i][0],ct,params[0][0])\n  else:ref.external_scalar_step(b,heads[i],N,D,params[i][0],ct,True,first=i==0)\n b.x(D[0]);b.free(D)\n\ndef terminal_one_phase(b,N,heads,params,ct,mask,pole,dirty):\n stop=len(heads)-1;D=b.alloc_qubits(24);b.x(D[0])\n for i in range(stop):ref.external_scalar_step(b,heads[i],N,D,params[i][0],ct,first=i==0)\n b.cz(N,mask);xsub.head_phase(b,heads[-1],D,mask,ct)\n scalepair(b,D,params[-1][0]);b.ccz(pole,D,mask);scalepair(b,D,inv(params[-1][0]))\n fm=b.alloc_bits(1);b.hmr(pole,fm[0]);b.free(pole)\n b.push_condition(fm[0])\n ref.external_scalar_step(b,heads[-1],N,D,params[-1][0],ct)\n clean_zero_phase24(b,D)\n ref.external_scalar_step(b,heads[-1],N,D,params[-1][0],ct,True)\n b.pop_condition();b.free(fm);b.free(mask)\n from replay_erase import scalar_erase_second24\n for i in reversed(range(stop)):\n  if i==1:scalar_erase_second24(b,heads[i],N,D,params[i][0],ct,params[0][0])\n  else:ref.external_scalar_step(b,heads[i],N,D,params[i][0],ct,True,first=i==0)\n b.x(D[0]);b.free(D)\n\ndef clean_zero_phase24_direct(b,x):\n # 22T/22scratch, only used after pole qubit is freed: peak <=886.\n b.x(x);nodes=b.alloc_qubits(len(x)-2)\n b.init_and(x[0],x[1],nodes[0])\n for j in range(1,len(nodes)):b.init_and(nodes[j-1],x[j+1],nodes[j])\n b.cz(nodes[-1],x[-1])\n for j in reversed(range(len(nodes))):\n  if j:b.del_and(nodes[j-1],x[j+1],nodes[j])\n  else:b.del_and(x[0],x[1],nodes[0])\n b.free(nodes);b.x(x)\n\ndef clean_zero_phase24_mixed(b,x):\n # Six AND3 + three AND2 flags. Produce15T, phase AND9 costs7T,\n # repair six AND3 flags at mean0.5T each; AND2 flags del_and freely.\n # Total25T; flag bank9 + phase ladder7 =16 clean scratch.\n b.x(x)\n groups=[x[j:j+3]for j in range(0,18,3)]+[x[j:j+2]for j in range(18,24,2)]\n flags=[frozen.and_flag(b,group)for group in groups]\n nodes=b.alloc_qubits(7);b.init_and(flags[0],flags[1],nodes[0])\n for j in range(1,7):b.init_and(nodes[j-1],flags[j+1],nodes[j])\n b.cz(nodes[-1],flags[-1])\n for j in reversed(range(1,7)):b.del_and(nodes[j-1],flags[j+1],nodes[j])\n b.del_and(flags[0],flags[1],nodes[0]);b.free(nodes)\n for group,F in zip(groups,flags):\n  if len(group)==2:b.del_and(group[0],group[1],F);b.free(F)\n  else:frozen.erase_and(b,group,F)\n b.x(x)\n\nclean_zero_phase24=clean_zero_phase24_mixed\n\ndef zero_flag24_grouped(b,x):\n # Eight AND3 flags (16T), AND8 (7T), eight repairs (mean4T).\n # Peak15 scratch, no chart value changes.\n b.x(x);groups=[x[j:j+3]for j in range(0,24,3)]\n flags=[frozen.and_flag(b,group)for group in groups]\n F=frozen.and_flag(b,flags)\n for group,flag in zip(groups,flags):frozen.erase_and(b,group,flag)\n b.x(x);return F\n\nfrom adaptive_zero import zero_flag24 as zero_flag24_grouped,clean_zero_phase24\nforward_chain=frozen.clone(frozen.external_node_chain,dict(frozen.external_node_chain.__globals__,zero_flag24=zero_flag24_grouped))\n\ndef external_node_chain(b,words,gi,reverse=False,token=None,normalizer=None):\n if not reverse:return forward_chain(b,words,gi,False,token,normalizer)\n gn=ref.groups[gi];N=combine(words[gn[\'word\']:gn[\'word\']+2]);L=combine(words[gn[\'word\']+2:gn[\'word\']+4])\n heads=[words[2*i:2*i+2]for i in range(2*gi)]\n params=[];c=0\n for i in range(2*gi):\n  q=ref.ks[i]^gn[\'kap\'];params.append((q,c));c^=inv(q)\n assert c==0\n pole=token[\'pole\'];D=b.alloc_qubits(24);b.x(D[0]);b.cswap(pole,N,D)\n for head,(q,c)in reversed(list(zip(heads,params))[1:]):ref.external_chain_step(b,head,N,D,L,q,gn[\'c\'],c,True)\n q0=params[0][0];scalepair(b,N,inv(q0));b.swap(N,D);scalepair(b,L,inv(q0))\n with b.mark(\'retained_normalize\'):norm,F=normalize_early(b,N,D,L)\n with b.mark(\'retained_direct\'):mask=direct_chart(b,D,F,token[\'norm\'],norm,N,L)\n a,h=heads[0];cx(b,h,a,gn[\'c\']);b.cx(combine(heads[0]),N);cx(b,h,a,gn[\'c\']);b.cx(h,L[:12])\n with b.mark(\'retained_terminal\'):terminal_two_phase(b,N,heads,params,gn[\'c\'],mask,pole,L)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/retimed.py': '"""Retire the paid N1 source and retain N0 instead:250T/15fresh.\n\nThe earlier cached287 graph and reviewed streamed.py remain unchanged.\nThis is one completed retiming, not a family-level count claim.\n"""\nfrom streamed import *\n\n\ndef scale6_log_key(b, x, key, inverse=False):\n    """Both GF8 halves times alpha^(+/-key),18T, no scratch."""\n    f = ar.field(3)\n    ops = linear_ops([f.mul(2, 1 << i) for i in range(3)])\n    for j in range(3):\n        before = [f.pow(1 << i, 1 << ((3 - j) % 3)) for i in range(3)]\n        after = [f.pow(1 << i, 1 << j) for i in range(3)]\n        for half in (x[:3], x[3:]):\n            ar.linear_inplace(b, half, before)\n            for op, i, t in (reversed(ops) if inverse else ops):\n                if op == \'swap\':\n                    b.cswap(key[j], half[i], half[t])\n                else:\n                    b.ccx(key[j], half[i], half[t])\n            ar.linear_inplace(b, half, after)\n\n\ndef inverse6_context_begin27(b, x):\n    """x->inv0(x), retain log(inv(norm(oldx)))3;27T/3fresh."""\n    ar.linear_inplace(b, x, TO6)\n    key = b.alloc_qubits(3)\n    norm_xor_noscratch(b, x, key, K6)\n    permutation3(b, key, \'log_inverse\')\n    b.cx(x[3:], x[:3])\n    scale6_log_key(b, x, key)\n    ar.linear_inplace(b, x, FROM6)\n    return key\n\n\ndef inverse6_context_end21(b, x, key):\n    """Use the retained log key, invert back and close its quadratic mask."""\n    ar.linear_inplace(b, x, TO6)\n    b.cx(x[3:], x[:3])\n    scale6_log_key(b, x, key, inverse=True)\n    permutation3(b, key, \'log_inverse\', reverse=True)\n    ar._erase_norm(b, x, key, K6)\n    b.free(key)\n    ar.linear_inplace(b, x, FROM6)\n\n\ndef source_low6(b, A, g, target):\n    """Low GF64 half of Norm24(A,g),30T, source bank6 included."""\n    source_linear(b, A, g, target, 0)\n    ar.linear_inplace(b, A, TO12)\n    ar.linear_inplace(b, g, TO12)\n    mul_xor_noscratch(b, A[:6], g[:6], target)\n    scale_constant(b, target, ar.field(6).invert(K12))\n    mul_xor_noscratch(b, A[6:], g[6:], target)\n    scale_constant(b, target, K12)\n    ar.linear_inplace(b, g, FROM12)\n    ar.linear_inplace(b, A, FROM12)\n\n\ndef source_high_chunk(b, A, g, target, chunk):\n    """One GF8 chunk of high GF64 half of Norm24(A,g),24T."""\n    source_linear(b, A, g, target, 1, chunk)\n    ar.linear_inplace(b, A, TO12)\n    ar.linear_inplace(b, g, TO12)\n    b.cx(A[6:], A[:6])\n    b.cx(g[6:], g[:6])\n    product6_chunk(b, A[:6], g[:6], target, chunk)\n    b.cx(g[6:], g[:6])\n    b.cx(A[6:], A[:6])\n    product6_chunk(b, A[:6], g[:6], target, chunk)\n    ar.linear_inplace(b, g, FROM12)\n    ar.linear_inplace(b, A, FROM12)\n\n\ndef phase_rational_retimed250(b, A, g, mask, paired_key=False):\n    """Complete ordinary-D phase m dot g/Norm24(A,g),250T/15fresh."""\n    nz = nonzero_classical(b, mask)\n    b.push_condition(nz)\n    R = b.alloc_qubits(6)\n    with b.mark(\'source_high\'):\n        source_high6(b, A, g, R)\n    D = b.alloc_qubits(6)\n    f = ar.field(6)\n    ar.linear_xor(b, R, D, [f.mul(K12, v) for v in ar.square_columns(6)])\n    for chunk in (0, 1):\n        P = b.alloc_qubits(3)\n        with b.mark(\'source_low_for_norm\'):\n            source_low_chunk(b, A, g, P, chunk)\n        ar.linear_xor(b, P, D, [f.square(FROM6[i + 3 * chunk]) for i in range(3)])\n        with b.mark(\'nested_norm_product\'):\n            partial_product(b, P, chunk, R, D)\n        erase_part(b, A, g, P, 0, chunk)\n        b.free(P)\n    with b.mark(\'inverse_nested_norm\'):\n        if paired_key:\n            key = inverse6_context_begin27(b, D)\n        else:\n            inverse6_unitary29(b, D)\n    with b.mark(\'high_numerator_phase\'):\n        product_phase_virtual(b, R, D, g, mask, both=True)\n    # The N1 character has been paid. Its source erasure is quadratic/free;\n    # its bank can now hold the whole N0, avoiding two partial phase products.\n    erase_part(b, A, g, R, 1)\n    b.free(R)\n    R = b.alloc_qubits(6)\n    with b.mark(\'source_low_for_phase\'):\n        source_low6(b, A, g, R)\n    with b.mark(\'low_numerator_phase\'):\n        product_phase_virtual(b, R, D, g, mask, both=False)\n    with b.mark(\'restore_nested_norm\'):\n        if paired_key:\n            inverse6_context_end21(b, D, key)\n        else:\n            inverse6_unitary29(b, D)\n    dm = b.alloc_bits(6)\n    b.hmr(D, dm)\n    b.free(D)\n    ar.phase_linear(b, R, dm, ar.square_columns(6))\n    source_phase(b, A, g, dm, [f.mul(K12, f.square(v)) for v in part_columns(1)])\n    # D\'s measurement has freed all six of its wires. Rebuild the whole N1\n    # bank here (30T), rather than two separately sourced GF8 chunks (48T).\n    P = b.alloc_qubits(6)\n    with b.mark(\'source_high_for_closure\'):\n        source_high6(b, A, g, P)\n    ar.phase_product(b, P, R, dm)\n    erase_part(b, A, g, P, 1)\n    b.free(P)\n    b.free(dm)\n    erase_part(b, A, g, R, 0)\n    b.free(R)\n    b.pop_condition()\n    b.free(nz)\n\n\ndef phase_rational_retimed240(b, A, g, mask):\n    """Keep the 3-bit log key through the bank swap:240T/15fresh."""\n    phase_rational_retimed250(b, A, g, mask, paired_key=True)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/reverse_exact.py': '"""Exact width-neutral synthesis kernels."""\nfrom exp.fold_joint import kernel as fold\nimport truncated_small as small\n\ndef low_product(b, head, A, B):\n    k = len(B)\n    if k in (2, 3):\n        small.accumulate(b, list(head[0])+list(head[1]),\n                         [q for w in A[:k] for q in w],\n                         [q for w in B for q in w], k)\n    else:\n        fold.low_product(b, head, A, B)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/root_fusion.py': '"""Monic-root full-product head restoration; separate from frozen candidate."""\nfrom pathlib import Path\nimport sys\nfrom types import SimpleNamespace\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nimport direct_converter as direct\nimport truncated_converter as trunc\nfrom converter import clone\n\n\ndef root_restore(b,f,g,ai,F):\n m=len(g);q0,q1=f[-2:]\n pivot(b,g,F);high=b.alloc_qubits(12)\n # G_original*(q or x*q) has degree m and known monic coefficient1.\n # Interpolation at m finite points aliases x^m by a known constant only.\n with dk.shifted_pair(b,[q0,q1],F) as q:\n  dk.conv(b,g,q,f[:-2]+[high],m)\n V,W=dk.k.vandermonde(m);field=ar.field(12)\n for j,word in enumerate(f[:-2]+[high]):\n  correction=0\n  for t in range(m):correction^=field.mul(W[j][t],field.pow(t,m))\n  for i in ar.bits(correction):b.x(word[i])\n pivot(b,g,F)\n mask=b.alloc_bits(12);b.hmr(q0,mask);b.swap(q0,high);b.free(high)\n # q0_old=lo*ai+c*ai^2, since the restored high coefficient is1.\n ar.phase_product(b,q0,ai,mask)\n ar.linear_inplace(b,ai,ar.square_columns(12))\n with dk.selected(b,g[-2],g[-3],F) as c:ar.phase_product(b,c,ai,mask)\n ar.linear_inplace(b,ai,[field.pow(1<<i,2048)for i in range(12)])\n b.free(mask)\n\n\ndef root_action(b,f,g,payloads,ai,Z,F,post,undo):\n restore=post!=undo\n if not restore:return ordinary_root_action(b,f,g,payloads,ai,Z,F,post,undo)\n a=g[-1];b.cx(Z,a[0])\n if post:\n  for v in payloads:old.scale(b,v[-1],a,ai,False)\n  for v in payloads:dk.selected_dot(b,g[:-1],v[:-1],v[-1],F)\n  for v in payloads:old.scale(b,v[-1],a,ai,True)\n  root_restore(b,f,g,ai,F)\n else:\n  root_restore(b,f,g,ai,F)\n  for v in payloads:dk.shifted_product(b,g[:-1],v[-1],v[:-1],F)\n  for v in payloads:old.scale(b,v[-1],a,ai,False)\n b.cx(Z,a[0])\n\n\ndef make_converter():\n original_factory=direct.mh.make_converter\n def factory(return_scope=False):\n  cv,scope,short=original_factory(True);oldconv=scope[\'convolution_xor\']\n  def convolution(b,x,y,z,reverse=False):\n   k=len(z)\n   if k in (2,3)and len(x)==k and len(y)==2:\n    trunc.small.accumulate(b,[q for w in x for q in w],[q for w in y for q in w],[q for w in z for q in w],k)\n   else:oldconv(b,x,y,z,reverse)\n  scope[\'convolution_xor\']=convolution;short[\'old\']=SimpleNamespace(**scope)\n  short[\'root_restore\']=clone(root_restore,short)\n  short[\'ordinary_root_action\']=clone(direct.root_action,short)\n  return (cv,scope,short)if return_scope else cv\n env=dict(vars(direct),mh=SimpleNamespace(make_converter=factory),root_action=root_action)\n return clone(direct.make_converter,env)()\nconvert_safe=make_converter()\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/streamed.py': '"""Original-D rational phase with small-field streaming and exact I6.\n\nNamed schedule: retain six N1 bits; stream three N0 bits at a time; D6\nis inverted by a unitary 29T/3-fresh circuit, hence no quartic inverse mask\nis deferred. The two cubic-source inverse calls cost 58T together.\nExpected complete count: 102+29+87+29+48 = 295T, 15 fresh.\n"""\nimport json\nfrom functools import lru_cache\nfrom rational import *\nfrom exp.noscratch.schedules import norm_xor_noscratch\nfrom exp.mul99.phase36 import phase_classical_parity\n\nPERMS = json.loads((Path(__file__).resolve().parent / \'permutations3.json\').read_text())\n\n\ndef permutation3(b, x, name, reverse=False):\n    gates = PERMS[name][\'gates\']\n    for gate in (reversed(gates) if reverse else gates):\n        getattr(b, gate[0])(*(x[i] for i in gate[1:]))\n\n\ndef linear_ops(columns):\n    n = len(columns)\n    rows = [sum(((c >> i) & 1) << j for j, c in enumerate(columns)) for i in range(n)]\n    ops = []\n    for j in range(n):\n        p = next(i for i in range(j, n) if rows[i] >> j & 1)\n        if p != j:\n            rows[p], rows[j] = rows[j], rows[p]\n            ops.append((\'swap\', p, j))\n        for i in range(n):\n            if i != j and rows[i] >> j & 1:\n                rows[i] ^= rows[j]\n                ops.append((\'cx\', j, i))\n    return tuple(reversed(ops))\n\n\ndef controlled_alpha(b, x, control):\n    f = ar.field(3)\n    ops = linear_ops([f.mul(2, 1 << j) for j in range(3)])\n    assert len(ops) == 3\n    for op, i, j in ops:\n        if op == \'swap\':\n            b.cswap(control, x[i], x[j])\n        else:\n            b.ccx(control, x[i], x[j])\n\n\ndef inverse6_unitary29(b, x):\n    """Exact inv0 permutation, 29T / 3 clean scratch, no deferred masks.\n\n    norm E is converted directly to log(inv(E)); logarithm 7 represents 0.\n    alpha^7=1, so the scale is reversible even on E=0. There x itself is0.\n    After both scalings, antilog changes E into norm(inv0(x)), closed by HMR.\n    """\n    to, frm, kval = ar.tower(6)\n    ar.linear_inplace(b, x, to)\n    E = b.alloc_qubits(3)\n    norm_xor_noscratch(b, x, E, kval)\n    permutation3(b, E, \'log_inverse\')\n    b.cx(x[3:], x[:3])\n    f = ar.field(3)\n    for j in range(3):\n        # F^j M_alpha F^-j = M_(alpha^(2^j)); F has order3.\n        before = [f.pow(1 << i, 1 << ((3 - j) % 3)) for i in range(3)]\n        after = [f.pow(1 << i, 1 << j) for i in range(3)]\n        for half in (x[:3], x[3:]):\n            ar.linear_inplace(b, half, before)\n            controlled_alpha(b, half, E[j])\n            ar.linear_inplace(b, half, after)\n    permutation3(b, E, \'log\', reverse=True)\n    ar._erase_norm(b, x, E, kval)\n    b.free(E)\n    ar.linear_inplace(b, x, frm)\n\n\nTO12, FROM12, K12 = ar.tower(12)\nTO6, FROM6, K6 = ar.tower(6)\n\n\n@lru_cache(None)\ndef part_columns(half, chunk=None):\n    columns = [(ar.apply(TO12, 1 << j) >> (6 * half)) & 63 for j in range(12)]\n    if chunk is not None:\n        columns = [(ar.apply(TO6, v) >> (3 * chunk)) & 7 for v in columns]\n    return tuple(columns)\n\n\ndef source_phase(b, A, g, mask, columns):\n    """Phase mask dot columns(Norm24(A,g)); quadratic closure is free."""\n    fullmask = b.alloc_bits(12)\n    b.bit_store(fullmask, False)\n    for j, value in enumerate(columns):\n        for i in ar.bits(value):\n            b.cx(mask[i], fullmask[j])\n    f = ar.field(12)\n    ar.phase_linear(b, A, fullmask, ar.square_columns(12))\n    ar.phase_linear(b, g, fullmask, [f.mul(K, v) for v in ar.square_columns(12)])\n    ar.phase_product(b, A, g, fullmask)\n    b.free(fullmask)\n\n\ndef erase_part(b, A, g, target, half, chunk=None):\n    mask = b.alloc_bits(len(target))\n    b.hmr(target, mask)\n    source_phase(b, A, g, mask, part_columns(half, chunk))\n    b.free(mask)\n\n\ndef source_linear(b, A, g, target, half, chunk=None, omitted=None):\n    cols = part_columns(half, chunk)\n    if omitted is not None:\n        cols = [v & ~(1 << omitted) for v in cols]\n    f = ar.field(12)\n    ar.linear_xor(b, A, target, [ar.apply(cols, v) for v in ar.square_columns(12)])\n    ar.linear_xor(b, g, target, [ar.apply(cols, f.mul(K, v)) for v in ar.square_columns(12)])\n\n\ndef source_high6(b, A, g, target):\n    """High GF64 component of N=Norm24(A,g), 30T / no scratch."""\n    source_linear(b, A, g, target, 1)\n    ar.linear_inplace(b, A, TO12)\n    ar.linear_inplace(b, g, TO12)\n    b.cx(A[6:], A[:6])\n    b.cx(g[6:], g[:6])\n    mul_xor_noscratch(b, A[:6], g[:6], target)\n    b.cx(g[6:], g[:6])\n    b.cx(A[6:], A[:6])\n    mul_xor_noscratch(b, A[:6], g[:6], target)\n    ar.linear_inplace(b, g, FROM12)\n    ar.linear_inplace(b, A, FROM12)\n\n\ndef scale_constant(b, x, c):\n    f = ar.field(len(x))\n    ar.linear_inplace(b, x, [f.mul(c, 1 << i) for i in range(len(x))])\n\n\ndef product3_projected(b, x, y, target, omitted=None):\n    if omitted is None:\n        mul_xor_noscratch(b, x, y, target)\n    else:\n        from exp.noscratch.kernels import tensor_xor_noscratch\n        terms = [(sm, vv & ~(1 << omitted)) for sm, vv in ar.tensor(3)\n                 if vv & ~(1 << omitted)]\n        tensor_xor_noscratch(b, x, y, target, terms)\n\n\ndef product6_chunk(b, x, y, target, chunk, omitted=None):\n    """One GF8 component of a GF64 product, 12T / no scratch."""\n    ar.linear_inplace(b, x, TO6)\n    ar.linear_inplace(b, y, TO6)\n    if chunk == 0:\n        mul_xor_noscratch(b, x[:3], y[:3], target)\n        scale_constant(b, target, ar.field(3).invert(K6))\n        mul_xor_noscratch(b, x[3:], y[3:], target)\n        scale_constant(b, target, K6)\n    else:\n        b.cx(x[3:], x[:3])\n        b.cx(y[3:], y[:3])\n        product3_projected(b, x[:3], y[:3], target, omitted)\n        b.cx(y[3:], y[:3])\n        b.cx(x[3:], x[:3])\n        product3_projected(b, x[:3], y[:3], target, omitted)\n    ar.linear_inplace(b, y, FROM6)\n    ar.linear_inplace(b, x, FROM6)\n\n\ndef source_low_chunk(b, A, g, target, chunk, cached=None):\n    """One GF8 component of low GF64 half of Norm24(A,g), 24T."""\n    omitted = 0 if cached is not None else None\n    assert cached is None or chunk == 1\n    if cached is not None:\n        b.cx(cached, target[0])\n    source_linear(b, A, g, target, 0, chunk, omitted=omitted)\n    ar.linear_inplace(b, A, TO12)\n    ar.linear_inplace(b, g, TO12)\n    product6_chunk(b, A[:6], g[:6], target, chunk, omitted=omitted)\n    scale_constant(b, A[6:], K12)\n    product6_chunk(b, A[6:], g[6:], target, chunk, omitted=omitted)\n    scale_constant(b, A[6:], ar.field(6).invert(K12))\n    ar.linear_inplace(b, g, FROM12)\n    ar.linear_inplace(b, A, FROM12)\n\n\ndef partial_product(b, p, chunk, x, output):\n    """output6 ^= embed(p3)*x6, 12T/no scratch, canonical GF64 x/out."""\n    ar.linear_inplace(b, x, TO6)\n    ar.linear_inplace(b, output, TO6)\n    if chunk == 0:\n        mul_xor_noscratch(b, p, x[:3], output[:3])\n        mul_xor_noscratch(b, p, x[3:], output[3:])\n    else:\n        scale_constant(b, output[:3], ar.field(3).invert(K6))\n        mul_xor_noscratch(b, p, x[3:], output[:3])\n        scale_constant(b, output[:3], K6)\n        b.cx(x[3:], x[:3])\n        mul_xor_noscratch(b, p, x[:3], output[3:])\n        b.cx(x[3:], x[:3])\n    ar.linear_inplace(b, output, FROM6)\n    ar.linear_inplace(b, x, FROM6)\n\n\ndef coefficient_parity(b, g, mask, h, a, c):\n    co = b.alloc_bits(12)\n    b.bit_store(co, False)\n    f = ar.field(12)\n    for j in range(12):\n        for i in ar.bits(f.mul(1 << j, h)):\n            b.cx(mask[i], co[j])\n    phase_classical_parity(b, a, c, g, co)\n    b.free(co)\n\n\ndef product_phase_virtual(b, x, y, g, mask, both):\n    """m dot g*embed(x6*y6), <=15T, virtual mask in live g."""\n    for sm, vv in ar.tensor(6):\n        h = ar.apply(FROM12, vv | ((vv << 6) if both else 0))\n        ix = list(ar.bits(sm))\n        p = ix[0]\n        for i in ix[1:]:\n            b.cx(x[i], x[p])\n            b.cx(y[i], y[p])\n        coefficient_parity(b, g, mask, h, x[p], y[p])\n        for i in reversed(ix[1:]):\n            b.cx(y[i], y[p])\n            b.cx(x[i], x[p])\n\n\ndef partial_phase(b, p, chunk, x, g=None, mask=None, classical=None):\n    """Phase embed(p3)*x6 with virtual g-dependent or classical mask."""\n    ar.linear_inplace(b, x, TO6)\n    configs = [(x[:3], 0, 1), (x[3:], 1, 1)] if chunk == 0 else [(x[3:], 0, K6), (x[:3], 1, 1)]\n    if chunk:\n        b.cx(x[3:], x[:3])\n    for controls, segment, constant in configs:\n        for sm, vv in ar.tensor(3):\n            vector = ar.apply(FROM6, ar.field(3).mul(constant, vv) << (3 * segment))\n            ix = list(ar.bits(sm))\n            pivot = ix[0]\n            for i in ix[1:]:\n                b.cx(p[i], p[pivot])\n                b.cx(controls[i], controls[pivot])\n            if classical is not None:\n                for i in ar.bits(vector):\n                    b.ccz(p[pivot], controls[pivot], classical[i])\n            else:\n                h = ar.apply(FROM12, vector)\n                coefficient_parity(b, g, mask, h, p[pivot], controls[pivot])\n            for i in reversed(ix[1:]):\n                b.cx(controls[i], controls[pivot])\n                b.cx(p[i], p[pivot])\n    if chunk:\n        b.cx(x[3:], x[:3])\n    ar.linear_inplace(b, x, FROM6)\n\n\ndef phase_rational_streamed295(b, A, g, mask, cache_source_bit=False):\n    """Phase m dot g/Norm24(A,g), 295T/15fresh or cached 287T/16fresh."""\n    nz = nonzero_classical(b, mask)\n    b.push_condition(nz)\n    R = b.alloc_qubits(6)\n    with b.mark(\'source_high\'):\n        source_high6(b, A, g, R)\n    D = b.alloc_qubits(6)\n    f = ar.field(6)\n    ar.linear_xor(b, R, D, [f.mul(K12, v) for v in ar.square_columns(6)])\n    cached = None\n    for chunk in (0, 1):\n        P = b.alloc_qubits(3)\n        with b.mark(\'source_low_for_norm\'):\n            source_low_chunk(b, A, g, P, chunk)\n        if cache_source_bit and chunk == 1:\n            cached = b.alloc_qubits(1)[0]\n            b.cx(P[0], cached)\n        ar.linear_xor(b, P, D, [f.square(FROM6[i + 3 * chunk]) for i in range(3)])\n        with b.mark(\'nested_norm_product\'):\n            partial_product(b, P, chunk, R, D)\n        erase_part(b, A, g, P, 0, chunk)\n        b.free(P)\n    with b.mark(\'inverse_nested_norm\'):\n        inverse6_unitary29(b, D)\n    with b.mark(\'retained_numerator_phase\'):\n        product_phase_virtual(b, R, D, g, mask, both=True)\n    for chunk in (0, 1):\n        P = b.alloc_qubits(3)\n        with b.mark(\'source_low_for_phase\'):\n            source_low_chunk(b, A, g, P, chunk, cached=cached if chunk == 1 else None)\n        with b.mark(\'streamed_numerator_phase\'):\n            partial_phase(b, P, chunk, D, g=g, mask=mask)\n        erase_part(b, A, g, P, 0, chunk)\n        b.free(P)\n    with b.mark(\'restore_nested_norm\'):\n        inverse6_unitary29(b, D)\n    dm = b.alloc_bits(6)\n    b.hmr(D, dm)\n    b.free(D)\n    ar.phase_linear(b, R, dm, [f.mul(K12, v) for v in ar.square_columns(6)])\n    source_phase(b, A, g, dm, [f.square(v) for v in part_columns(0)])\n    for chunk in (0, 1):\n        P = b.alloc_qubits(3)\n        with b.mark(\'source_low_for_closure\'):\n            source_low_chunk(b, A, g, P, chunk, cached=cached if chunk == 1 else None)\n        partial_phase(b, P, chunk, R, classical=dm)\n        erase_part(b, A, g, P, 0, chunk)\n        b.free(P)\n    if cached is not None:\n        cm = b.alloc_bits(1)\n        b.hmr(cached, cm[0])\n        source_phase(b, A, g, cm, [v & 1 for v in part_columns(0, 1)])\n        b.free(cm)\n        b.free([cached])\n    b.free(dm)\n    erase_part(b, A, g, R, 1)\n    b.free(R)\n    b.pop_condition()\n    b.free(nz)\n\n\ndef phase_rational_streamed287(b, A, g, mask):\n    """Retain one quadratic source bit: 287T/16fresh, all closures included."""\n    phase_rational_streamed295(b, A, g, mask, cache_source_bit=True)\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/synth_stream.py': '"""Consume the GF64 norm into its inverse; rebuild it after part scales."""\nfrom types import FunctionType,SimpleNamespace\nimport inspect\nfrom final_decoder import synthesis as folds\nfrom exp.noscratch.schedules import inverse_stream_noscratch\nimport norm_keep2\n\ndef clone(fn,env):\n out=FunctionType(fn.__code__,env,fn.__name__,fn.__defaults__,fn.__closure__)\n out.__kwdefaults__=fn.__kwdefaults__;return out\n\ndef make_synthesize(ref):\n source=inspect.getsource(folds.divide_fused)\n before=\'    t = inverse6_xor20(b, M)\\n\'\n after=\'\'\'    kept_M = norm_keep2.keep(b, M)\n    inverse_mask = inverse_stream_noscratch(b, M)\n    t = M\n    part_masks = []\n\'\'\'\n assert source.count(before)==1;source=source.replace(before,after)\n before=\'            ar.erase_product(b, out, M, part)\\n\'\n after=\'\'\'            pm = b.alloc_bits(6); b.hmr(part, pm); part_masks.append(pm)\n\'\'\'\n assert source.count(before)==1;source=source.replace(before,after)\n before=\'    for off, m in zip((0, 12), masks):\\n\'\n after=\'\'\'    ar.linear_inplace(b, N, to)\n    M = norm_keep2.rebuild(b, N, kept_M, k6)\n    ar.linear_inplace(b, N, frm)\n    b.cz(M, inverse_mask); b.free(inverse_mask)\n    for off in (0, 12):\n        half = x[off:off + 12]\n        ar.linear_inplace(b, half, to)\n        for j in (0, 6):\n            pm = part_masks.pop(0)\n            ar.phase_product(b, half[j:j + 6], M, pm); b.free(pm)\n        ar.linear_inplace(b, half, frm)\n    assert not part_masks\n    for off, m in zip((0, 12), masks):\n\'\'\'\n assert source.count(before)==1;source=source.replace(before,after)\n env=dict(folds.divide_fused.__globals__,inverse_stream_noscratch=inverse_stream_noscratch,norm_keep2=norm_keep2)\n exec(compile(source,__file__,\'exec\'),env)\n ug=dict(folds.ug)\n for key in (\'absorb\',\'absorb_even\'):\n  fn=ug[key];ug[key]=clone(fn,dict(fn.__globals__,divide_fused=env[\'divide_fused\']))\n proxy=SimpleNamespace(**dict(vars(folds),ug=ug))\n return clone(ref.synthesize,dict(ref.synthesize.__globals__,folds=proxy)),env[\'divide_fused\']\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/truncated_converter.py': '"""Direct candidate plus lead\'s checked k2/k3 truncated products:84T/pair."""\nfrom pathlib import Path\nimport sys,importlib.util\nfrom types import SimpleNamespace\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nfrom converter import clone\nimport direct_converter as direct\nspec=importlib.util.spec_from_file_location(\'bconvert_frozen_truncated_small\',Path(__file__).with_name(\'truncated_small.py\'))\nsmall=importlib.util.module_from_spec(spec);spec.loader.exec_module(small)\n\n\ndef make_converter():\n original_factory=direct.mh.make_converter\n def factory(return_scope=False):\n  cv,scope,short=original_factory(True);oldconv=scope[\'convolution_xor\']\n  def convolution(b,x,y,z,reverse=False):\n   k=len(z)\n   if k in (2,3) and len(x)==k and len(y)==2:\n    small.accumulate(b,[q for w in x for q in w],[q for w in y for q in w],[q for w in z for q in w],k)\n   else:oldconv(b,x,y,z,reverse)\n  scope[\'convolution_xor\']=convolution;short[\'old\']=SimpleNamespace(**scope)\n  return (cv,scope,short) if return_scope else cv\n return clone(direct.make_converter,dict(vars(direct),mh=SimpleNamespace(make_converter=factory)))()\nconvert_safe=make_converter()\n', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/truncated_small.py': '"""GF16 thickened cubic place: truncated lengths2/3 at99/153 Toffolis.\n\nInput banks may be shorter than k, treating missing high coefficients as zero.\nEverything is accumulated into a separate k-word target; no extra qubits.\n"""\nfrom functools import lru_cache\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[3]\nsys.path[:0]=[str(ROOT/\'playbook/kit\'),str(ROOT/\'playbook/kit/decoder\'),\n             str(ROOT/\'playbook/kit/decoder/exp/arithmetic\')]\nimport field as ar\nfrom exp.mul99.kernels import tensor4\n\n\n@lru_cache(None)\ndef construction(k):\n    assert k in (2,3)\n    f,sub=ar.field(12),ar.field(4)\n    mul=[[sub.mul(a,b) for b in range(16)] for a in range(16)]\n    def prod(a,b):\n        out=[0]*(len(a)+len(b)-1)\n        for i,u in enumerate(a):\n            for j,v in enumerate(b):out[i+j]^=mul[u][v]\n        return out\n    def mod(a,p):\n        out=list(a)\n        for i in range(len(out)-len(p),-1,-1):\n            c=out[i+len(p)-1]\n            if c:\n                for j,v in enumerate(p):out[i+j]^=mul[c][v]\n        return (out+[0]*(len(p)-1))[:len(p)-1]\n    def ev(a,x):\n        out=0\n        for c in reversed(a):out=mul[out][x]^c\n        return out\n    def pack(a):return sum(c<<(4*i) for i,c in enumerate(a))\n    def unpack(v,n):return [(v>>(4*i))&15 for i in range(n)]\n    generator=f.pow(f.primitive_element,4095//15)\n    def binary_eval(p,x):\n        out=0\n        for i in reversed(range(5)):out=f.mul(out,x)^((p>>i)&1)\n        return out\n    root=next(f.pow(generator,i) for i in range(1,16)\n              if binary_eval(sub.modulus,f.pow(generator,i))==0)\n    embed=[f.pow(root,i) for i in range(4)]\n    alpha=f.primitive_element\n    basis=[f.mul(v,f.pow(alpha,j)) for j in range(3) for v in embed]\n    to=ar.inverse_columns(basis)\n    p=unpack(ar.apply(to,f.pow(alpha,3)),3)+[1]\n    modulus=[1]\n    for _ in range(k):modulus=prod(modulus,p)\n    # The 4096-th power is the unique coefficient-field lift of t.\n    theta=[0,1]\n    for _ in range(12):theta=mod(prod(theta,theta),modulus)\n    tpowers=[[1],theta,mod(prod(theta,theta),modulus)]\n    ppowers=[[1]]\n    for _ in range(k-1):ppowers.append(prod(ppowers[-1],p))\n    cols=[]\n    for degree in range(k):\n        for bit in range(12):\n            co=unpack(to[bit],3)\n            v=[0]*(3*k)\n            for j,c in enumerate(co):\n                for i,x in enumerate(tpowers[j]):v[i]^=mul[c][x]\n            cols.append(pack(mod(prod(v,ppowers[degree]),modulus)))\n    inv=ar.inverse_columns(cols)\n    count=6*k-1\n    points=list(range(min(count,16)))\n    if count==17:points.append(None)\n    terms=[]\n    for point in points:\n        lag=[1]\n        denom=1\n        for other in points:\n            if other is not None and other!=point:\n                lag=prod(lag,[other,1])\n                if point is not None:denom=mul[denom][point^other]\n        lag=mod([mul[v][sub.invert(denom)] for v in lag],modulus)\n        vals=[(unpack(c,3*k)[-1] if point is None else ev(unpack(c,3*k),point))\n              for c in cols]\n        for smask,svec in tensor4():\n            mask=sum(((v&smask).bit_count()&1)<<i for i,v in enumerate(vals))\n            vector=ar.apply(inv,pack([mul[v][svec] for v in lag]))\n            if mask and vector:terms.append((mask,vector))\n    return tuple(terms)\n\n\n@lru_cache(None)\ndef tensor(k,nx=None,ny=None):\n    nx=k if nx is None else nx\n    ny=k if ny is None else ny\n    assert 1<=nx<=k and 1<=ny<=k\n    xm,ym=(1<<(12*nx))-1,(1<<(12*ny))-1\n    merged={}\n    for mask,v in construction(k):\n        a,c=mask&xm,mask&ym\n        if a and c:merged[a,c]=merged.get((a,c),0)^v\n    return tuple((a,c,v) for (a,c),v in merged.items() if v)\n\n\ndef accumulate(b,x,y,z,k):\n    assert len(x)%12==len(y)%12==0 and len(z)==12*k\n    assert len(set(map(str,list(x)+list(y)+list(z))))==len(x)+len(y)+len(z)\n    for am,bm,v in tensor(k,len(x)//12,len(y)//12):\n        ix,iy,iz=map(lambda q:list(ar.bits(q)),(am,bm,v))\n        p,q,t=ix[0],iy[0],iz[0]\n        for i in ix[1:]:b.cx(x[i],x[p])\n        for j in iy[1:]:b.cx(y[j],y[q])\n        for j in iz[1:]:b.cx(z[t],z[j])\n        b.ccx(x[p],y[q],z[t])\n        for j in reversed(iz[1:]):b.cx(z[t],z[j])\n        for j in reversed(iy[1:]):b.cx(y[j],y[q])\n        for i in reversed(ix[1:]):b.cx(x[i],x[p])\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/decode.py': '"""Our DQI decoder as decode(b, S, E), for check_decoder.py.\n\nThe stages, in order: mask projection, the initial linear maps, extraction (the jet kernel 289 times),\nsynthesis, conversion, scan, then conversion, synthesis, extraction and the linear maps undone, and the masks\nrestored. The jet kernel is exp/jet24/pro_jet.py: 683 Toffolis a pair (forward 290, reverse 393) at 43 private\nqubits, which sets Q at 883. The undo of the synthesis uses two narrower divisions than the forward\'s library\ndefault, so that it fits within 883 too: see unsynthesize below.\n"""\nimport sys\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nsys.path[:0] = [str(HERE.parent), str(HERE), str(HERE / \'exp/decoder\')]\n\nfrom common import *  # noqa: E402,F401,F403  (km is circ; ROOT is this folder)\nfrom exp.confluent_extract.kernel import initial, extract as extract_jets  # noqa: E402\nfrom exp.confluent_extract.model import constants as confluent_constants  # noqa: E402\nfrom exp.jet24.pro_jet import jet_forward as _jet_forward, jet_reverse as _jet_reverse  # noqa: E402\nimport exp.fold_cached_norm.kernel as FK  # noqa: E402\nfrom exp.fold_joint.kernel import divide414_callback  # noqa: E402\nfrom exp.mask_projection.kernel import project, restore, SELECTED  # noqa: E402\nsys.path.insert(0, str(HERE / \'exp/convert\'))\nfrom exp.convert_short_hankel.phase34 import convert_safe as convert_historyfree  # noqa: E402\nsys.path.insert(0, str(HERE / \'exp/scan\'))\nfrom scan import scan  # noqa: E402\n\n\ndef jet_forward(b, u, v, K=None):\n    assert K in (None, 2048)\n    return _jet_forward(b, u, v)\n\n\ndef jet_reverse(b, u, v, token, K=None):\n    assert K in (None, 2048)\n    return _jet_reverse(b, u, v, token)\n\n\ndef extract(b, words, reverse=False, transcript=None):\n    return extract_jets(b, words, jet_forward, jet_reverse, reverse, transcript)\n\n\ndef divide_zero414(b, x, y, K=None, extra_zero_mask=None, after=None):\n    """The outer fold\'s identity-at-zero division x /= y, at 25 private qubits: 414 Toffolis (the wide one, 324 at\n    49, would set Q at 889 here, since 840 qubits are live)."""\n    divide414_callback(b, x, y, K, extra_zero_mask, after)\n\n\nsynthesize = FK.synthesize\n# The forward synthesis multiplies; only its undo divides. At the inner folds\' divisions 853 qubits are live, so\n# they use the 24-scratch division of exp/mul99/schedules.py (369 Toffolis) instead of the retained-norm one\n# (323 at 36 private), and the outer fold uses divide_zero414.\nunsynthesize = FK.bind(FK.prior.unsynthesize, absorb=FK.prior.absorb_odd, absorb_even=FK.prior.absorb_even,\n                       outer=FK.bind(FK.prior.outer, divide_zero369=divide_zero414), odd_decode=FK.odd_decode,\n                       multiply_zero144=FK.multiply_zero141)\n\n\ndef decode(b, S, E):\n    with b.mark(\'mask_projection\'):\n        mask_token = project(b, S, E)\n    with b.mark(\'initial\'):\n        initial(b, S)\n    with b.mark(\'extract\'):\n        xh = extract(b, S)\n    with b.mark(\'synthesis\'):\n        A, B, yh = synthesize(b, S)\n    history_word = b.alloc_qubits(12)\n    with b.mark(\'convert\'):\n        phase_token = convert_historyfree(b, A, B, history_scratch12=history_word, monic_root=True)\n    with b.mark(\'scan\'):\n        scan(b, A, B, E, skip_points=SELECTED, node_polynomial=confluent_constants()[2])\n    with b.mark(\'unconvert\'):\n        convert_historyfree(b, A, B, True, history_scratch12=history_word, phase_token=phase_token)\n    b.free(history_word)\n    with b.mark(\'unsynthesis\'):\n        unsynthesize(b, yh)\n    with b.mark(\'unextract\'):\n        extract(b, S, True, xh)\n    with b.mark(\'uninitial\'):\n        initial(b, S, True)\n    with b.mark(\'restore_masks\'):\n        restore(b, E, mask_token)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/alternating_gauge/even_fold.py': '"""Full product/quotient jointly updates explicit odd continuant r, no shear."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.tail_alias.kernel import *\nfrom exp.mul99.schedules import divide_zero_bridge414\n\ndef full_product(b,head,A,qi,target):\n n=len(target);V,W=base.vandermonde(n)\n for t in range(n):old.rank_affine(b,head,[1,t],A+[qi],V[t][:len(A)+1],target,[W[j][t]for j in range(n)])\n\ndef full_quotient(b,head,A,qi,B,r,kap):\n k=len(B)\n if not k:ar.mul_xor0(b,head[1],qi,r);return\n words=A[1:]+[qi]\n for ac,bc,tc in reversed(base.quotient_terms(k+2,kap)):\n  old.rank_affine(b,head,ac,words,bc,B+[r],tc)\n\ndef absorb_even(b,head,A,B,qi,r,c,kap,reverse=False,transcript=None):\n """r=qnext->qprev in forward, inverse restores qnext; qi disjoint r."""\n k=len(B);out=head+B;target=out+[r]\n if not reverse:\n  value=old.eval_implicit(b,A,qi,c);F=dv.zero_flag(b,value);old.eval_implicit(b,A,qi,c,True)\n  b.cx(F,A[0][0]);spare=[b.alloc_qubits(12),b.alloc_qubits(12)];full=spare+B+[r]\n  linear_words(b,full,base.embedding_matrix(k+3,kap));full_product(b,head,A,qi,full);b.cx(F,A[0][0])\n  hm=b.alloc_bits(24);b.hmr(combine(head),hm)\n  for w,z in zip(head,spare):b.swap(w,z);b.free(z)\n  fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n  return out,A,(hm,fm)\n linear_words(b,target,base.embedding_matrix(k+3,kap),True)\n value=old.eval_implicit(b,A,qi,c);cx(b,head[1],head[0],c);hm,fm=transcript\n divide_zero_bridge414(b,combine(head),value,K=K,extra_zero_mask=fm[0]);b.free(fm)\n cx(b,head[1],head[0],c);b.cz(combine(head),hm);b.free(hm);old.eval_implicit(b,A,qi,c,True)\n full_quotient(b,head,A,qi,B,r,kap)\n return out,A,transcript\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/alternating_gauge/kernel.py': '"""Half-density continuant prefix: odd encoded heads, one rolling odd q.\n\nRaw b0 becomes a reversible physical q0=1 border; r carries its old value.\nThe border is freed after inner folds and r becomes the numerator top word.\nNo persistent quantum zero flag or second leading-coefficient bank is used.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.tail_alias.kernel import *\nfrom exp.alternating_gauge.even_fold import absorb_even\nfrom exp.alternating_gauge.scalar import prefix_reverse141,divide_zero117,multiply_zero144\n\ndef add_tail_a(b,a,q,d):\n ar.linear_inplace(b,q,ar.square_columns(12));ar.mul_xor0(b,a,q,d)\n ar.linear_inplace(b,q,[power(1<<j,2048)for j in range(12)])\n\ndef odd_encode(b,heads,i,r,d=None):\n a,x=heads[i];p=heads[i-2][1]if i>1 else heads[0][1]\n def after():\n  add_tail_a(b,a,r,d);ar.mul_xor0(b,x,r,d)\n return pg.forward(b,x,p,r,after_product=after if d is not None else None)\n\ndef odd_decode(b,heads,i,r,m,d=None):\n a,x=heads[i];p=heads[i-2][1]if i>1 else heads[0][1]\n if d is not None:ar.mul_xor0(b,x,r,d)\n prefix_reverse141(b,x,p,r,m)\n if d is not None:add_tail_a(b,a,r,d)\n\ndef even_step(b,heads,i,r,d=None,reverse=False):\n a,x=heads[i];q=heads[i-1][1]\n if d is not None and reverse:\n  add_tail_a(b,a,q,d);ar.mul_xor0(b,r,q,d)\n ar.mul_xor0(b,x,q,r)\n if d is not None and not reverse:\n  add_tail_a(b,a,q,d);ar.mul_xor0(b,r,q,d)\n\ndef synthesize(b,words):\n h=len(words)//2;assert h>=3 and h%2==1;cs,ks,_,_=old.constants(h)\n heads=[words[2*i:2*i+2]for i in range(h)];d=heads[0][0];border=heads[0][1]\n r=b.alloc_qubits(12);b.cx(border,r);b.cx(border,d);b.cx(r,border);b.x(border[0]);masks=[None]*h\n for i in range(1,h):\n  if i%2:masks[i]=odd_encode(b,heads,i,r,d)\n  else:even_step(b,heads,i,r,d)\n divide_zero117(b,d,r);A=[d];B=[];hist=[]\n for i in reversed(range(1,h)):\n  if i%2:\n   odd_decode(b,heads,i,r,masks[i]);qi=r;p=heads[i-2][1]if i>1 else border\n  else:\n   qi=heads[i-1][1];p=r\n  aa,bb=A,B\n  if i%2:A,B,token=absorb(b,heads[i],A,B,qi,p,None,0,cs[i],ks[i],wide=False)\n  else:A,B,token=absorb_even(b,heads[i],A,B,qi,r,cs[i],ks[i])\n  hist.append((i,aa,bb,qi,p,cs[i],ks[i],token))\n b.x(border[0]);b.free(border);C=B+[r];ot=outer(b,C,A,cs[0],ks[0],wide=True)\n return C,A,dict(h=h,heads=heads,d=d,r=r,border=border,history=hist,A=C,B=A,outer=ot)\n\ndef unsynthesize(b,t):\n h=t[\'h\'];heads=t[\'heads\'];r=t[\'r\'];cs,ks,_,_=old.constants(h)\n outer(b,t[\'A\'],t[\'B\'],cs[0],ks[0],True,t[\'outer\'],True)\n reclaim(b,[t[\'border\']]);b.x(t[\'border\'][0]);masks=[None]*h\n for i,A,B,qi,p,c,kap,token in reversed(t[\'history\']):\n  if i%2:\n   absorb(b,heads[i],A,B,qi,p,None,0,c,kap,True,token,False);masks[i]=odd_encode(b,heads,i,r)\n  else:absorb_even(b,heads[i],A,B,qi,r,c,kap,True,token)\n multiply_zero144(b,t[\'d\'],r)\n for i in reversed(range(1,h)):\n  if i%2:odd_decode(b,heads,i,r,masks[i],t[\'d\'])\n  else:even_step(b,heads,i,r,t[\'d\'],True)\n b.cx(r,t[\'d\']);b.x(t[\'border\'][0]);b.cx(r,t[\'border\']);b.cx(t[\'border\'],r);b.free(r)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/alternating_gauge/scalar.py': '"""25-private zero-aware scalar context; same117T as promised old divide."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.tail_alias.kernel import *\nfrom exp.inverse19_joint.kernel import inverse6_init19,inverse_zero\nfrom exp.arithmetic.lowwidth import phase_inverse_generic6\nfrom exp.safe_scalar.phase import phase_inverse36_zero\nfrom exp.noscratch.schedules import norm_xor_noscratch\n\n\ndef divide_zero117(b,x,y,before=None,after=None):\n """All-input x/=y or x when y0;117T/25private; hooks see F=[y0]."""\n to,frm,kk=ar.tower(12);N=b.alloc_qubits(6);ar.linear_inplace(b,y,to);norm_xor_noscratch(b,y,N,kk);ar.linear_inplace(b,y,frm)\n u,F=inverse6_init19(b,N,True)\n if before is not None:before(F)\n conj=[ar.field(12).frobenius(1<<i,6)for i in range(12)]\n ar.linear_inplace(b,y,conj);b.cx(F,y[0]);out=b.alloc_qubits(12);ar.mul_xor0(b,x,y,out)\n oldm=b.alloc_bits(12);b.hmr(x,oldm);b.swap(x,out);b.free(out);b.cx(F,y[0]);ar.linear_inplace(b,y,conj)\n ar.linear_inplace(b,x,to);b.cx(F,u[0]);b.cx(F,N[0])\n for off in(0,6):\n  half=x[off:off+6];out=b.alloc_qubits(6);ar.mul_xor0(b,half,u,out)\n  m=b.alloc_bits(6);b.hmr(half,m);b.swap(half,out);b.free(out);ar.phase_product(b,half,N,m);b.free(m)\n b.cx(F,N[0]);b.cx(F,u[0]);ar.linear_inplace(b,x,frm)\n b.cx(F,y[0]);ar.phase_product(b,x,y,oldm);b.cx(F,y[0]);b.free(oldm)\n if after is not None:after(F)\n fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F);b.z(fm[0]);zm=b.alloc_bits(6);b.bit_store(zm,False);b.cx(fm[0],zm[0]);ar.phase_product(b,N,u,zm);b.free(zm);b.free(fm)\n um=b.alloc_bits(6);b.hmr(u,um);b.free(u);phase_inverse_generic6(b,N,um);b.free(um)\n ar.linear_inplace(b,y,to);ar._erase_norm(b,y,N,kk);ar.linear_inplace(b,y,frm);b.free(N)\n\ndef multiply_zero144(b,x,y):\n """Inverse of divide_zero117,144T/25private, no deferred transcript."""\n u,F=inverse_zero(b,y);b.cx(F,y[0]);out=b.alloc_qubits(12);ar.mul_xor0(b,x,y,out)\n m=b.alloc_bits(12);b.hmr(x,m);b.swap(x,out);b.free(out);b.cx(F,y[0]);b.cx(F,u[0]);ar.phase_product(b,x,u,m);b.cx(F,u[0]);b.free(m)\n um=b.alloc_bits(12);fm=b.alloc_bits(1);b.hmr(u,um);b.hmr(F,fm[0]);b.free(u);b.free(F)\n phase_inverse36_zero(b,y,um,fm[0]);b.free(um);b.free(fm)\n\ndef prefix_reverse141(b,x,p,q,m):\n """Inverse oldpg.forward:141T/25private, including zero-q swap chart."""\n def before(F):b.cswap(F,x,p);b.cx(p,x);b.ccx(F,p,x)\n def after(F):\n  b.cz(x,m)\n  for j in range(12):b.ccz(F,x[j],m[j]);b.ccz(F,p[j],m[j])\n  b.free(m)\n divide_zero117(b,x,q,before,after)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/arithmetic/division.py': '"""Width-bounded GF24 scalar multiplication/division reference schedules.\n\nAll24-bit words use explicit a+b*y, y²+y=K. This file is separate from the\nverified primitive API while the scalar-normalization direction is measured.\n"""\nfrom field import *\n\n\ndef mul24_xor126(b,x,y,z,K=None):\n    """z ^=x*y,126Toffolis,1scratch; inputs/target disjoint."""\n    if K is None:K=extension_columns()[2]\n    f=field(12);a=x[:12];c=x[12:];d=y[:12];e=y[12:];za=z[:12];zb=z[12:]\n    # p=a*d contributes to both output halves.\n    b.cx(za,zb);mul_xor(b,a,d,za);b.cx(za,zb)\n    # q=c*e contributes Kq to the first half only.\n    b.gf2_imul(f.invert(K),target=km.array(za),field=f);mul_xor(b,c,e,za);b.gf2_imul(K,target=km.array(za),field=f)\n    # r=(a+c)(d+e) contributes to the second half.\n    b.cx(c,a);b.cx(e,d);mul_xor(b,a,d,zb);b.cx(e,d);b.cx(c,a)\n\n\n\ndef mul24_xor(b,x,y,z,K=None):\n    """Default105T/1scratch product; old126Tcomparison ismul24_xor126."""\n    mul24_xor105(b,x,y,z,K)\n\n\ndef phase_product24(b,x,y,mask,K=None):\n    if K is None:K=extension_columns()[2]\n    a=x[:12];c=x[12:];d=y[:12];e=y[12:];ma=mask[:12];mb=mask[12:]\n    phase_product(b,a,d,ma)\n    mk=b.alloc_bits(12);b.bit_store(mk,False)\n    # Transpose constant multiplication K into the classical phase mask.\n    for i in range(12):\n        for j in bits(field(12).mul(K,1<<i)):b.cx(ma[j],mk[i])\n    phase_product(b,c,e,mk);b.free(mk)\n    phase_product(b,a,e,mb);phase_product(b,c,d,mb);phase_product(b,c,e,mb)\n\n\ndef multiply_forward(b,x,y,K=None):\n    """105T forward x*=y, returns deferred oldx phase mask.\n\n    For y=0 this is correct only on valid subspace x=0. It is not an all-input\n    identity-at-zero permutation; caller must enforce that promise or add flag.\n    """\n    z=b.alloc_qubits(24);mul24_xor(b,x,y,z,K)\n    m=b.alloc_bits(24);b.hmr(x,m);b.swap(x,z);b.free(z);return m\n\n\ndef divide_preserving_factor(b,x,y,K=None):\n    """513T division x/=y preserving y,43workspace; valid x=0 when y=0."""\n    ym=inverse24_inplace_forward(b,y,K)\n    z=b.alloc_qubits(24);mul24_xor(b,x,y,z,K)\n    xm=b.alloc_bits(24);b.hmr(x,xm);b.swap(x,z);b.free(z)\n    inverse24_inplace_reverse(b,y,ym,K)\n    phase_product24(b,x,y,xm,K);b.free(xm)\n\n\ndef multiply_reverse(b,x,y,transcript,K=None):\n    divide_preserving_factor(b,x,y,K)\n    b.cz(x,transcript);b.free(transcript)\n\n\ndef divide_via_norm(b,x,y,K=None,retain_norm=True):\n    """Division x/=y:321T/43workspace or342T/37workspace.\n\n    y=0 requires promisedx=0. This all-valid-state exactness includes0 but does\n    NOT implement identity on arbitraryx at y=0.\n    """\n    import field as ar\n    if K is None:K=extension_columns()[2]\n    b.cx(y[12:],y[:12]);oldmask=multiply_forward(b,x,y,K);b.cx(y[12:],y[:12])\n    N=b.alloc_qubits(12);ar._norm_xor(b,y,N,K)\n    if retain_norm:u,D=ar._inverse_keep_norm(b,N)\n    else:u=inverse(b,N)\n    for offset in (0,12):\n        half=x[offset:offset+12];out=b.alloc_qubits(12);mul_xor(b,half,u,out)\n        erase_product(b,out,N,half);b.swap(half,out);b.free(out)\n    if retain_norm:\n        to,frm,K12=tower(12);linear_inplace(b,u,to)\n        v=b.alloc_qubits(6);ar._norm_xor(b,u,v,K12);linear_inplace(b,u,frm)\n        ar._erase_inverse_with_norm(b,N,u,v);b.free(u)\n        erase_inverse(b,D,v);b.free(v)\n        linear_inplace(b,N,to);ar._erase_norm(b,N,D,K12);b.free(D);linear_inplace(b,N,frm)\n    else:erase_inverse_releasing(b,N,u)\n    ar._erase_norm(b,y,N,K);b.free(N)\n    phase_product24(b,x,y,oldmask,K);b.free(oldmask)\n\n\ndef multiply_reverse_via_norm(b,x,y,transcript,K=None,retain_norm=True):\n    divide_via_norm(b,x,y,K,retain_norm=retain_norm)\n    b.cz(x,transcript);b.free(transcript)\n\n\ndef zero_flag(b,x):\n    """Compute [x=0], n-1ANDs; returnoneflag, clearallinternalANDs."""\n    n=len(x);b.x(x);ladder=b.alloc_qubits(n-1)\n    b.init_and(x[0],x[1],ladder[0])\n    for i in range(1,n-1):b.init_and(ladder[i-1],x[i+1],ladder[i])\n    for i in reversed(range(n-2)):\n        if i==0:b.del_and(x[0],x[1],ladder[0])\n        else:b.del_and(ladder[i-1],x[i+1],ladder[i])\n    b.x(x);b.free(ladder[:-1]);return ladder[-1]\n\n\ndef phase_zero(b,y,mask,head=None,headmask=None):\n    """Applyzero(y)*(mask+dot(head,headmask));22T+1CCZ for y24."""\n    n=len(y);b.x(y);ladder=b.alloc_qubits(n-2)\n    b.init_and(y[0],y[1],ladder[0])\n    for i in range(1,n-2):b.init_and(ladder[i-1],y[i+1],ladder[i])\n    b.ccz(ladder[-1],y[-1],mask)\n    if head is not None:\n        q=b.alloc_qubits(1)[0]\n        for xi,mi in zip(head,headmask):b.ccx(mi,xi,q)\n        b.ccz(ladder[-1],y[-1],q)\n        for xi,mi in reversed(list(zip(head,headmask))):b.ccx(mi,xi,q)\n        b.free(q)\n    for i in reversed(range(n-2)):\n        if i==0:b.del_and(y[0],y[1],ladder[0])\n        else:b.del_and(ladder[i-1],y[i+1],ladder[i])\n    b.free(ladder);b.x(y)\n\n\ndef multiply_zero_forward(b,x,y,K=None):\n    """152T multiplication, identityat y0. Deferredhead+zero transcripts."""\n    flag=zero_flag(b,y);z=b.alloc_qubits(24);mul24_xor(b,x,y,z,K);b.ccx(flag,x,z)\n    oldmask=b.alloc_bits(24);b.hmr(x,oldmask);b.swap(x,z);b.free(z)\n    fm=b.alloc_bits(1);b.hmr(flag,fm[0]);b.free(flag)\n    return oldmask,fm[0]\n\n\ndef divide_zero(b,x,y,K=None,extra_zero_mask=None):\n    """379T exact division x/=y, identityat y0,43workspace."""\n    import field as ar\n    if K is None:K=extension_columns()[2]\n    N=b.alloc_qubits(12);ar._norm_xor(b,y,N,K);flag=zero_flag(b,N)\n    b.cx(y[12:],y[:12]);z=b.alloc_qubits(24);mul24_xor(b,x,y,z,K);b.ccx(flag,x,z);b.cx(y[12:],y[:12])\n    oldmask=b.alloc_bits(24);b.hmr(x,oldmask);b.swap(x,z);b.free(z)\n    b.cx(flag,N[0]);fm=b.alloc_bits(1);b.hmr(flag,fm[0]);b.free(flag)\n    u,D=ar._inverse_keep_norm(b,N)\n    for offset in (0,12):\n        half=x[offset:offset+12];out=b.alloc_qubits(12);mul_xor(b,half,u,out)\n        erase_product(b,out,N,half);b.swap(half,out);b.free(out)\n    to,frm,K12=tower(12);linear_inplace(b,u,to);v=b.alloc_qubits(6);ar._norm_xor(b,u,v,K12);linear_inplace(b,u,frm)\n    ar._erase_inverse_with_norm(b,N,u,v);b.free(u);erase_inverse(b,D,v);b.free(v)\n    linear_inplace(b,N,to);ar._erase_norm(b,N,D,K12);b.free(D);linear_inplace(b,N,frm)\n    nm=b.alloc_bits(12);b.hmr(N,nm);b.free(N);ar._phase_norm(b,y,nm,K)\n    comb=b.alloc_bits(1);b.bit_store(comb,False);b.cx(fm[0],comb[0]);b.cx(nm[0],comb[0])\n    if extra_zero_mask is not None:b.cx(extra_zero_mask,comb[0])\n    phase_product24(b,x,y,oldmask,K);phase_zero(b,y,comb[0],x,oldmask)\n    b.free(oldmask);b.free(fm);b.free(nm);b.free(comb)\n\n\ndef multiply_zero_reverse(b,x,y,transcript,K=None):\n    oldmask,fm=transcript;divide_zero(b,x,y,K,extra_zero_mask=fm)\n    b.cz(x,oldmask);b.free(oldmask);b.free(fm)\n\n\n\ndef erase_inverse_releasing(b,x,y):\n    """Phase-only inverse erasure; measure/free target BEFORE recomputation.\n\n    Same9/45/153T recursion as field.erase_inverse, but permits early reuse\n    of y\'s qubits. This helper owns and frees y.\n    """\n    import field as ar\n    n=len(x)\n    if n==3:\n        mask=b.alloc_bits(3);b.hmr(y,mask);b.free(y)\n        for im,v in enumerate(inverse3_anf()):\n            ix=list(bits(im))\n            for j in bits(v):\n                if len(ix)==1:b.cz(x[ix[0]],mask[j])\n                elif len(ix)==2:b.ccz(x[ix[0]],x[ix[1]],mask[j])\n        b.free(mask);return\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,y,to)\n    mask=b.alloc_bits(n);b.hmr(y,mask);b.free(y)\n    linear_inplace(b,x,to);norm=b.alloc_qubits(k);ar._norm_xor(b,x,norm,K);invnorm=inverse(b,norm)\n    b.cx(x[k:],x[:k]);phase_product(b,x[:k],invnorm,mask[:k]);phase_product(b,x[k:],invnorm,mask[k:]);b.cx(x[k:],x[:k]);b.free(mask)\n    erase_inverse_releasing(b,norm,invnorm)\n    ar._erase_norm(b,x,norm,K);b.free(norm);linear_inplace(b,x,frm)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/arithmetic/field.py': '"""Low-width GF(2^m) arithmetic, standard polynomial basis.\n\nPublic: mul_xor(builder,x,y,z), inverse(builder,x,target=None),\nlinear_xor(builder, x, z, columns), linear_inplace(builder,x,columns),\nphase_product(builder,x,y,mask), erase_linear/erase_product/erase_inverse.\nAll array arguments can be lists or km.array. Arithmetic preserves inputs.\n`inverse` returns a fresh clean output, 0 maps to 0. Default GF2Field basis.\n"""\nfrom functools import lru_cache\nimport circ as km\n\n\ndef bits(x):\n    while x:\n        t=x&-x; yield t.bit_length()-1; x^=t\n\n\ndef apply(columns, value):\n    out=0\n    for i in bits(value): out ^= columns[i]\n    return out\n\n\ndef inverse_columns(columns):\n    n=len(columns)\n    rows=[sum(((c>>i)&1)<<j for j,c in enumerate(columns)) | (1<<(n+i)) for i in range(n)]\n    for j in range(n):\n        p=next(i for i in range(j,n) if rows[i]>>j&1)\n        rows[j],rows[p]=rows[p],rows[j]\n        for i in range(n):\n            if i!=j and rows[i]>>j&1: rows[i]^=rows[j]\n    return [sum(((rows[i]>>(n+j))&1)<<i for i in range(n)) for j in range(n)]\n\n\ndef linear_xor(b,x,z,columns):\n    for i,c in enumerate(columns):\n        for j in bits(c): b.cx(x[i],z[j])\n\n\ndef linear_inplace(b,x,columns):\n    """Invertible GF2 map, no ancillary qubits. columns[j] = image(1<<j)."""\n    n=len(x); rows=[sum(((c>>i)&1)<<j for j,c in enumerate(columns)) for i in range(n)]; ops=[]\n    for j in range(n):\n        p=next(i for i in range(j,n) if rows[i]>>j&1)\n        if p!=j: rows[p],rows[j]=rows[j],rows[p]; ops.append((\'swap\',p,j))\n        for i in range(n):\n            if i!=j and rows[i]>>j&1: rows[i]^=rows[j];ops.append((\'cx\',j,i))\n    for op,i,j in reversed(ops): getattr(b,op)(x[i],x[j])\n\n\ndef phase_linear(b,x,mask,columns):\n    for i,c in enumerate(columns):\n        for j in bits(c): b.cz(x[i],mask[j])\n\n\ndef erase_linear(b,x,z,columns):\n    mask=b.alloc_bits(len(z)); b.hmr(z,mask); phase_linear(b,x,mask,columns);b.free(mask)\n\n\n@lru_cache(None)\ndef field(n): return km.GF2Field(n)\n\n\n@lru_cache(None)\ndef tensor(n):\n    """Symmetric multiplication tensor (linear mask, output vector), rank3/6/15/42."""\n    f=field(n)\n    if n==2: masks=[1,2,3]\n    elif n==3: masks=[1,7,4,5,6,3]\n    elif n in (6,12,24):\n        k,d={6:(2,3),12:(3,4),24:(6,4)}[n]\n        sub=field(k); power=(2**n-1)//(2**k-1)\n        gen=f.pow(f.primitive_element,power)\n        def evalpoly(v,p):\n            q=0\n            for i in reversed(range(k+1)):q=f.mul(q,p)^((v>>i)&1)\n            return q\n        embroot=next(f.pow(gen,i) for i in range(1,2**k) if evalpoly(sub.modulus,f.pow(gen,i))==0)\n        embed=[f.pow(embroot,i) for i in range(k)]\n        theta=f.primitive_element\n        cols=[f.mul(embed[i],f.pow(theta,j)) for j in range(d) for i in range(k)]\n        inv=inverse_columns(cols)\n        lowmasks=[m for m,_ in tensor(k)]\n        masks=[]\n        for point in list(range(2*d-2))+[None]:\n            outcols=[]\n            for extbit in range(n):\n                coord=inv[extbit]\n                if point is None: value=(coord>>(k*(d-1)))&((1<<k)-1)\n                else:\n                    value=0\n                    for j in reversed(range(d)): value=sub.mul(value,point)^((coord>>(k*j))&((1<<k)-1))\n                outcols.append(value)\n            for lm in lowmasks:masks.append(sum(((v&lm).bit_count()%2)<<i for i,v in enumerate(outcols)))\n    else: raise ValueError(n)\n    # Each basis-pair equation determines output vector of rank terms.\n    rank=len(masks); piv={}\n    for i in range(n):\n        for j in range(i,n):\n            row=sum(((m>>i&1)&(m>>j&1))<<r for r,m in enumerate(masks)); rhs=f.mul(1<<i,1<<j)\n            while row:\n                p=(row&-row).bit_length()-1\n                if p in piv:row^=piv[p][0];rhs^=piv[p][1]\n                else:piv[p]=(row,rhs);break\n            if row==0: assert rhs==0,(n,i,j,rhs)\n    out=[0]*rank\n    for p,(row,rhs) in sorted(piv.items(),reverse=True):\n        for j in bits(row^(1<<p)):rhs^=out[j]\n        out[p]=rhs\n    assert all(out)\n    return tuple(zip(masks,out))\n\n\ndef parity_into(b,x,mask,t):\n    for i in bits(mask):b.cx(x[i],t)\n\n\ndef mul_xor(b,x,y,z):\n    """Accumulate product, 1 scratch when inputs disjoint; 3 if overlapping.\n\n    Linear forms are computed temporarily in a chosen input pivot, restored\n    after each rank-one term. The output must be disjoint from both inputs.\n    """\n    n=len(x);assert len(y)==len(z)==n\n    overlap=bool(set(map(str,x)) & set(map(str,y)))\n    if overlap:\n        a,c,p=b.alloc_qubits(3)\n        for mask,v in tensor(n):\n            parity_into(b,x,mask,a);parity_into(b,y,mask,c)\n            b.init_and(a,c,p)\n            for i in bits(v):b.cx(p,z[i])\n            b.del_and(a,c,p)\n            parity_into(b,y,mask,c);parity_into(b,x,mask,a)\n        b.free([a,c,p])\n    else:\n        p=b.alloc_qubits(1)[0]\n        for mask,v in tensor(n):\n            ix=list(bits(mask));pivot=ix[0];a=x[pivot];c=y[pivot]\n            for i in ix[1:]:b.cx(x[i],a);b.cx(y[i],c)\n            b.init_and(a,c,p)\n            for i in bits(v):b.cx(p,z[i])\n            b.del_and(a,c,p)\n            for i in reversed(ix[1:]):b.cx(y[i],c);b.cx(x[i],a)\n        b.free(p)\n\n\ndef phase_product(b,x,y,mask):\n    """Exact phase dot(mask,x*y), zero Toffolis, no quantum workspace."""\n    f=field(len(x))\n    for i in range(len(x)):\n        for j in range(len(y)):\n            v=f.mul(1<<i,1<<j)\n            for k in bits(v):b.ccz(x[i],y[j],mask[k])\n\n\ndef erase_product(b,x,y,z):\n    mask=b.alloc_bits(len(z));b.hmr(z,mask);phase_product(b,x,y,mask);b.free(mask)\n\n\n@lru_cache(None)\ndef tower(n):\n    """Coordinates a+b*y, y²+y=K; returns to/from tower and K."""\n    k=n//2;f=field(n);sub=field(k)\n    gen=f.pow(f.primitive_element,(2**n-1)//(2**k-1))\n    def ev(v,p):\n        out=0\n        for i in reversed(range(k+1)):out=f.mul(out,p)^((v>>i)&1)\n        return out\n    root=next(f.pow(gen,i) for i in range(1,2**k) if ev(sub.modulus,f.pow(gen,i))==0)\n    embed=[f.pow(root,i) for i in range(k)]\n    # Any trace-relative-one element has minimal polynomial y²+y+K.\n    y=next(v for v in range(2**n) if f.pow(v,2**k)^v==1)\n    cols=embed+[f.mul(v,y) for v in embed];inv=inverse_columns(cols)\n    kval=apply(inv,f.square(y)^y);assert kval<2**k\n    return tuple(inv),tuple(cols),kval\n\n\ndef square_columns(n):return [field(n).square(1<<i) for i in range(n)]\n\n\ndef _norm_xor(b,x,z,kval):\n    """x already in tower coordinates. Norm = a²+ab+K b²."""\n    k=len(z);f=field(k);a=x[:k];c=x[k:]\n    linear_xor(b,a,z,square_columns(k));linear_xor(b,c,z,[f.mul(kval,v) for v in square_columns(k)])\n    mul_xor(b,a,c,z)\n\n\ndef _phase_norm(b,x,mask,kval):\n    k=len(mask);f=field(k);a=x[:k];c=x[k:]\n    phase_linear(b,a,mask,square_columns(k));phase_linear(b,c,mask,[f.mul(kval,v) for v in square_columns(k)])\n    phase_product(b,a,c,mask)\n\n\ndef _erase_norm(b,x,z,kval):\n    mask=b.alloc_bits(len(z));b.hmr(z,mask);_phase_norm(b,x,mask,kval);b.free(mask)\n\n\n@lru_cache(None)\ndef inverse3_anf():\n    f=field(3);a=[0 if x==0 else f.invert(x) for x in range(8)]\n    for i in range(3):\n        for s in range(8):\n            if s>>i&1:a[s]^=a[s^(1<<i)]\n    assert a[7]==0\n    return tuple(a)\n\n\ndef inverse(b,x,target=None):\n    """Clean out-of-place inverse: I3=3,I6=21,I12=66,I24=192 Toffolis.\n\n    Target must be initially zero when passed. Allocating target externally can\n    increase peak width: omit target for just-in-time recursive allocation.\n    """\n    n=len(x)\n    if n==3:\n        if target is None:target=b.alloc_qubits(3)\n        anc=b.alloc_qubits(1)[0]\n        for mask,v in enumerate(inverse3_anf()):\n            ix=list(bits(mask))\n            if len(ix)==1:\n                for j in bits(v):b.cx(x[ix[0]],target[j])\n            elif len(ix)==2:\n                b.init_and(x[ix[0]],x[ix[1]],anc)\n                for j in bits(v):b.cx(anc,target[j])\n                b.del_and(x[ix[0]],x[ix[1]],anc)\n        b.free(anc);return target\n    k=n//2;to,frm,K=tower(n)\n    linear_inplace(b,x,to)\n    norm=b.alloc_qubits(k);_norm_xor(b,x,norm,K)\n    invnorm=inverse(b,norm)\n    _erase_norm(b,x,norm,K);b.free(norm)\n    if target is None:target=b.alloc_qubits(n)\n    # x conjugate is (a+b,b), modify temporarily in-place.\n    a=x[:k];c=x[k:];b.cx(c,a)\n    mul_xor(b,a,invnorm,target[:k]);mul_xor(b,c,invnorm,target[k:]);b.cx(c,a)\n    # norm(target) = invnorm, including zero.\n    _erase_norm(b,target,invnorm,K);b.free(invnorm)\n    linear_inplace(b,x,frm);linear_inplace(b,target,frm)\n    return target\n\n\ndef inverse_inplace_forward(b,x):\n    """Forward 192T (n24), classical n-bit deferred input phase transcript."""\n    y=inverse(b,x);mask=b.alloc_bits(len(x));b.hmr(x,mask);b.swap(x,y);b.free(y)\n    return mask\n\n\ndef erase_inverse(b,x,y):\n    """Erase y=inv(x) with a closed phase-only inverse recomputation.\n\n    Generic fallback costs Ierase3=0, Ierase6=9, Ierase12=45,\n    Ierase24=153. It is NOT the claimed 30T inverse12 erasure.\n    """\n    n=len(x)\n    if n==3:\n        mask=b.alloc_bits(3);b.hmr(y,mask)\n        for im,v in enumerate(inverse3_anf()):\n            ix=list(bits(im))\n            for j in bits(v):\n                if len(ix)==1:b.cz(x[ix[0]],mask[j])\n                elif len(ix)==2:b.ccz(x[ix[0]],x[ix[1]],mask[j])\n        b.free(mask);return\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to);linear_inplace(b,y,to)\n    norm=b.alloc_qubits(k);_norm_xor(b,x,norm,K);invnorm=inverse(b,norm)\n    mask=b.alloc_bits(n);b.hmr(y,mask)\n    b.cx(x[k:],x[:k]);phase_product(b,x[:k],invnorm,mask[:k]);phase_product(b,x[k:],invnorm,mask[k:]);b.cx(x[k:],x[:k]);b.free(mask)\n    erase_inverse(b,norm,invnorm);b.free(invnorm)\n    _erase_norm(b,x,norm,K);b.free(norm);linear_inplace(b,x,frm)\n\n\ndef inverse_inplace_reverse_fallback(b,x,transcript):\n    """Close forward transcript; exact345T reverse at degree24, width61.\n\n    Kept as a correctness reference, NOT competitive with the target222T.\n    """\n    y=inverse(b,x)\n    b.cz(y,transcript);b.free(transcript)\n    erase_inverse(b,y,x)\n    b.swap(x,y);b.free(y)\n\n\ndef _inverse_keep_norm(b,x):\n    """Initialized inverse with its half-size input norm kept quantum-live.\n\n    Cost unchanged from inverse(). Caller must eventually erase kept norm.\n    """\n    n=len(x);k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to)\n    norm=b.alloc_qubits(k);_norm_xor(b,x,norm,K)\n    invnorm=inverse(b,norm)\n    target=b.alloc_qubits(n)\n    b.cx(x[k:],x[:k]);mul_xor(b,x[:k],invnorm,target[:k]);mul_xor(b,x[k:],invnorm,target[k:]);b.cx(x[k:],x[:k])\n    _erase_norm(b,target,invnorm,K);b.free(invnorm)\n    linear_inplace(b,x,frm);linear_inplace(b,target,frm)\n    return target,norm\n\n\ndef _erase_inverse_with_norm(b,x,y,norm_y):\n    """Erase y=inv(x) when norm(y) is retained. Zero Toffolis.\n\n    y=conjugate(x)*norm(y), so the phase fix is bilinear.\n    Both x and y arrive in polynomial basis; norm_y in half-field basis.\n    """\n    n=len(x);k=n//2;to,frm,K=tower(n)\n    linear_inplace(b,x,to);linear_inplace(b,y,to)\n    b.cx(x[k:],x[:k]);erase_product(b,x[:k],norm_y,y[:k]);erase_product(b,x[k:],norm_y,y[k:]);b.cx(x[k:],x[:k])\n    linear_inplace(b,x,frm)\n\n\ndef inverse_inplace_reverse(b,x,transcript):\n    """Close inverse_inplace_forward: degree24 reverse216T, peak67 qubits.\n\n    Retain norm(norm(input))6 during inverse12. Stream the two output halves\n    so retained N12+t12+D6 add at most43 ancillas to input24. The final D\n    cleanup costs norm(t)15 + inverse6 phase9 =24 additional Toffolis.\n    """\n    n=len(x);assert n in (12,24)\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to)\n    N=b.alloc_qubits(k);_norm_xor(b,x,N,K)\n    t,D=_inverse_keep_norm(b,N)\n    outb=b.alloc_qubits(k);mul_xor(b,x[k:],t,outb)\n    erase_product(b,outb,N,x[k:])\n    mul_xor(b,x[:k],t,x[k:]);b.cx(outb,x[k:])\n    b.cx(outb,x[k:]);erase_product(b,x[k:],N,x[:k]);b.cx(outb,x[k:])\n    b.swap(x[:k],x[k:]);b.swap(x[k:],outb);b.free(outb)\n    # N=inv(t), and norm(N)=D was retained inside inverse(N).\n    _erase_inverse_with_norm(b,t,N,D);b.free(N)\n    tok,frmk,Kk=tower(k);linear_inplace(b,t,tok)\n    u=b.alloc_qubits(k//2);_norm_xor(b,t,u,Kk)\n    erase_inverse(b,u,D);b.free(D)\n    _erase_norm(b,t,u,Kk);b.free(u);linear_inplace(b,t,frmk)\n    _erase_norm(b,x,t,K);b.free(t)\n    linear_inplace(b,x,frm)\n    b.cz(x,transcript);b.free(transcript)\n\n\n@lru_cache(None)\ndef extension_columns(K=None):\n    """Map base12 pair a+b*y to standard GF24 polynomial basis.\n\n    Uses base GF12 modulus0x1053. K is any GF12 trace-one element, y²+y=K.\n    Omitting K picks the smallest trace-one word, as in BRIEF.md.\n    Return (to_polynomial_columns, from_polynomial_columns, K).\n    """\n    f=field(12)\n    def trace(v):\n        out=0\n        for i in range(12):out^=f.frobenius(v,i)\n        return out\n    if K is None:K=next(v for v in range(4096) if trace(v)==1)\n    if not 0<=K<4096 or trace(K)!=1:raise ValueError(\'K must have absolute trace1\')\n    _,frm,K0=tower(24)\n    c=next(v for v in range(4096) if f.square(v)^v==K^K0)\n    F=field(24);y=frm[12]^apply(frm[:12],c)\n    cols=tuple(frm[:12])+tuple(F.mul(v,y) for v in frm[:12])\n    return cols,tuple(inverse_columns(cols)),K\n\n\ndef inverse24_inplace_forward(b,x,K=None):\n    """192T forward inversion in explicit a+b*y coordinates; returns transcript."""\n    cols,invcols,_=extension_columns(K);linear_inplace(b,x,cols)\n    transcript=inverse_inplace_forward(b,x);linear_inplace(b,x,invcols)\n    return transcript\n\n\ndef inverse24_inplace_reverse(b,x,transcript,K=None):\n    """216T closed reverse,43 workspace, in explicit a+b*y coordinates."""\n    cols,invcols,_=extension_columns(K);linear_inplace(b,x,cols)\n    inverse_inplace_reverse(b,x,transcript);linear_inplace(b,x,invcols)\n\n\ndef _solve_output_vectors(rows,rank):\n    piv={}\n    for row,rhs in rows:\n        while row:\n            p=(row&-row).bit_length()-1\n            if p in piv:row^=piv[p][0];rhs^=piv[p][1]\n            else:piv[p]=(row,rhs);break\n        if row==0:assert rhs==0,\'tensor does not span function\'\n    out=[0]*rank\n    for p,(row,rhs) in sorted(piv.items(),reverse=True):\n        for j in bits(row^(1<<p)):rhs^=out[j]\n        out[p]=rhs\n    return out\n\n\n@lru_cache(None)\ndef cube4_tensor():\n    """Four ANDs plus linear gates for GF16 cube."""\n    f=field(4);pairs=[(i,j) for i in range(4) for j in range(i+1,4)]\n    quads=[sum(((f.pow((1<<i)^(1<<j),3)^f.pow(1<<i,3)^f.pow(1<<j,3))>>k&1)<<p for p,(i,j) in enumerate(pairs)) for k in range(4)]\n    span={0}\n    for q in quads:span|={v^q for v in list(span)}\n    selected=[];piv={}\n    for a in range(1,16):\n        for c in range(a+1,16):\n            q=sum((((a>>i&1)&(c>>j&1))^((a>>j&1)&(c>>i&1)))<<p for p,(i,j) in enumerate(pairs))\n            if q not in span:continue\n            rem=q\n            for p,v in sorted(piv.items()):\n                if rem>>p&1:rem^=v\n            if rem:\n                p=(rem&-rem).bit_length()-1;piv[p]=rem;selected.append((a,c))\n    assert len(selected)==4\n    rows=[]\n    for i,j in pairs:\n        row=sum((((a>>i&1)&(c>>j&1))^((a>>j&1)&(c>>i&1)))<<p for p,(a,c) in enumerate(selected))\n        rhs=f.pow((1<<i)^(1<<j),3)^f.pow(1<<i,3)^f.pow(1<<j,3);rows.append((row,rhs))\n    out=_solve_output_vectors(rows,len(selected))\n    return tuple((a,c,v) for (a,c),v in zip(selected,out))\n\n\n@lru_cache(None)\ndef cube12_tensor():\n    """Seven GF16 cubic evaluations, each four ANDs: rank28 quadratic map."""\n    f=field(12);sub=field(4);gen=f.pow(f.primitive_element,273)\n    def ev(v,p):\n        out=0\n        for i in reversed(range(5)):out=f.mul(out,p)^((v>>i)&1)\n        return out\n    root=next(f.pow(gen,i) for i in range(1,16) if ev(sub.modulus,f.pow(gen,i))==0)\n    embed=[f.pow(root,i) for i in range(4)]\n    coords=inverse_columns([f.mul(v,f.pow(f.primitive_element,j)) for j in range(3) for v in embed])\n    terms=[]\n    for point in range(7):\n        vals=[]\n        for co in coords:\n            v=0\n            for j in reversed(range(3)):v=sub.mul(v,point)^((co>>(4*j))&15)\n            vals.append(v)\n        for a,c,_ in cube4_tensor():\n            A=sum(((v&a).bit_count()%2)<<i for i,v in enumerate(vals));C=sum(((v&c).bit_count()%2)<<i for i,v in enumerate(vals));terms.append((A,C))\n    rows=[]\n    for i in range(12):\n        for j in range(i+1,12):\n            row=sum((((a>>i&1)&(c>>j&1))^((a>>j&1)&(c>>i&1)))<<p for p,(a,c) in enumerate(terms))\n            rhs=f.pow((1<<i)^(1<<j),3)^f.pow(1<<i,3)^f.pow(1<<j,3);rows.append((row,rhs))\n    out=_solve_output_vectors(rows,len(terms));assert all(out)\n    linear=[]\n    for i in range(12):\n        v=f.pow(1<<i,3)\n        for (a,c),o in zip(terms,out):\n            if a>>i&1 and c>>i&1:v^=o\n        linear.append(v)\n    return tuple((a,c,v) for (a,c),v in zip(terms,out)),tuple(linear)\n\n\ndef cube_xor(b,x,z):\n    """GF12 cube accumulator z ^=x³,28Toffolis and3workspace."""\n    assert len(x)==len(z)==12\n    terms,linear=cube12_tensor();linear_xor(b,x,z,linear)\n    a,c,p=b.alloc_qubits(3)\n    for A,C,v in terms:\n        parity_into(b,x,A,a);parity_into(b,x,C,c);b.init_and(a,c,p)\n        for i in bits(v):b.cx(p,z[i])\n        b.del_and(a,c,p);parity_into(b,x,C,c);parity_into(b,x,A,a)\n    b.free([a,c,p])\n\n\n\n@lru_cache(None)\ndef tensor24_extension(K=None):\n    """105rank multiplicationtensor in explicit a+b*y, y²+y=K basis."""\n    cols,invcols,_=extension_columns(K)\n    return tuple((sum(((m&v).bit_count()%2)<<i for i,v in enumerate(cols)),apply(invcols,out)) for m,out in tensor(24))\n\n\ndef mul24_xor105(b,x,y,z,K=None):\n    """Arbitrarytarget GF24 product in explicit quadraticbasis:105T,1scratch.\n\n    No default126T multiplicationcaller ischanged; this is anoptional newAPI.\n    Allthree24-qubit registers mustbedisjoint.\n    """\n    assert len(x)==len(y)==len(z)==24\n    assert not (set(map(str,x))&set(map(str,y)) or set(map(str,x))&set(map(str,z)) or set(map(str,y))&set(map(str,z)))\n    p=b.alloc_qubits(1)[0]\n    for mask,v in tensor24_extension(K):\n        ix=list(bits(mask));pivot=ix[0];a=x[pivot];c=y[pivot]\n        for i in ix[1:]:b.cx(x[i],a);b.cx(y[i],c)\n        b.init_and(a,c,p)\n        for i in bits(v):b.cx(p,z[i])\n        b.del_and(a,c,p)\n        for i in reversed(ix[1:]):b.cx(y[i],c);b.cx(x[i],a)\n    b.free(p)\n\n\ndef mul_xor0(b,x,y,z):\n    """Standardpolynomialbasis arbitrarytarget product withZEROworkspace.\n\n    Disjointx,y,z required. Conjugateeachoutputrankvectorintoonepivot,\n    applyCCXdirectlythere,thenundooutputconjugation. Same42/105Tcounts.\n    """\n    n=len(x);assert len(y)==len(z)==n\n    assert not(set(map(str,x))&set(map(str,y)) or set(map(str,x))&set(map(str,z)) or set(map(str,y))&set(map(str,z)))\n    for mask,v in tensor(n):\n        ix=list(bits(mask));iz=list(bits(v));p=ix[0];t=iz[0]\n        for i in ix[1:]:b.cx(x[i],x[p]);b.cx(y[i],y[p])\n        for j in iz[1:]:b.cx(z[t],z[j])\n        b.ccx(x[p],y[p],z[t])\n        for j in reversed(iz[1:]):b.cx(z[t],z[j])\n        for i in reversed(ix[1:]):b.cx(y[i],y[p]);b.cx(x[i],x[p])\n\n\ndef mul24_xor105_zero(b,x,y,z,K=None):\n    """Explicita+b*y-basis105T multiplication withZEROprivatequbits."""\n    assert len(x)==len(y)==len(z)==24\n    assert not(set(map(str,x))&set(map(str,y)) or set(map(str,x))&set(map(str,z)) or set(map(str,y))&set(map(str,z)))\n    for mask,v in tensor24_extension(K):\n        ix=list(bits(mask));iz=list(bits(v));p=ix[0];t=iz[0]\n        for i in ix[1:]:b.cx(x[i],x[p]);b.cx(y[i],y[p])\n        for j in iz[1:]:b.cx(z[t],z[j])\n        b.ccx(x[p],y[p],z[t])\n        for j in reversed(iz[1:]):b.cx(z[t],z[j])\n        for i in reversed(ix[1:]):b.cx(y[i],y[p]);b.cx(x[i],x[p])\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/arithmetic/lowwidth.py': '"""Streamed towerinversion and25-scratch preserved-factor division candidates."""\nimport field as ar\nfrom field import *\nfrom division import mul24_xor,phase_product24,multiply_forward\n\n\ndef classical_transpose(b,mask,columns):\n    out=b.alloc_bits(len(columns));b.bit_store(out,False)\n    for i,c in enumerate(columns):\n        for j in bits(c):b.cx(mask[j],out[i])\n    return out\n\n\ndef inverse_stream_forward(b,x):\n    """Initialized in-place inverse with deferredoldinputphase, W(n)=n+1.\n\n    T3=3,T6=21,T12=66,T24=192. Transcriptisintheoriginalpolynomialbasis.\n    Allinternalnorm transcriptsarefixedquadratically beforeparentinputchanges.\n    """\n    n=len(x)\n    if n==3:\n        y=inverse(b,x);mask=b.alloc_bits(n);b.hmr(x,mask);b.swap(x,y);b.free(y);return mask\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to)\n    N=b.alloc_qubits(k);ar._norm_xor(b,x,N,K)\n    nm=inverse_stream_forward(b,N);ar._phase_norm(b,x,nm,K);b.free(nm)\n    outb=b.alloc_qubits(k);mul_xor(b,x[k:],N,outb)\n    raw=b.alloc_bits(n);b.hmr(x[k:],raw[k:])\n    mul_xor(b,x[:k],N,x[k:]);b.cx(outb,x[k:]);b.hmr(x[:k],raw[:k])\n    b.swap(x[:k],x[k:]);b.swap(x[k:],outb);b.free(outb)\n    ar._erase_norm(b,x,N,K);b.free(N)\n    linear_inplace(b,x,frm)\n    mask=classical_transpose(b,raw,to);b.free(raw);return mask\n\n\ndef phase_product_quantum(b,x,y,mask):\n    """Phase dot(mask,x*y) with QUANTUMmask: rank(n)CCZ,0scratch."""\n    n=len(x);assert len(y)==len(mask)==n\n    for m,v in tensor(n):\n        ix=list(bits(m));iz=list(bits(v));i=ix[0];j=iz[0]\n        for k in ix[1:]:b.cx(x[k],x[i]);b.cx(y[k],y[i])\n        for k in iz[1:]:b.cx(mask[k],mask[j])\n        b.ccz(x[i],y[i],mask[j])\n        for k in reversed(iz[1:]):b.cx(mask[k],mask[j])\n        for k in reversed(ix[1:]):b.cx(y[k],y[i]);b.cx(x[k],x[i])\n\n\ndef phase_inverse_generic6(b,x,mask):\n    """9T inversephase on6inputbits,7scratch; maskclassical6."""\n    to,frm,K=tower(6);linear_inplace(b,x,to)\n    mt=classical_transpose(b,mask,frm)\n    N=b.alloc_qubits(3);ar._norm_xor(b,x,N,K);u=inverse(b,N)\n    b.cx(x[3:],x[:3]);phase_product(b,x[:3],u,mt[:3]);phase_product(b,x[3:],u,mt[3:]);b.cx(x[3:],x[:3])\n    erase_inverse(b,N,u);b.free(u);ar._erase_norm(b,x,N,K);b.free(N)\n    linear_inplace(b,x,frm);b.free(mt)\n\n\ndef phase_inverse_low12(b,x,mask):\n    """60T phaseofinverse12 using13scratch beyondinput12,maskclassical."""\n    to,frm,K=tower(12);linear_inplace(b,x,to)\n    mt=classical_transpose(b,mask,frm)\n    N=b.alloc_qubits(6);ar._norm_xor(b,x,N,K)\n    nm=inverse_stream_forward(b,N);ar._phase_norm(b,x,nm,K);b.free(nm)\n    b.cx(x[6:],x[:6]);phase_product(b,x[:6],N,mt[:6]);phase_product(b,x[6:],N,mt[6:]);b.cx(x[6:],x[:6])\n    um=b.alloc_bits(6);b.hmr(N,um);b.free(N)\n    N=b.alloc_qubits(6);ar._norm_xor(b,x,N,K);phase_inverse_generic6(b,N,um);b.free(um)\n    ar._erase_norm(b,x,N,K);b.free(N);linear_inplace(b,x,frm);b.free(mt)\n\n\ndef fix_scaled_half_masks(b,head,factor,masks,K):\n    """Phase sum_j dot(maskj,headj*norm(factor)):42CCZ,12scratch.\n\n    Thetwoheadhalves shareonequantum12-bittransposedmask, hence42not84CCZ.\n    """\n    L=b.alloc_qubits(12)\n    def coefficient_mask():\n        for i in range(12):\n            for half in range(2):\n                for j in range(12):\n                    for r in bits(field(12).mul(1<<i,1<<j)):\n                        b.ccx(masks[half][r],head[12*half+j],L[i])\n    coefficient_mask()\n    phase_linear(b,factor[:12],L,square_columns(12))\n    phase_linear(b,factor[12:],L,[field(12).mul(K,v) for v in square_columns(12)])\n    phase_product_quantum(b,factor[:12],factor[12:],L)\n    coefficient_mask();b.free(L)\n\n\ndef divide_lowwidth(b,x,y,K=None):\n    """441T division x/=y,preservingy,25workspace; promise x0wheny0."""\n    if K is None:K=extension_columns()[2]\n    b.cx(y[12:],y[:12]);oldmask=multiply_forward(b,x,y,K);b.cx(y[12:],y[:12])\n    u=b.alloc_qubits(12);ar._norm_xor(b,y,u,K)\n    nm=inverse_stream_forward(b,u);ar._phase_norm(b,y,nm,K);b.free(nm)\n    masks=[]\n    for offset in(0,12):\n        half=x[offset:offset+12];out=b.alloc_qubits(12);mul_xor(b,half,u,out)\n        m=b.alloc_bits(12);b.hmr(half,m);masks.append(m);b.swap(half,out);b.free(out)\n    fix_scaled_half_masks(b,x,y,masks,K)\n    for m in masks:b.free(m)\n    um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    N=b.alloc_qubits(12);ar._norm_xor(b,y,N,K);phase_inverse_low12(b,N,um);b.free(um)\n    ar._erase_norm(b,y,N,K);b.free(N)\n    phase_product24(b,x,y,oldmask,K);b.free(oldmask)\n\n\ndef phase_inverse36(b,x,mask):\n    """Inverse12phase:36Toffolis,13scratch beyondinput12.\n\n    Norm6cost15; nestednorm3+inverse3cost9; twoGF3trilinearphasescost12CCZ.\n    Theirthirdquantumcontroliscomputedoneparityatatimefromxandclassicalmask,\n    so no6-qubittransposed-maskbank isneeded.\n    """\n    to12,frm12,K12=tower(12);linear_inplace(b,x,to12)\n    mt=classical_transpose(b,mask,frm12)\n    D=b.alloc_qubits(6);ar._norm_xor(b,x,D,K12)\n    to6,frm6,K6=tower(6);linear_inplace(b,D,to6)\n    E=b.alloc_qubits(3);ar._norm_xor(b,D,E,K6);w=inverse(b,E)\n    anc=b.alloc_qubits(1)[0];f6=field(6)\n    def coefficient(h):\n        for j in range(6):\n            for r in bits(f6.mul(h,1<<j)):\n                b.ccx(mt[r],x[j],anc);b.ccx(mt[r],x[j+6],anc);b.ccx(mt[r+6],x[j+6],anc)\n    for segment in(0,1):\n        if segment==0:b.cx(D[3:],D[:3])\n        controls=D[:3] if segment==0 else D[3:]\n        for m,v in tensor(3):\n            h=apply(frm6,v<<(3*segment));ix=list(bits(m));p=ix[0]\n            coefficient(h)\n            for j in ix[1:]:b.cx(controls[j],controls[p]);b.cx(w[j],w[p])\n            b.ccz(controls[p],w[p],anc)\n            for j in reversed(ix[1:]):b.cx(w[j],w[p]);b.cx(controls[j],controls[p])\n            coefficient(h)\n        if segment==0:b.cx(D[3:],D[:3])\n    b.free(anc);erase_inverse(b,E,w);b.free(w)\n    ar._erase_norm(b,D,E,K6);b.free(E);linear_inplace(b,D,frm6)\n    ar._erase_norm(b,x,D,K12);b.free(D);linear_inplace(b,x,frm12);b.free(mt)\n\n\ndef divide_lowwidth375(b,x,y,K=None):\n    """375T preserved-factor division at25scratch,promisedx0wheny0."""\n    if K is None:K=extension_columns()[2]\n    b.cx(y[12:],y[:12]);oldmask=multiply_forward(b,x,y,K);b.cx(y[12:],y[:12])\n    u=b.alloc_qubits(12);ar._norm_xor(b,y,u,K)\n    nm=inverse_stream_forward(b,u);ar._phase_norm(b,y,nm,K);b.free(nm)\n    masks=[]\n    for offset in(0,12):\n        half=x[offset:offset+12];out=b.alloc_qubits(12);mul_xor(b,half,u,out)\n        m=b.alloc_bits(12);b.hmr(half,m);masks.append(m);b.swap(half,out);b.free(out)\n    um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    N=b.alloc_qubits(12);ar._norm_xor(b,y,N,K)\n    for offset,m in zip((0,12),masks):phase_product(b,x[offset:offset+12],N,m);b.free(m)\n    phase_inverse36(b,N,um);b.free(um)\n    ar._erase_norm(b,y,N,K);b.free(N)\n    phase_product24(b,x,y,oldmask,K);b.free(oldmask)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/confluent_extract/kernel.py': '"""Private fixed Hermite map and injectable singleton-first jet adapter.\n\nNo stable decoder modules are edited. Pass the arithmetic teammate\'s jet\nforward/reverse kernels; both operate on two24-bit data registers and return\nor consume classical transcripts, with no retained quantum cache.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.decoder.common import combine,cx,scalepair,linear_words,K\nfrom exp.inverse12_escape.joint import inverse24_forward190,inverse24_reverse209\ntry:\n    from .model import constants,plan\nexcept ImportError:\n    from model import constants,plan\n\ndef initial(b,words,reverse=False):\n    assert len(words)%2==0\n    linear_words(b,words,constants(len(words)//2)[3],reverse)\n\ndef translate(b,words,head,target,c):\n    a,z=words[2*head:2*head+2]\n    b.cx(a,words[target]);cx(b,z,words[target],c);b.cx(z,words[target+1]);b.cx(z,words[target+2])\n\ndef extract(b,words,jet_forward,jet_reverse,reverse=False,transcript=None,\n            inverse_forward=inverse24_forward190,inverse_reverse=inverse24_reverse209):\n    """Return/consume (kind,token) history; output head pairs alias input bank.\n\n    jet_forward(b,u24,d24,K) -> token\n    jet_reverse(b,u24,d24,token,K) restores both data and closes all phases.\n    """\n    h=len(words)//2;history=[]if not reverse else list(transcript)\n    for op in (reversed(plan(h))if reverse else plan(h)):\n        if op[\'kind\']==\'coordinates\':cx(b,words[2*op[\'head\']+1],words[2*op[\'head\']],op[\'c\']);continue\n        if op[\'kind\']==\'local\':\n            i=2*op[\'head\'];j=i+2;x=combine(words[j:j+2])\n            if not reverse:\n                b.cx(words[i+1],words[j]);history.append((\'local\',inverse_forward(b,x,K)))\n                cx(b,words[j+1],words[j],op[\'c\'])\n            else:\n                cx(b,words[j+1],words[j],op[\'c\']);kind,t=history.pop();assert kind==\'local\'\n                inverse_reverse(b,x,t,K);b.cx(words[i+1],words[j])\n            continue\n        j=op[\'target\'];u=combine(words[j:j+2]);d=combine(words[j+2:j+4]);q=op[\'q\']\n        if not reverse:\n            translate(b,words,op[\'head\'],j,op[\'c\']);history.append((\'jet\',jet_forward(b,u,d,K)))\n            scalepair(b,d,q);b.cx(u,d);scalepair(b,u,q)\n        else:\n            from exp.decoder.common import inv\n            scalepair(b,u,inv(q));b.cx(u,d);scalepair(b,d,inv(q))\n            kind,t=history.pop();assert kind==\'jet\';jet_reverse(b,u,d,t,K)\n            translate(b,words,op[\'head\'],j,op[\'c\'])\n    if reverse:assert not history\n    return history\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/confluent_extract/model.py': '"""Singleton-first confluent quadratic-node map and zero-safe jet EXTRACT.\n\nAll polynomial/rational references are classical. No decoder/verifier claim.\n"""\nfrom functools import lru_cache\nfrom pathlib import Path\nimport sys,json,random,time\nROOT=Path(__file__).resolve().parents[2]\nsys.path.insert(0,str(ROOT/\'playbook/exp/cf_check\'))\nfrom audit import Field\nimport audit as cf_audit\nf=Field(12,0x1053);assert f.K==2048\n\ndef eadd(a,b):return a^b\ndef esquare(a):return f.emul(a,a)\ndef ehead(a,b,c):return (a^f.mul(b,c))|(b<<12)\ndef coords(u,c):\n    b=u>>12;return (u&4095)^f.mul(b,c),b\ndef get(bank,j):return bank[j]|(bank[j+1]<<12)\ndef put(bank,j,x):bank[j]=x&4095;bank[j+1]=x>>12\n\ndef synthetic(poly,gamma):\n    assert len(poly)>=2\n    Q=[0]*(len(poly)-1);Q[-1]=poly[-1]\n    for i in reversed(range(len(Q)-1)):Q[i]=poly[i+1]^f.emul(gamma,Q[i+1])\n    return Q,poly[0]^f.emul(gamma,Q[0])\n\n@lru_cache(None)\ndef constants(h=35):\n    assert h%2 and h>=1;d=(h-1)//2\n    groups=[dict(c=0,kappa=f.K,gamma=4096,head=0,word=0,double=False)]\n    headcs=[0]\n    for j in range(1,d+1):\n        c=2*j;groups.append(dict(c=c,kappa=f.mul(c,c)^c^f.K,gamma=c|4096,head=2*j-1,word=4*j-2,double=True));headcs.extend([c,c])\n    kappas=[f.mul(c,c)^c^f.K for c in headcs];P=[1]\n    for kap in kappas:P=f.pmul(P,[kap,1,1])\n    matrix=[]\n    for group in groups:\n        Q,rem=synthetic(P,group[\'gamma\']);assert rem==0\n        matrix.append([q&4095 for q in Q]);matrix.append([q>>12 for q in Q])\n        if group[\'double\']:\n            QQ,rem=synthetic(Q,group[\'gamma\']);assert rem==0\n            QQ=QQ+[0]\n            matrix.append([q&4095 for q in QQ]);matrix.append([q>>12 for q in QQ])\n    assert len(matrix)==2*h and all(len(r)==2*h for r in matrix)\n    return tuple(headcs),tuple(kappas),tuple(P),tuple(tuple(r)for r in matrix),tuple(groups)\n\n@lru_cache(None)\ndef inverse_matrix(h=35):\n    M=constants(h)[3];n=len(M);a=[list(row)+[int(i==j)for j in range(n)]for i,row in enumerate(M)]\n    for j in range(n):\n        p=next(i for i in range(j,n)if a[i][j]);a[j],a[p]=a[p],a[j]\n        iv=f.inv(a[j][j]);a[j]=[f.mul(v,iv)for v in a[j]]\n        for i in range(n):\n            if i!=j and a[i][j]:\n                q=a[i][j];a[i]=[x^f.mul(q,y)for x,y in zip(a[i],a[j])]\n    assert all(a[i][j]==int(i==j)for i in range(n)for j in range(n))\n    return tuple(tuple(row[n:])for row in a)\n\ndef matrix_apply(M,s):\n    out=[]\n    for row in M:\n        v=0\n        for a,x in zip(row,s):v^=f.mul(a,x)\n        out.append(v)\n    return out\n\ndef initial(s,h=35,reverse=False):return matrix_apply(inverse_matrix(h)if reverse else constants(h)[3],s)\n\n@lru_cache(None)\ndef plan(h=35):\n    groups=constants(h)[4];ops=[]\n    for i,g in enumerate(groups):\n        ops.append(dict(kind=\'coordinates\',head=g[\'head\'],c=g[\'c\']))\n        for target in groups[i+1:]:ops.append(dict(kind=\'jet\',head=g[\'head\'],target=target[\'word\'],c=target[\'c\'],q=g[\'kappa\']^target[\'kappa\']))\n        if g[\'double\']:\n            ops.append(dict(kind=\'local\',head=g[\'head\'],c=g[\'c\']))\n            for target in groups[i+1:]:ops.append(dict(kind=\'jet\',head=g[\'head\']+1,target=target[\'word\'],c=target[\'c\'],q=g[\'kappa\']^target[\'kappa\']))\n    assert sum(op[\'kind\']==\'jet\'for op in ops)==((h-1)//2)**2\n    assert sum(op[\'kind\']==\'local\'for op in ops)==(h-1)//2\n    return tuple(ops)\n\ndef J(u,v):\n    z=int(u==0);ui=f.einv(u);return ui,f.emul(v,esquare(ui^z))\n\ndef extract(bank,h=35,reverse=False,events=None):\n    bank=list(bank);assert len(bank)==2*h\n    for op in (reversed(plan(h))if reverse else plan(h)):\n        if op[\'kind\']==\'coordinates\':bank[2*op[\'head\']]^=f.mul(op[\'c\'],bank[2*op[\'head\']+1]);continue\n        if op[\'kind\']==\'local\':\n            i=2*op[\'head\'];j=i+2\n            if not reverse:\n                u=get(bank,j)^bank[i+1]\n                if events is not None and not u:events.append(dict(kind=\'local_zero_inverse\',head=op[\'head\']+1))\n                a,b=coords(f.einv(u),op[\'c\']);bank[j:j+2]=[a,b]\n            else:\n                a,b=bank[j:j+2];put(bank,j,f.einv(ehead(a,b,op[\'c\']))^bank[i+1])\n            continue\n        i=2*op[\'head\'];j=op[\'target\'];a,b=bank[i:i+2];u=get(bank,j);v=get(bank,j+2);q=op[\'q\'];assert q\n        if not reverse:\n            u^=ehead(a,b,op[\'c\']);v^=b\n            if events is not None and not u:events.append(dict(kind=\'jet_zero_value\',head=op[\'head\'],target=j//2,derivative_nonzero=bool(v)))\n            u,v=J(u,v);v=u^f.escale(v,q);u=f.escale(u,q)\n        else:\n            u=f.escale(u,f.inv(q));v=f.escale(v^u,f.inv(q));u,v=J(u,v);u^=ehead(a,b,op[\'c\']);v^=b\n        put(bank,j,u);put(bank,j+2,v)\n    return bank\n\ndef pair_from_residues(roots,weights):\n    D=[1]\n    for root in roots:D=f.pmul(D,[root,1])\n    N=[0]\n    for root,w in zip(roots,weights):\n        Q,rem=f.pdiv(D,[root,1]);assert rem==[0];N=f.add(N,f.scale(Q,w))\n    return N,D\n\ndef instance(h,roots,weights):\n    P=constants(h)[2];syndrome=[0]*(2*h);errors=[]\n    for root,w in zip(roots,weights):\n        pb=f.ev(P,root);assert pb and not pb>>12\n        start=f.div(w,pb);errors.append(f.div(start,root));v=start\n        for j in range(2*h):syndrome[j]^=v;v=f.mul(v,root)\n    N,D=pair_from_residues(roots,weights)\n    return dict(h=h,roots=roots,residues=weights,errors=errors,syndrome=syndrome,N=N,D=D)\n\ndef direct_jets(roots,weights,h=35):\n    out=[]\n    for group in constants(h)[4]:\n        u=v=0\n        for root,w in zip(roots,weights):\n            t=f.einv(group[\'gamma\']^root);u^=f.escale(t,w);v^=f.escale(esquare(t),w)\n        out.extend([u&4095,u>>12])\n        if group[\'double\']:out.extend([v&4095,v>>12])\n    return out\n\ndef polynomial_oracle(N,D,h=35):\n    cs,ks,*_=constants(h);heads=[];term=None\n    for i,(c,kap)in enumerate(zip(cs,ks)):\n        gamma=c|4096;de=f.ev(D,gamma)\n        if not de:return dict(heads=heads,termination=term,pole=i)\n        u=f.emul(f.ev(N,gamma),f.einv(de));a,b=coords(u,c);heads.extend([a,b])\n        H=f.add(N,f.pmul([a,b],D));Q,rem=f.pdiv(H,[kap,1,1]);assert rem==[0]\n        if H==[0]:term=i;heads.extend([0]*(2*h-len(heads)));break\n        N,D=D,Q\n    return dict(heads=heads,termination=term,pole=None)\n\ndef trial(case):\n    h=case[\'h\'];s=case[\'syndrome\'];jet=initial(s,h);assert jet==direct_jets(case[\'roots\'],case[\'residues\'],h)\n    assert initial(jet,h,True)==s\n    events=[];heads=extract(jet,h,events=events);assert extract(heads,h,True)==jet\n    reference=polynomial_oracle(case[\'N\'],case[\'D\'],h)\n    match=heads==reference[\'heads\']if reference[\'pole\']is None else None\n    raw_ratio=None;raw_padding=None;raw_squarefree=None\n    if reference[\'pole\']is None:\n        assert match\n        pairs=[heads[i:i+2]for i in range(0,len(heads),2)]\n        A,B,tail=cf_audit.synth(f,constants(h)[1],pairs)\n        raw_ratio=f.pmul(A,case[\'D\'])==f.pmul(B,case[\'N\']);assert raw_ratio\n        expected=case[\'D\'][:]\n        for i in range(len(case[\'roots\'])+1,h,2):expected=f.pmul(expected,[constants(h)[1][i],1,1])\n        if not len(case[\'roots\'])%2:expected=[0]+expected\n        raw_padding=B==expected;assert raw_padding\n        raw_squarefree=f.gcd(B,f.derivative(B))==[1];assert raw_squarefree\n        assert len(B)==h+1 and B[-1]==1\n    else:assert heads[:len(reference[\'heads\'])]==reference[\'heads\']\n    return dict(h=h,weight=len(case[\'roots\']),oracle_pole=reference[\'pole\'],termination=reference[\'termination\'],\n        heads_match=match,jet_zero_events=sum(e[\'kind\']==\'jet_zero_value\'for e in events),\n        nonzero_derivative_zero_events=sum(e[\'kind\']==\'jet_zero_value\'and e[\'derivative_nonzero\']for e in events),\n        local_zero_inverses=sum(e[\'kind\']==\'local_zero_inverse\'for e in events),roundtrip=True,\n        raw_synthesis_ratio=raw_ratio,raw_synthesis_padding=raw_padding,raw_synthesis_squarefree=raw_squarefree)\n\ndef main(repeats=4):\n    start=time.time();rng=random.Random(202610270901);tests=[];arbitrary=[]\n    for h in[3,9,35]:\n        inverse_matrix(h)\n        for weight in range(h+1):\n            for repeat in range(repeats):\n                roots=rng.sample(range(1,4096),weight);weights=[rng.randrange(1,4096)for _ in roots]\n                tests.append(trial(instance(h,roots,weights)))\n        for j in range(300):\n            bank=[rng.randrange(4096)for _ in range(2*h)]\n            if j%3==0:\n                for group in constants(h)[4][1:]:bank[group[\'word\']]=bank[group[\'word\']+1]=0\n            if j%3==1:bank=[0 if rng.randrange(2)else v for v in bank]\n            assert extract(extract(bank,h),h,True)==bank\n        arbitrary.append(dict(h=h,cases=300,includes_zero_values_and_zero_jets=True))\n        print(\'h\',h,\'done\',flush=True)\n    peer=json.loads((ROOT/\'exp/confluent_synthesis/fixtures.json\').read_text());compared=0\n    for fixture in peer:\n        h=fixture[\'h\'];case=instance(h,fixture[\'roots\'],fixture[\'weights\']);heads=extract(initial(case[\'syndrome\'],h),h)\n        assert heads==[v for head in fixture[\'heads\']for v in head];compared+=1\n        A,B,_=cf_audit.synth(f,constants(h)[1],[heads[i:i+2]for i in range(0,len(heads),2)])\n        assert A==fixture[\'rawA\']and B==fixture[\'rawB\']\n    result=dict(tests=tests,arbitrary=arbitrary,peer_fixture_heads_compared=compared,\n        matrix_ranks={h:2*h for h in[3,9,35]},counts={h:dict(jets=sum(o[\'kind\']==\'jet\'for o in plan(h)),local_inverses=sum(o[\'kind\']==\'local\'for o in plan(h)))for h in[3,9,35]},\n        seconds=time.time()-start,meaning=\'Classical exactness and fixed-linear map; no full decoder/verifier\')\n    print(json.dumps(dict(cases=len(tests),heads_bad=sum(t[\'heads_match\']is False for t in tests),poles=sum(t[\'oracle_pole\']is not None for t in tests),\n        peer_fixture_heads_compared=compared,arbitrary=arbitrary,seconds=result[\'seconds\']),indent=2))\n    Path(__file__).with_name(\'results.json\').write_text(json.dumps(result,indent=2)+\'\\n\')\n\nif __name__==\'__main__\':main()\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/confluent_synthesis/kernel.py': '"""Read-only constants adapter for frozen alternating-gauge synthesis."""\nfrom pathlib import Path\nimport sys\nfrom functools import lru_cache\nfrom types import FunctionType,SimpleNamespace\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.alternating_gauge import kernel as base\nfrom exp.decoder.common import mul,K,pmul\n@lru_cache(None)\ndef constants(h=35):\n assert h>=3 and h%2==1\n cs=[0]+[2*j for j in range(1,(h+1)//2)for _ in range(2)]\n ks=[mul(c,c)^c^K for c in cs];P=[1]\n for k in ks:P=pmul(P,[k,1,1])\n return cs,ks,P,()\nold=SimpleNamespace(**dict(vars(base.old),constants=constants))\nsynthesize=FunctionType(base.synthesize.__code__,dict(base.synthesize.__globals__,old=old),\'synthesize\',base.synthesize.__defaults__,base.synthesize.__closure__)\nunsynthesize=FunctionType(base.unsynthesize.__code__,dict(base.unsynthesize.__globals__,old=old),\'unsynthesize\',base.unsynthesize.__defaults__,base.unsynthesize.__closure__)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/convert/convert.py': '"""Reversible fixed-generic-chart half-degree CONVERT.\n\nPublic convert(builder,A[35] words,B[35] words,reverse=False).\nA,B hold GF12 polynomial coefficients; B35=1 implicit.\nSafe divisors make roundtrip exact even on degree gaps; forward decode is only\nproved for generic Euclid degrees. Runtime gap2 routing is not yet implemented.\n"""\nfrom contextlib import contextmanager\nimport sys\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parent))\nfrom kernels import ar,cx,convolution_xor,hankel\nfrom lowwidth import phase_inverse36\n\ndef zero_flag(b,a):\n    for q in a:b.x(q)\n    nodes=[];left=a[0]\n    for q in a[1:]:\n        out=b.alloc_qubits(1)[0];b.init_and(left,q,out);nodes.append((left,q,out));left=out\n    for x,y,z in reversed(nodes[:-1]):b.del_and(x,y,z);b.free(z)\n    for q in a:b.x(q)\n    return nodes[-1][2]\n\ndef erase_zero_flag(b,a,flag):\n    for q in a:b.x(q)\n    nodes=[];left=a[0]\n    for q in a[1:-1]:\n        out=b.alloc_qubits(1)[0];b.init_and(left,q,out);nodes.append((left,q,out));left=out\n    b.del_and(left,a[-1],flag);b.free(flag)\n    for x,y,z in reversed(nodes):b.del_and(x,y,z);b.free(z)\n    for q in a:b.x(q)\n\n@contextmanager\ndef inverse_context(b,a):\n    flag=zero_flag(b,a);b.cx(flag,a[0])\n    ai=ar.inverse(b,a)\n    yield ai\n    mask=b.alloc_bits(12);b.hmr(ai,mask);b.free(ai)\n    phase_inverse36(b,a,mask);b.free(mask)\n    b.cx(flag,a[0]);erase_zero_flag(b,a,flag)\n\ndef scale(b,x,a,ai,divide=True):\n    out=b.alloc_qubits(12);ar.mul_xor(b,x,ai if divide else a,out)\n    ar.erase_product(b,a if divide else ai,out,x)\n    b.swap(x,out);b.free(out)\n\ndef runops(b,ops,a,ai,undo=False):\n    for op in (reversed(ops) if undo else ops):\n        if op[0]==\'s\':scale(b,op[1],a,ai,op[2]^undo)\n        elif op[0]==\'m\':ar.mul_xor(b,op[1],op[2],op[3])\n        elif op[0]==\'c\':convolution_xor(b,op[1],op[2],op[3],undo)\n        else:raise ValueError(op[0])\n\ndef pre(b,f,g,payloads,undo=False):\n    m=len(f)-1;n=m-1;a=g[-1];q0=f[-2]\n    with inverse_context(b,a) as ai:\n        q1=ai if f[-1] is None else f[-1]\n        ops=[(\'s\',v[-1],True) for v in payloads]\n        ops += [(\'m\',g[i],v[-1],v[i]) for v in payloads for i in range(n)]\n        if f[-1] is not None:ops += [(\'s\',q1,True)]\n        ops += [(\'m\',q1,g[-2],q0),(\'s\',q0,True),(\'c\',g[:-1],[q0,q1],f[:-2])]\n        runops(b,ops,a,ai,undo)\n\ndef post(b,f,g,payloads,undo=False):\n    n=len(g)-1;a=g[-1];q0=f[-2]\n    def dots():\n        for v in payloads:\n            for i in range(n):ar.mul_xor(b,g[i],v[i],v[-1])\n    if f[-1] is None:\n        with inverse_context(b,a) as ai:\n            q1=ai\n            ops=[(\'s\',v[-1],False) for v in payloads]\n            ops += [(\'m\',g[i],v[i],v[-1]) for v in payloads for i in range(n)]\n            ops += [(\'s\',v[-1],True) for v in payloads]\n            ops += [(\'c\',g[:-1],[q0,q1],f[:-2]),(\'s\',q0,False),(\'m\',q1,g[-2],q0)]\n            runops(b,ops,a,ai,undo)\n    else:\n        q1=f[-1]\n        def qscale(reverse):\n            with inverse_context(b,q1) as qi:\n                for v in payloads:scale(b,v[-1],q1,qi,not reverse)\n        def last(reverse):\n            with inverse_context(b,a) as ai:\n                ops=[(\'s\',v[-1],True) for v in payloads]\n                ops += [(\'c\',g[:-1],[q0,q1],f[:-2]),(\'s\',q0,False),(\'m\',q1,g[-2],q0),(\'s\',q1,False)]\n                runops(b,ops,a,ai,reverse)\n        if not undo:qscale(False);dots();last(False)\n        else:last(True);dots();qscale(True)\n\ndef raw_solve(b,f,g,payloads,reverse=False):\n    m=len(f)-1\n    assert len(g)==m and all(len(v)==m for v in payloads)\n    if m==1:\n        divisors=[p for p in (f[-1],g[-1]) if p is not None]\n        for a in (reversed(divisors) if reverse else divisors):\n            with inverse_context(b,a) as ai:\n                for v in payloads:scale(b,v[0],a,ai,not reverse)\n        return\n    low=[v[:-1] for v in payloads]\n    if not reverse:\n        pre(b,f,g,payloads);raw_solve(b,g,f[:-2],low);post(b,f,g,payloads)\n    else:\n        post(b,f,g,payloads,True);raw_solve(b,g,f[:-2],low,True);pre(b,f,g,payloads,True)\n\ndef bez_solve(b,O,E,payloads,reverse=False):\n    def reduce():\n        for i in range(len(O)):ar.mul_xor(b,E[-1],O[i],E[i])\n    reduce();raw_solve(b,O+[None],E[:-1],payloads,reverse);reduce()\n\ndef frob(b,words,power):\n    f=ar.field(12);cols=[f.pow(1<<i,power) for i in range(12)]\n    for x in words:ar.linear_inplace(b,x,cols)\n\ndef convert(b,A,B,reverse=False):\n    assert len(A)==len(B)==35\n    E=B[::2];O=B[1::2];u=A[1::2];w=A[:34:2];top=A[-1]\n    if reverse:hankel(b,B,A,True)\n    frob(b,B,2048);frob(b,A,2048)\n    if reverse:\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n        for i in range(17):ar.mul_xor(b,O[i],u[i],top)\n        bez_solve(b,O,E,[u,w],True)\n        for i in range(17):ar.mul_xor(b,top,O[i],w[i])\n    else:\n        for i in range(17):ar.mul_xor(b,top,O[i],w[i])\n        bez_solve(b,O,E,[u,w])\n        for i in range(17):ar.mul_xor(b,O[i],u[i],top)\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n    frob(b,A,2);frob(b,B,2)\n    if not reverse:hankel(b,B,A)\n\ndef raw_solve_blocked(b,f,g,payloads,block_size=2,reverse=False):\n    """Closed checkpointed scalar schedule grouped into quotient blocks.\n\n    This is the no-speedup control construction: at most2*block_size quotient\n    metadata words remain in already consumed f/g coefficients; no matrix or\n    evaluation bank is materialized. Product fusion is deliberately absent.\n    """\n    frames=[]\n    for _ in range(block_size):\n        if len(f)==2:break\n        frames.append((f,g,payloads))\n        if not reverse:pre(b,f,g,payloads)\n        else:post(b,f,g,payloads,True)\n        m=len(f)-1;f,g=g,f[:-2];payloads=[v[:-1]for v in payloads]\n    if len(f)==2:raw_solve(b,f,g,payloads,reverse)\n    else:raw_solve_blocked(b,f,g,payloads,block_size,reverse)\n    for f,g,payloads in reversed(frames):\n        if not reverse:post(b,f,g,payloads)\n        else:pre(b,f,g,payloads,True)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/convert/hankel_check.py': '"""Independent GF(2^12) rank-one, in-place Hankel audit; no verifier."""\nimport random, json\nfrom functools import lru_cache\nMOD=0x1053\n\ndef slowmul(a,b):\n    c=0\n    for _ in range(12):\n        if b&1:c^=a\n        b>>=1;a<<=1\n        if a&4096:a^=MOD\n    return c\nEXP=[1]*8190;LOG=[-1]*4096\nfor i in range(1,4095):EXP[i]=slowmul(EXP[i-1],2)\nassert len(set(EXP[:4095]))==4095\nfor i in range(4095):LOG[EXP[i]]=i;EXP[i+4095]=EXP[i]\ndef mul(a,b):return EXP[LOG[a]+LOG[b]] if a and b else 0\ndef inv(a):assert a;return EXP[4095-LOG[a]]\ndef power(a,n):return EXP[(LOG[a]*n)%4095] if a else int(n==0)\ndef dot(a,b):\n    out=0\n    for x,y in zip(a,b):out^=mul(x,y)\n    return out\n\ndef mi(a):\n    n=len(a);a=[r[:]+[int(i==j) for j in range(n)] for i,r in enumerate(a)]\n    for j in range(n):\n        p=next(i for i in range(j,n) if a[i][j]);a[p],a[j]=a[j],a[p]\n        t=inv(a[j][j]);a[j]=[mul(t,v) for v in a[j]]\n        for i in range(n):\n            if i!=j:\n                t=a[i][j];a[i]=[x^mul(t,y) for x,y in zip(a[i],a[j])]\n    return [r[n:] for r in a]\n\n@lru_cache(None)\ndef terms(k,m):\n    n=k+m-1;points=list(range(n));V=[[power(t,j) for j in range(n)] for t in points];W=mi(V)\n    # Cross = sum_t source(reverse(low),V[t]) * coefficient(kernel,W[:,t]) * V[t,:m]\n    return [(V[t][:k][::-1],[W[j][t] for j in range(n)],V[t][:m]) for t in range(n)]\n\ndef cross(q,low,high):\n    count=0\n    for source,coef,target in terms(len(low),len(high)):\n        p=mul(dot(source,low),dot(coef,q));count+=1\n        for i,c in enumerate(target):high[i]^=mul(c,p)\n    return count\n\ndef tri(q,x,reverse=False):\n    n=len(x)\n    if n==1:return 0\n    k=n//2;m=n-k;lo=x[:k];hi=x[k:]\n    if not reverse:\n        count=tri(q[:m],hi);count+=cross(q[1:n],lo,hi);count+=tri(q[:k],lo)\n    else:\n        count=tri(q[:k],lo,True);count+=cross(q[1:n],lo,hi);count+=tri(q[:m],hi,True)\n    x[:]=lo+hi\n    return count\n\ndef main():\n    rng=random.Random(102617147);tests=0;rank_counts={}\n    for n in (2,3,4,8,16,17,35):\n        for _ in range(100):\n            q=[1]+[rng.randrange(4096) for _ in range(n-1)];x=[rng.randrange(4096) for _ in range(n)];orig=x[:]\n            target=[dot(q[:i+1],orig[i::-1]) for i in range(n)]\n            c=tri(q,x);assert x==target\n            cr=tri(q,x,True);assert x==orig and c==cr\n            rank_counts[n]=c;tests+=1\n    # H_ij=B_(i+j+1), coefficients B_35=1, higher 0. H=J*T with q_s=B_(35-s).\n    for _ in range(100):\n        b=[rng.randrange(4096) for _ in range(35)]+[1];x=[rng.randrange(4096) for _ in range(35)];old=x[:]\n        expected=[dot((b[i+j+1] if i+j+1<=35 else 0 for j in range(35)),old) for i in range(35)]\n        c=tri(list(reversed(b[1:])),x);x.reverse();assert x==expected and c==147\n        x.reverse();tri(list(reversed(b[1:])),x,True);assert x==old\n    out={\'GF12_modulus\':hex(MOD),\'triangular_forward_reverse_cases\':tests,\'hankel35_forward_reverse_cases\':100,\'products\':rank_counts,\'H35_forward_T\':147*42,\'H35_pair_T\':294*42,\'extra_words\':0,\'inversions\':0,\'measurement_phase_corrections\':0}\n    print(json.dumps(out,indent=2))\nif __name__==\'__main__\':main()\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/convert/kernels.py': '"""In-place structured GF12 kernels using exp/arithmetic primitives."""\nimport sys\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/\'arithmetic\'))\nimport field as ar\nfrom hankel_check import mul,inv,mi,power\nfrom functools import lru_cache\n\ndef ccols(c):return [mul(c,1<<i) for i in range(12)]\ndef cx(b,x,y,c=1):\n    if c==1:b.cx(x,y)\n    elif c:ar.linear_xor(b,x,y,ccols(c))\ndef scale_const(b,x,c):\n    if c!=1:ar.linear_inplace(b,x,ccols(c))\n\ndef expose(b,words,coeffs):\n    p=next(i for i,c in enumerate(coeffs) if c);c=coeffs[p]\n    scale_const(b,words[p],c)\n    for i,a in enumerate(coeffs):\n        if i!=p:cx(b,words[i],words[p],a)\n    return words[p],p\n\ndef unexpose(b,words,coeffs,p):\n    for i in reversed(range(len(coeffs))):\n        if i!=p:cx(b,words[i],words[p],coeffs[i])\n    scale_const(b,words[p],inv(coeffs[p]))\n\ndef rank_update(b,aw,ac,bw,bc,tw,tc):\n    x,ap=expose(b,aw,ac);y,bp=expose(b,bw,bc)\n    p=next(i for i,c in enumerate(tc) if c);v=tc[p];vi=inv(v)\n    for i,c in enumerate(tc):\n        if i!=p:cx(b,tw[p],tw[i],mul(c,vi))\n    scale_const(b,tw[p],vi)\n    ar.mul_xor(b,x,y,tw[p])\n    scale_const(b,tw[p],v)\n    for i in reversed(range(len(tc))):\n        if i!=p:cx(b,tw[p],tw[i],mul(tc[i],vi))\n    unexpose(b,bw,bc,bp);unexpose(b,aw,ac,ap)\n\n@lru_cache(None)\ndef vandermonde(n):\n    V=[[power(t,j) for j in range(n)] for t in range(n)]\n    return V,mi(V)\n\n@lru_cache(None)\ndef cross_terms(k,m):\n    n=k+m-1;V,W=vandermonde(n)\n    return [(V[t][:k][::-1],[W[j][t] for j in range(n)],V[t][:m]) for t in range(n)]\n\ndef tri(b,q,x,reverse=False):\n    # q=(q1,...,q_(n-1)), q0=1 implicit.\n    n=len(x)\n    if n==1:return\n    k=n//2;m=n-k\n    if not reverse:tri(b,q[:m-1],x[k:])\n    else:tri(b,q[:k-1],x[:k],True)\n    ts=cross_terms(k,m)\n    for ac,bc,tc in (reversed(ts) if reverse else ts):rank_update(b,x[:k],ac,q[:n-1],bc,x[k:],tc)\n    if not reverse:tri(b,q[:k-1],x[:k])\n    else:tri(b,q[:m-1],x[k:],True)\n\ndef hankel(b,B,A,reverse=False):\n    # H_ij=B_(i+j+1), implicit monic B_degree=len(A).\n    assert len(B)==len(A)\n    q=list(reversed(B[1:]))\n    if reverse:\n        for i in range(len(A)//2):b.swap(A[i],A[-1-i])\n        tri(b,q,A,True)\n    else:\n        tri(b,q,A)\n        for i in range(len(A)//2):b.swap(A[i],A[-1-i])\n\ndef convolution_xor(b,a,c,out,reverse=False):\n    # Coefficients 0..len(out)-1 of a*c; supports short output truncation.\n    N=len(a)+len(c)-1;V,W=vandermonde(N)\n    terms=range(N-1,-1,-1) if reverse else range(N)\n    for t in terms:\n        tc=[W[j][t] for j in range(len(out))]\n        if any(tc):rank_update(b,a,V[t][:len(a)],c,V[t][:len(c)],out,tc)\n\ndef convolution_transpose_left_xor(b,source,c,out,reverse=False):\n    # out[j] += sum_i source[i]*c[i-j], transpose of outpoly*c truncated to source.\n    N=len(out)+len(c)-1;V,W=vandermonde(N)\n    terms=range(N-1,-1,-1) if reverse else range(N)\n    for t in terms:\n        ac=[W[i][t] for i in range(len(source))]\n        if any(ac):rank_update(b,source,ac,c,V[t][:len(c)],out,V[t][:len(out)])\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/convert/repair.py': '"""Adaptive one-gap Barnett repair for raw Bezout solve.\n\nSeparate fromdefaultgenericconvert.py. Firstleading-zeroisrepairedbymultiplygbyx\nandprecompanionMx. More/largegapsandrepairwithf(0)=0decodeincorrectlybut\nforward/reversearestilllocallyphase-correctpermutations. No deferredtranscripts.\n"""\nfrom contextlib import contextmanager\nfrom convert import ar,zero_flag,erase_zero_flag,scale,pre,post,raw_solve,frob\nfrom lowwidth import phase_inverse36\nfrom kernels import hankel\n\n@contextmanager\ndef nonzero_inverse(b,a):\n    ai=ar.inverse(b,a)\n    yield ai\n    mask=b.alloc_bits(12);b.hmr(ai,mask);b.free(ai);phase_inverse36(b,a,mask);b.free(mask)\n\n@contextmanager\ndef safe_word(b,a):\n    flag=zero_flag(b,a);b.cx(flag,a[0])\n    yield\n    b.cx(flag,a[0]);erase_zero_flag(b,a,flag)\n\ndef rotate(b,v,F,reverse=False):\n    for j in (range(1,len(v)) if reverse else range(len(v)-1,0,-1)):\n        b.cswap(F,v[j],v[j-1])\n\ndef coefficient_update(b,f,u,target):\n    if f[-1] is None:b.x(f[0][0])\n    else:b.cx(f[-1],f[0])\n    ar.mul_xor(b,f[0],u,target)\n    if f[-1] is None:b.x(f[0][0])\n    else:b.cx(f[-1],f[0])\n\ndef companion(b,f,payloads,F,reverse=False):\n    """Conditionallymultiplyeachpayloadbyx modf, oritsinverse.\n\n    f0 andleadingaresafenedonlyduringthispermutation, soalwaysinvertible.\n    Masku=F*lowest/bbeforeupdates; afterupdatesu=F*lowest/f0.\n    """\n    lead=f[-1]\n    @contextmanager\n    def lead_safe():\n        if lead is None:yield\n        else:\n            with safe_word(b,lead):yield\n    @contextmanager\n    def invctx(a):\n        if a is None:yield None\n        else:\n            with nonzero_inverse(b,a) as ai:yield ai\n    with safe_word(b,f[0]),lead_safe():\n        if not reverse:\n            for v in payloads:rotate(b,v,F)\n        masks=[]\n        divisor=f[0] if reverse else lead\n        with invctx(divisor) as di:\n            for v in payloads:\n                u=b.alloc_qubits(12)\n                for j in range(12):b.init_and(F,v[0][j],u[j])\n                if divisor is not None:scale(b,u,divisor,di,True)\n                if not reverse:\n                    for i in range(1,len(v)):ar.mul_xor(b,f[i],u,v[i])\n                    coefficient_update(b,f,u,v[0])\n                else:\n                    coefficient_update(b,f,u,v[0])\n                    for i in reversed(range(1,len(v))):ar.mul_xor(b,f[i],u,v[i])\n                mask=b.alloc_bits(12);b.hmr(u,mask);b.free(u);masks.append(mask)\n        recovery=lead if reverse else f[0]\n        with invctx(recovery) as ri:\n            for v,mask in zip(payloads,masks):\n                if recovery is None:\n                    for j in range(12):b.ccz(F,v[0][j],mask[j])\n                else:\n                    tmp=b.alloc_qubits(12);ar.mul_xor(b,v[0],ri,tmp)\n                    for j in range(12):b.ccz(F,tmp[j],mask[j])\n                    ar.erase_product(b,v[0],ri,tmp);b.free(tmp)\n                b.free(mask)\n        if reverse:\n            for v in payloads:rotate(b,v,F,True)\n\ndef equal_code(b,H,code):\n    for i,q in enumerate(H):\n        if code>>i&1:b.x(q)\n    flag=zero_flag(b,H)\n    for i,q in enumerate(H):\n        if code>>i&1:b.x(q)\n    return flag\n\ndef erase_equal(b,H,code,flag):\n    for i,q in enumerate(H):\n        if code>>i&1:b.x(q)\n    erase_zero_flag(b,H,flag)\n    for i,q in enumerate(H):\n        if code>>i&1:b.x(q)\n\ndef history_arm(b,H,a,code):\n    hz=zero_flag(b,H);az=zero_flag(b,a);F=b.alloc_qubits(1)[0];b.ccx(hz,az,F)\n    erase_zero_flag(b,a,az);erase_zero_flag(b,H,hz)\n    for i,q in enumerate(H):\n        if code>>i&1:b.cx(F,q)\n    return F\n\ndef history_clear_flag(b,H,code,F):\n    eq=equal_code(b,H,code);b.cx(eq,F);erase_equal(b,H,code,eq);b.free(F)\n\ndef history_finish(b,H,a,code,F):\n    for i,q in enumerate(H):\n        if code>>i&1:b.cx(F,q)\n    hz=zero_flag(b,H);az=zero_flag(b,a);b.ccx(hz,az,F)\n    erase_zero_flag(b,a,az);erase_zero_flag(b,H,hz);b.free(F)\n\ndef repair_solve(b,f,g,payloads,H,reverse=False):\n    m=len(f)-1\n    if m==1:return raw_solve(b,f,g,payloads,reverse)\n    F=history_arm(b,H,g[-1],m)\n    rotate(b,g,F)\n    if not reverse:companion(b,f,payloads,F)\n    history_clear_flag(b,H,m,F)\n    low=[v[:-1]for v in payloads]\n    if not reverse:\n        pre(b,f,g,payloads);repair_solve(b,g,f[:-2],low,H);post(b,f,g,payloads)\n    else:\n        post(b,f,g,payloads,True);repair_solve(b,g,f[:-2],low,H,True);pre(b,f,g,payloads,True)\n    F=equal_code(b,H,m)\n    if reverse:companion(b,f,payloads,F,True)\n    rotate(b,g,F,True)\n    history_finish(b,H,g[-1],m,F)\n\ndef bez_repair(b,O,E,payloads,reverse=False):\n    def reduce():\n        for i in range(len(O)):ar.mul_xor(b,E[-1],O[i],E[i])\n    reduce();H=list(b.alloc_qubits(5));repair_solve(b,O+[None],E[:-1],payloads,H,reverse);b.free(H);reduce()\n\ndef convert_repair(b,A,B,reverse=False):\n    assert len(A)==len(B)==35\n    E=B[::2];O=B[1::2];u=A[1::2];w=A[:34:2];top=A[-1]\n    if reverse:hankel(b,B,A,True)\n    frob(b,B,2048);frob(b,A,2048)\n    if reverse:\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n        for i in range(17):ar.mul_xor(b,O[i],u[i],top)\n        bez_repair(b,O,E,[u,w],True)\n        for i in range(17):ar.mul_xor(b,top,O[i],w[i])\n    else:\n        for i in range(17):ar.mul_xor(b,top,O[i],w[i])\n        bez_repair(b,O,E,[u,w])\n        for i in range(17):ar.mul_xor(b,O[i],u[i],top)\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n    frob(b,A,2);frob(b,B,2)\n    if not reverse:hankel(b,B,A)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/convert_short_hankel/kernel.py': '"""Drop-in guarded CONVERT with exact short-H boundary.\n\nDefault preserves the stable arbitrary borrowed-history completion. Optional\nmonic_root=True requires the supplied history word initially zero; if absent\nit is allocated zero. B/payloads otherwise arbitrary, including all gaps.\nReverse infers the option from its forward phase token.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.safe_scalar import converter as old\n\n\ndef _root_ops(b,ops,a,ai,undo=False):\n for op in (reversed(ops)if undo else ops):\n  if op[0]==\'copy\':b.x(op[1][0]);b.cx(ai,op[1])\n  else:old.runops(b,[op],a,ai,undo)\n\n\ndef _pre_root(b,f,g,payloads,undo=False,incoming=None,defer=False):\n m=len(f)-1;n=m-1;a=g[-1];q0,q1=f[-2:]\n def action(ai):\n  ops=[(\'s\',v[-1],True)for v in payloads]\n  ops +=[(\'m\',g[i],v[-1],v[i])for v in payloads for i in range(n)]\n  ops +=[(\'copy\',q1),(\'m\',q1,g[-2],q0),(\'s\',q0,True),(\'c\',g[:-1],[q0,q1],f[:-2])]\n  _root_ops(b,ops,a,ai,undo)\n return old.inverse_apply(b,a,action,incoming,defer)\n\n\ndef _post_root(b,f,g,payloads,reverse=False,incoming=None,defer=False):\n n=len(g)-1;a=g[-1];q0,q1=f[-2:]\n def action(ai):\n  # q1=1/(a+[a0]), so division by q1 is multiplication by the\n  # already clipped a in this same context, even on scalar-zero charts.\n  ops=[(\'s\',v[-1],False)for v in payloads]\n  ops +=[(\'m\',g[i],v[i],v[-1])for v in payloads for i in range(n)]\n  ops +=[(\'s\',v[-1],True)for v in payloads]\n  ops +=[(\'c\',g[:-1],[q0,q1],f[:-2]),(\'s\',q0,False),(\'m\',q1,g[-2],q0),(\'copy\',q1)]\n  _root_ops(b,ops,a,ai,reverse)\n return old.inverse_apply(b,a,action,incoming,defer)\n\n\ndef _look_root(b,f,g,payloads,reverse,phase):\n m=len(f)-1;F=old.fresh_zero_flag(b,g[-1]);old.rotate(b,g,F)\n if not reverse:carry=_pre_root(b,f,g,payloads,defer=True)\n else:carry=_post_root(b,f,g,payloads,True,phase[\'a\'].pop(m),True)\n old.canonical_permute(b,f,g,F);old.clear_flag(b,g[-1],F)\n parent=([v[-1]for v in payloads],f[-1],f[-2])\n old.look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,parent,reverse,phase)\n F=old.fresh_zero_flag(b,g[-1]);old.canonical_permute(b,f,g,F,True)\n if not reverse:phase[\'a\'][m]=_post_root(b,f,g,payloads,False,carry,True)\n else:_pre_root(b,f,g,payloads,True,carry)\n old.rotate(b,g,F,True);old.clear_flag(b,g[-1],F)\n\n\ndef _bez_monic(b,O,E,payloads,reverse,history_scratch12,phase):\n def reduce():\n  for i in range(len(O)):old.ar.mul_xor0(b,E[-1],O[i],E[i])\n reduce();lead=list(b.alloc_qubits(12))if history_scratch12 is None else list(history_scratch12)\n b.x(lead[0]);_look_root(b,O+[lead],E[:-1],payloads,reverse,phase);b.x(lead[0])\n if history_scratch12 is None:b.free(lead)\n reduce()\n\n\ndef convert_safe(b,A,B,reverse=False,history_scratch12=None,phase_token=None,*,monic_root=False):\n assert len(A)==len(B)==35\n phase={\'a\':{},\'q\':{},\'_short_monic\':bool(monic_root)}if phase_token is None else phase_token\n assert not reverse or phase_token is not None\n use_monic=phase[\'_short_monic\']\n # Frobenius first on BOTH directions: reverse input is the completed C.\n old.frob(b,B,2048);old.frob(b,A,2048)\n E=B[::2];O=B[1::2];u=A[1::2];w=A[:34:2];top=A[-1]\n def bez(rev):\n  if use_monic:_bez_monic(b,O,E,[u,w],rev,history_scratch12,phase)\n  else:old.bez_lookahead(b,O,E,[u,w],rev,history_scratch12,phase)\n if not reverse:\n  for i in range(17):old.ar.mul_xor0(b,top,O[i],w[i])\n  bez(False)\n  for i in range(17):old.ar.mul_xor0(b,E[i+1],w[i],top)\n  for i in range(17):b.swap(A[2*i],A[2*i+1])\n  old.hankel(b,B[1:],A[:34])\n  for i in reversed(range(34)):b.swap(A[i],A[i+1])\n else:\n  for i in range(34):b.swap(A[i],A[i+1])\n  old.hankel(b,B[1:],A[:34],True)\n  for i in range(17):b.swap(A[2*i],A[2*i+1])\n  for i in range(17):old.ar.mul_xor0(b,E[i+1],w[i],top)\n  bez(True)\n  for i in range(17):old.ar.mul_xor0(b,top,O[i],w[i])\n old.frob(b,A,2);old.frob(b,B,2)\n if reverse:assert not phase[\'a\']and not phase[\'q\']\n return phase if not reverse else None\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/convert_short_hankel/phase34.py': '"""Private34T phase substitution; never mutates stable module globals.\n\nSame API as kernel.convert_safe. Saving2T per ultimate inverse closure;\nphase helper21private fits below the existing39-private converter peak.\n"""\nfrom types import FunctionType,SimpleNamespace\nfrom contextlib import contextmanager\nfrom exp.safe_scalar import converter as stable\nfrom exp.fold_cached_norm.phase34 import phase_inverse34\nfrom exp.convert_short_hankel import kernel as short\n\n\ndef clone(fn,scope,name=None):\n out=FunctionType(fn.__code__,scope,name or fn.__name__,fn.__defaults__,fn.__closure__)\n out.__kwdefaults__=None if fn.__kwdefaults__ is None else dict(fn.__kwdefaults__)\n return out\n\n# Erase/inverse_apply_safe are imported closures with different globals;\n# clone the chain explicitly before rebinding the converter namespace.\n_erase=clone(stable.erase,dict(stable.erase.__globals__,phase_inverse36_zero=phase_inverse34))\n_apply=clone(stable.inverse_apply_safe,dict(stable.inverse_apply_safe.__globals__,erase=_erase))\n_scope=dict(vars(stable),erase=_erase,inverse_apply_safe=_apply)\n_qgen=clone(stable.q_inverse.__wrapped__,_scope,\'q_inverse\')\n_scope[\'q_inverse\']=contextmanager(_qgen)\nfor _name in [\'scale\',\'runops\',\'zero_toggle\',\'inverse_apply\',\'pre\',\'canonical_permute\',\'masked_product\',\'fresh_zero_flag\',\'clear_flag\',\'q_post\',\'post_look\',\'base\',\'look_solve\',\'bez_lookahead\']:\n _scope[_name]=clone(getattr(stable,_name),_scope,_name)\n_private=SimpleNamespace(**_scope)\n\n# Clone short-H and optional monic-root functions against that private graph.\n_short_scope=dict(vars(short),old=_private)\nfor _name in [\'_root_ops\',\'_pre_root\',\'_post_root\',\'_look_root\',\'_bez_monic\',\'convert_safe\']:\n _short_scope[_name]=clone(getattr(short,_name),_short_scope,_name)\nconvert_safe=_short_scope[\'convert_safe\']\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/decoder/common.py': '"""Lead\'s finite-field and reversible linear utilities (GF12 word bank)."""\nfrom functools import lru_cache\nfrom pathlib import Path\nimport sys\nimport circ as km\n\nROOT=Path(__file__).resolve().parents[2]\nsys.path.insert(0,str(ROOT))\nsys.path.insert(0,str(ROOT/\'exp/arithmetic\'))\nimport field as ar\ngf=km.GF2Field(12)\nK=2048\n\ndef mul(a,b):return gf.mul(a,b)\ndef inv(a):return gf.invert(a) if a else 0\ndef power(a,n):return gf.pow(a,n)\ndef xor(xs):\n    out=0\n    for x in xs:out^=x\n    return out\ndef trim(p):\n    p=list(p)\n    while len(p)>1 and not p[-1]:p.pop()\n    return p\ndef add(a,b):return trim([(a[i] if i<len(a) else 0)^(b[i] if i<len(b) else 0) for i in range(max(len(a),len(b)))])\ndef pmul(a,b):\n    c=[0]*(len(a)+len(b)-1)\n    for i,x in enumerate(a):\n        for j,y in enumerate(b):c[i+j]^=mul(x,y)\n    return trim(c)\ndef divmodp(a,b):\n    a=trim(a);out=[0]*max(1,len(a)-len(b)+1)\n    assert b[-1]\n    bi=inv(b[-1])\n    for k in range(len(a)-len(b),-1,-1):\n        t=mul(a[k+len(b)-1],bi);out[k]=t\n        for j,v in enumerate(b):a[k+j]^=mul(t,v)\n    return trim(out),trim(a)\ndef emul(x,y,k=K):\n    a,b=x&4095,x>>12;c,d=y&4095,y>>12;bd=mul(b,d)\n    return (mul(a,c)^mul(k,bd))|((mul(a,d)^mul(b,c)^bd)<<12)\ndef einv(x,k=K):\n    a,b=x&4095,x>>12;t=inv(mul(a,a)^mul(a,b)^mul(k,mul(b,b)))\n    return mul(a^b,t)|(mul(b,t)<<12)\ndef eev(p,x):\n    v=0\n    for a in reversed(p):v=emul(v,x)^a\n    return v\n\n@lru_cache(None)\ndef constcols(c):return tuple(mul(c,1<<i) for i in range(12))\ndef cx(b,x,z,c=1):\n    if c==1:b.cx(x,z)\n    elif c:ar.linear_xor(b,x,z,constcols(c))\ndef scale(b,x,c):\n    if c!=1:ar.linear_inplace(b,x,constcols(c))\ndef scalepair(b,x,c):\n    scale(b,x[:12],c);scale(b,x[12:],c)\ndef combine(words):return km.array([q for w in words for q in w])\ndef wordlist(reg):return [km.array(list(reg)[i:i+12]) for i in range(0,len(reg),12)]\n\n@lru_cache(None)\ndef matrix_ops(rows):\n    """Elementary row operations reducing an invertible GF12 matrix to I."""\n    a=[list(r) for r in rows];n=len(a);ops=[]\n    assert all(len(r)==n for r in a)\n    for j in range(n):\n        p=next(i for i in range(j,n) if a[i][j])\n        if p!=j:a[p],a[j]=a[j],a[p];ops.append((\'swap\',p,j,0))\n        v=inv(a[j][j])\n        if v!=1:a[j]=[mul(x,v) for x in a[j]];ops.append((\'scale\',j,0,v))\n        for i in range(n):\n            if i!=j and a[i][j]:\n                v=a[i][j];a[i]=[x^mul(y,v) for x,y in zip(a[i],a[j])];ops.append((\'add\',j,i,v))\n    return tuple(ops)\n\ndef linear_words(b,words,rows,reverse=False):\n    ops=matrix_ops(tuple(tuple(r) for r in rows))\n    for op,i,j,v in (ops if reverse else reversed(ops)):\n        if op==\'swap\':b.swap(words[i],words[j])\n        elif op==\'scale\':scale(b,words[i],v if reverse else inv(v))\n        else:cx(b,words[i],words[j],v)\n\ndef zero_hold(b,x):\n    """Return a held all-zero ladder, len(x)-1 Toffolis."""\n    b.x(x);lad=b.alloc_qubits(len(x)-1)\n    b.init_and(x[0],x[1],lad[0])\n    for j in range(2,len(x)):b.init_and(lad[j-2],x[j],lad[j-1])\n    b.x(x)\n    return lad\ndef zero_free(b,x,lad):\n    b.x(x)\n    for j in reversed(range(2,len(x))):b.del_and(lad[j-2],x[j],lad[j-1])\n    b.del_and(x[0],x[1],lad[0]);b.free(lad);b.x(x)\n\ndef reclaim(b,regs):\n    """Reclaim specified known-free physical wires, without new wire IDs."""\n    wanted={str(q) for reg in regs for q in reg}\n    available=b.alloc_qubits(b.finish_circuit().num_qubits-b.num_allocated_qubits)\n    have={str(q) for q in available}\n    assert wanted<=have,(wanted-have)\n    b.free(km.array([q for q in available if str(q) not in wanted]))\n\ndef extmul_xor(b,x,y,z):\n    """126T product in explicit a+b*y basis; one scratch bit."""\n    a,c=x[:12],x[12:];d,e=y[:12],y[12:]\n    b.cx(z[:12],z[12:]);ar.mul_xor(b,a,d,z[:12]);b.cx(z[:12],z[12:])\n    scale(b,z[:12],inv(K))\n    ar.mul_xor(b,c,e,z[:12])\n    scale(b,z[:12],K)\n    b.cx(c,a);b.cx(e,d);ar.mul_xor(b,a,d,z[12:]);b.cx(e,d);b.cx(c,a)\n\ndef phase_extmul(b,x,y,mask):\n    for i in range(24):\n        for j in range(24):\n            v=emul(1<<i,1<<j)\n            for k in ar.bits(v):b.ccz(x[i],y[j],mask[k])\n\ndef write(s,qs,vals):\n    import numpy as np\n    vals=np.asarray(vals,dtype=np.uint64)\n    s.write_across_shots(km.array(qs),((vals[None,:]>>np.arange(len(qs),dtype=np.uint64)[:,None])&1).astype(bool))\ndef read(s,qs):\n    import numpy as np\n    return (s.read_across_shots(km.array(qs)).astype(np.uint64)*(np.uint64(1)<<np.arange(len(qs),dtype=np.uint64))[:,None]).sum(axis=0)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/decoder/extract.py': '"""In-place conjugate-node initial map and triangular EXTRACT."""\nfrom functools import lru_cache\nfrom common import *\nfrom exp.inverse12_escape.joint import inverse24_forward190 as inverse24_forward20,inverse24_reverse209 as inverse24_reverse20\n\n@lru_cache(None)\ndef constants(h=35):\n    cs=list(range(0,2*h,2));ks=[mul(c,c)^c^K for c in cs];P=[1]\n    for k in ks:P=pmul(P,[k,1,1])\n    matrix=[]\n    for c in cs:\n        g=c|(1<<12);Q=[0]*(2*h);Q[-1]=1\n        for j in reversed(range(2*h-1)):Q[j]=P[j+1]^emul(g,Q[j+1])\n        assert P[0]^emul(g,Q[0])==0\n        matrix.append(tuple(v&4095 for v in Q));matrix.append(tuple(v>>12 for v in Q))\n    return cs,ks,P,tuple(matrix)\n\ndef initial(b,words,reverse=False):\n    linear_words(b,words,constants(len(words)//2)[3],reverse=reverse)\n\ndef extract(b,words,reverse=False,transcript=None):\n    h=len(words)//2;cs,ks,_,_=constants(h);bank=[combine(words[2*j:2*j+2]) for j in range(h)]\n    if not reverse:\n        history=[]\n        for i in range(h):\n            cx(b,words[2*i+1],words[2*i],cs[i])\n            for j in range(i+1,h):\n                b.cx(words[2*i],words[2*j]);cx(b,words[2*i+1],words[2*j],cs[j]);b.cx(words[2*i+1],words[2*j+1])\n                history.append(inverse24_forward20(b,bank[j],K))\n                scalepair(b,bank[j],ks[i]^ks[j])\n        return history\n    history=list(transcript)\n    for i in reversed(range(h)):\n        for j in reversed(range(i+1,h)):\n            scalepair(b,bank[j],inv(ks[i]^ks[j]))\n            inverse24_reverse20(b,bank[j],history.pop(),K)\n            b.cx(words[2*i+1],words[2*j+1]);cx(b,words[2*i+1],words[2*j],cs[j]);b.cx(words[2*i],words[2*j])\n        cx(b,words[2*i+1],words[2*i],cs[i])\n    assert not history\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/decoder/safe_bridge.py': '"""Zero-safe division by temporarily replacing a zero factor by one."""\nfrom common import *\nimport division as dv\nfrom exp.mul99.schedules import divide_zero_bridge414\n\ndef divide_zero_bridge(b,x,y,extra_zero_mask=None):\n    return divide_zero_bridge414(b,x,y,K=K,extra_zero_mask=extra_zero_mask)\n\nif __name__==\'__main__\':\n    import numpy as np,json\n    rng=np.random.default_rng(18171026);n=16384\n    xv=rng.integers(0,1<<24,n,dtype=np.uint64);yv=rng.integers(0,1<<24,n,dtype=np.uint64);yv[:4096]=0\n    b=km.CircuitBuilder();x=b.alloc_qubits(24);y=b.alloc_qubits(24)\n    old,fm=dv.multiply_zero_forward(b,x,y,K);mm=b.alloc_bits(24);b.cz(x,mm);b.cz(x[2],x[13])\n    divide_zero_bridge(b,x,y,fm);b.free(fm);b.cz(x,old);b.free(old)\n    c=b.finish_circuit();s=km.Simulator(batch_size=n,seed=188182);ms=rng.integers(0,1<<24,n,dtype=np.uint64)\n    write(s,x,xv);write(s,y,yv);write(s,mm,ms);s.do(c)\n    assert np.array_equal(read(s,x),xv) and np.array_equal(read(s,y),yv)\n    z=[emul(int(a),int(bb)) if bb else int(a) for a,bb in zip(xv,yv)]\n    want=np.array([bool((a&int(m)).bit_count()%2)^bool(((a>>2)&1)&((a>>13)&1)) for a,m in zip(z,ms)])\n    assert np.array_equal(want,s.read_phase_flipped_across_shots())\n    for q in range(48,c.num_qubits):assert not s.read_across_shots(km.q(q)).any(),q\n    r=dict(shots=n,pair_T=c.operation_statistics().num_toffoli_like,division_T=c.operation_statistics().num_toffoli_like-152,scratch=c.num_qubits-48,value_errors=0,phase_errors=0)\n    print(json.dumps(r,indent=2));(ROOT/\'exp/decoder/safe_bridge.json\').write_text(json.dumps(r,indent=2)+\'\\n\')\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/decoder/synthesis.py': '"""Head-absorption synthesis with a complete deferred measurement transcript.\n\nThe rational interpolation chart is exact on valid heads without extension\npoles. Zero evaluation factors are replaced by one in the remainder map, so\narbitrary heads are preserved; that repair can change the decoded rational\nfunction after an extension pole. The top-coefficient promises are separate.\n"""\nfrom common import *\nfrom extract import constants\nimport division as dv\nfrom lowwidth import divide_lowwidth375\nsys.path.insert(0,str(ROOT/\'exp/convert\'))\nfrom kernels import rank_update,vandermonde\nimport kernels as ck\nfrom exp.noscratch.kernels import mul_xor_noscratch\nfrom convert import inverse_context\nfrom safe_bridge import divide_zero_bridge\n\ndef tail_recur(b,heads,p,q,d,reverse=False):\n    for a,z in (reversed(heads) if reverse else heads):\n        if reverse:b.swap(p,q);ar.mul_xor(b,z,q,p)\n        ar.mul_xor(b,p,q,d)\n        b.cx(z,a);ar.linear_inplace(b,q,ar.square_columns(12));ar.mul_xor(b,a,q,d)\n        ar.linear_inplace(b,q,[power(1<<j,2048) for j in range(12)]);b.cx(z,a)\n        if not reverse:ar.mul_xor(b,z,q,p);b.swap(p,q)\n\n@lru_cache(None)\ndef quotient_terms(n,kap):\n    """floor(l*A/q): A0 does not contribute; n rank-one products."""\n    V,W=vandermonde(n);out=[]\n    for t in range(n):\n        polynomial=[0]+[W[j][t] for j in range(n)]\n        quotient,_=divmodp(polynomial,[kap,1,1]);quotient+= [0]*(n-1-len(quotient))\n        out.append(([1,t],V[t][:n-1],quotient))\n    return out\n\ndef quotient_xor(b,head,A,B,kap,reverse=False):\n    assert len(B)==len(A)-1\n    terms=quotient_terms(len(A),kap)\n    for ac,bc,tc in (reversed(terms) if reverse else terms):\n        if any(tc):rank_update(b,head,ac,A[1:],bc,B,tc)\n\ndef full_product_xor(b,head,A,target):\n    n=len(A)+1;V,W=vandermonde(n)\n    for t in range(n):rank_update_noanc(b,head,[1,t],A,V[t][:len(A)],target,[W[j][t] for j in range(n)])\n\ndef rank_update_noanc(b,aw,ac,bw,bc,tw,tc):\n    x,ap=ck.expose(b,aw,ac);y,bp=ck.expose(b,bw,bc)\n    p=next(i for i,c in enumerate(tc) if c);v=tc[p];vi=inv(v)\n    for i,c in enumerate(tc):\n        if i!=p:cx(b,tw[p],tw[i],mul(c,vi))\n    scale(b,tw[p],vi);mul_xor_noscratch(b,x,y,tw[p]);scale(b,tw[p],v)\n    for i in reversed(range(len(tc))):\n        if i!=p:cx(b,tw[p],tw[i],mul(tc[i],vi))\n    ck.unexpose(b,bw,bc,bp);ck.unexpose(b,aw,ac,ap)\n\n@lru_cache(None)\ndef embedding_matrix(n,kap):\n    # Input [remainder0,remainder1,quotient0,...] -> q*quotient+remainder.\n    rows=[[0]*n for _ in range(n)];rows[0][0]=1;rows[1][1]=1\n    for j in range(n-2):\n        for k,c in enumerate([kap,1,1]):rows[j+k][j+2]=c\n    return tuple(tuple(r) for r in rows)\n\n@lru_cache(None)\ndef evaluation_coefficients(n,c):\n    g=c|(1<<12);out=[1]\n    for j in range(1,n):out.append(emul(out[-1],g))\n    return tuple(out)\n\ndef expose_eval(b,A,c,reverse=False):\n    es=evaluation_coefficients(len(A),c)\n    if reverse:\n        for i in reversed(range(2,len(A))):cx(b,A[i],A[1],es[i]>>12)\n        for i in reversed(range(2,len(A))):cx(b,A[i],A[0],es[i]&4095)\n        cx(b,A[1],A[0],c)\n    else:\n        cx(b,A[1],A[0],c)\n        for i in range(2,len(A)):cx(b,A[i],A[0],es[i]&4095)\n        for i in range(2,len(A)):cx(b,A[i],A[1],es[i]>>12)\n    return combine(A[:2])\n\ndef absorb(b,head,A,B,c,kap,reverse=False,transcript=None):\n    out=head+B\n    if not reverse:\n        value=expose_eval(b,A,c);flag=dv.zero_flag(b,value);expose_eval(b,A,c,True)\n        # Zero-safe factor: A(gamma)=0 gets factor1. Adding1 to A changes\n        # l*A only in its remainder, so the quotient map is unchanged.\n        b.cx(flag,A[0][0])\n        spare=[b.alloc_qubits(12),b.alloc_qubits(12)];target=spare+B\n        linear_words(b,target,embedding_matrix(len(target),kap))\n        full_product_xor(b,head,A,target)\n        b.cx(flag,A[0][0])\n        transcript=b.alloc_bits(24);b.hmr(combine(head),transcript)\n        for h,z in zip(head,spare):b.swap(h,z);b.free(z)\n        flagmask=b.alloc_bits(1);b.hmr(flag,flagmask[0]);b.free(flag)\n        return out,A,(transcript,flagmask)\n    linear_words(b,out,embedding_matrix(len(out),kap),True)\n    value=expose_eval(b,A,c)\n    cx(b,head[1],head[0],c)\n    headmask,flagmask=transcript\n    divide_zero_bridge(b,combine(head),value,flagmask[0]);b.free(flagmask)\n    cx(b,head[1],head[0],c)\n    b.cz(combine(head),headmask);b.free(headmask)\n    expose_eval(b,A,c,True)\n    quotient_xor(b,head,A,B,kap,True)\n    return out,A,transcript\n\ndef rotate(b,words,control,reverse=False):\n    # multiply by x using a top-zero cyclic register, controlled by even chart.\n    for i in (range(1,len(words)) if reverse else reversed(range(1,len(words)))):\n        b.cswap(control,words[i-1],words[i])\n\ndef flag_zero(b,q,flag):\n    ladder=zero_hold(b,q);b.cx(ladder[-1],flag);zero_free(b,q,ladder)\n\ndef tail_scale_forward(b,p,q,d):\n    with inverse_context(b,q) as t:\n        out=b.alloc_qubits(12);mul_xor_noscratch(b,d,t,out)\n        mask=b.alloc_bits(12);b.hmr(d,mask);b.swap(d,out);b.free(out)\n    return mask\n\ndef tail_scale_reverse(b,p,q,d,mask):\n    with inverse_context(b,q) as t:\n        out=b.alloc_qubits(12);mul_xor_noscratch(b,d,q,out)\n        newer=b.alloc_bits(12);b.hmr(d,newer);b.swap(d,out);b.free(out)\n        ar.phase_product(b,d,t,newer);b.free(newer)\n    b.cz(d,mask);b.free(mask)\n\ndef synthesize(b,words,keep_leading=True):\n    """Return A,B, transcript. Exactly70datawords after forward completion."""\n    h=len(words)//2;assert h%2==1\n    cs,ks,_,_=constants(h);heads=[words[2*i:2*i+2] for i in range(h)]\n    p,q,d=[b.alloc_qubits(12) for _ in range(3)];b.x(q[0])\n    tail_recur(b,heads,p,q,d)\n    dmask=tail_scale_forward(b,p,q,d)\n    # Uniform tail: when q=0,d=0, [d/q,p]/q is x times the old even\n    # chart [p]/0. Every transfer is polynomial-linear, so the required\n    # padding x is already present; no parity bit or outer shift is needed.\n    A=[d,p];B=[q];history=[]\n    for i in reversed(range(h)):\n        oldA,oldB=A,B\n        A,B,m=absorb(b,heads[i],A,B,cs[i],ks[i])\n        history.append((heads[i],oldA,oldB,cs[i],ks[i],m))\n    # Highest A coefficient=0 and highest B coefficient=1 are determinant\n    # identities. The second A coefficient can be nonzero on a bad tail.\n    extra=[A[h],A[h+1],B[h]];b.x(B[h][0])\n    freed=extra[1:] if keep_leading else extra\n    for reg in freed:b.free(reg)\n    transcript=dict(h=h,heads=heads,p=p,q=q,d=d,dmask=dmask,history=history,A=A,B=B,extra=extra,freed=freed,keep_leading=keep_leading)\n    return A[:h],B[:h],transcript\n\ndef unsynthesize(b,transcript):\n    t=transcript;h=t[\'h\'];A=t[\'A\'];B=t[\'B\']\n    reclaim(b,t[\'freed\'])\n    b.x(B[h][0])\n    for head,oldA,oldB,c,kap,mask in reversed(t[\'history\']):absorb(b,head,oldA,oldB,c,kap,True,mask)\n    p,q,d=t[\'p\'],t[\'q\'],t[\'d\']\n    tail_scale_reverse(b,p,q,d,t[\'dmask\'])\n    tail_recur(b,t[\'heads\'],p,q,d,True);b.x(q[0])\n    b.free(p);b.free(q);b.free(d)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/fold_cached_norm/kernel.py': '"""Spend spare global width on a retained norm inside quadratic fold division."""\nfrom pathlib import Path\nimport sys\nfrom types import FunctionType\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.mul99.schedules import multiply_forward99,phase_inverse36_pivot\nfrom exp.noscratch.schedules import norm_xor_noscratch\nfrom exp.noscratch.kernels import mul_xor_noscratch\nfrom exp.inverse12_escape.joint import inverse_initialized\nfrom exp.arithmetic.division import phase_product24\nfrom exp.arithmetic.field import erase_product\nfrom exp.zero_padding_synthesis import kernel as prior\nfrom exp.inverse12_escape.zero1 import inverse_zero\nfrom exp.fold_cached_norm.phase34 import phase_inverse34\nfrom exp.inverse12_escape.zero1 import inverse6_zero20\nfrom exp.phase24_tradeoff.phase6small import phase_inverse6_classical6\nfrom exp.alternating_gauge import scalar as scalar_prior\n\ndef divide325(b,x,y,K=None):\n    """x/=y, preserve y,325 executed T/36 private; y!=0 or x=0 promise.\n\n    Inner synthesis explicitly clips its factor before this call. The norm\n    stays live through both base-field output scales, allowing immediate\n    old-half phase repayment and avoiding its42T reconstruction.\n    """\n    assert K is not None\n    b.cx(y[12:],y[:12]);old=multiply_forward99(b,x,y,K);b.cx(y[12:],y[:12])\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,y,N,K);u=inverse_initialized(b,N)\n    for off in (0,12):\n        half=x[off:off+12];out=b.alloc_qubits(12)\n        mul_xor_noscratch(b,half,u,out);erase_product(b,out,N,half)\n        b.swap(half,out);b.free(out)\n    mask=b.alloc_bits(12);b.hmr(u,mask);b.free(u)\n    phase_inverse36_pivot(b,N,mask);b.free(mask)\n    prior.ar._erase_norm(b,y,N,K);b.free(N)\n    phase_product24(b,x,y,old,K);b.free(old)\n\ndef bind(fn,**changes):\n    return FunctionType(fn.__code__,dict(fn.__globals__,**changes),fn.__name__,fn.__defaults__,fn.__closure__)\n\ndivide323=bind(divide325,phase_inverse36_pivot=phase_inverse34)\ndivide323.__doc__=\'Unit factor division:323T/36private; retained norm and34T/21private inverse phase.\'\n\ndef divide_zero324(b,x,y,K=None,extra_zero_mask=None,after=None):\n    """Identity-at-zero division,324T/49 private, all HMR phases closed."""\n    assert K is not None\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,y,N,K);u,F=inverse_zero(b,N)\n    b.cx(y[12:],y[:12]);b.cx(F,y[0]);old=multiply_forward99(b,x,y,K)\n    b.cx(F,y[0]);b.cx(y[12:],y[:12]);b.cx(F,u[0])\n    for off in (0,12):\n        half=x[off:off+12];out=b.alloc_qubits(12);mul_xor_noscratch(b,half,u,out)\n        b.cx(F,N[0]);erase_product(b,out,N,half);b.cx(F,N[0])\n        b.swap(half,out);b.free(out)\n    b.cx(F,u[0]);um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    b.cx(F,y[0]);phase_product24(b,x,y,old,K);b.cx(F,y[0]);b.free(old)\n    if after is not None:after(F)\n    fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n    if extra_zero_mask is not None:b.cx(extra_zero_mask,fm[0])\n    phase_inverse34(b,N,um,fm[0]);b.free(um);b.free(fm)\n    prior.ar._erase_norm(b,y,N,K);b.free(N)\n\nabsorb=bind(prior.absorb_odd,divide_lowwidth369=divide323)\nabsorb_even=bind(prior.absorb_even,divide_lowwidth369=divide323)\nouter=bind(prior.outer,divide_zero369=divide_zero324)\n\ndef _inverse6_zero(b,x,with_zero):\n    assert with_zero\n    return inverse6_zero20(b,x)\n\ndivide_zero113=bind(scalar_prior.divide_zero117,inverse6_init19=_inverse6_zero,phase_inverse_generic6=phase_inverse6_classical6)\nmultiply_zero141=bind(scalar_prior.multiply_zero144,inverse_zero=inverse_zero,phase_inverse36_zero=phase_inverse34)\nprefix_reverse137=bind(scalar_prior.prefix_reverse141,divide_zero117=divide_zero113)\nodd_decode=bind(prior.synthesize.__globals__[\'odd_decode\'],prefix_reverse141=prefix_reverse137)\nsynthesize=bind(prior.synthesize,absorb=absorb,absorb_even=absorb_even,outer=outer,odd_decode=odd_decode,divide_zero117=divide_zero113)\nunsynthesize=bind(prior.unsynthesize,absorb=absorb,absorb_even=absorb_even,outer=outer,odd_decode=odd_decode,multiply_zero144=multiply_zero141)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/fold_cached_norm/phase34.py': '"""Use the held I6 DAG with a classical mask:34T,21 private, optional zero."""\nfrom exp.jet24.kernel import phase_inverse12_quantum64\n\ndef phase_inverse34(b,x,mask,zero_mask=None):\n    return phase_inverse12_quantum64(b,x,mask,zero_mask)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/fold_joint/divide.py': '"""Zero-aware streamed factor division24, no edits to arithmetic modules."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.mul99.schedules import *\nfrom exp.safe_scalar.phase import phase_inverse36_zero\nfrom exp.noscratch.kernels import inverse3_noscratch\nfrom exp.arithmetic.lowwidth import classical_transpose\n\ndef inverse_stream_zero(b,x):\n    """Overwrite12-bit x with inv0(x); expose zero via bottom norm,68T.\n\n    Same recursive streamed schedule as old66T inverse, with2AND flag only\n    after deepest inverse\'s old-input wires are erased. Peak12+1 scratch.\n    """\n    n=len(x)\n    if n==3:\n        y=inverse3_noscratch(b,x);mask=b.alloc_bits(3);b.hmr(x,mask);b.swap(x,y);b.free(y)\n        b.x(x);u=b.alloc_qubits(1)[0];F=b.alloc_qubits(1)[0]\n        b.init_and(x[0],x[1],u);b.init_and(u,x[2],F);b.del_and(x[0],x[1],u);b.free(u);b.x(x)\n        return mask,F\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to)\n    N=b.alloc_qubits(k);norm_xor_noscratch(b,x,N,K)\n    nm,F=inverse_stream_zero(b,N);ar._phase_norm(b,x,nm,K);b.free(nm)\n    outb=b.alloc_qubits(k);mul_xor_noscratch(b,x[k:],N,outb)\n    raw=b.alloc_bits(n);b.hmr(x[k:],raw[k:])\n    mul_xor_noscratch(b,x[:k],N,x[k:]);b.cx(outb,x[k:]);b.hmr(x[:k],raw[:k])\n    b.swap(x[:k],x[k:]);b.swap(x[k:],outb);b.free(outb)\n    ar._erase_norm(b,x,N,K);b.free(N);linear_inplace(b,x,frm)\n    mask=classical_transpose(b,raw,to);b.free(raw);return mask,F\n\ndef divide_zero371(b,x,y,K=None,extra_zero_mask=None,after=None):\n    """x/=y for y!=0, identity when y0. <=371T,37 private, all-input safe.\n\n    Norm\'s zero flag costs2T in streamed inverse, not a23-bit factor test.\n    Its phase is fused into the existing36T inverse-norm phase correction.\n    WIDTH OBSTRUCTION: u12 remains live across a24-bit product target, +F.\n    """\n    if K is None:K=extension_columns()[2]\n    u=b.alloc_qubits(12);norm_xor_noscratch(b,y,u,K)\n    nm,F=inverse_stream_zero(b,u);ar._phase_norm(b,y,nm,K);b.free(nm)\n    b.cx(y[12:],y[:12]);b.cx(F,y[0]);oldmask=multiply_forward99(b,x,y,K);b.cx(F,y[0]);b.cx(y[12:],y[:12])\n    b.cx(F,u[0]);masks=[]\n    for off in(0,12):\n        half=x[off:off+12];out=b.alloc_qubits(12);mul_xor_noscratch(b,half,u,out)\n        mask=b.alloc_bits(12);b.hmr(half,mask);masks.append(mask);b.swap(half,out);b.free(out)\n    b.cx(F,u[0]);um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,y,N,K)\n    b.cx(F,N[0])\n    for off,mask in zip((0,12),masks):phase_product(b,x[off:off+12],N,mask);b.free(mask)\n    b.cx(F,N[0]);b.cx(F,y[0]);phase_product24(b,x,y,oldmask,K);b.cx(F,y[0]);b.free(oldmask)\n    if after is not None:after(F)\n    fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n    if extra_zero_mask is not None:b.cx(extra_zero_mask,fm[0])\n    phase_inverse36_zero(b,N,um,fm[0]);b.free(um);b.free(fm)\n    ar._erase_norm(b,y,N,K);b.free(N)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/fold_joint/kernel.py': '"""CRT joint remainder/Taylor fold. Live BEST modules are read only.\n\nThe factor flag remains live across quotient restoration, so its existing\ncompletion can be shared without another zero test or controlled word bank.\nThe polynomial and zero-factor completion match BEST for arbitrary heads.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.partial_gauge import synthesis_noflag as old\nfrom exp.mul99.kernels import mul24_xor99\nfrom exp.mul99.schedules import divide_lowwidth369\nfrom exp.fold_joint.divide import divide_zero371\nfrom exp.decoder.common import *\nbase=old.base;pg=old.pg;ck=old.ck;dv=old.dv\n\ndef epow(c,k):\n    ans=1;g=c|4096\n    for _ in range(k):ans=emul(ans,g)\n    return ans\ndef escale(b,R,c):ar.linear_inplace(b,combine(R),[emul(1<<i,c)for i in range(24)])\n\ndef q_times_low(b,B,kap,reverse=False):\n    n=len(B)\n    if not n:return\n    rows=[[kap if i==j else 1 if i-j in(1,2)else 0 for j in range(n)]for i in range(n)]\n    linear_words(b,B,rows,reverse)\n\ndef low_product(b,head,A,B):\n    """l*A mod x^k, k low Acoeffs; rank0/1/k+1 at k0/1/>=2."""\n    k=len(B)\n    if not k:return\n    V,W=base.vandermonde(k+1)\n    for t in range(k+1):\n        old.rank_affine(b,head,[1,t],A[:k],V[t][:k],B,[W[j][t]for j in range(k)])\n\ndef R_to_tops(b,R,B,c,reverse=False):\n    """CRT(E-remainder, lowTaylor)->(topcoeffs, unchangedlowTaylor)."""\n    k=len(B);powers=base.evaluation_coefficients(k,c)\n    if reverse:\n        cx(b,R[1],R[0],c);escale(b,R,epow(c,k))\n        for word,v in zip(B,powers):cx(b,word,R[0],v&4095);cx(b,word,R[1],v>>12)\n    else:\n        for word,v in zip(B,powers):cx(b,word,R[0],v&4095);cx(b,word,R[1],v>>12)\n        escale(b,R,einv(epow(c,k)));cx(b,R[1],R[0],c)\n\ndef divide414_callback(b,x,y,K,extra_zero_mask,after):\n    F=dv.zero_flag(b,y);b.cx(F,y[0]);divide_lowwidth369(b,x,y,K);b.cx(F,y[0])\n    after(F)\n    mask=b.alloc_bits(1);b.hmr(F,mask[0]);b.free(F);b.cx(extra_zero_mask,mask[0])\n    dv.phase_zero(b,y,mask[0]);b.free(mask)\n\ndef absorb(b,head,A,B,qi,p,virtual,pconst,c,kap,reverse=False,transcript=None,wide=False):\n    k=len(B);assert len(A)==k+1;out=B+head\n    if not reverse:\n        value=old.eval_implicit(b,A,qi,c);flag=dv.zero_flag(b,value)\n        spare=[b.alloc_qubits(12),b.alloc_qubits(12)];b.cx(flag,value[0]);cx(b,head[1],head[0],c)\n        mul24_xor99(b,combine(head),value,combine(spare),K)\n        cx(b,head[1],head[0],c);b.cx(flag,value[0]);old.eval_implicit(b,A,qi,c,True)\n        q_times_low(b,B,kap);b.cx(flag,A[0][0]);low_product(b,head,A,B);b.cx(flag,A[0][0])\n        v=epow(c,k+2);old.add_alias(b,p,spare,[v&4095,v>>12],virtual,pconst)\n        R_to_tops(b,spare,B,c)\n        hm=b.alloc_bits(24);b.hmr(combine(head),hm)\n        for h,z in zip(head,spare):b.swap(h,z);b.free(z)\n        fm=b.alloc_bits(1);b.hmr(flag,fm[0]);b.free(flag)\n        return out,A,(hm,fm)\n    R_to_tops(b,head,B,c,True)\n    v=epow(c,k+2);old.add_alias(b,p,head,[v&4095,v>>12],virtual,pconst)\n    value=old.eval_implicit(b,A,qi,c);hm,fm=transcript\n    def after(F):\n        cx(b,head[1],head[0],c);old.eval_implicit(b,A,qi,c,True)\n        b.cx(F,A[0][0]);low_product(b,head,A,B);b.cx(F,A[0][0]);q_times_low(b,B,kap,True)\n        old.eval_implicit(b,A,qi,c);cx(b,head[1],head[0],c)\n    if wide:divide_zero371(b,combine(head),value,K=K,extra_zero_mask=fm[0],after=after)\n    else:divide414_callback(b,combine(head),value,K,fm[0],after)\n    b.free(fm)\n    cx(b,head[1],head[0],c);b.cz(combine(head),hm);b.free(hm)\n    old.eval_implicit(b,A,qi,c,True)\n    return out,A,transcript\n\ndef synthesize(b,words,keep_leading=True,wide=False):\n    h=len(words)//2;assert h>=3 and h%2==1\n    cs,ks,_,_=old.constants(h);heads=[words[2*i:2*i+2]for i in range(h)]\n    d=b.alloc_qubits(12);F=pg.zero_compact(b,heads[0][1]);masks=[]\n    for i in range(h):masks.append(old.prefix_encode(b,heads,i,F,d))\n    old.divide_safe138(b,d,heads[-1][1]);A=[d];B=[];hist=[]\n    for i in reversed(range(h)):\n        old.prefix_decode(b,heads,i,F,masks[i]);qi=heads[i-1][1]if i else None;p,v,pc=old.p_alias(heads,i,F)\n        oldA,oldB=A,B;A,B,t=absorb(b,heads[i],A,B,qi,p,v,pc,cs[i],ks[i],wide=wide)\n        hist.append((i,oldA,oldB,qi,p,v,pc,cs[i],ks[i],t))\n    extra=[A[h]];freed=[]if keep_leading else extra\n    for word in freed:b.free(word)\n    fmask=b.alloc_bits(1);b.hmr(F,fmask[0]);b.free(F)\n    t=dict(h=h,heads=heads,d=d,F=F,history=hist,A=A,B=B,extra=extra,freed=freed,keep_leading=keep_leading,fmask=fmask,wide=wide)\n    return A[:h],B,t\n\ndef unsynthesize(b,t):\n    if t[\'freed\']:reclaim(b,t[\'freed\'])\n    heads=t[\'heads\'];F=t[\'F\'];masks=[None]*t[\'h\']\n    for i,A,B,qi,p,v,pc,c,kap,token in reversed(t[\'history\']):\n        absorb(b,heads[i],A,B,qi,p,v,pc,c,kap,True,token,t[\'wide\']);masks[i]=old.prefix_encode(b,heads,i,F)\n        if i==0:\n            reclaim(b,[[F]]);lad=base.zero_hold(b,heads[0][1]);b.cx(lad[-1],F);base.zero_free(b,heads[0][1],lad)\n            b.cz(F,t[\'fmask\'][0]);b.free(t[\'fmask\'])\n    old.multiply_safe165(b,t[\'d\'],heads[-1][1])\n    for i in reversed(range(t[\'h\'])):old.prefix_decode(b,heads,i,F,masks[i],t[\'d\'])\n    pg.zero_compact_free(b,heads[0][1],F);b.free(t[\'d\'])\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse12_escape/joint.py': '"""Correlated inverse-norm cache lowers the paired I12 reverse to67T/Q33.\nThe initialized inverse is unchanged64T/Q31; no <=63 isolated claim.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse19_joint.kernel import *\nfrom exp.inverse12_escape.zero1 import erase_raw_cache\n\ndef inverse6_keep_inner19(b,x):\n    """Preserve x; return inv0(x) and norm(inv0(x))3,19T/Q19."""\n    to,frm,K=tower(6);linear_inplace(b,x,to);g=b.alloc_qubits(6)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,x,l,r)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(x)+list(g);forms=ref.norm_forms(K);w=ref.inverse3_virtual(b,prefix,forms)\n    linear_inplace(b,g,LAY[\'raw_columns\']);R,E=g[:4],g[4:];z=list(E)+list(b.alloc_qubits(4));source=list(x)+list(R)+list(w)\n    for j,e in enumerate(LAY[\'outputs\']):ref.form_xor(b,source,e&65535,z[j])\n    for i,(l,r)in enumerate(LAY[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(LAY[\'outputs\']))\n        with ref.expose_pair(b,source,l,r)as(p,q):rankone_xor(b,p,q,z,v)\n    erase_raw_cache(b,x,R,LAY[\'raw_rows\'][:4]);linear_inplace(b,z,LAY[\'output_inverse_columns\'])\n    linear_inplace(b,x,frm);linear_inplace(b,z,frm);return z,w\n\ndef inverse12_reverse67(b,x,mask):\n    assert len(x)==12;to,frm,K=tower(12);linear_inplace(b,x,to)\n    N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,K);t,v=inverse6_keep_inner19(b,N);out=b.alloc_qubits(6)\n    mul_xor_noscratch(b,x[6:],t,out);erase_product(b,out,N,x[6:]);mul_xor_noscratch(b,x[:6],t,x[6:]);b.cx(out,x[6:])\n    b.cx(out,x[6:]);erase_product(b,x[6:],N,x[:6]);b.cx(out,x[6:]);b.swap(x[:6],x[6:]);b.swap(x[6:],out);b.free(out)\n    D=inverse3_noscratch(b,v);ar._erase_inverse_with_norm(b,t,N,D);b.free(N);erase_inverse(b,v,D);b.free(D)\n    to6,frm6,K6=tower(6);linear_inplace(b,t,to6);ar._erase_norm(b,t,v,K6);b.free(v);linear_inplace(b,t,frm6)\n    ar._erase_norm(b,x,t,K);b.free(t);linear_inplace(b,x,frm);b.cz(x,mask);b.free(mask)\n\ninverse12_forward64=inverse_inplace_forward20\n\ndef phase_inverse6_quantum19(b,u,L):\n    """Phase L dot inv0(u), bothquantum6;19T,9private, exactclosedHMR."""\n    to,frm,K=tower(6);linear_inplace(b,u,to);transpose=[sum(((frm[i]>>j)&1)<<i for i in range(6))for j in range(6)];linear_inplace(b,L,transpose)\n    g=b.alloc_qubits(6)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,u,l,r)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(u)+list(g);forms=ref.norm_forms(K);w=ref.inverse3_virtual(b,prefix,forms);source=prefix+list(w)\n    for j,e in enumerate(basic.DAG19[\'outputs\']):\n        if e&1:b.z(L[j])\n        for r in bits((e&65535)>>1):b.cz(L[j],source[r])\n    for i,(l,r)in enumerate(basic.DAG19[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(basic.DAG19[\'outputs\']));ix=list(bits(v));piv=ix[0]\n        for j in ix[1:]:b.cx(L[j],L[piv])\n        with ref.expose_pair(b,source,l,r)as(p,q):b.ccz(p,q,L[piv])\n        for j in reversed(ix[1:]):b.cx(L[j],L[piv])\n    m=b.alloc_bits(3);b.hmr(w,m);b.free(w);ref.phase_inverse3_virtual(b,prefix,forms,m);b.free(m)\n    for i,(l,r)in reversed(list(enumerate(basic.raw_forms19()))):\n        with ref.expose_pair(b,u,l,r)as(p,q):b.del_and(p,q,g[i])\n    b.free(g);linear_inplace(b,L,inverse_columns(transpose));linear_inplace(b,u,frm)\n\ndef inverse12_keep_inverse_norm64(b,x):\n    """Preserve x12, return t=inv(x),u=norm(t)6,64T;Q31."""\n    to,frm,K=tower(12);linear_inplace(b,x,to);D=b.alloc_qubits(6);norm_xor_noscratch(b,x,D,K);u=inverse6_init19(b,D);ar._erase_norm(b,x,D,K);b.free(D)\n    t=b.alloc_qubits(12);b.cx(x[6:],x[:6]);mul_xor_noscratch(b,x[:6],u,t[:6]);mul_xor_noscratch(b,x[6:],u,t[6:]);b.cx(x[6:],x[:6]);linear_inplace(b,x,frm);linear_inplace(b,t,frm);return t,u\n\ndef erase_norm_via_inverse_cache19(b,t,N,u):\n    """Erase N=inv0(t),u=norm(t)6;19T vsoldDcachecleanup24T.\n\n    Joint phase mask for N is linear in t and classical HMR outcomes; evaluate\n    inverse6 with that quantum mask, then erase u quadratically as norm(t).\n    """\n    to,frm,K=tower(12);linear_inplace(b,t,to);linear_inplace(b,N,to);m=b.alloc_bits(12);b.hmr(N,m);b.free(N);L=b.alloc_qubits(6);f=field(6)\n    def masks():\n        for j in range(6):\n            for i in range(6):\n                for r in bits(f.mul(1<<i,1<<j)):\n                    b.ccx(m[r],t[i],L[j]);b.ccx(m[r],t[i+6],L[j]);b.ccx(m[r+6],t[i+6],L[j])\n    masks();phase_inverse6_quantum19(b,u,L);masks();b.free(L);b.free(m);ar._erase_norm(b,t,u,K);b.free(u);linear_inplace(b,t,frm)\n\ndef inverse24_reverse209(b,x,mask,K=None):\n    cols,back,_=extension_columns(K);linear_inplace(b,x,cols);to,frm,K24=tower(24);linear_inplace(b,x,to)\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,x,N,K24);t,u=inverse12_keep_inverse_norm64(b,N);out=b.alloc_qubits(12)\n    mul_xor_noscratch(b,x[12:],t,out);erase_product(b,out,N,x[12:]);mul_xor_noscratch(b,x[:12],t,x[12:]);b.cx(out,x[12:]);b.cx(out,x[12:]);erase_product(b,x[12:],N,x[:12]);b.cx(out,x[12:]);b.swap(x[:12],x[12:]);b.swap(x[12:],out);b.free(out)\n    erase_norm_via_inverse_cache19(b,t,N,u);ar._erase_norm(b,x,t,K24);b.free(t);linear_inplace(b,x,frm);b.cz(x,mask);b.free(mask);linear_inplace(b,x,back)\n\ninverse24_forward190=basic.inverse24_forward19\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse12_escape/zero1.py': '"""65T/Q31 zero-aware I12 from a one-AND correlated zero output.\nNot an improvement to isolated64T I12 inversion; no live module is changed.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse19_joint.kernel import *\nfrom types import FunctionType\nZLAY=json.loads(Path(__file__).with_name(\'zero_layout.json\').read_text())\n\ndef erase_raw_cache(b,x,cache,rows):\n    m=b.alloc_bits(len(cache));b.hmr(cache,m);b.free(cache)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,x,l,r)as(p,q):\n            for j,row in enumerate(rows):\n                if row>>i&1:b.ccz(p,q,m[j])\n    b.free(m)\n\ndef inverse6_zero20(b,x):\n    to,frm,K=tower(6);linear_inplace(b,x,to);g=b.alloc_qubits(6)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,x,l,r)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(x)+list(g);forms=ref.norm_forms(K);w=ref.inverse3_virtual(b,prefix,forms)\n    linear_inplace(b,g,LAY[\'raw_columns\']);R,E=g[:4],g[4:];z=list(E)+list(b.alloc_qubits(4));source=list(x)+list(R)+list(w)\n    for j,e in enumerate(LAY[\'outputs\']):ref.form_xor(b,source,e&65535,z[j])\n    for i,(l,r)in enumerate(LAY[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(LAY[\'outputs\']))\n        with ref.expose_pair(b,source,l,r)as(p,q):rankone_xor(b,p,q,z,v)\n    # One norm coordinate lies in the4D live raw-control span. Keep it, erase3.\n    linear_inplace(b,R,ZLAY[\'cache_columns\']);erase_raw_cache(b,x,R[1:],ZLAY[\'raw_cache_rows\'][1:])\n    source=list(x)+[R[0]]+list(w);left=1|(ZLAY[\'left_w\']<<8);right=1^ZLAY[\'input_form\']^(1<<7)^(ZLAY[\'right_w\']<<8)\n    F=b.alloc_qubits(1)[0]\n    with ref.expose_pair(b,source,left,right)as(p,q):b.init_and(p,q,F)\n    erase_raw_cache(b,x,R[:1],ZLAY[\'raw_cache_rows\'][:1]);linear_inplace(b,z,LAY[\'output_inverse_columns\']);ar._erase_norm(b,z,w,K);b.free(w)\n    linear_inplace(b,x,frm);linear_inplace(b,z,frm);return z,F\n\ndef inverse_zero(b,x):\n    to,frm,K=tower(12);linear_inplace(b,x,to);N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,K)\n    t,F=inverse6_zero20(b,N);ar._erase_norm(b,x,N,K);b.free(N)\n    out=b.alloc_qubits(12);b.cx(x[6:],x[:6]);mul_xor_noscratch(b,x[:6],t,out[:6]);mul_xor_noscratch(b,x[6:],t,out[6:]);b.cx(x[6:],x[:6])\n    ar._erase_norm(b,out,t,K);b.free(t);linear_inplace(b,x,frm);linear_inplace(b,out,frm);return out,F\n\nfrom exp.inverse19_joint import kernel as parentkernel\n_old=parentkernel.inverse_apply_safe\ninverse_apply_safe=FunctionType(_old.__code__,globals(),\'inverse_apply_safe\',_old.__defaults__,_old.__closure__)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse19_joint/kernel.py': '"""Width-audited19AND inverse substitutions; stable imports remain unchanged."""\nfrom pathlib import Path\nimport sys,json\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse_width_subspaces import kernel19 as basic\nfrom exp.inverse_width_subspaces.kernel19 import *\nfrom exp.safe_scalar import kernel as safe\nfrom exp.safe_scalar.phase import phase_inverse36_zero\nfrom exp.noscratch.schedules import norm_xor_noscratch\nLAY=json.loads(Path(__file__).with_name(\'layout.json\').read_text())\n\ndef inverse6_init19(b,x,with_zero=False):\n    """Fresh inverse6:19T/Q19, or inverse+zero21T/Q19; preserve x.\n\n    Reuse two redundant raw norm directions as output. Four other norm\n    directions remain until all final products are added. Compute zero only\n    AFTER those four wires are erased, using the still-live inverse norm3.\n    """\n    to,frm,K=tower(6);linear_inplace(b,x,to);g=b.alloc_qubits(6)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,x,l,r)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(x)+list(g);forms=ref.norm_forms(K);w=ref.inverse3_virtual(b,prefix,forms)\n    linear_inplace(b,g,LAY[\'raw_columns\']);R,E=g[:4],g[4:]\n    z=list(E)+list(b.alloc_qubits(4));source=list(x)+list(R)+list(w)\n    for j,e in enumerate(LAY[\'outputs\']):ref.form_xor(b,source,e&65535,z[j])\n    for i,(l,r)in enumerate(LAY[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(LAY[\'outputs\']));assert v\n        with ref.expose_pair(b,source,l,r)as(p,q):rankone_xor(b,p,q,z,v)\n    # Four retained raw products have only quadratic phase corrections.\n    rm=b.alloc_bits(4);b.hmr(R,rm);b.free(R)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,x,l,r)as(p,q):\n            for j,row in enumerate(LAY[\'raw_rows\'][:4]):\n                if row>>i&1:b.ccz(p,q,rm[j])\n    b.free(rm);linear_inplace(b,z,LAY[\'output_inverse_columns\'])\n    F=None\n    if with_zero:\n        b.x(w);u=b.alloc_qubits(1)[0];F=b.alloc_qubits(1)[0]\n        b.init_and(w[0],w[1],u);b.init_and(u,w[2],F);b.del_and(w[0],w[1],u);b.free(u);b.x(w)\n    ar._erase_norm(b,z,w,K);b.free(w);linear_inplace(b,x,frm);linear_inplace(b,z,frm)\n    return(z,F)if with_zero else z\n\ndef inverse6_xor19(b,x,z=None,keep_norm=False):\n    if z is None and not keep_norm:return inverse6_init19(b,x)\n    return basic.inverse6_xor19(b,x,z,keep_norm)\n\n# Local recursive schedules, no mutation of imported globals.\nfrom types import FunctionType\ninverse6_xor20=inverse6_xor19\nfor _name in(\'inverse_initialized\',\'inverse12_keep_norm\',\'inverse_inplace_forward20\',\'inverse_inplace_reverse20\'):\n    _old=getattr(ref,_name);globals()[_name]=FunctionType(_old.__code__,globals(),_name,_old.__defaults__,_old.__closure__)\n\ndef inverse_zero(b,x):\n    """GF12 inverse0 +zero:66T/Q31, preserve input; closure unchanged36T."""\n    to,frm,K=tower(12);linear_inplace(b,x,to);N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,K)\n    t,F=inverse6_init19(b,N,True);ar._erase_norm(b,x,N,K);b.free(N)\n    out=b.alloc_qubits(12);b.cx(x[6:],x[:6]);mul_xor_noscratch(b,x[:6],t,out[:6]);mul_xor_noscratch(b,x[6:],t,out[6:]);b.cx(x[6:],x[:6])\n    ar._erase_norm(b,out,t,K);b.free(t);linear_inplace(b,x,frm);linear_inplace(b,out,frm);return out,F\n\nconsume=safe.consume;erase=safe.erase\n\ndef inverse_apply_safe(b,a,action,incoming=None,defer=False):\n    ai,F=inverse_zero(b,a);consume(b,ai,F,incoming)\n    b.cx(F,a[0]);b.cx(F,ai[0]);action(ai);b.cx(F,ai[0]);b.cx(F,a[0])\n    return erase(b,a,ai,F,defer)\n\nfrom exp.arithmetic.lowwidth import phase_inverse_generic6,phase_product_quantum\n\ndef inverse12_xor73(b,x,z,base6=inverse6_init19):\n    """Arbitrary-target XOR inverse12:73T/Q43, preserve x and target phase.\n\n    Initialized inverse schedules are NOT arbitrary-target oracles: they erase\n    t using norm(output), which is wrong for a preexisting target. Keep N and\n    close t by a9T phase inverse instead.\n    """\n    to,frm,K=tower(12);linear_inplace(b,x,to);linear_inplace(b,z,to)\n    N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,K);t=base6(b,N)\n    b.cx(x[6:],x[:6]);mul_xor_noscratch(b,x[:6],t,z[:6]);mul_xor_noscratch(b,x[6:],t,z[6:]);b.cx(x[6:],x[:6])\n    m=b.alloc_bits(6);b.hmr(t,m);b.free(t);phase_inverse_generic6(b,N,m);b.free(m)\n    ar._erase_norm(b,x,N,K);b.free(N);linear_inplace(b,x,frm);linear_inplace(b,z,frm)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_search/baseline.py': '"""21T correlated inverse6 XOR seed, arbitrary target, clean private state."""\nimport sys\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.noscratch.kernels import *\n\ndef inverse6_xor21(b,x,z):\n    assert len(x)==len(z)==6\n    to,frm,K=tower(6);linear_inplace(b,x,to);linear_inplace(b,z,to)\n    N=b.alloc_qubits(3);f=field(3)\n    linear_xor(b,x[:3],N,square_columns(3))\n    linear_xor(b,x[3:],N,[f.mul(K,v)for v in square_columns(3)])\n    mul_xor_noscratch(b,x[:3],x[3:],N)\n    w=inverse3_noscratch(b,N)\n    b.cx(x[3:],x[:3]);mul_xor_noscratch(b,x[:3],w,z[:3]);b.cx(x[3:],x[:3])\n    mul_xor_noscratch(b,x[3:],w,z[3:])\n    erase_inverse(b,N,w);b.free(w);ar._erase_norm(b,x,N,K);b.free(N)\n    linear_inplace(b,x,frm);linear_inplace(b,z,frm)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_search/inverse20.py': '"""Explicit 20-AND GF64 inverse XOR circuit with exact measurement cleanup.\n\nAll masks below describe physical linear forms. Prefix source order is x0..5,\nsix raw norm rank-products, then the three inverse-norm bits. Constant is bit0\nof a form mask; physical source i is bit(i+1).\n"""\ntry:\n    from .baseline import *\nexcept ImportError:\n    from baseline import *\nfrom contextlib import contextmanager\nimport json\n\nDAG=json.loads(Path(__file__).with_name(\'optimized.json\').read_text())\n\n@contextmanager\ndef expose_pair(b,registers,left,right):\n    u,v=left>>1,right>>1;p=next(bits(u));ops=[]\n    for j in bits(u^(1<<p)):b.cx(registers[j],registers[p]);ops.append((j,p))\n    if v>>p&1:v^=u^(1<<p)\n    q=next(bits(v&~(1<<p)))\n    for j in bits(v^(1<<q)):b.cx(registers[j],registers[q]);ops.append((j,q))\n    if left&1:b.x(registers[p])\n    if right&1:b.x(registers[q])\n    yield registers[p],registers[q]\n    if right&1:b.x(registers[q])\n    if left&1:b.x(registers[p])\n    for j,k in reversed(ops):b.cx(registers[j],registers[k])\n\ndef form_xor(b,registers,mask,target):\n    if mask&1:b.x(target)\n    for i in bits(mask>>1):b.cx(registers[i],target)\n\ndef norm_forms(K):\n    forms=[0]*3;f=field(3)\n    for i,v in enumerate(square_columns(3)):\n        for j in bits(v):forms[j]^=1<<(1+i)\n        for j in bits(f.mul(K,v)):forms[j]^=1<<(1+3+i)\n    for i,(_,v)in enumerate(tensor(3)):\n        for j in bits(v):forms[j]^=1<<(1+6+i)\n    return forms\n\ndef inverse3_virtual(b,registers,forms):\n    out=b.alloc_qubits(3)\n    for mask,vector in enumerate(inverse3_anf()):\n        if not vector:continue\n        ix=list(bits(mask))\n        if len(ix)==1:\n            for j in bits(vector):form_xor(b,registers,forms[ix[0]],out[j])\n        else:\n            assert len(ix)==2\n            with expose_pair(b,registers,forms[ix[0]],forms[ix[1]])as(p,q):rankone_xor(b,p,q,out,vector)\n    return out\n\ndef phase_inverse3_virtual(b,registers,forms,mask):\n    for monomial,vector in enumerate(inverse3_anf()):\n        if not vector:continue\n        ix=list(bits(monomial))\n        if len(ix)==1:\n            assert not(forms[ix[0]]&1)\n            for i in bits(forms[ix[0]]>>1):\n                for j in bits(vector):b.cz(registers[i],mask[j])\n        else:\n            with expose_pair(b,registers,forms[ix[0]],forms[ix[1]])as(p,q):\n                for j in bits(vector):b.ccz(p,q,mask[j])\n\ndef inverse6_xor20(b,x,z=None,keep_norm=False):\n    """z ^= inverse(x), inv0=0; 20T,21 total Q for input6+output6.\n\n    Arbitrary target is supported. If omitted, allocate clean six-bit target.\n    Exactly six norm ANDs, three inverse3 ANDs, eleven final ANDs; cleanup0T.\n    """\n    assert len(x)==6\n    to,frm,K=tower(6);linear_inplace(b,x,to)\n    if z is not None:linear_inplace(b,z,to)\n    g=b.alloc_qubits(6)\n    for i,(mask,_)in enumerate(tensor(3)):\n        with expose_pair(b,x,mask<<1,mask<<4)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(x)+list(g);forms=norm_forms(K)\n    w=inverse3_virtual(b,prefix,forms);source=prefix+list(w)\n    if z is None:z=b.alloc_qubits(6)\n    for j,expression in enumerate(DAG[\'outputs\']):form_xor(b,source,expression&65535,z[j])\n    for i,(left,right)in enumerate(DAG[\'gates\']):\n        vector=sum(((expression>>(16+i))&1)<<j for j,expression in enumerate(DAG[\'outputs\']))\n        assert vector\n        with expose_pair(b,source,left,right)as(p,q):rankone_xor(b,p,q,z,vector)\n    wm=b.alloc_bits(3);b.hmr(w,wm);b.free(w)\n    phase_inverse3_virtual(b,prefix,forms,wm);b.free(wm)\n    kept=None\n    if keep_norm:\n        kept=b.alloc_qubits(3)\n        for j,form in enumerate(forms):form_xor(b,prefix,form,kept[j])\n    for i,(mask,_)in reversed(list(enumerate(tensor(3)))):\n        with expose_pair(b,x,mask<<1,mask<<4)as(p,q):b.del_and(p,q,g[i])\n    b.free(g);linear_inplace(b,x,frm);linear_inplace(b,z,frm)\n    return (z,kept)if keep_norm else z\n\ndef inverse_initialized(b,x,target=None):\n    """No-scratch recursive initialized inverse using the20T base at n6."""\n    n=len(x)\n    if n==3:return inverse3_noscratch(b,x,target)\n    if n==6:return inverse6_xor20(b,x,target)\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to)\n    norm=b.alloc_qubits(k);f=field(k)\n    linear_xor(b,x[:k],norm,square_columns(k))\n    linear_xor(b,x[k:],norm,[f.mul(K,v)for v in square_columns(k)])\n    mul_xor_noscratch(b,x[:k],x[k:],norm)\n    invnorm=inverse_initialized(b,norm)\n    ar._erase_norm(b,x,norm,K);b.free(norm)\n    if target is None:target=b.alloc_qubits(n)\n    b.cx(x[k:],x[:k]);mul_xor_noscratch(b,x[:k],invnorm,target[:k]);mul_xor_noscratch(b,x[k:],invnorm,target[k:]);b.cx(x[k:],x[:k])\n    ar._erase_norm(b,target,invnorm,K);b.free(invnorm)\n    linear_inplace(b,x,frm);linear_inplace(b,target,frm)\n    return target\n\ndef norm_xor(b,x,z,K):\n    k=len(z);f=field(k)\n    linear_xor(b,x[:k],z,square_columns(k))\n    linear_xor(b,x[k:],z,[f.mul(K,v)for v in square_columns(k)])\n    mul_xor_noscratch(b,x[:k],x[k:],z)\n\ndef inverse12_keep_norm(b,x):\n    """65T inverse12 keeping its6-bit norm; used by closed reverse24."""\n    assert len(x)==12\n    to,frm,K=tower(12);linear_inplace(b,x,to)\n    N=b.alloc_qubits(6);norm_xor(b,x,N,K);w=inverse6_xor20(b,N)\n    target=b.alloc_qubits(12);b.cx(x[6:],x[:6])\n    mul_xor_noscratch(b,x[:6],w,target[:6]);mul_xor_noscratch(b,x[6:],w,target[6:]);b.cx(x[6:],x[:6])\n    ar._erase_norm(b,target,w,K);b.free(w)\n    linear_inplace(b,x,frm);linear_inplace(b,target,frm)\n    return target,N\n\ndef inverse_inplace_forward20(b,x):\n    target=inverse_initialized(b,x);mask=b.alloc_bits(len(x))\n    b.hmr(x,mask);b.swap(x,target);b.free(target);return mask\n\ndef inverse_inplace_reverse20(b,x,mask):\n    """Closed reverse12/24 at71/215T,21/42 private qubits."""\n    n=len(x);assert n in(12,24);k=n//2\n    to,frm,K=tower(n);linear_inplace(b,x,to)\n    N=b.alloc_qubits(k);norm_xor(b,x,N,K)\n    t,D=inverse12_keep_norm(b,N)if n==24 else inverse6_xor20(b,N,keep_norm=True)\n    out=b.alloc_qubits(k);mul_xor_noscratch(b,x[k:],t,out)\n    erase_product(b,out,N,x[k:]);mul_xor_noscratch(b,x[:k],t,x[k:]);b.cx(out,x[k:])\n    b.cx(out,x[k:]);erase_product(b,x[k:],N,x[:k]);b.cx(out,x[k:])\n    b.swap(x[:k],x[k:]);b.swap(x[k:],out);b.free(out)\n    ar._erase_inverse_with_norm(b,t,N,D);b.free(N)\n    tok,frmk,Kk=tower(k);linear_inplace(b,t,tok)\n    u=b.alloc_qubits(k//2);norm_xor(b,t,u,Kk);erase_inverse(b,u,D);b.free(D)\n    ar._erase_norm(b,t,u,Kk);b.free(u);linear_inplace(b,t,frmk)\n    ar._erase_norm(b,x,t,K);b.free(t);linear_inplace(b,x,frm)\n    b.cz(x,mask);b.free(mask)\n\ndef inverse24_forward20(b,x,K=None):\n    cols,frm,_=extension_columns(K);linear_inplace(b,x,cols)\n    mask=inverse_inplace_forward20(b,x);linear_inplace(b,x,frm);return mask\n\ndef inverse24_reverse20(b,x,mask,K=None):\n    cols,frm,_=extension_columns(K);linear_inplace(b,x,cols)\n    inverse_inplace_reverse20(b,x,mask);linear_inplace(b,x,frm)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_search/inverse20_iso.py': '"""Width-preserving20T GF64 inverse using the lambda2 norm isotopy.\n\nInitialized output reuses three redundant norm directions as output wires.\nNo core modules are edited or monkeypatched.\n"""\ntry:\n    from . import inverse20 as base\n    from .inverse20 import *\nexcept ImportError:\n    import inverse20 as base\n    from inverse20 import *\nDAG=json.loads(Path(__file__).with_name(\'optimized_isotopy2.json\').read_text())\nEXTRA_ROWS=[11,52,34]\nNORM_ROWS=[42,49,54]\nassert [(e>>7)&63 for e in DAG[\'outputs\']]==[11,0,52,34,52,52]\nassert all(not((l|r)&(63<<7))for l,r in DAG[\'gates\'])\n\ndef raw_forms():\n    f=field(3);scale=2;inv=f.invert(scale);forms=[]\n    for mask,_ in tensor(3):\n        left=sum(((mask&f.mul(scale,1<<i)).bit_count()&1)<<(1+i)for i in range(3))\n        right=sum(((mask&f.mul(inv,1<<i)).bit_count()&1)<<(4+i)for i in range(3))\n        forms.append((left,right))\n    return forms\n\ndef xw_form(mask):return(mask&127)|((mask>>6)&(7<<7))\n\ndef erase_extras(b,x,E):\n    masks=b.alloc_bits(3);b.hmr(E,masks);b.free(E)\n    for i,(left,right)in enumerate(raw_forms()):\n        with expose_pair(b,x,left,right)as(p,q):\n            for j,row in enumerate(EXTRA_ROWS):\n                if row>>i&1:b.ccz(p,q,masks[j])\n    b.free(masks)\n\ndef inverse6_xor20(b,x,z=None,keep_norm=False):\n    """20T exact inverse6: fresh-output Q15; arbitrary-target Q18.\n\n    keep_norm=True returns(input_norm3) too, initialized Q18. All private\n    erasures are explicit0T quadratic phase fixes. Zero maps to zero.\n    """\n    assert len(x)==6;fresh=z is None\n    to,frm,K=tower(6);linear_inplace(b,x,to)\n    if not fresh:linear_inplace(b,z,to)\n    g=b.alloc_qubits(6)\n    for i,(left,right)in enumerate(raw_forms()):\n        with expose_pair(b,x,left,right)as(p,q):b.init_and(p,q,g[i])\n    rows=NORM_ROWS+EXTRA_ROWS\n    cols=[sum(((r>>j)&1)<<i for i,r in enumerate(rows))for j in range(6)]\n    linear_inplace(b,g,cols);N,E=g[:3],g[3:]\n    for j,form in enumerate(norm_forms(K)):form_xor(b,x,form&127,N[j])\n    if not fresh:\n        b.cx(E[0],z[0]);b.cx(E[1],[z[2],z[4],z[5]]);b.cx(E[2],z[3])\n        erase_extras(b,x,E)\n    w=inverse3_noscratch(b,N)\n    if fresh:\n        if not keep_norm:ar._erase_norm(b,x,N,K);b.free(N)\n        extra=b.alloc_qubits(3)\n        # Fixed physical layout: these data wires are now output coordinates.\n        z=[E[0],extra[0],E[1],E[2],extra[1],extra[2]]\n        b.cx(z[2],[z[4],z[5]])\n    source=list(x)+list(w)\n    for j,e in enumerate(DAG[\'outputs\']):form_xor(b,source,xw_form(e),z[j])\n    for i,(left,right)in enumerate(DAG[\'gates\']):\n        vector=sum(((e>>(16+i))&1)<<j for j,e in enumerate(DAG[\'outputs\']))\n        with expose_pair(b,source,xw_form(left),xw_form(right))as(p,q):rankone_xor(b,p,q,z,vector)\n    if fresh:\n        ar._erase_norm(b,z,w,K);b.free(w)\n    else:\n        erase_inverse(b,N,w);b.free(w)\n        if not keep_norm:ar._erase_norm(b,x,N,K);b.free(N)\n    linear_inplace(b,x,frm);linear_inplace(b,z,frm)\n    return(z,N)if keep_norm else z\n\n# These higher-level schedules are unchanged structurally. Rebind their private\n# globals to this plugin\'s inverse6 implementation; no imported module changes.\nfrom types import FunctionType\nfor _name in (\'inverse_initialized\',\'inverse12_keep_norm\',\'inverse_inplace_forward20\',\n              \'inverse_inplace_reverse20\',\'inverse24_forward20\',\'inverse24_reverse20\'):\n    _original=getattr(base,_name)\n    globals()[_name]=FunctionType(_original.__code__,globals(),_name,_original.__defaults__,_original.__closure__)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_width/kernel.py': '"""39-private paired GF24 inverse via a three-bit norm cache.\n\nForward191T is unchanged iso20. Reverse220T adds five Toffolis while\nreducing reverse private workspace42->39. Explicit transcript inclinv0.\nNo stable module is edited or monkeypatched.\n"""\nimport sys,json\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse_search import inverse20_iso as iso\nfrom exp.inverse_search import inverse20 as ref\nfrom exp.inverse_search.inverse20_iso import *\nfrom exp.arithmetic.lowwidth import classical_transpose\nPART=json.loads(Path(__file__).with_name(\'partial5.json\').read_text())\nFULL=json.loads((ROOT/\'exp/inverse_search/optimized.json\').read_text())\n\ndef coefficient_xor(b,t,nm,e,anc):\n    """anc ^=dot(nm,conjugate(t)*e), e GF64 public, t tower12."""\n    f6=field(6)\n    for i in range(6):\n        v=f6.mul(e,1<<i)\n        for r in bits(v):\n            b.ccx(nm[r],t[i],anc);b.ccx(nm[r],t[i+6],anc);b.ccx(nm[r+6],t[i+6],anc)\n\ndef phase_partial_inverse6(b,u,t,nm,cm):\n    """14T: quantum-mask upper3(inv(u)), classical-mask lower3(inv(u)).\n\n    u canonicalGF64, t alreadytower12; nm12 classical, cm3classical.\n    Upper/lower refer to the standardGF8 tower6, not polynomialGF64 bits.\n    All u/norm/inverse prefix state and the one parity ancilla are restored.\n    """\n    to,frm,K=tower(6);linear_inplace(b,u,to);g=b.alloc_qubits(6)\n    for i,(mask,_) in enumerate(tensor(3)):\n        with ref.expose_pair(b,u,mask<<1,mask<<4)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(u)+list(g);forms=ref.norm_forms(K);w=ref.inverse3_virtual(b,prefix,forms);source=prefix+list(w)\n    # Retained lower3 cache\'s ordinary classical HMR correction costs0T.\n    for j,e in enumerate(FULL[\'outputs\'][:3]):\n        assert not(e&1)\n        for r in bits(e&65535):b.cz(source[r-1],cm[j])\n    cb=b.alloc_bits(1)[0];b.bit_store(cb,False)\n    for i,(left,right)in enumerate(FULL[\'gates\']):\n        js=[j for j,e in enumerate(FULL[\'outputs\'][:3]) if e>>(16+i)&1]\n        if not js:continue\n        for j in js:b.cx(cm[j],cb)\n        with ref.expose_pair(b,source,left,right)as(p,q):b.ccz(p,q,cb)\n        for j in reversed(js):b.cx(cm[j],cb)\n    b.free(cb)\n    # Exactly five products span the three missing output functions.\n    anc=b.alloc_qubits(1)[0]\n    def coeff(v):coefficient_xor(b,t,nm,apply(frm,v<<3),anc)\n    for r in range(16):\n        v=sum(((e>>r)&1)<<j for j,e in enumerate(PART[\'outputs\']))\n        if not v:continue\n        coeff(v)\n        if r:b.cz(anc,source[r-1])\n        else:b.z(anc)\n        coeff(v)\n    for i,(left,right)in enumerate(PART[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(PART[\'outputs\']))\n        assert v\n        coeff(v)\n        with ref.expose_pair(b,source,left,right)as(p,q):b.ccz(p,q,anc)\n        coeff(v)\n    b.free(anc)\n    wm=b.alloc_bits(3);b.hmr(w,wm);b.free(w);ref.phase_inverse3_virtual(b,prefix,forms,wm);b.free(wm)\n    for i,(mask,_)in reversed(list(enumerate(tensor(3)))):\n        with ref.expose_pair(b,u,mask<<1,mask<<4)as(p,q):b.del_and(p,q,g[i])\n    b.free(g);linear_inplace(b,u,frm)\n\ndef compress_D(b,N,D):\n    """Retain lower3 tower6 coordinates, eraseupper3 with quadratic norm phase."""\n    to6,frm6,K6=tower(6);linear_inplace(b,D,to6);m=b.alloc_bits(3);b.hmr(D[3:],m);b.free(D[3:])\n    mt=b.alloc_bits(6);b.bit_store(mt,False);b.cx(m,mt[3:]);canon=classical_transpose(b,mt,to6);b.free(mt);b.free(m)\n    to12,frm12,K12=tower(12);linear_inplace(b,N,to12);ar._phase_norm(b,N,canon,K12);linear_inplace(b,N,frm12);b.free(canon)\n    return D[:3]\n\ndef erase_N_cache(b,t,N,C):\n    """Joint cleanup29T: norm(t)15 +sharedprefix9 +fivequantumCCZ."""\n    to12,frm12,K12=tower(12);linear_inplace(b,t,to12);linear_inplace(b,N,to12)\n    nm=b.alloc_bits(12);b.hmr(N,nm);b.free(N);frm6=tower(6)[1];f6=field(6)\n    for j in range(3):\n        for i in range(6):\n            for r in bits(f6.mul(frm6[j],1<<i)):\n                b.ccz(C[j],t[i],nm[r]);b.ccz(C[j],t[i+6],nm[r]);b.ccz(C[j],t[i+6],nm[r+6])\n    cm=b.alloc_bits(3);b.hmr(C,cm);b.free(C)\n    u=b.alloc_qubits(6);iso.norm_xor(b,t,u,K12);phase_partial_inverse6(b,u,t,nm,cm);b.free(nm);b.free(cm)\n    ar._erase_norm(b,t,u,K12);b.free(u);linear_inplace(b,t,frm12)\n\ndef inverse_inplace_reverse39(b,x,mask):\n    assert len(x)==24;k=12;to,frm,K=tower(24);linear_inplace(b,x,to)\n    N=b.alloc_qubits(12);iso.norm_xor(b,x,N,K);t,D=iso.inverse12_keep_norm(b,N);C=compress_D(b,N,D)\n    out=b.alloc_qubits(12);mul_xor_noscratch(b,x[12:],t,out)\n    erase_product(b,out,N,x[12:]);mul_xor_noscratch(b,x[:12],t,x[12:]);b.cx(out,x[12:])\n    b.cx(out,x[12:]);erase_product(b,x[12:],N,x[:12]);b.cx(out,x[12:])\n    b.swap(x[:12],x[12:]);b.swap(x[12:],out);b.free(out)\n    erase_N_cache(b,t,N,C);ar._erase_norm(b,x,t,K);b.free(t);linear_inplace(b,x,frm);b.cz(x,mask);b.free(mask)\n\ndef inverse24_forward39(b,x,K=None):return iso.inverse24_forward20(b,x,K)\n\ndef inverse24_reverse39(b,x,mask,K=None):\n    cols,frm,_=extension_columns(K);linear_inplace(b,x,cols);inverse_inplace_reverse39(b,x,mask);linear_inplace(b,x,frm)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_width_subspaces/kernel19.py': '"""New19-AND GF64 inverse at norm isotopy6; I24 pair190/219T,39private.\n\nNo imported module is changed. The six-bit XOR inverse supports arbitrary\noutput targets; initialized totalQ21 is wider than iso20, but nested I24 peaks\nremain36forward/39reverse because the larger subcall is not the global peak.\n"""\nimport sys,json\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse_search import inverse20 as ref\nfrom exp.inverse_search import inverse20_iso as iso\nfrom exp.inverse_search.inverse20_iso import *\nfrom exp.inverse_width.kernel import compress_D,erase_N_cache\nDAG19=json.loads(Path(__file__).with_name(\'inverse19.json\').read_text())\n\ndef raw_forms19():\n    f=field(3);scale=6;inv=f.invert(scale);out=[]\n    for mask,v in tensor(3):\n        left=sum(((mask&f.mul(scale,1<<i)).bit_count()&1)<<(1+i)for i in range(3))\n        right=sum(((mask&f.mul(inv,1<<i)).bit_count()&1)<<(4+i)for i in range(3));out.append((left,right))\n    return out\n\ndef inverse6_xor19(b,x,z=None,keep_norm=False):\n    assert len(x)==6;to,frm,K=tower(6);linear_inplace(b,x,to)\n    if z is not None:linear_inplace(b,z,to)\n    g=b.alloc_qubits(6)\n    for i,(left,right)in enumerate(raw_forms19()):\n        with ref.expose_pair(b,x,left,right)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(x)+list(g);forms=ref.norm_forms(K);w=ref.inverse3_virtual(b,prefix,forms);source=prefix+list(w)\n    if z is None:z=b.alloc_qubits(6)\n    for j,e in enumerate(DAG19[\'outputs\']):ref.form_xor(b,source,e&65535,z[j])\n    for i,(left,right)in enumerate(DAG19[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(DAG19[\'outputs\']));assert v\n        with ref.expose_pair(b,source,left,right)as(p,q):rankone_xor(b,p,q,z,v)\n    wm=b.alloc_bits(3);b.hmr(w,wm);b.free(w);ref.phase_inverse3_virtual(b,prefix,forms,wm);b.free(wm)\n    kept=None\n    if keep_norm:\n        kept=b.alloc_qubits(3)\n        for j,f in enumerate(forms):ref.form_xor(b,prefix,f,kept[j])\n    for i,(left,right)in reversed(list(enumerate(raw_forms19()))):\n        with ref.expose_pair(b,x,left,right)as(p,q):b.del_and(p,q,g[i])\n    b.free(g);linear_inplace(b,x,frm);linear_inplace(b,z,frm)\n    return (z,kept)if keep_norm else z\n\n# Rebind only this module\'s private copies of the recursive schedules.\nfrom types import FunctionType\ninverse6_xor20=inverse6_xor19\nfor _name in(\'inverse_initialized\',\'inverse12_keep_norm\',\'inverse_inplace_forward20\'):\n    _old=getattr(ref,_name);globals()[_name]=FunctionType(_old.__code__,globals(),_name,_old.__defaults__,_old.__closure__)\n\ndef inverse24_forward19(b,x,K=None):\n    cols,frm,_=extension_columns(K);linear_inplace(b,x,cols)\n    mask=inverse_inplace_forward20(b,x);linear_inplace(b,x,frm);return mask\n\ndef inverse_inplace_reverse19(b,x,mask):\n    to,frm,K=tower(24);linear_inplace(b,x,to);N=b.alloc_qubits(12);iso.norm_xor(b,x,N,K)\n    t,D=inverse12_keep_norm(b,N);C=compress_D(b,N,D);out=b.alloc_qubits(12);mul_xor_noscratch(b,x[12:],t,out)\n    erase_product(b,out,N,x[12:]);mul_xor_noscratch(b,x[:12],t,x[12:]);b.cx(out,x[12:])\n    b.cx(out,x[12:]);erase_product(b,x[12:],N,x[:12]);b.cx(out,x[12:])\n    b.swap(x[:12],x[12:]);b.swap(x[12:],out);b.free(out)\n    erase_N_cache(b,t,N,C);ar._erase_norm(b,x,t,K);b.free(t);linear_inplace(b,x,frm);b.cz(x,mask);b.free(mask)\n\ndef inverse24_reverse19(b,x,mask,K=None):\n    cols,frm,_=extension_columns(K);linear_inplace(b,x,cols);inverse_inplace_reverse19(b,x,mask);linear_inplace(b,x,frm)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/jet24/kernel.py': '"""Completed dual-number inversion reference; allinput, explicitextension basis.\nJ(u,v)=(inv0u,v*(inv0u+[u0])²). Forward290T; closedpair686T/49private.\nNo persistent quantum cache; return only two classical masks.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse12_escape import joint as base\nfrom exp.inverse12_escape import zero1\nfrom exp.inverse12_escape.joint import *\nfrom exp.mul99.kernels import mul_xor99_zero,phase_product_quantum99\nfrom exp.arithmetic.lowwidth import classical_transpose,phase_product_quantum\n\n\ndef phase_zero_virtual(b,prefix,forms,w,mask):\n    """zero(prefix norm) phase,0T while its norm3 andinverse3 forms live."""\n    b.z(mask);source=list(prefix)+list(w);f=field(3)\n    for i in range(3):\n        for j in range(3):\n            if f.mul(1<<i,1<<j)&1:\n                with ref.expose_pair(b,source,forms[i],1<<(1+len(prefix)+j))as(p,q):b.ccz(p,q,mask)\n\n\ndef phase_outputs_classical(b,source,outputs,gates,m):\n    for j,e in enumerate(outputs):\n        if e&1:b.z(m[j])\n        for r in bits((e&65535)>>1):b.cz(source[r],m[j])\n    cb=b.alloc_bits(1)[0];b.bit_store(cb,False)\n    for i,(l,r)in enumerate(gates):\n        js=[j for j,e in enumerate(outputs)if e>>(16+i)&1]\n        for j in js:b.cx(m[j],cb)\n        with ref.expose_pair(b,source,l,r)as(p,q):b.ccz(p,q,cb)\n        for j in reversed(js):b.cx(m[j],cb)\n    b.free(cb)\n\n\ndef phase_inverse12_quantum64(b,x,L,zero_mask=None):\n    """Phase L.inv0(x),quantumL12,64T withpaid-innerprefixretained."""\n    to12,frm12,K12=tower(12);linear_inplace(b,x,to12);T=[sum(((frm12[i]>>j)&1)<<i for i in range(12))for j in range(12)];linear_inplace(b,L,T)\n    D=b.alloc_qubits(6);norm_xor_noscratch(b,x,D,K12);to6,frm6,K6=tower(6);linear_inplace(b,D,to6);g=b.alloc_qubits(6)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,D,l,r)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(D)+list(g);forms=ref.norm_forms(K6);w=ref.inverse3_virtual(b,prefix,forms);source=prefix+list(w);out=b.alloc_qubits(6)\n    for j,e in enumerate(basic.DAG19[\'outputs\']):ref.form_xor(b,source,e&65535,out[j])\n    for i,(l,r)in enumerate(basic.DAG19[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(basic.DAG19[\'outputs\']))\n        with ref.expose_pair(b,source,l,r)as(p,q):rankone_xor(b,p,q,out,v)\n    linear_inplace(b,out,frm6);b.cx(x[6:],x[:6]);phase_product_quantum(b,x[:6],out,L[:6]);phase_product_quantum(b,x[6:],out,L[6:]);b.cx(x[6:],x[:6])\n    if zero_mask is not None:phase_zero_virtual(b,prefix,forms,w,zero_mask)\n    m=b.alloc_bits(6);b.hmr(out,m);b.free(out);mt=classical_transpose(b,m,frm6);b.free(m);phase_outputs_classical(b,source,basic.DAG19[\'outputs\'],basic.DAG19[\'gates\'],mt);b.free(mt)\n    wm=b.alloc_bits(3);b.hmr(w,wm);b.free(w);ref.phase_inverse3_virtual(b,prefix,forms,wm);b.free(wm)\n    for i,(l,r)in reversed(list(enumerate(basic.raw_forms19()))):\n        with ref.expose_pair(b,D,l,r)as(p,q):b.del_and(p,q,g[i])\n    b.free(g);linear_inplace(b,D,frm6);ar._erase_norm(b,x,D,K12);b.free(D);linear_inplace(b,L,inverse_columns(T));linear_inplace(b,x,frm12)\n\n\ndef phase_inverse24_classical106(b,x,mask,zero_mask=None):\n    """Phase mask.inv0(x)+zero_mask*[x0],106T/45private."""\n    to,frm,K=tower(24);linear_inplace(b,x,to);mt=classical_transpose(b,mask,frm);N=b.alloc_qubits(12);norm_xor_noscratch(b,x,N,K);L=b.alloc_qubits(12);f=field(12)\n    def coeff():\n        for j in range(12):\n            for i in range(12):\n                for r in bits(f.mul(1<<i,1<<j)):\n                    b.ccx(mt[r],x[i],L[j]);b.ccx(mt[r],x[i+12],L[j]);b.ccx(mt[r+12],x[i+12],L[j])\n    coeff();phase_inverse12_quantum64(b,N,L,zero_mask);coeff();b.free(L);ar._erase_norm(b,x,N,K);b.free(N);b.free(mt);linear_inplace(b,x,frm)\n\n\ndef inverse24_initialized_zero191(b,x):\n    to,frm,K=tower(24);linear_inplace(b,x,to);N=b.alloc_qubits(12);norm_xor_noscratch(b,x,N,K);t,F=zero1.inverse_zero(b,N);ar._erase_norm(b,x,N,K);b.free(N)\n    z=b.alloc_qubits(24);b.cx(x[12:],x[:12]);mul_xor_noscratch(b,x[:12],t,z[:12]);mul_xor_noscratch(b,x[12:],t,z[12:]);b.cx(x[12:],x[:12]);ar._erase_norm(b,z,t,K);b.free(t);linear_inplace(b,x,frm);linear_inplace(b,z,frm);return z,F\n\n\ndef safe_square(b,x,F,undo=False):\n    sq=square_columns(24)\n    if undo:linear_inplace(b,x,inverse_columns(sq));b.cx(F,x[0])\n    else:b.cx(F,x[0]);linear_inplace(b,x,sq)\n\n\ndef forward_canonical(b,u,v):\n    z,F=inverse24_initialized_zero191(b,u);out=b.alloc_qubits(24);safe_square(b,z,F);mul_xor99_zero(b,v,z,out);safe_square(b,z,F,True)\n    m=b.alloc_bits(24);b.hmr(v,m);safe_square(b,u,F);phase_product_quantum99(b,out,u,m);safe_square(b,u,F,True);b.free(m);b.swap(v,out);b.free(out)\n    mu=b.alloc_bits(24);fz=b.alloc_bits(1);b.hmr(u,mu);b.swap(u,z);b.free(z);b.hmr(F,fz[0]);b.free(F);return mu,fz\n\n\ndef jet_forward(b,u,v,K=None):\n    """290T/49private;returns classical(mu24,zero1),noquantumcache."""\n    cols,frm,_=extension_columns(K);linear_inplace(b,u,cols);linear_inplace(b,v,cols);token=forward_canonical(b,u,v);linear_inplace(b,u,frm);linear_inplace(b,v,frm);return token\n\n\ndef jet_reverse(b,u,v,token,K=None):\n    """396T/49private closedreverse ofjet_forward;106T inversephase."""\n    cols,frm,_=extension_columns(K);linear_inplace(b,u,cols);linear_inplace(b,v,cols);mu2,fz2=forward_canonical(b,u,v);mu,fz=token;b.cz(u,mu);b.free(mu);b.cx(fz,fz2);b.free(fz);phase_inverse24_classical106(b,u,mu2,fz2[0]);b.free(mu2);b.free(fz2);linear_inplace(b,u,frm);linear_inplace(b,v,frm)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/jet24/pro_jet.py': '"""Complete jet kernel:269 forward+389 reverse=658 Toffolis,43 private qubits.\nOnly supplied-library imports and the documented builder API are used.\nTokens contain only classical bits; no per-call state survives elsewhere.\nStatic masks are little endian, with affine bit0=constant; linear maps list columns.\nPlans r/g/p change and retire a source basis, grow outputs, or add one product.\nEvery retired function\'s exact quadratic measurement phase is repaid immediately.\n"""\nfrom field import bits, field, extension_columns, linear_inplace, square_columns, inverse_columns, tower\nfrom exp.mul99.kernels import mul_xor99_zero, phase_product_quantum99\nfrom exp.noscratch.schedules import norm_xor_noscratch\nfrom exp.noscratch.kernels import rankone_xor, mul_xor_noscratch\nfrom field import tensor, apply, linear_xor\nfrom exp.jet24.kernel import safe_square\nimport field as ar\nfrom exp.inverse_search import inverse20 as ref\nLEAF_OLD12 = ([3334, 28416, 16128, 9744], [[86, 118], [84, 112], [76, 96], [70, 126], [64, 122]], 7, 10)\nLEAF1 = ([15939726, 1292428, 48148704, 42382372, 51177556, 58773552], [[130, 530], [326, 658], [474, 542], [220, 840], [456, 664], [176, 304], [318, 512], [310, 690], [236, 776], [394, 638]], 10, 16)\n\ndef forward_canonical(b, u, v):\n    z, F = inverse24(b, u)\n    mu = b.alloc_bits(24)\n    b.hmr(u, mu)\n    b.swap(u, z)\n    b.free(z)\n    out = b.alloc_qubits(24)\n    safe_square(b, u, F)\n    with b.mark(\'multiply v\'): mul_xor99_zero(b, v, u, out)\n    safe_square(b, u, F, True)\n    mv = b.alloc_bits(24)\n    b.hmr(v, mv)\n    b.swap(v, out)\n    b.free(out)\n    fz = b.alloc_bits(1)\n    b.hmr(F, fz[0])\n    b.free(F)\n    return (mu, mv, fz)\n\ndef scale_halves(b, x, t, N):\n    """Scale each half by nonzero t; N is its multiplicative inverse."""\n    for offset in (0, 12):\n        half = x[offset:offset + 12]\n        out = b.alloc_qubits(12)\n        mul_xor_noscratch(b, half, t, out)\n        ar.erase_product(b, out, N, half)\n        b.swap(half, out)\n        b.free(out)\n\ndef erase_inverse_cached(b, t, N, k, F, fz):\n    """Erase N=inv0(t), k=Norm(t), F=[t=0]; include forward flag debt."""\n    to, frm, K = tower(12)\n    linear_inplace(b, t, to)\n    linear_inplace(b, N, to)\n    m = b.alloc_bits(12)\n    b.hmr(N, m)\n    b.free(N)\n    L = b.alloc_qubits(6)\n    f = field(6)\n\n    def coeff():\n        for j in range(6):\n            for i in range(6):\n                for r in bits(f.mul(1 << i, 1 << j)):\n                    b.ccx(m[r], t[i], L[j])\n                    b.ccx(m[r], t[i + 6], L[j])\n                    b.ccx(m[r + 6], t[i + 6], L[j])\n    coeff()\n    phase_inverse6_heldflag17(b, k, L, F, fz[0])\n    coeff()\n    b.free(L)\n    b.free(m)\n    b.free(fz)\n    ar._erase_norm(b, t, k, K)\n    b.free(k)\n    linear_inplace(b, t, frm)\n\ndef start_inverse(b, N):\n    """Keep N; return k=inv0(Norm(N)) and F=[N=0]."""\n    to, frm, K = tower(12)\n    linear_inplace(b, N, to)\n    D = b.alloc_qubits(6)\n    norm_xor_noscratch(b, N, D, K)\n    k, F = inverse6_zero18(b, D)\n    ar._erase_norm(b, N, D, K)\n    b.free(D)\n    linear_inplace(b, N, frm)\n    return (k, F)\n\ndef finish_inverse(b, N, k):\n    """For safe nonzero N, finish t=inv(N), retaining k=Norm(t)."""\n    to, frm, K = tower(12)\n    linear_inplace(b, N, to)\n    t = b.alloc_qubits(12)\n    b.cx(N[6:], N[:6])\n    mul_xor_noscratch(b, N[:6], k, t[:6])\n    mul_xor_noscratch(b, N[6:], k, t[6:])\n    b.cx(N[6:], N[:6])\n    linear_inplace(b, N, frm)\n    linear_inplace(b, t, frm)\n    return t\n\ndef reverse_canonical(b, u, v, token):\n    mu, mv, fz = token\n    to, frm, K = tower(24)\n    linear_inplace(b, u, to)\n    N = b.alloc_qubits(12)\n    with b.mark(\'reverse norm\'): norm_xor_noscratch(b, u, N, K)\n    with b.mark(\'reverse start inverse with flag\'): k, F = start_inverse(b, N)\n    b.cx(u[12:], u[:12])\n    linear_inplace(b, u, frm)\n    safe_square(b, u, F)\n    out = b.alloc_qubits(24)\n    with b.mark(\'reverse multiply v\'): mul_xor99_zero(b, v, u, out)\n    safe_square(b, u, F, True)\n    linear_inplace(b, u, to)\n    b.cx(u[12:], u[:12])\n    vm = b.alloc_bits(24)\n    b.hmr(v, vm)\n    b.swap(v, out)\n    b.free(out)\n    b.cx(F, N[0])\n    b.cx(F, k[0])\n    with b.mark(\'reverse finish inverse\'): t = finish_inverse(b, N, k)\n    linear_inplace(b, v, to)\n    sq = square_columns(12)\n    linear_inplace(b, t, sq)\n    linear_inplace(b, N, sq)\n    with b.mark(\'reverse scale v\'): scale_halves(b, v, t, N)\n    linear_inplace(b, N, inverse_columns(sq))\n    linear_inplace(b, t, inverse_columns(sq))\n    linear_inplace(b, v, frm)\n    linear_inplace(b, u, frm)\n    safe_square(b, u, F)\n    phase_product_quantum99(b, v, u, vm)\n    safe_square(b, u, F, True)\n    b.free(vm)\n    linear_inplace(b, u, to)\n    b.cx(F, u[0])\n    b.cx(u[12:], u[:12])\n    with b.mark(\'reverse scale u\'): scale_halves(b, u, t, N)\n    b.cx(F, u[0])\n    b.cx(F, N[0])\n    b.cx(F, t[0])\n    b.cx(F, k[0])\n    with b.mark(\'reverse cached phase inverse\'): erase_inverse_cached(b, t, N, k, F, fz)\n    ar._erase_norm(b, u, t, K)\n    b.free(t)\n    linear_inplace(b, u, frm)\n    b.cz(u, mu)\n    b.cz(v, mv)\n    b.free(mu)\n    b.free(mv)\n\ndef jet_forward(b, u, v):\n    """Apply J in the decoder basis; return classical (mu, mv, fz)."""\n    cols, frm, _ = extension_columns(None)\n    linear_inplace(b, u, cols)\n    linear_inplace(b, v, cols)\n    token = forward_canonical(b, u, v)\n    linear_inplace(b, u, frm)\n    linear_inplace(b, v, frm)\n    return token\n\ndef jet_reverse(b, u, v, token):\n    """Restore the input and repay every deferred measurement sign."""\n    cols, frm, _ = extension_columns(None)\n    linear_inplace(b, u, cols)\n    linear_inplace(b, v, cols)\n    reverse_canonical(b, u, v, token)\n    linear_inplace(b, u, frm)\n    linear_inplace(b, v, frm)\n\ndef inverse6_zero18_plan(b, x):\n    return inverse6_fixed18(b, x, True)\nPLAN12 = {\'n\': 12, \'terms\': [(33, 2112, 35), (56, 3584, 905), (25, 1600, 362), (55, 3520, 3), (32, 2048, 334), (23, 1472, 397), (21, 1344, 40), (6, 384, 918), (19, 1216, 190), (3, 192, 50), (30, 1920, 656), (29, 1856, 34), (48, 3072, 18), (28, 1792, 87), (44, 2816, 197)], \'affine\': [1, 4, 16, 3, 12, 48, 63, 57, 33, 2, 8, 32], \'reduce\': [65, 129, 3, 256, 1, 512, 4, 8, 26, 55], \'retire\': [24619, 15261, 374, 2885], \'out\': [3216, 2004, 529, 338, 206, 123, 1, 2, 4, 8, 16, 32], \'size\': 10, \'leaves\': [([33, 56, 2112, 3584, 135168, 229376], [9, 42, 585, 2730], LEAF_OLD12), ([55, 32, 3520, 2048, 225280, 131072], [14, 13, 910, 845], LEAF_OLD12), ([21, 6, 1344, 384, 86016, 24576], [22, 62, 1430, 4030], LEAF_OLD12), ([3, 30, 192, 1920, 12288, 122880], [16, 34, 1040, 2210], LEAF_OLD12), ([48, 28, 3072, 1792, 196608, 114688], [23, 5, 1495, 325], LEAF_OLD12)]}\nPLAN24 = {\'n\': 24, \'terms\': [(1776, 1114384, 419755), (1969, 6951184, 1511959), (1697, 332048, 3917), (321, 8257536, 1627580), (2016, 8065296, 1628913), (272, 1315137, 3418), (592, 15793935, 1270639), (3439, 2494223, 177757), (608, 200463, 2329), (3903, 14020608, 628018), (3423, 14089999, 629803), (3855, 15990591, 3908), (3726, 9349354, 1770621), (1207, 12966122, 538163), (3165, 2963690, 1869), (2617, 6701056, 1626702), (1636, 4946154, 1627395), (2282, 10721849, 382), (1877, 9365742, 1084187), (3879, 8165614, 1481953), (1993, 641262, 1299), (2162, 16494592, 604154), (4027, 15890670, 602857), (2286, 8857714, 2546), (119, 2089470, 1642329), (714, 3359230, 238442), (820, 3420670, 1033), (701, 1609728, 39987), (393, 2925054, 38970), (510, 2871997, 1891), (2300, 16154471, 1179523), (233, 16314215, 1165379), (3982, 7810919, 412), (2069, 7974912, 903104), (1947, 958311, 902748), (3943, 8476693, 2527), (888, 8947848, 1473400), (984, 11864200, 1664433), (2896, 8554632, 415), (160, 12517376, 917193), (3056, 4032648, 917334), (2184, 655520, 1070)], \'affine\': [755, 2929, 2347, 572, 2191, 2631, 3604, 1538, 3661, 1633, 237, 499, 3208, 725, 2900, 3574, 1837, 3303, 873, 3492, 1637, 2503, 1978, 3771], \'reduce\': [1, 2, 4, 9, 22, 61, 127, 251, 394, 632, 2036, 3457, 4096, 12288, 20480, 61440, 77824, 143360, 286720, 524288, 1056768], \'retire\': [], \'out\': [4910144, 9817152, 3062784, 1540096, 111680, 186368, 266304, 535616, 46080, 25600, 7232, 10304, 65, 1027, 2053, 15, 18, 34, 134, 256, 515, 1, 2, 4], \'size\': 21, \'leaves\': [([321, 272, 2016, 8257536, 6950912, 1114112, 4563402752, 29796335616, 28470935552], [1468, 2801, 3418, 9332966, 6014396, 11475697], LEAF1), ([3903, 3855, 3423, 14020608, 2490368, 15790080, 64676167680, 9932111872, 10200547328], [1330, 3115, 3908, 10971766, 5449010, 12762155], LEAF1), ([2617, 2282, 1636, 6701056, 12963840, 9347072, 38285606912, 62511906816, 53099888640], [590, 1283, 382, 3343152, 2417230, 5256451], LEAF1), ([2162, 2286, 4027, 16494592, 8163328, 9363456, 38352715776, 31490834432, 33436991488], [2042, 745, 2546, 14716424, 8366074, 3052265], LEAF1), ([701, 510, 393, 1609728, 3358720, 2088960, 8556380160, 1996488704, 13757317120], [3123, 2106, 1891, 11864912, 12794931, 8628282], LEAF1), ([2069, 3943, 1947, 7974912, 16310272, 16150528, 66152562688, 38587596800, 66806874112], [1984, 1628, 2527, 14810655, 8128448, 6669916], LEAF1), ([160, 2184, 3056, 12517376, 11862016, 8945664, 36641439744, 14898167808, 48586817536], [3785, 3926, 1070, 11434727, 15507145, 16084822], LEAF1)]}\n\ndef cached_inverse(b, x, plan, with_zero=True):\n    n = plan[\'n\']\n    h = n // 2\n    to, frm, K = tower(n)\n    linear_inplace(b, x, to)\n    terms = plan[\'terms\']\n    with b.mark(\'inverse%d norm\' % n): raw = raw_cache(b, x, plan)\n    D = raw[:h]\n    linear_xor(b, x, D, plan[\'affine\'])\n    with b.mark(\'inverse%d scalar\' % n):\n        value = inverse6_fixed18(b, D, with_zero) if h == 6 else cached_inverse(b, D, PLAN12, with_zero)\n    k, F = value\n    linear_xor(b, x, D, plan[\'affine\'])\n    linear_inplace(b, raw, plan[\'reduce\'])\n    kept = plan[\'size\'] - len(plan[\'retire\'])\n    for q, row in zip(raw[kept:], plan[\'retire\']):\n        mask = b.alloc_bits(1)[0]\n        b.hmr(q, mask)\n        for j in bits(row):\n            left, right, _ = terms[j]\n            with ref.expose_pair(b, x, left << 1, right << 1) as (p, r): b.ccz(p, r, mask)\n        b.free(mask)\n        b.free(q)\n    out = list(raw[:kept]) + list(b.alloc_qubits(n - kept))\n    linear_inplace(b, out, plan[\'out\'])\n    source = list(x) + list(k)\n    with b.mark(\'inverse%d products\' % n):\n        for forms, vectors, leaf in plan[\'leaves\']:\n            outputs, gates, low, offset = leaf\n\n            def remap(form):\n                result = form & 1\n                for j in bits(form >> 1): result ^= forms[j] << 1\n                return result\n            for e, v in zip(outputs, vectors):\n                if e & 1:\n                    for j in bits(v): b.x(out[j])\n                for a in bits((e & (1 << low) - 1) >> 1):\n                    for i in bits(forms[a]):\n                        for j in bits(v): b.cx(source[i], out[j])\n            for i, (left, right) in enumerate(gates):\n                vec = 0\n                for e, v in zip(outputs, vectors):\n                    if e >> offset + i & 1: vec ^= v\n                with ref.expose_pair(b, source, remap(left), remap(right)) as (p, q): rankone_xor(b, p, q, out, vec)\n    ar._erase_norm(b, out, k, K)\n    b.free(k)\n    linear_inplace(b, x, frm)\n    linear_inplace(b, out, frm)\n    return (out, F)\n\ndef inverse24(b, x):\n    return cached_inverse(b, x, PLAN24, True)\ninverse6_zero18 = inverse6_zero18_plan\nTO6 = [1, 17, 37, 54, 49, 30]\nFROM6 = [1, 14, 23, 58, 3, 18]\n\ndef raw_cache(b, x, plan):\n    raw = b.alloc_qubits(plan[\'size\'])\n    for left, right, vec in plan[\'terms\']:\n        with ref.expose_pair(b, x, left << 1, right << 1) as (p, q): rankone_xor(b, p, q, raw, vec)\n    return raw\nW14_JOINT18 = {\'K\': 1, \'scale\': 1, \'raw_gates\': [[2, 16], [14, 112], [8, 64], [10, 80], [12, 96], [6, 48]], \'gates\': [[63688, 65242], [47190, 50256], [42244, 64160], [42064, 65236], [37240, 62320], [11090, 43290], [9360, 33288], [48208, 51212], [8560, 51442]], \'outputs\': [31410304, 13183220, 23895034, 16089926, 14693266, 6608146, 67091], \'AND\': 18}\n\ndef phase_inverse6_heldflag17(b, x, L, F, zero_mask=None):\n    """Apply inverse phase; consume the supplied exact F=[x=0]."""\n    dag = W14_JOINT18\n    frm = FROM6\n    linear_inplace(b, x, TO6)\n    g = b.alloc_qubits(6)\n    for j, (l, r) in enumerate(dag[\'raw_gates\']):\n        with ref.expose_pair(b, x, l, r) as (p, q): b.init_and(p, q, g[j])\n    prefix = list(x) + list(g)\n    forms = ref.norm_forms(1)\n    w = ref.inverse3_virtual(b, prefix, forms)\n    source = prefix + list(w)\n    trans = [sum(((frm[i] >> j & 1) << i for i in range(6))) for j in range(6)]\n    linear_inplace(b, L, trans)\n    for j, e in enumerate(dag[\'outputs\'][:6]):\n        uses_g0 = e >> 16 & 1\n        lin = e & 65535 ^ (1555 if uses_g0 else 0)\n        if lin & 1: b.z(L[j])\n        for r in bits(lin >> 1): b.cz(L[j], source[r])\n        if uses_g0: b.cz(F, L[j])\n    for i, (l, r) in enumerate(dag[\'gates\'][1:], 1):\n        vector = sum(((e >> 16 + i & 1) << j for j, e in enumerate(dag[\'outputs\'][:6])))\n        js = list(bits(vector))\n        if not js: continue\n        pL = js[0]\n        for j in js[1:]: b.cx(L[j], L[pL])\n        with ref.expose_pair(b, source, l, r) as (p, q): b.ccz(p, q, L[pL])\n        for j in reversed(js[1:]): b.cx(L[j], L[pL])\n    if zero_mask is not None: b.cz(F, zero_mask)\n    fm = b.alloc_bits(1)[0]\n    b.hmr(F, fm)\n    b.free(F)\n    b.z(fm)\n    for r in bits(1555 >> 1): b.cz(source[r], fm)\n    with ref.expose_pair(b, source, *dag[\'gates\'][0]) as (p, q): b.ccz(p, q, fm)\n    b.free(fm)\n    m = b.alloc_bits(3)\n    b.hmr(w, m)\n    b.free(w)\n    ref.phase_inverse3_virtual(b, prefix, forms, m)\n    b.free(m)\n    for i, (l, r) in reversed(list(enumerate(dag[\'raw_gates\']))):\n        with ref.expose_pair(b, x, l, r) as (p, q): b.del_and(p, q, g[i])\n    b.free(g)\n    linear_inplace(b, L, inverse_columns(trans))\n    linear_inplace(b, x, frm)\nRAW = [[2, 16], [14, 112], [8, 64], [10, 80], [12, 96], [6, 48]]\nPLAN = [(\'r\', [397, 80, 58, 2, 153, 25, 9, 5, 1], 9, []), (\'g\', 1, None, None), (\'p\', 2948, 4256, 1), (\'r\', [1, 2, 4, 8, 16, 64, 304, 32, 128], 9, []), (\'g\', 1, None, None), (\'p\', 7928, 12528, 2), (\'r\', [1, 2, 4, 8, 16, 32, 160, 64, 256], 9, []), (\'g\', 1, None, None), (\'p\', 19728, 9864, 4), (\'r\', [1, 2, 4, 8, 16, 32, 64, 128, 256], 9, []), (\'p\', 17360, 46220, 7), (\'r\', [1, 2, 4, 8, 16, 32, 64, 256, 128], 8, [None]), (\'p\', 27760, 21746, 13), (\'r\', [1, 2, 4, 8, 16, 32, 64, 128], 7, [None]), (\'p\', 7250, 15002, 27), (\'r\', [1, 2, 4, 8, 16, 32, 64], 5, [(11, [(4, 2), (4, 3), (5, 1), (6, 1), (6, 3)]), None]), (\'g\', 1, None, None), (\'p\', 200, 474, 64), (\'r\', [17, 14, 3, 10, 12], 4, [(104, [(4, 1), (4, 2), (5, 1), (5, 3), (6, 2), (7, 0), (8, 0), (10, 0)])]), (\'p\', 214, 464, 41), (\'r\', [5, 10, 1, 2], 2, [(62, [(4, 2), (4, 3), (5, 1), (5, 3), (6, 1), (6, 2), (7, 0)]), (45, [(1, 0), (3, 0), (3, 2), (4, 0), (4, 1), (4, 2), (4, 3), (5, 0), (5, 3), (5, 4), (6, 0), (6, 1), (6, 2), (6, 5), (7, 0), (7, 2), (7, 3), (7, 6), (8, 0), (8, 1), (8, 2), (8, 3), (8, 7), (9, 5), (9, 6), (9, 7), (10, 0), (10, 1), (10, 4), (10, 5), (10, 7), (10, 8), (11, 0), (11, 2), (11, 3), (11, 5), (11, 9), (11, 10)])]), (\'p\', 208, 468, 59), (\'r\', [1, 2], 0, [(126, [(2, 0), (2, 1), (3, 0), (4, 1), (4, 2), (4, 3), (5, 0), (5, 4), (6, 0), (6, 1), (7, 2), (7, 4), (8, 0), (8, 7), (9, 2), (9, 4), (10, 0), (10, 1), (10, 6), (10, 7), (10, 9), (11, 7), (11, 8), (12, 8), (12, 9)]), (75, [(2, 0), (2, 1), (4, 2), (5, 1), (5, 2), (6, 2), (6, 3), (6, 4), (7, 0), (7, 2), (7, 3), (7, 4), (8, 2), (8, 5), (9, 1), (9, 6), (9, 8), (10, 2), (10, 8)])])]\nOUTPUT = [45, 9, 63, 23, 15, 59, 75]\nLINEAR = [0, 116, 122, 70, 18, 18, 19]\n\ndef inverse6_joint18_12extra(b, x):\n    frm = FROM6\n    linear_inplace(b, x, TO6)\n    raw = b.alloc_qubits(6)\n    for j, (l, r) in enumerate(RAW):\n        with ref.expose_pair(b, x, l, r) as (p, q): b.init_and(p, q, raw[j])\n    w = ref.inverse3_virtual(b, list(x) + list(raw), ref.norm_forms(1))\n    raw = list(raw) + list(w)\n    out = []\n    for kind, a, c, d in PLAN:\n        if kind == \'r\':\n            linear_inplace(b, raw, a)\n            for j, action in enumerate(d, c):\n                if action is None:\n                    out.append(raw[j])\n                    continue\n                coeff, phase = action\n                for k in bits(coeff): b.cx(raw[j], out[k])\n                m = b.alloc_bits(1)[0]\n                b.hmr(raw[j], m)\n                live = list(x) + raw[:c] + raw[j + 1:] + out\n                for i, k in phase:\n                    if i == 0: b.z(m)\n                    elif k == 0 or i == k: b.cz(live[i - 1], m)\n                    else:\n                        b.ccz(live[i - 1], live[k - 1], m)\n                b.free(m)\n                b.free(raw[j])\n            raw = list(raw[:c])\n        elif kind == \'g\': out.extend(b.alloc_qubits(a))\n        else:\n            with ref.expose_pair(b, list(x) + raw, a, c) as (p, q): rankone_xor(b, p, q, out, d)\n    linear_inplace(b, out, OUTPUT)\n    for j, form in enumerate(LINEAR): ref.form_xor(b, x, form, out[j])\n    linear_inplace(b, x, frm)\n    linear_inplace(b, out[:6], frm)\n    return (out[:6], out[6])\n\ndef inverse6_fixed18(b, x, joint=True):\n    return inverse6_joint18_12extra(b, x)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/mask_projection/kernel.py': '"""Classical mask projection onto the70 syndrome coordinates,0 Toffolis.\n\nInterpolate a degree<=70 polynomial F with F(0)=0 through70 selected masks\nin the trace-dual field basis. Pay Tr(sum S_k F_(k+1)) directly, subtract\nF\'s evaluations from every mask, and omit the70 now-zero scan positions.\nAn additive FFT avoids a dense4095x70 classical CNOT transform.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.decoder.common import *\n\nSELECTED=tuple(range(70))\n\ndef ev(poly,x):\n    y=0\n    for a in reversed(poly):y=mul(y,x)^a\n    return y\n\ndef tr(x):\n    y=0\n    for _ in range(12):y^=x;x=mul(x,x)\n    assert y in(0,1)\n    return y\n\n@lru_cache(None)\ndef trace_columns():\n    return tuple(sum(tr(mul(1<<i,1<<j))<<i for i in range(12))for j in range(12))\n\n@lru_cache(None)\ndef subspace_polynomials():\n    out=[[0,1]]\n    for j in range(12):\n        s=out[-1];d=ev(s,1<<j);assert d\n        z=[0]*(2*len(s)-1)\n        for i,c in enumerate(s):z[2*i]^=mul(c,c);z[i]^=mul(d,c)\n        out.append(z)\n    return out\n\n@lru_cache(None)\ndef novel_basis(active):\n    ss=subspace_polynomials();out=[]\n    for i in range(active):\n        p=[1]\n        for j in range(i.bit_length()):\n            if i>>j&1:p=pmul(p,ss[j])\n        assert len(p)==i+1 and p[-1]==1\n        out.append(tuple(p))\n    return tuple(out)\n\ndef to_novel(b,words,active,reverse=False):\n    basis=novel_basis(active)\n    for i in(range(1,active)if reverse else range(active-1,0,-1)):\n        for j,c in enumerate(basis[i][:-1]):cx(b,words[i],words[j],c)\n\ndef additive_fft(b,words,m=12,offset=0,active=71,reverse=False):\n    """Novel-basis coefficients -> values at offset+span(1,...,2^(m-1)).\n\n    Only the first active coefficients may be nonzero on entry. The inverse\n    restores the unused classical wires tozero. Output order is numeric field\n    representation, not primitive-element exponent order.\n    """\n    assert len(words)==1<<m and 0<active<=len(words)\n    if not m:return\n    h=1<<(m-1);lo,hi=words[:h],words[h:];high=max(0,active-h)\n    nextactive=min(active,h);ss=subspace_polynomials()[m-1]\n    c=ev(ss,offset);d=ev(ss,1<<(m-1));assert d\n    if reverse:\n        additive_fft(b,lo,m-1,offset,nextactive,True)\n        additive_fft(b,hi,m-1,offset^(1<<(m-1)),nextactive,True)\n    for j in range(nextactive):\n        if j>=high:b.cx(lo[j],hi[j])\n        elif not reverse:\n            cx(b,hi[j],lo[j],c);scale(b,hi[j],d);b.cx(lo[j],hi[j])\n        else:\n            b.cx(lo[j],hi[j]);scale(b,hi[j],inv(d));cx(b,hi[j],lo[j],c)\n    if not reverse:\n        additive_fft(b,lo,m-1,offset,nextactive)\n        additive_fft(b,hi,m-1,offset^(1<<(m-1)),nextactive)\n\n@lru_cache(None)\ndef interpolation_matrix():\n    alpha=gf.primitive_element\n    return tuple(tuple(power(power(alpha,j),k+1)for k in range(70))for j in SELECTED)\n\ndef residual_masks(b,E,F):\n    beta=1;T=trace_columns()\n    for mask in E:\n        ar.linear_xor(b,F[beta],mask,T);beta=mul(beta,gf.primitive_element)\n    assert beta==1\n\ndef project(b,S,E):\n    assert len(S)==70 and len(E)==4095\n    F=[b.alloc_bits(12)for _ in range(4096)]\n    b.bit_store(combine(F),False)\n    Ti=ar.inverse_columns(trace_columns())\n    for k,j in enumerate(SELECTED):\n        b.cx(E[j],F[k+1]);ar.linear_inplace(b,F[k+1],Ti)\n    linear_words(b,F[1:71],interpolation_matrix(),reverse=True)\n    for s,f in zip(S,F[1:71]):\n        ar.linear_inplace(b,f,trace_columns());b.cz(s,f);ar.linear_inplace(b,f,Ti)\n    to_novel(b,F,71);additive_fft(b,F)\n    residual_masks(b,E,F)\n    return F\n\ndef restore(b,E,F):\n    residual_masks(b,E,F)\n    # F is classical deterministic working memory, not quantum garbage.\n    for word in F:b.free(word)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/mul99/kernels.py': '"""GF24 multiplication: GF16 degree6, 11 evaluations times rank9 =99.\n\nSeparate experimental implementation; does not edit the stable arithmetic API.\n"""\nfrom functools import lru_cache\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2]\nsys.path.insert(0,str(ROOT/\'exp/arithmetic\'))\nimport field as ar\nfrom field import *\n\n\ndef extension_tensor(n,k,d,subterms):\n    f=field(n);sub=field(k);gen=f.pow(f.primitive_element,(2**n-1)//(2**k-1))\n    def ev(v,p):\n        out=0\n        for i in reversed(range(k+1)):out=f.mul(out,p)^((v>>i)&1)\n        return out\n    root=next(f.pow(gen,i) for i in range(1,2**k) if ev(sub.modulus,f.pow(gen,i))==0)\n    embed=[f.pow(root,i) for i in range(k)]\n    cols=[f.mul(v,f.pow(f.primitive_element,j)) for j in range(d) for v in embed]\n    inv=inverse_columns(cols);masks=[]\n    for point in list(range(2*d-2))+[None]:\n        values=[]\n        for co in inv:\n            if point is None:value=(co>>(k*(d-1)))&((1<<k)-1)\n            else:\n                value=0\n                for j in reversed(range(d)):value=sub.mul(value,point)^((co>>(k*j))&((1<<k)-1))\n            values.append(value)\n        for mask,_ in subterms:\n            masks.append(sum(((v&mask).bit_count()%2)<<j for j,v in enumerate(values)))\n    equations=[]\n    for i in range(n):\n        for j in range(i,n):\n            row=sum((((m>>i)&1)&((m>>j)&1))<<r for r,m in enumerate(masks))\n            equations.append((row,f.mul(1<<i,1<<j)))\n    vectors=ar._solve_output_vectors(equations,len(masks));assert all(vectors)\n    return tuple(zip(masks,vectors))\n\n\n@lru_cache(None)\ndef tensor4():return extension_tensor(4,2,2,tensor(2))\n\n\n@lru_cache(None)\ndef tensor99():\n    out=extension_tensor(24,4,6,tensor4());assert len(out)==99\n    return out\n\n\ndef multiplier(b,x,y,z,terms):\n    assert not(set(map(str,x))&set(map(str,y)) or set(map(str,x))&set(map(str,z)) or set(map(str,y))&set(map(str,z)))\n    for mask,v in terms:\n        ix=list(bits(mask));iz=list(bits(v));p=ix[0];t=iz[0]\n        for i in ix[1:]:b.cx(x[i],x[p]);b.cx(y[i],y[p])\n        for j in iz[1:]:b.cx(z[t],z[j])\n        b.ccx(x[p],y[p],z[t])\n        for j in reversed(iz[1:]):b.cx(z[t],z[j])\n        for i in reversed(ix[1:]):b.cx(y[i],y[p]);b.cx(x[i],x[p])\n\n\ndef mul_xor99_zero(b,x,y,z):return multiplier(b,x,y,z,tensor99())\n\n\n@lru_cache(None)\ndef tensor99_extension(K=None):\n    cols,invcols,_=extension_columns(K)\n    return tuple((sum(((mask&col).bit_count()%2)<<j for j,col in enumerate(cols)),apply(invcols,v))\n                 for mask,v in tensor99())\n\n\ndef mul24_xor99_zero(b,x,y,z,K=None):return multiplier(b,x,y,z,tensor99_extension(K))\n\n\ndef phase_product_quantum99(b,x,y,mask):\n    for m,v in tensor99():\n        ix=list(bits(m));iz=list(bits(v));p=ix[0];t=iz[0]\n        for i in ix[1:]:b.cx(x[i],x[p]);b.cx(y[i],y[p])\n        for j in iz[1:]:b.cx(mask[j],mask[t])\n        b.ccz(x[p],y[p],mask[t])\n        for j in reversed(iz[1:]):b.cx(mask[j],mask[t])\n        for i in reversed(ix[1:]):b.cx(y[i],y[p]);b.cx(x[i],x[p])\n\n\n# Stable public spellings: no zero-target promise is required.\nmul_xor99=mul_xor99_zero\nmul24_xor99=mul24_xor99_zero\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/mul99/phase36.py': '"""Inverse12 phase with12 scratch and at most36 EXECUTED Toffolis.\n\nThe coefficient mask is classical. A classical first-nonzero pivot exposes\nits parity in an existing input bit, avoiding the13th quantum wire.\n"""\ntry:\n    from .kernels import *\nexcept ImportError:\n    from kernels import *\nsys.path.insert(0,str(ROOT))\nfrom exp.noscratch.kernels import inverse3_noscratch\nfrom lowwidth import classical_transpose\n\n\ndef phase_classical_parity(b,a,c,x,co):\n    """CCZ(a,c,parity(co*x)), zero quantum scratch, <=1 executed CCZ."""\n    first=b.alloc_bits(len(x));b.bit_store(first,False)\n    seen=b.alloc_bits(1)[0];b.bit_store(seen,False)\n    for j in range(len(x)):\n        b.x(seen);b.push_condition(seen)\n        b.bit_store(first[j],True,control=co[j])\n        b.pop_condition();b.x(seen)\n        b.bit_store(seen,True,control=co[j])\n    for j in range(len(x)):\n        b.push_condition(first[j])\n        for k in range(len(x)):\n            if k!=j:b.ccx(co[k],x[k],x[j])\n        b.ccz(a,c,x[j])\n        for k in reversed(range(len(x))):\n            if k!=j:b.ccx(co[k],x[k],x[j])\n        b.pop_condition()\n    b.free(seen);b.free(first)\n\n\nTRACE = []\n\n\ndef phase_inverse36_pivot(b,x,mask):\n    """Choose an exact classical-mask inverse phase by current live width."""\n    live = b.num_allocated_qubits\n    use_virtual = live + 15 <= 889\n    caller = sys._getframe(1)\n    TRACE.append(dict(live=live, virtual=use_virtual,\n                      caller=caller.f_code.co_name, line=caller.f_lineno,\n                      file=caller.f_code.co_filename))\n    if use_virtual:\n        # Import only at call time, after exact_subs has finished initialization.\n        # Do not rebind its implementation or route it through this selector.\n        from exact_subs import phase_inverse12_virtual\n        return phase_inverse12_virtual(b,x,mask)\n    return phase_inverse36_pivot_original(b,x,mask)\n\n\ndef phase_inverse36_pivot_original(b,x,mask):\n    active=b.alloc_bits(1)[0]\n    b.bit_store(active,False)\n    for bit in mask:b.bit_store(active,True,control=bit)\n    b.push_condition(active)\n    phase_inverse36_pivot_nonzero(b,x,mask)\n    b.pop_condition()\n    b.free(active)\n\ndef phase_inverse36_pivot_nonzero(b,x,mask):\n    """Phase mask dot inverse0(x): <=36 mean T /12 quantum scratch.\n\n    Static count168T due to12 mutually exclusive classical branches per CCZ.\n    No quantum input controls branch selection; all branches use HMR-mask bits.\n    """\n    assert len(x)==len(mask)==12\n    to12,frm12,K12=tower(12);linear_inplace(b,x,to12)\n    mt=classical_transpose(b,mask,frm12)\n    D=b.alloc_qubits(6);ar._norm_xor(b,x,D,K12)\n    to6,frm6,K6=tower(6);linear_inplace(b,D,to6)\n    E=b.alloc_qubits(3);ar._norm_xor(b,D,E,K6);w=inverse3_noscratch(b,E)\n    f6=field(6)\n    for segment in(0,1):\n        if segment==0:b.cx(D[3:],D[:3])\n        controls=D[:3] if segment==0 else D[3:]\n        for m,v in tensor(3):\n            h=apply(frm6,v<<(3*segment));co=b.alloc_bits(12);b.bit_store(co,False)\n            for j in range(6):\n                for r in bits(f6.mul(h,1<<j)):\n                    b.cx(mt[r],co[j]);b.cx(mt[r],co[j+6]);b.cx(mt[r+6],co[j+6])\n            ix=list(bits(m));p=ix[0]\n            for j in ix[1:]:b.cx(controls[j],controls[p]);b.cx(w[j],w[p])\n            phase_classical_parity(b,controls[p],w[p],x,co)\n            for j in reversed(ix[1:]):b.cx(w[j],w[p]);b.cx(controls[j],controls[p])\n            b.free(co)\n        if segment==0:b.cx(D[3:],D[:3])\n    erase_inverse(b,E,w);b.free(w)\n    ar._erase_norm(b,D,E,K6);b.free(E);linear_inplace(b,D,frm6)\n    ar._erase_norm(b,x,D,K12);b.free(D);linear_inplace(b,x,frm12);b.free(mt)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/mul99/schedules.py': '"""Explicit compositions; no patching or edits of existing arithmetic modules."""\ntry:\n    from .kernels import *\nexcept ImportError:\n    from kernels import *\nsys.path.insert(0,str(ROOT))\nfrom exp.noscratch.schedules import norm_xor_noscratch,inverse_stream_noscratch\nfrom exp.noscratch.phase39 import phase_inverse39\nfrom exp.noscratch.kernels import mul_xor_noscratch\nfrom division import phase_product24,zero_flag,phase_zero\ntry:\n    from .phase36 import phase_inverse36_pivot\nexcept ImportError:\n    from phase36 import phase_inverse36_pivot\n\n\ndef multiply_forward99(b,x,y,K=None):\n    """x*=y:99T/24scratch, old-x phase deferred. Promise x=0 when y=0."""\n    z=b.alloc_qubits(24);mul24_xor99(b,x,y,z,K)\n    m=b.alloc_bits(24);b.hmr(x,m);b.swap(x,z);b.free(z);return m\n\n\ndef divide_lowwidth372(b,x,y,K=None):\n    """x/=y, preserves y:372T/24scratch. Promise x=0 when y=0."""\n    return _divide_lowwidth(b,x,y,K,phase_inverse39)\n\n\ndef divide_lowwidth369(b,x,y,K=None):\n    """Same division, <=369 EXECUTED T /24scratch; static count501T."""\n    return _divide_lowwidth(b,x,y,K,phase_inverse36_pivot)\n\n\ndef _divide_lowwidth(b,x,y,K,phase_kernel):\n    if K is None:K=extension_columns()[2]\n    b.cx(y[12:],y[:12]);oldmask=multiply_forward99(b,x,y,K);b.cx(y[12:],y[:12])\n    u=b.alloc_qubits(12);norm_xor_noscratch(b,y,u,K)\n    nm=inverse_stream_noscratch(b,u);ar._phase_norm(b,y,nm,K);b.free(nm)\n    masks=[]\n    for offset in(0,12):\n        half=x[offset:offset+12];out=b.alloc_qubits(12);mul_xor_noscratch(b,half,u,out)\n        m=b.alloc_bits(12);b.hmr(half,m);masks.append(m);b.swap(half,out);b.free(out)\n    um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,y,N,K)\n    for offset,m in zip((0,12),masks):phase_product(b,x[offset:offset+12],N,m);b.free(m)\n    phase_kernel(b,N,um);b.free(um)\n    ar._erase_norm(b,y,N,K);b.free(N)\n    phase_product24(b,x,y,oldmask,K);b.free(oldmask)\n\n\ndef divide_zero_bridge417(b,x,y,K=None,extra_zero_mask=None):\n    """x/=y if y!=0, else identity:417T/25scratch, all inputs exact."""\n    flag=zero_flag(b,y);b.cx(flag,y[0])\n    divide_lowwidth372(b,x,y,K)\n    b.cx(flag,y[0])\n    mask=b.alloc_bits(1);b.hmr(flag,mask[0]);b.free(flag)\n    if extra_zero_mask is not None:b.cx(extra_zero_mask,mask[0])\n    phase_zero(b,y,mask[0]);b.free(mask)\n\n\ndef divide_zero_bridge414(b,x,y,K=None,extra_zero_mask=None):\n    """All-input exact zero bridge: <=414 EXECUTED T /25scratch."""\n    flag=zero_flag(b,y);b.cx(flag,y[0])\n    divide_lowwidth369(b,x,y,K)\n    b.cx(flag,y[0])\n    mask=b.alloc_bits(1);b.hmr(flag,mask[0]);b.free(flag)\n    if extra_zero_mask is not None:b.cx(extra_zero_mask,mask[0])\n    phase_zero(b,y,mask[0]);b.free(mask)\n\n\ndef multiply_zero_forward99(b,x,y,K=None):\n    """x*=y if y!=0, else identity:146T/25scratch; deferred masks."""\n    flag=zero_flag(b,y)\n    z=b.alloc_qubits(24);mul24_xor99(b,x,y,z,K);b.ccx(flag,x,z)\n    old=b.alloc_bits(24);b.hmr(x,old);b.swap(x,z);b.free(z)\n    mask=b.alloc_bits(1);b.hmr(flag,mask[0]);b.free(flag)\n    return old,mask[0]\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/noscratch/kernels.py': '"""Scratch-free rankone bilinear accumulation; no coremodule edits."""\nimport sys\nfrom pathlib import Path\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT/\'exp/arithmetic\'))\nimport field as ar\nfrom field import *\n\ndef rankone_xor(b,x,y,z,vector):\n    """z^=vector*(xANDy), arbitraryz, zeroancilla, oneCCX.\n\n    x,y arebits; vector is nonzero bitmaskoverz. Inputsdisjointfromtarget.\n    """\n    out=list(bits(vector));p=out[0]\n    for j in out[1:]:b.cx(z[p],z[j])\n    b.ccx(x,y,z[p])\n    for j in reversed(out[1:]):b.cx(z[p],z[j])\n\ndef tensor_xor_noscratch(b,x,y,z,terms):\n    assert len(x)==len(y)==len(z)\n    assert not(set(map(str,x))&set(map(str,y))or set(map(str,x))&set(map(str,z))or set(map(str,y))&set(map(str,z)))\n    for mask,vector in terms:\n        ix=list(bits(mask));p=ix[0]\n        for j in ix[1:]:b.cx(x[j],x[p]);b.cx(y[j],y[p])\n        rankone_xor(b,x[p],y[p],z,vector)\n        for j in reversed(ix[1:]):b.cx(y[j],y[p]);b.cx(x[j],x[p])\n\ndef mul_xor_noscratch(b,x,y,z):\n    """DisjointGF2^n productaccumulate; n2/3/6/12, rank3/6/15/42,0scratch."""\n    tensor_xor_noscratch(b,x,y,z,tensor(len(x)))\n\ndef mul24_xor_noscratch(b,x,y,z,K=None):\n    """ExplicitquadraticGF24 productaccumulate, arbitrarytarget,105T/0scratch."""\n    assert len(x)==24;tensor_xor_noscratch(b,x,y,z,ar.tensor24_extension(K))\n\ndef inverse3_xor_noscratch(b,x,z):\n    """z^=inverseGF8(x), inverse0=0, arbitrarytarget,3T/0scratch."""\n    assert len(x)==len(z)==3 and not(set(map(str,x))&set(map(str,z)))\n    for mask,vector in enumerate(inverse3_anf()):\n        if not vector:continue\n        ix=list(bits(mask))\n        if len(ix)==1:\n            for j in bits(vector):b.cx(x[ix[0]],z[j])\n        elif len(ix)==2:rankone_xor(b,x[ix[0]],x[ix[1]],z,vector)\n        else:raise ValueError((mask,vector))\n\ndef inverse3_noscratch(b,x,target=None):\n    """Freshcleantargetinverse, or XORintogiventarget,3T/0privatequbits."""\n    if target is None:target=b.alloc_qubits(3)\n    inverse3_xor_noscratch(b,x,target);return target\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/noscratch/phase39.py': '"""Inverse GF4096 phase with twelve private qubits, 39 Toffolis.\n\nOnly imports core modules; this plugin never patches or edits them.\n"""\ntry:\n    from .kernels import *\nexcept ImportError:\n    from kernels import *\nfrom lowwidth import classical_transpose\n\n\ndef norm_bit_xor(b,x,z,K,bit=0):\n    """XOR one GF8 norm coordinate into arbitrary single-qubit z, 3T."""\n    assert len(x)==6\n    a,c=x[:3],x[3:];f=field(3)\n    for j,v in enumerate(square_columns(3)):\n        if v>>bit&1:b.cx(a[j],z)\n        if f.mul(K,v)>>bit&1:b.cx(c[j],z)\n    for mask,vector in tensor(3):\n        if not(vector>>bit&1):continue\n        ix=list(bits(mask));p=ix[0]\n        for j in ix[1:]:b.cx(a[j],a[p]);b.cx(c[j],c[p])\n        b.ccx(a[p],c[p],z)\n        for j in reversed(ix[1:]):b.cx(c[j],c[p]);b.cx(a[j],a[p])\n\n\ndef phase_inverse39(b,x,mask):\n    """Apply (-1)^dot(mask,inverse(x)); inv(0)=0, x unchanged.\n\n    E[0] is HMR-erased and its quadratic norm phase repaid before the\n    mask-parity qubit is allocated. Once that qubit is freed, E[0] is\n    reconstructed with 3 CCX, enabling the ordinary inverse erasure.\n    Peak D6 + E2 + w3 + parity1 = 12 private quantum wires.\n    """\n    assert len(x)==len(mask)==12\n    to12,frm12,K12=tower(12);linear_inplace(b,x,to12)\n    mt=classical_transpose(b,mask,frm12)\n    D=b.alloc_qubits(6);ar._norm_xor(b,x,D,K12)\n    to6,frm6,K6=tower(6);linear_inplace(b,D,to6)\n    E=b.alloc_qubits(3);ar._norm_xor(b,D,E,K6)\n    w=inverse3_noscratch(b,E)\n    em=b.alloc_bits(3);b.bit_store(em,False);b.hmr(E[:1],em[:1])\n    ar._phase_norm(b,D,em,K6);b.free(em);b.free(E[:1])\n    anc=b.alloc_qubits(1)[0];f6=field(6)\n    def coefficient(h):\n        for j in range(6):\n            for r in bits(f6.mul(h,1<<j)):\n                b.ccx(mt[r],x[j],anc)\n                b.ccx(mt[r],x[j+6],anc)\n                b.ccx(mt[r+6],x[j+6],anc)\n    for segment in (0,1):\n        if segment==0:b.cx(D[3:],D[:3])\n        controls=D[:3] if segment==0 else D[3:]\n        for m,v in tensor(3):\n            h=apply(frm6,v<<(3*segment));ix=list(bits(m));p=ix[0]\n            coefficient(h)\n            for j in ix[1:]:b.cx(controls[j],controls[p]);b.cx(w[j],w[p])\n            b.ccz(controls[p],w[p],anc)\n            for j in reversed(ix[1:]):b.cx(w[j],w[p]);b.cx(controls[j],controls[p])\n            coefficient(h)\n        if segment==0:b.cx(D[3:],D[:3])\n    b.free(anc)\n    e0=b.alloc_qubits(1);norm_bit_xor(b,D,e0[0],K6)\n    E=[e0[0],E[1],E[2]]\n    erase_inverse(b,E,w);b.free(w)\n    ar._erase_norm(b,D,E,K6);b.free(E);linear_inplace(b,D,frm6)\n    ar._erase_norm(b,x,D,K12);b.free(D)\n    linear_inplace(b,x,frm12);b.free(mt)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/noscratch/schedules.py': '"""Reference compositions of the plugins; no changes to arithmetic modules."""\ntry:\n    from .phase39 import *\nexcept ImportError:\n    from phase39 import *\nfrom division import phase_product24\n\n# Shared by privately rebound copies, which retain this empty-list object.\nSTREAM18_TRACE=[]\n\n\ndef norm_xor_noscratch(b,x,z,K):\n    k=len(z);f=field(k)\n    linear_xor(b,x[:k],z,square_columns(k))\n    linear_xor(b,x[k:],z,[f.mul(K,v)for v in square_columns(k)])\n    mul_xor_noscratch(b,x[:k],x[k:],z)\n\n\ndef inverse_stream_noscratch(b,x):\n    """Streamed inv0, with width-safe 18T GF64 leaves (else 21T)."""\n    n=len(x)\n    if n==6:\n        live=b.num_allocated_qubits\n        selected=live+12<=889\n        stage=getattr(b,\'_stage\',None)\n        paths=getattr(b,\'_mark_paths\',None)\n        STREAM18_TRACE.append(dict(live=live,selected=selected,\n                                  stage=list(paths[stage]) if paths is not None else []))\n        if selected:\n            # Initialized inversion is phase exact. Only this measurement\'s\n            # character of the ORIGINAL x is returned to our caller.\n            from exact_subs import inverse6_init18\n            y=inverse6_init18(b,x)\n            mask=b.alloc_bits(6)\n            b.hmr(x,mask);b.swap(x,y);b.free(y)\n            return mask\n    if n==3:\n        y=inverse3_noscratch(b,x);mask=b.alloc_bits(n)\n        b.hmr(x,mask);b.swap(x,y);b.free(y);return mask\n    k=n//2;to,frm,K=tower(n);linear_inplace(b,x,to)\n    N=b.alloc_qubits(k);norm_xor_noscratch(b,x,N,K)\n    nm=inverse_stream_noscratch(b,N);ar._phase_norm(b,x,nm,K);b.free(nm)\n    outb=b.alloc_qubits(k);mul_xor_noscratch(b,x[k:],N,outb)\n    raw=b.alloc_bits(n);b.hmr(x[k:],raw[k:])\n    mul_xor_noscratch(b,x[:k],N,x[k:]);b.cx(outb,x[k:]);b.hmr(x[:k],raw[:k])\n    b.swap(x[:k],x[k:]);b.swap(x[k:],outb);b.free(outb)\n    ar._erase_norm(b,x,N,K);b.free(N);linear_inplace(b,x,frm)\n    mask=classical_transpose(b,raw,to);b.free(raw);return mask\n\n\ndef multiply_forward_noscratch(b,x,y,K=None):\n    z=b.alloc_qubits(24);mul24_xor_noscratch(b,x,y,z,K)\n    m=b.alloc_bits(24);b.hmr(x,m);b.swap(x,z);b.free(z);return m\n\n\ndef divide_lowwidth378(b,x,y,K=None):\n    """378T/24 scratch, promise x=0 when y=0; preserves y."""\n    if K is None:K=extension_columns()[2]\n    b.cx(y[12:],y[:12]);oldmask=multiply_forward_noscratch(b,x,y,K);b.cx(y[12:],y[:12])\n    u=b.alloc_qubits(12);norm_xor_noscratch(b,y,u,K)\n    nm=inverse_stream_noscratch(b,u);ar._phase_norm(b,y,nm,K);b.free(nm)\n    masks=[]\n    for offset in(0,12):\n        half=x[offset:offset+12];out=b.alloc_qubits(12);mul_xor_noscratch(b,half,u,out)\n        m=b.alloc_bits(12);b.hmr(half,m);masks.append(m);b.swap(half,out);b.free(out)\n    um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,y,N,K)\n    for offset,m in zip((0,12),masks):phase_product(b,x[offset:offset+12],N,m);b.free(m)\n    phase_inverse39(b,N,um);b.free(um)\n    ar._erase_norm(b,y,N,K);b.free(N)\n    phase_product24(b,x,y,oldmask,K);b.free(oldmask)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/partial_gauge/prefix.py': '"""Closed local q-prefix step with the historical swap zero chart.\n\nInputs (b,p,q) -> (q*b+p,p,q) when q!=0, or (p,b,q) whenq=0.\nForward65T, reverse152T using divide117; pair217T,29private peak.\nThe reverse is exact for all (b,p,q), not only adjacent-nonzero continuants.\n"""\ntry:\n    from .scalar import *\nexcept ImportError:\n    from scalar import *\n\ndef zero_hold(b,x):\n    b.x(x);lad=b.alloc_qubits(len(x)-1);b.init_and(x[0],x[1],lad[0])\n    for j in range(2,len(x)):b.init_and(lad[j-2],x[j],lad[j-1])\n    b.x(x);return lad\n\ndef zero_free(b,x,lad):\n    b.x(x)\n    for j in reversed(range(2,len(x))):b.del_and(lad[j-2],x[j],lad[j-1])\n    b.del_and(x[0],x[1],lad[0]);b.free(lad);b.x(x)\n\ndef forward(b,x,p,q,virtual_zero=None,after_product=None):\n    out=b.alloc_qubits(12);mul_xor0(b,x,q,out);b.cx(p,out)\n    if virtual_zero is not None:b.ccx(virtual_zero,p,out)\n    lad=zero_hold(b,q);b.cswap(lad[-1],x,p)\n    m=b.alloc_bits(12);b.hmr(x,m);b.swap(x,out);b.free(out)\n    if after_product is not None:after_product()\n    zero_free(b,q,lad);return m\n\ndef reverse(b,x,p,q,m):\n    lad=zero_hold(b,q);f=lad[-1];b.cswap(f,x,p);b.cx(p,x);b.ccx(f,p,x)\n    b.cx(f,q[0]);divide117(b,x,q);b.cx(f,q[0])\n    b.cz(x,m)\n    for j in range(12):b.ccz(f,x[j],m[j]);b.ccz(f,p[j],m[j])\n    b.free(m);zero_free(b,q,lad)\n\ndef zero_compact(b,x):\n    lad=zero_hold(b,x);b.x(x)\n    for j in reversed(range(2,len(x)-1)):b.del_and(lad[j-2],x[j],lad[j-1]);b.free(lad[j-1])\n    b.del_and(x[0],x[1],lad[0]);b.free(lad[0]);b.x(x)\n    return lad[-1]\n\ndef zero_compact_free(b,x,f):\n    b.x(x);lad=b.alloc_qubits(len(x)-2);b.init_and(x[0],x[1],lad[0])\n    for j in range(2,len(x)-1):b.init_and(lad[j-2],x[j],lad[j-1])\n    b.del_and(lad[-1],x[-1],f);b.free(f)\n    for j in reversed(range(2,len(x)-1)):b.del_and(lad[j-2],x[j],lad[j-1])\n    b.del_and(x[0],x[1],lad[0]);b.free(lad);b.x(x)\n\ndef reverse_compact(b,x,p,q,m,virtual_zero=None):\n    """162T reverse,19private; paired with forward gives227T/23private."""\n    f=zero_compact(b,q);b.cswap(f,x,p);b.cx(p,x);b.ccx(f,p,x)\n    if virtual_zero is not None:b.ccx(virtual_zero,p,x)\n    b.cx(f,q[0]);divide117(b,x,q);b.cx(f,q[0]);b.cz(x,m)\n    for j in range(12):b.ccz(f,x[j],m[j]);b.ccz(f,p[j],m[j])\n    b.free(m);zero_compact_free(b,q,f)\n\ndef boundary_forward(b,x,q,F,after_product=None):\n    """i=1 special chart; F=zero(original q1),66T,12private."""\n    b.cx(F,q[0]);out=b.alloc_qubits(12);mul_xor0(b,x,q,out)\n    m=b.alloc_bits(12);b.hmr(x,m);b.swap(x,out);b.free(out);b.cx(F,q[0]);b.x(x[0])\n    if after_product is not None:after_product()\n    b.ccx(F,x,q);b.cx(F,q[0]);b.ccx(F,q,x)\n    return m\n\ndef boundary_reverse(b,x,q,F,m):\n    """129T,18private; exact inverse, closes boundary_forward\'s mask."""\n    b.cswap(F,q,x);b.cx(F,q[0]);b.x(x[0]);b.cx(F,x[0])\n    b.cx(F,q[0]);divide117(b,x,q);b.cx(F,q[0]);b.cz(x,m);b.free(m)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/partial_gauge/scalar.py': '"""Retained-norm GF12 division:117T,18 private, complete local HMR phase."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT/\'exp/arithmetic\'))\nfrom field import *\nimport field as ar\nfrom lowwidth import phase_inverse_generic6\n\ndef divide117(b,x,y):\n    """x/=y, y preserved; y=0 requires x=0. 117T/18 private qubits."""\n    f=field(12);to,frm,K=tower(12);conj=[f.frobenius(1<<i,6) for i in range(12)]\n    linear_inplace(b,y,conj);out=b.alloc_qubits(12);mul_xor0(b,x,y,out)\n    old=b.alloc_bits(12);b.hmr(x,old);b.swap(x,out);b.free(out);linear_inplace(b,y,conj)\n    N=b.alloc_qubits(6);linear_inplace(b,y,to);ar._norm_xor(b,y,N,K);linear_inplace(b,y,frm)\n    u=inverse(b,N);linear_inplace(b,x,to)\n    for off in(0,6):\n        half=x[off:off+6];out=b.alloc_qubits(6);mul_xor0(b,half,u,out)\n        m=b.alloc_bits(6);b.hmr(half,m);b.swap(half,out);b.free(out);phase_product(b,half,N,m);b.free(m)\n    um=b.alloc_bits(6);b.hmr(u,um);b.free(u);phase_inverse_generic6(b,N,um);b.free(um)\n    linear_inplace(b,y,to);ar._erase_norm(b,y,N,K);linear_inplace(b,y,frm);b.free(N)\n    linear_inplace(b,x,frm);phase_product(b,x,y,old);b.free(old)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/partial_gauge/synthesis_noflag.py': '"""Partial-q-gauge synthesis. Same public A,B,transcript API as decoder.\n\n70 input field words plus one rolling d field word and one boundary flag.\nOnly lower A/B coefficients are stored; implicit high coefficients are aliases\nof earlier q-prefix slots. extra[0] is the physical A[h] zero on valid heads.\nNo edits to the live decoder; local measurement transcript is fully closed.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/\'exp/decoder\'))\nfrom common import *\nfrom extract import constants\nfrom exp.decoder import synthesis as base\nfrom exp.partial_gauge import prefix as pg\nfrom exp.partial_gauge.scalar import divide117\nfrom exp.partial_gauge.tail import divide_safe138,multiply_safe165,controlled_linear_xor\nfrom exp.mul99.schedules import divide_zero_bridge414\nimport division as dv\nck=base.ck\n\ndef xword(b,w,v):\n    for j in ar.bits(v):b.x(w[j])\n\ndef add_alias(b,src,targets,coeffs,virtual_zero=None,constant=0):\n    if src is None:\n        for z,c in zip(targets,coeffs):xword(b,z,mul(constant,c))\n    else:\n        for z,c in zip(targets,coeffs):cx(b,src,z,c)\n        if virtual_zero is not None:controlled_linear_xor(b,virtual_zero,src,targets,coeffs)\n\ndef eval_implicit(b,A,qi,c,reverse=False):\n    if len(A)==1:\n        assert qi is not None\n        return base.expose_eval(b,A+[qi],c,reverse)\n    v=base.evaluation_coefficients(len(A)+1,c)[-1]\n    if reverse:\n        add_alias(b,qi,A[:2],[v&4095,v>>12],constant=1)\n        return base.expose_eval(b,A,c,True)\n    out=base.expose_eval(b,A,c)\n    add_alias(b,qi,A[:2],[v&4095,v>>12],constant=1);return out\n\ndef rank_affine(b,head,ac,words,bc,target,tc,constant=0):\n    if not any(tc):return\n    x,ap=ck.expose(b,head,ac);y,bp=ck.expose(b,words,bc)\n    xword(b,y,constant)\n    p=next(i for i,c in enumerate(tc) if c);v=tc[p];vi=inv(v)\n    for i,c in enumerate(tc):\n        if i!=p:cx(b,target[p],target[i],mul(c,vi))\n    scale(b,target[p],vi);ar.mul_xor0(b,x,y,target[p]);scale(b,target[p],v)\n    for i in reversed(range(len(tc))):\n        if i!=p:cx(b,target[p],target[i],mul(tc[i],vi))\n    xword(b,y,constant);ck.unexpose(b,words,bc,bp);ck.unexpose(b,head,ac,ap)\n\ndef projected_product(b,head,A,qi,target,kap):\n    k=len(A)-1;n=k+3;V,W=base.vandermonde(n)\n    words=A+([qi] if qi is not None else [])\n    for t in range(n):\n        tc=[W[j][t] for j in range(k+2)];top=W[k+2][t]\n        tc[k]^=mul(kap,top);tc[k+1]^=top\n        rank_affine(b,head,[1,t],words,V[t][:len(words)],target,tc,V[t][k+1] if qi is None else 0)\n\ndef truncated_quotient(b,head,A,qi,B,kap):\n    k=len(B)\n    if not k:return\n    words=A[1:]+([qi] if qi is not None else [])\n    for ac,bc,tc in reversed(base.quotient_terms(k+2,kap)):\n        rank_affine(b,head,ac,words,bc[:len(words)],B,tc[:k],bc[k] if qi is None else 0)\n\ndef prefix_encode(b,heads,i,F,d=None):\n    a,x=heads[i]\n    if not i:\n        if d is not None:b.cx(a,d);b.cx(x,d)\n        return None\n    q=heads[i-1][1]\n    def after():\n        ar.linear_inplace(b,q,ar.square_columns(12));ar.mul_xor0(b,a,q,d)\n        ar.linear_inplace(b,q,[power(1<<j,2048) for j in range(12)]);ar.mul_xor0(b,x,q,d)\n    callback=after if d is not None else None\n    if i==1:return pg.boundary_forward(b,x,q,F,callback)\n    return pg.forward(b,x,heads[i-2][1],q,F if i==2 else None,callback)\n\ndef prefix_decode(b,heads,i,F,m,d=None):\n    a,x=heads[i]\n    if not i:\n        if d is not None:b.cx(a,d);b.cx(x,d)\n        return\n    q=heads[i-1][1]\n    if d is not None:\n        ar.mul_xor0(b,x,q,d)\n        if i==1:b.ccx(F,q,d)\n    if i==1:pg.boundary_reverse(b,x,q,F,m)\n    else:pg.reverse_compact(b,x,heads[i-2][1],q,m,F if i==2 else None)\n    if d is not None:\n        ar.linear_inplace(b,q,ar.square_columns(12));ar.mul_xor0(b,a,q,d)\n        ar.linear_inplace(b,q,[power(1<<j,2048) for j in range(12)])\n\ndef p_alias(heads,i,F):\n    return (None,None,0) if i==0 else (None,None,1) if i==1 else (heads[i-2][1],F if i==2 else None,0)\n\ndef absorb(b,head,A,B,qi,p,virtual,pconst,c,kap,reverse=False,transcript=None):\n    k=len(B);assert len(A)==k+1;out=head+B\n    if not reverse:\n        value=eval_implicit(b,A,qi,c);flag=dv.zero_flag(b,value);eval_implicit(b,A,qi,c,True)\n        b.cx(flag,A[0][0]);spare=[b.alloc_qubits(12),b.alloc_qubits(12)];target=spare+B\n        linear_words(b,target,base.embedding_matrix(k+2,kap));projected_product(b,head,A,qi,target,kap)\n        b.cx(flag,A[0][0]);cs=[0]*(k+2);cs[k]=kap;cs[k+1]=1\n        add_alias(b,p,target,cs,virtual,pconst)\n        hm=b.alloc_bits(24);b.hmr(combine(head),hm)\n        for h,z in zip(head,spare):b.swap(h,z);b.free(z)\n        fm=b.alloc_bits(1);b.hmr(flag,fm[0]);b.free(flag)\n        return out,A,(hm,fm)\n    linear_words(b,out,base.embedding_matrix(k+2,kap),True)\n    quo,rem=divmodp([0]*(k+2)+[1],[kap,1,1]);coeffs=[rem[j] if j<len(rem) else 0 for j in range(2)]+[quo[j] if j<len(quo) else 0 for j in range(k)]\n    add_alias(b,p,out,coeffs,virtual,pconst)\n    value=eval_implicit(b,A,qi,c);cx(b,head[1],head[0],c);hm,fm=transcript\n    divide_zero_bridge414(b,combine(head),value,K=K,extra_zero_mask=fm[0]);b.free(fm)\n    cx(b,head[1],head[0],c);b.cz(combine(head),hm);b.free(hm)\n    eval_implicit(b,A,qi,c,True);truncated_quotient(b,head,A,qi,B,kap)\n    return out,A,transcript\n\ndef synthesize(b,words,keep_leading=True):\n    h=len(words)//2;assert h>=3 and h%2==1\n    cs,ks,_,_=constants(h);heads=[words[2*i:2*i+2] for i in range(h)]\n    d=b.alloc_qubits(12);F=pg.zero_compact(b,heads[0][1]);masks=[]\n    for i in range(h):masks.append(prefix_encode(b,heads,i,F,d))\n    divide_safe138(b,d,heads[-1][1]);A=[d];B=[];history=[]\n    for i in reversed(range(h)):\n        prefix_decode(b,heads,i,F,masks[i]);qi=heads[i-1][1] if i else None;p,virtual,pc=p_alias(heads,i,F)\n        oldA,oldB=A,B;A,B,m=absorb(b,heads[i],A,B,qi,p,virtual,pc,cs[i],ks[i]);history.append((i,oldA,oldB,qi,p,virtual,pc,cs[i],ks[i],m))\n    extra=[A[h]];freed=[] if keep_leading else extra\n    for w in freed:b.free(w)\n    fmask=b.alloc_bits(1);b.hmr(F,fmask[0]);b.free(F)\n    t=dict(fmask=fmask,kind=\'partial_q_gauge_noflag\',h=h,heads=heads,d=d,F=F,history=history,A=A,B=B,extra=extra,freed=freed,keep_leading=keep_leading)\n    return A[:h],B,t\n\ndef unsynthesize(b,t):\n    if t[\'freed\']:reclaim(b,t[\'freed\'])\n    heads=t[\'heads\'];F=t[\'F\'];masks=[None]*t[\'h\']\n    for i,A,B,qi,p,virtual,pc,c,kap,m in reversed(t[\'history\']):\n        absorb(b,heads[i],A,B,qi,p,virtual,pc,c,kap,True,m);masks[i]=prefix_encode(b,heads,i,F)\n        if i==0:\n            reclaim(b,[[F]]);lad=base.zero_hold(b,heads[0][1]);b.cx(lad[-1],F);base.zero_free(b,heads[0][1],lad)\n            b.cz(F,t[\'fmask\'][0]);b.free(t[\'fmask\'])\n    multiply_safe165(b,t[\'d\'],heads[-1][1])\n    for i in reversed(range(t[\'h\'])):prefix_decode(b,heads,i,F,masks[i],t[\'d\'])\n    pg.zero_compact_free(b,heads[0][1],F);b.free(t[\'d\'])\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/partial_gauge/tail.py': '"""Closed valid-tail scale pair; q=0 requires d=0 throughout."""\ntry:\n    from .scalar import *\nexcept ImportError:\n    from scalar import *\nfrom lowwidth import phase_inverse36\n\ndef multiply_closed144(b,x,q):\n    """x*=q, 144T/19private, q0 requires x0; all HMR phases closed."""\n    out=b.alloc_qubits(12);mul_xor0(b,x,q,out);m=b.alloc_bits(12);b.hmr(x,m);b.swap(x,out);b.free(out)\n    iq=inverse(b,q);phase_product(b,x,iq,m);b.free(m)\n    im=b.alloc_bits(12);b.hmr(iq,im);b.free(iq);phase_inverse36(b,q,im);b.free(im)\n\ndef divide_safe138(b,x,q):\n    """All-input x/=q when q!=0, identity whenq0;138T/20private."""\n    try:from .prefix import zero_compact,zero_compact_free\n    except ImportError:from prefix import zero_compact,zero_compact_free\n    f=zero_compact(b,q);b.cx(f,q[0]);divide117(b,x,q);b.cx(f,q[0]);zero_compact_free(b,q,f)\n\ndef multiply_safe165(b,x,q):\n    """Inverse all-input tail map,165T/20private, no deferred phase."""\n    try:from .prefix import zero_compact,zero_compact_free\n    except ImportError:from prefix import zero_compact,zero_compact_free\n    f=zero_compact(b,q);b.cx(f,q[0]);multiply_closed144(b,x,q);b.cx(f,q[0]);zero_compact_free(b,q,f)\n\ndef controlled_linear_xor(b,F,x,targets,coeffs):\n    """targets[j]^=F*coeffs[j]*x,12T,0scratch; GF12 coefficients classical."""\n    f=field(12);p=next(i for i,v in enumerate(coeffs) if v);v=coeffs[p];vi=f.invert(v)\n    for i,c in enumerate(coeffs):\n        if i!=p:linear_xor(b,targets[p],targets[i],[f.mul(f.mul(c,vi),1<<j) for j in range(12)])\n    linear_inplace(b,targets[p],[f.mul(vi,1<<j) for j in range(12)])\n    b.ccx(F,x,targets[p]);linear_inplace(b,targets[p],[f.mul(v,1<<j) for j in range(12)])\n    for i in reversed(range(len(coeffs))):\n        if i!=p:linear_xor(b,targets[p],targets[i],[f.mul(f.mul(coeffs[i],vi),1<<j) for j in range(12)])\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/phase24_tradeoff/kernel.py': '"""Inverse24 classical-mask phase tradeoff; frozen jet source is read-only."""\nfrom exp.jet24.kernel import *\nfrom exp.mul99.phase36 import phase_classical_parity\n\n\ndef rank(rows):\n    p={}\n    for v in rows:\n        while v:\n            j=v.bit_length()-1\n            if j in p:v^=p[j]\n            else:p[j]=v;break\n    return len(p)\n\n\ndef cols_from_rows(rows):return[sum(((r>>j)&1)<<i for i,r in enumerate(rows))for j in range(len(rows))]\n\n\ndef layout():\n    rows=[]\n    for l,r in LAY[\'gates\']:\n        for v in(l>>1,r>>1):\n            if rank(rows+[v])>len(rows):rows.append(v)\n    assert len(rows)==12\n    pivot=next(j for j in range(6)if rank(rows+[1<<j])==13);rows.append(1<<pivot);cols=cols_from_rows(rows);inv=inverse_columns(cols)\n    def form(e):return(e&1)|sum((((((e>>1)&65535)&inv[j]).bit_count()&1)<<(j+1))for j in range(13))\n    outputs=[(e&~65535)|form(e&65535)for e in LAY[\'outputs\']];gates=[tuple(form(e)for e in g)for g in LAY[\'gates\']]\n    assert all(not(e>>13&1)for g in gates for e in g)\n    v=sum(((e>>13)&1)<<j for j,e in enumerate(outputs));assert rank([1,2,v])==3\n    rest=[]\n    for j in range(2,6):\n        if rank([1,2,v]+rest+[1<<j])>3+len(rest):rest.append(1<<j)\n    B=[1,2]+rest+[v];assert len(B)==6;U=inverse_columns(B)\n    def outputmix(values):return[sum(0 for _ in[])for j in[]]\n    newoutputs=[]\n    for j in range(6):\n        e=0\n        for i in range(6):\n            if U[i]>>j&1:e^=outputs[i]\n        newoutputs.append(e)\n    assert[(e>>13)&1 for e in newoutputs]==[0,0,0,0,0,1]\n    return dict(pivot=pivot,source_columns=cols,source_inverse=inv,output_B=B,outputs=newoutputs,gates=gates)\n\nL30=layout()\n\n\ndef norm_bit_xor(b,N,target,K12,to6,j):\n    """One tower6 component ofnorm12(N),6CCX andnoquantumscratch.\n    N is12towercoordinates; outputcanonical6isthenmappedbyto6.\n    """\n    f=field(6)\n    for i in range(6):\n        if apply(to6,f.square(1<<i))>>j&1:b.cx(N[i],target)\n        if apply(to6,f.mul(K12,f.square(1<<i)))>>j&1:b.cx(N[i+6],target)\n        row=sum(((apply(to6,f.mul(1<<i,1<<k))>>j)&1)<<k for k in range(6));ix=list(bits(row));p=ix[0]\n        for k in ix[1:]:b.cx(N[6+k],N[6+p])\n        b.ccx(N[i],N[6+p],target)\n        for k in reversed(ix[1:]):b.cx(N[6+k],N[6+p])\n\n\ndef virtual_mask_phase(b,x24,mask24,N12,out6,frm12):\n    """30 executedCCZ,0scratch; L12 virtual inx24 andclassicalmask."""\n    f=field(12)\n    for segment in(0,1):\n        if segment==0:b.cx(N12[6:],N12[:6])\n        xx=N12[:6]if segment==0 else N12[6:]\n        for m,v in tensor(6):\n            h=apply(frm12,v<<(6*segment));co=b.alloc_bits(24);b.bit_store(co,False)\n            for i in range(12):\n                for r in bits(f.mul(h,1<<i)):\n                    b.cx(mask24[r],co[i]);b.cx(mask24[r],co[i+12]);b.cx(mask24[r+12],co[i+12])\n            ix=list(bits(m));p=ix[0]\n            for k in ix[1:]:b.cx(xx[k],xx[p]);b.cx(out6[k],out6[p])\n            phase_classical_parity(b,xx[p],out6[p],x24,co)\n            for k in reversed(ix[1:]):b.cx(out6[k],out6[p]);b.cx(xx[k],xx[p])\n            b.free(co)\n        if segment==0:b.cx(N12[6:],N12[:6])\n\n\ndef phase_inverse24_canonical118(b,x,mask,zero_mask=None):\n    """<=118 executedT/30private, inverse0phase; inputcanonical24."""\n    to24,frm24,K24=tower(24);linear_inplace(b,x,to24);mt=classical_transpose(b,mask,frm24)\n    N=b.alloc_qubits(12);norm_xor_noscratch(b,x,N,K24);to12,frm12,K12=tower(12);linear_inplace(b,N,to12)\n    D=b.alloc_qubits(6);norm_xor_noscratch(b,N,D,K12);to6,frm6,K6=tower(6);linear_inplace(b,D,to6);g=b.alloc_qubits(6)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,D,l,r)as(p,q):b.init_and(p,q,g[i])\n    prefix=list(D)+list(g);forms=ref.norm_forms(K6);w=ref.inverse3_virtual(b,prefix,forms)\n    linear_inplace(b,g,LAY[\'raw_columns\']);R,E=g[:4],g[4:];source=list(D)+list(R)+list(w);linear_inplace(b,source,L30[\'source_columns\']);C=source[:12];z=list(E)+list(b.alloc_qubits(3))+source[12:]\n    for j,e in enumerate(L30[\'outputs\']):ref.form_xor(b,C,e&8191,z[j])\n    for i,(l,r)in enumerate(L30[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(L30[\'outputs\']))\n        with ref.expose_pair(b,C,l,r)as(p,q):rankone_xor(b,p,q,z,v)\n    linear_inplace(b,z,L30[\'output_B\']);linear_inplace(b,z,LAY[\'output_inverse_columns\']);linear_inplace(b,z,frm6)\n    virtual_mask_phase(b,x,mt,N,z,frm12);b.free(mt)\n    # OutputHMR first frees6 before restoring the one originalD coordinate.\n    zm=b.alloc_bits(6);b.hmr(z,zm);b.free(z);ym=classical_transpose(b,zm,frm6);b.free(zm);mm=classical_transpose(b,ym,LAY[\'output_inverse_columns\']);b.free(ym)\n    dj=b.alloc_qubits(1)[0];norm_bit_xor(b,N,dj,K12,to6,L30[\'pivot\']);S=list(C)+[dj];linear_inplace(b,S,L30[\'source_inverse\']);DD=S[:6];RR=S[6:10];ww=S[10:]\n    phase_outputs_classical(b,S,LAY[\'outputs\'],LAY[\'gates\'],mm)\n    for i,(l,r)in enumerate(basic.raw_forms19()):\n        with ref.expose_pair(b,DD,l,r)as(p,q):\n            for j,row in enumerate(LAY[\'raw_rows\'][4:]):\n                if row>>i&1:b.ccz(p,q,mm[j])\n    b.free(mm)\n    EE=b.alloc_qubits(3);norm_xor_noscratch(b,DD,EE,K6)\n    if zero_mask is not None:\n        b.z(zero_mask)\n        for i in range(3):\n            for j in range(3):\n                if field(3).mul(1<<i,1<<j)&1:b.ccz(EE[i],ww[j],zero_mask)\n    erase_inverse(b,EE,ww);b.free(ww);ar._erase_norm(b,DD,EE,K6);b.free(EE);zero1.erase_raw_cache(b,DD,RR,LAY[\'raw_rows\'][:4]);linear_inplace(b,DD,frm6)\n    dm=b.alloc_bits(6);b.hmr(DD,dm);b.free(DD);ar._phase_norm(b,N,dm,K12);b.free(dm);linear_inplace(b,N,frm12);ar._erase_norm(b,x,N,K24);b.free(N);linear_inplace(b,x,frm24)\n\n\ndef phase_inverse24_extension118(b,x,mask,K=None,zero_mask=None):\n    cols,frm,_=extension_columns(K);linear_inplace(b,x,cols);mt=classical_transpose(b,mask,frm);phase_inverse24_canonical118(b,x,mt,zero_mask);b.free(mt);linear_inplace(b,x,frm)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/phase24_tradeoff/phase6small.py': '"""Classical-mask inverse6 phase:<=6executedT/5scratch; exact64pointSLP.\nClassicalmask-dependent constantfieldscaling maps63nonzero components toone.\n"""\nfrom exp.phase24_tradeoff.kernel import *\nimport json\nP6=json.loads(Path(__file__).with_name(\'phase6_six_search.json\').read_text())[\'candidate\']\n\ndef classical_equal(b,x,value):\n    chain=b.alloc_bits(len(x)+1);b.bit_store(chain,False);b.x(chain[0])\n    for j,q in enumerate(x):\n        if not(value>>j&1):b.x(q)\n        b.push_condition(chain[j]);b.bit_store(chain[j+1],True,control=q);b.pop_condition()\n        if not(value>>j&1):b.x(q)\n    b.free(chain[:-1]);return chain[-1]\n\n\ndef fixed_phase6(b,x,do_inverse=True,zero_mask=None):\n    to,frm,K=tower(6);linear_inplace(b,x,to);ng=len(P6[\'raw_forms\']);g=b.alloc_qubits(ng)\n    for i,(l,r)in enumerate(P6[\'raw_forms\']):\n        with ref.expose_pair(b,x,l,r)as(p,q):b.init_and(p,q,g[i])\n    w=P6[\'w\'];base=[]\n    for j in range(ng):\n        if rank(base+[w,1<<j])==len(base)+2:base.append(1<<j)\n    B=base+[w];assert len(B)==ng;iv=inverse_columns(B);linear_inplace(b,g,iv)\n    def form(v):\n        return(v&63)|sum((((v>>6)&co).bit_count()&1)<<(6+j)for j,co in enumerate(B))\n    p,q=form(P6[\'p\']),form(P6[\'q\']);assert not(((p|q)>>(6+ng-1))&1);source=list(x)+list(g[:ng-1])\n    with ref.expose_pair(b,source,p<<1,q<<1)as(a,c):b.ccx(a,c,g[ng-1])\n    linear_inplace(b,g,B);S=list(x)+list(g)\n    for term in(P6[\'terms\']if do_inverse else[]):\n        if len(term)==1:b.z(S[term[0]])\n        else:b.cz(S[term[0]],S[term[1]])\n    if zero_mask is not None:\n        b.z(zero_mask);b.push_condition(zero_mask)\n        for term in P6[\'zero_terms\']:\n            if len(term)==1:b.z(S[term[0]])\n            else:b.cz(S[term[0]],S[term[1]])\n        b.pop_condition()\n    linear_inplace(b,g,iv);hm=b.alloc_bits(1)[0];b.hmr(g[ng-1],hm)\n    with ref.expose_pair(b,source,p<<1,q<<1)as(a,c):b.ccz(a,c,hm)\n    rows=[sum(((iv[i]>>j)&1)<<i for i in range(ng))for j in range(ng)]\n    for i,(l,r)in enumerate(P6[\'raw_forms\']):\n        if rows[ng-1]>>i&1:\n            with ref.expose_pair(b,x,l,r)as(a,c):b.ccz(a,c,hm)\n    b.free(hm);b.free(g[ng-1]);mm=b.alloc_bits(ng-1);b.hmr(g[:ng-1],mm);b.free(g[:ng-1])\n    for i,(l,r)in enumerate(P6[\'raw_forms\']):\n        with ref.expose_pair(b,x,l,r)as(a,c):\n            for j in range(ng-1):\n                if rows[j]>>i&1:b.ccz(a,c,mm[j])\n    b.free(mm);linear_inplace(b,x,frm)\n\n\ndef mask_factors():\n    f=field(6);out={}\n    for s in range(1,64):\n        si=f.invert(s);m=sum(((f.mul(si,1<<j)>>P6[\'output_bit\'])&1)<<j for j in range(6));assert m not in out;out[m]=s\n    return out\nFACTORS=mask_factors()\n\n\ndef phase_inverse6_classical6(b,x,mask,zero_mask=None):\n    """Exactmask·inv0(x),0→0,<=6executedT/5scratch,arbitraryclassicalmask."""\n    for m,s in sorted(FACTORS.items()):\n        eq=classical_equal(b,mask,m);b.push_condition(eq);co=[field(6).mul(s,1<<j)for j in range(6)];linear_inplace(b,x,co);fixed_phase6(b,x,zero_mask=zero_mask);linear_inplace(b,x,inverse_columns(co));b.pop_condition();b.free(eq)\n\n    if zero_mask is not None:\n        eq=classical_equal(b,mask,0);b.push_condition(eq);b.push_condition(zero_mask);fixed_phase6(b,x,do_inverse=False,zero_mask=zero_mask);b.pop_condition();b.pop_condition();b.free(eq)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/safe_scalar/converter.py': '# Correct residues on coprime degree profiles with gaps at most two.\n# Zero-clipped scalar permutations preserve restoration on unsupported profiles.\n"""Guarded history-free multi-h2 Bezout lookahead, with borrowed monic-root word.\n\nOn proper coprime half-degree charts with drops1or2, parent gap flags are\nrecomputed from the deliberatelyzero child quotient metadata. No persistent\nquantum history register. Crosspass token is classical and phaseclosed after\nreverse around a diagonal oracle. See RESTORATION.md for the all-input\nrestoration argument; larger gaps may still produce wrong decoded phases.\n"""\nimport sys\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/\'convert\'))\nfrom convert import ar,frob\nfrom repair import rotate\nfrom kernels import hankel,convolution_xor\nfrom lowwidth import phase_inverse36\nfrom contextlib import contextmanager\n\ndef scale(b,x,a,ai,divide=True):\n    out=b.alloc_qubits(12)\n    ar.mul_xor0(b,x,ai if divide else a,out)\n    ar.erase_product(b,a if divide else ai,out,x)\n    b.swap(x,out);b.free(out)\n\ndef runops(b,ops,a,ai,undo=False):\n    for op in (reversed(ops)if undo else ops):\n        if op[0]==\'s\':scale(b,op[1],a,ai,op[2]^undo)\n        elif op[0]==\'m\':ar.mul_xor0(b,op[1],op[2],op[3])\n        elif op[0]==\'c\':convolution_xor(b,op[1],op[2],op[3],undo)\n        else:raise ValueError(op[0])\n\ndef zero_toggle(b,a,target):\n    # Holdthe11ANDladderuntiltheflagcopy, thenuncomputeallforfree.\n    for q in a:b.x(q)\n    ladder=list(b.alloc_qubits(len(a)-1));b.init_and(a[0],a[1],ladder[0])\n    for i in range(1,len(ladder)):b.init_and(ladder[i-1],a[i+1],ladder[i])\n    b.cx(ladder[-1],target)\n    for i in reversed(range(len(ladder))):\n        if i:b.del_and(ladder[i-1],a[i+1],ladder[i])\n        else:b.del_and(a[0],a[1],ladder[0])\n    b.free(ladder)\n    for q in a:b.x(q)\n\nfrom exp.inverse12_escape.zero1 import inverse_zero,consume,erase,inverse_apply_safe\n\n@contextmanager\ndef q_inverse(b,a,m,reverse,phase):\n    ai,F=inverse_zero(b,a)\n    if reverse:consume(b,ai,F,phase[\'q\'].pop(m))\n    b.cx(F,a[0]);b.cx(F,ai[0]);yield ai;b.cx(F,ai[0]);b.cx(F,a[0])\n    token=erase(b,a,ai,F,not reverse)\n    if not reverse:phase[\'q\'][m]=token\n\ndef inverse_apply(b,a,action,incoming=None,defer=False):\n    return inverse_apply_safe(b,a,action,incoming,defer)\n\ndef pre(b,f,g,payloads,undo=False,incoming=None,defer=False):\n    m=len(f)-1;n=m-1;a=g[-1];q0=f[-2];q1=f[-1]\n    def action(ai):\n        ops=[(\'s\',v[-1],True)for v in payloads]\n        ops +=[(\'m\',g[i],v[-1],v[i])for v in payloads for i in range(n)]\n        ops +=[(\'s\',q1,True),(\'m\',q1,g[-2],q0),(\'s\',q0,True),(\'c\',g[:-1],[q0,q1],f[:-2])]\n        runops(b,ops,a,ai,undo)\n    return inverse_apply(b,a,action,incoming,defer)\n\ndef canonical_permute(b,f,g,F,reverse=False):\n    # [G0,F0,G1,F1,...,G_(m-2),F_(m-2),G_(m-1)] leftcyclic.\n    cyc=[]\n    for i in range(len(g)-1):cyc +=[g[i],f[i]]\n    cyc +=[g[-1]]\n    seq=range(len(cyc)-2,-1,-1) if reverse else range(len(cyc)-1)\n    for i in seq:b.cswap(F,cyc[i],cyc[i+1])\n\ndef masked_product(b,F,x,c,target):\n    u=b.alloc_qubits(12)\n    for j in range(12):b.init_and(F,x[j],u[j])\n    ar.mul_xor0(b,u,c,target)\n    mask=b.alloc_bits(12);b.hmr(u,mask);b.free(u)\n    for j in range(12):b.ccz(F,x[j],mask[j])\n    b.free(mask)\n\ndef fresh_zero_flag(b,a):\n    F=b.alloc_qubits(1)[0];zero_toggle(b,a,F);return F\n\ndef clear_flag(b,a,F):zero_toggle(b,a,F);b.free(F)\n\ndef q_post(b,q,payloads,H,m,parent,reverse=False,phase=None):\n    if parent is None:\n        with q_inverse(b,q,m,reverse,phase) as qi:\n            for v in payloads:scale(b,v[-1],q,qi,not reverse)\n        return\n    tops,p,c=parent;F=fresh_zero_flag(b,q)\n    b.cswap(F,q,p)\n    if not reverse:\n        for v,t in zip(payloads,tops):b.cswap(F,v[-1],t)\n    with q_inverse(b,q,m,reverse,phase) as qi:\n        if not reverse:\n            for v in payloads:scale(b,v[-1],q,qi,True)\n            for v,t in zip(payloads,tops):masked_product(b,F,v[-1],c,t)\n        else:\n            for v,t in zip(payloads,tops):masked_product(b,F,v[-1],c,t)\n            for v in payloads:scale(b,v[-1],q,qi,False)\n    if reverse:\n        for v,t in zip(payloads,tops):b.cswap(F,v[-1],t)\n    b.cswap(F,q,p);clear_flag(b,q,F)\n\ndef post_look(b,f,g,payloads,H,parent,reverse=False,incoming=None,defer=False,phase=None):\n    m=len(f)-1;n=m-1;a=g[-1];q0=f[-2];q1=f[-1]\n    def dots():\n        for v in payloads:\n            for i in range(n):ar.mul_xor0(b,g[i],v[i],v[-1])\n    def last(undo,inc=None,delay=False):\n        def action(ai):\n            ops=[(\'s\',v[-1],True)for v in payloads]\n            ops +=[(\'c\',g[:-1],[q0,q1],f[:-2]),(\'s\',q0,False),(\'m\',q1,g[-2],q0),(\'s\',q1,False)]\n            runops(b,ops,a,ai,undo)\n        return inverse_apply(b,a,action,inc,delay)\n    if not reverse:q_post(b,q1,payloads,H,m,parent,phase=phase);dots();return last(False,incoming,defer)\n    else:\n        carry=last(True,incoming,defer);dots();q_post(b,q1,payloads,H,m,parent,True,phase);return carry\n\ndef base(b,f,g,payloads,H,parent,reverse=False,phase=None):\n    a=g[0];q=f[-1]\n    def first(undo,inc=None,delay=False):\n        def action(ai):runops(b,[(\'s\',v[0],True)for v in payloads]+[(\'s\',q,True)],a,ai,undo)\n        return inverse_apply(b,a,action,inc,delay)\n    def last(undo,inc=None,delay=False):\n        def action(ai):runops(b,[(\'s\',v[0],True)for v in payloads]+[(\'s\',q,False)],a,ai,undo)\n        return inverse_apply(b,a,action,inc,delay)\n    if not reverse:\n        carry=first(False,delay=True);q_post(b,q,payloads,H,1,parent,phase=phase);phase[\'a\'][1]=last(False,carry,True)\n    else:\n        carry=last(True,phase[\'a\'].pop(1),True);q_post(b,q,payloads,H,1,parent,True,phase);first(True,carry)\n\ndef look_solve(b,f,g,payloads,H=None,parent=None,reverse=False,phase=None):\n    m=len(f)-1\n    if m==1:return base(b,f,g,payloads,H,parent,reverse,phase)\n    F=fresh_zero_flag(b,g[-1]);rotate(b,g,F)\n    if not reverse:carry=pre(b,f,g,payloads,defer=True)\n    else:carry=post_look(b,f,g,payloads,H,parent,True,incoming=phase[\'a\'].pop(m),defer=True,phase=phase)\n    canonical_permute(b,f,g,F);clear_flag(b,g[-1],F)\n    childparent=([v[-1]for v in payloads],f[-1],f[-2])\n    look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,childparent,reverse,phase)\n    F=fresh_zero_flag(b,g[-1]);canonical_permute(b,f,g,F,True)\n    if not reverse:phase[\'a\'][m]=post_look(b,f,g,payloads,H,parent,incoming=carry,defer=True,phase=phase)\n    else:pre(b,f,g,payloads,True,incoming=carry)\n    rotate(b,g,F,True);clear_flag(b,g[-1],F)\n\ndef bez_lookahead(b,O,E,payloads,reverse=False,history_scratch12=None,phase=None):\n    def reduce():\n        for i in range(len(O)):ar.mul_xor0(b,E[-1],O[i],E[i])\n    reduce()\n    lead=list(b.alloc_qubits(12))if history_scratch12 is None else list(history_scratch12)\n    assert len(lead)==12\n    b.x(lead[0])\n    look_solve(b,O+[lead],E[:-1],payloads,reverse=reverse,phase=phase)\n    b.x(lead[0])\n    if history_scratch12 is None:b.free(lead)\n    reduce()\n\ndef convert_safe(b,A,B,reverse=False,history_scratch12=None,phase_token=None):\n    """Optional12zero wires borrowed as monic rootleadingword; returnedzero."""\n    phase={\'a\':{},\'q\':{}}if phase_token is None else phase_token\n    assert not reverse or phase_token is not None\n    E=B[::2];O=B[1::2];u=A[1::2];w=A[:34:2];top=A[-1]\n    if reverse:hankel(b,B,A,True)\n    frob(b,B,2048);frob(b,A,2048)\n    if reverse:\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n        for i in range(17):ar.mul_xor0(b,O[i],u[i],top)\n        bez_lookahead(b,O,E,[u,w],True,history_scratch12,phase)\n        for i in range(17):ar.mul_xor0(b,top,O[i],w[i])\n    else:\n        for i in range(17):ar.mul_xor0(b,top,O[i],w[i])\n        bez_lookahead(b,O,E,[u,w],False,history_scratch12,phase)\n        for i in range(17):ar.mul_xor0(b,O[i],u[i],top)\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n    frob(b,A,2);frob(b,B,2)\n    if not reverse:hankel(b,B,A)\n\n    if reverse:assert not phase[\'a\']and not phase[\'q\']\n    return phase if not reverse else None\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/safe_scalar/kernel.py': '"""Zero-aware inverse12: expose zero cheaply at the bottom tower norm."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.inverse_search.inverse20_iso import *\nfrom exp.noscratch.schedules import norm_xor_noscratch\nfrom exp.safe_scalar.phase import phase_inverse36_zero\n\ndef inverse_zero(b,x):\n    """Preserve x, return inverse0(x) and [x=0],67T with clean scratch."""\n    to,frm,K=tower(12);linear_inplace(b,x,to)\n    N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,K)\n    t,D=inverse6_xor20(b,N,keep_norm=True)\n    b.x(D);u=b.alloc_qubits(1)[0];F=b.alloc_qubits(1)[0]\n    b.init_and(D[0],D[1],u);b.init_and(u,D[2],F);b.del_and(D[0],D[1],u);b.free(u);b.x(D)\n    to6,frm6,K6=tower(6);linear_inplace(b,N,to6)\n    ar._erase_norm(b,N,D,K6);b.free(D);linear_inplace(b,N,frm6)\n    ar._erase_norm(b,x,N,K);b.free(N)\n    out=b.alloc_qubits(12);b.cx(x[6:],x[:6])\n    mul_xor_noscratch(b,x[:6],t,out[:6]);mul_xor_noscratch(b,x[6:],t,out[6:]);b.cx(x[6:],x[:6])\n    ar._erase_norm(b,out,t,K);b.free(t)\n    linear_inplace(b,x,frm);linear_inplace(b,out,frm)\n    return out,F\n\ndef consume(b,ai,F,token):\n    if token is not None:\n        m,z=token;b.cz(ai,m);b.cz(F,z[0]);b.free(m);b.free(z)\n\ndef erase(b,a,ai,F,defer):\n    m=b.alloc_bits(12);z=b.alloc_bits(1)\n    b.hmr(ai,m);b.hmr(F,z[0]);b.free(ai);b.free(F)\n    if defer:return m,z\n    phase_inverse36_zero(b,a,m,z[0]);b.free(m);b.free(z)\n\ndef inverse_apply_safe(b,a,action,incoming=None,defer=False):\n    ai,F=inverse_zero(b,a);consume(b,ai,F,incoming)\n    b.cx(F,a[0]);b.cx(F,ai[0]);action(ai);b.cx(F,ai[0]);b.cx(F,a[0])\n    return erase(b,a,ai,F,defer)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/safe_scalar/phase.py': '"""Inverse12 phase with12 scratch and at most36 EXECUTED Toffolis.\n\nThe coefficient mask is classical. A classical first-nonzero pivot exposes\nits parity in an existing input bit, avoiding the13th quantum wire.\n"""\ntry:\n    from exp.mul99.kernels import *\nexcept ImportError:\n    from exp.mul99.kernels import *\nsys.path.insert(0,str(ROOT))\nfrom exp.noscratch.kernels import inverse3_noscratch\nfrom lowwidth import classical_transpose\n\n\ndef phase_classical_parity(b,a,c,x,co):\n    """CCZ(a,c,parity(co*x)), zero quantum scratch, <=1 executed CCZ."""\n    first=b.alloc_bits(len(x));b.bit_store(first,False)\n    seen=b.alloc_bits(1)[0];b.bit_store(seen,False)\n    for j in range(len(x)):\n        b.x(seen);b.push_condition(seen)\n        b.bit_store(first[j],True,control=co[j])\n        b.pop_condition();b.x(seen)\n        b.bit_store(seen,True,control=co[j])\n    for j in range(len(x)):\n        b.push_condition(first[j])\n        for k in range(len(x)):\n            if k!=j:b.ccx(co[k],x[k],x[j])\n        b.ccz(a,c,x[j])\n        for k in reversed(range(len(x))):\n            if k!=j:b.ccx(co[k],x[k],x[j])\n        b.pop_condition()\n    b.free(seen);b.free(first)\n\n\ndef phase_inverse36_zero(b,x,mask,zero_mask):\n    """Phase mask dot inverse0(x): <=36 mean T /12 quantum scratch.\n\n    Static count168T due to12 mutually exclusive classical branches per CCZ.\n    No quantum input controls branch selection; all branches use HMR-mask bits.\n    """\n    assert len(x)==len(mask)==12\n    to12,frm12,K12=tower(12);linear_inplace(b,x,to12)\n    mt=classical_transpose(b,mask,frm12)\n    D=b.alloc_qubits(6);ar._norm_xor(b,x,D,K12)\n    to6,frm6,K6=tower(6);linear_inplace(b,D,to6)\n    E=b.alloc_qubits(3);ar._norm_xor(b,D,E,K6);w=inverse3_noscratch(b,E)\n    b.z(zero_mask);zm=b.alloc_bits(3);b.bit_store(zm,False);b.cx(zero_mask,zm[0])\n    phase_product(b,E,w,zm);b.free(zm)\n    f6=field(6)\n    for segment in(0,1):\n        if segment==0:b.cx(D[3:],D[:3])\n        controls=D[:3] if segment==0 else D[3:]\n        for m,v in tensor(3):\n            h=apply(frm6,v<<(3*segment));co=b.alloc_bits(12);b.bit_store(co,False)\n            for j in range(6):\n                for r in bits(f6.mul(h,1<<j)):\n                    b.cx(mt[r],co[j]);b.cx(mt[r],co[j+6]);b.cx(mt[r+6],co[j+6])\n            ix=list(bits(m));p=ix[0]\n            for j in ix[1:]:b.cx(controls[j],controls[p]);b.cx(w[j],w[p])\n            phase_classical_parity(b,controls[p],w[p],x,co)\n            for j in reversed(ix[1:]):b.cx(w[j],w[p]);b.cx(controls[j],controls[p])\n            b.free(co)\n        if segment==0:b.cx(D[3:],D[:3])\n    erase_inverse(b,E,w);b.free(w)\n    ar._erase_norm(b,D,E,K6);b.free(E);linear_inplace(b,D,frm6)\n    ar._erase_norm(b,x,D,K12);b.free(D);linear_inplace(b,x,frm12);b.free(mt)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/scan/scan.py': '"""Exact45045-Toffoli Chien/residue phase scan,11workspacequbits.\n\nA,B:35GF12 words each, degree<35 numerator and monicdegree35 denominator.\nE:4095classical12-bit masks, indexed beta=alpha**j. Bothbanks and E restored.\nA is already residue-converted; phase is dot(Ej,A(beta)/(beta*P(beta)))\nwhen B(beta)=0, and0otherwise. P is the node polynomial from extract.constants.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2]\nsys.path.insert(0,str(ROOT/\'exp/decoder\'))\nfrom common import *\nfrom extract import constants\n\n\ndef scan(b,A,B,E,*,points=4095,skip_points=(),node_polynomial=None):\n    if len(A)!=35 or len(B)!=35 or len(E)<points:raise ValueError(\'need35+35words and one maskperpoint\')\n    if points!=4095:raise ValueError(\'fullcycle4095points required for bank restoration\')\n    alpha=gf.primitive_element;P=constants()[2]if node_polynomial is None else node_polynomial\n    powers=[power(alpha,k) for k in range(35)]\n    skip_points=frozenset(skip_points)\n    assert all(0<=j<4095 for j in skip_points)\n    beta=1;lead=1;leadstep=power(alpha,35);m=b.alloc_bits(12)\n    active=b.alloc_bits(1)[0]\n    for j in range(4095):\n        # The original coefficientk now holds Ak*beta**k (likewiseB).\n        if j not in skip_points:\n            for k in range(1,35):b.cx(A[k],A[0]);b.cx(B[k],B[0])\n            b.gf2_iadd(lead,target=B[0])\n            pv=0\n            for v in reversed(P):pv=mul(pv,beta)^v\n            c=inv(mul(beta,pv));assert c\n            # Classically apply transpose of multiplicationc to maskEj.\n            b.bit_store(m,False)\n            for t in range(12):\n                for i in ar.bits(mul(c,1<<t)):b.cx(E[j][i],m[t])\n            b.bit_store(active,False)\n            for bit in m:b.bit_store(active,True,control=bit)\n            b.push_condition(active)\n            ladder=zero_hold(b,B[0]);flag=ladder[-1]\n            b.ccz(flag,A[0],m)\n            zero_free(b,B[0],ladder)\n            b.pop_condition()\n            b.gf2_iadd(lead,target=B[0])\n            for k in range(1,35):b.cx(A[k],A[0]);b.cx(B[k],B[0])\n        for k in range(1,35):scale(b,A[k],powers[k]);scale(b,B[k],powers[k])\n        beta=mul(beta,alpha);lead=mul(lead,leadstep)\n    assert beta==lead==1\n    b.free(active)\n    b.free(m)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/tail_alias/divide.py': '"""Zero-aware initialized inverse fits below the product peak:369T safe divide."""\nfrom pathlib import Path\nimport sys\nfrom types import FunctionType\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.fold_joint import divide as prior\nfrom exp.inverse19_joint.kernel import inverse_zero\n\ndef inverse_overwrite_zero66(b,x):\n    """Overwrite inverse0+zero,66T,31 localQ; old input mask deferred."""\n    out,F=inverse_zero(b,x);m=b.alloc_bits(12);b.hmr(x,m);b.swap(x,out);b.free(out)\n    return m,F\n\n_globals=dict(prior.divide_zero371.__globals__,inverse_stream_zero=inverse_overwrite_zero66)\n_fn=prior.divide_zero371\ndivide_zero369=FunctionType(_fn.__code__,_globals,\'divide_zero369\',_fn.__defaults__,_fn.__closure__)\ndivide_zero369.__doc__=\'All-input safe GF24 division: <=369 executed T,37 private. Old initialized inverse+flag replaces streamed68T with66T below37-bit product peak.\'\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/tail_alias/kernel.py': '"""Tail d aliases raw a0; quadratic companion action closes outer fold.\n\nNo additional persistent field word. Inputs and returned A/B use70 words.\nAll maps are total: factors zero are replaced by factor1 consistently.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.fold_joint.kernel import *\nfrom exp.tail_alias.divide import divide_zero369\nfrom types import FunctionType\n# Private binding: imported CRT arithmetic modules are never mutated.\n_absorb=absorb\nabsorb=FunctionType(_absorb.__code__,dict(_absorb.__globals__,divide_zero371=divide_zero369),\n    \'absorb\',_absorb.__defaults__,_absorb.__closure__)\n\ndef expose_quotient(b,C,D):\n    """Top C pair -> quotient l of q*C by monic D, h>=3; involution."""\n    b.cx(C[-1],C[-2]);ar.mul_xor0(b,C[-1],D[-1],C[-2])\n\ndef outer(b,C,D,c,kap,reverse=False,transcript=None,wide=True):\n    """C <- q*C mod (D+[D(gamma)=0]), D monic degree h unchanged.\n\n    D lists its h low coefficients, C its h coefficients. Constant completion\n    leaves quotient l unaffected for h>=3. CRT output is lowerC then top pair.\n    """\n    h=len(C);assert len(D)==h and h>=3;head=C[-2:];low=C[:-2]\n    if not reverse:\n        expose_quotient(b,C,D)\n        value=old.eval_implicit(b,D,None,c);F=dv.zero_flag(b,value)\n        spare=[b.alloc_qubits(12),b.alloc_qubits(12)]\n        b.cx(F,value[0]);cx(b,head[1],head[0],c)\n        mul24_xor99(b,combine(head),value,combine(spare),K)\n        cx(b,head[1],head[0],c);b.cx(F,value[0]);old.eval_implicit(b,D,None,c,True)\n        q_times_low(b,low,kap);b.cx(F,D[0][0]);low_product(b,head,D,low);b.cx(F,D[0][0])\n        R_to_tops(b,spare,low,c)\n        hm=b.alloc_bits(24);b.hmr(combine(head),hm)\n        for w,z in zip(head,spare):b.swap(w,z);b.free(z)\n        fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n        return hm,fm\n    R_to_tops(b,head,low,c,True)\n    value=old.eval_implicit(b,D,None,c);hm,fm=transcript\n    def after(F):\n        cx(b,head[1],head[0],c);old.eval_implicit(b,D,None,c,True)\n        b.cx(F,D[0][0]);low_product(b,head,D,low);b.cx(F,D[0][0]);q_times_low(b,low,kap,True)\n        old.eval_implicit(b,D,None,c);cx(b,head[1],head[0],c)\n    if wide:divide_zero369(b,combine(head),value,K=K,extra_zero_mask=fm[0],after=after)\n    else:divide414_callback(b,combine(head),value,K,fm[0],after)\n    b.free(fm);cx(b,head[1],head[0],c);b.cz(combine(head),hm);b.free(hm)\n    old.eval_implicit(b,D,None,c,True);expose_quotient(b,C,D)\n    return transcript\n\ndef synthesize(b,words,wide=True):\n    h=len(words)//2;assert h>=3 and h%2==1\n    cs,ks,_,_=old.constants(h);heads=[words[2*i:2*i+2]for i in range(h)]\n    d=heads[0][0];F=pg.zero_compact(b,heads[0][1]);b.cx(heads[0][1],d);masks=[None]*h\n    for i in range(1,h):masks[i]=old.prefix_encode(b,heads,i,F,d)\n    old.divide_safe138(b,d,heads[-1][1]);A=[d];B=[];hist=[]\n    for i in reversed(range(1,h)):\n        old.prefix_decode(b,heads,i,F,masks[i]);qi=heads[i-1][1];p,v,pc=old.p_alias(heads,i,F)\n        oldA,oldB=A,B;A,B,t=absorb(b,heads[i],A,B,qi,p,v,pc,cs[i],ks[i],wide=wide)\n        hist.append((i,oldA,oldB,qi,p,v,pc,cs[i],ks[i],t))\n    C=B+[heads[0][1]];ot=outer(b,C,A,cs[0],ks[0],wide=wide)\n    fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n    return C,A,dict(h=h,heads=heads,d=d,F=F,history=hist,A=C,B=A,outer=ot,wide=wide,fmask=fm)\n\ndef unsynthesize(b,t):\n    h=t[\'h\'];heads=t[\'heads\'];F=t[\'F\'];cs,ks,_,_=old.constants(h)\n    outer(b,t[\'A\'],t[\'B\'],cs[0],ks[0],True,t[\'outer\'],t[\'wide\'])\n    reclaim(b,[[F]]);lad=base.zero_hold(b,heads[0][1]);b.cx(lad[-1],F);base.zero_free(b,heads[0][1],lad)\n    b.cz(F,t[\'fmask\'][0]);b.free(t[\'fmask\']);masks=[None]*h\n    for i,A,B,qi,p,v,pc,c,kap,token in reversed(t[\'history\']):\n        absorb(b,heads[i],A,B,qi,p,v,pc,c,kap,True,token,t[\'wide\'])\n        masks[i]=old.prefix_encode(b,heads,i,F)\n    old.multiply_safe165(b,t[\'d\'],heads[-1][1])\n    for i in reversed(range(1,h)):old.prefix_decode(b,heads,i,F,masks[i],t[\'d\'])\n    b.cx(heads[0][1],t[\'d\']);pg.zero_compact_free(b,heads[0][1],F)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/zero_padding_synthesis/kernel.py': '"""One-flag Taylor folds with common x-power padding on valid heads."""\nfrom pathlib import Path\nimport sys\nfrom types import FunctionType\nROOT=Path(__file__).resolve().parents[2];sys.path.insert(0,str(ROOT))\nfrom exp.tail_alias.kernel import *\nfrom exp.alternating_gauge import kernel as alt\nfrom exp.alternating_gauge.even_fold import full_product,full_quotient\nfrom exp.confluent_synthesis import kernel as ca\n\n\ndef zero_finish(b,value,F,fm):\n m=b.alloc_bits(1);b.hmr(F,m[0]);b.free(F);b.cx(fm[0],m[0]);b.free(fm)\n dv.phase_zero(b,value,m[0]);b.free(m)\n\n\ndef absorb_odd(b,head,A,B,Q,p,virtual,pconst,c,kap,reverse=False,transcript=None,wide=False):\n k=len(B);assert len(A)==k+1 and virtual is None and pconst==0\n out=B+head;v=epow(c,k+2);co=[v&4095,v>>12]\n if not reverse:\n  value=old.eval_implicit(b,A,Q,c);F=dv.zero_flag(b,value)\n  if k<4:\n   spare=[b.alloc_qubits(12),b.alloc_qubits(12)];b.cx(F,value[0]);cx(b,head[1],head[0],c)\n   mul24_xor99(b,combine(head),value,combine(spare),K)\n   cx(b,head[1],head[0],c);b.cx(F,value[0]);old.eval_implicit(b,A,Q,c,True)\n   q_times_low(b,B,kap);b.cx(F,A[0][0]);low_product(b,head,A,B);b.cx(F,A[0][0])\n   old.add_alias(b,p,spare,co);old.controlled_linear_xor(b,F,p,spare,co);R_to_tops(b,spare,B,c)\n  else:\n   old.eval_implicit(b,A,Q,c,True)\n   spare=[b.alloc_qubits(12),b.alloc_qubits(12)]\n   if k:\n    b.cx(B[-1],spare[0]);b.cx(B[-1],spare[1])\n    if k>=2:b.cx(B[-2],spare[0])\n   q_times_low(b,B,kap)\n   b.cx(F,A[0][0]);old.projected_product(b,head,A,Q,B+spare,kap);b.cx(F,A[0][0])\n   old.add_alias(b,p,spare,[kap,1]);old.controlled_linear_xor(b,F,p,spare,[kap,1])\n  hm=b.alloc_bits(24);b.hmr(combine(head),hm)\n  for x,z in zip(head,spare):b.swap(x,z);b.free(z)\n  fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n  return out,A,(hm,fm)\n R_to_tops(b,head,B,c,True);old.add_alias(b,p,head,co)\n value=old.eval_implicit(b,A,Q,c);F=dv.zero_flag(b,value);hm,fm=transcript\n old.controlled_linear_xor(b,F,p,head,co);b.cx(F,value[0]);divide_lowwidth369(b,combine(head),value,K);b.cx(F,value[0])\n cx(b,head[1],head[0],c);old.eval_implicit(b,A,Q,c,True)\n b.cx(F,A[0][0]);low_product(b,head,A,B);b.cx(F,A[0][0]);q_times_low(b,B,kap,True)\n value=old.eval_implicit(b,A,Q,c);zero_finish(b,value,F,fm);old.eval_implicit(b,A,Q,c,True)\n b.cz(combine(head),hm);b.free(hm)\n return out,A,transcript\n\n\ndef absorb_even(b,head,A,B,Q,r,c,kap,reverse=False,transcript=None):\n k=len(B);out=head+B;target=out+[r]\n if not reverse:\n  value=old.eval_implicit(b,A,Q,c);F=dv.zero_flag(b,value);old.eval_implicit(b,A,Q,c,True)\n  b.cx(F,A[0][0]);spare=[b.alloc_qubits(12),b.alloc_qubits(12)];full=spare+B+[r]\n  linear_words(b,full,base.embedding_matrix(k+3,kap));full_product(b,head,A,Q,full);b.cx(F,A[0][0])\n  old.controlled_linear_xor(b,F,r,full[-3:-1],[kap,1])\n  hm=b.alloc_bits(24);b.hmr(combine(head),hm)\n  for x,z in zip(head,spare):b.swap(x,z);b.free(z)\n  fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n  return out,A,(hm,fm)\n value=old.eval_implicit(b,A,Q,c);F=dv.zero_flag(b,value);old.eval_implicit(b,A,Q,c,True)\n old.controlled_linear_xor(b,F,r,out[-2:],[kap,1]);linear_words(b,target,base.embedding_matrix(k+3,kap),True)\n value=old.eval_implicit(b,A,Q,c);cx(b,head[1],head[0],c);hm,fm=transcript\n b.cx(F,value[0]);divide_lowwidth369(b,combine(head),value,K);b.cx(F,value[0]);cx(b,head[1],head[0],c)\n b.cz(combine(head),hm);b.free(hm);old.eval_implicit(b,A,Q,c,True)\n full_quotient(b,head,A,Q,B,r,kap)\n value=old.eval_implicit(b,A,Q,c);zero_finish(b,value,F,fm);old.eval_implicit(b,A,Q,c,True)\n return out,A,transcript\n\nsynthesize=FunctionType(alt.synthesize.__code__,dict(alt.synthesize.__globals__,old=ca.old,absorb=absorb_odd,absorb_even=absorb_even),\'synthesize\',alt.synthesize.__defaults__,alt.synthesize.__closure__)\nunsynthesize=FunctionType(alt.unsynthesize.__code__,dict(alt.unsynthesize.__globals__,old=ca.old,absorb=absorb_odd,absorb_even=absorb_even),\'unsynthesize\',alt.unsynthesize.__defaults__,alt.unsynthesize.__closure__)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/final_decoder/conversion.py': '"""Reuse Euclid gap predicates with a sparse quantum flag bank.\n\nKEEP_FULL holds nine flags through the diagonal middle stages. KEEP_CHILD\nalso holds level3 through its child recursion; that flag is measured at\nforward return and reconstructed with its matching phase at reverse entry.\nOther levels stream masks between identical adjacent predicates. Every\nfinal flag measurement is repaid by its exact zero-predicate phase.\n"""\nfrom pathlib import Path\nimport sys\nfrom types import SimpleNamespace\nfrom contextlib import contextmanager\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom exp.convert_short_hankel import phase34 as base\nfrom types import FunctionType\nfrom exp.fold_cached_norm.phase34 import phase_inverse34\nfrom exp.decoder.common import ar\n\ndef clone(fn,scope):\n    out=FunctionType(fn.__code__,scope,fn.__name__,fn.__defaults__,fn.__closure__)\n    out.__kwdefaults__=fn.__kwdefaults__\n    return out\n\ndef clear_flag(b,a,F):\n    m=b.alloc_bits(1);b.hmr(F,m[0]);b.free(F)\n    b.push_condition(m[0]);b.x(a)\n    nodes=b.alloc_qubits(len(a)-2)\n    b.init_and(a[0],a[1],nodes[0])\n    for i in range(1,len(nodes)):b.init_and(nodes[i-1],a[i+1],nodes[i])\n    b.ccz(nodes[-1],a[-1],m[0])\n    for i in reversed(range(len(nodes))):\n        if i:b.del_and(nodes[i-1],a[i+1],nodes[i])\n        else:b.del_and(a[0],a[1],nodes[0])\n    b.free(nodes);b.x(a);b.pop_condition();b.free(m)\n\ndef hold_flag(b,F):\n    m=b.alloc_bits(1);b.hmr(F,m[0]);b.free(F);return m\n\ndef consume_flag(b,F,m):\n    b.cz(F,m[0]);b.free(m)\n\nKEEP_FULL=frozenset((2,4,6,8,10,12,14,16,17))\nKEEP_CHILD=KEEP_FULL|{3}\n\n\ndef q_post(b,q,payloads,H,m,parent,reverse=False,phase=None):\n    if parent is None:\n        with q_inverse(b,q,m,reverse,phase) as qi:\n            for v in payloads:scale(b,v[-1],q,qi,not reverse)\n        return\n    tops,p,c=parent\n    if reverse or m+1 in KEEP_CHILD:F=phase[\'given_gap\'].pop(m+1)\n    else:\n        F=fresh_zero_flag(b,q)\n        consume_flag(b,F,phase[\'incoming_gap\'].pop(m+1))\n    b.cswap(F,q,p)\n    if not reverse:\n        for v,t in zip(payloads,tops):b.cswap(F,v[-1],t)\n    with q_inverse(b,q,m,reverse,phase) as qi:\n        if not reverse:\n            for v in payloads:scale(b,v[-1],q,qi,True)\n            for v,t in zip(payloads,tops):masked_product(b,F,v[-1],c,t)\n        else:\n            for v,t in zip(payloads,tops):masked_product(b,F,v[-1],c,t)\n            for v in payloads:scale(b,v[-1],q,qi,False)\n    if reverse:\n        for v,t in zip(payloads,tops):b.cswap(F,v[-1],t)\n    b.cswap(F,q,p)\n    if reverse and m+1 not in KEEP_CHILD:phase[\'return_mask\'][m+1]=hold_flag(b,F)\n    else:phase[\'returned_gap\'][m+1]=F\n\n\ndef look_solve(b,f,g,payloads,H=None,parent=None,reverse=False,phase=None):\n    m=len(f)-1\n    if m==1:return base_case(b,f,g,payloads,H,parent,reverse,phase)\n    keep=m in KEEP_CHILD;full=m in KEEP_FULL\n    F=phase[\'held_gap\'].pop(m) if reverse and full else fresh_zero_flag(b,g[-1])\n    if reverse and not full:consume_flag(b,F,phase[\'gap_flags\'].pop(m))\n    rotate(b,g,F)\n    if not reverse:carry=pre(b,f,g,payloads,defer=True)\n    else:carry=post_look(b,f,g,payloads,H,parent,True,incoming=phase[\'a\'].pop(m),defer=True,phase=phase)\n    canonical_permute(b,f,g,F)\n    if reverse or keep:phase[\'given_gap\'][m]=F\n    else:phase[\'incoming_gap\'][m]=hold_flag(b,F)\n    childparent=([v[-1]for v in payloads],f[-1],f[-2])\n    look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,childparent,reverse,phase)\n    if reverse and not keep:\n        F=fresh_zero_flag(b,g[-1]);consume_flag(b,F,phase[\'return_mask\'].pop(m))\n    else:F=phase[\'returned_gap\'].pop(m)\n    canonical_permute(b,f,g,F,True)\n    if not reverse:phase[\'a\'][m]=post_look(b,f,g,payloads,H,parent,incoming=carry,defer=True,phase=phase)\n    else:pre(b,f,g,payloads,True,incoming=carry)\n    rotate(b,g,F,True)\n    if reverse:clear_flag(b,g[-1],F)\n    elif full:phase[\'held_gap\'][m]=F\n    else:phase[\'gap_flags\'][m]=hold_flag(b,F)\n\n\ndef erase_product_zero(b, a, ai, F):\n    mask = b.alloc_bits(1)\n    b.hmr(F, mask[0]); b.free(F)\n    b.z(mask[0])\n    word = b.alloc_bits(12)\n    b.bit_store(word, False); b.cx(mask[0], word[0])\n    ar.phase_product(b, a, ai, word)\n    b.free(word); b.free(mask)\n\n\ndef cached_root_action(b, f, g, payloads, ai, Z, post, undo):\n    a = g[-1]; q0, q1 = f[-2:]\n    n = len(g)-1\n    if post:\n        operations = [(\'s\', v[-1], False) for v in payloads]\n        operations += [(\'m\', g[i], v[i], v[-1]) for v in payloads for i in range(n)]\n        operations += [(\'s\', v[-1], True) for v in payloads]\n        operations += [(\'c\', g[:-1], [q0, q1], f[:-2]),\n                       (\'s\', q0, False), (\'m\', q1, g[-2], q0)]\n    else:\n        operations = [(\'s\', v[-1], True) for v in payloads]\n        operations += [(\'m\', g[i], v[-1], v[i]) for v in payloads for i in range(n)]\n        operations += [(\'m\', q1, g[-2], q0), (\'s\', q0, True),\n                       (\'c\', g[:-1], [q0, q1], f[:-2])]\n    b.cx(Z, a[0])\n    old.runops(b, operations, a, ai, undo)\n    b.cx(Z, a[0])\n\n\ndef look_root_cached(b, O, g, payloads, reverse, phase):\n    m = len(O); keep = m in KEEP_CHILD; full = m in KEEP_FULL\n    F = phase[\'held_gap\'].pop(m) if reverse and full else old.fresh_zero_flag(b, g[-1])\n    if reverse and not full:\n        consume_flag(b, F, phase[\'gap_flags\'].pop(m))\n    old.rotate(b, g, F)\n    a = g[-1]\n    ai, Z = old.inverse_zero(b, a)\n    b.cx(Z, ai[0])\n    if reverse:\n        mask = phase.pop(\'_root_inverse_mask\')\n        b.cz(ai, mask); b.free(mask)\n    f = list(O) + [ai]\n    cached_root_action(b, f, g, payloads, ai, Z, post=reverse, undo=reverse)\n    erase_product_zero(b, a, ai, Z)\n    old.canonical_permute(b, f, g, F)\n    if reverse or keep:\n        phase[\'given_gap\'][m] = F\n    else:\n        phase[\'incoming_gap\'][m] = hold_flag(b, F)\n    parent = ([v[-1] for v in payloads], f[-1], f[-2])\n    old.look_solve(b, g, f[:-2], [v[:-1] for v in payloads], None,\n                   parent, reverse, phase)\n    if reverse and not keep:\n        F = old.fresh_zero_flag(b, g[-1])\n        consume_flag(b, F, phase[\'return_mask\'].pop(m))\n    else:\n        F = phase[\'returned_gap\'].pop(m)\n    old.canonical_permute(b, f, g, F, True)\n    Z = old.fresh_zero_flag(b, a)\n    cached_root_action(b, f, g, payloads, ai, Z, post=not reverse, undo=reverse)\n    erase_product_zero(b, a, ai, Z)\n    mask = b.alloc_bits(12)\n    b.hmr(ai, mask); b.free(ai)\n    if reverse:\n        phase_inverse34(b, a, mask, zero_mask=mask[0]); b.free(mask)\n    else:\n        phase[\'_root_inverse_mask\'] = mask\n    old.rotate(b, g, F, True)\n    if reverse:\n        clear_flag(b, g[-1], F)\n    elif full:\n        phase[\'held_gap\'][m] = F\n    else:\n        phase[\'gap_flags\'][m] = hold_flag(b, F)\n\n\ndef _bez_monic(b, O, E, payloads, reverse, history_scratch12, phase):\n    def reduce():\n        for i in range(len(O)):\n            ar.mul_xor0(b, E[-1], O[i], E[i])\n    reduce()\n    look_root_cached(b, O, E[:-1], payloads, reverse, phase)\n    reduce()\n\n\n\nscope = dict(vars(base._private), KEEP_CHILD=KEEP_CHILD, KEEP_FULL=KEEP_FULL, hold_flag=hold_flag,\n             consume_flag=consume_flag, clear_flag=clear_flag)\nfor name in [\'scale\', \'runops\', \'zero_toggle\', \'inverse_apply\', \'pre\',\n             \'canonical_permute\', \'masked_product\', \'fresh_zero_flag\',\n             \'post_look\', \'base\', \'bez_lookahead\']:\n    scope[name] = clone(getattr(base._private, name), scope)\nscope[\'base_case\'] = scope[\'base\']\nscope[\'q_inverse\'] = contextmanager(clone(base._private.q_inverse.__wrapped__, scope))\nscope[\'q_post\'] = clone(q_post, scope)\nscope[\'look_solve\'] = clone(look_solve, scope)\nshort = dict(base._short_scope, KEEP_CHILD=KEEP_CHILD, KEEP_FULL=KEEP_FULL, old=SimpleNamespace(**scope),\n             hold_flag=hold_flag, consume_flag=consume_flag, clear_flag=clear_flag,\n             ar=ar, erase_product_zero=erase_product_zero,\n             phase_inverse34=phase_inverse34)\nfor name, fn in [(\'cached_root_action\', cached_root_action),\n                 (\'look_root_cached\', look_root_cached), (\'_bez_monic\', _bez_monic)]:\n    short[name] = clone(fn, short)\n_core_convert = clone(base._short_scope[\'convert_safe\'], short)\n\n\ndef convert_safe(b, A, B, reverse=False, history_scratch12=None,\n                 phase_token=None, *, monic_root=False):\n    keys = [\'gap_flags\', \'given_gap\', \'incoming_gap\', \'return_mask\',\n            \'returned_gap\', \'held_gap\']\n    if phase_token is None:\n        phase_token = {\'a\': {}, \'q\': {}, \'_short_monic\': bool(monic_root)}\n        for key in keys:\n            phase_token[key] = {}\n    result = _core_convert(b, A, B, reverse, history_scratch12, phase_token,\n                          monic_root=monic_root)\n    if reverse:\n        assert all(not phase_token[key] for key in keys)\n        assert \'_root_inverse_mask\' not in phase_token\n    return result\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/final_decoder/decode.py': '"""Nine homogeneous external chains, a trailing jet, and exact phase replay.\n\nThe only new dependencies are final_decoder/synthesis.py and conversion.py.\nEvery other import is an unchanged decoderkit module.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[1]\nsys.path[:0]=[str(ROOT),str(ROOT.parent)]\nimport decode as baseline\nfrom exp.decoder.common import *\nfrom exp.confluent_extract.model import synthetic\nfrom exp.inverse12_escape.joint import inverse24_forward190,inverse24_reverse209\nfrom exp.jet24.pro_jet import jet_forward,jet_reverse\nfrom exp.inverse19_joint.kernel import inverse_initialized\nfrom exp.fold_cached_norm.phase34 import phase_inverse34\nfrom exp.convert_short_hankel.phase34 import _private as guarded\nfrom final_decoder import synthesis as folds\nfrom final_decoder.conversion import convert_safe,clear_flag\n\n\ndef geometry():\n    groups=[];qs=[];P=[0,0,1]\n    for i in range(17):\n        c=2*i;kap=mul(c,c)^c^K\n        groups.append(dict(zero=False,c=c,kap=kap,gamma=c|4096,\n                           count=2,head=2*i,word=4*i,t=1))\n        qs.extend([(kap,1)]*2)\n        for _ in range(2):P=pmul(P,[kap,1,1])\n    groups.append(dict(zero=True,c=0,kap=0,gamma=0,\n                       count=1,head=34,word=68,t=0))\n    qs.append((0,0));matrix=[]\n    for g in groups[:-1]:\n        Q=P\n        for _ in range(2):\n            Q,r=synthetic(Q,g[\'gamma\']);assert r==0\n            q=Q+[0]*(70-len(Q))\n            matrix.extend([tuple(x&4095 for x in q),tuple(x>>12 for x in q)])\n    matrix.extend([tuple(P[1:]+[0]*(71-len(P))),\n                   tuple(P[2:]+[0]*(72-len(P)))])\n    return groups,qs,P,matrix\n\n\ngroups,qs,P,matrix=geometry()\ncs=[g[\'c\']for g in groups[:-1]for _ in range(2)]\nks=[kap for kap,_ in qs[:-1]]\nPscan,rem=divmodp(P,[ks[0],1,1]);assert rem==[0]\ncorrections=[];c=0\nfor kap in ks:corrections.append(c);c^=inv(kap)\nassert c==0\n\n\ndef square12(b,x,reverse=False):\n    ar.linear_inplace(b,x,[power(1<<j,2048 if reverse else 2)for j in range(12)])\n\n\ndef scale_word(b,x,f,fi):\n    out=b.alloc_qubits(12);ar.mul_xor0(b,x,f,out)\n    ar.erase_product(b,out,fi,x);b.swap(x,out);b.free(out)\n\n\ndef masked_update(b,x,f,F):\n    """Return the phase mask of F*x_old, after x += F*x_old*(f+1)."""\n    u=b.alloc_qubits(12)\n    for a,z in zip(x,u):b.init_and(F,a,z)\n    b.x(f[0]);ar.mul_xor0(b,u,f,x);b.x(f[0])\n    m=b.alloc_bits(12);b.hmr(u,m);b.free(u);return m\n\n\ndef phase_masked_product(b,x,f,F,m):\n    t=b.alloc_qubits(1)[0];field=ar.field(12)\n    for i,a in enumerate(x):\n        b.init_and(F,a,t)\n        for j,c in enumerate(f):\n            for k in ar.bits(field.mul(1<<i,1<<j)):b.ccz(t,c,m[k])\n        b.del_and(F,a,t)\n    b.free(t);b.free(m)\n\n\ndef reverse_masked_multiply(b,x,f,fi,F,m):\n    fresh=masked_update(b,x,fi,F)\n    for a,bit in zip(x,m):b.ccz(F,a,bit)\n    b.free(m);phase_masked_product(b,x,f,F,fresh)\n\n\ndef deferred_multiply(b,x,f):\n    out=b.alloc_qubits(12);ar.mul_xor0(b,x,f,out)\n    m=b.alloc_bits(12);b.hmr(x,m);b.swap(x,out);b.free(out);return m\n\n\ndef reverse_deferred_multiply(b,x,f,fi,m):\n    fresh=deferred_multiply(b,x,fi)\n    b.cz(x,m);b.free(m);ar.phase_product(b,x,f,fresh);b.free(fresh)\n\n\ndef controlled_inverse(b,x,F,reverse=False,token=None):\n    xi=inverse_initialized(b,x)\n    if reverse:b.cz(xi,token);b.free(token)\n    b.cswap(F,x,xi)\n    m=b.alloc_bits(12);b.hmr(xi,m);b.free(xi)\n    if reverse:phase_inverse34(b,x,m);b.free(m)\n    else:return m\n\n\ndef term_correction_with_inverse(b,p,q,d,u,v,pole,qi,reverse=False,token=None,Z=None):\n    if reverse:\n        Z=guarded.fresh_zero_flag(b,d);b.cz(Z,token[0]);b.free(token)\n    F=b.alloc_qubits(1)[0];b.cx(Z,F);b.ccx(Z,pole,F)\n    def add_product(x,target):\n        w=b.alloc_qubits(12);ar.mul_xor0(b,x,qi,w)\n        b.ccx(F,w,target);ar.erase_product(b,x,qi,w);b.free(w)\n    if reverse:add_product(v,u);add_product(p,v)\n    else:add_product(p,v);add_product(v,u)\n    b.ccx(Z,pole,F);b.cx(Z,F);b.free(F)\n    if reverse:return Z\n    zm=b.alloc_bits(1);b.hmr(Z,zm[0]);b.free(Z);return zm\n\n\ndef tail_finish(b,p,q,d,u,v,pole,reverse=False,token=None,Z=None):\n    """Share q\'s inverse0 across correction and clipped pole products."""\n    if reverse:b.cswap(pole,v,d);b.cswap(pole,u,v)\n    qi,Zq=guarded.inverse_zero(b,q)\n    if reverse:guarded.consume(b,qi,Zq,token[\'inverse\'])\n    if not reverse:tt=term_correction_with_inverse(b,p,q,d,u,v,pole,qi,Z=Z)\n    b.cx(Zq,q[0]);b.cx(Zq,qi[0])\n    if reverse:\n        reverse_masked_multiply(b,d,qi,q,pole,token[\'d\'])\n        reverse_masked_multiply(b,v,q,qi,pole,token[\'v\'])\n    else:\n        vm=masked_update(b,v,q,pole);dm=masked_update(b,d,qi,pole)\n    b.cx(Zq,qi[0]);b.cx(Zq,q[0])\n    if reverse:Z=term_correction_with_inverse(b,p,q,d,u,v,pole,qi,True,token[\'term\'])\n    qi_token=guarded.erase(b,q,qi,Zq,not reverse)\n    if reverse:return Z\n    if not reverse:\n        b.cswap(pole,u,v);b.cswap(pole,v,d)\n        return dict(inverse=qi_token,term=tt,v=vm,d=dm)\n\n\ndef tail_inplace(b,p,q,d,u,v,pole,reverse=False,token=None):\n    """(d,u,v) <-> (delta,N0,N1), with p,q the implicit leaders."""\n    if not reverse:\n        ar.mul_xor0(b,q,v,p)\n        square12(b,q);ar.mul_xor0(b,q,u,d);square12(b,q,True)\n        masks=[]\n        def divide(pi):\n            b.x(pole);masks.append(masked_update(b,d,pi,pole));b.x(pole)\n        tp=guarded.inverse_apply(b,p,divide,defer=True);tp=(tp,masks[0]);ar.mul_xor0(b,q,v,p)\n        def multiply(di,m):\n            b.x(pole);vm=masked_update(b,v,d,pole);b.x(pole)\n            ar.mul_xor0(b,q,u,v);um=deferred_multiply(b,u,d)\n            return vm,um\n        Z=guarded.fresh_zero_flag(b,d);b.cx(Z,d[0]);td=multiply(None,None);b.cx(Z,d[0])\n        tf=tail_finish(b,p,q,d,u,v,pole,Z=Z)\n        return dict(p=tp,d=td,finish=tf)\n    Z=tail_finish(b,p,q,d,u,v,pole,True,token[\'finish\'])\n    def divide(di,m):\n        vm,um=m\n        reverse_deferred_multiply(b,u,d,di,um);ar.mul_xor0(b,q,u,v)\n        b.x(pole);reverse_masked_multiply(b,v,d,di,pole,vm);b.x(pole)\n    di=inverse_initialized(b,d);b.cx(Z,d[0]);b.cx(Z,di[0])\n    divide(di,token[\'d\']);b.cx(Z,di[0]);b.cx(Z,d[0])\n    guarded.erase(b,d,di,Z,False);ar.mul_xor0(b,q,v,p)\n    def multiply(pi):\n        b.x(pole);reverse_masked_multiply(b,d,pi,p,pole,token[\'p\'][1]);b.x(pole)\n    guarded.inverse_apply(b,p,multiply,incoming=token[\'p\'][0]);ar.mul_xor0(b,q,v,p)\n    square12(b,q);ar.mul_xor0(b,q,u,d);square12(b,q,True)\n\n\ndef synthesize(b,words,pole):\n    heads=[words[2*i:2*i+2]for i in range(34)];u,v=words[-2:]\n    d,border=heads[0];r=b.alloc_qubits(12)\n    b.cx(border,r);b.cx(border,d);b.cx(r,border);b.x(border[0])\n    for i in range(1,34):\n        qi,target=(r,border)if i%2 else(border,r)\n        ar.mul_xor0(b,heads[i][1],qi,target)\n        folds.add_tail(b,heads[i],qi,d,bool(i%2))\n    ar.mul_xor0(b,r,border,d);tt=tail_inplace(b,r,border,d,u,v,pole)\n    # The exact chart remains live through the diagonal middle stages.\n    A,B,hist=[u,v],[d],[]\n    for i in reversed(range(1,34)):\n        aa,bb=A,B\n        if i%2:\n            ar.mul_xor0(b,heads[i][1],r,border);qi,p=r,border\n            A,B,t=folds.sg[\'absorb\'](b,heads[i],A,B,qi,p,None,0,cs[i],ks[i],wide=False)\n        else:\n            qi,p=border,r\n            A,B,t=folds.sg[\'absorb_even\'](b,heads[i],A,B,qi,r,cs[i],ks[i])\n        hist.append((i,aa,bb,qi,p,t))\n    b.x(border[0]);b.free(border);C=B+[r]\n    def undo():\n        folds.ug[\'reclaim\'](b,[border]);b.x(border[0])\n        for i,aa,bb,qi,p,t in reversed(hist):\n            if i%2:\n                folds.ug[\'absorb\'](b,heads[i],aa,bb,qi,p,None,0,cs[i],ks[i],True,t,False)\n                ar.mul_xor0(b,heads[i][1],r,border)\n            else:folds.ug[\'absorb_even\'](b,heads[i],aa,bb,qi,r,cs[i],ks[i],True,t)\n        tail_inplace(b,r,border,d,u,v,pole,True,tt);ar.mul_xor0(b,r,border,d)\n        for i in reversed(range(1,34)):\n            qi,target=(r,border)if i%2 else(border,r)\n            folds.add_tail(b,heads[i],qi,d,bool(i%2));ar.mul_xor0(b,heads[i][1],qi,target)\n        b.cx(r,d);b.x(border[0]);b.cx(r,border);b.cx(border,r);b.free(r)\n    return C,A,undo\n\n\ndef step(b,head,N,D,L,kap,c,reverse=False):\n    a,h=head\n    if reverse:scale(b,N,inv(kap));b.swap(N,D);ar.mul_xor0(b,a,D,N);scale(b,L,inv(kap))\n    cx(b,a,h,c);square12(b,D);ar.mul_xor0(b,h,D,L);square12(b,D,True);cx(b,a,h,c)\n    if not reverse:scale(b,L,kap);ar.mul_xor0(b,a,D,N);b.swap(N,D);scale(b,N,kap)\n\n\ndef first_step(b,head,N,D,L,kap):\n    b.cx(head[0],N);b.swap(N,D);scale(b,N,kap)\n    b.cx(head[1],L);scale(b,L,kap)\n\n\ndef scalar_step(b,head,N,D,kap,reverse=False,first=False):\n    if reverse:scale(b,N,inv(kap));b.swap(N,D)\n    if first:b.cx(head[0],N)\n    else:ar.mul_xor0(b,head[0],D,N)\n    if not reverse:b.swap(N,D);scale(b,N,kap)\n\n\ndef normal_form(b,N,D,L):\n    """Normalize by the guaranteed nonzero D; defer only its inverse mask."""\n    di=inverse_initialized(b,D);scale_word(b,N,di,D)\n    square12(b,D);square12(b,di);scale_word(b,L,di,D)\n    square12(b,di,True);square12(b,D,True)\n    m=b.alloc_bits(12);b.hmr(di,m);b.free(di);return m\n\n\ndef extraction(b,words,reverse=False,token=None):\n    heads=[words[2*i:2*i+2]for i in range(34)];N,L=words[-2:]\n    if not reverse:\n        ext=external_extraction(b,words);D=b.alloc_qubits(12);b.x(D[0])\n        first_step(b,heads[0],N,D,L,ks[0])\n        for head,kap,c in zip(heads[1:],ks[1:],corrections[1:]):step(b,head,N,D,L,kap,c)\n        pole=b.alloc_qubits(1)[0];guarded.zero_toggle(b,D,pole);b.cswap(pole,N,D)\n        norm=normal_form(b,N,D,L);ci=controlled_inverse(b,L,pole)\n        gauge=b.alloc_bits(12);b.hmr(D,gauge);b.free(D)\n        return dict(external=ext,pole=pole,normalization=norm,gauge_mask=gauge,chart_inverse=ci)\n    pole=token[\'pole\'];controlled_inverse(b,L,pole,True,token[\'chart_inverse\'])\n    D=b.alloc_qubits(12);b.x(D[0]);b.cswap(pole,N,D)\n    for head,kap,c in reversed(list(zip(heads,ks,corrections))[1:]):step(b,head,N,D,L,kap,c,True)\n    scale(b,N,inv(ks[0]));b.swap(N,D);scale(b,L,inv(ks[0]))\n    extra_z_mask=normal_form(b,N,D,L)\n    b.cx(heads[0][0],N);b.cx(heads[0][1],L)\n    b.cz(D,token[\'normalization\']);b.free(token[\'normalization\'])\n    inverse_mask=b.alloc_bits(12);b.hmr(D,inverse_mask);b.free(D)\n    z_mask=token[\'gauge_mask\'];b.cx(extra_z_mask,z_mask);b.free(extra_z_mask)\n    D=b.alloc_qubits(12);b.x(D[0]);scalar_step(b,heads[0],N,D,ks[0],first=True)\n    for head,kap in zip(heads[1:],ks[1:]):scalar_step(b,head,N,D,kap)\n    b.cswap(pole,N,D);b.cz(D,z_mask);b.free(z_mask)\n    phase_inverse34(b,D,inverse_mask);b.free(inverse_mask);b.cswap(pole,N,D)\n    guarded.zero_toggle(b,D,pole)\n    for head,kap in reversed(list(zip(heads[1:],ks[1:]))):scalar_step(b,head,N,D,kap,True)\n    scalar_step(b,heads[0],N,D,ks[0],True,first=True)\n    b.x(D[0]);b.free(D);b.free(pole);external_extraction(b,words,True,token[\'external\'])\n\n\ndef decode(b,S,E):\n    with b.mark(\'mask_projection\'):mt=baseline.project(b,S,E)\n    with b.mark(\'initial\'):linear_words(b,S,matrix)\n    with b.mark(\'extract\'):et=extraction(b,S)\n    with b.mark(\'synthesis\'):A,B,undo=synthesize(b,S,et[\'pole\'])\n    with b.mark(\'convert\'):ct=convert_safe(b,A,B,monic_root=True)\n    with b.mark(\'scan\'):baseline.scan(b,A,B,E,skip_points=baseline.SELECTED,node_polynomial=Pscan)\n    with b.mark(\'unconvert\'):convert_safe(b,A,B,True,phase_token=ct)\n    with b.mark(\'unsynthesis\'):undo()\n    with b.mark(\'unextract\'):extraction(b,S,True,et)\n    with b.mark(\'uninitial\'):linear_words(b,S,matrix,True)\n    with b.mark(\'restore_masks\'):baseline.restore(b,E,mt)\n\n\nfrom functools import lru_cache\nfrom exp.mul99.kernels import mul24_xor99\nfrom exp.mul99.schedules import multiply_forward99\nfrom exp.arithmetic.division import phase_product24,zero_flag\nfrom exp.noscratch.schedules import norm_xor_noscratch,inverse_stream_noscratch\nfrom exp.noscratch.kernels import mul_xor_noscratch\nfrom exp.mul99.phase36 import phase_inverse36_pivot\n\nEXTERNAL_HOMOGENEOUS=frozenset(range(8,17))\n\n@lru_cache(None)\ndef external_square_columns(reverse=False):\n    def epow(x,n):\n        out=1\n        while n:\n            if n&1:out=emul(out,x)\n            x=emul(x,x);n>>=1\n        return out\n    return tuple(epow(1<<i,1<<23 if reverse else 2)for i in range(24))\n\n\ndef external_square24(b,x,reverse=False):ar.linear_inplace(b,x,external_square_columns(reverse))\n\n\ndef external_mul_head(b,head,D,N,ct):\n    a,h=head;cx(b,h,a,ct)\n    mul24_xor99(b,combine(head),D,N,K)\n    cx(b,h,a,ct)\n\n\ndef external_corrected_product(b,head,D,L,ct,c):\n    a,h=head\n    if not c:\n        ar.mul_xor0(b,h,D[:12],L[:12]);ar.mul_xor0(b,h,D[12:],L[12:])\n    else:\n        scale(b,a,c);cx(b,h,a,mul(c,ct)^1);scale(b,h,c)\n        mul24_xor99(b,combine(head),D,L,K)\n        scale(b,h,inv(c));cx(b,h,a,mul(c,ct)^1);scale(b,a,inv(c))\n\n\ndef external_chain_step(b,head,N,D,L,q,ct,c,reverse=False,first=False):\n    if reverse:\n        scalepair(b,N,inv(q));b.swap(N,D);external_mul_head(b,head,D,N,ct);scalepair(b,L,inv(q))\n    if first and not reverse:\n        b.cx(head[1],L[:12]);scalepair(b,L,q)\n        a,h=head;cx(b,h,a,ct);b.cx(combine(head),N);cx(b,h,a,ct)\n        b.swap(N,D);scalepair(b,N,q)\n        return\n    external_square24(b,D);external_corrected_product(b,head,D,L,ct,c);external_square24(b,D,True)\n    if not reverse:\n        scalepair(b,L,q);external_mul_head(b,head,D,N,ct);b.swap(N,D);scalepair(b,N,q)\n\n\ndef external_scalar_step(b,head,N,D,q,ct,reverse=False,first=False):\n    if reverse:scalepair(b,N,inv(q));b.swap(N,D)\n    if first:\n        a,h=head;cx(b,h,a,ct);b.cx(combine(head),N);cx(b,h,a,ct)\n    else:external_mul_head(b,head,D,N,ct)\n    if not reverse:b.swap(N,D);scalepair(b,N,q)\n\n\ndef external_ordinary_node(b,words,gi,reverse=False,token=None):\n    g=groups[gi];u=combine(words[g[\'word\']:g[\'word\']+2]);v=combine(words[g[\'word\']+2:g[\'word\']+4])\n    history=[]if not reverse else list(token)\n    for h in reversed(range(2*gi))if reverse else range(2*gi):\n        a,d=words[2*h:2*h+2];q=ks[h]^g[\'kap\']\n        def translate():\n            b.cx(a,u[:12]);cx(b,d,u[:12],g[\'c\']);b.cx(d,u[12:]);b.cx(d,v[:12])\n        if not reverse:\n            translate();history.append(jet_forward(b,u,v));scalepair(b,v,q);b.cx(u,v);scalepair(b,u,q)\n        else:\n            scalepair(b,u,inv(q));b.cx(u,v);scalepair(b,v,inv(q));jet_reverse(b,u,v,history.pop());translate()\n    if reverse:assert not history\n    return history\n\n\ndef external_normalize_paired(b,N,D,L,reverse=False,token=None):\n    b.cx(D[12:],D[:12]);nm=multiply_forward99(b,N,D,K)\n    external_square24(b,D);lm=multiply_forward99(b,L,D,K);external_square24(b,D,True)\n    b.cx(D[12:],D[:12])\n    u=b.alloc_qubits(12);norm_xor_noscratch(b,D,u,K)\n    if reverse:b.cz(u,token);b.free(token)\n    m=inverse_stream_noscratch(b,u);ar._phase_norm(b,D,m,K);b.free(m)\n    masks=[]\n    for word,squared in [(N,False),(L,True)]:\n        if squared:square12(b,u)\n        row=[]\n        for off in (0,12):\n            half=word[off:off+12];out=b.alloc_qubits(12)\n            mul_xor_noscratch(b,half,u,out);hm=b.alloc_bits(12)\n            b.hmr(half,hm);b.swap(half,out);b.free(out);row.append(hm)\n        masks.append(row)\n        if squared:square12(b,u,True)\n    if reverse:um=u\n    else:\n        um=b.alloc_bits(12);b.hmr(u,um);b.free(u)\n    norm=b.alloc_qubits(12);norm_xor_noscratch(b,D,norm,K)\n    for word,row,squared in [(N,masks[0],False),(L,masks[1],True)]:\n        if squared:square12(b,norm)\n        for off,hm in zip((0,12),row):\n            ar.phase_product(b,word[off:off+12],norm,hm);b.free(hm)\n        if squared:square12(b,norm,True)\n    if not reverse:ar._erase_norm(b,D,norm,K);b.free(norm)\n    phase_product24(b,N,D,nm,K);b.free(nm)\n    external_square24(b,D);phase_product24(b,L,D,lm,K);b.free(lm);external_square24(b,D,True)\n    return (norm,um) if reverse else um\n\n\ndef external_inverse_phase_norm_pair(b,x,mask,reciprocal,norm):\n    b.cx(x[12:],x[:12])\n    ar.phase_product(b,x[:12],reciprocal,mask[:12]);ar.phase_product(b,x[12:],reciprocal,mask[12:])\n    b.cx(x[12:],x[:12])\n    im=b.alloc_bits(12);b.hmr(reciprocal,im);b.free(reciprocal)\n    phase_inverse36_pivot(b,norm,im);b.free(im)\n    nm=b.alloc_bits(12);b.hmr(norm,nm);b.free(norm)\n    ar._phase_norm(b,x,nm,K);b.free(nm)\n\n\ndef external_node_chain(b,words,gi,reverse=False,token=None,normalizer=None):\n    g=groups[gi];N=combine(words[g[\'word\']:g[\'word\']+2]);L=combine(words[g[\'word\']+2:g[\'word\']+4])\n    heads=[words[2*i:2*i+2]for i in range(2*gi)]\n    params=[];c=0\n    for i in range(2*gi):\n        q=ks[i]^g[\'kap\'];assert q\n        params.append((q,c));c^=inv(q)\n    assert c==0\n    if not reverse:\n        D=b.alloc_qubits(24);b.x(D[0])\n        for i,(head,(q,c))in enumerate(zip(heads,params)):\n            external_chain_step(b,head,N,D,L,q,g[\'c\'],c,first=i==0)\n        pole=zero_flag(b,D);b.cswap(pole,N,D)\n        norm=external_normalize_paired(b,N,D,L)\n        gauge=b.alloc_bits(24);b.hmr(D,gauge);b.free(D)\n        return dict(pole=pole,gauge=gauge,norm=norm)\n    pole=token[\'pole\'];D=b.alloc_qubits(24);b.x(D[0]);b.cswap(pole,N,D)\n    for head,(q,c)in reversed(list(zip(heads,params))[1:]):external_chain_step(b,head,N,D,L,q,g[\'c\'],c,True)\n    q0=params[0][0];scalepair(b,N,inv(q0));b.swap(N,D);scalepair(b,L,inv(q0))\n    norm=external_normalize_paired(b,N,D,L,True,token[\'norm\'])\n    a,h=heads[0];cx(b,h,a,g[\'c\']);b.cx(combine(heads[0]),N);cx(b,h,a,g[\'c\']);b.cx(h,L[:12])\n    invmask=b.alloc_bits(24);b.hmr(D,invmask);b.free(D)\n    D=b.alloc_qubits(24);b.x(D[0])\n    for i,(head,(q,c))in enumerate(zip(heads,params)):external_scalar_step(b,head,N,D,params[i][0],g[\'c\'],first=i==0)\n    b.cswap(pole,N,D);b.cz(D,token[\'gauge\']);b.free(token[\'gauge\'])\n    reciprocal,direct=norm\n    external_inverse_phase_norm_pair(b,D,invmask,reciprocal,direct);b.free(invmask);b.cswap(pole,N,D)\n    clear_flag(b,D,pole)\n    for i in reversed(range(len(heads))):external_scalar_step(b,heads[i],N,D,params[i][0],g[\'c\'],True,first=i==0)\n    b.x(D[0]);b.free(D)\n\n\ndef external_extraction(b,words,reverse=False,token=None):\n    history=[]if not reverse else list(token)\n    for gi in reversed(range(17))if reverse else range(17):\n        g=groups[gi];u=combine(words[g[\'word\']:g[\'word\']+2]);v=combine(words[g[\'word\']+2:g[\'word\']+4])\n        a,d=words[g[\'word\']:g[\'word\']+2]\n        if not reverse:\n            nt=external_node_chain(b,words,gi,normalizer=None)if gi in EXTERNAL_HOMOGENEOUS else external_ordinary_node(b,words,gi)\n            cx(b,d,a,g[\'c\']);b.cx(d,v[:12]);local=inverse24_forward190(b,v,K)\n            cx(b,words[g[\'word\']+3],words[g[\'word\']+2],g[\'c\'])\n            history.append((nt,local))\n        else:\n            nt,local=history.pop();cx(b,words[g[\'word\']+3],words[g[\'word\']+2],g[\'c\'])\n            inverse24_reverse209(b,v,local,K);b.cx(d,v[:12]);cx(b,d,a,g[\'c\'])\n            if gi in EXTERNAL_HOMOGENEOUS:external_node_chain(b,words,gi,True,nt,None)\n            else:external_ordinary_node(b,words,gi,True,nt)\n    assert not reverse or not history\n    return history\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/final_decoder/synthesis.py': '"""Synthesis with additive continuants, shared norms, and no outer fold.\n\nEvery imported dependency is an unchanged file in decoderkit.\nThe caller compensates for the omitted fixed quadratic in the scan.\nAll arithmetic uses circ builder gates.\n"""\nimport sys\nfrom pathlib import Path\nHERE = Path(__file__).resolve().parents[1]\nsys.path[:0] = [str(HERE), str(HERE.parent)]\nimport decode as baseline\nfrom exp.decoder.common import ar\nfrom exp.mul99.schedules import multiply_forward99\nfrom exp.noscratch.schedules import norm_xor_noscratch\nfrom exp.noscratch.kernels import mul_xor_noscratch\nfrom exp.arithmetic.division import phase_product24\nfrom exp.inverse_search.inverse20_iso import inverse6_xor20\nfrom exp.phase24_tradeoff.phase6small import phase_inverse6_classical6\nfrom exp.zero_padding_synthesis.kernel import *\nfrom exp.fold_cached_norm import kernel as FK\n\n\n\ndef divide_fused(b, x, y, F, fm, after, K=None):\n    """Consume F=[y=0] and prior phase fm; divide with identity at zero.\n\n    The callback sees restored y and the live flag.  It may change x, but\n    must restore y before returning.  Every norm/flag measurement is closed.\n    """\n    b.cx(F, y[0])\n    b.cx(y[12:], y[:12])\n    oldmask = multiply_forward99(b, x, y, K)\n    b.cx(y[12:], y[:12])\n    N = b.alloc_qubits(12)\n    norm_xor_noscratch(b, y, N, K)\n    conj = [ar.field(12).frobenius(1 << j, 6) for j in range(12)]\n    ar.linear_inplace(b, N, conj)\n    masks = []\n    for off in (0, 12):\n        half = x[off:off + 12]\n        out = b.alloc_qubits(12)\n        ar.mul_xor0(b, half, N, out)\n        m = b.alloc_bits(12)\n        b.hmr(half, m)\n        b.swap(half, out)\n        b.free(out)\n        masks.append(m)\n    ar.linear_inplace(b, N, conj)\n    to, frm, k6 = ar.tower(12)\n    ar.linear_inplace(b, N, to)\n    M = b.alloc_qubits(6)\n    norm_xor_noscratch(b, N, M, k6)\n    ar.linear_inplace(b, N, frm)\n    t = inverse6_xor20(b, M)\n    for off in (0, 12):\n        half = x[off:off + 12]\n        ar.linear_inplace(b, half, to)\n        for j in (0, 6):\n            part = half[j:j + 6]\n            out = b.alloc_qubits(6)\n            mul_xor_noscratch(b, part, t, out)\n            ar.erase_product(b, out, M, part)\n            b.swap(part, out)\n            b.free(out)\n        ar.linear_inplace(b, half, frm)\n    for off, m in zip((0, 12), masks):\n        ar.phase_product(b, x[off:off + 12], N, m)\n        b.free(m)\n    phase_product24(b, x, y, oldmask, K)\n    b.free(oldmask)\n\n    # Norms of a clipped zero factor differ only in their constant bit.\n    b.cx(F, y[0])\n    b.cx(F, N[0])\n    b.cx(F, M[0])\n    after()\n    fmask = b.alloc_bits(1)\n    b.hmr(F, fmask[0])\n    b.free(F)\n    b.cx(fm[0], fmask[0])\n    b.free(fm)\n    b.z(fmask[0])\n    pm = b.alloc_bits(6)\n    b.bit_store(pm, False)\n    b.cx(fmask[0], pm[0])\n    ar.phase_product(b, M, t, pm)\n    b.free(pm)\n    b.free(fmask)\n    tm = b.alloc_bits(6)\n    b.hmr(t, tm)\n    b.free(t)\n    # t=inv0(M)+[M=0], so its constant output bit also masks the zero phase.\n    phase_inverse6_classical6(b, M, tm, zero_mask=tm[0])\n    b.free(tm)\n    ar.linear_inplace(b, N, to)\n    ar._erase_norm(b, N, M, k6)\n    b.free(M)\n    ar.linear_inplace(b, N, frm)\n    ar._erase_norm(b, y, N, K)\n    b.free(N)\n\n\ndef absorb_odd_fused(b, head, A, B, Q, p, virtual, pconst, c, kap,\n                     reverse=False, transcript=None, wide=False):\n    if not reverse:\n        return FK.prior.absorb_odd(b, head, A, B, Q, p, virtual, pconst,\n                                   c, kap, reverse, transcript, wide)\n    k = len(B)\n    v = epow(c, k + 2)\n    co = [v & 4095, v >> 12]\n    R_to_tops(b, head, B, c, True)\n    old.add_alias(b, p, head, co)\n    value = old.eval_implicit(b, A, Q, c)\n    F = dv.zero_flag(b, value)\n    hm, fm = transcript\n    old.controlled_linear_xor(b, F, p, head, co)\n    def after():\n        cx(b, head[1], head[0], c)\n        old.eval_implicit(b, A, Q, c, True)\n        b.cx(F, A[0][0])\n        low_product(b, head, A, B)\n        b.cx(F, A[0][0])\n        q_times_low(b, B, kap, True)\n        old.eval_implicit(b, A, Q, c)\n    divide_fused(b, combine(head), value, F, fm, after, K)\n    b.cz(combine(head), hm)\n    b.free(hm)\n    old.eval_implicit(b, A, Q, c, True)\n    return B + head, A, transcript\n\n\ndef absorb_even_fused(b, head, A, B, Q, r, c, kap,\n                      reverse=False, transcript=None):\n    if not reverse:\n        return FK.prior.absorb_even(b, head, A, B, Q, r, c, kap,\n                                    reverse, transcript)\n    k = len(B)\n    out = head + B\n    target = out + [r]\n    value = old.eval_implicit(b, A, Q, c)\n    F = dv.zero_flag(b, value)\n    old.eval_implicit(b, A, Q, c, True)\n    old.controlled_linear_xor(b, F, r, out[-2:], [kap, 1])\n    linear_words(b, target, base.embedding_matrix(k + 3, kap), True)\n    value = old.eval_implicit(b, A, Q, c)\n    cx(b, head[1], head[0], c)\n    hm, fm = transcript\n    def after():\n        cx(b, head[1], head[0], c)\n        old.eval_implicit(b, A, Q, c, True)\n        full_quotient(b, head, A, Q, B, r, kap)\n        old.eval_implicit(b, A, Q, c)\n    divide_fused(b, combine(head), value, F, fm, after, K)\n    b.cz(combine(head), hm)\n    b.free(hm)\n    old.eval_implicit(b, A, Q, c, True)\n    return out, A, transcript\n\n\n# Bind only the two inner fold reverses into the original private graph.\n_shared_unsynthesize = FK.bind(baseline.unsynthesize,\n    absorb=absorb_odd_fused, absorb_even=absorb_even_fused)\n\nsg = baseline.synthesize.__globals__\nug = _shared_unsynthesize.__globals__\nar = sg[\'ar\']\n\n\ndef add_tail(b, head, q, d, odd):\n    a, x = head\n    if not odd:\n        b.cx(x, a)\n    sg[\'add_tail_a\'](b, a, q, d)\n    if not odd:\n        b.cx(x, a)\n', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/playbook/exp/cf_check/audit.py': '"""Standalone classical finite-field EXTRACT/SYNTHESIZE audit (no circuits/verifier)."""\nimport argparse, json, random, time\nfrom collections import Counter\n\nclass Field:\n    def __init__(self, m, modulus):\n        self.m=m; self.n=1<<m; self.mask=self.n-1\n        self.exp=[]; self.log=[-1]*self.n; x=1\n        for i in range(self.n-1):\n            assert self.log[x]<0\n            self.log[x]=i; self.exp.append(x)\n            x<<=1\n            if x&self.n:x^=modulus\n        assert x==1\n        self.exp*=2\n        self.K=next(x for x in range(1,self.n) if self.trace(x)==1)\n    def mul(self,a,b):return self.exp[self.log[a]+self.log[b]] if a and b else 0\n    def inv(self,a):return self.exp[self.n-1-self.log[a]] if a else 0\n    def div(self,a,b):return self.mul(a,self.inv(b))\n    def trace(self,a):\n        t=0\n        for _ in range(self.m):t^=a;a=self.mul(a,a)\n        return t\n    def emul(self,u,v):\n        a=u&self.mask;b=u>>self.m;c=v&self.mask;d=v>>self.m\n        bd=self.mul(b,d)\n        return (self.mul(a,c)^self.mul(self.K,bd)) | ((self.mul(a,d)^self.mul(b,c)^bd)<<self.m)\n    def einv(self,u):\n        a=u&self.mask;b=u>>self.m\n        ni=self.inv(self.mul(a,a)^self.mul(a,b)^self.mul(self.K,self.mul(b,b)))\n        return self.mul(a^b,ni)|(self.mul(b,ni)<<self.m)\n    def escale(self,u,a):return self.mul(u&self.mask,a)|(self.mul(u>>self.m,a)<<self.m)\n    def ev(self,p,u):\n        out=0\n        for a in reversed(p):out=self.emul(out,u)^a\n        return out\n    def trim(self,p):\n        while len(p)>1 and not p[-1]:p.pop()\n        return p\n    def add(self,a,b):\n        c=[0]*max(len(a),len(b))\n        for i,x in enumerate(a):c[i]^=x\n        for i,x in enumerate(b):c[i]^=x\n        return self.trim(c)\n    def pmul(self,a,b):\n        c=[0]*(len(a)+len(b)-1)\n        for i,x in enumerate(a):\n            for j,y in enumerate(b):c[i+j]^=self.mul(x,y)\n        return self.trim(c)\n    def scale(self,p,c):return self.trim([self.mul(x,c) for x in p])\n    def pdiv(self,a,b):\n        a=a.copy();out=[0]*max(1,len(a)-len(b)+1);bi=self.inv(b[-1])\n        assert b!=[0]\n        while a!=[0] and len(a)>=len(b):\n            d=len(a)-len(b);v=self.mul(a[-1],bi);out[d]=v\n            for i,x in enumerate(b):a[i+d]^=self.mul(v,x)\n            self.trim(a)\n        return self.trim(out),a\n    def gcd(self,a,b):\n        while b!=[0]:a,b=b,self.pdiv(a,b)[1]\n        return self.scale(a,self.inv(a[-1]))\n    def derivative(self,p):return self.trim([p[i] if i%2 else 0 for i in range(1,len(p))] or [0])\n\ndef data(f, cs, betas, ws):\n    kappas=[f.mul(c,c)^c^f.K for c in cs]\n    nodes=[c|f.n for c in cs]\n    D=[1]\n    for b in betas:D=f.pmul(D,[b,1])\n    N=[0]\n    for b,w in zip(betas,ws):\n        quotient,rem=f.pdiv(D,[b,1]);assert rem==[0]\n        N=f.add(N,f.scale(quotient,w))\n    vals=[f.emul(f.ev(N,g),f.einv(f.ev(D,g))) for g in nodes]\n    return kappas,nodes,N,D,vals\n\ndef extract(f,cs,kappas,vals):\n    vals=vals.copy();heads=[];zeros=[]\n    for i,c in enumerate(cs):\n        b=vals[i]>>f.m;a=(vals[i]&f.mask)^f.mul(b,c);heads.append((a,b))\n        for j in range(i+1,len(cs)):\n            u=vals[j]^a^f.mul(b,cs[j])^(b<<f.m)\n            if not u:zeros.append((i,j))\n            vals[j]=f.escale(f.einv(u),kappas[i]^kappas[j])\n    return heads,zeros\n\ndef synth(f,kappas,heads):\n    p,q,d=0,1,0\n    for a,b in heads:p,q,d=q,p^f.mul(b,q),d^f.mul(p,q)^f.mul(a^b,f.mul(q,q))\n    tail=(p,q,d)\n    if q:A,B=[f.div(d,q),p],[q]\n    else:A,B=[p],[0]\n    for (a,b),kappa in reversed(list(zip(heads,kappas))):\n        A,B=f.add(f.pmul([a,b],A),f.pmul([kappa,1,1],B)),A\n    if not q:A,B=[0]+A,[0]+B\n    return f.trim(A),f.trim(B),tail\n\ndef rational_audit(f,kappas,nodes,N,D,heads):\n    """Follow exact rational transforms until the first genuine pole or termination."""\n    first_poles=[];termination=None\n    for i,(a,b) in enumerate(heads):\n        if f.ev(D,nodes[i])==0:return {\'head_pole\':i,\'poles\':first_poles}\n        assert f.ev(N,nodes[i])==f.emul(a^f.mul(b,nodes[i]&f.mask)^(b<<f.m),f.ev(D,nodes[i]))\n        H=f.add(N,f.pmul([a,b],D))\n        if H==[0]:termination=i;break\n        H,rem=f.pdiv(H,[kappas[i],1,1]);assert rem==[0]\n        poles=[j for j in range(i+1,len(nodes)) if f.ev(H,nodes[j])==0]\n        if poles:return {\'first_nonterminal_zero_step\':i,\'poles\':poles,\'residual_degree\':len(H)-1}\n        N,D=D,H\n    return {\'termination\':termination,\'poles\':first_poles}\n\ndef validate(m,modulus,h,repeats,seed):\n    f=Field(m,modulus);rng=random.Random(seed);cs=list(range(0,2*h,2));assert max(cs)<f.n\n    results=Counter();examples=[]\n    for t in range(h+1):\n        for trial in range(repeats):\n            betas=rng.sample(range(1,f.n),t);ws=[rng.randrange(1,f.n) for _ in betas]\n            kappas,nodes,N,D,vals=data(f,cs,betas,ws)\n            heads,zeros=extract(f,cs,kappas,vals);A,B,tail=synth(f,kappas,heads)\n            ra=rational_audit(f,kappas,nodes,N,D,heads)\n            correct=f.pmul(A,D)==f.pmul(B,N)\n            sqfree=f.gcd(B,f.derivative(B))==[1]\n            degree=len(B)-1\n            result={\'t\':t,\'trial\':trial,\'correct\':correct,\'degree\':degree,\'monic\':B[-1]==1,\'squarefree\':sqfree,\'tail\':tail,\'rational\':ra,\'zeros\':len(zeros)}\n            results[\'cases\']+=1;results[\'correct\']+=correct;results[\'monic\']+=B[-1]==1;results[\'degree_h\']+=degree==h;results[\'squarefree\']+=sqfree\n            if \'termination\' in ra and ra[\'termination\'] is not None:\n                term=ra[\'termination\'];results[\'terminated\']+=1\n                assert all(a==b==0 for a,b in heads[term+1:])\n                results[\'later_heads_zero\']+=1\n            if not correct or not sqfree or degree!=h or not result[\'monic\']:\n                if len(examples)<12:examples.append(dict(result,betas=betas,ws=ws,heads=heads,A=A,B=B))\n            if correct:\n                pad,rem=f.pdiv(B,D);assert rem==[0]\n                expected=D.copy()\n                for j in range(t+1,h,2):expected=f.pmul(expected,[kappas[j],1,1])\n                if not t%2:expected=[0]+expected\n                assert expected==B,(\'padding index pattern\',t,B,expected)\n                allowed=[kappas[i] for i in range(h)]\n                if not tail[1]:\n                    pad,rem=f.pdiv(pad,[0,1]);assert rem==[0]\n                for k in allowed:\n                    quotient,rem=f.pdiv(pad,[k,1,1])\n                    if rem==[0]:pad=quotient\n                assert pad==[1],pad\n                assert all(f.ev(B,b)!=0 for b in range(1,f.n) if b not in betas)\n    return {\'m\':m,\'K\':f.K,\'h\':h,\'repeats_per_weight\':repeats,\'counts\':dict(results),\'examples\':examples}\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--m\',type=int,default=12);parser.add_argument(\'--h\',type=int,default=35);parser.add_argument(\'--repeats\',type=int,default=3);parser.add_argument(\'--seed\',type=int,default=1)\n    args=parser.parse_args();mods={3:0b1011,4:0b10011,6:0b1000011,12:0x1053}\n    start=time.time();result=validate(args.m,mods[args.m],args.h,args.repeats,args.seed);result[\'seconds\']=time.time()-start\n    print(json.dumps(result,indent=2))\n', 'blocks/dqi/pro-q8q9-889/source/prior/decode_backhalf_q7.py': '"""Exact back-half improvements to the uploaded narrow240 decoder.\n\nThis file imports only unchanged files from the uploaded kit. All new\narithmetic, scan and measurement-phase helpers are defined here.\n"""\nfrom functools import lru_cache\nfrom contextlib import contextmanager\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport importlib.util\nimport inspect\nimport sys\n\nROOT = Path(__file__).resolve().parent\nBASE_DIR = ROOT / \'exp/r-main/narrow240\'\nsys.path[:0] = [str(BASE_DIR), str(ROOT/\'playbook/kit/decoder\'),\n               str(ROOT/\'playbook/kit\'),\n               str(ROOT/\'playbook/kit/decoder/exp/convert\')]\nspec = importlib.util.spec_from_file_location(\'uploaded_default\',\n                                             BASE_DIR/\'decode_retimed.py\')\nbase = importlib.util.module_from_spec(spec)\nspec.loader.exec_module(base)\nimport high_stream\nimport direct_converter\nimport direct_kernels as dk\nimport truncated_small as small\nfrom converter import clone\nfrom exp.convert import kernels as k\nfrom exp.decoder.common import ar, zero_hold, zero_free\nfrom exp.safe_scalar.phase import phase_classical_parity\nfrom exp.inverse19_joint.kernel import inverse_initialized\nfrom exp.scan.scan import constants, gf, power, mul, inv, scale\n\ndef masked_products(b, F, payloads, c, targets):\n    """Add F*c*v[-1] to each target, with one shared AND bank.\n\n    HMR of u_i=F*c_i adds r_i*F*c_i to the phase. CCZ(F,c_i,r_i)\n    repays it exactly, for every outcome and also when F or c is zero.\n    """\n    u = b.alloc_qubits(12)\n    for i in range(12):\n        b.init_and(F, c[i], u[i])\n    for v, t in zip(payloads, targets):\n        ar.mul_xor0(b, u, v[-1], t)\n    mask = b.alloc_bits(12)\n    b.hmr(u, mask)\n    b.free(u)\n    for i in range(12):\n        b.ccz(F, c[i], mask[i])\n    b.free(mask)\n\n\ndef q_post(b, q, payloads, H, m, parent, reverse=False, phase=None):\n    if parent is None:\n        with q_inverse(b, q, m, reverse, phase) as qi:\n            for v in payloads:\n                scale(b, v[-1], q, qi, not reverse)\n        return\n    tops, p, c = parent\n    if reverse or m + 1 in KEEP_CHILD:\n        F = phase[\'given_gap\'].pop(m + 1)\n    else:\n        F = fresh_zero_flag(b, q)\n        consume_flag(b, F, phase[\'incoming_gap\'].pop(m + 1))\n    b.cswap(F, q, p)\n    if not reverse:\n        for v, t in zip(payloads, tops):\n            b.cswap(F, v[-1], t)\n    with q_inverse(b, q, m, reverse, phase) as qi:\n        if not reverse:\n            for v in payloads:\n                scale(b, v[-1], q, qi, True)\n            masked_products(b, F, payloads, c, tops)\n        else:\n            masked_products(b, F, payloads, c, tops)\n            for v in payloads:\n                scale(b, v[-1], q, qi, False)\n    if reverse:\n        for v, t in zip(payloads, tops):\n            b.cswap(F, v[-1], t)\n    b.cswap(F, q, p)\n    if reverse and m + 1 not in KEEP_CHILD:\n        phase[\'return_mask\'][m + 1] = hold_flag(b, F)\n    else:\n        phase[\'returned_gap\'][m + 1] = F\n\n\ndef selected_pre(b,f,g,payloads,F,undo=False,incoming=None,defer=False):\n    if not undo:return ordinary_pre(b,f,g,payloads,F,False,incoming,defer)\n    # F=[g[-1]=0] is already live. Thus the old 24-bit zero test equals\n    # F AND [g[-2]=0]. The temporary zero ladder is erased exactly.\n    ladder=zero_hold(b,g[-2]);Z0=b.alloc_qubits(1)[0]\n    b.init_and(F,ladder[-1],Z0);zero_free(b,g[-2],ladder)\n    b.cx(Z0,g[-2][0])\n    masks=high_full_restore(b,f,g,F)\n    b.cx(Z0,g[-2][0]);pivot(b,g,F);a=g[-1]\n    ai,Z=inverse_zero(b,a);b.cx(Z,Z0);b.free(Z0);consume(b,ai,Z,incoming)\n    b.cx(Z,a[0]);b.cx(Z,ai[0]);high_close_heads(b,f,g,ai,F,masks)\n    for v in payloads:dk.shifted_product(b,g[:-1],v[-1],v[:-1],F)\n    for v in payloads:scale(b,v[-1],a,ai,False)\n    b.cx(Z,ai[0]);b.cx(Z,a[0]);out=erase(b,a,ai,Z,defer)\n    pivot(b,g,F);return out\n\n\ndef triple_square_phase(b,x,y,z,mask):\n    """Apply (-1)**dot(mask, x*y*z**2), with no fresh quantum wire."""\n    assert len(x)==len(y)==len(z)==len(mask)==12\n    f=ar.field(12)\n    for linear,vector in ar.tensor(12):\n        co=b.alloc_bits(12)\n        b.bit_store(co,False)\n        for j in range(12):\n            for k in ar.bits(f.mul(vector,f.square(1<<j))):\n                b.cx(mask[k],co[j])\n        ix=list(ar.bits(linear));p=ix[0]\n        for j in ix[1:]:\n            b.cx(x[j],x[p]);b.cx(y[j],y[p])\n        phase_classical_parity(b,x[p],y[p],z,co)\n        for j in reversed(ix[1:]):\n            b.cx(y[j],y[p]);b.cx(x[j],x[p])\n        b.free(co)\n\n\ndef high_close_heads(b,f,g,ai,F,masks):\n    lo,hi=f[-2:]\n    ar.phase_product(b,hi,ai,masks[1])\n    ar.phase_product(b,lo,ai,masks[0])\n    with dk.selected(b,g[-2],g[-3],F) as c:\n        triple_square_phase(b,hi,c,ai,masks[0])\n    for mask in masks:b.free(mask)\n\n\ndef close_heads(b,f,g,ai,masks):\n    lo,hi=f[-2:]\n    ar.phase_product(b,hi,ai,masks[1])\n    ar.phase_product(b,lo,ai,masks[0])\n    triple_square_phase(b,hi,g[-2],ai,masks[0])\n    for mask in masks:b.free(mask)\n\n\nCUBE_TERMS = ((1211, 2209, 1392), (1841, 3016, 2582), (179, 3076, 80), (41, 3038, 2255), (8, 774, 1831), (1875, 2483, 3690), (1651, 2511, 2931), (280, 3214, 3979), (26, 3467, 589), (929, 3190, 2015), (303, 631, 540), (1973, 2217, 1107), (595, 3124, 703), (127, 2465, 2410), (440, 756, 735), (688, 3342, 3259), (894, 2259, 2441), (453, 3665, 525), (1504, 2527, 499), (212, 3164, 1395), (728, 3409, 2134), (497, 2243, 2285), (211, 3422, 1259), (1315, 3054, 1230))\nCUBE_LINEAR = (1453, 3308, 1839, 1801, 3614, 2307, 1204, 3804, 23, 166, 1183, 1118)\n\n\ndef quadratic_accumulate(b, x, z, terms, linear):\n    assert len(set(map(str, list(x)+list(z)))) == len(x)+len(z)\n    for i, value in enumerate(linear):\n        for j in ar.bits(value):\n            b.cx(x[i], z[j])\n    for A, C, value in terms:\n        assert A and C and A != C and value\n        left = list(ar.bits(A))\n        p = left[0]\n        for i in left[1:]:\n            b.cx(x[i], x[p])\n        transformed_C = C ^ ((A ^ (1 << p)) if (C >> p) & 1 else 0)\n        q = next(i for i in ar.bits(transformed_C) if i != p)\n        right = [i for i in ar.bits(transformed_C) if i != q]\n        for i in right:\n            b.cx(x[i], x[q])\n        output = list(ar.bits(value))\n        t = output[0]\n        for j in output[1:]:\n            b.cx(z[t], z[j])\n        b.ccx(x[p], x[q], z[t])\n        for j in reversed(output[1:]):\n            b.cx(z[t], z[j])\n        for i in reversed(right):\n            b.cx(x[i], x[q])\n        for i in reversed(left[1:]):\n            b.cx(x[i], x[p])\n\n\ndef cube_xor(b, x, z):\n    quadratic_accumulate(b, x, z, CUBE_TERMS, CUBE_LINEAR)\n\n\ndef cube_phase(b, x, mask):\n    field = ar.field(12)\n    linear = [field.pow(1 << i, 3) for i in range(12)]\n    for i, value in enumerate(linear):\n        for bit in ar.bits(value):\n            b.cz(x[i], mask[bit])\n    for i in range(12):\n        for j in range(i):\n            value = field.pow((1 << i) ^ (1 << j), 3) ^ linear[i] ^ linear[j]\n            for bit in ar.bits(value):\n                b.ccz(x[i], x[j], mask[bit])\n\n\ndef hankel_direct(b, q, x):\n    """Apply T(q) to len(x)=3 or 4 words, q[0] is coefficient of z."""\n    n = len(x)\n    assert n in (3, 4)\n    copies = [b.alloc_qubits(12) for _ in range(n-2)]\n    for source, target in zip(x[1:-1], copies):\n        b.cx(source, target)\n    small.accumulate(b, [v for w in [x[0]]+copies for v in w],\n                     [v for w in q[:n-1] for v in w],\n                     [v for w in x[1:] for v in w], n-1)\n    masks = [b.alloc_bits(12) for _ in copies]\n    for source, mask in zip(copies, masks):\n        b.hmr(source, mask)\n        b.free(source)\n    # old x1 = new x1 + q1*x0.\n    b.cz(x[1], masks[0])\n    ar.phase_product(b, q[0], x[0], masks[0])\n    if n == 4:\n        # old x2 = new x2 + q1*new x1 + (q1^2+q2)*x0.\n        b.cz(x[2], masks[1])\n        ar.phase_product(b, q[0], x[1], masks[1])\n        ar.phase_product(b, q[1], x[0], masks[1])\n        ar.linear_inplace(b, q[0], ar.square_columns(12))\n        ar.phase_product(b, q[0], x[0], masks[1])\n        ar.linear_inplace(b, q[0], [ar.field(12).pow(1 << i, 2048) for i in range(12)])\n    for mask in masks:\n        b.free(mask)\n\n\n@lru_cache(None)\ndef reverse_partition(n):\n    if n == 1:\n        return 0, None\n    choices = []\n    if n == 3:\n        choices.append((99, None))\n    if n == 4:\n        choices.append((153, None))\n    for left in range(1, n):\n        choices.append((reverse_partition(left)[0]\n                        + reverse_partition(n-left)[0] + 42*(n-1), left))\n    return min(choices, key=lambda pair: pair[0])\n\n\ndef hankel(b, B, A, reverse=False):\n    if not reverse:\n        return k.hankel(b, B, A)\n    cube = b.alloc_qubits(12)\n    cube_xor(b, B[-1], cube)\n\n    def reverse_tri(b, q, x, reverse=False):\n        assert reverse\n        n = len(x)\n        if n == 1:\n            return\n        _, left = reverse_partition(n)\n        if left is None:\n            if n == 4:\n                b.cx(cube, q[2])\n            ar.linear_xor(b, q[0], q[1], ar.square_columns(12))\n            hankel_direct(b, q, x)\n            ar.linear_xor(b, q[0], q[1], ar.square_columns(12))\n            if n == 4:\n                b.cx(cube, q[2])\n            return\n        right = n-left\n        reverse_tri(b, q[:left-1], x[:left], True)\n        for ac, bc, tc in reversed(k.cross_terms(left, right)):\n            k.rank_update(b, x[:left], ac, q[:n-1], bc, x[left:], tc)\n        reverse_tri(b, q[:right-1], x[left:], True)\n\n    clone(k.hankel, dict(vars(k), tri=reverse_tri))(b, B, A, True)\n    mask = b.alloc_bits(12)\n    b.hmr(cube, mask)\n    b.free(cube)\n    cube_phase(b, B[-1], mask)\n    b.free(mask)\n\n\ndef scan(b, A, B, E, *, points=4095, skip_points=(), node_polynomial=None):\n    if len(A) != 35 or len(B) != 35 or len(E) < points:\n        raise ValueError("need 35+35 words and one mask per point")\n    if points != 4095:\n        raise ValueError("full cycle required for restoration")\n    alpha = gf.primitive_element\n    P = constants()[2] if node_polynomial is None else node_polynomial\n    powers = [power(alpha, k) for k in range(35)]\n    skip_points = frozenset(skip_points)\n    assert all(0 <= j < 4095 for j in skip_points)\n    beta = lead = 1\n    leadstep = power(alpha, 35)\n    m = b.alloc_bits(12)\n    nonzero = b.alloc_bits(1)\n    for j in range(4095):\n        if j not in skip_points:\n            # OR is implemented entirely on classical wires, using the\n            # builder\'s conditional store.  No quantum input is observed.\n            b.bit_store(nonzero, False)\n            for bit in E[j]:\n                b.bit_store(nonzero, True, control=bit)\n            b.push_condition(nonzero[0])\n            for k in range(1, 35):\n                b.cx(A[k], A[0])\n                b.cx(B[k], B[0])\n            b.gf2_iadd(lead, target=B[0])\n            ladder = zero_hold(b, B[0])\n            pv = 0\n            for v in reversed(P):\n                pv = mul(pv, beta) ^ v\n            c = inv(mul(beta, pv))\n            assert c\n            b.bit_store(m, False)\n            for t in range(12):\n                for i in ar.bits(mul(c, 1 << t)):\n                    b.cx(E[j][i], m[t])\n            b.ccz(ladder[-1], A[0], m)\n            zero_free(b, B[0], ladder)\n            b.gf2_iadd(lead, target=B[0])\n            for k in range(1, 35):\n                b.cx(A[k], A[0])\n                b.cx(B[k], B[0])\n            b.pop_condition()\n        for k in range(1, 35):\n            scale(b, A[k], powers[k])\n            scale(b, B[k], powers[k])\n        beta = mul(beta, alpha)\n        lead = mul(lead, leadstep)\n    assert beta == lead == 1\n    b.free(nonzero)\n    b.free(m)\n\n\ndef make_converter():\n    original = direct_converter.mh.make_converter\n\n    def factory(return_scope=False):\n        cv, scope, short = original(True)\n        scope[\'masked_products\'] = clone(masked_products, scope)\n        scope[\'q_post\'] = clone(q_post, scope)\n        short[\'old\'] = SimpleNamespace(**scope)\n        return (cv, scope, short) if return_scope else cv\n\n    direct = SimpleNamespace(**dict(vars(direct_converter),\n                                   mh=SimpleNamespace(make_converter=factory)))\n    cv = clone(high_stream.make_converter,\n               dict(vars(high_stream), direct=direct))()\n    cells = dict(zip(cv.__code__.co_freevars, cv.__closure__))\n    short = cells[\'core\'].cell_contents.__globals__\n    scope = short[\'old\'].direct_post.__globals__\n    scope.update(zero_hold=zero_hold, zero_free=zero_free,\n                 KEEP_FULL=frozenset({16}), THRESHOLD=6)\n    short[\'KEEP_FULL\'] = scope[\'KEEP_FULL\']\n    scope[\'direct_pre\'] = clone(selected_pre, scope)\n    scope[\'high_close_heads\'] = high_close_heads\n    scope[\'close_heads\'] = close_heads\n    short[\'old\'].hankel = hankel\n    return cv\n\n\ndef gap_zero_action(b,f,g,payloads,ai,F,H,post,undo):\n    a=g[-1]\n    b.ccx(F,H,a[0])\n    root_action(b,f,g,payloads,ai,False,F,post,undo)\n    b.ccx(F,H,a[0])\n\n\ndef look_root_cached(b,O,g,payloads,reverse,phase):\n    m=len(O);keep=m in KEEP_CHILD;full=m in KEEP_FULL\n    F=phase[\'held_gap\'].pop(m) if reverse and full else old.fresh_zero_flag(b,g[-1])\n    if reverse and not full:consume_flag(b,F,phase[\'gap_flags\'].pop(m))\n    pivot(b,g,F);a=g[-1];ai,Z=old.inverse_zero(b,a);b.cx(Z,ai[0])\n    if reverse:\n        mask=phase.pop(\'_root_inverse_mask\');b.cz(ai,mask);b.free(mask)\n    f=list(O)+[ai]\n    if reverse:\n        H=phase[\'held_gap\'][m-1]\n        mask=b.alloc_bits(1);b.hmr(Z,mask[0]);b.free(Z)\n        b.ccz(F,H,mask[0]);b.free(mask)\n        gap_zero_action(b,f,g,payloads,ai,F,H,post=True,undo=True)\n    else:\n        root_action(b,f,g,payloads,ai,Z,F,post=False,undo=False)\n        erase_product_zero(b,a,ai,Z)\n    pivot(b,g,F);aligned(b,f,g,F)\n    if reverse or keep:phase[\'given_gap\'][m]=F\n    else:phase[\'incoming_gap\'][m]=hold_flag(b,F)\n    parent=([v[-1]for v in payloads],f[-1],f[-2])\n    old.look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,parent,reverse,phase)\n    if reverse and not keep:\n        F=old.fresh_zero_flag(b,g[-1]);consume_flag(b,F,phase[\'return_mask\'].pop(m))\n    else:F=phase[\'returned_gap\'].pop(m)\n    aligned(b,f,g,F);pivot(b,g,F)\n    if not reverse:\n        H=phase[\'held_gap\'][m-1]\n        gap_zero_action(b,f,g,payloads,ai,F,H,post=True,undo=False)\n    else:\n        Z=old.fresh_zero_flag(b,a)\n        root_action(b,f,g,payloads,ai,Z,F,post=False,undo=True)\n        erase_product_zero(b,a,ai,Z)\n    mask=b.alloc_bits(12);b.hmr(ai,mask);b.free(ai)\n    if reverse:phase_inverse34(b,a,mask,zero_mask=mask[0]);b.free(mask)\n    else:phase[\'_root_inverse_mask\']=mask\n    pivot(b,g,F)\n    if reverse:clear_flag(b,g[-1],F)\n    elif full:phase[\'held_gap\'][m]=F\n    else:phase[\'gap_flags\'][m]=hold_flag(b,F)\n\n\ndef consume_clipped(b,ai,F,H,token):\n    if token is None:return\n    mask,zm=token\n    b.cz(ai,mask)\n    b.ccz(F,H,mask[0]);b.ccz(F,H,zm[0])\n    b.free(mask);b.free(zm)\n\n\ndef retire_clipped(b,ai):\n    mask=b.alloc_bits(12);b.hmr(ai,mask);b.free(ai)\n    zm=b.alloc_bits(1);b.bit_store(zm,False);b.cx(mask[0],zm[0])\n    return mask,zm\n\n\ndef post(b,f,g,payloads,F,H,parent,reverse=False,incoming=None,defer=False,phase=None):\n    m=len(f)-1;G=phase[\'held_gap\'][m-1];q0,q1=f[-2:]\n    assert defer\n    if not reverse:\n        q_post(b,q1,payloads,H,m,parent,phase=phase)\n        pivot(b,g,F);a=g[-1]\n        for v in payloads:dk.selected_dot(b,g[:-1],v[:-1],v[-1],F)\n        b.ccx(F,G,a[0]);ai=inverse_initialized(b,a)\n        consume_clipped(b,ai,F,G,incoming)\n        for v in payloads:scale(b,v[-1],a,ai,True)\n        out=retire_clipped(b,ai)\n        pivot(b,g,F);phase[\'heads\'][m]=high_full_restore(b,f,g,F)\n        # The product flag implies F=1, so the selected scalar is here.\n        b.ccx(F,G,g[-2][0])\n        return out\n    pivot(b,g,F);a=g[-1]\n    b.ccx(F,G,a[0]);ai=inverse_initialized(b,a)\n    consume_clipped(b,ai,F,G,incoming)\n    scale(b,q1,a,ai,True);cross(b,g,q1,q0,F);scale(b,q0,a,ai,True)\n    for word,mask in zip([q0,q1],phase[\'heads\'].pop(m)):\n        b.cz(word,mask);b.free(mask)\n    dk.shifted_conv(b,g[:-1],[q0,q1],f[:-2],F,True)\n    for v in payloads:scale(b,v[-1],a,ai,False)\n    b.ccx(F,G,a[0]);out=retire_clipped(b,ai)\n    for v in payloads:dk.selected_dot(b,g[:-1],v[:-1],v[-1],F)\n    pivot(b,g,F);q_post(b,q1,payloads,H,m,parent,True,phase)\n    return out\n\n\ndef install_retained_gaps(cv):\n    cells=dict(zip(cv.__code__.co_freevars,cv.__closure__))\n    short=cells[\'core\'].cell_contents.__globals__\n    scope=short[\'old\'].direct_post.__globals__\n    scope[\'KEEP_FULL\']=frozenset({15,16,17})\n    short[\'KEEP_FULL\']=scope[\'KEEP_FULL\']\n    old_inverse=scope[\'q_inverse\']\n    @contextmanager\n    def q_inverse(b,a,m,reverse,phase):\n        if m!=16 or not phase[\'_short_monic\']:\n            with old_inverse(b,a,m,reverse,phase) as ai:\n                yield ai\n            return\n        ai=inverse_initialized(b,a)\n        if reverse:\n            mask=phase[\'q\'].pop(m);b.cz(ai,mask);b.free(mask)\n        yield ai\n        mask=b.alloc_bits(12);b.hmr(ai,mask);b.free(ai)\n        if reverse:\n            short[\'phase_inverse34\'](b,a,mask);b.free(mask)\n        else:phase[\'q\'][m]=mask\n    scope[\'q_inverse\']=q_inverse\n    short[\'gap_zero_action\']=clone(gap_zero_action,short)\n    short[\'look_root_cached\']=clone(look_root_cached,short)\n    original=scope[\'direct_post\']\n    scope.update(consume_clipped=consume_clipped,retire_clipped=retire_clipped,\n                 inverse_initialized=inverse_initialized)\n    selected=clone(post,scope)\n    def direct_post(b,f,g,payloads,*args,**kwargs):\n        return (selected if len(f)-1==16 else original)(b,f,g,payloads,*args,**kwargs)\n    scope[\'direct_post\']=direct_post\n    return cv\n\n\ndef make_gate(b, F, H):\n    """G=H(1+F)."""\n    G = b.alloc_qubits(1)[0]\n    b.x(F); b.init_and(F, H, G); b.x(F)\n    return G\n\n\ndef erase_gate(b, F, H, G):\n    mask = b.alloc_bits(1)\n    b.hmr(G, mask[0]); b.free(G)\n    b.cz(H, mask[0]); b.ccz(F, H, mask[0])\n    b.free(mask)\n\n\ndef quotient_inverse(b, p, g, F, H, K):\n    # FH+GK = H+G(1+K), since G=H(1+F).\n    t, s = g[-1], g[-2]\n    G = make_gate(b, F, H)\n    r = b.alloc_qubits(12)\n    b.cx(t, r); b.cx(H, r[0])\n    b.cx(s, t); b.cx(K, t[0]); b.x(t[0])\n    b.ccx(G, t, r)\n    b.x(t[0]); b.cx(K, t[0]); b.cx(s, t)\n    erase_gate(b, F, H, G)\n    qi = b.alloc_qubits(12)\n    b.cx(F, p[0]); ar.mul_xor0(b, p, r, qi); b.cx(F, p[0])\n    mask = b.alloc_bits(12)\n    b.hmr(r, mask); b.free(r)\n    b.cz(t, mask); b.cz(H, mask[0])\n    G = make_gate(b, F, H)\n    b.ccz(G, t, mask); b.ccz(G, s, mask)\n    b.ccz(G, K, mask[0]); b.cz(G, mask[0])\n    erase_gate(b, F, H, G); b.free(mask)\n    return qi\n\n\ndef close_quotient_phase(b, p, g, F, H, K, mask):\n    # p*F=0 after the parent swap, and H*t=0 for t=g[-1].\n    # Therefore qi=(p+F)*t + H*p*(s+K) + F*H.\n    t, s = g[-1], g[-2]\n    b.cx(F, p[0]); ar.phase_product(b, p, t, mask); b.cx(F, p[0])\n    b.ccz(F, H, mask[0])\n    b.cx(K, s[0])\n    field = ar.field(12)\n    for j in range(12):\n        coefficients = b.alloc_bits(12); b.bit_store(coefficients, False)\n        for i in range(12):\n            for bit in ar.bits(field.mul(1 << i, 1 << j)):\n                b.cx(mask[bit], coefficients[i])\n        phase_classical_parity(b, H, s[j], p, coefficients)\n        b.free(coefficients)\n    b.cx(K, s[0])\n\n\ndef root_child_post(b, q, payloads, H, m, parent, reverse=False, phase=None):\n    if m != 16 or not phase[\'_short_monic\']:\n        return prior_q_post(b, q, payloads, H, m, parent, reverse, phase)\n    assert parent is not None\n    tops, p, c = parent\n    g, current, child = phase[\'_child_scalar_context\']\n    if reverse or m+1 in KEEP_CHILD:\n        F = phase[\'given_gap\'].pop(m+1)\n    else:\n        F = fresh_zero_flag(b, q)\n        consume_flag(b, F, phase[\'incoming_gap\'].pop(m+1))\n    b.cswap(F, q, p)\n    if not reverse:\n        for v, top in zip(payloads, tops):\n            b.cswap(F, v[-1], top)\n    qi = quotient_inverse(b, p, g, F, current, child)\n    if reverse:\n        mask = phase[\'q\'].pop(m); b.cz(qi, mask); b.free(mask)\n    if not reverse:\n        for v in payloads:\n            scale(b, v[-1], q, qi, True)\n        masked_products(b, F, payloads, c, tops)\n    else:\n        masked_products(b, F, payloads, c, tops)\n        for v in payloads:\n            scale(b, v[-1], q, qi, False)\n    mask = b.alloc_bits(12)\n    b.hmr(qi, mask); b.free(qi)\n    if reverse:\n        close_quotient_phase(b, p, g, F, current, child, mask); b.free(mask)\n    else:\n        phase[\'q\'][m] = mask\n    if reverse:\n        for v, top in zip(payloads, tops):\n            b.cswap(F, v[-1], top)\n    b.cswap(F, q, p)\n    if reverse and m+1 not in KEEP_CHILD:\n        phase[\'return_mask\'][m+1] = hold_flag(b, F)\n    else:\n        phase[\'returned_gap\'][m+1] = F\n\n\ndef install_child_inverse(cv):\n    cells = dict(zip(cv.__code__.co_freevars, cv.__closure__))\n    short = cells[\'core\'].cell_contents.__globals__\n    scope = short[\'old\'].direct_post.__globals__\n    # The selected post dispatcher is a closure; its selected implementation\n    # still reads the original private converter globals.\n    if \'old_look_solve\' not in scope:\n        candidates = dict(zip(short[\'old\'].direct_post.__code__.co_freevars,\n                              short[\'old\'].direct_post.__closure__))\n        scope = candidates[\'selected\'].cell_contents.__globals__\n    prior_post = scope[\'direct_post\']\n    scope.update(prior_q_post=scope[\'q_post\'], quotient_inverse=quotient_inverse,\n                 close_quotient_phase=close_quotient_phase)\n    scope[\'q_post\'] = clone(root_child_post, scope)\n    def post(b, f, g, payloads, F, H, parent, reverse=False,\n             incoming=None, defer=False, phase=None):\n        selected = len(f)-1 == 16 and phase[\'_short_monic\']\n        if selected:\n            phase[\'_child_scalar_context\'] = (g, F, phase[\'held_gap\'][15])\n        try:\n            return prior_post(b, f, g, payloads, F, H, parent,\n                              reverse, incoming, defer, phase)\n        finally:\n            if selected:\n                del phase[\'_child_scalar_context\']\n    scope[\'direct_post\'] = post\n    return cv\n\n\ndef finish_reverse_gap(b, m, g, F, phase):\n    if m+1 in ZERO_HANDOFF_LEVELS:\n        phase.setdefault(\'_undo_child_flags\', {})[m] = F\n    else:\n        clear_flag(b, g[-1], F)\n\n\ndef receive_reverse_gap(b, m, f, F, phase):\n    if m not in ZERO_HANDOFF_LEVELS:\n        return\n    H = phase[\'_undo_child_flags\'].pop(m-1)\n    Z0 = b.alloc_qubits(1)[0]; b.init_and(F, H, Z0)\n    clear_flag(b, f[-3], H)\n    phase.setdefault(\'_undo_parent_zeros\', {})[m] = Z0\n\n\ndef handed_aligned_pre(b, f, g, payloads, F, incoming, defer, Z0):\n    b.cx(Z0, g[-2][0]); masks = high_full_restore(b, f, g, F)\n    b.cx(Z0, g[-2][0]); pivot(b, g, F); a = g[-1]\n    ai, Z = inverse_zero(b, a); b.cx(Z, Z0); b.free(Z0)\n    consume(b, ai, Z, incoming)\n    b.cx(Z, a[0]); b.cx(Z, ai[0])\n    high_close_heads(b, f, g, ai, F, masks)\n    for v in payloads: dk.shifted_product(b, g[:-1], v[-1], v[:-1], F)\n    for v in payloads: scale(b, v[-1], a, ai, False)\n    b.cx(Z, ai[0]); b.cx(Z, a[0]); out = erase(b, a, ai, Z, defer)\n    pivot(b, g, F)\n    return out\n\n\ndef handed_plain_pre(b, f, g, payloads, incoming, defer, Z0):\n    a = g[-1]\n    b.cx(Z0, a[0]); masks = full_restore(b, f, g); b.cx(Z0, a[0])\n    ai, Z = inverse_zero(b, a); b.cx(Z, Z0); b.free(Z0)\n    consume(b, ai, Z, incoming)\n    b.cx(Z, a[0]); b.cx(Z, ai[0]); close_heads(b, f, g, ai, masks)\n    for v in payloads:\n        for i in range(len(g)-1): ar.mul_xor0(b, g[i], v[-1], v[i])\n    for v in payloads: scale(b, v[-1], a, ai, False)\n    b.cx(Z, ai[0]); b.cx(Z, a[0])\n    return erase(b, a, ai, Z, defer)\n\n\ndef base_with_returned_zero(b, f, g, payloads, H, parent, reverse=False, phase=None):\n    a = g[0]; q = f[-1]\n    def first(undo, inc=None, delay=False):\n        def action(ai):\n            runops(b, [(\'s\', v[0], True) for v in payloads]+[(\'s\', q, True)], a, ai, undo)\n        if not undo or 2 not in ZERO_HANDOFF_LEVELS:\n            return inverse_apply(b, a, action, inc, delay)\n        ai, Z = inverse_zero(b, a); consume(b, ai, Z, inc)\n        b.cx(Z, a[0]); b.cx(Z, ai[0]); action(ai)\n        b.cx(Z, ai[0]); b.cx(Z, a[0])\n        mask = b.alloc_bits(12); b.hmr(ai, mask); b.free(ai)\n        phase_inverse_for_gap(b, a, mask); b.free(mask)\n        phase.setdefault(\'_undo_child_flags\', {})[1] = Z\n    def last(undo, inc=None, delay=False):\n        def action(ai):\n            runops(b, [(\'s\', v[0], True) for v in payloads]+[(\'s\', q, False)], a, ai, undo)\n        return inverse_apply(b, a, action, inc, delay)\n    if not reverse:\n        carry = first(False, delay=True)\n        q_post(b, q, payloads, H, 1, parent, phase=phase)\n        phase[\'a\'][1] = last(False, carry, True)\n    else:\n        carry = last(True, phase[\'a\'].pop(1), True)\n        q_post(b, q, payloads, H, 1, parent, True, phase)\n        first(True, carry)\n\n\ndef install_undo_zeros(cv, levels=range(2,17)):\n    cells = dict(zip(cv.__code__.co_freevars, cv.__closure__))\n    short = cells[\'core\'].cell_contents.__globals__\n    scope = short[\'old\'].direct_post.__globals__\n    scope[\'ZERO_HANDOFF_LEVELS\'] = frozenset(levels)\n    scope[\'phase_inverse_for_gap\'] = short[\'phase_inverse34\']\n    for function in (finish_reverse_gap, receive_reverse_gap,\n                     handed_aligned_pre, handed_plain_pre, base_with_returned_zero):\n        scope[function.__name__] = clone(function, scope)\n    for name, newname in [(\'look_solve\', \'handoff_direct_solve\'),\n                          (\'old_look_solve\', \'handoff_generic_solve\')]:\n        source = inspect.getsource(scope[name])\n        needle = \'if reverse:clear_flag(b,g[-1],F)\'\n        assert source.count(needle) == 1\n        source = source.replace(needle, \'if reverse:finish_reverse_gap(b,m,g,F,phase)\')\n        if name == \'look_solve\':\n            needle = " else:F=phase[\'returned_gap\'].pop(m)\\n aligned(b,f,g,F)"\n            replacement = " else:F=phase[\'returned_gap\'].pop(m)\\n if reverse:receive_reverse_gap(b,m,f,F,phase)\\n aligned(b,f,g,F)"\n            call = \'direct_pre(b,f,g,payloads,F,True,incoming=carry)\'\n            newcall = "direct_pre(b,f,g,payloads,F,True,incoming=carry,handoff_zero=phase.get(\'_undo_parent_zeros\',{}).pop(m,None))"\n        else:\n            needle = "    else:F=phase[\'returned_gap\'].pop(m)\\n    canonical_permute(b,f,g,F,True)"\n            replacement = "    else:F=phase[\'returned_gap\'].pop(m)\\n    if reverse:receive_reverse_gap(b,m,f,F,phase)\\n    canonical_permute(b,f,g,F,True)"\n            call = \'pre(b,f,g,payloads,True,incoming=carry)\'\n            newcall = "pre(b,f,g,payloads,True,incoming=carry,handoff_zero=phase.get(\'_undo_parent_zeros\',{}).pop(m,None))"\n        assert source.count(needle) == source.count(call) == 1\n        source = source.replace(needle, replacement).replace(call, newcall)\n        source = source.replace(\'def look_solve(\', \'def \'+newname+\'(\', 1)\n        exec(compile(source, __file__, \'exec\'), scope)\n        scope[name] = scope[newname]\n    original, plain_original = scope[\'direct_pre\'], scope[\'pre\']\n    handed, plain_handed = scope[\'handed_aligned_pre\'], scope[\'handed_plain_pre\']\n    def direct_pre(b, f, g, payloads, F, undo=False, incoming=None,\n                   defer=False, handoff_zero=None):\n        if handoff_zero is None:\n            return original(b, f, g, payloads, F, undo, incoming, defer)\n        assert undo\n        return handed(b, f, g, payloads, F, incoming, defer, handoff_zero)\n    def pre(b, f, g, payloads, undo=False, incoming=None, defer=False, handoff_zero=None):\n        if handoff_zero is None:\n            return plain_original(b, f, g, payloads, undo, incoming, defer)\n        assert undo\n        return plain_handed(b, f, g, payloads, incoming, defer, handoff_zero)\n    scope[\'direct_pre\'] = direct_pre; scope[\'pre\'] = pre\n    scope[\'base_case\'] = scope[\'base_with_returned_zero\']\n    short[\'old\'].look_solve = scope[\'look_solve\']\n    return cv\n\n\ndef base_with_fused_zero(b, f, g, payloads, H, parent, reverse=False, phase=None):\n    a = g[0]; q = f[-1]\n    def first(undo, inc=None, delay=False):\n        def action(ai):\n            runops(b, [(\'s\', v[0], True) for v in payloads]+[(\'s\', q, True)], a, ai, undo)\n        if not undo:\n            return inverse_apply(b, a, action, inc, delay)\n        ai, Z = inverse_zero(b, a); consume(b, ai, Z, inc)\n        b.cx(Z, a[0]); b.cx(Z, ai[0]); action(ai)\n        b.cx(Z, ai[0]); b.cx(Z, a[0])\n        F = phase[\'returned_gap\'][2]\n        Z0 = b.alloc_qubits(1)[0]; b.init_and(F, Z, Z0)\n        erase_product_zero(b, a, ai, Z)\n        phase.setdefault(\'_undo_parent_zeros\', {})[2] = Z0\n        mask = b.alloc_bits(12); b.hmr(ai, mask); b.free(ai)\n        phase_inverse_for_gap(b, a, mask); b.free(mask)\n    def last(undo, inc=None, delay=False):\n        def action(ai):\n            runops(b, [(\'s\', v[0], True) for v in payloads]+[(\'s\', q, False)], a, ai, undo)\n        return inverse_apply(b, a, action, inc, delay)\n    if not reverse:\n        carry = first(False, delay=True)\n        q_post(b, q, payloads, H, 1, parent, phase=phase)\n        phase[\'a\'][1] = last(False, carry, True)\n    else:\n        carry = last(True, phase[\'a\'].pop(1), True)\n        q_post(b, q, payloads, H, 1, parent, True, phase)\n        first(True, carry)\n\n\ndef install_base_zero(cv, keep=(2, 3)):\n    cells = dict(zip(cv.__code__.co_freevars, cv.__closure__))\n    short = cells[\'core\'].cell_contents.__globals__\n    scope = short[\'old\'].direct_post.__globals__\n    assert 2 in keep and 2 in scope[\'ZERO_HANDOFF_LEVELS\']\n    scope[\'KEEP_CHILD\'] = short[\'KEEP_CHILD\'] = frozenset(keep)\n    scope[\'erase_product_zero\'] = short[\'erase_product_zero\']\n    scope[\'base_case\'] = clone(base_with_fused_zero, scope)\n    original_receive = scope[\'receive_reverse_gap\']\n    def receive(b, m, f, F, phase):\n        if m == 2:\n            assert 2 in phase[\'_undo_parent_zeros\']\n            return\n        original_receive(b, m, f, F, phase)\n    scope[\'receive_reverse_gap\'] = receive\n    return cv\n\n\ndef dot_cache_core(b,A,B,reverse=False,history_scratch12=None,phase_token=None,*,monic_root=False):\n    assert len(A)==len(B)==35\n    phase={\'a\':{},\'q\':{},\'_short_monic\':bool(monic_root)} if phase_token is None else phase_token\n    assert not reverse or phase_token is not None\n    old.frob(b,B,2048);old.frob(b,A,2048)\n    E=B[::2];O=B[1::2];u=A[1::2];w=A[:34:2];top=A[-1]\n    def bez(rev):\n        if phase[\'_short_monic\']:_bez_monic(b,O,E,[u,w],rev,history_scratch12,phase)\n        else:old.bez_lookahead(b,O,E,[u,w],rev,history_scratch12,phase)\n    if not reverse:\n        for i in range(17):old.ar.mul_xor0(b,top,O[i],w[i])\n        bez(False)\n        delta=b.alloc_qubits(12)\n        for i in range(17):old.ar.mul_xor0(b,E[i+1],w[i],delta)\n        b.cx(delta,top);phase[\'_top_dot\']=delta\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n        old.hankel(b,B[1:],A[:34])\n        for i in reversed(range(34)):b.swap(A[i],A[i+1])\n    else:\n        for i in range(34):b.swap(A[i],A[i+1])\n        # Top is outside the 34-word Hankel. Free delta before that map.\n        delta=phase.pop(\'_top_dot\');b.cx(delta,top)\n        mask=b.alloc_bits(12);b.hmr(delta,mask);b.free(delta)\n        old.hankel(b,B[1:],A[:34],True)\n        for i in range(17):b.swap(A[2*i],A[2*i+1])\n        for i in range(17):old.ar.phase_product(b,E[i+1],w[i],mask)\n        b.free(mask)\n        bez(True)\n        for i in range(17):old.ar.mul_xor0(b,top,O[i],w[i])\n    old.frob(b,A,2);old.frob(b,B,2)\n    if reverse:assert not phase[\'a\'] and not phase[\'q\']\n    return phase if not reverse else None\n\n\n@lru_cache(None)\ndef restricted_tensor16(ar):\n    k,nx,width,extension=3,2,4,3\n    f,sub=ar.field(12),ar.field(width)\n    def product(a,b):\n        out=[0]*(len(a)+len(b)-1)\n        for i,x in enumerate(a):\n            for j,y in enumerate(b):out[i+j]^=sub.mul(x,y)\n        return out\n    def remainder(a,p):\n        out=list(a)\n        for i in range(len(out)-len(p),-1,-1):\n            c=out[i+len(p)-1]\n            for j,v in enumerate(p):out[i+j]^=sub.mul(c,v)\n        return (out+[0]*(len(p)-1))[:len(p)-1]\n    def evaluate(a,x):\n        out=0\n        for c in reversed(a):out=sub.mul(out,x)^c\n        return out\n    def pack(a):return sum(c<<(width*i) for i,c in enumerate(a))\n    def unpack(value,n):return [(value>>(width*i))&15 for i in range(n)]\n    generator=f.pow(f.primitive_element,273)\n    def binary_eval(p,x):\n        out=0\n        for i in reversed(range(width+1)):out=f.mul(out,x)^((p>>i)&1)\n        return out\n    root=next(f.pow(generator,i) for i in range(1,16)\n              if binary_eval(sub.modulus,f.pow(generator,i))==0)\n    embed=[f.pow(root,i) for i in range(width)]\n    alpha=f.primitive_element\n    to=ar.inverse_columns([f.mul(value,f.pow(alpha,j)) for j in range(extension) for value in embed])\n    p=unpack(ar.apply(to,f.pow(alpha,extension)),extension)+[1]\n    modulus=[1]\n    for _ in range(k):modulus=product(modulus,p)\n    theta=[0,1]\n    for _ in range(12):theta=remainder(product(theta,theta),modulus)\n    powers=[[1],theta,remainder(product(theta,theta),modulus)]\n    test=[0]*(extension*k)\n    power=[1]\n    for c in p:\n        for i,value in enumerate(power):test[i]^=sub.mul(c,value)\n        power=remainder(product(power,theta),modulus)\n    assert not any(test)\n    epsilon=list(theta);epsilon[1]^=1\n    epowers=[[1]]\n    for _ in range(k):epowers.append(remainder(product(epowers[-1],epsilon),modulus))\n    assert not any(epowers[k])\n    # Frozen exact unit from the constant nullspace over this field basis.\n    assert p==[2,13,7,1]\n    bound=2*k;u=[0,8,1]\n    assert any(remainder(u,p))\n    for value in powers:assert not any(remainder(product(u,value),modulus)[bound+1:])\n    columns=[]\n    for degree in range(k):\n        for bit in range(12):\n            value=[0]*(extension*k)\n            for c,theta_power in zip(unpack(to[bit],extension),powers):\n                for i,a in enumerate(theta_power):value[i]^=sub.mul(c,a)\n            columns.append(pack(remainder(product(value,epowers[degree]),modulus)))\n    weighted=[pack(remainder(product(unpack(value,extension*k),u),modulus)) for value in columns]\n    inverse=ar.inverse_columns(weighted)\n    assert all(not (value>>(width*(bound+nx))) for value in weighted[:12*nx])\n    left_degree=max((value.bit_length()-1)//width for value in weighted[:12*nx] if value)\n    right_degree=max((value.bit_length()-1)//width for value in columns if value)\n    points=list(range(left_degree+right_degree+1))\n    assert len(points)<=16\n    local=((1,9),(12,3),(13,10),(15,14),(4,2),(11,12),(14,1),(8,6),(6,7))\n    for i in range(4):\n        for j in range(4):\n            check=0\n            for mask,vector in local:\n                if (mask>>i)&(mask>>j)&1:check^=vector\n            assert check==sub.mul(1<<i,1<<j)\n    terms=[]\n    for point in points:\n        lag,denominator=[1],1\n        for other in points:\n            if other==point:continue\n            lag=product(lag,[other,1]);denominator=sub.mul(denominator,point^other)\n        lag=[sub.mul(a,sub.invert(denominator)) for a in lag]\n        left=[evaluate(unpack(value,extension*k),point) for value in weighted[:12*nx]]\n        right=[evaluate(unpack(value,extension*k),point) for value in columns]\n        for mask,vector in local:\n            am=sum(((a&mask).bit_count()&1)<<i for i,a in enumerate(left))\n            bm=sum(((a&mask).bit_count()&1)<<i for i,a in enumerate(right))\n            output=ar.apply(inverse,pack(remainder([sub.mul(a,vector) for a in lag],modulus)))\n            if am and bm and output:terms.append((am,bm,output))\n    merged={}\n    for a,b,value in terms:merged[a,b]=merged.get((a,b),0)^value\n    return tuple((a,b,value) for (a,b),value in merged.items() if value)\n\n\n@lru_cache(None)\ndef weighted_tensor64(ar, k, nx):\n    if (k, nx) == (3, 2):\n        return restricted_tensor16(ar)\n    def product(a, b, f):\n        out = [0]*(len(a)+len(b)-1)\n        for i, x in enumerate(a):\n            for j, y in enumerate(b): out[i+j] ^= f.mul(x, y)\n        return out\n\n\n    def remainder(a, p, f):\n        out = list(a)\n        for i in range(len(out)-len(p), -1, -1):\n            c = out[i+len(p)-1]\n            for j, v in enumerate(p): out[i+j] ^= f.mul(c, v)\n        return (out+[0]*(len(p)-1))[:len(p)-1]\n\n\n    def evaluate(a, x, f):\n        out = 0\n        for c in reversed(a): out = f.mul(out, x) ^ c\n        return out\n\n\n    def pack(a): return sum(c << (6*i) for i, c in enumerate(a))\n    def unpack(x, n): return [(x >> (6*i)) & 63 for i in range(n)]\n    assert 2<=k<=9 and 1<=nx<=k\n    f,sub=ar.field(12),ar.field(6)\n    generator=f.pow(f.primitive_element,65)\n    def binary_eval(p,x):\n        out=0\n        for i in reversed(range(7)):out=f.mul(out,x)^((p>>i)&1)\n        return out\n    root=next(f.pow(generator,i) for i in range(1,64)\n              if binary_eval(sub.modulus,f.pow(generator,i))==0)\n    embed=[f.pow(root,i) for i in range(6)];alpha=f.primitive_element\n    to=ar.inverse_columns(embed+[f.mul(v,alpha) for v in embed])\n    p=unpack(ar.apply(to,f.square(alpha)),2)+[1]\n    modulus=[1]\n    for _ in range(k):modulus=product(modulus,p,sub)\n    theta=[0,1]\n    for _ in range(12):theta=remainder(product(theta,theta,sub),modulus,sub)\n    epsilon=list(theta);epsilon[1]^=1\n    epowers=[[1]]\n    for _ in range(k):epowers.append(remainder(product(epowers[-1],epsilon,sub),modulus,sub))\n    assert not any(epowers[k])\n    # (Y+alpha)^k=v(Y)+alpha*u(Y), and theta*u=v modulo P^k.\n    u=[0]*(k+1);v=[0]*(k+1)\n    for j in range(k+1):\n        if (j&k)==j:v[j],u[j]=unpack(ar.apply(to,f.pow(alpha,k-j)),2)\n    assert any(remainder(u,p,sub))\n    assert remainder(product(theta,u,sub),modulus,sub)==remainder(v,modulus,sub)\n    columns=[]\n    for degree in range(k):\n        for bit in range(12):\n            c0,c1=unpack(to[bit],2)\n            value=[sub.mul(c1,a) for a in theta];value[0]^=c0\n            columns.append(pack(remainder(product(value,epowers[degree],sub),modulus,sub)))\n    weighted=[pack(remainder(product(unpack(value,2*k),u,sub),modulus,sub)) for value in columns]\n    inverse=ar.inverse_columns(weighted)\n    assert all(not (value>>(6*(k+nx))) for value in weighted[:12*nx])\n    left_degree=max((value.bit_length()-1)//6 for value in weighted[:12*nx] if value)\n    right_degree=max((value.bit_length()-1)//6 for value in columns if value)\n    points=list(range(left_degree+right_degree+1));terms=[]\n    assert len(points)<=64\n    for point in points:\n        lag,denominator=[1],1\n        for other in points:\n            if other==point:continue\n            lag=product(lag,[other,1],sub);denominator=sub.mul(denominator,point^other)\n        lag=[sub.mul(a,sub.invert(denominator)) for a in lag]\n        left=[evaluate(unpack(value,2*k),point,sub) for value in weighted[:12*nx]]\n        right=[evaluate(unpack(value,2*k),point,sub) for value in columns]\n        for mask,vector in ar.tensor(6):\n            am=sum(((a&mask).bit_count()&1)<<i for i,a in enumerate(left))\n            bm=sum(((a&mask).bit_count()&1)<<i for i,a in enumerate(right))\n            poly=[sub.mul(a,vector) for a in lag]\n            output=ar.apply(inverse,pack(remainder(poly,modulus,sub)))\n            if am and bm and output:terms.append((am,bm,output))\n    merged={}\n    for a,b,value in terms:merged[a,b]=merged.get((a,b),0)^value\n    return tuple((a,b,value) for (a,b),value in merged.items() if value)\n\n\ndef install_weighted_hankel(reference, cv, maximum=8, low_lengths=(3, 4)):\n    ar, k = reference.ar, reference.k\n    flat = lambda words: [bit for word in words for bit in word]\n    tensor = lambda degree, sources: weighted_tensor64(ar, degree, sources)\n\n    def accumulate(b, x, q, out, terms):\n        assert len(set(map(str, x+q+out))) == len(x)+len(q)+len(out)\n        for am, bm, vector in terms:\n            ix, iy, iz = [list(ar.bits(v)) for v in (am, bm, vector)]\n            p, r, t = ix[0], iy[0], iz[0]\n            for i in ix[1:]: b.cx(x[i], x[p])\n            for i in iy[1:]: b.cx(q[i], q[r])\n            for i in iz[1:]: b.cx(out[t], out[i])\n            b.ccx(x[p], q[r], out[t])\n            for i in reversed(iz[1:]): b.cx(out[t], out[i])\n            for i in reversed(iy[1:]): b.cx(q[i], q[r])\n            for i in reversed(ix[1:]): b.cx(x[i], x[p])\n\n    def erase_copies(b, q, x, copies):\n        masks = [b.alloc_bits(12) for _ in copies]\n        for source, mask in zip(copies, masks):\n            b.hmr(source, mask)\n            b.free(source)\n        b.cz(x[1], masks[0])\n        ar.phase_product(b, q[0], x[0], masks[0])\n        b.cz(x[2], masks[1])\n        ar.phase_product(b, q[0], x[1], masks[1])\n        ar.phase_product(b, q[1], x[0], masks[1])\n        ar.linear_inplace(b, q[0], ar.square_columns(12))\n        ar.phase_product(b, q[0], x[0], masks[1])\n        ar.linear_inplace(b, q[0], [ar.field(12).pow(1 << i, 2048) for i in range(12)])\n        for mask in masks: b.free(mask)\n\n    cells = dict(zip(cv.__code__.co_freevars, cv.__closure__))\n    scope = cells[\'core\'].cell_contents.__globals__\n    def original_tri(b, q, x, reverse=False):\n        assert not reverse and 2 <= len(x) <= 4\n        if len(x) == 2:\n            return ar.mul_xor0(b, q[0], x[0], x[1])\n        return hankel_direct(b, q, x)\n\n    @lru_cache(None)\n    def partition(n):\n        if n == 1: return 0, None\n        leaves = {2: 42, 3: 99, 4: 153, 5: 42+len(tensor(4, 3))}\n        if 6 <= n <= maximum:\n            leaves[n] = partition(n-3)[0]+len(tensor(n-1, 3))\n        choices = [(leaves[n], None)] if n in leaves else []\n        for left in range(1, n):\n            choices.append((partition(left)[0]+partition(n-left)[0]+42*(n-1), left))\n        return min(choices, key=lambda pair: pair[0])\n\n    def direct(b, q, x):\n        n = len(x)\n        if n == 5:\n            ar.mul_xor0(b, q[0], x[3], x[4])\n        else:\n            tri(b, q[:n-4], x[3:])\n        copies = [b.alloc_qubits(12) for _ in range(2)]\n        for source, target in zip(x[1:3], copies): b.cx(source, target)\n        accumulate(b, flat([x[0]]+copies), flat(q[:n-1]), flat(x[1:]), tensor(n-1, 3))\n        erase_copies(b, q, x, copies)\n\n    def tri(b, q, x, reverse=False):\n        assert not reverse\n        n = len(x)\n        if n == 1: return\n        _, left = partition(n)\n        if left is None:\n            if n >= 5: return direct(b, q, x)\n            return original_tri(b, q, x)\n        right = n-left\n        tri(b, q[:right-1], x[left:])\n        for ac, bc, tc in k.cross_terms(left, right):\n            k.rank_update(b, x[:left], ac, q[:n-1], bc, x[left:], tc)\n        tri(b, q[:left-1], x[:left])\n\n    def hankel(b, B, A, reverse=False):\n        if reverse: return reference.hankel(b, B, A, True)\n        return reference.clone(k.hankel, dict(vars(k), tri=tri))(b, B, A)\n    scope[\'old\'].hankel = hankel\n    direct_scope = scope[\'old\'].direct_post.__globals__\n    prior = direct_scope[\'convolution_xor\']\n\n    def convolution(b, x, y, out, reverse=False):\n        degree = len(out)\n        if degree in low_lengths and len(x) == degree and len(y) == 2:\n            return accumulate(b, flat(y), flat(x), flat(out), tensor(degree, 2))\n        return prior(b, x, y, out, reverse)\n    direct_scope[\'convolution_xor\'] = convolution\n    return cv\n\n\ndef install_dot_cache(cv):\n    cells = dict(zip(cv.__code__.co_freevars, cv.__closure__))\n    short = cells[\'core\'].cell_contents.__globals__\n    cells[\'core\'].cell_contents = clone(dot_cache_core, short)\n    reference = SimpleNamespace(ar=ar, k=k, hankel=hankel, clone=clone)\n    return install_weighted_hankel(reference, cv)\n\n\nconvert_safe = install_dot_cache(install_base_zero(install_undo_zeros(install_child_inverse(install_retained_gaps(make_converter())))))\nbaseline = SimpleNamespace(**dict(vars(base.ref.baseline), scan=scan))\ndecode = clone(base.decode, dict(base.decode.__globals__,\n                               convert_safe=convert_safe, baseline=baseline))\n\n\ndef make_prefix_boundary(original):\n    env=original.__globals__\n    folds=env[\'folds\'];sg=folds.sg[\'absorb\'].__globals__\n    old,dv=sg[\'old\'],sg[\'dv\']\n    c,kap=env[\'cs\'][1],env[\'ks\'][1]\n    def synthesis():\n        source=inspect.getsource(original)\n        needle=\'    for i in reversed(range(1,34)):\\n        aa,bb=A,B\\n\'\n        assert source.count(needle)==1\n        source=source.replace(needle,\'    for i in reversed(range(2,34)):\\n        aa,bb=A,B\\n\')\n        needle=\'    b.x(border[0]);b.free(border);C=B+[r]\\n\'\n        assert source.count(needle)==1\n        source=source.replace(needle,\'    C=A+[r];D=B+heads[1]\\n\')\n        needle="        folds.ug[\'reclaim\'](b,[border]);b.x(border[0])\\n"\n        assert source.count(needle)==1\n        source=source.replace(needle,needle+\'        ar.mul_xor0(b,heads[1][1],r,border)\\n\')\n        needle=\'    return C,A,undo\\n\'\n        assert source.count(needle)==1\n        source=source.replace(needle,\'    return C,D,border,undo\\n\')\n        scope=dict(original.__globals__)\n        exec(compile(source,__file__,\'exec\'),scope)\n        return scope[\'synthesize\']\n\n\n    def materialize(b,A,D,prefix):\n        low=A[:-1];Q=A[-1];B=D[:-2];head=D[-2:]\n        value=old.eval_implicit(b,low,Q,c)\n        F=dv.zero_flag(b,value)\n        old.eval_implicit(b,low,Q,c,True)\n        spare=[b.alloc_qubits(12),b.alloc_qubits(12)]\n        b.cx(B[-2],spare[0]);b.cx(B[-1],spare[0]);b.cx(B[-1],spare[1])\n        b.gf2_iadd(kap,target=spare[0]);b.x(spare[1][0])\n        sg[\'q_times_low\'](b,B,kap)\n        b.cx(F,low[0][0])\n        # Full product degree is at most 35. Its leading coefficient is\n        # head[1]*Q = prefix+1, so only 35 finite evaluations are needed.\n        targets=B+spare;n=len(targets)\n        V,W=k.vandermonde(n);field=ar.field(12)\n        for t in range(n):\n            old.rank_affine(b,head,[1,t],A,V[t][:len(A)],\n                                 targets,[W[j][t] for j in range(n)])\n        for j in range(n):\n            co=0\n            for t in range(n):co^=field.mul(W[j][t],field.pow(t,n))\n            if j==n-2:co^=kap\n            if j==n-1:co^=1\n            if co:\n                k.cx(b,prefix,targets[j],co)\n                b.gf2_iadd(co,target=targets[j])\n        b.cx(F,low[0][0])\n        for bit in ar.bits(kap):b.cx(F,spare[0][bit])\n        b.cx(F,spare[1][0])\n        pm=b.alloc_bits(12);b.hmr(prefix,pm);b.free(prefix)\n        b.z(pm[0]);ar.phase_product(b,head[1],Q,pm);b.free(pm)\n        hm=b.alloc_bits(24);b.hmr(head[0]+head[1],hm)\n        for source,new in zip(head,spare):b.swap(source,new);b.free(new)\n        fm=b.alloc_bits(1);b.hmr(F,fm[0]);b.free(F)\n        return hm,fm\n\n    return synthesis(),materialize\n\n\ndef install_factorized_boundary(reference_decode):\n    from exp.decoder.common import ar\n    from exp.convert import kernels as k\n    from exp.noscratch.schedules import norm_xor_noscratch\n    from exp.mul99.schedules import multiply_forward99\n    from exp.arithmetic.division import phase_product24,zero_flag\n    from exp.inverse19_joint.kernel import inverse_initialized\n    from exp.fold_cached_norm.phase34 import phase_inverse34\n\n    original=reference_decode.__globals__[\'synthesize\']\n    env=original.__globals__\n    folds=env[\'folds\']\n    sg=folds.sg[\'absorb\'].__globals__\n    old,dv=sg[\'old\'],sg[\'dv\']\n    c,kap=env[\'cs\'][1],env[\'ks\'][1]\n    K=sg[\'K\']\n\n\n    synthesize, materialize = make_prefix_boundary(original)\n\n    def divide_cached(b,x,y,N,F,fm,after):\n        \'Divide by clipped y with its raw norm already held; close all signs.\'\n        b.cx(F,y[0]);b.cx(F,N[0])\n        b.cx(y[12:],y[:12])\n        oldmask=multiply_forward99(b,x,y,K)\n        b.cx(y[12:],y[:12])\n        ai=inverse_initialized(b,N)\n        for offset in (0,12):\n            part=x[offset:offset+12];out=b.alloc_qubits(12)\n            ar.mul_xor0(b,part,ai,out)\n            ar.erase_product(b,out,N,part)\n            b.swap(part,out);b.free(out)\n        phase_product24(b,x,y,oldmask,K);b.free(oldmask)\n        b.cx(F,y[0]);b.cx(F,N[0])\n        after()\n        mask=b.alloc_bits(1);b.hmr(F,mask[0]);b.free(F)\n        b.cx(fm[0],mask[0]);b.free(fm);b.z(mask[0])\n        pm=b.alloc_bits(12);b.bit_store(pm,False);b.cx(mask[0],pm[0])\n        ar.phase_product(b,N,ai,pm);b.free(pm);b.free(mask)\n        im=b.alloc_bits(12);b.hmr(ai,im);b.free(ai)\n        phase_inverse34(b,N,im,zero_mask=im[0]);b.free(im)\n        ar._erase_norm(b,y,N,K);b.free(N)\n\n\n    def dematerialize_cached(b,A,D,token):\n        head=D[-2:];B=D[:-2];low=A[:-1];Q=A[-1]\n        g=folds.ug[\'absorb\'].__globals__\n        g[\'R_to_tops\'](b,head,B,c,True)\n        v=g[\'epow\'](c,len(B)+2);co=[v&4095,v>>12]\n        one=[True]+[False]*11\n        old.add_alias(b,one,head,co)\n        value=old.eval_implicit(b,low,Q,c)\n        N=b.alloc_qubits(12);norm_xor_noscratch(b,value,N,K)\n        F=zero_flag(b,N)\n        hm,fm=token\n        old.controlled_linear_xor(b,F,one,head,co)\n        def after():\n            k.cx(b,head[1],head[0],c)\n            old.eval_implicit(b,low,Q,c,True)\n            b.cx(F,low[0][0]);g[\'low_product\'](b,head,low,B);b.cx(F,low[0][0])\n            g[\'q_times_low\'](b,B,kap,True)\n            old.eval_implicit(b,low,Q,c)\n        divide_cached(b,head[0]+head[1],value,N,F,fm,after)\n        b.cz(head[0]+head[1],hm);b.free(hm)\n        old.eval_implicit(b,low,Q,c,True)\n\n\n    def make_decoder():\n        g=reference_decode.__globals__\n        undo_materialize=dematerialize_cached\n        def decode(b,S,E):\n            with b.mark(\'mask_projection\'):mt=g[\'baseline\'].project(b,S,E)\n            with b.mark(\'initial\'):g[\'linear_words\'](b,S,g[\'matrix\'])\n            with b.mark(\'extract\'):et=g[\'extraction\'](b,S)\n            with b.mark(\'synthesis\'):A,B,prefix,undo=synthesize(b,S,et[\'pole\'])\n            with b.mark(\'materialize_boundary\'):ft=materialize(b,A,B,prefix)\n            with b.mark(\'convert\'):ct=g[\'convert_safe\'](b,A,B,monic_root=True)\n            with b.mark(\'scan\'):g[\'baseline\'].scan(b,A,B,E,skip_points=g[\'baseline\'].SELECTED,node_polynomial=g[\'Pscan\'])\n            with b.mark(\'unconvert\'):g[\'convert_safe\'](b,A,B,True,phase_token=ct)\n            with b.mark(\'dematerialize_boundary\'):undo_materialize(b,A,B,ft)\n            with b.mark(\'unsynthesis\'):undo()\n            with b.mark(\'unextract\'):g[\'extraction\'](b,S,True,et)\n            with b.mark(\'uninitial\'):g[\'linear_words\'](b,S,g[\'matrix\'],True)\n            with b.mark(\'restore_masks\'):g[\'baseline\'].restore(b,E,mt)\n        return decode\n\n\n    return make_decoder()\n\n\ndecode = install_factorized_boundary(decode)\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a01/ported_exact_cycle.py': '"""Carry-over converter plus combo exact substitutions, cycle-safe scope clone."""\nfrom pathlib import Path\nimport sys\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport ported as p\nfrom rebind_graph import private_rebind\nconvert_safe = private_rebind(p.convert_safe, p.base.replacements, p.ROOT)\ndecode = p.clone(p.decode, dict(p.decode.__globals__, convert_safe=convert_safe))\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a01/rebind_graph.py': '"""Private rebinding for recursive function/closure graphs.\n\nOnly Python construction changes: no circuit operation or simulator state is\nused. New closure cells are memoized with the function before their values are\ncopied, so mutually recursive kernel schedules retain the same topology.\n"""\nfrom contextlib import contextmanager\nfrom pathlib import Path\nfrom types import CellType, CodeType, FunctionType, ModuleType, SimpleNamespace\n\n\ndef private_rebind(entry, replacements, root):\n    root = Path(root).resolve()\n    memo, scopes, processed = {}, {}, set()\n\n    def owned(filename):\n        try:\n            return Path(filename).resolve().is_relative_to(root)\n        except (TypeError, ValueError):\n            return False\n\n    def names(code):\n        result = set(code.co_names)\n        for value in code.co_consts:\n            if isinstance(value, CodeType):\n                result.update(names(value))\n        return result\n\n    def copy(value):\n        key = id(value)\n        if key in replacements:\n            return replacements[key]\n        if key in memo:\n            return memo[key]\n        if isinstance(value, FunctionType) and owned(value.__code__.co_filename):\n            old = value.__globals__\n            scope = scopes.setdefault(id(old), dict(old))\n            closure = None if value.__closure__ is None else tuple(\n                CellType() for _ in value.__closure__)\n            out = FunctionType(value.__code__, scope, value.__name__,\n                               value.__defaults__, closure)\n            memo[key] = out\n            out.__kwdefaults__ = value.__kwdefaults__\n            out.__annotations__ = dict(value.__annotations__)\n            if closure is not None:\n                for target, original in zip(closure, value.__closure__):\n                    try:\n                        contents = original.cell_contents\n                    except ValueError:\n                        continue\n                    target.cell_contents = copy(contents)\n            for name in names(value.__code__):\n                token = id(old), name\n                if token not in processed and name in old:\n                    processed.add(token)\n                    scope[name] = copy(old[name])\n            return out\n        if isinstance(value, FunctionType) and getattr(value, \'__wrapped__\', None) is not None:\n            wrapped = value.__wrapped__\n            if isinstance(wrapped, FunctionType) and owned(wrapped.__code__.co_filename):\n                out = contextmanager(copy(wrapped))\n                memo[key] = out\n                return out\n        if isinstance(value, SimpleNamespace) or (\n                isinstance(value, ModuleType) and owned(getattr(value, \'__file__\', None))):\n            out = ModuleType(value.__name__) if isinstance(value, ModuleType) else SimpleNamespace()\n            memo[key] = out\n            for name, item in vars(value).items():\n                if name != \'__builtins__\':\n                    setattr(out, name, copy(item))\n            return out\n        if isinstance(value, dict):\n            out = {}\n            memo[key] = out\n            out.update((k, copy(v)) for k, v in value.items())\n            return out\n        if isinstance(value, (list, tuple)) and any(\n                isinstance(v, (FunctionType, SimpleNamespace, dict)) for v in value):\n            out = [copy(v) for v in value]\n            out = tuple(out) if isinstance(value, tuple) else out\n            memo[key] = out\n            return out\n        return value\n\n    return copy(entry)\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a02/cache_support.py': '"""Exact retained three-bit inverse-tower caches in selected converter frames.\n\nNo decision depends on quantum values or checker samples. The cache is\ninv0(Norm_GF64/GF8(Norm_GF4096/GF64(a))) in fixed tower coordinates.\n"""\nfrom contextlib import contextmanager\nfrom pathlib import Path\nimport importlib.util\nimport inspect\nimport sys\n\nROOT = Path(__file__).resolve().parents[3]\nsys.path.insert(0, str(ROOT/\'research/agents/a01\'))\nimport ported as p\nfrom exp.inverse19_joint import kernel as joint\nfrom exp.noscratch.schedules import norm_xor_noscratch\nfrom exp.noscratch.kernels import mul_xor_noscratch\nfrom exp.arithmetic.lowwidth import classical_transpose\nfrom exp.decoder.common import ar\nfrom exp.jet24 import pro_jet\nimport exact_subs as exact\n\n\ndef close_spilled_phase(b,x,mask,zero_mask,debt):\n    """Close the ordinary inverse and a measured inner-cache character jointly."""\n    source=inspect.getsource(exact.phase_inverse12_virtual_nonzero)\n    source=source.replace(\'def phase_inverse12_virtual_nonzero(b, x, mask, zero_mask=None):\',\n                          \'def joint_phase(b, x, mask, zero_mask=None, debt=None):\')\n    needle=\'    w = ref.inverse3_virtual(b, prefix, forms)\\n\'\n    assert source.count(needle)==1\n    source=source.replace(needle,needle+\'    if debt is not None:b.cz(w,debt)\\n\')\n    env=dict(exact.phase_inverse12_virtual_nonzero.__globals__)\n    exec(compile(source,__file__,\'exec\'),env)\n    active=b.alloc_bits(1)[0];b.bit_store(active,False)\n    for bit in mask:b.bit_store(active,True,control=bit)\n    if zero_mask is not None:b.bit_store(active,True,control=zero_mask)\n    if debt is not None:\n        for bit in debt:b.bit_store(active,True,control=bit)\n    b.push_condition(active);env[\'joint_phase\'](b,x,mask,zero_mask,debt)\n    b.pop_condition();b.free(active)\n    if debt is not None:b.free(debt)\n\n\ndef inverse6_keep_cache(b,x):\n    """The supplied initialized inverse, retaining its existing inner inverse."""\n    to,frm,K=ar.tower(6);ar.linear_inplace(b,x,to);g=b.alloc_qubits(6)\n    for i,(l,r) in enumerate(joint.basic.raw_forms19()):\n        with joint.ref.expose_pair(b,x,l,r) as (u,v):b.init_and(u,v,g[i])\n    prefix=list(x)+list(g);forms=joint.ref.norm_forms(K)\n    w=joint.ref.inverse3_virtual(b,prefix,forms)\n    ar.linear_inplace(b,g,joint.LAY[\'raw_columns\']);R,E=g[:4],g[4:]\n    z=list(E)+list(b.alloc_qubits(4));source=list(x)+list(R)+list(w)\n    for j,e in enumerate(joint.LAY[\'outputs\']):joint.ref.form_xor(b,source,e&65535,z[j])\n    for i,(l,r) in enumerate(joint.LAY[\'gates\']):\n        v=sum(((e>>(16+i))&1)<<j for j,e in enumerate(joint.LAY[\'outputs\']))\n        with joint.ref.expose_pair(b,source,l,r) as (u,vv):joint.rankone_xor(b,u,vv,z,v)\n    rm=b.alloc_bits(4);b.hmr(R,rm);b.free(R)\n    for i,(l,r) in enumerate(joint.basic.raw_forms19()):\n        with joint.ref.expose_pair(b,x,l,r) as (u,v):\n            for j,row in enumerate(joint.LAY[\'raw_rows\'][:4]):\n                if row>>i&1:b.ccz(u,v,rm[j])\n    b.free(rm);ar.linear_inplace(b,z,joint.LAY[\'output_inverse_columns\'])\n    F=zero_from_cache(b,w)\n    ar.linear_inplace(b,x,frm);ar.linear_inplace(b,z,frm)\n    return z,F,w\n\n\ndef zero_from_cache(b,w):\n    b.x(w);u=b.alloc_qubits(1)[0];F=b.alloc_qubits(1)[0]\n    b.init_and(w[0],w[1],u);b.init_and(u,w[2],F)\n    b.del_and(w[0],w[1],u);b.free(u);b.x(w)\n    return F\n\n\ndef inverse6_keep18(b,x):\n    """Keep a copy of the existing 18-AND producer\'s inner inverse prefix."""\n    ar.linear_inplace(b,x,pro_jet.TO6)\n    raw=b.alloc_qubits(6)\n    for j,(l,r) in enumerate(pro_jet.RAW):\n        with joint.ref.expose_pair(b,x,l,r) as (u,v):b.init_and(u,v,raw[j])\n    w=joint.ref.inverse3_virtual(b,list(x)+list(raw),joint.ref.norm_forms(1))\n    cache=b.alloc_qubits(3);b.cx(w,cache)\n    raw=list(raw)+list(w);out=[]\n    for kind,a,c,d in pro_jet.PLAN:\n        if kind==\'r\':\n            ar.linear_inplace(b,raw,a)\n            for j,action in enumerate(d,c):\n                if action is None:\n                    out.append(raw[j]);continue\n                coeff,phase=action\n                for k in ar.bits(coeff):b.cx(raw[j],out[k])\n                m=b.alloc_bits(1)[0];b.hmr(raw[j],m)\n                live=list(x)+raw[:c]+raw[j+1:]+out\n                for i,k in phase:\n                    if i==0:b.z(m)\n                    elif k==0 or i==k:b.cz(live[i-1],m)\n                    else:b.ccz(live[i-1],live[k-1],m)\n                b.free(m);b.free(raw[j])\n            raw=list(raw[:c])\n        elif kind==\'g\':out.extend(b.alloc_qubits(a))\n        else:\n            with joint.ref.expose_pair(b,list(x)+raw,a,c) as (u,v):\n                joint.rankone_xor(b,u,v,out,d)\n    ar.linear_inplace(b,out,pro_jet.OUTPUT)\n    for j,form in enumerate(pro_jet.LINEAR):joint.ref.form_xor(b,x,form,out[j])\n    ar.linear_inplace(b,x,pro_jet.FROM6);ar.linear_inplace(b,out[:6],pro_jet.FROM6)\n    return out[:6],out[6],cache\n\n\ndef inverse_zero_cached(b,x,w=None,producer18=False):\n    """Return inv0(x), [x=0], and the unchanged three-bit inverse cache."""\n    to,frm,K=ar.tower(12);ar.linear_inplace(b,x,to)\n    N=b.alloc_qubits(6);norm_xor_noscratch(b,x,N,K)\n    if w is None:\n        t,F,w=(inverse6_keep18 if producer18 else inverse6_keep_cache)(b,N)\n    else:\n        to6,frm6,_=ar.tower(6);ar.linear_inplace(b,N,to6)\n        t=b.alloc_qubits(6)\n        b.cx(N[3:],N[:3]);mul_xor_noscratch(b,N[:3],w,t[:3])\n        mul_xor_noscratch(b,N[3:],w,t[3:]);b.cx(N[3:],N[:3])\n        ar.linear_inplace(b,N,frm6);ar.linear_inplace(b,t,frm6)\n        F=zero_from_cache(b,w)\n    ar._erase_norm(b,x,N,K);b.free(N)\n    out=b.alloc_qubits(12);b.cx(x[6:],x[:6])\n    mul_xor_noscratch(b,x[:6],t,out[:6]);mul_xor_noscratch(b,x[6:],t,out[6:])\n    b.cx(x[6:],x[:6]);ar._erase_norm(b,out,t,K);b.free(t)\n    ar.linear_inplace(b,x,frm);ar.linear_inplace(b,out,frm)\n    return out,F,w\n\n\ndef close_cached_phase(b,x,mask,zero_mask,w):\n    """Repay inverse/zero characters and retire the cache with its own character."""\n    to12,frm12,K12=ar.tower(12);ar.linear_inplace(b,x,to12)\n    mt=classical_transpose(b,mask,frm12)\n    D=b.alloc_qubits(6);norm_xor_noscratch(b,x,D,K12)\n    to6,frm6,K6=ar.tower(6);ar.linear_inplace(b,D,to6)\n    anc=b.alloc_qubits(1)[0];f6=ar.field(6)\n    def coefficient(h):\n        for j in range(6):\n            for r in ar.bits(f6.mul(h,1<<j)):\n                b.ccx(mt[r],x[j],anc);b.ccx(mt[r],x[j+6],anc)\n                b.ccx(mt[r+6],x[j+6],anc)\n    for segment in (0,1):\n        if segment==0:b.cx(D[3:],D[:3])\n        controls=D[:3] if segment==0 else D[3:]\n        for m,v in ar.tensor(3):\n            h=ar.apply(frm6,v<<(3*segment));ix=list(ar.bits(m));q=ix[0]\n            coefficient(h)\n            for j in ix[1:]:b.cx(controls[j],controls[q]);b.cx(w[j],w[q])\n            b.ccz(controls[q],w[q],anc)\n            for j in reversed(ix[1:]):b.cx(w[j],w[q]);b.cx(controls[j],controls[q])\n            coefficient(h)\n        if segment==0:b.cx(D[3:],D[:3])\n    b.free(anc)\n    if zero_mask is not None:\n        b.x(w);b.push_condition(zero_mask);b.ccz(w[0],w[1],w[2]);b.pop_condition();b.x(w)\n    E=b.alloc_qubits(3);norm_xor_noscratch(b,D,E,K6)\n    ar.erase_inverse(b,E,w);b.free(w)\n    ar._erase_norm(b,D,E,K6);b.free(E)\n    ar.linear_inplace(b,D,frm6);ar._erase_norm(b,x,D,K12);b.free(D)\n    ar.linear_inplace(b,x,frm12);b.free(mt)\n\n\ndef install(cv,levels=(14,),trace=False,split=False,producer18=False,spill=False):\n    """Retain one cache per listed degree between first pre and final undo pre."""\n    cells=dict(zip(cv.__code__.co_freevars,cv.__closure__))\n    short=cells[\'core\'].cell_contents.__globals__\n    scope=short[\'old\'].direct_post.__globals__\n    levels=frozenset(levels)\n    frames=[]\n    old_inverse=scope[\'inverse_zero\'];old_erase=scope[\'erase\']\n    old_gap_phase=scope[\'phase_inverse_for_gap\']\n    old_solve=scope[\'look_solve\']\n    def selected(x):\n        if not frames:return None\n        frame=frames[-1]\n        return frame if frame[0] in levels and list(x)==list(frame[2][-1]) else None\n    def inverse_zero(b,x):\n        frame=selected(x)\n        if frame is None:return old_inverse(b,x)\n        m,phase,g,reverse=frame\n        cache=phase.setdefault(\'_inverse3_cache\',{})\n        if trace:print(\'CACHE-INVERSE\',m,len(cache),b.num_allocated_qubits,file=sys.stderr)\n        ai,F,w=inverse_zero_cached(b,x,cache.get(m),producer18)\n        cache[m]=w\n        if (spill or split and reverse) and m in phase.get(\'_inverse3_debt\',{}):\n            debt=phase[\'_inverse3_debt\'].pop(m);b.cz(w,debt);b.free(debt)\n        return ai,F\n    def erase(b,a,ai,F,defer):\n        frame=selected(a)\n        if frame is None:return old_erase(b,a,ai,F,defer)\n        m,phase,g,reverse=frame\n        mask=b.alloc_bits(12);zm=b.alloc_bits(1)\n        b.hmr(ai,mask);b.hmr(F,zm[0]);b.free(ai);b.free(F)\n        if defer:return mask,zm\n        w=phase[\'_inverse3_cache\'].pop(m,None)\n        if w is not None:close_cached_phase(b,a,mask,zm[0],w)\n        else:close_spilled_phase(b,a,mask,zm[0],phase[\'_inverse3_debt\'].pop(m,None))\n        b.free(mask);b.free(zm)\n    def inverse_apply_safe(b,a,action,incoming=None,defer=False):\n        ai,F=inverse_zero(b,a);scope[\'consume\'](b,ai,F,incoming)\n        b.cx(F,a[0]);b.cx(F,ai[0]);action(ai)\n        b.cx(F,ai[0]);b.cx(F,a[0])\n        return erase(b,a,ai,F,defer)\n    def gap_phase(b,a,mask,zero_mask=None):\n        frame=selected(a)\n        if frame is None:return old_gap_phase(b,a,mask,zero_mask)\n        m,phase,g,reverse=frame\n        w=phase[\'_inverse3_cache\'].pop(m,None)\n        if w is not None:close_cached_phase(b,a,mask,zero_mask,w)\n        else:close_spilled_phase(b,a,mask,zero_mask,phase[\'_inverse3_debt\'].pop(m,None))\n    def retire_for_workspace(b):\n        if not frames:return\n        m,phase,g,reverse=frames[-1]\n        w=phase.get(\'_inverse3_cache\',{}).pop(m,None)\n        if w is None:return\n        debt=b.alloc_bits(3);b.hmr(w,debt);b.free(w)\n        assert m not in phase.setdefault(\'_inverse3_debt\',{})\n        phase[\'_inverse3_debt\'][m]=debt\n    def solve(b,f,g,payloads,H=None,parent=None,reverse=False,phase=None):\n        m=len(f)-1\n        frames.append((m,phase,g,reverse))\n        try:\n            answer=old_solve(b,f,g,payloads,H,parent,reverse,phase)\n            if split and not reverse and m in levels and m in phase.get(\'_inverse3_cache\',{}):\n                cache=phase[\'_inverse3_cache\'].pop(m)\n                debt=b.alloc_bits(3);b.hmr(cache,debt);b.free(cache)\n                phase.setdefault(\'_inverse3_debt\',{})[m]=debt\n            return answer\n        finally:frames.pop()\n    scope.update(inverse_zero=inverse_zero,erase=erase,inverse_apply_safe=inverse_apply_safe,\n                 look_solve=solve,phase_inverse_for_gap=gap_phase)\n    if spill:\n        for name in [\'scale\',\'high_close_heads\',\'close_heads\']:\n            old=scope[name]\n            def call(b,*args,_old=old,**kw):\n                retire_for_workspace(b)\n                return _old(b,*args,**kw)\n            scope[name]=call\n        short[\'old\'].scale=scope[\'scale\']\n    short[\'old\'].look_solve=solve\n    return cv\n\n\ndef decoder(levels,trace=False,split=False,exact=False,producer18=False,spill=False):\n    cv=install(p.convert_safe,levels,trace,split,producer18,spill)\n    if exact:\n        from rebind_graph import private_rebind\n        cv=private_rebind(cv,p.base.replacements,p.ROOT)\n    return p.clone(p.decode,dict(p.decode.__globals__,convert_safe=cv))\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a03/local_norm.py': '"""Keep selected pivot-norm coordinates only across the final undo action.\n\nThe saved coordinates are copied from a norm that the ordinary producer\nalready computed.  The final inverse character reconstructs only its missing\ncoordinates, then erases the complete norm with its exact quadratic phase.\n"""\nfrom pathlib import Path\nfrom contextlib import contextmanager\nimport inspect\nimport sys\n\nROOT = Path(__file__).resolve().parents[3]\nsys.path.insert(0, str(ROOT / \'research/agents/a01\'))\nimport ported_exact_cycle as baseline\nimport exact_subs as xs\nimport norm_cache as nc\nfrom exp.noscratch.schedules import norm_xor_noscratch\nfrom exp.noscratch.kernels import mul_xor_noscratch\n\np = baseline.p\nar = p.ar\nTRACE = []\n\n\ndef inverse_zero_kept(b, x, count):\n    to, frm, kval = ar.tower(12)\n    ar.linear_inplace(b, x, to)\n    norm = b.alloc_qubits(6)\n    norm_xor_noscratch(b, x, norm, kval)\n    projection, _ = nc.layout(count)\n    saved = b.alloc_qubits(count)\n    ar.linear_xor(b, norm, saved, [v & ((1 << count)-1) for v in projection])\n    invnorm, zero = xs.inverse6_joint18_12extra(b, norm)\n    ar._erase_norm(b, x, norm, kval)\n    b.free(norm)\n    out = b.alloc_qubits(12)\n    b.cx(x[6:], x[:6])\n    mul_xor_noscratch(b, x[:6], invnorm, out[:6])\n    mul_xor_noscratch(b, x[6:], invnorm, out[6:])\n    b.cx(x[6:], x[:6])\n    ar._erase_norm(b, out, invnorm, kval)\n    b.free(invnorm)\n    ar.linear_inplace(b, x, frm)\n    ar.linear_inplace(b, out, frm)\n    return out, zero, saved\n\n\ndef make_cached_phase():\n    source = inspect.getsource(xs.phase_inverse12_virtual_nonzero)\n    source = source.replace(\'def phase_inverse12_virtual_nonzero(b, x, mask, zero_mask=None):\',\n                            \'def phase_inverse_cached(b, x, mask, saved, zero_mask=None):\')\n    needle = \'    D = b.alloc_qubits(6)\\n    norm_xor_noscratch(b, x, D, K)\'\n    assert source.count(needle) == 1\n    source = source.replace(needle, \'    D = rebuild_norm(b, x, saved, K)\')\n    scope = dict(xs.phase_inverse12_virtual_nonzero.__globals__, rebuild_norm=nc.rebuild)\n    exec(compile(source, __file__, \'exec\'), scope)\n    return scope[\'phase_inverse_cached\']\n\n\nphase_inverse_cached = make_cached_phase()\n\n\ndef erase_kept(b, x, inv, zero, saved):\n    mask = b.alloc_bits(12)\n    zm = b.alloc_bits(1)\n    b.hmr(inv, mask)\n    b.hmr(zero, zm[0])\n    b.free(inv)\n    b.free(zero)\n    phase_inverse_cached(b, x, mask, saved, zm[0])\n    b.free(mask)\n    b.free(zm)\n\n\ndef make_decoder(count=1, levels=range(6, 17), quotient_count=0, plain_count=0,\n                 plain_levels=range(2, 6), factorized=False, converter=None,\n                 reference_decode=None):\n    cv = baseline.convert_safe if converter is None else converter\n    cells = dict(zip(cv.__code__.co_freevars, cv.__closure__))\n    short = cells[\'core\'].cell_contents.__globals__\n    scope = short[\'old\'].direct_post.__globals__\n    dispatcher = scope[\'direct_pre\']\n    d_cells = dict(zip(dispatcher.__code__.co_freevars, dispatcher.__closure__))\n    original = d_cells[\'handed\'].cell_contents\n    scope = original.__globals__\n    levels = frozenset(levels)\n\n    def handed(b, f, g, payloads, flag, incoming, defer, z0):\n        if len(f)-1 not in levels:\n            return original(b, f, g, payloads, flag, incoming, defer, z0)\n        assert not defer\n        b.cx(z0, g[-2][0])\n        masks = scope[\'high_full_restore\'](b, f, g, flag)\n        b.cx(z0, g[-2][0])\n        scope[\'pivot\'](b, g, flag)\n        a = g[-1]\n        ai, zero, saved = inverse_zero_kept(b, a, count)\n        b.cx(zero, z0)\n        b.free(z0)\n        scope[\'consume\'](b, ai, zero, incoming)\n        b.cx(zero, a[0])\n        b.cx(zero, ai[0])\n        scope[\'high_close_heads\'](b, f, g, ai, flag, masks)\n        for v in payloads:\n            scope[\'dk\'].shifted_product(b, g[:-1], v[-1], v[:-1], flag)\n        for v in payloads:\n            scope[\'scale\'](b, v[-1], a, ai, False)\n        b.cx(zero, ai[0])\n        b.cx(zero, a[0])\n        erase_kept(b, a, ai, zero, saved)\n        scope[\'pivot\'](b, g, flag)\n\n    # install_undo_zeros captures this function directly in its dispatcher.\n    d_cells[\'handed\'].cell_contents = handed\n    scope[\'handed_aligned_pre\'] = handed\n\n    if quotient_count:\n        old_q_inverse = scope[\'q_inverse\']\n\n        @contextmanager\n        def q_inverse(b, a, m, reverse, phase):\n            if not reverse or m == 16:\n                with old_q_inverse(b, a, m, reverse, phase) as ai:\n                    yield ai\n                return\n            ai, zero, saved = inverse_zero_kept(b, a, quotient_count)\n            scope[\'consume\'](b, ai, zero, phase[\'q\'].pop(m))\n            b.cx(zero, a[0]); b.cx(zero, ai[0])\n            yield ai\n            b.cx(zero, ai[0]); b.cx(zero, a[0])\n            erase_kept(b, a, ai, zero, saved)\n\n        scope[\'q_inverse\'] = q_inverse\n\n    if plain_count:\n        plain_levels = frozenset(plain_levels)\n        plain_dispatcher = scope[\'pre\']\n        plain_cells = dict(zip(plain_dispatcher.__code__.co_freevars,\n                               plain_dispatcher.__closure__))\n        plain_original = plain_cells[\'plain_handed\'].cell_contents\n\n        def plain(b, f, g, payloads, incoming, defer, z0):\n            if len(f)-1 not in plain_levels:\n                return plain_original(b, f, g, payloads, incoming, defer, z0)\n            assert not defer\n            a = g[-1]\n            b.cx(z0, a[0]); masks = scope[\'full_restore\'](b, f, g); b.cx(z0, a[0])\n            ai, zero, saved = inverse_zero_kept(b, a, plain_count)\n            b.cx(zero, z0); b.free(z0)\n            scope[\'consume\'](b, ai, zero, incoming)\n            b.cx(zero, a[0]); b.cx(zero, ai[0])\n            scope[\'close_heads\'](b, f, g, ai, masks)\n            for v in payloads:\n                for i in range(len(g)-1): ar.mul_xor0(b, g[i], v[-1], v[i])\n            for v in payloads: scope[\'scale\'](b, v[-1], a, ai, False)\n            b.cx(zero, ai[0]); b.cx(zero, a[0])\n            erase_kept(b, a, ai, zero, saved)\n\n        plain_cells[\'plain_handed\'].cell_contents = plain\n        scope[\'handed_plain_pre\'] = plain\n    reference = baseline.decode if reference_decode is None else reference_decode\n    decode = p.clone(reference, dict(reference.__globals__, convert_safe=cv))\n    if not factorized:\n        return decode\n    from rebind_graph import private_rebind\n    return private_rebind(p.install_factorized_boundary(decode),\n                          p.base.replacements, p.ROOT)\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a07/blocked_three_noscratch.py': '"""Exact three-frame forward payload fusion in a retained middle basis.\n\nCoefficient high heads are processed normally. The third long coefficient\nupdate is postponed until all low payload updates have been applied in the\nmiddle polynomial pair. Every flag chart has its full polynomial weight.\nThe ordinary reverse converter consumes the unchanged transcript interface.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[3]\nsys.path[:0]=[str(ROOT/\'research/agents/a01\'),str(ROOT/\'research/agents/a03\')]\nimport ported as p\nfrom rebind_graph import private_rebind\nfrom exp.mul99.phase36 import phase_classical_parity\n\n\ndef mask_word(b,x,F,G,not_f=False,not_g=False):\n    if not_f:b.x(F)\n    if not_g:b.x(G)\n    h=b.alloc_qubits(1)[0];b.init_and(F,G,h)\n    out=b.alloc_qubits(12)\n    for a,c in zip(x,out):b.init_and(h,a,c)\n    b.del_and(F,G,h);b.free(h)\n    if not_g:b.x(G)\n    if not_f:b.x(F)\n    return out\n\n\ndef erase_mask(b,x,out,F,G,not_f=False,not_g=False):\n    r=b.alloc_bits(12);b.hmr(out,r);b.free(out)\n    if not_f:b.x(F)\n    if not_g:b.x(G)\n    phase_classical_parity(b,F,G,x,r)\n    if not_g:b.x(G)\n    if not_f:b.x(F)\n    b.free(r)\n\n\ndef copy_flag(b,F,x,y,complement=False):\n    if complement:b.x(F)\n    for a,c in zip(x,y):b.ccx(F,a,c)\n    if complement:b.x(F)\n\n\ndef phase_flag(b,F,x,r,complement=False):\n    if complement:b.x(F)\n    for a,c in zip(x,r):b.ccz(F,a,c)\n    if complement:b.x(F)\n\n\ndef payload_middle(b,lo_f,lo_g,heads,out,q,F0,F1,F2):\n    """Accumulate all three deferred low updates, for every flag pattern.\n\n    heads=(s0,s1,s2), q=(q10,q11). P has degree<=1, Q degree<=2.\n    Each ephemeral word is measured only while its defining sources survive.\n    """\n    s0,s1,s2=heads;q0,q1=q\n    # M=(1+F0)(1+F1)s0. Its AND flag is retired before both outputs exist.\n    M=mask_word(b,s0,F0,F1,True,True)\n    P0=b.alloc_qubits(12);ar.mul_xor0(b,q0,M,P0)\n    P1=b.alloc_qubits(12);ar.mul_xor0(b,q1,M,P1)\n    b.cx(M,P0);b.cx(M,P1)\n    copy_flag(b,F0,s0,P0,True);copy_flag(b,F1,s1,P0,True)\n    copy_flag(b,F1,s0,P1,True)\n    convolution(b,lo_f,[P0,P1],out)\n    r0=b.alloc_bits(12);r1=b.alloc_bits(12)\n    b.hmr(P0,r0);b.hmr(P1,r1);b.free(P0);b.free(P1)\n    ar.phase_product(b,q0,M,r0);ar.phase_product(b,q1,M,r1)\n    b.cz(M,r0);b.cz(M,r1)\n    phase_flag(b,F0,s0,r0,True);phase_flag(b,F1,s1,r0,True)\n    phase_flag(b,F1,s0,r1,True)\n    b.free(r0);b.free(r1);erase_mask(b,s0,M,F0,F1,True,True)\n\n    # M=(1+F0)F1s0. It becomes Q0 during the convolution, then is restored.\n    M=mask_word(b,s0,F0,F1,True,False)\n    Q1=b.alloc_qubits(12);ar.mul_xor0(b,q0,M,Q1)\n    Q2=b.alloc_qubits(12);ar.mul_xor0(b,q1,M,Q2)\n    b.cx(M,Q1)\n    copy_flag(b,F1,s0,Q1);copy_flag(b,F1,s1,Q1);copy_flag(b,F2,s2,Q1)\n    copy_flag(b,F0,s0,M,True);copy_flag(b,F2,s2,M,True)\n    convolution(b,lo_g,[M,Q1,Q2],out)\n    copy_flag(b,F2,s2,M,True);copy_flag(b,F0,s0,M,True)\n    r1=b.alloc_bits(12);r2=b.alloc_bits(12)\n    b.hmr(Q1,r1);b.hmr(Q2,r2);b.free(Q1);b.free(Q2)\n    ar.phase_product(b,q0,M,r1);ar.phase_product(b,q1,M,r2);b.cz(M,r1)\n    phase_flag(b,F1,s0,r1);phase_flag(b,F1,s1,r1);phase_flag(b,F2,s2,r1)\n    b.free(r1);b.free(r2);erase_mask(b,s0,M,F0,F1,True,False)\n\n\ndef pre_partial(b,f,g,payloads,F,cutoff,defer_coefficient=False):\n    pivot(b,g,F);a=g[-1];q0,q1=f[-2:]\n    def action(ai):\n        for v in payloads:scale(b,v[-1],a,ai,True)\n        # Only the future head strip is needed before the block\'s low update.\n        for i in range(cutoff,len(g)-1):\n            with dk.selected(b,g[i],g[i-1],F) as coefficient:\n                for v in payloads:ar.mul_xor0(b,coefficient,v[-1],v[i])\n        scale(b,q1,a,ai,True);cross(b,g,q1,q0,F);scale(b,q0,a,ai,True)\n        if not defer_coefficient:dk.shifted_conv(b,g[:-1],[q0,q1],f[:-2],F)\n    result=inverse_apply(b,a,action,None,True)\n    pivot(b,g,F)\n    return result\n\n\ndef look_solve(b,f,g,payloads,H=None,parent=None,reverse=False,phase=None):\n    m=len(f)-1\n    if reverse or m not in BLOCK_LEVELS:\n        return ordinary_solve(b,f,g,payloads,H,parent,reverse,phase)\n    assert m-2>=THRESHOLD and not any(n in KEEP_CHILD for n in (m,m-1,m-2))\n    cutoff=m-3;frames=[]\n    for i in range(3):\n        n=len(f)-1;F=fresh_zero_flag(b,g[-1])\n        carry=pre_partial(b,f,g,payloads,F,cutoff,i==2)\n        frames.append((n,f,g,payloads,H,parent,F,carry))\n        if i==2:break\n        aligned(b,f,g,F)\n        phase[\'incoming_gap\'][n]=hold_flag(b,F)\n        childparent=([v[-1]for v in payloads],f[-1],f[-2])\n        f,g,payloads,H,parent=g,f[:-2],[v[:-1]for v in payloads],None,childparent\n    # These are the same predicates as the two already measured parent flags.\n    F0=fresh_zero_flag(b,frames[1][1][-1])\n    F1=fresh_zero_flag(b,frames[2][1][-1])\n    F2=frames[2][6]\n    for j in range(len(payloads)):\n        heads=[frame[3][j][-1] for frame in frames]\n        payload_middle(b,f[:cutoff],g[:cutoff],heads,payloads[j][:cutoff],\n                       frames[1][1][-2:],F0,F1,F2)\n    for flag,n in ((F0,m),(F1,m-1)):\n        mask=hold_flag(b,flag)\n        b.cx(mask,phase[\'incoming_gap\'][n]);b.free(mask)\n    # Finish the postponed third coefficient division in its original frame.\n    pivot(b,g,F2)\n    dk.shifted_conv(b,g[:-1],f[-2:],f[:-2],F2)\n    pivot(b,g,F2)\n    aligned(b,f,g,F2)\n    phase[\'incoming_gap\'][m-2]=hold_flag(b,F2)\n    childparent=([v[-1]for v in payloads],f[-1],f[-2])\n    look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,childparent,False,phase)\n    for n,f,g,payloads,H,parent,unused,carry in reversed(frames):\n        F=phase[\'returned_gap\'].pop(n)\n        aligned(b,f,g,F)\n        phase[\'a\'][n]=direct_post(b,f,g,payloads,F,H,parent,\n                                 incoming=carry,defer=True,phase=phase)\n        if n in KEEP_FULL:phase[\'held_gap\'][n]=F\n        else:phase[\'gap_flags\'][n]=hold_flag(b,F)\n\n\ndef make_decoder(levels=(16,)):\n    cv=p.install_dot_cache(p.install_base_zero(p.install_undo_zeros(\n        p.install_child_inverse(p.install_retained_gaps(p.make_converter())))))\n    cells=dict(zip(cv.__code__.co_freevars,cv.__closure__))\n    short=cells[\'core\'].cell_contents.__globals__\n    scope=short[\'old\'].direct_post.__globals__\n    scope.update(BLOCK_LEVELS=frozenset(levels),ordinary_solve=scope[\'look_solve\'],\n                 convolution=lambda b,x,y,out: scope[\'dk\'].conv(b,x,y,out,len(x)+len(y)-1),phase_classical_parity=phase_classical_parity)\n    for fun in (mask_word,erase_mask,copy_flag,phase_flag,payload_middle,pre_partial,look_solve):\n        scope[fun.__name__]=p.clone(fun,scope)\n    short[\'old\'].look_solve=scope[\'look_solve\']\n    cv=private_rebind(cv,p.base.replacements,p.ROOT)\n    import local_norm\n    return local_norm.make_decoder(count=1,plain_count=1,plain_levels=(3,4,5),\n                                   converter=cv,reference_decode=p.decode,factorized=True)\n\n\ndecode=make_decoder()\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a07/ephemeral_root_block.py': '"""Retire the root inverse during forward descent; rebuild it at child return.\n\nThe returned child still holds its quotient head q and raw divisor g.\nWith F=[q=0], H=[g[-1]=0], K the next held gap, and clipped selected\nchild pivot a_c, the clipped original root pivot is\n    R=q*a_c + F*g[-1] + F*H.\nIt is nonzero on every completed coefficient bank. The original root\ninverse is therefore inv(R), and its measured character is consumed there.\n"""\nimport inspect\n\n\ndef restore_root(b,f,g,Fchild,phase):\n    held,mask=phase.pop(\'_ephemeral_root\')\n    q=f[-1]\n    Froot=fresh_zero_flag(b,q)\n    K=phase[\'held_gap\'][15]\n    R=b.alloc_qubits(12)\n    pivot(b,g,Fchild)\n    b.ccx(Fchild,K,g[-1][0])\n    ar.mul_xor0(b,q,g[-1],R)\n    b.ccx(Fchild,K,g[-1][0])\n    pivot(b,g,Fchild)\n    for x,y in zip(g[-1],R):b.ccx(Froot,x,y)\n    b.ccx(Froot,Fchild,R[0])\n    phase[\'_root_early_flag\']=Froot\n    replacement=inverse_initialized(b,R)\n    rm=b.alloc_bits(12);b.hmr(R,rm);b.free(R)\n    phase[\'_root_pivot_debt\']=rm\n    held[:]=replacement\n    b.cz(held,mask);b.free(mask)\n\n\n\ndef root_pre_partial(b,f,g,payloads,ai,Z,F):\n    a=g[-1];q0,q1=f[-2:];b.cx(Z,a[0])\n    for v in payloads:old.scale(b,v[-1],a,ai,True)\n    cutoff=len(g)-3\n    for i in range(cutoff,len(g)-1):\n        with dk.selected(b,g[i],g[i-1],F) as coefficient:\n            for v in payloads:ar.mul_xor0(b,coefficient,v[-1],v[i])\n    with dk.selected(b,g[-2],g[-3],F) as c:ar.mul_xor0(b,q1,c,q0)\n    old.scale(b,q0,a,ai,True)\n    dk.shifted_conv(b,g[:-1],[q0,q1],f[:-2],F)\n    b.cx(Z,a[0])\n\ndef install(cv,p):\n    cells=dict(zip(cv.__code__.co_freevars,cv.__closure__))\n    short=cells[\'core\'].cell_contents.__globals__\n    scope=short[\'old\'].direct_post.__globals__\n    scope[\'phase_inverse_for_root\']=short[\'phase_inverse34\']\n    scope[\'restore_root\']=p.clone(restore_root,scope)\n    original=scope[\'direct_post\']\n    def direct_post(b,f,g,payloads,F,H,parent,reverse=False,incoming=None,defer=False,phase=None):\n        if not reverse and len(f)-1==16 and \'_ephemeral_root\' in phase:\n            scope[\'restore_root\'](b,f,g,F,phase)\n        result=original(b,f,g,payloads,F,H,parent,reverse,incoming,defer,phase)\n        if not reverse and len(f)-1==16 and \'_root_pivot_debt\' in phase:\n            debt=phase.pop(\'_root_pivot_debt\');Froot=phase[\'returned_gap\'][17]\n            b.cz(f[-1],debt)\n            for x,y in zip(g[-1],debt):b.ccz(Froot,x,y)\n            b.ccz(Froot,F,debt[0]);b.free(debt)\n        return result\n    scope[\'direct_post\']=direct_post\n    qsource=inspect.getsource(scope[\'q_post\'])\n    needle=\'        F = fresh_zero_flag(b, q)\'\n    assert qsource.count(needle)==1\n    qsource=qsource.replace(needle,"        F = phase.pop(\'_root_early_flag\') if \'_root_early_flag\' in phase else fresh_zero_flag(b, q)")\n    exec(compile(qsource,__file__,\'exec\'),scope)\n    scope[\'q_post\']=scope[\'root_child_post\']\n    short[\'root_pre_partial\']=p.clone(root_pre_partial,short)\n    source=inspect.getsource(short[\'look_root_cached\'])\n    needle=\'    old.look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,parent,reverse,phase)\'\n    assert source.count(needle)==1\n    replacement="""    if not reverse:\n        root_mask=b.alloc_bits(12);b.hmr(ai,root_mask);b.free(ai)\n        phase[\'_ephemeral_root\']=(ai,root_mask)\n"""+needle\n    source=source.replace(needle,replacement)\n    needle=\'        root_action(b,f,g,payloads,ai,Z,F,post=False,undo=False)\'\n    assert source.count(needle)==1\n    source=source.replace(needle,\'        root_pre_partial(b,f,g,payloads,ai,Z,F)\')\n    needle=\'    parent=([v[-1]for v in payloads],f[-1],f[-2])\'\n    assert source.count(needle)==1\n    source=source.replace(needle,needle+"\\n    if not reverse:phase[\'_root_block_payloads\']=payloads")\n    exec(compile(source,__file__,\'exec\'),short)\n    return cv\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a07/middle_fusion.py': '"""Install the exact root/16/15 forward fusion on a private carried converter.\n\nThis is the composable installation interface. The three-frame arithmetic is\nfrozen in blocked_three_noscratch.py and root_fused_three.py. See PROOF.md for\nthe completed polynomial recurrence and the four/eight-frame handoff.\n\nInstall before private_rebind, and before or after other independent forward\nHankel installers. The reverse converter and its transcript interface remain\nthe original carried converter\'s.\n"""\nfrom pathlib import Path\nimport sys\n\nHERE = Path(__file__).resolve().parent\nsys.path.insert(0, str(HERE))\nimport blocked_three_noscratch as arithmetic\nfrom root_fused_three import look_solve\nfrom ephemeral_root_block import install as install_root_lifetime\n\n\ndef install(cv):\n    p = arithmetic.p\n    cells = dict(zip(cv.__code__.co_freevars, cv.__closure__))\n    short = cells[\'core\'].cell_contents.__globals__\n    scope = short[\'old\'].direct_post.__globals__\n    scope.update(\n        ordinary_solve=scope[\'look_solve\'],\n        convolution=lambda b, x, y, out: scope[\'dk\'].conv(\n            b, x, y, out, len(x) + len(y) - 1),\n        phase_classical_parity=arithmetic.phase_classical_parity,\n    )\n    for fun in (\n        arithmetic.mask_word, arithmetic.erase_mask,\n        arithmetic.copy_flag, arithmetic.phase_flag,\n        arithmetic.payload_middle, arithmetic.pre_partial, look_solve,\n    ):\n        scope[fun.__name__] = p.clone(fun, scope)\n    short[\'old\'].look_solve = scope[\'look_solve\']\n    install_root_lifetime(cv, p)\n    return cv\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a07/root_fused_three.py': '"""Exact middle fusion of the root pre and its first two child pres."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[3]\nsys.path[:0]=[str(Path(__file__).resolve().parent),str(ROOT/\'research/agents/a01\'),\n              str(ROOT/\'research/agents/a03\'),str(ROOT/\'research/agents/a09\')]\nimport blocked_three_noscratch as base\np=base.p\nfrom rebind_graph import private_rebind\n\n\ndef look_solve(b,f,g,payloads,H=None,parent=None,reverse=False,phase=None):\n    if reverse or \'_root_block_payloads\' not in phase:\n        return ordinary_solve(b,f,g,payloads,H,parent,reverse,phase)\n    m=len(f)-1;assert m==16\n    root_payloads=phase.pop(\'_root_block_payloads\')\n    cutoff=m-2;frames=[]\n    for i in range(2):\n        n=len(f)-1;F=fresh_zero_flag(b,g[-1])\n        carry=pre_partial(b,f,g,payloads,F,cutoff,i==1)\n        frames.append((n,f,g,payloads,H,parent,F,carry))\n        if i==1:break\n        aligned(b,f,g,F)\n        phase[\'incoming_gap\'][n]=hold_flag(b,F)\n        childparent=([v[-1]for v in payloads],f[-1],f[-2])\n        f,g,payloads,H,parent=g,f[:-2],[v[:-1]for v in payloads],None,childparent\n    F0=fresh_zero_flag(b,frames[0][1][-1])\n    F1=fresh_zero_flag(b,frames[1][1][-1])\n    F2=frames[1][6]\n    for j in range(len(payloads)):\n        heads=[root_payloads[j][-1]]+[frame[3][j][-1] for frame in frames]\n        payload_middle(b,f[:cutoff],g[:cutoff],heads,payloads[j][:cutoff],\n                       frames[0][1][-2:],F0,F1,F2)\n    for flag,n in ((F0,m+1),(F1,m)):\n        mask=hold_flag(b,flag)\n        b.cx(mask,phase[\'incoming_gap\'][n]);b.free(mask)\n    pivot(b,g,F2);dk.shifted_conv(b,g[:-1],f[-2:],f[:-2],F2);pivot(b,g,F2)\n    aligned(b,f,g,F2);phase[\'incoming_gap\'][m-1]=hold_flag(b,F2)\n    childparent=([v[-1]for v in payloads],f[-1],f[-2])\n    look_solve(b,g,f[:-2],[v[:-1]for v in payloads],None,childparent,False,phase)\n    for n,f,g,payloads,H,parent,unused,carry in reversed(frames):\n        F=phase[\'returned_gap\'].pop(n);aligned(b,f,g,F)\n        phase[\'a\'][n]=direct_post(b,f,g,payloads,F,H,parent,\n                                 incoming=carry,defer=True,phase=phase)\n        if n in KEEP_FULL:phase[\'held_gap\'][n]=F\n        else:phase[\'gap_flags\'][n]=hold_flag(b,F)\n\n\ndef make_decoder(hankel=False):\n    cv=p.install_dot_cache(p.install_base_zero(p.install_undo_zeros(\n        p.install_child_inverse(p.install_retained_gaps(p.make_converter())))))\n    cells=dict(zip(cv.__code__.co_freevars,cv.__closure__))\n    short=cells[\'core\'].cell_contents.__globals__\n    scope=short[\'old\'].direct_post.__globals__\n    scope.update(ordinary_solve=scope[\'look_solve\'],\n                 convolution=lambda b,x,y,out: scope[\'dk\'].conv(b,x,y,out,len(x)+len(y)-1),\n                 phase_classical_parity=base.phase_classical_parity)\n    for fun in (base.mask_word,base.erase_mask,base.copy_flag,base.phase_flag,\n                base.payload_middle,base.pre_partial,look_solve):\n        scope[fun.__name__]=p.clone(fun,scope)\n    short[\'old\'].look_solve=scope[\'look_solve\']\n    from ephemeral_root_block import install\n    install(cv,p)\n    if hankel:\n        from descending_overlap import install as install_hankel\n        install_hankel(cv,maximum=18,max_sources=8)\n    cv=private_rebind(cv,p.base.replacements,p.ROOT)\n    import local_norm\n    return local_norm.make_decoder(count=1,plain_count=1,plain_levels=(3,4,5),\n                                   converter=cv,reference_decode=p.decode,factorized=True)\n\n\ndecode=make_decoder()\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a08/scalar_compact_v3.py': '"""Exact local norm-pair retention in the reverse converter.\n\nOnly the A01 port, its private graph copier, and unchanged kit arithmetic\nare dependencies. No research diagnostic or losing candidate is imported.\n"""\nfrom pathlib import Path\nfrom contextlib import contextmanager\nimport sys\nROOT=Path(__file__).resolve().parents[3]\nsys.path.insert(0,str(ROOT/\'research/agents/a01\'))\nimport ported as p\nfrom rebind_graph import private_rebind\nimport exact_subs as xs\nfrom exp.noscratch.schedules import norm_xor_noscratch\nfrom exp.noscratch.kernels import mul_xor_noscratch\nfrom exp.arithmetic.lowwidth import classical_transpose\nar=p.ar\n\n\ndef prefix(b,x):\n    to,frm,K=ar.tower(12);ar.linear_inplace(b,x,to)\n    norm=b.alloc_qubits(6);norm_xor_noscratch(b,x,norm,K)\n    invnorm,Z=xs.inverse6_joint18_12extra(b,norm)\n    ar.linear_inplace(b,x,frm)\n    return Z,(norm,invnorm)\n\n\ndef finish(b,x,saved):\n    norm,invnorm=saved\n    to,frm,K=ar.tower(12);ar.linear_inplace(b,x,to)\n    out=b.alloc_qubits(12);b.cx(x[6:],x[:6])\n    mul_xor_noscratch(b,x[:6],invnorm,out[:6])\n    mul_xor_noscratch(b,x[6:],invnorm,out[6:])\n    b.cx(x[6:],x[:6]);ar.linear_inplace(b,x,frm);ar.linear_inplace(b,out,frm)\n    return out\n\n\ndef inverse_zero_normpair(b,x):\n    to,frm,K=ar.tower(12);ar.linear_inplace(b,x,to)\n    norm=b.alloc_qubits(6);norm_xor_noscratch(b,x,norm,K)\n    invnorm,Z=xs.inverse6_joint18_12extra(b,norm)\n    out=b.alloc_qubits(12);b.cx(x[6:],x[:6])\n    mul_xor_noscratch(b,x[:6],invnorm,out[:6])\n    mul_xor_noscratch(b,x[6:],invnorm,out[6:])\n    b.cx(x[6:],x[:6]);ar.linear_inplace(b,x,frm);ar.linear_inplace(b,out,frm)\n    return out,Z,(norm,invnorm)\n\n\ndef erase_normpair(b,x,ai,Z,saved,handoff=None):\n    norm,invnorm=saved\n    mask=b.alloc_bits(12);zm=b.alloc_bits(1)\n    b.hmr(ai,mask)\n    if handoff is not None:\n        b.free(ai);F,phase=handoff\n        Zparent=b.alloc_qubits(1)[0];b.init_and(F,Z,Zparent)\n        phase.setdefault(\'_undo_parent_zeros\',{})[2]=Zparent\n        b.hmr(Z,zm[0]);b.free(Z)\n    else:\n        b.hmr(Z,zm[0]);b.free(ai);b.free(Z)\n    to,frm,K=ar.tower(12);ar.linear_inplace(b,x,to)\n    mt=classical_transpose(b,mask,frm)\n    b.cx(x[6:],x[:6])\n    ar.phase_product(b,x[:6],invnorm,mt[:6])\n    ar.phase_product(b,x[6:],invnorm,mt[6:])\n    b.cx(x[6:],x[:6]);b.free(mt);b.free(mask)\n    b.z(zm[0]);zero_mask=b.alloc_bits(6);b.bit_store(zero_mask,False)\n    b.cx(zm[0],zero_mask[0]);ar.phase_product(b,norm,invnorm,zero_mask)\n    b.free(zero_mask);b.free(zm)\n    im=b.alloc_bits(6);b.hmr(invnorm,im);b.free(invnorm)\n    xs.phase_inverse6_quad4(b,norm,im);b.free(im)\n    ar._erase_norm(b,x,norm,K);b.free(norm);ar.linear_inplace(b,x,frm)\n\n\ndef scale_before_inverse(b,x,a):\n    out=b.alloc_qubits(12);ar.mul_xor0(b,x,a,out)\n    mask=b.alloc_bits(12);b.hmr(x,mask);b.swap(x,out);b.free(out)\n    return mask\n\n\n@contextmanager\ndef selected_partial(b,x,y,F,borrow=11):\n    b.cswap(F,x[:borrow],y[:borrow])\n    z=b.alloc_qubits(12-borrow)\n    for j in range(borrow,12):\n        b.cx(y[j],x[j]);b.init_and(F,x[j],z[j-borrow]);b.cx(y[j],x[j])\n        b.cx(x[j],z[j-borrow])\n    yield list(x[:borrow])+list(z)\n    mask=b.alloc_bits(12-borrow);b.hmr(z,mask);b.free(z)\n    for j in range(borrow,12):\n        b.cz(x[j],mask[j-borrow]);b.cx(y[j],x[j])\n        b.ccz(F,x[j],mask[j-borrow]);b.cx(y[j],x[j])\n    b.free(mask);b.cswap(F,x[:borrow],y[:borrow])\n\n\ndef converter_of(decoder):\n    cells=dict(zip(decoder.__code__.co_freevars,decoder.__closure__))\n    return cells[\'g\'].cell_contents[\'convert_safe\']\n\n\ndef scope_of(cv):\n    core=dict(zip(cv.__code__.co_freevars,cv.__closure__))[\'core\'].cell_contents\n    return core.__globals__[\'old\'].direct_post.__globals__\n\n\ndef install_pre(cv,levels=range(6,17),plain_levels=(2,3,4,5)):\n    scope=scope_of(cv)\n    dcells=dict(zip(scope[\'direct_pre\'].__code__.co_freevars,scope[\'direct_pre\'].__closure__))\n    original=dcells[\'handed\'].cell_contents;scope=original.__globals__\n    levels=frozenset(levels);plain_levels=frozenset(plain_levels)\n\n    def close_heads(b,f,g,ai,F,masks):\n        lo,hi=f[-2:]\n        ar.phase_product(b,hi,ai,masks[1]);ar.phase_product(b,lo,ai,masks[0])\n        with selected_partial(b,g[-2],g[-3],F)as c:\n            p.triple_square_phase(b,hi,c,ai,masks[0])\n        for mask in masks:b.free(mask)\n\n    def handed(b,f,g,payloads,F,incoming,defer,Z0):\n        if len(f)-1 not in levels:return original(b,f,g,payloads,F,incoming,defer,Z0)\n        assert not defer\n        b.cx(Z0,g[-2][0]);masks=scope[\'high_full_restore\'](b,f,g,F);b.cx(Z0,g[-2][0])\n        scope[\'pivot\'](b,g,F);a=g[-1]\n        for v in payloads:scope[\'dk\'].shifted_product(b,g[:-1],v[-1],v[:-1],F)\n        b.cx(Z0,a[0]);debts=[scale_before_inverse(b,v[-1],a)for v in payloads];b.cx(Z0,a[0])\n        zm=b.alloc_bits(1);b.hmr(Z0,zm[0]);b.free(Z0)\n        ai,Z,saved=inverse_zero_normpair(b,a)\n        b.cz(Z,zm[0]);b.free(zm);scope[\'consume\'](b,ai,Z,incoming)\n        b.cx(Z,a[0]);b.cx(Z,ai[0])\n        for v,mask in zip(payloads,debts):ar.phase_product(b,ai,v[-1],mask);b.free(mask)\n        close_heads(b,f,g,ai,F,masks)\n        b.cx(Z,ai[0]);b.cx(Z,a[0]);erase_normpair(b,a,ai,Z,saved)\n        scope[\'pivot\'](b,g,F)\n\n    dcells[\'handed\'].cell_contents=handed;scope[\'handed_aligned_pre\']=handed\n    pcells=dict(zip(scope[\'pre\'].__code__.co_freevars,scope[\'pre\'].__closure__))\n    plain_original=pcells[\'plain_handed\'].cell_contents\n\n    def plain(b,f,g,payloads,incoming,defer,Z0):\n        if len(f)-1 not in plain_levels:return plain_original(b,f,g,payloads,incoming,defer,Z0)\n        assert not defer\n        a=g[-1];b.cx(Z0,a[0]);masks=scope[\'full_restore\'](b,f,g);b.cx(Z0,a[0])\n        for v in payloads:\n            for i in range(len(g)-1):ar.mul_xor0(b,g[i],v[-1],v[i])\n        b.cx(Z0,a[0]);debts=[scale_before_inverse(b,v[-1],a)for v in payloads];b.cx(Z0,a[0])\n        zm=b.alloc_bits(1);b.hmr(Z0,zm[0]);b.free(Z0)\n        ai,Z,saved=inverse_zero_normpair(b,a)\n        b.cz(Z,zm[0]);b.free(zm);scope[\'consume\'](b,ai,Z,incoming)\n        b.cx(Z,a[0]);b.cx(Z,ai[0])\n        for v,mask in zip(payloads,debts):ar.phase_product(b,ai,v[-1],mask);b.free(mask)\n        scope[\'close_heads\'](b,f,g,ai,masks)\n        b.cx(Z,ai[0]);b.cx(Z,a[0]);erase_normpair(b,a,ai,Z,saved)\n\n    pcells[\'plain_handed\'].cell_contents=plain;scope[\'handed_plain_pre\']=plain\n    return cv\n\n\ndef install_quotient(decoder,levels=range(1,16)):\n    scope=scope_of(converter_of(decoder))[\'q_post\'].__globals__\n    original=scope[\'prior_q_post\'];levels=frozenset(levels)\n\n    def q_post(b,q,payloads,H,m,parent,reverse=False,phase=None):\n        if not reverse or m not in levels or parent is None:\n            return original(b,q,payloads,H,m,parent,reverse,phase)\n        tops,p,c=parent\n        F=phase[\'given_gap\'].pop(m+1);b.cswap(F,q,p)\n        scope[\'masked_products\'](b,F,payloads,c,tops)\n        Z,saved=prefix(b,q)\n        b.cx(Z,q[0]);debts=[scale_before_inverse(b,v[-1],q)for v in payloads];b.cx(Z,q[0])\n        qi=finish(b,q,saved);scope[\'consume\'](b,qi,Z,phase[\'q\'].pop(m))\n        b.cx(Z,q[0]);b.cx(Z,qi[0])\n        for v,mask in zip(payloads,debts):ar.phase_product(b,qi,v[-1],mask);b.free(mask)\n        b.cx(Z,qi[0]);b.cx(Z,q[0]);erase_normpair(b,q,qi,Z,saved)\n        for v,t in zip(payloads,tops):b.cswap(F,v[-1],t)\n        b.cswap(F,q,p)\n        if m+1 not in scope[\'KEEP_CHILD\']:\n            phase[\'return_mask\'][m+1]=scope[\'hold_flag\'](b,F)\n        else:phase[\'returned_gap\'][m+1]=F\n\n    scope[\'prior_q_post\']=q_post\n    return decoder\n\n\ndef install_base(decoder):\n    scope=scope_of(converter_of(decoder));original=scope[\'base_case\']\n\n    def base_case(b,f,g,payloads,H,parent,reverse=False,phase=None):\n        if not reverse:return original(b,f,g,payloads,H,parent,reverse,phase)\n        a=g[0];q=f[-1]\n        def action(ai):\n            scope[\'runops\'](b,[(\'s\',v[0],True)for v in payloads]+[(\'s\',q,False)],a,ai,True)\n        carry=scope[\'inverse_apply\'](b,a,action,phase[\'a\'].pop(1),True)\n        scope[\'q_post\'](b,q,payloads,H,1,parent,True,phase)\n        Z,saved=prefix(b,a)\n        b.cx(Z,a[0]);words=[v[0]for v in payloads]+[q]\n        debts=[scale_before_inverse(b,x,a)for x in words];b.cx(Z,a[0])\n        ai=finish(b,a,saved);scope[\'consume\'](b,ai,Z,carry)\n        b.cx(Z,a[0]);b.cx(Z,ai[0])\n        for x,mask in zip(words,debts):ar.phase_product(b,ai,x,mask);b.free(mask)\n        b.cx(Z,ai[0]);b.cx(Z,a[0])\n        erase_normpair(b,a,ai,Z,saved,(phase[\'returned_gap\'][2],phase))\n\n    scope[\'base_case\']=base_case\n    return decoder\n\n\ndef make_decoder(converter=None,reference_decode=None):\n    cv=private_rebind(p.convert_safe,p.base.replacements,p.ROOT)if converter is None else converter\n    install_pre(cv)\n    reference=p.decode if reference_decode is None else reference_decode\n    decoder=p.clone(reference,dict(reference.__globals__,convert_safe=cv))\n    decoder=private_rebind(p.install_factorized_boundary(decoder),p.base.replacements,p.ROOT)\n    install_quotient(decoder);install_base(decoder)\n    return decoder\n\n\ndecode=make_decoder()\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a09/hankel_compact.py': '"""Exact two-copy Hankel schedule with a frozen public partition.\n\nStandalone installer: imports only A01\'s port and immutable kit arithmetic.\nIt retains the top dot and changes only the forward Hankel map. Every\nselected public bilinear tensor is verified during construction.\n"""\nfrom pathlib import Path\nfrom functools import lru_cache\nimport sys\nROOT=Path(__file__).resolve().parents[3]\nsys.path.insert(0,str(ROOT/\'research/agents/a01\'))\nimport ported as p\nar,kernels=p.ar,p.k\n\n\ndef product(a,b,f):\n    out=[0]*(len(a)+len(b)-1)\n    for i,x in enumerate(a):\n        for j,y in enumerate(b):out[i+j]^=f.mul(x,y)\n    return out\n\ndef remainder(a,mod,f):\n    out=list(a)\n    for i in range(len(out)-len(mod),-1,-1):\n        c=out[i+len(mod)-1]\n        for j,v in enumerate(mod):out[i+j]^=f.mul(c,v)\n    return (out+[0]*(len(mod)-1))[:len(mod)-1]\n\ndef evaluate(a,x,f):\n    out=0\n    for c in reversed(a):out=f.mul(out,x)^c\n    return out\n\ndef pack(a):return sum(c<<(6*i) for i,c in enumerate(a))\n\ndef unpack(x,n):return [(x>>(6*i))&63 for i in range(n)]\n\n@lru_cache(None)\ndef thickening(k):\n    f,sub=ar.field(12),ar.field(6)\n    generator=f.pow(f.primitive_element,65)\n    def binary_eval(poly,x):\n        out=0\n        for i in reversed(range(7)):out=f.mul(out,x)^((poly>>i)&1)\n        return out\n    root=next(f.pow(generator,i) for i in range(1,64)\n              if binary_eval(sub.modulus,f.pow(generator,i))==0)\n    embed=[f.pow(root,i) for i in range(6)]\n    alpha=f.primitive_element\n    to=ar.inverse_columns(embed+[f.mul(v,alpha) for v in embed])\n    minimal=unpack(ar.apply(to,f.square(alpha)),2)+[1]\n    mod=[1]\n    for _ in range(k):mod=product(mod,minimal,sub)\n    theta=[0,1]\n    for _ in range(12):theta=remainder(product(theta,theta,sub),mod,sub)\n    epsilon=list(theta);epsilon[1]^=1\n    epowers=[[1]]\n    for _ in range(k):epowers.append(remainder(product(epowers[-1],epsilon,sub),mod,sub))\n    assert not any(epowers[k])\n    unit=[0]*(k+1);other=[0]*(k+1)\n    for j in range(k+1):\n        if (j&k)==j:other[j],unit[j]=unpack(ar.apply(to,f.pow(alpha,k-j)),2)\n    assert any(remainder(unit,minimal,sub))\n    assert remainder(product(theta,unit,sub),mod,sub)==remainder(other,mod,sub)\n    columns=[]\n    for degree in range(k):\n        for bit in range(12):\n            c0,c1=unpack(to[bit],2)\n            value=[sub.mul(c1,a) for a in theta];value[0]^=c0\n            columns.append(pack(remainder(product(value,epowers[degree],sub),mod,sub)))\n    weighted=[pack(remainder(product(unpack(v,2*k),unit,sub),mod,sub)) for v in columns]\n    double=[pack(remainder(product(unpack(v,2*k),unit,sub),mod,sub)) for v in weighted]\n    return mod,tuple(weighted),tuple(ar.inverse_columns(double))\n\n@lru_cache(None)\ndef two_sided(k,nx,ny):\n    assert 1<=nx<=k and 1<=ny<=k\n    sub=ar.field(6)\n    mod,weighted,decode=thickening(k)\n    left=weighted[:12*nx];right=weighted[:12*ny]\n    ld=max((v.bit_length()-1)//6 for v in left if v)\n    rd=max((v.bit_length()-1)//6 for v in right if v)\n    points=range(ld+rd+1)\n    assert len(points)<=64\n    terms=[]\n    for point in points:\n        lag,den=[1],1\n        for other in points:\n            if other==point:continue\n            lag=product(lag,[other,1],sub);den=sub.mul(den,point^other)\n        lag=[sub.mul(a,sub.invert(den)) for a in lag]\n        lv=[evaluate(unpack(v,2*k),point,sub) for v in left]\n        rv=[evaluate(unpack(v,2*k),point,sub) for v in right]\n        for mask,vector in ar.tensor(6):\n            a=sum(((v&mask).bit_count()&1)<<i for i,v in enumerate(lv))\n            b=sum(((v&mask).bit_count()&1)<<i for i,v in enumerate(rv))\n            out=ar.apply(decode,pack(remainder([sub.mul(v,vector) for v in lag],mod,sub)))\n            if a and b and out:terms.append((a,b,out))\n    merged={}\n    for a,b,out in terms:merged[a,b]=merged.get((a,b),0)^out\n    result=tuple((a,b,out) for (a,b),out in merged.items() if out)\n    return result\n\n@lru_cache(None)\ndef trace_pairing():\n    f=ar.field(12)\n    def trace(x):\n        y=x;out=0\n        for _ in range(12):out^=y;y=f.square(y)\n        assert out in (0,1)\n        return out\n    columns=tuple(sum(trace(f.mul(1<<i,1<<j))<<i for i in range(12)) for j in range(12))\n    return columns,tuple(ar.inverse_columns(columns))\n\ndef reverse_transform(value,words,columns):\n    return sum(ar.apply(columns,(value>>(12*i))&4095)<<(12*(words-1-i)) for i in range(words))\n\ndef verify_middle(terms,k,sources,prefix):\n    f=ar.field(12)\n    for i in range(12*sources):\n        result=[0]*(12*k)\n        for a,b,out in terms:\n            if not ((a>>i)&1):continue\n            for j in ar.bits(b):result[j]^=out\n        for j,value in enumerate(result):\n            degree=i//12+j//12\n            expected=(f.mul(1<<(i%12),1<<(j%12))<<(12*(degree-prefix+1))\n                      if prefix-1<=degree<k else 0)\n            assert value==expected,(k,sources,prefix,i,j,value,expected)\n    return True\n\n@lru_cache(None)\ndef middle_tensor(k,sources,overlap=2):\n    prefix=sources-overlap\n    outputs=k-prefix+1\n    assert prefix>=1 and 1<=outputs<=k\n    paired,unpaired=trace_pairing()\n    terms=[]\n    for a,b,out in two_sided(k,sources,outputs):\n        terms.append((a,reverse_transform(out,k,paired),reverse_transform(b,outputs,unpaired)))\n    merged={}\n    for a,b,out in terms:merged[a,b]=merged.get((a,b),0)^out\n    result=tuple((a,b,out) for (a,b),out in merged.items() if a and b and out)\n    verify_middle(result,k,sources,prefix)\n    return result\n\nclass NativeMiddle:\n    def __init__(self,k,sources,overlap):\n        self.k,self.sources=k,sources\n        self.prefix=sources-overlap\n        self.outputs=k-self.prefix+1\n        N=sources+self.outputs-1\n        V,W=kernels.vandermonde(N)\n        self.terms=tuple((tuple(V[t][:sources]),\n                         tuple(W[k-1-j][t] for j in range(k)),\n                         tuple(V[t][self.outputs-1-j] for j in range(self.outputs)))\n                        for t in range(N))\n        self.terms=tuple((a,b,c) for a,b,c in self.terms if any(a) and any(b) and any(c))\n        self.verify()\n\n    def verify(self):\n        f=ar.field(12)\n        for i in range(self.sources):\n            for j in range(self.k):\n                out=[0]*self.outputs\n                for a,b,c in self.terms:\n                    ab=f.mul(a[i],b[j])\n                    for l,v in enumerate(c):out[l]^=f.mul(ab,v)\n                expected=[int(i+j==self.prefix-1+l) for l in range(self.outputs)]\n                assert out==expected,(self.k,self.sources,i,j,out,expected)\n        return True\n\n    def __len__(self):\n        return 42*len(self.terms)\n\n@lru_cache(None)\ndef hybrid_tensor(k,sources,overlap=2):\n    if k<=17:\n        return middle_tensor(k,sources,overlap)\n    return NativeMiddle(k,sources,overlap)\n\ndef accumulate_binary(b,x,q,out,terms):\n    assert len(set(map(str,x+q+out)))==len(x)+len(q)+len(out)\n    for am,bm,vector in terms:\n        ix,iy,iz=[list(ar.bits(v)) for v in (am,bm,vector)]\n        a,c,t=ix[0],iy[0],iz[0]\n        for i in ix[1:]: b.cx(x[i],x[a])\n        for i in iy[1:]: b.cx(q[i],q[c])\n        for i in iz[1:]: b.cx(out[t],out[i])\n        b.ccx(x[a],q[c],out[t])\n        for i in reversed(iz[1:]): b.cx(out[t],out[i])\n        for i in reversed(iy[1:]): b.cx(q[i],q[c])\n        for i in reversed(ix[1:]): b.cx(x[i],x[a])\n\ndef accumulate(b,x,q,out,terms):\n    if not isinstance(terms,NativeMiddle):\n        return accumulate_binary(b,x,q,out,terms)\n    assert len(set(map(str,x+q+out)))==len(x)+len(q)+len(out)\n    words=lambda bits:[bits[i:i+12] for i in range(0,len(bits),12)]\n    aw,bw,tw=map(words,(x,q,out))\n    for ac,bc,tc in terms.terms:\n        p.dk.rank_update(b,aw,ac,bw,bc,tw,tc)\n\ndef retire_descending(b,q,x,old_sources,copies,prefix):\n    for index in reversed(range(prefix,len(old_sources))):\n        copied=copies[index-prefix]\n        mask=b.alloc_bits(12)\n        b.hmr(copied,mask);b.free(copied)\n        b.cz(x[index],mask)\n        for j in range(index):\n            ar.phase_product(b,q[index-j-1],old_sources[j],mask)\n        b.free(mask)\n\nflat=lambda words:[bit for word in words for bit in word]\nPLANS={1:(\'empty\',),2:(\'small\',),4:(\'small\',),6:(\'leaf\',4),\n       10:(\'leaf\',6),14:(\'leaf\',8),18:(\'leaf\',8),34:(\'leaf\',16)}\n\n\ndef install(cv):\n    cells=dict(zip(cv.__code__.co_freevars,cv.__closure__))\n    scope=cells[\'core\'].cell_contents.__globals__\n    maximum,max_sources,overlap,force_sources=34,18,2,None\n\n    @lru_cache(None)\n    def partition(n):\n        assert maximum==34 and max_sources==18 and overlap==2 and force_sources is None\n        return 0,PLANS[n]\n\n    def tri(b,q,x,reverse=False):\n        assert not reverse\n        _,plan=partition(len(x))\n        if plan[0]==\'empty\':return\n        if plan[0]==\'small\':\n            if len(x)==2:return ar.mul_xor0(b,q[0],x[0],x[1])\n            return p.hankel_direct(b,q,x)\n        assert plan[0]==\'leaf\'\n        sources=plan[1];prefix=sources-overlap\n        tri(b,q[:len(x)-sources-1],x[sources:])\n        copies=[b.alloc_qubits(12) for _ in range(overlap)]\n        for source,target in zip(x[prefix:sources],copies):b.cx(source,target)\n        old_sources=x[:prefix]+copies\n        accumulate(b,flat(old_sources),flat(q[:len(x)-1]),flat(x[prefix:]),\n                   hybrid_tensor(len(x)-1,sources,overlap))\n        retire_descending(b,q,x,old_sources,copies,prefix)\n        tri(b,q[:prefix-1],x[:prefix])\n\n    def hankel(b,B,A,reverse=False):\n        if reverse:return p.hankel(b,B,A,True)\n        return p.clone(kernels.hankel,dict(vars(kernels),tri=tri))(b,B,A)\n    scope[\'old\'].hankel=hankel\n    return cv\n\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a11/hankel_cache_even.py': '"""Exact one-word handoff from forward Hankel to its measured inverse.\n\nThe public plans impose an even root source count for cyclic top-dot sharing.  The only additional\nlive state through scan is one original GF4096 word from the final triple.\n"""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[3]\nsys.path[:0]=[str(ROOT/\'research/agents/a09\'),str(ROOT/\'research/agents/a01\')]\nimport hankel_compact as h\np,ar,k=h.p,h.ar,h.kernels\n\nFORWARD={(34,True):18,(16,False):8,(8,False):4,(6,False):4,\n         (16,True):9,(7,False):4,(7,True):5}\nREVERSE_FIRST={34:18,18:10,10:6,6:3}\n\n\ndef install(cv):\n    core=dict(zip(cv.__code__.co_freevars,cv.__closure__))[\'core\'].cell_contents\n    owner=core.__globals__[\'old\']\n    saved=[]\n\n    def forward(b,q,x,first=True):\n        n=len(x)\n        if n in (2,3,4):\n            if first:\n                assert n==3 and not saved\n                word=b.alloc_qubits(12);b.cx(x[-1],word);saved.append(word)\n            if n==2:return ar.mul_xor0(b,q[0],x[0],x[1])\n            return p.hankel_direct(b,q,x)\n        sources=FORWARD[n,first];prefix=sources-2\n        forward(b,q[:n-sources-1],x[sources:],False)\n        copies=[b.alloc_qubits(12)for _ in range(2)]\n        for source,target in zip(x[prefix:sources],copies):b.cx(source,target)\n        old_sources=x[:prefix]+copies\n        h.accumulate(b,h.flat(old_sources),h.flat(q[:n-1]),h.flat(x[prefix:]),\n                     h.hybrid_tensor(n-1,sources,2))\n        h.retire_descending(b,q,x,old_sources,copies,prefix)\n        forward(b,q[:prefix-1],x[:prefix],first)\n\n    def hankel(b,B,A,reverse=False):\n        qfull=list(reversed(B[1:]))\n        if not reverse:\n            assert not saved\n            forward(b,qfull,A)\n            for i in range(len(A)//2):b.swap(A[i],A[-1-i])\n            assert len(saved)==1\n            return\n        assert len(saved)==1\n        cube=b.alloc_qubits(12);p.cube_xor(b,B[-1],cube)\n\n        def solve(q,x,first=False):\n            n=len(x)\n            if n==1:return\n            if first and n==3:\n                solve(q[:1],x[:2])\n                mask=b.alloc_bits(12);b.hmr(x[2],mask)\n                word=saved.pop();b.swap(x[2],word);b.free(word)\n                b.cz(x[2],mask)\n                ar.phase_product(b,q[1],x[0],mask)\n                ar.phase_product(b,q[0],x[1],mask)\n                b.free(mask)\n                return\n            if first:\n                left=REVERSE_FIRST[n]\n            else:\n                _,left=p.reverse_partition(n)\n            if left is None:\n                if n==4:b.cx(cube,q[2])\n                ar.linear_xor(b,q[0],q[1],ar.square_columns(12))\n                p.hankel_direct(b,q,x)\n                ar.linear_xor(b,q[0],q[1],ar.square_columns(12))\n                if n==4:b.cx(cube,q[2])\n                return\n            solve(q[:left-1],x[:left],first)\n            for ac,bc,tc in reversed(k.cross_terms(left,n-left)):\n                k.rank_update(b,x[:left],ac,q[:n-1],bc,x[left:],tc)\n            solve(q[:n-left-1],x[left:])\n\n        for i in range(len(A)//2):b.swap(A[i],A[-1-i])\n        solve(qfull,A,True)\n        assert not saved\n        mask=b.alloc_bits(12);b.hmr(cube,mask);b.free(cube)\n        p.cube_phase(b,B[-1],mask);b.free(mask)\n\n    owner.hankel=hankel\n    return cv\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a12/forward_norm_cache.py': '"""Exact forward inverse-norm retention in the retired-root lifetime.\n\nInstall after A08\'s scalar installer on the private converter graph.  The\ndefault two six-bit caches occupy the twelve wires that A07 frees during\nforward descent; neither cache survives the selected frame\'s return.\n"""\nfrom pathlib import Path\nimport sys\n\nROOT = Path(__file__).resolve().parents[3]\nsys.path.insert(0, str(ROOT / \'research/agents/a08\'))\nimport scalar_compact_v3 as scalar\n\nar, xs = scalar.ar, scalar.xs\n\n\ndef first_inverse_with_t(b, x):\n    """Return inv0(x), [x=0], and t=inv0(Norm(x)); retire raw Norm(x)."""\n    to, frm, K = ar.tower(12)\n    ar.linear_inplace(b, x, to)\n    norm = b.alloc_qubits(6)\n    scalar.norm_xor_noscratch(b, x, norm, K)\n    t, F = xs.inverse6_joint18_12extra(b, norm)\n    ar._erase_norm(b, x, norm, K)\n    b.free(norm)\n    out = b.alloc_qubits(12)\n    b.cx(x[6:], x[:6])\n    scalar.mul_xor_noscratch(b, x[:6], t, out[:6])\n    scalar.mul_xor_noscratch(b, x[6:], t, out[6:])\n    b.cx(x[6:], x[:6])\n    ar.linear_inplace(b, x, frm)\n    ar.linear_inplace(b, out, frm)\n    return out, F, t\n\n\ndef inverse_from_t_and_retire(b, x, t, zero_flag):\n    """Reuse t, then repay its HMR character against Norm(inv0(x))."""\n    to, frm, K = ar.tower(12)\n    F = zero_flag(b, t)\n    ar.linear_inplace(b, x, to)\n    out = b.alloc_qubits(12)\n    b.cx(x[6:], x[:6])\n    scalar.mul_xor_noscratch(b, x[:6], t, out[:6])\n    scalar.mul_xor_noscratch(b, x[6:], t, out[6:])\n    b.cx(x[6:], x[:6])\n    # inv0(Norm(x)) = Norm(inv0(x)), also at x=0.\n    ar._erase_norm(b, out, t, K)\n    b.free(t)\n    ar.linear_inplace(b, x, frm)\n    ar.linear_inplace(b, out, frm)\n    return out, F\n\n\ndef install(cv, levels=(11, 5)):\n    """Patch only forward inverse production; keep all existing phase tokens."""\n    cells = dict(zip(cv.__code__.co_freevars, cv.__closure__))\n    short = cells[\'core\'].cell_contents.__globals__\n    scope = scalar.scope_of(cv)\n    original_inverse = scope[\'inverse_zero\']\n    original_solve = scope[\'look_solve\']\n    levels = frozenset(levels)\n    frames = []\n\n    def inverse_zero(b, x):\n        if not frames:\n            return original_inverse(b, x)\n        m, phase, g, reverse = frames[-1]\n        if reverse or m not in levels or list(x) != list(g[-1]):\n            return original_inverse(b, x)\n        caches = phase.setdefault(\'_a12_forward_norm\', {})\n        if m not in caches:\n            ai, F, caches[m] = first_inverse_with_t(b, x)\n            return ai, F\n        return inverse_from_t_and_retire(\n            b, x, caches.pop(m), scope[\'fresh_zero_flag\'])\n\n    def inverse_apply_safe(b, a, action, incoming=None, defer=False):\n        ai, F = inverse_zero(b, a)\n        scope[\'consume\'](b, ai, F, incoming)\n        b.cx(F, a[0])\n        b.cx(F, ai[0])\n        action(ai)\n        b.cx(F, ai[0])\n        b.cx(F, a[0])\n        return scope[\'erase\'](b, a, ai, F, defer)\n\n    def solve(b, f, g, payloads, H=None, parent=None, reverse=False, phase=None):\n        m = len(f) - 1\n        frames.append((m, phase, g, reverse))\n        try:\n            out = original_solve(b, f, g, payloads, H, parent, reverse, phase)\n            if not reverse and m in levels:\n                assert m not in phase.get(\'_a12_forward_norm\', {})\n            return out\n        finally:\n            frames.pop()\n\n    scope.update(inverse_zero=inverse_zero,\n                 inverse_apply_safe=inverse_apply_safe, look_solve=solve)\n    short[\'old\'].look_solve = solve\n    return cv\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a12/known_zero_inverse.py': '"""Use a supplied exact zero flag in the initialized GF64 inverse plan."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[3]\nsys.path[:0]=[str(ROOT/\'research/agents/a08\'),str(ROOT/\'research/agents/a01\')]\nimport scalar_compact_v3 as scalar\nfrom exp.jet24 import pro_jet as jet\nar=scalar.ar\n\n\ndef inverse6_known_zero(b,x,Z):\n    """Return inv0(x), preserving x and supplied Z=[x=0]."""\n    ar.linear_inplace(b,x,jet.TO6)\n    raw=b.alloc_qubits(6)\n    for j,(l,r) in enumerate(jet.RAW):\n        with jet.ref.expose_pair(b,x,l,r) as (p,q):b.init_and(p,q,raw[j])\n    w=jet.ref.inverse3_virtual(b,list(x)+list(raw),jet.ref.norm_forms(1))\n    raw=list(raw)+list(w);out=[]\n    for step,(kind,a,c,d) in enumerate(jet.PLAN):\n        if kind==\'r\':\n            ar.linear_inplace(b,raw,a)\n            for j,action in enumerate(d,c):\n                if action is None:out.append(raw[j]);continue\n                coeff,phase=action\n                for k in ar.bits(coeff):b.cx(raw[j],out[k])\n                mask=b.alloc_bits(1)[0];b.hmr(raw[j],mask)\n                live=list(x)+raw[:c]+raw[j+1:]+out\n                for i,k in phase:\n                    if i==0:b.z(mask)\n                    elif k==0 or i==k:b.cz(live[i-1],mask)\n                    else:b.ccz(live[i-1],live[k-1],mask)\n                b.free(mask);b.free(raw[j])\n            raw=list(raw[:c])\n        elif kind==\'g\':out.extend(b.alloc_qubits(a))\n        elif step==17:\n            assert (a,c,d,len(raw),len(out))==(200,474,64,5,7)\n            # Exact all-field identity in this live prefix\'s coordinates.\n            b.x(out[6]);b.cx(x[0],out[6]);b.cx(x[3],out[6])\n            b.cx(raw[1],out[6]);b.cx(Z,out[6])\n        else:\n            with jet.ref.expose_pair(b,list(x)+raw,a,c) as (p,q):\n                jet.rankone_xor(b,p,q,out,d)\n    ar.linear_inplace(b,out,jet.OUTPUT)\n    for j,form in enumerate(jet.LINEAR):jet.ref.form_xor(b,x,form,out[j])\n    ar.linear_inplace(b,x,jet.FROM6)\n    ar.linear_inplace(b,out[:6],jet.FROM6)\n    b.cx(Z,out[6]);b.free(out[6])\n    return out[:6]\n\n\ndef inverse_normpair_known_zero(b,a,Z):\n    to,frm,K=ar.tower(12);ar.linear_inplace(b,a,to)\n    N=b.alloc_qubits(6);scalar.norm_xor_noscratch(b,a,N,K)\n    t=inverse6_known_zero(b,N,Z)\n    ar.linear_inplace(b,a,frm)\n    ai=scalar.finish(b,a,(N,t))\n    return ai,Z,(N,t)\n\n\ndef install(cv):\n    scope=scalar.scope_of(cv)\n    dcells=dict(zip(scope[\'direct_pre\'].__code__.co_freevars,scope[\'direct_pre\'].__closure__))\n    pcells=dict(zip(scope[\'pre\'].__code__.co_freevars,scope[\'pre\'].__closure__))\n    aligned_original=dcells[\'handed\'].cell_contents\n    plain_original=pcells[\'plain_handed\'].cell_contents\n    acells=dict(zip(aligned_original.__code__.co_freevars,aligned_original.__closure__))\n    close_heads=acells[\'close_heads\'].cell_contents\n    aligned_levels=frozenset(range(6,17));plain_levels=frozenset((2,3,4,5))\n\n    def handed(b,f,g,payloads,F,incoming,defer,Z0):\n        if len(f)-1 not in aligned_levels:return aligned_original(b,f,g,payloads,F,incoming,defer,Z0)\n        assert not defer\n        b.cx(Z0,g[-2][0]);masks=scope[\'high_full_restore\'](b,f,g,F);b.cx(Z0,g[-2][0])\n        scope[\'pivot\'](b,g,F);a=g[-1]\n        for v in payloads:scope[\'dk\'].shifted_product(b,g[:-1],v[-1],v[:-1],F)\n        b.cx(Z0,a[0]);debts=[scalar.scale_before_inverse(b,v[-1],a)for v in payloads];b.cx(Z0,a[0])\n        ai,Z,saved=inverse_normpair_known_zero(b,a,Z0);scope[\'consume\'](b,ai,Z,incoming)\n        b.cx(Z,a[0]);b.cx(Z,ai[0])\n        for v,mask in zip(payloads,debts):ar.phase_product(b,ai,v[-1],mask);b.free(mask)\n        close_heads(b,f,g,ai,F,masks)\n        b.cx(Z,ai[0]);b.cx(Z,a[0]);scalar.erase_normpair(b,a,ai,Z,saved)\n        scope[\'pivot\'](b,g,F)\n\n    def plain(b,f,g,payloads,incoming,defer,Z0):\n        if len(f)-1 not in plain_levels:return plain_original(b,f,g,payloads,incoming,defer,Z0)\n        assert not defer\n        a=g[-1];b.cx(Z0,a[0]);masks=scope[\'full_restore\'](b,f,g);b.cx(Z0,a[0])\n        for v in payloads:\n            for i in range(len(g)-1):ar.mul_xor0(b,g[i],v[-1],v[i])\n        b.cx(Z0,a[0]);debts=[scalar.scale_before_inverse(b,v[-1],a)for v in payloads];b.cx(Z0,a[0])\n        ai,Z,saved=inverse_normpair_known_zero(b,a,Z0);scope[\'consume\'](b,ai,Z,incoming)\n        b.cx(Z,a[0]);b.cx(Z,ai[0])\n        for v,mask in zip(payloads,debts):ar.phase_product(b,ai,v[-1],mask);b.free(mask)\n        scope[\'close_heads\'](b,f,g,ai,masks)\n        b.cx(Z,ai[0]);b.cx(Z,a[0]);scalar.erase_normpair(b,a,ai,Z,saved)\n\n    dcells[\'handed\'].cell_contents=handed;scope[\'handed_aligned_pre\']=handed\n    pcells[\'plain_handed\'].cell_contents=plain;scope[\'handed_plain_pre\']=plain\n    return cv\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a12/reverse_root_lifetime.py': '"""Complete experimental reverse root retirement with exact reconstruction."""\nfrom pathlib import Path\nimport sys\nROOT = Path(__file__).resolve().parents[3]\nsys.path[:0] = [str(Path(__file__).resolve().parent),\n                str(ROOT/\'research/integrated\'), str(ROOT/\'research/agents/a08\'),\n                str(ROOT/\'research/agents/a01\')]\nimport decode_compact_integrated as reference\nimport scalar_compact_v3 as scalar\nfrom rebind_graph import private_rebind\np = scalar.p\n\n\ndef look_root_cached(b, O, g, payloads, reverse, phase):\n    m=len(O); keep=m in KEEP_CHILD; full=m in KEEP_FULL\n    F=phase[\'held_gap\'].pop(m) if reverse and full else old.fresh_zero_flag(b,g[-1])\n    if reverse and not full:consume_flag(b,F,phase[\'gap_flags\'].pop(m))\n    pivot(b,g,F); a=g[-1]; ai,Z=old.inverse_zero(b,a); b.cx(Z,ai[0])\n    if reverse:\n        mask=phase.pop(\'_root_inverse_mask\'); b.cz(ai,mask); b.free(mask)\n    f=list(O)+[ai]\n    if reverse:\n        H=phase[\'held_gap\'][m-1]\n        mask=b.alloc_bits(1); b.hmr(Z,mask[0]); b.free(Z)\n        b.ccz(F,H,mask[0]); b.free(mask)\n        gap_zero_action(b,f,g,payloads,ai,F,H,post=True,undo=True)\n    else:\n        root_pre_partial(b,f,g,payloads,ai,Z,F)\n        erase_product_zero(b,a,ai,Z)\n    pivot(b,g,F); aligned(b,f,g,F)\n    if reverse or keep:phase[\'given_gap\'][m]=F\n    else:phase[\'incoming_gap\'][m]=hold_flag(b,F)\n    parent=([v[-1] for v in payloads],f[-1],f[-2])\n    if not reverse:\n        phase[\'_root_block_payloads\']=payloads\n        root_mask=b.alloc_bits(12); b.hmr(ai,root_mask); b.free(ai)\n        phase[\'_ephemeral_root\']=(ai,root_mask)\n    old.look_solve(b,g,f[:-2],[v[:-1] for v in payloads],None,parent,reverse,phase)\n    if reverse and not keep:\n        F=old.fresh_zero_flag(b,g[-1]); consume_flag(b,F,phase[\'return_mask\'].pop(m))\n    else:F=phase[\'returned_gap\'].pop(m)\n    aligned(b,f,g,F); pivot(b,g,F)\n    if not reverse:\n        H=phase[\'held_gap\'][m-1]\n        gap_zero_action(b,f,g,payloads,ai,F,H,post=True,undo=False)\n    else:\n        saved,mask=phase.pop(\'_a12_reverse_root\')\n        replacement,Z=old.inverse_zero(b,a)\n        b.cx(Z,replacement[0]); saved[:]=replacement\n        b.cz(ai,mask); b.free(mask)\n        root_action(b,f,g,payloads,ai,Z,F,post=False,undo=True)\n        erase_product_zero(b,a,ai,Z)\n    mask=b.alloc_bits(12); b.hmr(ai,mask); b.free(ai)\n    if reverse:phase_inverse34(b,a,mask,zero_mask=mask[0]); b.free(mask)\n    else:phase[\'_root_inverse_mask\']=mask\n    pivot(b,g,F)\n    if reverse:clear_flag(b,g[-1],F)\n    elif full:phase[\'held_gap\'][m]=F\n    else:phase[\'gap_flags\'][m]=hold_flag(b,F)\n\n\ndef install(cv):\n    cells=dict(zip(cv.__code__.co_freevars,cv.__closure__))\n    short=cells[\'core\'].cell_contents.__globals__\n    scope=scalar.scope_of(cv)\n    original=scope[\'direct_post\']\n\n    def direct_post(b,f,g,payloads,F,H,parent,reverse=False,incoming=None,defer=False,phase=None):\n        result=original(b,f,g,payloads,F,H,parent,reverse,incoming,defer,phase)\n        if reverse and len(f)-1==16 and phase[\'_short_monic\']:\n            held=parent[1]\n            mask=b.alloc_bits(12); b.hmr(held,mask); b.free(held)\n            phase[\'_a12_reverse_root\']=(held,mask)\n        return result\n\n    scope[\'direct_post\']=direct_post\n    short[\'look_root_cached\']=p.clone(look_root_cached,short)\n    return cv\n\n\ndef make_decoder():\n    decoder=private_rebind(reference.decode,{},p.ROOT)\n    install(scalar.converter_of(decoder))\n    return decoder\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a13/inverse_character_width.py': '"""Exact, compact inverse-character phase schedules for private converters.\n\nThe ordinary branch has two stored products and two cubic phases.  The\nzero-mixed and zero-only branches have three stored products and one cubic\nphase.  This file never builds or changes a decoder on import.\n"""\nfrom pathlib import Path\nfrom types import CodeType, FunctionType, ModuleType, SimpleNamespace\nimport sys\n\nROOT = Path(__file__).resolve().parents[3]\nsys.path[:0] = [str(ROOT/\'exp/r-main/narrow240\'),\n                str(ROOT/\'research/agents/a01\')]\nimport exact_subs as xs\nfrom rebind_graph import private_rebind\n\nUSED = False\nORDINARY_FORMS = ((12,52), (154,94))\nORDINARY_TERMS = ((2,), (4,), (5,), (7,), (0,6), (1,6), (1,7),\n                  (2,3), (2,5), (2,7), (3,7))\nORDINARY_COLUMNS = (10,19,34,65,5,129,1,2)\nORDINARY_INVERSE = (64,128,80,129,194,132,72,96)\nMIXED_FORMS = ((23,78), (60,234), (15,182))\nMIXED_TERMS = ((), (1,), (2,3), (2,4), (2,5), (3,4), (3,8),\n              (4,), (4,5), (4,8), (5,8), (7,), (8,))\nMIXED_COLUMNS = (10,19,37,66,135,257,1,2,4)\nMIXED_INVERSE = (64,128,256,129,194,324,136,464,96)\n\n\ndef quadratic_phase(b, source, terms):\n    for term in terms:\n        if not term:\n            b.z(True)\n        elif len(term) == 1:\n            b.z(source[term[0]])\n        else:\n            b.cz(source[term[0]], source[term[1]])\n\n\ndef cubic_phase(b, source, columns, inverse):\n    """Phase l0*l1*(1+l2), restoring every source coordinate."""\n    xs.linear_inplace(b, source, columns)\n    b.cz(source[0], source[1])\n    b.ccz(source[0], source[1], source[2])\n    xs.linear_inplace(b, source, inverse)\n\n\ndef character(b, x, forms, terms, columns, inverse, extra=False):\n    work = b.alloc_qubits(len(forms))\n    source = list(x)+list(work)\n    for i, pair in enumerate(forms):\n        with xs.expose_pair(b, source[:6+i], *pair) as (left,right):\n            b.init_and(left,right,work[i])\n    quadratic_phase(b, source, terms)\n    cubic_phase(b, source, columns, inverse)\n    if extra:\n        b.ccz(x[0], x[2], x[3])\n    for i in reversed(range(len(forms))):\n        with xs.expose_pair(b, source[:6+i], *forms[i]) as (left,right):\n            b.del_and(left,right,work[i])\n    b.free(work)\n\n\ndef ordinary(b, x):\n    character(b,x,ORDINARY_FORMS,ORDINARY_TERMS,\n              ORDINARY_COLUMNS,ORDINARY_INVERSE,True)\n\n\ndef mixed(b, x):\n    character(b,x,MIXED_FORMS,MIXED_TERMS,MIXED_COLUMNS,MIXED_INVERSE)\n\n\ndef zero_phase(b, x):\n    b.x(x)\n    work = b.alloc_qubits(3)\n    b.init_and(x[0], x[1], work[0])\n    for i in range(1,3):\n        b.init_and(work[i-1], x[i+1], work[i])\n    b.ccz(work[2], x[4], x[5])\n    for i in reversed(range(1,3)):\n        b.del_and(work[i-1], x[i+1], work[i])\n    b.del_and(x[0], x[1], work[0])\n    b.free(work)\n    b.x(x)\n\n\ndef phase_inverse6(b, x, mask, zero_mask=None):\n    global USED\n    USED = True\n    assert len(x) == len(mask) == 6\n    for m in range(1,64):\n        eq = xs.classical_equal(b, mask, m)\n        b.push_condition(eq)\n        xs.linear_inplace(b, x, xs.SCALES[m])\n        if zero_mask is None:\n            ordinary(b, x)\n        else:\n            b.push_condition(zero_mask)\n            mixed(b, x)\n            b.pop_condition()\n            b.x(zero_mask)\n            b.push_condition(zero_mask)\n            ordinary(b, x)\n            b.pop_condition()\n            b.x(zero_mask)\n        xs.linear_inplace(b, x, xs.UNSCALES[m])\n        b.pop_condition()\n        b.free(eq)\n    if zero_mask is not None:\n        eq = xs.classical_equal(b, mask, 0)\n        b.push_condition(eq)\n        b.push_condition(zero_mask)\n        zero_phase(b, x)\n        b.pop_condition()\n        b.pop_condition()\n        b.free(eq)\n\n\ndef target_ids(entry):\n    """Find original helper copies only in this converter\'s private graph."""\n    seen, targets = set(), set()\n\n    def owned(path):\n        try:\n            return Path(path).resolve().is_relative_to(ROOT)\n        except (TypeError,ValueError):\n            return False\n\n    def names(code):\n        result = set(code.co_names)\n        for item in code.co_consts:\n            if isinstance(item,CodeType):\n                result.update(names(item))\n        return result\n\n    def visit(value):\n        key = id(value)\n        if key in seen:\n            return\n        seen.add(key)\n        if isinstance(value,FunctionType) and owned(value.__code__.co_filename):\n            if value.__name__ == \'phase_inverse6_quad4\' and Path(value.__code__.co_filename).name == \'exact_subs.py\':\n                targets.add(key)\n            for name in names(value.__code__):\n                if name in value.__globals__:\n                    visit(value.__globals__[name])\n            for cell in value.__closure__ or ():\n                try:\n                    visit(cell.cell_contents)\n                except ValueError:\n                    pass\n        elif isinstance(value,SimpleNamespace) or (\n                isinstance(value,ModuleType) and owned(getattr(value,\'__file__\',None))):\n            for name,item in vars(value).items():\n                if name != \'__builtins__\':\n                    visit(item)\n        elif isinstance(value,dict):\n            for item in value.values():\n                visit(item)\n        elif isinstance(value,(tuple,list)):\n            for item in value:\n                visit(item)\n        elif isinstance(value,FunctionType) and getattr(value,\'__wrapped__\',None) is not None:\n            visit(value.__wrapped__)\n    visit(entry)\n    return targets\n\n\ndef install_converter(cv):\n    ids = target_ids(cv)\n    assert ids, \'The intended inverse-character API was not found\'\n    return private_rebind(cv, {key:phase_inverse6 for key in ids}, ROOT)\n\n\ndef install_decoder(decoder):\n    cells = dict(zip(decoder.__code__.co_freevars,decoder.__closure__))\n    scope = cells[\'g\'].cell_contents\n    scope[\'convert_safe\'] = install_converter(scope[\'convert_safe\'])\n    return decoder\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a14/reverse_caches.py': '"""Exact mixed-size reverse inverse-norm caches in the retired-root lifetime."""\nfrom pathlib import Path\nimport sys\nROOT=Path(__file__).resolve().parents[3]\nsys.path[:0]=[str(ROOT/\'research/agents/a12\'),str(ROOT/\'research/agents/a08\')]\nimport forward_norm_cache as cache\nimport scalar_compact_v3 as scalar\nar=scalar.ar\nsys.path.insert(0,str(ROOT/\'research/agents/a02\'))\nimport cache_support as small_cache\n\n\ndef cached_pair_from_w(b,x,w,Z):\n    to,frm,K=ar.tower(12);ar.linear_inplace(b,x,to)\n    N=b.alloc_qubits(6);scalar.norm_xor_noscratch(b,x,N,K)\n    to6,frm6,K6=ar.tower(6);ar.linear_inplace(b,N,to6)\n    t=b.alloc_qubits(6);b.cx(N[3:],N[:3])\n    scalar.mul_xor_noscratch(b,N[:3],w,t[:3])\n    scalar.mul_xor_noscratch(b,N[3:],w,t[3:])\n    b.cx(N[3:],N[:3]);ar.linear_inplace(b,N,frm6)\n    # In tower coordinates w = Norm(t), including t=0.\n    ar._erase_norm(b,t,w,K6);b.free(w);ar.linear_inplace(b,t,frm6)\n    ar.linear_inplace(b,x,frm)\n    return scalar.finish(b,x,(N,t)),Z,(N,t)\n\n\n\ndef install(cv):\n    kinds={11:\'t6\',5:\'w3\'}\n    levels=tuple(kinds)\n    cells=dict(zip(cv.__code__.co_freevars,cv.__closure__))\n    short=cells[\'core\'].cell_contents.__globals__\n    scope=scalar.scope_of(cv)\n    inverse_original=scope[\'inverse_zero\']\n    solve_original=scope[\'look_solve\']\n    frames=[]\n    levels=frozenset(levels)\n\n    def selected(x):\n        if not frames:return None\n        m,phase,g,reverse=frames[-1]\n        if reverse and m in levels and list(x)==list(g[-1]):return m,phase\n        return None\n\n    def inverse_zero(b,x):\n        match=selected(x)\n        if match is None:return inverse_original(b,x)\n        m,phase=match\n        caches=phase.setdefault(\'_a12_reverse_norm\',{})\n        assert m not in caches\n        if kinds[m]==\'w3\':\n            ai,Z,caches[m]=small_cache.inverse_zero_cached(b,x,producer18=True)\n        else:\n            assert kinds[m]==\'t6\'\n            ai,Z,caches[m]=cache.first_inverse_with_t(b,x)\n        return ai,Z\n\n    def inverse_apply_safe(b,a,action,incoming=None,defer=False):\n        ai,Z=inverse_zero(b,a)\n        scope[\'consume\'](b,ai,Z,incoming)\n        b.cx(Z,a[0]);b.cx(Z,ai[0]);action(ai)\n        b.cx(Z,ai[0]);b.cx(Z,a[0])\n        return scope[\'erase\'](b,a,ai,Z,defer)\n\n    def solve(b,f,g,payloads,H=None,parent=None,reverse=False,phase=None):\n        m=len(f)-1\n        frames.append((m,phase,g,reverse))\n        try:\n            out=solve_original(b,f,g,payloads,H,parent,reverse,phase)\n            if reverse and m in levels:\n                assert m not in phase.get(\'_a12_reverse_norm\',{})\n            return out\n        finally:frames.pop()\n\n    def producer(original):\n        def inverse_zero_normpair(b,x,Z):\n            match=selected(x)\n            if match is None:return original(b,x,Z)\n            m,phase=match\n            t=phase[\'_a12_reverse_norm\'].pop(m)\n            if kinds[m]==\'w3\':return cached_pair_from_w(b,x,t,Z)\n            to,frm,K=ar.tower(12);ar.linear_inplace(b,x,to)\n            N=b.alloc_qubits(6);scalar.norm_xor_noscratch(b,x,N,K)\n            ar.linear_inplace(b,x,frm)\n            ai=scalar.finish(b,x,(N,t))\n            return ai,Z,(N,t)\n        return inverse_zero_normpair\n\n    dcells=dict(zip(scope[\'direct_pre\'].__code__.co_freevars,\n                    scope[\'direct_pre\'].__closure__))\n    pcells=dict(zip(scope[\'pre\'].__code__.co_freevars,scope[\'pre\'].__closure__))\n    seen=set()\n    for function in (dcells[\'handed\'].cell_contents,pcells[\'plain_handed\'].cell_contents):\n        namespace=function.__globals__\n        if id(namespace) not in seen:\n            seen.add(id(namespace))\n            namespace[\'inverse_normpair_known_zero\']=producer(namespace[\'inverse_normpair_known_zero\'])\n    scope.update(inverse_zero=inverse_zero,inverse_apply_safe=inverse_apply_safe,look_solve=solve)\n    short[\'old\'].look_solve=solve\n    return cv\n\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a14/reverse_fusion.py': '"""Exact reverse middle fusion at the public root and degree-14 blocks.\n\nThe zero predicate is X-measured only while the three weight words coexist.\nIt is reconstructed from the restored pivot and its character is consumed\nbefore the ordinary scalar restoration continues.\n"""\nfrom pathlib import Path\nfrom types import FunctionType\nimport sys\n\nROOT=Path(__file__).resolve().parents[3]\nsys.path[:0]=[str(ROOT/\'research/agents/a08\'),\n              str(ROOT/\'research/agents/a12\'),\n              str(ROOT/\'research/agents/a01\')]\nimport scalar_compact_v3 as scalar\nimport reverse_root_lifetime\np,ar=scalar.p,scalar.ar\n\n\ndef install(cv):\n    starts=(17,14)\n    scope=scalar.scope_of(cv)\n    short=dict(zip(cv.__code__.co_freevars,cv.__closure__))[\'core\'].cell_contents.__globals__\n    reverse_root_lifetime.install(cv)\n    original_solve=scope[\'look_solve\']\n    original_root=short[\'look_root_cached\']\n    original_product=scope[\'dk\'].shifted_product\n    pre=scope[\'direct_pre\']\n    dcells=dict(zip(pre.__code__.co_freevars,pre.__closure__))\n    original_handed=dcells[\'handed\'].cell_contents\n    hc=dict(zip(original_handed.__code__.co_freevars,original_handed.__closure__))\n    handed_scope=hc[\'scope\'].cell_contents\n    close_heads=hc[\'close_heads\'].cell_contents\n    original_restore=handed_scope[\'high_full_restore\']\n    starts=frozenset(starts)\n    bottom_to_top={m-2:m for m in starts}\n    frame_to_top={m-j:m for m in starts for j in range(3)}\n    assert len(frame_to_top)==3*len(starts)\n    frames=[]\n    original_hold=scope[\'hold_flag\']\n    retained_starts=frozenset((min(starts),))\n    retained_levels=frozenset(n for top in retained_starts for n in (top,top-1))\n\n    def hold_flag(b,F):\n        if frames and frames[-1][\'reverse\'] and not frames[-1][\'restored\']:\n            frame=frames[-1];n=frame[\'m\']+1\n            if n in retained_levels:\n                held=frame[\'phase\'].setdefault(\'_a14_held_reverse_gaps\',{})\n                assert n not in held\n                held[n]=F\n                mask=b.alloc_bits(1);b.bit_store(mask,False)\n                return mask\n        return original_hold(b,F)\n\n    # The quotient pipeline owns closure dictionaries as well as function\n    # globals. Patch all of those owners, not merely the dispatch dictionary.\n    owners={};seen=set()\n    def visit(value):\n        if id(value) in seen:return\n        seen.add(id(value))\n        if isinstance(value,dict):\n            if \'hold_flag\' in value:\n                owners[id(value)]=value\n                for name in (\'q_post\',\'prior_q_post\'):\n                    if name in value:visit(value[name])\n        elif isinstance(value,FunctionType):\n            visit(value.__globals__)\n            for cell in value.__closure__ or ():\n                try:child=cell.cell_contents\n                except ValueError:continue\n                if isinstance(child,(dict,FunctionType)):visit(child)\n    visit(scope);visit(handed_scope)\n    for owner in owners.values():owner[\'hold_flag\']=hold_flag\n\n    def solve(b,f,g,payloads,H=None,parent=None,reverse=False,phase=None):\n        frames.append(dict(m=len(f)-1,f=f,g=g,payloads=payloads,\n                           phase=phase,reverse=reverse,restored=False))\n        try:return original_solve(b,f,g,payloads,H,parent,reverse,phase)\n        finally:frames.pop()\n\n    def root(b,O,g,payloads,reverse,phase):\n        frames.append(dict(m=len(O),f=None,g=g,payloads=payloads,\n                           phase=phase,reverse=reverse,restored=False))\n        try:return original_root(b,O,g,payloads,reverse,phase)\n        finally:frames.pop()\n\n    def fuse(b,f,g,F):\n        frame=frames[-1];m=frame[\'m\'];top=bottom_to_top[m]\n        block=frames[-3:];cutoff=top-3\n        assert [row[\'m\'] for row in block]==[top,top-1,top-2]\n        assert block[-1][\'f\'] is f and block[-1][\'g\'] is g\n        if top in retained_starts:\n            held=frame[\'phase\'][\'_a14_held_reverse_gaps\']\n            F0=held.pop(top);F1=held.pop(top-1)\n        else:\n            F0=scope[\'fresh_zero_flag\'](b,block[1][\'f\'][-1])\n            F1=scope[\'fresh_zero_flag\'](b,block[2][\'f\'][-1])\n        for j in range(len(frame[\'payloads\'])):\n            heads=[row[\'payloads\'][j][-1] for row in block]\n            scope[\'payload_middle\'](b,f[:cutoff],g[:cutoff],heads,\n                frame[\'payloads\'][j][:cutoff],block[1][\'f\'][-2:],F0,F1,F)\n        for flag,n in ((F0,top),(F1,top-1)):\n            mask=original_hold(b,flag)\n            b.cx(mask,frame[\'phase\'][\'return_mask\'][n]);b.free(mask)\n        for row in block:row[\'restored\']=True\n\n    def shifted_product(b,g,x,out,F):\n        if not frames:return original_product(b,g,x,out,F)\n        frame=frames[-1];m=frame[\'m\']\n        if not frame[\'reverse\'] or not frame[\'restored\'] or m not in frame_to_top:\n            return original_product(b,g,x,out,F)\n        cutoff=frame_to_top[m]-3\n        assert len(out)==m-1\n        for i in range(cutoff,len(out)):\n            with scope[\'dk\'].selected(b,g[i],g[i-1],F) as coefficient:\n                ar.mul_xor0(b,coefficient,x,out[i])\n\n    def handed(b,f,g,payloads,F,incoming,defer,Z0):\n        m=len(f)-1\n        if m not in bottom_to_top:\n            return original_handed(b,f,g,payloads,F,incoming,defer,Z0)\n        assert frames[-1][\'reverse\'] and frames[-1][\'m\']==m and not defer\n        b.cx(Z0,g[-2][0]);masks=original_restore(b,f,g,F);b.cx(Z0,g[-2][0])\n        zm=b.alloc_bits(1);b.hmr(Z0,zm[0]);b.free(Z0)\n        fuse(b,f,g,F)\n        scope[\'pivot\'](b,g,F);a=g[-1]\n        for v in payloads:shifted_product(b,g[:-1],v[-1],v[:-1],F)\n        ai,Z,saved=scalar.inverse_zero_normpair(b,a)\n        b.cz(Z,zm[0]);b.free(zm)\n        scope[\'consume\'](b,ai,Z,incoming)\n        b.cx(Z,a[0]);b.cx(Z,ai[0])\n        for v in payloads:scope[\'scale\'](b,v[-1],a,ai,False)\n        close_heads(b,f,g,ai,F,masks)\n        b.cx(Z,ai[0]);b.cx(Z,a[0]);scalar.erase_normpair(b,a,ai,Z,saved)\n        scope[\'pivot\'](b,g,F)\n\n    dcells[\'handed\'].cell_contents=handed\n    scope[\'look_solve\']=solve\n    scope[\'dk\'].shifted_product=shifted_product\n    handed_scope[\'dk\'].shifted_product=shifted_product\n    short[\'old\'].look_solve=solve\n    short[\'look_root_cached\']=root\n    return cv\n\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a16/dot_one_word.py': '"""One-word transpose dot: accumulate its constant output in the target."""\n\n\ndef selected_dot(b, g, v, target, F):\n    """target ^= D0 + F*(D0+D1), retaining only D0+D1.\n\n    At each rank t, direct accumulation into target contributes the old D0\n    term; the sole temporary receives (1+t) times the same rank product.\n    Its X measurement is repaid by that exact transposed bilinear character.\n    """\n    d = b.alloc_qubits(12)\n    N = len(g)\n    V, W = k.vandermonde(N)\n    for t in range(N):\n        rank_update(b, g, V[t], v, [W[i][t] for i in range(N)],\n                    [target, d], [1, 1 ^ t])\n    for j in range(12):\n        b.ccx(F, d[j], target[j])\n    r = b.alloc_bits(12)\n    b.hmr(d, r)\n    b.free(d)\n    field = ar.field(12)\n    for t in range(N):\n        mask = b.alloc_bits(12)\n        b.bit_store(mask, False)\n        for j in range(12):\n            value = field.mul(1 ^ t, 1 << j)\n            for i in ar.bits(value):\n                b.cx(r[i], mask[j])\n        x, p = k.expose(b, g, V[t])\n        weights = [W[i][t] for i in range(N)]\n        y, q = k.expose(b, v, weights)\n        ar.phase_product(b, x, y, mask)\n        k.unexpose(b, v, weights, q)\n        k.unexpose(b, g, V[t], p)\n        b.free(mask)\n    b.free(r)\n\n\ndef install(cv, scalar):\n    from types import FunctionType\n    scope = scalar.scope_of(cv)\n    dk = scope[\'dk\']\n    dk.selected_dot = FunctionType(selected_dot.__code__,\n                                   dict(vars(dk)), \'selected_dot\')\n    return cv\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a16/forward_cache_shift.py': '"""Move only the selected forward inverse-norm cache schedule."""\nfrom types import FunctionType\n\ndef install(cv, scalar, levels=(8, 5)):\n    scope = scalar.scope_of(cv)\n    seen = set(); changed = []\n    def visit(fn):\n        if not isinstance(fn, FunctionType) or id(fn) in seen: return\n        seen.add(id(fn))\n        cells = dict(zip(fn.__code__.co_freevars, fn.__closure__ or ()))\n        if fn.__code__.co_filename.endswith(\'/a12/forward_norm_cache.py\') and \'levels\' in cells:\n            cells[\'levels\'].cell_contents = frozenset(levels)\n            changed.append(fn.__name__)\n        for cell in fn.__closure__ or ():\n            try: value = cell.cell_contents\n            except ValueError: continue\n            if isinstance(value, FunctionType): visit(value)\n    visit(scope[\'inverse_zero\']); visit(scope[\'look_solve\'])\n    assert sorted(changed) == [\'inverse_zero\', \'solve\'], changed\n    return cv\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a16/forward_return.py': '"""Three-frame forward return in the pre-aligned middle basis.\n\nThe whole quotient-head actions are hoisted in their original order.\nA prefix of the existing middle aligned route moves before the low dots;\nits last pivot edge is restored just across the bottom scalar operation.\nEvery correlation and product measurement is repaid before its sources move.\n"""\n\ndef install(cv, scalar, top=15):\n    scope = scalar.scope_of(cv)\n    short = dict(zip(cv.__code__.co_freevars, cv.__closure__))[\'core\'].cell_contents.__globals__\n    dk, ar = scope[\'dk\'], scope[\'ar\']\n    original_solve, original_post = scope[\'look_solve\'], scope[\'direct_post\']\n    original_dot = dk.selected_dot\n    active = []\n    phases = []\n    pre_frames = {}\n    original_partial = scope[\'pre_partial\']\n    original_aligned = scope[\'aligned\']\n    TOP = top\n    assert TOP in (12, 15)\n    CUT = TOP - 3\n\n    def corr(b, v, c, count):\n        out = [b.alloc_qubits(12) for _ in range(count)]\n        N = len(c) + count - 1\n        V, W = dk.k.vandermonde(N)\n        for t in range(N):\n            ac = [W[i][t] for i in range(len(v))]\n            if any(ac):\n                dk.rank_update(b, v, ac, c, V[t][:len(c)], out, V[t][:count])\n        return out\n\n\n\n\n\n\n    def select_copy(b, x, y, target, F):\n        b.cx(x, target); b.cx(x, y)\n        for u, v in zip(y, target): b.ccx(F, u, v)\n        b.cx(x, y)\n\n    def select_phase(b, x, y, mask, F):\n        b.cz(x, mask); b.cx(x, y)\n        for u, v in zip(y, mask): b.ccz(F, u, v)\n        b.cx(x, y)\n\n    def retire_one(b, v, c, word, count, index):\n        r = b.alloc_bits(12); b.hmr(word, r); b.free(word)\n        N = len(c) + count - 1\n        V, W = dk.k.vandermonde(N)\n        field = ar.field(12)\n        for t in range(N):\n            ac = [W[i][t] for i in range(len(v))]\n            if not any(ac): continue\n            mask = b.alloc_bits(12); b.bit_store(mask, False)\n            for bit in range(12):\n                for i in ar.bits(field.mul(V[t][index], 1 << bit)):\n                    b.cx(r[i], mask[bit])\n            x, xp = dk.k.expose(b, v, ac)\n            cc = V[t][:len(c)]\n            y, yp = dk.k.expose(b, c, cc)\n            ar.phase_product(b, x, y, mask)\n            dk.k.unexpose(b, c, cc, yp); dk.k.unexpose(b, v, ac, xp)\n            b.free(mask)\n        b.free(r)\n\n\n    def flag_copy(b, F, x, out, complement=False):\n        if complement: b.x(F)\n        for u, v in zip(x, out): b.ccx(F, u, v)\n        if complement: b.x(F)\n\n    def fused(b, f, g, payloads, F2, phase):\n        f0, g0, v0, parent0 = phase[f\'_a16_{TOP}_forward_block\']\n        f1 = g0\n        assert len(f0)-1 == TOP and len(f1)-1 == TOP-1\n        assert f == f0[:-2]\n        # These ordinary q_post actions do not depend on any lower dot or\n        # head scaling. Their order and every token are unchanged.\n        parent1 = ([row[-1] for row in v0], f0[-1], f0[-2])\n        prior_qpost(b, f1[-1], [row[:-1] for row in v0], None,\n                        TOP-1, parent1, phase=phase)\n        prior_qpost(b, f0[-1], v0, None, TOP, parent0, phase=phase)\n        phase[f\'_a16_{TOP}_hoisted\'] = {TOP-1, TOP}\n        outer = phase[\'returned_gap\'].pop(TOP+1)\n        phase[f\'_a16_{TOP}_parent_debt\'] = scope[\'hold_flag\'](b, outer)\n        F0 = phase[\'returned_gap\'][TOP]\n        F1 = phase[\'returned_gap\'][TOP-1]\n        q0, q1 = f1[-2:]\n        # Undo the low prefix of the middle aligned route. The original\n        # parent return later performs only its remaining suffix.\n        with b.mark(\'aligned_route\'):\n            for x, y in zip(f[:CUT], g[:CUT]): b.cswap(F1, x, y)\n        phase[f\'_a16_{TOP}_prefix_route\'] = F1\n        for j, v in enumerate(payloads):\n            low = v[:CUT]\n            heads = [v0[j][-1], v0[j][-2], v0[j][-3]]\n            fs = corr(b, low, f[:CUT], 3)\n            # d0 = (1+F0)*q0*d1 + q1*e + (1+F0)*G0 + F0*F1corr.\n            b.cswap(F1, fs[1], fs[2])\n            ar.mul_xor0(b, q1, fs[1], heads[0])\n            b.cswap(F1, fs[1], fs[2])\n            flag_copy(b, F0, fs[1], heads[0])\n            retire_one(b, low, f[:CUT], fs[2], 3, 2)\n            b.cswap(F1, fs[0], fs[1])\n            b.cx(fs[0], heads[1])\n            product = b.alloc_qubits(12)\n            ar.mul_xor0(b, q0, fs[0], product)\n            flag_copy(b, F0, product, heads[0], True)\n            rm = b.alloc_bits(12); b.hmr(product, rm); b.free(product)\n            ar.phase_product(b, q0, fs[0], rm); b.free(rm)\n            b.cswap(F1, fs[0], fs[1])\n            temp = b.alloc_qubits(12)\n            select_copy(b, fs[0], fs[1], temp, F2)\n            flag_copy(b, F1, temp, heads[2])\n            rm = b.alloc_bits(12); b.hmr(temp, rm); b.free(temp)\n            select_phase(b, fs[0], fs[1], rm, F2); b.free(rm)\n            retire_one(b, low, f[:CUT], fs[0], 3, 0)\n            retire_one(b, low, f[:CUT], fs[1], 3, 1)\n\n            gs = corr(b, low, g[:CUT], 2)\n            flag_copy(b, F0, gs[0], heads[0], True)\n            temp = b.alloc_qubits(12)\n            select_copy(b, gs[0], gs[1], temp, F2)\n            flag_copy(b, F1, temp, heads[2], True)\n            rm = b.alloc_bits(12); b.hmr(temp, rm); b.free(temp)\n            select_phase(b, gs[0], gs[1], rm, F2); b.free(rm)\n            retire_one(b, low, g[:CUT], gs[0], 2, 0)\n            retire_one(b, low, g[:CUT], gs[1], 2, 1)\n\n    def bottom_post(b, f, g, payloads, F, H, parent, incoming, defer, phase):\n        m = len(f)-1\n        assert m == TOP-2 and defer\n        scope[\'q_post\'](b, f[-1], payloads, H, m, parent, phase=phase)\n        # Retire the one-bit zero predicate across the three-word pass.\n        # Recreate it with the ordinary inverse producer and repay its X debt.\n        scope[\'pivot\'](b, g, F); a = g[-1]\n        # Keep the explicit strip for the parameterized public prefix.\n        for i in range(CUT, len(g)-1):\n            with dk.selected(b, g[i], g[i-1], F) as coefficient:\n                for v in payloads: ar.mul_xor0(b, coefficient, v[i], v[-1])\n        Z0 = scope[\'fresh_zero_flag\'](b, a)\n        b.cx(Z0, a[0]); scope[\'pivot\'](b, g, F)\n        phase[\'heads\'][m] = scope[\'high_full_restore\'](b, f, g, F)\n        b.cx(Z0, g[-2][0])\n        zm0 = b.alloc_bits(1); b.hmr(Z0, zm0[0]); b.free(Z0)\n        fused(b, f, g, payloads, F, phase)\n        # Restore only the pivot edge while its scalar inverse is consumed.\n        # All correlation characters have already been repaid in the new basis.\n        middle_gap = phase[\'returned_gap\'][TOP-1]\n        b.cswap(middle_gap, f[CUT-1], g[CUT-1])\n        scope[\'pivot\'](b, g, F); a = g[-1]\n        ai, Z = scope[\'inverse_zero\'](b, a)\n        b.cz(Z, zm0[0]); b.free(zm0)\n        scope[\'consume\'](b, ai, Z, incoming)\n        b.cx(Z, a[0]); b.cx(Z, ai[0])\n        for v in payloads: scope[\'scale\'](b, v[-1], a, ai, True)\n        b.cx(Z, ai[0]); r = b.alloc_bits(12); b.hmr(ai, r); b.free(ai)\n        b.cx(Z, a[0]); scope[\'pivot\'](b, g, F)\n        b.cswap(middle_gap, f[CUT-1], g[CUT-1])\n        zm = b.alloc_bits(1); b.hmr(Z, zm[0]); b.free(Z)\n        return r, zm\n\n    def selected_dot(b, g, v, target, F):\n        if not active or active[-1][\'m\'] not in (TOP, TOP-1):\n            return original_dot(b, g, v, target, F)\n        frame = active[-1]\n        if frame[\'done\']: return\n        frame[\'done\'] = True\n        for i in range(CUT, len(g)):\n            with dk.selected(b, g[i], g[i-1], F) as c:\n                for row in frame[\'payloads\']:\n                    ar.mul_xor0(b, c, row[i], row[-1])\n\n    prior_qpost = scope[\'q_post\']\n    def q_post(b, q, payloads, H, m, parent, reverse=False, phase=None):\n        if not reverse and m in phase.get(f\'_a16_{TOP}_hoisted\', ()):\n            phase[f\'_a16_{TOP}_hoisted\'].remove(m)\n            return\n        return prior_qpost(b, q, payloads, H, m, parent, reverse, phase)\n\n    def post(b, f, g, payloads, F, H, parent, reverse=False,\n             incoming=None, defer=False, phase=None):\n        m = len(f)-1\n        if reverse or m not in (TOP, TOP-1, TOP-2) or f\'_a16_{TOP}_forward_block\' not in phase:\n            return original_post(b, f, g, payloads, F, H, parent,\n                                 reverse, incoming, defer, phase)\n        if m == TOP-2:\n            return bottom_post(b, f, g, payloads, F, H, parent, incoming, defer, phase)\n        active.append(dict(m=m, payloads=payloads, done=False))\n        try:\n            result = original_post(b, f, g, payloads, F, H, parent,\n                                   reverse, incoming, defer, phase)\n            if m == TOP:\n                outer = scope[\'fresh_zero_flag\'](b, f[-1])\n                scope[\'consume_flag\'](b, outer, phase.pop(f\'_a16_{TOP}_parent_debt\'))\n                phase[\'returned_gap\'][TOP+1] = outer\n                del phase[f\'_a16_{TOP}_forward_block\']\n                assert not phase.pop(f\'_a16_{TOP}_hoisted\')\n            return result\n        finally:\n            active.pop()\n\n    def aligned(b, f, g, F):\n        if phases and len(f)-1 == TOP-1 and f\'_a16_{TOP}_prefix_route\' in phases[-1]:\n            assert phases[-1].pop(f\'_a16_{TOP}_prefix_route\') == F\n            with b.mark(\'aligned_route\'):\n                for x, y in zip(f[:-2][CUT:], g[:-1][CUT:]): b.cswap(F, x, y)\n            return\n        return original_aligned(b, f, g, F)\n\n    def partial(b, f, g, payloads, F, cutoff, defer_coefficient=False):\n        m = len(f)-1\n        if m == TOP+1:\n            pre_frames[m] = (f, g, payloads)\n        if m == TOP:\n            pf, pg, pv = pre_frames.pop(TOP+1)\n            parent = ([row[-1] for row in pv], pf[-1], pf[-2])\n            assert f\'_a16_{TOP}_forward_block\' not in phases[-1]\n            phases[-1][f\'_a16_{TOP}_forward_block\'] = (f, g, payloads, parent)\n        return original_partial(b, f, g, payloads, F, cutoff, defer_coefficient)\n\n    def solve(b, f, g, payloads, H=None, parent=None, reverse=False, phase=None):\n        phases.append(phase)\n        try:\n            return original_solve(b, f, g, payloads, H, parent, reverse, phase)\n        finally:\n            phases.pop()\n\n    scope[\'aligned\'] = aligned\n    scope[\'pre_partial\'] = partial\n    scope[\'look_solve\'] = solve\n    short[\'old\'].look_solve = solve\n    scope[\'direct_post\'] = post\n    scope[\'q_post\'] = q_post\n    dk.selected_dot = selected_dot\n    return cv\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a18/three_borrowed_bit_v2.py': '"""Borrow a clean bit from complementary gap-supported words.\n\nOnly public frame dimensions and builder wire identities select the source.\nNo quantum values or simulator state are read by this installer.\n"""\nfrom pathlib import Path\nimport sys\nfrom types import FunctionType\nROOT=Path(__file__).resolve().parents[3]\nsys.path[:0]=[str(ROOT/\'research/agents/a08\'),str(ROOT/\'research/agents/a01\')]\nimport scalar_compact_v3 as scalar\np=scalar.p\n\ndef payload_middle(b,lo_f,lo_g,heads,out,q,F0,F1,F2,zero_word):\n    s0,s1,s2=heads; q0,q1=q\n    # F0=[q1=0] is an exact completed-frame identity.\n    M=mask_word(b,s0,F0,F1,True,True)\n    P0=b.alloc_qubits(12); ar.mul_xor0(b,q0,M,P0)\n    b.cx(M,P0)\n    copy_flag(b,F0,s0,P0,True); copy_flag(b,F1,s1,P0,True)\n    P1=b.alloc_qubits(12); ar.mul_xor0(b,q1,s0,P1)\n    b.cx(M,P1); copy_flag(b,F1,s0,P1,True)\n    erase_mask(b,s0,M,F0,F1,True,True)\n    Q2=b.alloc_qubits(12); b.cswap(F1,P1,Q2)\n    convolution(b,lo_f,[P0,P1],out)\n    r1=b.alloc_bits(12)\n    b.hmr(P1,r1); b.free(P1)\n    ar.phase_product(b,q1,s0,r1); b.cz(Q2,r1)\n    b.x(F1); phase_classical_parity(b,F0,F1,s0,r1); b.x(F1)\n    b.free(r1)\n    # P0 becomes Q1; M_P+M_Q=(1+F0)*s0.\n    M=b.alloc_qubits(12); b.x(F0)\n    for bit,target in zip(s0,M):b.init_and(F0,bit,target)\n    b.x(F0)\n    ar.mul_xor0(b,q0,M,P0)\n    mask=b.alloc_bits(12); b.hmr(M,mask); b.free(M)\n    phase_flag(b,F0,s0,mask,True); b.free(mask)\n    Q1=P0\n    b.cx(s1,Q1); copy_flag(b,F1,s0,Q1)\n    # F1*zero_word=0 and (1+F1)*Q2=0. The swap exposes clean Q2[0].\n    M=b.alloc_qubits(12)\n    b.cswap(F1,zero_word[0],Q2[0])\n    b.x(F0);b.x(F1);b.init_and(F0,F1,Q2[0])\n    for bit,target in zip(s0,M):b.ccx(Q2[0],bit,target)\n    b.del_and(F0,F1,Q2[0]);b.x(F1);b.x(F0)\n    b.cswap(F1,zero_word[0],Q2[0])\n    b.cx(s2,M)\n    b.cx(M,Q1); copy_flag(b,F2,s2,M); b.cx(M,Q1)\n    convolution(b,lo_g,[M,Q1,Q2],out)\n    r0=b.alloc_bits(12); r1=b.alloc_bits(12); r2=b.alloc_bits(12)\n    b.hmr(M,r0); b.hmr(Q1,r1); b.hmr(Q2,r2)\n    b.free(M); b.free(Q1); b.free(Q2)\n    M=mask_word(b,s0,F0,F1,True,False)\n    b.cz(M,r0); phase_flag(b,F0,s0,r0,True); phase_flag(b,F2,s2,r0,True)\n    ar.phase_product(b,q0,M,r1); ar.phase_product(b,q1,M,r2); b.cz(M,r1)\n    phase_flag(b,F1,s0,r1); phase_flag(b,F1,s1,r1); phase_flag(b,F2,s2,r1)\n    b.free(r0); b.free(r1); b.free(r2)\n    erase_mask(b,s0,M,F0,F1,True,False)\n\ndef install(cv):\n    scope=scalar.scope_of(cv)\n    short=dict(zip(cv.__code__.co_freevars,cv.__closure__))[\'core\'].cell_contents.__globals__\n    original=scope[\'look_solve\'];frames=[]\n    arithmetic=p.clone(payload_middle,scope)\n\n    def solve(b,f,g,payloads,H=None,parent=None,reverse=False,phase=None):\n        frames.append((f,g))\n        try:return original(b,f,g,payloads,H,parent,reverse,phase)\n        finally:frames.pop()\n\n    def middle(b,lo_f,lo_g,heads,out,q,F0,F1,F2):\n        # In both completed three-frame traversals lo_f is bottom f[:-2].\n        # Enclosing frames retain the same physical coefficient bank prefix.\n        candidates=[]\n        for pair in frames:\n            for bank in pair:\n                if len(bank)>=len(lo_f)+2 and bank[:len(lo_f)]==lo_f:\n                    candidates.append(bank[len(lo_f)+1])\n        assert candidates and all(word==candidates[0] for word in candidates)\n        return arithmetic(b,lo_f,lo_g,heads,out,q,F0,F1,F2,candidates[0])\n\n    scope[\'look_solve\']=solve;short[\'old\'].look_solve=solve\n    owners={};seen=set()\n    def visit(value):\n        if id(value) in seen:return\n        seen.add(id(value))\n        if isinstance(value,dict):\n            if \'payload_middle\' in value:owners[id(value)]=value\n            for name in (\'look_solve\',\'direct_pre\',\'direct_post\',\'handed_aligned_pre\',\'look_root_cached\'):\n                if name in value:visit(value[name])\n        elif isinstance(value,FunctionType):\n            visit(value.__globals__)\n            for cell in value.__closure__ or ():\n                try:child=cell.cell_contents\n                except ValueError:continue\n                if isinstance(child,(dict,FunctionType)):visit(child)\n    visit(scope);visit(original);visit(short[\'look_root_cached\'])\n    for owner in owners.values():owner[\'payload_middle\']=middle\n    return cv\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/a22/root_forward_return_v3.py': '"""Three-frame forward return in the pre-aligned middle basis.\n\nThe whole quotient-head actions are hoisted in their original order.\nA prefix of the existing middle aligned route moves before the low dots;\nits last pivot edge is restored just across the bottom scalar operation.\nEvery correlation and product measurement is repaid before its sources move.\n"""\n\ndef install(cv, scalar, retire_root=True):\n    scope = scalar.scope_of(cv)\n    short = dict(zip(cv.__code__.co_freevars, cv.__closure__))[\'core\'].cell_contents.__globals__\n    dk, ar = scope[\'dk\'], scope[\'ar\']\n    original_solve, original_post = scope[\'look_solve\'], scope[\'direct_post\']\n    original_dot = dk.selected_dot\n    active = []\n    phases = []\n    pre_frames = {}\n    original_partial = scope[\'pre_partial\']\n    original_aligned = scope[\'aligned\']\n    original_root_pre = short[\'root_pre_partial\']\n    original_root_action = short[\'root_action\']\n    TOP = 17\n    CUT = TOP - 3\n\n    def corr(b, v, c, count):\n        out = [b.alloc_qubits(12) for _ in range(count)]\n        N = len(c) + count - 1\n        V, W = dk.k.vandermonde(N)\n        for t in range(N):\n            ac = [W[i][t] for i in range(len(v))]\n            if any(ac):\n                dk.rank_update(b, v, ac, c, V[t][:len(c)], out, V[t][:count])\n        return out\n\n\n\n\n\n\n    def select_copy(b, x, y, target, F):\n        b.cx(x, target); b.cx(x, y)\n        for u, v in zip(y, target): b.ccx(F, u, v)\n        b.cx(x, y)\n\n    def select_phase(b, x, y, mask, F):\n        b.cz(x, mask); b.cx(x, y)\n        for u, v in zip(y, mask): b.ccz(F, u, v)\n        b.cx(x, y)\n\n    def retire_one(b, v, c, word, count, index):\n        r = b.alloc_bits(12); b.hmr(word, r); b.free(word)\n        N = len(c) + count - 1\n        V, W = dk.k.vandermonde(N)\n        field = ar.field(12)\n        for t in range(N):\n            ac = [W[i][t] for i in range(len(v))]\n            if not any(ac): continue\n            mask = b.alloc_bits(12); b.bit_store(mask, False)\n            for bit in range(12):\n                for i in ar.bits(field.mul(V[t][index], 1 << bit)):\n                    b.cx(r[i], mask[bit])\n            x, xp = dk.k.expose(b, v, ac)\n            cc = V[t][:len(c)]\n            y, yp = dk.k.expose(b, c, cc)\n            ar.phase_product(b, x, y, mask)\n            dk.k.unexpose(b, c, cc, yp); dk.k.unexpose(b, v, ac, xp)\n            b.free(mask)\n        b.free(r)\n\n\n    def flag_copy(b, F, x, out, complement=False):\n        if complement: b.x(F)\n        for u, v in zip(x, out): b.ccx(F, u, v)\n        if complement: b.x(F)\n\n    def fused(b, f, g, payloads, F2, phase):\n        f0, g0, v0, parent0 = phase[f\'_a16_{TOP}_forward_block\']\n        f1 = g0\n        assert len(f0)-1 == TOP and len(f1)-1 == TOP-1\n        assert f == f0[:-2]\n        # The special child quotient uses the reconstructed root inverse.\n        Fchild = phase[\'returned_gap\'][16]\n        # The special child formula reads this selected-pivot edge.\n        b.cswap(Fchild, f1[-3], f0[-4])\n        R = restore_root_keep(b, f1, f0[:-2], Fchild, F2, phase)\n        r_early=b.alloc_bits(12); b.hmr(R,r_early); b.free(R)\n        parent1 = ([row[-1] for row in v0], f0[-1], f0[-2])\n        phase[\'_child_scalar_context\'] = (f0[:-2], Fchild, F2)\n        try:\n            prior_qpost(b, f1[-1], [row[:-1] for row in v0], None,\n                        16, parent1, phase=phase)\n        finally:\n            del phase[\'_child_scalar_context\']\n        phase[f\'_a16_{TOP}_hoisted\'] = {16}\n        R=root_pivot(b,f1,f0[:-2],Fchild,F2,phase[\'returned_gap\'][17])\n        b.cz(R,r_early); b.free(r_early)\n        # Hoist the root\'s multiply before its dot. R is the exact clipped\n        # root pivot, and f0[-1] is its reconstructed inverse.\n        for row in v0:\n            short[\'old\'].scale(b, row[-1], R, f0[-1], False)\n        rm = b.alloc_bits(12); b.hmr(R, rm); b.free(R)\n        phase[\'_root_pivot_debt\'] = rm\n        if retire_root:\n            mask = b.alloc_bits(12); b.hmr(f0[-1], mask); b.free(f0[-1])\n            phase[\'_a22_root_inverse_return\'] = (f0[-1], mask)\n        F0 = phase[\'returned_gap\'][TOP]\n        F1 = phase[\'returned_gap\'][TOP-1]\n        q0, q1 = f1[-2:]\n        # Undo the low prefix of the middle aligned route. The original\n        # parent return later performs only its remaining suffix.\n        with b.mark(\'aligned_route\'):\n            for x, y in zip(f[:CUT], g[:CUT]): b.cswap(F1, x, y)\n        phase[f\'_a16_{TOP}_prefix_route\'] = F1\n        for j, v in enumerate(payloads):\n            low = v[:CUT]\n            heads = [v0[j][-1], v0[j][-2], v0[j][-3]]\n            fs = corr(b, low, f[:CUT], 3)\n            # d0 = (1+F0)*q0*d1 + q1*e + (1+F0)*G0 + F0*F1corr.\n            b.cswap(F1, fs[1], fs[2])\n            ar.mul_xor0(b, q1, fs[1], heads[0])\n            b.cswap(F1, fs[1], fs[2])\n            flag_copy(b, F0, fs[1], heads[0])\n            retire_one(b, low, f[:CUT], fs[2], 3, 2)\n            b.cswap(F1, fs[0], fs[1])\n            b.cx(fs[0], heads[1])\n            product = b.alloc_qubits(12)\n            ar.mul_xor0(b, q0, fs[0], product)\n            flag_copy(b, F0, product, heads[0], True)\n            rm = b.alloc_bits(12); b.hmr(product, rm); b.free(product)\n            ar.phase_product(b, q0, fs[0], rm); b.free(rm)\n            b.cswap(F1, fs[0], fs[1])\n            temp = b.alloc_qubits(12)\n            select_copy(b, fs[0], fs[1], temp, F2)\n            flag_copy(b, F1, temp, heads[2])\n            rm = b.alloc_bits(12); b.hmr(temp, rm); b.free(temp)\n            select_phase(b, fs[0], fs[1], rm, F2); b.free(rm)\n            retire_one(b, low, f[:CUT], fs[0], 3, 0)\n            retire_one(b, low, f[:CUT], fs[1], 3, 1)\n\n            gs = corr(b, low, g[:CUT], 2)\n            flag_copy(b, F0, gs[0], heads[0], True)\n            temp = b.alloc_qubits(12)\n            select_copy(b, gs[0], gs[1], temp, F2)\n            flag_copy(b, F1, temp, heads[2], True)\n            rm = b.alloc_bits(12); b.hmr(temp, rm); b.free(temp)\n            select_phase(b, gs[0], gs[1], rm, F2); b.free(rm)\n            retire_one(b, low, g[:CUT], gs[0], 2, 0)\n            retire_one(b, low, g[:CUT], gs[1], 2, 1)\n\n    def bottom_post(b, f, g, payloads, F, H, parent, incoming, defer, phase):\n        m = len(f)-1\n        assert m == TOP-2 and defer\n        scope[\'q_post\'](b, f[-1], payloads, H, m, parent, phase=phase)\n        # Retire the one-bit zero predicate across the three-word pass.\n        # Recreate it with the ordinary inverse producer and repay its X debt.\n        scope[\'pivot\'](b, g, F); a = g[-1]\n        # Keep the explicit strip for the parameterized public prefix.\n        for i in range(CUT, len(g)-1):\n            with dk.selected(b, g[i], g[i-1], F) as coefficient:\n                for v in payloads: ar.mul_xor0(b, coefficient, v[i], v[-1])\n        Z0 = scope[\'fresh_zero_flag\'](b, a)\n        b.cx(Z0, a[0]); scope[\'pivot\'](b, g, F)\n        phase[\'heads\'][m] = scope[\'high_full_restore\'](b, f, g, F)\n        b.cx(Z0, g[-2][0])\n        zm0 = b.alloc_bits(1); b.hmr(Z0, zm0[0]); b.free(Z0)\n        fused(b, f, g, payloads, F, phase)\n        # Restore only the pivot edge while its scalar inverse is consumed.\n        # All correlation characters have already been repaid in the new basis.\n        middle_gap = phase[\'returned_gap\'][TOP-1]\n        b.cswap(middle_gap, f[CUT-1], g[CUT-1])\n        b.cswap(middle_gap, f[-2], g[-1])\n        scope[\'pivot\'](b, g, F); a = g[-1]\n        ai, Z = scope[\'inverse_zero\'](b, a)\n        b.cz(Z, zm0[0]); b.free(zm0)\n        scope[\'consume\'](b, ai, Z, incoming)\n        b.cx(Z, a[0]); b.cx(Z, ai[0])\n        for v in payloads: scope[\'scale\'](b, v[-1], a, ai, True)\n        b.cx(Z, ai[0]); r = b.alloc_bits(12); b.hmr(ai, r); b.free(ai)\n        b.cx(Z, a[0]); scope[\'pivot\'](b, g, F)\n        b.cswap(middle_gap, f[CUT-1], g[CUT-1])\n        b.cswap(middle_gap, f[-2], g[-1])\n        zm = b.alloc_bits(1); b.hmr(Z, zm[0]); b.free(Z)\n        return r, zm\n\n    def selected_dot(b, g, v, target, F):\n        if not active or active[-1][\'m\'] not in (TOP, TOP-1):\n            return original_dot(b, g, v, target, F)\n        frame = active[-1]\n        if frame[\'done\']: return\n        frame[\'done\'] = True\n        for i in range(CUT, len(g)):\n            with dk.selected(b, g[i], g[i-1], F) as c:\n                for row in frame[\'payloads\']:\n                    ar.mul_xor0(b, c, row[i], row[-1])\n\n    prior_qpost = scope[\'q_post\']\n    def q_post(b, q, payloads, H, m, parent, reverse=False, phase=None):\n        if not reverse and m in phase.get(f\'_a16_{TOP}_hoisted\', ()):\n            phase[f\'_a16_{TOP}_hoisted\'].remove(m)\n            return\n        return prior_qpost(b, q, payloads, H, m, parent, reverse, phase)\n\n    def post(b, f, g, payloads, F, H, parent, reverse=False,\n             incoming=None, defer=False, phase=None):\n        m = len(f)-1\n        if reverse or m not in (TOP, TOP-1, TOP-2) or f\'_a16_{TOP}_forward_block\' not in phase:\n            return original_post(b, f, g, payloads, F, H, parent,\n                                 reverse, incoming, defer, phase)\n        if m == TOP-2:\n            return bottom_post(b, f, g, payloads, F, H, parent, incoming, defer, phase)\n        active.append(dict(m=m, payloads=payloads, done=False))\n        try:\n            result = original_post(b, f, g, payloads, F, H, parent,\n                                   reverse, incoming, defer, phase)\n            return result\n        finally:\n            active.pop()\n\n    def aligned(b, f, g, F):\n        if phases and len(f)-1 == TOP-1 and f\'_a16_{TOP}_prefix_route\' in phases[-1]:\n            assert phases[-1].pop(f\'_a16_{TOP}_prefix_route\') == F\n            with b.mark(\'aligned_route\'):\n                for x, y in zip(f[:-2][CUT+1:], g[:-1][CUT+1:]): b.cswap(F, x, y)\n            return\n        return original_aligned(b, f, g, F)\n\n    def root_pivot(b,f,g,Fchild,K,Froot):\n        q=f[-1]\n        R = b.alloc_qubits(12)\n        scope[\'pivot\'](b, g, Fchild)\n        b.ccx(Fchild, K, g[-1][0])\n        ar.mul_xor0(b, q, g[-1], R)\n        b.ccx(Fchild, K, g[-1][0])\n        scope[\'pivot\'](b, g, Fchild)\n        for x, y in zip(g[-1], R): b.ccx(Froot, x, y)\n        b.ccx(Froot, Fchild, R[0])\n        return R\n\n    def restore_root_keep(b, f, g, Fchild, K, phase):\n        held, mask = phase.pop(\'_ephemeral_root\')\n        q = f[-1]\n        Froot = scope[\'fresh_zero_flag\'](b, q)\n        R = root_pivot(b,f,g,Fchild,K,Froot)\n        phase[\'_root_early_flag\'] = Froot\n        replacement = scope[\'inverse_initialized\'](b, R)\n        held[:] = replacement\n        b.cz(held, mask); b.free(mask)\n        return R\n\n    def root_pre(b, f, g, payloads, ai, Z, F):\n        root_frames.append((f,g,payloads))\n        return original_root_pre(b,f,g,payloads,ai,Z,F)\n\n    root_frames=[]\n    def partial(b, f, g, payloads, F, cutoff, defer_coefficient=False):\n        if len(f)-1 == 16 and root_frames:\n            rf,rg,rv=root_frames.pop()\n            phases[-1][f\'_a16_{TOP}_forward_block\']=(rf,rg,rv,None)\n        return original_partial(b,f,g,payloads,F,cutoff,defer_coefficient)\n\n    def root_action(b,f,g,payloads,ai,Z,F,post,undo):\n        if not (post and not undo and root_phases and\n                f\'_a16_{TOP}_forward_block\' in root_phases[-1]):\n            return original_root_action(b,f,g,payloads,ai,Z,F,post,undo)\n        phase=root_phases[-1]\n        assert Z is False\n        if retire_root:\n            saved,mask=phase.pop(\'_a22_root_inverse_return\')\n            replacement=scope[\'inverse_initialized\'](b,g[-1])\n            saved[:]=replacement\n            b.cz(ai,mask); b.free(mask)\n        for i in range(CUT,len(g)-1):\n            with dk.selected(b,g[i],g[i-1],F) as c:\n                for row in payloads: ar.mul_xor0(b,c,row[i],row[-1])\n        for row in payloads: short[\'old\'].scale(b,row[-1],g[-1],ai,True)\n        short[\'root_restore\'](b,f,g,ai,F)\n        del phase[f\'_a16_{TOP}_forward_block\']\n        assert not phase.pop(f\'_a16_{TOP}_hoisted\')\n\n    root_phases=[]\n    original_root=short[\'look_root_cached\']\n    def root(b,O,g,payloads,reverse,phase):\n        root_phases.append(phase)\n        try:return original_root(b,O,g,payloads,reverse,phase)\n        finally:root_phases.pop()\n\n    def solve(b, f, g, payloads, H=None, parent=None, reverse=False, phase=None):\n        phases.append(phase)\n        try:\n            return original_solve(b, f, g, payloads, H, parent, reverse, phase)\n        finally:\n            phases.pop()\n\n    short[\'root_pre_partial\'] = root_pre\n    short[\'root_action\'] = root_action\n    short[\'look_root_cached\'] = root\n    scope[\'aligned\'] = aligned\n    short[\'aligned\'] = aligned\n    scope[\'pre_partial\'] = partial\n    scope[\'look_solve\'] = solve\n    short[\'old\'].look_solve = solve\n    scope[\'direct_post\'] = post\n    scope[\'q_post\'] = q_post\n    dk.selected_dot = selected_dot\n    return cv\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/r06_joint_top/joint_cached.py': '"""Compose cyclic top-row sharing with an existing one-word Hankel handoff."""\nfrom functools import lru_cache\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport inspect\nimport sys\nROOT = Path(__file__).resolve().parents[3]\nsys.path[:0] = [str(ROOT/\'research/agents/a09\'), str(ROOT/\'research/agents/a01\')]\nimport hankel_compact as h\np, ar, k = h.p, h.ar, h.kernels\n\n\n@lru_cache(None)\ndef root_terms(sources):\n    f = ar.field(12)\n    zeta = f.pow(f.primitive_element, 117)\n    points = [f.pow(zeta, i) for i in range(35)]\n    assert len(set(points)) == 35\n    V = [[f.pow(point, j) for j in range(35)] for point in points]\n    W = k.mi(V)\n    outputs = 36-sources\n    terms = tuple((tuple(V[t][:sources]),\n                   tuple(W[32-j][t] for j in range(33)),\n                   tuple(V[t][outputs-1-j] for j in range(outputs))+(V[t][34],))\n                  for t in range(35))\n    for i in range(sources):\n        for j in range(33):\n            out = [0]*(outputs+1)\n            for a, b, c in terms:\n                ab = f.mul(a[i], b[j])\n                for index, coefficient in enumerate(c):\n                    out[index] ^= f.mul(ab, coefficient)\n            expected = [int(i+j == sources-3+index) for index in range(outputs)]\n            expected.append(int(i > 0 and i+j == 33))\n            assert out == expected, (sources, i, j, out, expected)\n    return terms\n\n\ndef install(cv, root_sources=16):\n    cells = dict(zip(cv.__code__.co_freevars, cv.__closure__))\n    core = cells[\'core\'].cell_contents\n    scope = core.__globals__\n    old_hankel = scope[\'old\'].hankel\n    hcells = dict(zip(old_hankel.__code__.co_freevars, old_hankel.__closure__))\n    forward_name = \'forward_tri\' if \'forward_tri\' in hcells else \'forward\'\n    old_forward = hcells[forward_name].cell_contents\n    saved = hcells[\'saved\'].cell_contents\n    current = {\'delta\': None}\n\n    def accumulate(b, x, q, out, terms):\n        assert current[\'delta\'] is not None\n        assert len(x) == 12*root_sources and len(q) == 12*33\n        words = lambda flat: [flat[i:i+12] for i in range(0,len(flat),12)]\n        aw, bw, tw = map(words, (x,q,out))\n        tw.append(current[\'delta\'])\n        for ac, bc, tc in root_terms(root_sources):\n            p.dk.rank_update(b, aw, ac, bw, bc, tw, tc)\n\n    # Only the outer call uses these globals. Recursive calls retain the old\n    # closure and its original helper, so the extra row is produced once.\n    env = dict(old_forward.__globals__)\n    env[\'h\'] = SimpleNamespace(**dict(vars(env[\'h\']), accumulate=accumulate))\n    forward = p.clone(old_forward, env)\n\n    def hankel(b, B, A, reverse=False, delta=None):\n        if reverse:\n            assert delta is None\n            return old_hankel(b,B,A,True)\n        assert delta is not None and not saved\n        current[\'delta\'] = delta\n        try:\n            forward(b,list(reversed(B[1:])),A)\n        finally:\n            current[\'delta\'] = None\n        for i in range(len(A)//2):\n            b.swap(A[i],A[-1-i])\n        assert len(saved) == 1\n\n    scope[\'old\'].hankel = hankel\n    source = inspect.getsource(core)\n    needle = \'for i in range(17):old.ar.mul_xor0(b,E[i+1],w[i],delta)\'\n    assert source.count(needle) == 1\n    source = source.replace(needle,\n        f\'for i in range({root_sources//2},17):old.ar.mul_xor0(b,E[i+1],w[i],delta)\')\n    needle = "b.cx(delta,top);phase[\'_top_dot\']=delta"\n    assert source.count(needle) == 1\n    source = source.replace(needle, "phase[\'_top_dot\']=delta")\n    needle = \'        old.hankel(b,B[1:],A[:34])\'\n    assert source.count(needle) == 1\n    source = source.replace(needle,\n        f\'        for j in range(2,{root_sources},2):old.ar.mul_xor0(b,B[j+1],A[j],delta)\\n\'\n        \'        old.hankel(b,B[1:],A[:34],delta=delta)\\n\'\n        \'        b.cx(delta,top)\')\n    exec(compile(source, __file__, \'exec\'), scope)\n    cells[\'core\'].cell_contents = scope[core.__name__]\n    return cv\n', 'blocks/dqi/pro-q8q9-889/source/research/agents/r08_weight_handoff/weight_handoff_v3.py': '"""Share q1*s0 and retain P0 as the second-pass Q1 weight."""\nfrom pathlib import Path\nimport sys\nROOT = Path(__file__).resolve().parents[3]\nsys.path[:0] = [str(ROOT/\'research/agents/a01\'), str(ROOT/\'research/agents/a08\')]\nimport ported as p\nimport scalar_compact_v3 as scalar\n\n\ndef payload_middle(b,lo_f,lo_g,heads,out,q,F0,F1,F2):\n    s0,s1,s2=heads; q0,q1=q\n    # F0=[q1=0] is an exact completed-frame identity.\n    M=mask_word(b,s0,F0,F1,True,True)\n    P0=b.alloc_qubits(12); ar.mul_xor0(b,q0,M,P0)\n    b.cx(M,P0)\n    copy_flag(b,F0,s0,P0,True); copy_flag(b,F1,s1,P0,True)\n    P1=b.alloc_qubits(12); ar.mul_xor0(b,q1,s0,P1)\n    b.cx(M,P1); copy_flag(b,F1,s0,P1,True)\n    erase_mask(b,s0,M,F0,F1,True,True)\n    Q2=b.alloc_qubits(12); b.cswap(F1,P1,Q2)\n    convolution(b,lo_f,[P0,P1],out)\n    r1=b.alloc_bits(12)\n    b.hmr(P1,r1); b.free(P1)\n    ar.phase_product(b,q1,s0,r1); b.cz(Q2,r1)\n    b.x(F1); phase_classical_parity(b,F0,F1,s0,r1); b.x(F1)\n    b.free(r1)\n    # P0 becomes Q1; M_P+M_Q=(1+F0)*s0.\n    M=b.alloc_qubits(12); b.x(F0)\n    for bit,target in zip(s0,M):b.init_and(F0,bit,target)\n    b.x(F0)\n    ar.mul_xor0(b,q0,M,P0)\n    mask=b.alloc_bits(12); b.hmr(M,mask); b.free(M)\n    phase_flag(b,F0,s0,mask,True); b.free(mask)\n    Q1=P0\n    b.cx(s1,Q1); copy_flag(b,F1,s0,Q1)\n    # Twelve output wires also hold the conjunction while eleven bits form.\n    # The last masked bit uses Q1[0] as restored arbitrary dirty workspace.\n    M=b.alloc_qubits(12); b.x(F0); b.x(F1)\n    b.init_and(F0,F1,M[0])\n    for j in range(1,12):b.ccx(M[0],s0[j],M[j])\n    b.del_and(F0,F1,M[0])\n    b.ccx(F0,F1,Q1[0]); b.ccx(Q1[0],s0[0],M[0])\n    b.ccx(F0,F1,Q1[0]); b.ccx(Q1[0],s0[0],M[0])\n    b.x(F1); b.x(F0)\n    b.cx(s2,M)\n    b.cx(M,Q1); copy_flag(b,F2,s2,M); b.cx(M,Q1)\n    convolution(b,lo_g,[M,Q1,Q2],out)\n    r0=b.alloc_bits(12); r1=b.alloc_bits(12); r2=b.alloc_bits(12)\n    b.hmr(M,r0); b.hmr(Q1,r1); b.hmr(Q2,r2)\n    b.free(M); b.free(Q1); b.free(Q2)\n    M=mask_word(b,s0,F0,F1,True,False)\n    b.cz(M,r0); phase_flag(b,F0,s0,r0,True); phase_flag(b,F2,s2,r0,True)\n    ar.phase_product(b,q0,M,r1); ar.phase_product(b,q1,M,r2); b.cz(M,r1)\n    phase_flag(b,F1,s0,r1); phase_flag(b,F1,s1,r1); phase_flag(b,F2,s2,r1)\n    b.free(r0); b.free(r1); b.free(r2)\n    erase_mask(b,s0,M,F0,F1,True,False)\n\n\ndef install(decoder):\n    cv=scalar.converter_of(decoder)\n    core=dict(zip(cv.__code__.co_freevars,cv.__closure__))[\'core\'].cell_contents\n    scope=core.__globals__[\'old\'].look_solve.__globals__\n    assert \'payload_middle\' in scope\n    scope[\'payload_middle\']=p.clone(payload_middle,scope)\n    return decoder\n', 'blocks/dqi/pro-q8q9-889/source/research/integrated/decode_compact_integrated.py': '"""Native middle products, root and degree14 fusion, and pipelined reverse norm pairs."""\nfrom pathlib import Path\nimport sys\nimport inspect\n\nROOT = Path(__file__).resolve().parents[2]\nsys.path[:0] = [str(ROOT / \'research/agents\' / name)\n                for name in (\'a09\', \'a08\', \'a07\', \'a01\')]\nimport hankel_compact\nimport middle_fusion\nimport blocked_three_noscratch\nimport root_fused_three\nimport scalar_compact_v3\nfrom rebind_graph import private_rebind\n\np = hankel_compact.p\ncv = p.install_dot_cache(p.install_base_zero(p.install_undo_zeros(\n    p.install_child_inverse(p.install_retained_gaps(p.make_converter())))))\ncv = middle_fusion.install(cv)\ncells = dict(zip(cv.__code__.co_freevars, cv.__closure__))\nshort = cells[\'core\'].cell_contents.__globals__\nscope = short[\'old\'].direct_post.__globals__\n# The lifetime wrapper owns a small closure, so take the solver\'s actual globals.\nscope = short[\'old\'].look_solve.__globals__\nscope[\'BLOCK_LEVELS\'] = frozenset((14,))\nscope[\'block_solve\'] = p.clone(blocked_three_noscratch.look_solve, scope)\nsource = inspect.getsource(root_fused_three.look_solve)\nneedle = \'return ordinary_solve(b,f,g,payloads,H,parent,reverse,phase)\'\nassert source.count(needle) == 1\nsource = source.replace(needle, \'return block_solve(b,f,g,payloads,H,parent,reverse,phase)\')\nexec(compile(source, __file__, \'exec\'), scope)\nshort[\'old\'].look_solve = scope[\'look_solve\']\ncv = hankel_compact.install(cv)\ncv = private_rebind(cv, p.base.replacements, p.ROOT)\ndecode = scalar_compact_v3.make_decoder(converter=cv, reference_decode=p.decode)\n', 'blocks/dqi/pro-q8q9-889/source/research/production/final_root_v3.py': '"""Paid monic-root forward return and the degree-12 interior return."""\nfrom pathlib import Path\nimport importlib.util\nimport sys\n\nROOT = Path(__file__).resolve().parents[2]\nsys.path[:0] = [str(ROOT / \'research/agents\' / name)\n                for name in (\'a22\', \'a16\', \'a18\', \'a08\', \'a01\')]\nspec = importlib.util.spec_from_file_location(\n    \'root_return_clean_control\', ROOT / \'research/production/selected.py\')\nreference = importlib.util.module_from_spec(spec)\nsys.modules[spec.name] = reference\nspec.loader.exec_module(reference)\n\nimport scalar_compact_v3 as scalar\nfrom rebind_graph import private_rebind\nimport three_borrowed_bit_v2 as borrowed\nimport dot_one_word\nimport forward_return\nimport forward_cache_shift\nimport root_forward_return_v3\n\ndecode = private_rebind(reference.decode, {}, scalar.p.ROOT)\ncv = scalar.converter_of(decode)\nborrowed.install(cv)\ndot_one_word.install(cv, scalar)\nforward_cache_shift.install(cv, scalar, levels=(8, 5))\nroot_forward_return_v3.install(cv, scalar, retire_root=True)\nforward_return.install(cv, scalar, top=12)\n', 'blocks/dqi/pro-q8q9-889/source/research/production/ported.py': '"""Load the supplied prior helpers, with only the four audited entry changes.\n\nThis module avoids duplicating the supplied arithmetic in the delivered entry.\nAll newly developed arithmetic is installed separately on a private graph.\n"""\nfrom pathlib import Path\n\nKIT_ROOT = Path(__file__).resolve().parents[2]\nSOURCE_PATH = KIT_ROOT / \'prior/decode_backhalf_q7.py\'\n\n\ndef _carried_source():\n    source = SOURCE_PATH.read_text()\n    edits = (\n        (\'ROOT = Path(__file__).resolve().parent\', \'ROOT = KIT_ROOT\'),\n        ("BASE_DIR/\'decode_retimed.py\'", "BASE_DIR/\'decode_combo.py\'"),\n        (\'baseline = SimpleNamespace(**dict(vars(base.ref.baseline), scan=scan))\',\n         "baseline = SimpleNamespace(**dict(vars(base.decode.__globals__[\'baseline\']), scan=scan))"),\n        (\'\\ndecode = install_factorized_boundary(decode)\\n\', \'\\n\'),\n    )\n    for before, after in edits:\n        assert source.count(before) == 1, before\n        source = source.replace(before, after)\n    return source\n\n\nexec(compile(_carried_source(), str(SOURCE_PATH), \'exec\'), globals())\n', 'blocks/dqi/pro-q8q9-889/source/research/production/selected.py': '"""Selected back half, assembled without displaced intermediate installers.\n\nAll arithmetic helpers remain in their frozen research source files. The\nprivate-graph copy between the forward assembly and the reverse additions is\nintentional: nested closure dictionaries and their shared caches must retain\nthe same ownership as the measured composition.\n"""\nfrom pathlib import Path\nimport inspect\nimport sys\n\nROOT = Path(__file__).resolve().parents[2]\nsys.path[:0] = [str(ROOT / \'research/agents\' / name)\n                for name in (\'a12\', \'a11\', \'r06_joint_top\', \'a09\',\n                             \'a08\', \'a07\', \'a01\')]\nimport hankel_compact\nimport hankel_cache_even\nimport joint_cached\nimport forward_norm_cache\nimport known_zero_inverse\nimport middle_fusion\nimport blocked_three_noscratch\nimport root_fused_three\nimport scalar_compact_v3\nfrom rebind_graph import private_rebind\n\np = hankel_compact.p\ncv = p.install_dot_cache(p.install_base_zero(p.install_undo_zeros(\n    p.install_child_inverse(p.install_retained_gaps(p.make_converter())))))\ncv = middle_fusion.install(cv)\ncells = dict(zip(cv.__code__.co_freevars, cv.__closure__))\nshort = cells[\'core\'].cell_contents.__globals__\nscope = short[\'old\'].look_solve.__globals__\nscope[\'BLOCK_LEVELS\'] = frozenset((14,))\nscope[\'block_solve\'] = p.clone(blocked_three_noscratch.look_solve, scope)\nsource = inspect.getsource(root_fused_three.look_solve)\nneedle = \'return ordinary_solve(b,f,g,payloads,H,parent,reverse,phase)\'\nassert source.count(needle) == 1\nsource = source.replace(needle, \'return block_solve(b,f,g,payloads,H,parent,reverse,phase)\')\nexec(compile(source, __file__, \'exec\'), scope)\nshort[\'old\'].look_solve = scope[\'look_solve\']\ncv = hankel_cache_even.install(cv)\ncv = joint_cached.install(cv, root_sources=18)\ncv = private_rebind(cv, p.base.replacements, p.ROOT)\ndecode = scalar_compact_v3.make_decoder(converter=cv, reference_decode=p.decode)\nforward_norm_cache.install(scalar_compact_v3.converter_of(decode), levels=(11, 5))\nknown_zero_inverse.install(scalar_compact_v3.converter_of(decode))\n\nsys.path[:0] = [str(ROOT / \'research/agents\' / name)\n                for name in (\'a14\', \'r08_weight_handoff\', \'a13\')]\nimport reverse_fusion\nimport reverse_caches\nimport weight_handoff_v3\n\ndecode = private_rebind(decode, {}, p.ROOT)\ncv = scalar_compact_v3.converter_of(decode)\nscope = scalar_compact_v3.scope_of(cv)\nscope[\'payload_middle\'] = p.clone(weight_handoff_v3.payload_middle, scope)\nreverse_caches.install(cv)\nreverse_fusion.install(cv)\n\nfrom inverse_character_width import install_decoder\ninstall_decoder(decode)\n', 'blocks/dqi/pro-q8q9-889/source/source_sha256.json': '{\n"build.py": "678972359897dd3cfb2e633e8527bf2e13e40e0d4ac1de7c402fe1f343646e5c",\n"decode_frontier.py": "5bb75d8e5192d3f5797b1a7d41686337e3549b295dd9e0bb9f5cbd02a6dd8422",\n"decode_q8q9.py": "f9e6f30b7d639f30f46ec8048630b1286fa348e7a31fa2e45a09528a0e472dd1",\n"exp/frontier/a03/consuming_plan.json": "3b7a8711649b90384c5c86114ea29612c7526eed41290da36360b165cd572e6e",\n"exp/frontier/a03/inverse_consume17.py": "b1eb87c0528c4f38ab2cafc42e9aec6114c27f0c5604fd46de7c1eb5478246ea",\n"exp/frontier/a03/inverse_consume17_front.py": "93e7af744d8f850a330524534e57c1262dbddb52184923ca4b56b42567ccc927",\n"exp/frontier/a03/jet_norm6.py": "aca16b3f895ed0af8b7fe3f3ebd410b1c882990f97242bd12ba5e4ff7dd44093",\n"exp/frontier/a03/jet_norm6_front.py": "d2a7083dac844941bc0290d718a47aff4e44e141cb2581b374a25969ff0716c9",\n"exp/frontier/a04/nonzero_consume16.py": "7b1f3a34a0cdd63b04f4e3f455114c050d58d9dbeeddc3cd7862de2a3d9fad99",\n"exp/frontier/a04/nonzero_plan16.json": "d1d761c70a9b11c1e16709d3c55b41f074ced37ddade40303a97a9ac1d2e9684",\n"exp/frontier/a05/scalar_projection.py": "f3e9718dcdf0ef54282475e34b0d6937efdd91014b00d9ca8888c45d11f8704c",\n"exp/frontier/a05/scalar_projection12.py": "7752169a35caaf544ece5067b8e2fd74dea8503a7738dcd1d31b6238a2bfbb96",\n"exp/frontier/a05/scalar_projection_multi.py": "ba73c99ac484d4f630358a0706405caa6aaa7e4b37d7932441be4eeddbe41f6b",\n"exp/frontier/a07/decode_correction_and.py": "c622bd9fc46d9fe31e73b56409b9b7ff327a802b41a7bc7af0e4f75cfb18d0e3",\n"exp/frontier/a07/decode_prefix_gauge.py": "d978f3d9c3dd13f0c243a6fe68d2c43b8ac454e26158f7fa39b2f8e2326276c1",\n"exp/frontier/a08/decode_unitary3.py": "103ef90fdf01a03212d43e4eccdc3377d3780a300348527c2829ec43fe0b7f91",\n"exp/frontier/a08/generic19_plan16.json": "f4293936179250c08b07da8e3a8e6e559c976453250519c10109576b628416c5",\n"exp/frontier/a08/generic20_plan.json": "01024b001d85da02baa932e9d09ad0961b949bd9b531f82f2cd3fb9d1edc9f35",\n"exp/frontier/a08/inverse19_consume16.py": "aaa1960906fbbe16168d4077292e114cfdc3260379a99ec6f8c5bdaaed4ee2e2",\n"exp/frontier/a08/inverse20_consume14.py": "35695069d28bb1b2527eedf29c556af695eb7be707b39f3cf7303e5ba10409a9",\n"exp/frontier/a08/preserve_streams.py": "054e2f83527bbfc10fc2efe312b5fd3235ffa48f714afdd0187cda9a3d94beb1",\n"exp/frontier/a11/decode_low_crt.py": "a3c6c676badc43d253db3bd120b2ec9742afd2375c54e662862d0207b431a2e4",\n"exp/frontier/a11/decode_outer35.py": "ebcea2f1f32f18c2b81a073bd4a80737d2c64a35589fc384829ec985b8a696d4",\n"exp/frontier/a11/decode_outer_alias_constant.py": "a9acdddcf37da7f4d7182da27163f50b3cd718a0f267eaf13a602ba2c7f6729a",\n"exp/frontier/a11/decode_root_v8.py": "9cbb0aac09a6de8f69ae64ca77e582f4484601c19cf4680e718fe51dbfb856ea",\n"exp/frontier/a11/decode_virtual_leader.py": "8c4ad5e6637d5736587cf7e11e0e536da40cd41a69a8d41e0ca2dbf9b634912c",\n"exp/frontier/a12/decode_companion_preserve.py": "07c581855c9a53d8b9e7a71c826c9e3086e0806ec88215ad77f19d300d012fb5",\n"exp/frontier/a12/decode_trailing_companion.py": "43edd778e415aecf9fb902a1100bbd4dd6c1e84f8acc865e5f7a5f8aed83aee3",\n"exp/frontier/a12/decode_trailing_normcache.py": "7585a2ac1bf63ce1f5c96d1512d2217277084d3fb043e1a6dac599ea35e144e9",\n"exp/frontier/a14/decode_inverse20_12.py": "9eebd696711b0eb675eea23e92ad684c7d6c6ddd9cf16dc6d5744cc86a992d38",\n"exp/frontier/a14/decode_retained19.py": "3d21aa26c070483ace9a2f07c1cf551c8123a01ab88986967e8aaba28be599c6",\n"exp/frontier/a14/decode_root_v11_six.py": "0e99785e43c9d82fddb8506bbbb3231a6b1ecdfe67b56c5ef0bc62a6ff924cd0",\n"exp/frontier/a14/new19_retained2_b14_plan.json": "fa357669acffedbfc6001b4455fa8f6e9d68399b3ce77f94cdbb249f27a975fe",\n"exp/frontier/a14/new_consuming12_plan.json": "4f01021f90f32cd6f7f1f5dd4ed3b0d2febc3490f75cdb65da945f3564b14fb6",\n"exp/frontier/a16/best_pair.json": "e5dfbfc373f08c543eebac14c2efad1d0632286c5644d0987e0116628e4ff9a9",\n"exp/frontier/a16/decode_pair.py": "1aa8d476d2217ba289306397aaaaf922a51efb75a08e351849b1ffa662b07a69",\n"exp/frontier/a16/decode_partial_many.py": "9ffff9004060d9c400ab1f5d7bd49b8571acd3c3a97f76a1a57db06eaf2cb535",\n"exp/frontier/a16/fused_characters.json": "f839a6d3f50c920c3114239577f3639f70a5a948a63465bac0c6ea1d4f9da15f",\n"exp/frontier/a16/fused_pair.json": "130e8d0550788c035b6a5d3a87bdfae18a20129bdb0bea1216b7e220d05b5f9c",\n"exp/frontier/a16/phase_many_fused.py": "44867eac97c60b8db490e8ecfb31fa37a80727a61537f49afe1a1e7d2950ba07",\n"exp/frontier/a16/phase_pair.py": "9540f37cdf53c7fccc826472060a7dc29c1286de8f15dd5d0f135a77c972aa3f",\n"exp/frontier/a16/phase_pair_fused.py": "9dbe674b38066bc4cc839cc456999f9ad2c603a3e11efd6b715731420c8d227e",\n"exp/frontier/a17/a18_17_initialized_b16_cubic_plan.json": "7b8afc5934c377a16cac84143fd682bc0048363905bc22a31ad1cc4a489919c4",\n"exp/frontier/a17/a18_dag17_frozen.json": "cc7da224c3be6ce0ba0ac5b45dda4806f46362ecf356596584e8122416adf2cc",\n"exp/frontier/a17/decode_initialized19.py": "209075b5db287651e5bd23ba63db5f9d2460b6c4b9379c5a7b51584698f4b7ec",\n"exp/frontier/a17/decode_nonzero19_stream.py": "c109ec87c59b59a4f793fdb815519c5fd01a6ffd158b25d18d9e6c9f50ad42b7",\n"exp/frontier/a17/initialized17_factored.py": "cac9227a70be1946ab1e5a33a574db26eb8e1493be5c671e8f0e1e05fd736414",\n"exp/frontier/a17/nonzero19_stream_plan.json": "1e2563a65cb287b45695cb7bd0e04ca3d56f1d254207fe5e16b0aec8bae6884d",\n"exp/frontier/a17/source_preserving19_b16_result.json": "81e35c35cb3c2672a7bc87151a5c887d0281d557def7147c9a43a6e929a486d0",\n"exp/frontier/a18/control_plan.json": "ef23d37883c69c078ca78e85a78066d9012716275780f264952ae2477a483633",\n"exp/frontier/a18/decode_initialized17_retire_last.py": "7e2e81a5306e65cdaddd1c75d64f65a2c7c806781c556bed3795b719fb313729",\n"exp/frontier/a18/decode_vector_control.py": "a479d3c88bfa418a332dddd431334308efa6ef2278260973a6b219e1f6efe755",\n"exp/frontier/a18/retire_last_plan.json": "69a7d6a150c6b052e35bd14c90f073991dee3a024f60f84ea68c307564307a6b",\n"exp/frontier/a20/final_cubic_search.json": "579beda90a72bbe80598af9a7511b52a54264e03a980049d6899c664c91125d9",\n"exp/frontier/a20/initialized17_linked.py": "b3ea211876e7f2c83d552a3542370a9bbf033a6bd5330bdb32e078811b4a6485",\n"exp/frontier/root/decode_basic18.py": "8a0b7c5e76f746f6413ae3fa366cb95edb89826546081d45e76bc68b125a4293",\n"exp/frontier/root/decode_cached12.py": "960a1b3c8cd6baf753cfea1fd8261d2d4ffa4a311b65a99ecd51a0d47a57aa20",\n"exp/frontier/root/decode_cached_leaf20.py": "380d5182ac10697bf1a18d1905c9eeb90a11987b43694b54fe26bf05eb4329d4",\n"exp/frontier/root/decode_combo_v11.py": "d6328c0ab1caefe6da3057ae9c7abce8623e81b1318234cdfabf5393203f2728",\n"exp/frontier/root/decode_combo_v12.py": "c879950eaa8a9523a4068f0e9158ff90e7177ce61200e1703329abb46a69042c",\n"exp/frontier/root/decode_combo_v13.py": "bba2266c8a979fd698fa0798942aeb381ce2df25686e1edaa0d426820b117cc3",\n"exp/frontier/root/decode_combo_v14.py": "576a51deab2050ff5ecc33a5660ef7e1edff386f7dd4240bfec55b7652e1d063",\n"exp/frontier/root/decode_combo_v15.py": "0427705231814b0b88cab44fb5a0601861f1886a955c2e79941fffaf44d92aea",\n"exp/frontier/root/decode_combo_v16.py": "7da4da726679bd5adbecd67d9cd519c665806ee0de0bc29f861d9bade4b3562f",\n"exp/frontier/root/decode_combo_v18.py": "2db16922e086248c54b844e37b41e096cb75d69e25d69f3fa0a96ac959b5b18e",\n"exp/frontier/root/decode_combo_v3.py": "316564002b3f5f022eb54e879cb99aec04bdc7938104a52519d27e13fc620e0a",\n"exp/frontier/root/decode_combo_v4.py": "546e9f75459ffe01702d81e405d0fb428d37ad7a3501d2062b19570a9467b80a",\n"exp/frontier/root/decode_combo_v5.py": "8289e01576bc99d27c4461f5d4ec0b27d5580d06bd97e4a52c3762ccf9060d3a",\n"exp/frontier/root/decode_combo_v6.py": "e824c56dbdd79a82c35e441a97c96794348bae452e8103d976b43753125e3448",\n"exp/frontier/root/decode_combo_v7.py": "33b40f71e9197b5f244c8b718494abd3a50a6d1d402a9d83e6832c747afef0d5",\n"exp/frontier/root/decode_combo_v8.py": "db7001c4ad21edd793dfc01595b3a20873277916afad5469efe7d9614035d297",\n"exp/frontier/root/decode_combo_v9.py": "3d72bb33b69b4c4089bf8ed0f093e184fc7e26f652c97c46c04c8450a24d6d7f",\n"exp/frontier/root/decode_flagfree_cached.py": "cc7ebc6ba9a80067d30da170fd51970f6708d3cb3e8d1f61f3db77cb33fb5e53",\n"exp/frontier/root/decode_fullfold.py": "0e1ca40d7dd94eccaaa6df4e41bcb012020567e526846eae3fb1c22910431f1b",\n"exp/frontier/root/decode_inverse_retained.py": "972913d7bf3f94f3380785d8f9d36d1d4852e4e8ed827425025027518b501359",\n"exp/frontier/root/decode_local_forward.py": "4156e97207268b8ee77c52fad92b7c2f829a1db7224181e0bcdc2162c2071b3f",\n"exp/frontier/root/decode_local_forward_partial.py": "898ce361145c18ff840a0ef6d1338f8b363651a48db5e40152c0d1e056168cfa",\n"exp/frontier/root/decode_local_forward_sheared.py": "54b0f72646900a070023035634c2b5686c6fed68b8b5580cc7faa864a76892c8",\n"exp/frontier/root/decode_local_norm.py": "7005634d92fd182410570313d0d5f9c6972335a6d07e82ff7a2356490f75d295",\n"exp/frontier/root/decode_local_norm_partial.py": "e39912c1fe9af14b51ea43a4dae3bf607571a36c83a3b58e463489b3f762e788",\n"exp/frontier/root/decode_quantum_inverse_phase.py": "95c0a83272ac7a1fab4892392320cba36aea126e7fa98c253f01f167789c1b20",\n"exp/frontier/root/decode_shared_correction.py": "aa9b8283a6d832d1b9ab731ae0eddf3325843930924dc441d8b35252a5353199",\n"exp/frontier/root/decode_shared_correction_front.py": "e3f900249ab1e0637047fd74d0e5bee5a7a448524ea601eef7a82cd22d4bab06",\n"exp/frontier/root/decode_stream12_cached.py": "936a190f7a1e5698e5efe79cda5c4046a6ac59489613552039ef2b5e8a6417d7",\n"exp/frontier/root/quantum_phase6.py": "11ecdce1428f58a1ddea410bd814db6340db40dfbe8b41d0cf28aab9a77e8c3b",\n"exp/frontier/root/retained_inverse_k2_b14_all.json": "85364be6f0c83915c9375c0c529bca9337c8eefdd5bbb660e3047e31255cdf03",\n"exp/frontier/root/search_quantum_phase6.json": "95661d7267c2c8dfadc4b1382575cf9fc23e0d73f0d8114123b269603274b2f5",\n"exp/r-main/narrow240/adaptive_zero.py": "ba171d9bea2a1d6adc15e70bedc47a232d8777a05545ed9fb83696cf5d5aa4ba",\n"exp/r-main/narrow240/chart_retimed.py": "d5c2fb9a7575ce39353181f4d4b4453f5e8fc304436e1b07fd3c66b55b563c67",\n"exp/r-main/narrow240/converter.py": "495d357de638624718fd8fb9d5867f6f8b7eeb5f6580bf98283025fb315233f6",\n"exp/r-main/narrow240/core90.py": "202b37ada82448f9046f32c5a0435be75e5fb60cefc43c17ce8660a5d41025fc",\n"exp/r-main/narrow240/decode_combo.py": "caf90442a61dba079c736e0590f16429e489ce56fa2dfe2ccc31848677d471dc",\n"exp/r-main/narrow240/decode_retimed.py": "d63bc0866115b738c19951ffe8a2717131501bd4a19112ee95de03d064c2fa3c",\n"exp/r-main/narrow240/decode_reverse.py": "7ba16e3946fea9972f4aa8e149a99d251f2388227113bc90a3d5978fcbe01ab4",\n"exp/r-main/narrow240/decode_subs.py": "af3b439a83adeca7079d30e0cd765a7856020b5679cb9233b45a9c12dd51fc49",\n"exp/r-main/narrow240/decode_subs_combined.py": "2d80ae64aed85d1db65c82fa5e640bf75496da785f4a7739092bee4a85ea8ed8",\n"exp/r-main/narrow240/direct_converter.py": "88513922dbc03990de5ced3fbe9670602a426bd35b484f99e7e2e9c9edbff6f2",\n"exp/r-main/narrow240/direct_kernels.py": "76483349f8b6baedb3e72fc5db8d9bd3890785342d34b3e12b9d1400348bc4ff",\n"exp/r-main/narrow240/exact_subs.py": "64eb3974768c5647106860b743fbd688a836d62cbd27734637d0a41531dd9d17",\n"exp/r-main/narrow240/frozen_chart.py": "7db312bb1083148fef70037d357c9fd1d083f617604021a47d5d64f11a44f407",\n"exp/r-main/narrow240/head_stream.py": "d6996bfcb2a09344ad152d857a57d741bdbeb455847777125dc4cb2c4fa936d2",\n"exp/r-main/narrow240/high_stream.py": "e8d764bc9080546da9e690e5a59dc01b9224d4815ea20248bae75f74641115be",\n"exp/r-main/narrow240/measured_heads.py": "23a12730db559e80c448738d83a50e666ed81dea6f15786c48a7ac215df15e32",\n"exp/r-main/narrow240/norm_cache.py": "b4ea4ae17a8cdfb06471bb53495ff4821fea28d30d5b0a2519ba87214fb52892",\n"exp/r-main/narrow240/norm_keep2.py": "dea1090333845dc101174972e4f3a636039626835a88ae40b184a915225f2467",\n"exp/r-main/narrow240/permutations3.json": "e3e8be856ebd2980249c8f2ea1de625d7b829f2ff46bfa91c3780294f5781064",\n"exp/r-main/narrow240/phase_core.py": "853e647a814e0a906d06e38e8a10e626b3f9ff50778de10baab01f8cf87d8b4c",\n"exp/r-main/narrow240/phase_quantum.py": "09cc2d297e633f8c7f1eedee72f10a0db20e233b1b2c6c6d38ee06067d08e4b7",\n"exp/r-main/narrow240/rational.py": "a002afc6459113d22a24bd74d9c909978bbb927dd31df8e72cd99c29ab7e03c0",\n"exp/r-main/narrow240/replay_erase.py": "825e8feadeaedd90bf986463553e7034c1ebf6430869d6edcf057f3962d742f2",\n"exp/r-main/narrow240/retained_core_safe27.py": "77d9fc500382448045476202d5872456a5bf086ec66765c3cdc4b0d1ebda64bf",\n"exp/r-main/narrow240/retimed.py": "9437bc50457d341a7e38da4f7af86d438151e1cb7eaa06fc40bb911d051469a7",\n"exp/r-main/narrow240/reverse_exact.py": "713fac58b1dea72c682dc9bfe2506220256b5cb9afc84e3e407de3e44f1ca855",\n"exp/r-main/narrow240/root_fusion.py": "0b1fdb6e4737f01d6e308b4d2cf78d65a6fca8c54c5939164cecf194c27d79e1",\n"exp/r-main/narrow240/streamed.py": "ba56f5041f57eaf75eb029bd154f4f88b9a29b7f6508f61390a680f20265e454",\n"exp/r-main/narrow240/synth_stream.py": "1b4ef2d59237988ee64d03e3624d9aa230a98e22d977916afe968b97ef2a099b",\n"exp/r-main/narrow240/truncated_converter.py": "f2d5784890f0bcd6527249ac8f9e951cd8d3e00cda540d20faae39326da4cf25",\n"exp/r-main/narrow240/truncated_small.py": "dda93ccee7d4b7998674c1c690921e18cabe918319d44df4a1d4134f9a3f69fc",\n"playbook/kit/decoder/decode.py": "21b5c2dfe972b7f873216f0eea708d21f20f7c5206d26684bb413548d09741bb",\n"playbook/kit/decoder/exp/alternating_gauge/even_fold.py": "784cf9b2873b5fa04da4233a528d7ffd8acf9d30fb2e39dc9992d46b1438cd16",\n"playbook/kit/decoder/exp/alternating_gauge/kernel.py": "d99aa5eaef8a0df416270d8eee0fa55a9847ee7c08c3aa6ee437352e1abdd05a",\n"playbook/kit/decoder/exp/alternating_gauge/scalar.py": "2fb3e5c4ec62df647de519647a57f51c765907ce1b6ecfe2d512035a91523b15",\n"playbook/kit/decoder/exp/arithmetic/division.py": "28567073545c805e32636668cc0bb33f0dbc441a1de0dee2825774b3a33d2832",\n"playbook/kit/decoder/exp/arithmetic/field.py": "d1346ff195b02fabfda21904094410291aac6fc08e0c182a1b639c14ab743274",\n"playbook/kit/decoder/exp/arithmetic/lowwidth.py": "c536fa2b42393b83aae2bef80eeda22d83a6c799ef555178e94e09f1f5abc22a",\n"playbook/kit/decoder/exp/confluent_extract/kernel.py": "3acfc52dbf5994a85671b975be506b7967d2d075a3e1fc39f7c7cd16a8586ff3",\n"playbook/kit/decoder/exp/confluent_extract/model.py": "9b8d6fbaad6aec5da87d5247a11276c2fa177883adff9eca030a9bc2f3286387",\n"playbook/kit/decoder/exp/confluent_synthesis/kernel.py": "2eaabffce88e50ab3ce4719e764b3ed43c8bd6d8a6acf22a4461574268691c54",\n"playbook/kit/decoder/exp/convert/convert.py": "7afdafa60806df5afbc0af8fd01417a3e09fc15c5dc83fdd28d68079e78df7a0",\n"playbook/kit/decoder/exp/convert/hankel_check.py": "6d0ac6f55a70418049f46300863eeabb90a7c38b476b84db6e83d395edc3d72c",\n"playbook/kit/decoder/exp/convert/kernels.py": "f5efa2f2b008cce86a0b751d400ee75777b77b6872d1875d6fae281cd7f09ef9",\n"playbook/kit/decoder/exp/convert/repair.py": "8beceea9091a9656901290498fbdc0e4373963374b5ee8f7e98d257ece1daeaf",\n"playbook/kit/decoder/exp/convert_short_hankel/kernel.py": "6dd64a123e4bd668bbf30e7d3c6449e738c8bf693f9a31bce1c9c39810ba7ddb",\n"playbook/kit/decoder/exp/convert_short_hankel/phase34.py": "620985ba489a03cef51095157af60b7459732b72b68f85a10265f34905ed203f",\n"playbook/kit/decoder/exp/decoder/common.py": "60f9db94c70dd746a429147b25121381934737fe6a576a5526ef70e1b12007a5",\n"playbook/kit/decoder/exp/decoder/extract.py": "6597bd52d2bebedae8860acaa4aaeaf980c3d21977781382ba8926833c8f7696",\n"playbook/kit/decoder/exp/decoder/safe_bridge.py": "023c81c775967a71ef44c571200bbe3ffb8b3da443fc2816e751856fc3f0d8d9",\n"playbook/kit/decoder/exp/decoder/synthesis.py": "fdd7de226b7445007689ce2e50dbabbe77fdbc20e994ec41ac821a6ad2474c49",\n"playbook/kit/decoder/exp/fold_cached_norm/kernel.py": "6a148ebe23eae9194ec0ddb921ff6da1d362565218913170a36691c45dd7c084",\n"playbook/kit/decoder/exp/fold_cached_norm/phase34.py": "974d76939483c061e7b7995600fcdcb05e91529297725acfba41ce5ce808661f",\n"playbook/kit/decoder/exp/fold_joint/divide.py": "8456426c171f36a716a39b457a0de5206b5d4db6c8553f1bfc919248ba1de851",\n"playbook/kit/decoder/exp/fold_joint/kernel.py": "d4764e763c6ed1ff538b58c1cc628df0c9577300aae5a4d83a5338e0290e018f",\n"playbook/kit/decoder/exp/inverse12_escape/joint.py": "4bcc5614693d646bd910b9ed4748b278d2ec8dc1584bf8a45d3e465dd757cdc1",\n"playbook/kit/decoder/exp/inverse12_escape/zero1.py": "c9274f825d452082d88a98ded25a269aab247626a17032287c26ca1d8a71c724",\n"playbook/kit/decoder/exp/inverse12_escape/zero_layout.json": "45d1361fcf582282d15cc6fabeab290f8e9fd9dc4c68a3e14ffff86e9f9a537f",\n"playbook/kit/decoder/exp/inverse19_joint/kernel.py": "035152593a105b8178522a6edbf7cda3b709d8a86f51ae2148e71b01a1274913",\n"playbook/kit/decoder/exp/inverse19_joint/layout.json": "0adee49170ae597ed5a7df4d4d0df0e495fba49013f6acd9ae6a3f06b9ad4ab4",\n"playbook/kit/decoder/exp/inverse_search/baseline.py": "7aecb36f951122861fe0443eee92f1d65a10484fc86f2e535f3db44f5dd553da",\n"playbook/kit/decoder/exp/inverse_search/inverse20.py": "e36162777e51361a7ceb47ae6b86a34ce492b23c2fde8083113b03985b98060d",\n"playbook/kit/decoder/exp/inverse_search/inverse20_iso.py": "1b4ffd1d35b12ca4f327c8885063fb76eef459eeabed1fddae371e2e4bb4f635",\n"playbook/kit/decoder/exp/inverse_search/optimized.json": "b18cfd6025658d02e38adfbe0ea7eca09960bbbdd458abe1ee1d539e4d3f6ca7",\n"playbook/kit/decoder/exp/inverse_search/optimized_isotopy2.json": "28424aa2f6b0b92b7b3b4f6bc62feda566746feb9a9f21660ed5103420eae606",\n"playbook/kit/decoder/exp/inverse_width/kernel.py": "accdd344715e6972ebe22e1679ca872d455a75466f1b1236b83bb636f6466409",\n"playbook/kit/decoder/exp/inverse_width/partial5.json": "202ddf3a0a62530ac4f4028774e87c729cba6cc85dd465570a59e882a83cf4f7",\n"playbook/kit/decoder/exp/inverse_width_subspaces/inverse19.json": "c868b9fc56f05ec89d34375ad086490691237211ecdc99b4a0c0b6e86b468a3f",\n"playbook/kit/decoder/exp/inverse_width_subspaces/kernel19.py": "2cb588e60d5fe22ea59d625164e0a0d285cbcd352a80f5bf86f53f76050c9f4b",\n"playbook/kit/decoder/exp/jet24/kernel.py": "f490e6205c678135dc2273d1f02e5ce9540087995f52c37e174b51dce0b4a5f6",\n"playbook/kit/decoder/exp/jet24/pro_jet.py": "168975c442003084f644852b5bc012c44f7ea681d224ca736bf438fffffd377b",\n"playbook/kit/decoder/exp/mask_projection/kernel.py": "80148ee048d1fd9aaace9637d2e652a7f46dbcb4306913f053437af4a2119088",\n"playbook/kit/decoder/exp/mul99/kernels.py": "8f93d0a23f465cf72045e02f702b0b40d629b72d1618952a8fc79c22e23ee6ec",\n"playbook/kit/decoder/exp/mul99/phase36.py": "f6806918afd6cd6c09c23221511483cf19962df916689b87e1d718c459ef6852",\n"playbook/kit/decoder/exp/mul99/schedules.py": "9d16c576747f9d556b857f846c4029131f977cd6eb15d8a37cff4b84b0919c66",\n"playbook/kit/decoder/exp/noscratch/kernels.py": "fa64bb5d5c359c507ac9a488c3f73363d5390bd43a4697daa67df57c322d7370",\n"playbook/kit/decoder/exp/noscratch/phase39.py": "a47a3ad3ce8787f868b93b40156e3496eead5200b187244b5db13ec0f0c79f1a",\n"playbook/kit/decoder/exp/noscratch/schedules.py": "5a6ea45a0596cbf0d96bac07a8d93dc473e7d51013d6776d1163deec9f7d7c72",\n"playbook/kit/decoder/exp/partial_gauge/prefix.py": "dba10093b9314daf696ebd8fbc85543dbcf73534eb45d2303dc9ce53189b0652",\n"playbook/kit/decoder/exp/partial_gauge/scalar.py": "7b3cb8eece8ccbc42462f13dcab9ef6b253581a06ce4d14906d88a541774de29",\n"playbook/kit/decoder/exp/partial_gauge/synthesis_noflag.py": "cb316d80cdcfa636735156171a2b4d0bb57be22e4c5495806709f224fd0ea8ae",\n"playbook/kit/decoder/exp/partial_gauge/tail.py": "cef30a29c419aef478fbf9c98f4277ab4a1fc63ca52e95d6025d592f16c7aff3",\n"playbook/kit/decoder/exp/phase24_tradeoff/kernel.py": "984fc30f4c834b8f3cc1a7276f26f54801c01c8fd53b757e9ba767325059711a",\n"playbook/kit/decoder/exp/phase24_tradeoff/phase6_six_search.json": "afb39dcf8748b1b1419a5e8db6b5440be74db6687464e6e0a9052a08b168fdb0",\n"playbook/kit/decoder/exp/phase24_tradeoff/phase6small.py": "28590db3c6a0843fedc2894bb9899df45a7b99ea4544ce6b7517e739e672fe32",\n"playbook/kit/decoder/exp/safe_scalar/converter.py": "771a242b7910c3e59f27bc82c72e8f7b86f9cb201d84ca3712ba7fd7e28a2841",\n"playbook/kit/decoder/exp/safe_scalar/kernel.py": "72fbedc923066741e50cf4372de75ade749844c3554005b2f72ea5594190e84a",\n"playbook/kit/decoder/exp/safe_scalar/phase.py": "30fdcd43d2234c6476344c1608449684199c8f3e8fb403f96aca80ae2379b048",\n"playbook/kit/decoder/exp/scan/scan.py": "cae856cc1c26fad1a9f7528f87edf0f04428aabdc6db172c3755e99c9cd07b09",\n"playbook/kit/decoder/exp/tail_alias/divide.py": "46d4c7c4c94a06e9e244c030934ab45092ad6793b530c501226e2edb62a3ee5c",\n"playbook/kit/decoder/exp/tail_alias/kernel.py": "3452ed2c039c73d3b1f45743f1ddeb40b88ffc32178b4052115f17bf18ba48ff",\n"playbook/kit/decoder/exp/zero_padding_synthesis/kernel.py": "da56f931ec61c2e51b44039e4c7938206534b24090690f03a61286c8d1f51149",\n"playbook/kit/decoder/final_decoder/conversion.py": "b3d05d20180fec5ab2f484cdca9e69da537bc1b5988ebf61984be68ef1383462",\n"playbook/kit/decoder/final_decoder/decode.py": "a5f5dcb7be195b8a5e345c3d475c0eb807dbdf5e62192d0753088442f12627ff",\n"playbook/kit/decoder/final_decoder/synthesis.py": "56ca15de2471de8dbe583294069283c25e306393e9f2630ad9420ec3eb8999a2",\n"playbook/kit/decoder/playbook/exp/cf_check/audit.py": "c3211bbccebce2183b1a24385812dc505d1041ca7244274bbdffa75a95f37b33",\n"prior/decode_backhalf_q7.py": "dc57b9cdcf84db7979c9bfca5dd3ce16a87fc6388ca5f4713d59557cd324e4d7",\n"research/agents/a01/ported_exact_cycle.py": "03a7be08a98dc25d83d7c7267afae52bcaa6ffde80646e20cda033e390733f5c",\n"research/agents/a01/rebind_graph.py": "05d21da19d16e6bb07d498e79ac5bc6784597bcdbbab5c5be1d4ebd469a76cdc",\n"research/agents/a02/cache_support.py": "07f22152a66aca40c2dde437f2c409e411db3334898ef231476a02face02ce3f",\n"research/agents/a03/local_norm.py": "58ff8c71bbcd61b5ee52c3697c6ebb4c9cf0451776b2850ae8fa4354b3e94cad",\n"research/agents/a07/blocked_three_noscratch.py": "c3910e2a210a19515d21a562ac62c5505ad1fae5f1b05412dbbce9c4583a6ea5",\n"research/agents/a07/ephemeral_root_block.py": "061484b59ae8cf456558bbb3605c3102219312e89d31a6600eefc19649ad4f04",\n"research/agents/a07/middle_fusion.py": "300e1a0ae81b98b196f34f7217dd367d414e4b96a1599082d2ef931084f69093",\n"research/agents/a07/root_fused_three.py": "7105bdd4341a4a7b54daf3c475093cacb62e056df485e37037ba150706c6864a",\n"research/agents/a08/scalar_compact_v3.py": "eef91b4196d4190c47d7fdf411714076ce8921f92d6ab863d062cdbf333933ff",\n"research/agents/a09/hankel_compact.py": "8c5801a08b01078639909a2cda78c8e362a28f292df1cb7b6ee7142c7b1df3f5",\n"research/agents/a11/hankel_cache_even.py": "e0299322eba37a434c9550607617295bb3e8dd7548b02676b4b3c890da1143c0",\n"research/agents/a12/forward_norm_cache.py": "fdc3ca3c6827fc3e8360c451e7be18dc7026a4312d2c1bfa918b19822579c086",\n"research/agents/a12/known_zero_inverse.py": "a4bfe3d447ec2acc0328d652af4e86ea5c9152b4572a25b2253cfa04953cba02",\n"research/agents/a12/reverse_root_lifetime.py": "ef1aa163adb7385fd9c039debffdd1eb9678f5c5a63f8638d665600a18126e92",\n"research/agents/a13/inverse_character_width.py": "c7547ac14707e26c765c2affa0ba648c1e8f7ea05163b1ad4b9d6576fb2d0acd",\n"research/agents/a14/reverse_caches.py": "62d0ba19518d2917920db5b5b4fe439d40c74831d6c480233d9be119267822bb",\n"research/agents/a14/reverse_fusion.py": "abe7a0ecb077c31b74b58de196246771cf5cdad2d08eff1eba3325a3122983bc",\n"research/agents/a16/dot_one_word.py": "442aee4cc58de694a4d175d3f6de6dabddc00148e00c861a696d6388f7813525",\n"research/agents/a16/forward_cache_shift.py": "06eb3a11489589b3fd24571cb46ad8a6edad8dd783c27e94688fce27e9902de9",\n"research/agents/a16/forward_return.py": "83b195271a39d75ab6a5ca79a8be11450ea75f6834e598fd9ea92f0ec3b1d7e9",\n"research/agents/a18/three_borrowed_bit_v2.py": "f55e132d7eb34f14013b19742977c92a44c2c93b9e6f0d033f9a74f189a81211",\n"research/agents/a22/root_forward_return_v3.py": "22e51ca14af4d96c488ed8e4d6a9116bd50844f7c811ed8d23aaa2c141c193b0",\n"research/agents/r06_joint_top/joint_cached.py": "8c261be25ce6aedc37fe3b59b3521128126c570df84bbf01839c7c3104058a5f",\n"research/agents/r08_weight_handoff/weight_handoff_v3.py": "b283ff0a6719b96cc81fd3542e738c7587bd6e686595fba7d109237896007295",\n"research/integrated/decode_compact_integrated.py": "71bd6c09d275ed21ceeffa9715de2c06e61a0104f6dd7f4923a268e9737a1c8f",\n"research/production/final_root_v3.py": "41cb8e7f79f8b32ae6f8f9e73a7c826c04996252bd9a932ae7feba9ddab561f1",\n"research/production/ported.py": "1a32f91845c5ec02a515ca8097596a14ac1db7bc00bdd888fc32e5140f165eb3",\n"research/production/selected.py": "3532c37aa7967f48f1f204e0f40b42744b6480f6d11c55ebbe44c949d4d2becb"\n}\n'}
SOURCE_HASHES = {'notebooks/launch.py': '72179eb4e59baa4adbcab0fa3f8b56cccde7d6ca359a7f06e17d45efaeb14d7b', 'notebooks/take.py': '64b527642010d9b07d9ad3e9b1aeec112feecac14a1053b208cfb46033652e8a', 'notebooks/check-submission': 'b1a0e5190fb0d96df85cc10184c834d33e100f1627b2fbbc4f5604e0f240284b', 'notebooks/extra-format.py': '6593250be3d89ee0942e01daf7251d0e5b351148786aa851d7f07dfe5dcac588', 'LICENSE': 'cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30', 'blocks/dqi/pro-q8q9-889/source/build.py': '678972359897dd3cfb2e633e8527bf2e13e40e0d4ac1de7c402fe1f343646e5c', 'blocks/dqi/pro-q8q9-889/source/decode_frontier.py': '5bb75d8e5192d3f5797b1a7d41686337e3549b295dd9e0bb9f5cbd02a6dd8422', 'blocks/dqi/pro-q8q9-889/source/decode_q8q9.py': 'f9e6f30b7d639f30f46ec8048630b1286fa348e7a31fa2e45a09528a0e472dd1', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a03/inverse_consume17.py': 'b1eb87c0528c4f38ab2cafc42e9aec6114c27f0c5604fd46de7c1eb5478246ea', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a03/inverse_consume17_front.py': '93e7af744d8f850a330524534e57c1262dbddb52184923ca4b56b42567ccc927', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a03/jet_norm6.py': 'aca16b3f895ed0af8b7fe3f3ebd410b1c882990f97242bd12ba5e4ff7dd44093', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a03/jet_norm6_front.py': 'd2a7083dac844941bc0290d718a47aff4e44e141cb2581b374a25969ff0716c9', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a04/nonzero_consume16.py': '7b1f3a34a0cdd63b04f4e3f455114c050d58d9dbeeddc3cd7862de2a3d9fad99', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a05/scalar_projection.py': 'f3e9718dcdf0ef54282475e34b0d6937efdd91014b00d9ca8888c45d11f8704c', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a05/scalar_projection12.py': '7752169a35caaf544ece5067b8e2fd74dea8503a7738dcd1d31b6238a2bfbb96', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a05/scalar_projection_multi.py': 'ba73c99ac484d4f630358a0706405caa6aaa7e4b37d7932441be4eeddbe41f6b', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a07/decode_correction_and.py': 'c622bd9fc46d9fe31e73b56409b9b7ff327a802b41a7bc7af0e4f75cfb18d0e3', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a07/decode_prefix_gauge.py': 'd978f3d9c3dd13f0c243a6fe68d2c43b8ac454e26158f7fa39b2f8e2326276c1', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a08/decode_unitary3.py': '103ef90fdf01a03212d43e4eccdc3377d3780a300348527c2829ec43fe0b7f91', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a08/inverse19_consume16.py': 'aaa1960906fbbe16168d4077292e114cfdc3260379a99ec6f8c5bdaaed4ee2e2', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a08/inverse20_consume14.py': '35695069d28bb1b2527eedf29c556af695eb7be707b39f3cf7303e5ba10409a9', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a08/preserve_streams.py': '054e2f83527bbfc10fc2efe312b5fd3235ffa48f714afdd0187cda9a3d94beb1', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a11/decode_low_crt.py': 'a3c6c676badc43d253db3bd120b2ec9742afd2375c54e662862d0207b431a2e4', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a11/decode_outer35.py': 'ebcea2f1f32f18c2b81a073bd4a80737d2c64a35589fc384829ec985b8a696d4', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a11/decode_outer_alias_constant.py': 'a9acdddcf37da7f4d7182da27163f50b3cd718a0f267eaf13a602ba2c7f6729a', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a11/decode_root_v8.py': '9cbb0aac09a6de8f69ae64ca77e582f4484601c19cf4680e718fe51dbfb856ea', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a11/decode_virtual_leader.py': '8c4ad5e6637d5736587cf7e11e0e536da40cd41a69a8d41e0ca2dbf9b634912c', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a12/decode_companion_preserve.py': '07c581855c9a53d8b9e7a71c826c9e3086e0806ec88215ad77f19d300d012fb5', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a12/decode_trailing_companion.py': '43edd778e415aecf9fb902a1100bbd4dd6c1e84f8acc865e5f7a5f8aed83aee3', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a12/decode_trailing_normcache.py': '7585a2ac1bf63ce1f5c96d1512d2217277084d3fb043e1a6dac599ea35e144e9', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a14/decode_inverse20_12.py': '9eebd696711b0eb675eea23e92ad684c7d6c6ddd9cf16dc6d5744cc86a992d38', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a14/decode_retained19.py': '3d21aa26c070483ace9a2f07c1cf551c8123a01ab88986967e8aaba28be599c6', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a14/decode_root_v11_six.py': '0e99785e43c9d82fddb8506bbbb3231a6b1ecdfe67b56c5ef0bc62a6ff924cd0', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/decode_pair.py': '1aa8d476d2217ba289306397aaaaf922a51efb75a08e351849b1ffa662b07a69', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/decode_partial_many.py': '9ffff9004060d9c400ab1f5d7bd49b8571acd3c3a97f76a1a57db06eaf2cb535', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/phase_many_fused.py': '44867eac97c60b8db490e8ecfb31fa37a80727a61537f49afe1a1e7d2950ba07', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/phase_pair.py': '9540f37cdf53c7fccc826472060a7dc29c1286de8f15dd5d0f135a77c972aa3f', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/phase_pair_fused.py': '9dbe674b38066bc4cc839cc456999f9ad2c603a3e11efd6b715731420c8d227e', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a17/decode_initialized19.py': '209075b5db287651e5bd23ba63db5f9d2460b6c4b9379c5a7b51584698f4b7ec', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a17/decode_nonzero19_stream.py': 'c109ec87c59b59a4f793fdb815519c5fd01a6ffd158b25d18d9e6c9f50ad42b7', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a17/initialized17_factored.py': 'cac9227a70be1946ab1e5a33a574db26eb8e1493be5c671e8f0e1e05fd736414', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a18/decode_initialized17_retire_last.py': '7e2e81a5306e65cdaddd1c75d64f65a2c7c806781c556bed3795b719fb313729', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a18/decode_vector_control.py': 'a479d3c88bfa418a332dddd431334308efa6ef2278260973a6b219e1f6efe755', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a20/initialized17_linked.py': 'b3ea211876e7f2c83d552a3542370a9bbf033a6bd5330bdb32e078811b4a6485', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_basic18.py': '8a0b7c5e76f746f6413ae3fa366cb95edb89826546081d45e76bc68b125a4293', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_cached12.py': '960a1b3c8cd6baf753cfea1fd8261d2d4ffa4a311b65a99ecd51a0d47a57aa20', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_cached_leaf20.py': '380d5182ac10697bf1a18d1905c9eeb90a11987b43694b54fe26bf05eb4329d4', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v11.py': 'd6328c0ab1caefe6da3057ae9c7abce8623e81b1318234cdfabf5393203f2728', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v12.py': 'c879950eaa8a9523a4068f0e9158ff90e7177ce61200e1703329abb46a69042c', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v13.py': 'bba2266c8a979fd698fa0798942aeb381ce2df25686e1edaa0d426820b117cc3', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v14.py': '576a51deab2050ff5ecc33a5660ef7e1edff386f7dd4240bfec55b7652e1d063', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v15.py': '0427705231814b0b88cab44fb5a0601861f1886a955c2e79941fffaf44d92aea', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v16.py': '7da4da726679bd5adbecd67d9cd519c665806ee0de0bc29f861d9bade4b3562f', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v18.py': '2db16922e086248c54b844e37b41e096cb75d69e25d69f3fa0a96ac959b5b18e', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v3.py': '316564002b3f5f022eb54e879cb99aec04bdc7938104a52519d27e13fc620e0a', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v4.py': '546e9f75459ffe01702d81e405d0fb428d37ad7a3501d2062b19570a9467b80a', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v5.py': '8289e01576bc99d27c4461f5d4ec0b27d5580d06bd97e4a52c3762ccf9060d3a', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v6.py': 'e824c56dbdd79a82c35e441a97c96794348bae452e8103d976b43753125e3448', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v7.py': '33b40f71e9197b5f244c8b718494abd3a50a6d1d402a9d83e6832c747afef0d5', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v8.py': 'db7001c4ad21edd793dfc01595b3a20873277916afad5469efe7d9614035d297', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_combo_v9.py': '3d72bb33b69b4c4089bf8ed0f093e184fc7e26f652c97c46c04c8450a24d6d7f', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_flagfree_cached.py': 'cc7ebc6ba9a80067d30da170fd51970f6708d3cb3e8d1f61f3db77cb33fb5e53', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_fullfold.py': '0e1ca40d7dd94eccaaa6df4e41bcb012020567e526846eae3fb1c22910431f1b', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_inverse_retained.py': '972913d7bf3f94f3380785d8f9d36d1d4852e4e8ed827425025027518b501359', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_local_forward.py': '4156e97207268b8ee77c52fad92b7c2f829a1db7224181e0bcdc2162c2071b3f', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_local_forward_partial.py': '898ce361145c18ff840a0ef6d1338f8b363651a48db5e40152c0d1e056168cfa', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_local_forward_sheared.py': '54b0f72646900a070023035634c2b5686c6fed68b8b5580cc7faa864a76892c8', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_local_norm.py': '7005634d92fd182410570313d0d5f9c6972335a6d07e82ff7a2356490f75d295', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_local_norm_partial.py': 'e39912c1fe9af14b51ea43a4dae3bf607571a36c83a3b58e463489b3f762e788', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_quantum_inverse_phase.py': '95c0a83272ac7a1fab4892392320cba36aea126e7fa98c253f01f167789c1b20', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_shared_correction.py': 'aa9b8283a6d832d1b9ab731ae0eddf3325843930924dc441d8b35252a5353199', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_shared_correction_front.py': 'e3f900249ab1e0637047fd74d0e5bee5a7a448524ea601eef7a82cd22d4bab06', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/decode_stream12_cached.py': '936a190f7a1e5698e5efe79cda5c4046a6ac59489613552039ef2b5e8a6417d7', 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/quantum_phase6.py': '11ecdce1428f58a1ddea410bd814db6340db40dfbe8b41d0cf28aab9a77e8c3b', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/adaptive_zero.py': 'ba171d9bea2a1d6adc15e70bedc47a232d8777a05545ed9fb83696cf5d5aa4ba', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/chart_retimed.py': 'd5c2fb9a7575ce39353181f4d4b4453f5e8fc304436e1b07fd3c66b55b563c67', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/converter.py': '495d357de638624718fd8fb9d5867f6f8b7eeb5f6580bf98283025fb315233f6', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/core90.py': '202b37ada82448f9046f32c5a0435be75e5fb60cefc43c17ce8660a5d41025fc', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/decode_combo.py': 'caf90442a61dba079c736e0590f16429e489ce56fa2dfe2ccc31848677d471dc', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/decode_retimed.py': 'd63bc0866115b738c19951ffe8a2717131501bd4a19112ee95de03d064c2fa3c', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/decode_reverse.py': '7ba16e3946fea9972f4aa8e149a99d251f2388227113bc90a3d5978fcbe01ab4', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/decode_subs.py': 'af3b439a83adeca7079d30e0cd765a7856020b5679cb9233b45a9c12dd51fc49', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/decode_subs_combined.py': '2d80ae64aed85d1db65c82fa5e640bf75496da785f4a7739092bee4a85ea8ed8', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/direct_converter.py': '88513922dbc03990de5ced3fbe9670602a426bd35b484f99e7e2e9c9edbff6f2', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/direct_kernels.py': '76483349f8b6baedb3e72fc5db8d9bd3890785342d34b3e12b9d1400348bc4ff', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/exact_subs.py': '64eb3974768c5647106860b743fbd688a836d62cbd27734637d0a41531dd9d17', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/frozen_chart.py': '7db312bb1083148fef70037d357c9fd1d083f617604021a47d5d64f11a44f407', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/head_stream.py': 'd6996bfcb2a09344ad152d857a57d741bdbeb455847777125dc4cb2c4fa936d2', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/high_stream.py': 'e8d764bc9080546da9e690e5a59dc01b9224d4815ea20248bae75f74641115be', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/measured_heads.py': '23a12730db559e80c448738d83a50e666ed81dea6f15786c48a7ac215df15e32', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/norm_cache.py': 'b4ea4ae17a8cdfb06471bb53495ff4821fea28d30d5b0a2519ba87214fb52892', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/norm_keep2.py': 'dea1090333845dc101174972e4f3a636039626835a88ae40b184a915225f2467', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/phase_core.py': '853e647a814e0a906d06e38e8a10e626b3f9ff50778de10baab01f8cf87d8b4c', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/phase_quantum.py': '09cc2d297e633f8c7f1eedee72f10a0db20e233b1b2c6c6d38ee06067d08e4b7', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/rational.py': 'a002afc6459113d22a24bd74d9c909978bbb927dd31df8e72cd99c29ab7e03c0', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/replay_erase.py': '825e8feadeaedd90bf986463553e7034c1ebf6430869d6edcf057f3962d742f2', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/retained_core_safe27.py': '77d9fc500382448045476202d5872456a5bf086ec66765c3cdc4b0d1ebda64bf', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/retimed.py': '9437bc50457d341a7e38da4f7af86d438151e1cb7eaa06fc40bb911d051469a7', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/reverse_exact.py': '713fac58b1dea72c682dc9bfe2506220256b5cb9afc84e3e407de3e44f1ca855', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/root_fusion.py': '0b1fdb6e4737f01d6e308b4d2cf78d65a6fca8c54c5939164cecf194c27d79e1', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/streamed.py': 'ba56f5041f57eaf75eb029bd154f4f88b9a29b7f6508f61390a680f20265e454', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/synth_stream.py': '1b4ef2d59237988ee64d03e3624d9aa230a98e22d977916afe968b97ef2a099b', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/truncated_converter.py': 'f2d5784890f0bcd6527249ac8f9e951cd8d3e00cda540d20faae39326da4cf25', 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/truncated_small.py': 'dda93ccee7d4b7998674c1c690921e18cabe918319d44df4a1d4134f9a3f69fc', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/decode.py': '21b5c2dfe972b7f873216f0eea708d21f20f7c5206d26684bb413548d09741bb', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/alternating_gauge/even_fold.py': '784cf9b2873b5fa04da4233a528d7ffd8acf9d30fb2e39dc9992d46b1438cd16', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/alternating_gauge/kernel.py': 'd99aa5eaef8a0df416270d8eee0fa55a9847ee7c08c3aa6ee437352e1abdd05a', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/alternating_gauge/scalar.py': '2fb3e5c4ec62df647de519647a57f51c765907ce1b6ecfe2d512035a91523b15', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/arithmetic/division.py': '28567073545c805e32636668cc0bb33f0dbc441a1de0dee2825774b3a33d2832', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/arithmetic/field.py': 'd1346ff195b02fabfda21904094410291aac6fc08e0c182a1b639c14ab743274', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/arithmetic/lowwidth.py': 'c536fa2b42393b83aae2bef80eeda22d83a6c799ef555178e94e09f1f5abc22a', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/confluent_extract/kernel.py': '3acfc52dbf5994a85671b975be506b7967d2d075a3e1fc39f7c7cd16a8586ff3', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/confluent_extract/model.py': '9b8d6fbaad6aec5da87d5247a11276c2fa177883adff9eca030a9bc2f3286387', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/confluent_synthesis/kernel.py': '2eaabffce88e50ab3ce4719e764b3ed43c8bd6d8a6acf22a4461574268691c54', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/convert/convert.py': '7afdafa60806df5afbc0af8fd01417a3e09fc15c5dc83fdd28d68079e78df7a0', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/convert/hankel_check.py': '6d0ac6f55a70418049f46300863eeabb90a7c38b476b84db6e83d395edc3d72c', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/convert/kernels.py': 'f5efa2f2b008cce86a0b751d400ee75777b77b6872d1875d6fae281cd7f09ef9', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/convert/repair.py': '8beceea9091a9656901290498fbdc0e4373963374b5ee8f7e98d257ece1daeaf', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/convert_short_hankel/kernel.py': '6dd64a123e4bd668bbf30e7d3c6449e738c8bf693f9a31bce1c9c39810ba7ddb', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/convert_short_hankel/phase34.py': '620985ba489a03cef51095157af60b7459732b72b68f85a10265f34905ed203f', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/decoder/common.py': '60f9db94c70dd746a429147b25121381934737fe6a576a5526ef70e1b12007a5', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/decoder/extract.py': '6597bd52d2bebedae8860acaa4aaeaf980c3d21977781382ba8926833c8f7696', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/decoder/safe_bridge.py': '023c81c775967a71ef44c571200bbe3ffb8b3da443fc2816e751856fc3f0d8d9', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/decoder/synthesis.py': 'fdd7de226b7445007689ce2e50dbabbe77fdbc20e994ec41ac821a6ad2474c49', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/fold_cached_norm/kernel.py': '6a148ebe23eae9194ec0ddb921ff6da1d362565218913170a36691c45dd7c084', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/fold_cached_norm/phase34.py': '974d76939483c061e7b7995600fcdcb05e91529297725acfba41ce5ce808661f', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/fold_joint/divide.py': '8456426c171f36a716a39b457a0de5206b5d4db6c8553f1bfc919248ba1de851', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/fold_joint/kernel.py': 'd4764e763c6ed1ff538b58c1cc628df0c9577300aae5a4d83a5338e0290e018f', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse12_escape/joint.py': '4bcc5614693d646bd910b9ed4748b278d2ec8dc1584bf8a45d3e465dd757cdc1', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse12_escape/zero1.py': 'c9274f825d452082d88a98ded25a269aab247626a17032287c26ca1d8a71c724', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse19_joint/kernel.py': '035152593a105b8178522a6edbf7cda3b709d8a86f51ae2148e71b01a1274913', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_search/baseline.py': '7aecb36f951122861fe0443eee92f1d65a10484fc86f2e535f3db44f5dd553da', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_search/inverse20.py': 'e36162777e51361a7ceb47ae6b86a34ce492b23c2fde8083113b03985b98060d', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_search/inverse20_iso.py': '1b4ffd1d35b12ca4f327c8885063fb76eef459eeabed1fddae371e2e4bb4f635', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_width/kernel.py': 'accdd344715e6972ebe22e1679ca872d455a75466f1b1236b83bb636f6466409', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_width_subspaces/kernel19.py': '2cb588e60d5fe22ea59d625164e0a0d285cbcd352a80f5bf86f53f76050c9f4b', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/jet24/kernel.py': 'f490e6205c678135dc2273d1f02e5ce9540087995f52c37e174b51dce0b4a5f6', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/jet24/pro_jet.py': '168975c442003084f644852b5bc012c44f7ea681d224ca736bf438fffffd377b', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/mask_projection/kernel.py': '80148ee048d1fd9aaace9637d2e652a7f46dbcb4306913f053437af4a2119088', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/mul99/kernels.py': '8f93d0a23f465cf72045e02f702b0b40d629b72d1618952a8fc79c22e23ee6ec', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/mul99/phase36.py': 'f6806918afd6cd6c09c23221511483cf19962df916689b87e1d718c459ef6852', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/mul99/schedules.py': '9d16c576747f9d556b857f846c4029131f977cd6eb15d8a37cff4b84b0919c66', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/noscratch/kernels.py': 'fa64bb5d5c359c507ac9a488c3f73363d5390bd43a4697daa67df57c322d7370', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/noscratch/phase39.py': 'a47a3ad3ce8787f868b93b40156e3496eead5200b187244b5db13ec0f0c79f1a', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/noscratch/schedules.py': '5a6ea45a0596cbf0d96bac07a8d93dc473e7d51013d6776d1163deec9f7d7c72', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/partial_gauge/prefix.py': 'dba10093b9314daf696ebd8fbc85543dbcf73534eb45d2303dc9ce53189b0652', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/partial_gauge/scalar.py': '7b3cb8eece8ccbc42462f13dcab9ef6b253581a06ce4d14906d88a541774de29', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/partial_gauge/synthesis_noflag.py': 'cb316d80cdcfa636735156171a2b4d0bb57be22e4c5495806709f224fd0ea8ae', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/partial_gauge/tail.py': 'cef30a29c419aef478fbf9c98f4277ab4a1fc63ca52e95d6025d592f16c7aff3', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/phase24_tradeoff/kernel.py': '984fc30f4c834b8f3cc1a7276f26f54801c01c8fd53b757e9ba767325059711a', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/phase24_tradeoff/phase6small.py': '28590db3c6a0843fedc2894bb9899df45a7b99ea4544ce6b7517e739e672fe32', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/safe_scalar/converter.py': '771a242b7910c3e59f27bc82c72e8f7b86f9cb201d84ca3712ba7fd7e28a2841', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/safe_scalar/kernel.py': '72fbedc923066741e50cf4372de75ade749844c3554005b2f72ea5594190e84a', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/safe_scalar/phase.py': '30fdcd43d2234c6476344c1608449684199c8f3e8fb403f96aca80ae2379b048', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/scan/scan.py': 'cae856cc1c26fad1a9f7528f87edf0f04428aabdc6db172c3755e99c9cd07b09', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/tail_alias/divide.py': '46d4c7c4c94a06e9e244c030934ab45092ad6793b530c501226e2edb62a3ee5c', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/tail_alias/kernel.py': '3452ed2c039c73d3b1f45743f1ddeb40b88ffc32178b4052115f17bf18ba48ff', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/zero_padding_synthesis/kernel.py': 'da56f931ec61c2e51b44039e4c7938206534b24090690f03a61286c8d1f51149', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/final_decoder/conversion.py': 'b3d05d20180fec5ab2f484cdca9e69da537bc1b5988ebf61984be68ef1383462', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/final_decoder/decode.py': 'a5f5dcb7be195b8a5e345c3d475c0eb807dbdf5e62192d0753088442f12627ff', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/final_decoder/synthesis.py': '56ca15de2471de8dbe583294069283c25e306393e9f2630ad9420ec3eb8999a2', 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/playbook/exp/cf_check/audit.py': 'c3211bbccebce2183b1a24385812dc505d1041ca7244274bbdffa75a95f37b33', 'blocks/dqi/pro-q8q9-889/source/prior/decode_backhalf_q7.py': 'dc57b9cdcf84db7979c9bfca5dd3ce16a87fc6388ca5f4713d59557cd324e4d7', 'blocks/dqi/pro-q8q9-889/source/research/agents/a01/ported_exact_cycle.py': '03a7be08a98dc25d83d7c7267afae52bcaa6ffde80646e20cda033e390733f5c', 'blocks/dqi/pro-q8q9-889/source/research/agents/a01/rebind_graph.py': '05d21da19d16e6bb07d498e79ac5bc6784597bcdbbab5c5be1d4ebd469a76cdc', 'blocks/dqi/pro-q8q9-889/source/research/agents/a02/cache_support.py': '07f22152a66aca40c2dde437f2c409e411db3334898ef231476a02face02ce3f', 'blocks/dqi/pro-q8q9-889/source/research/agents/a03/local_norm.py': '58ff8c71bbcd61b5ee52c3697c6ebb4c9cf0451776b2850ae8fa4354b3e94cad', 'blocks/dqi/pro-q8q9-889/source/research/agents/a07/blocked_three_noscratch.py': 'c3910e2a210a19515d21a562ac62c5505ad1fae5f1b05412dbbce9c4583a6ea5', 'blocks/dqi/pro-q8q9-889/source/research/agents/a07/ephemeral_root_block.py': '061484b59ae8cf456558bbb3605c3102219312e89d31a6600eefc19649ad4f04', 'blocks/dqi/pro-q8q9-889/source/research/agents/a07/middle_fusion.py': '300e1a0ae81b98b196f34f7217dd367d414e4b96a1599082d2ef931084f69093', 'blocks/dqi/pro-q8q9-889/source/research/agents/a07/root_fused_three.py': '7105bdd4341a4a7b54daf3c475093cacb62e056df485e37037ba150706c6864a', 'blocks/dqi/pro-q8q9-889/source/research/agents/a08/scalar_compact_v3.py': 'eef91b4196d4190c47d7fdf411714076ce8921f92d6ab863d062cdbf333933ff', 'blocks/dqi/pro-q8q9-889/source/research/agents/a09/hankel_compact.py': '8c5801a08b01078639909a2cda78c8e362a28f292df1cb7b6ee7142c7b1df3f5', 'blocks/dqi/pro-q8q9-889/source/research/agents/a11/hankel_cache_even.py': 'e0299322eba37a434c9550607617295bb3e8dd7548b02676b4b3c890da1143c0', 'blocks/dqi/pro-q8q9-889/source/research/agents/a12/forward_norm_cache.py': 'fdc3ca3c6827fc3e8360c451e7be18dc7026a4312d2c1bfa918b19822579c086', 'blocks/dqi/pro-q8q9-889/source/research/agents/a12/known_zero_inverse.py': 'a4bfe3d447ec2acc0328d652af4e86ea5c9152b4572a25b2253cfa04953cba02', 'blocks/dqi/pro-q8q9-889/source/research/agents/a12/reverse_root_lifetime.py': 'ef1aa163adb7385fd9c039debffdd1eb9678f5c5a63f8638d665600a18126e92', 'blocks/dqi/pro-q8q9-889/source/research/agents/a13/inverse_character_width.py': 'c7547ac14707e26c765c2affa0ba648c1e8f7ea05163b1ad4b9d6576fb2d0acd', 'blocks/dqi/pro-q8q9-889/source/research/agents/a14/reverse_caches.py': '62d0ba19518d2917920db5b5b4fe439d40c74831d6c480233d9be119267822bb', 'blocks/dqi/pro-q8q9-889/source/research/agents/a14/reverse_fusion.py': 'abe7a0ecb077c31b74b58de196246771cf5cdad2d08eff1eba3325a3122983bc', 'blocks/dqi/pro-q8q9-889/source/research/agents/a16/dot_one_word.py': '442aee4cc58de694a4d175d3f6de6dabddc00148e00c861a696d6388f7813525', 'blocks/dqi/pro-q8q9-889/source/research/agents/a16/forward_cache_shift.py': '06eb3a11489589b3fd24571cb46ad8a6edad8dd783c27e94688fce27e9902de9', 'blocks/dqi/pro-q8q9-889/source/research/agents/a16/forward_return.py': '83b195271a39d75ab6a5ca79a8be11450ea75f6834e598fd9ea92f0ec3b1d7e9', 'blocks/dqi/pro-q8q9-889/source/research/agents/a18/three_borrowed_bit_v2.py': 'f55e132d7eb34f14013b19742977c92a44c2c93b9e6f0d033f9a74f189a81211', 'blocks/dqi/pro-q8q9-889/source/research/agents/a22/root_forward_return_v3.py': '22e51ca14af4d96c488ed8e4d6a9116bd50844f7c811ed8d23aaa2c141c193b0', 'blocks/dqi/pro-q8q9-889/source/research/agents/r06_joint_top/joint_cached.py': '8c261be25ce6aedc37fe3b59b3521128126c570df84bbf01839c7c3104058a5f', 'blocks/dqi/pro-q8q9-889/source/research/agents/r08_weight_handoff/weight_handoff_v3.py': 'b283ff0a6719b96cc81fd3542e738c7587bd6e686595fba7d109237896007295', 'blocks/dqi/pro-q8q9-889/source/research/integrated/decode_compact_integrated.py': '71bd6c09d275ed21ceeffa9715de2c06e61a0104f6dd7f4923a268e9737a1c8f', 'blocks/dqi/pro-q8q9-889/source/research/production/final_root_v3.py': '41cb8e7f79f8b32ae6f8f9e73a7c826c04996252bd9a932ae7feba9ddab561f1', 'blocks/dqi/pro-q8q9-889/source/research/production/ported.py': '1a32f91845c5ec02a515ca8097596a14ac1db7bc00bdd888fc32e5140f165eb3', 'blocks/dqi/pro-q8q9-889/source/research/production/selected.py': '3532c37aa7967f48f1f204e0f40b42744b6480f6d11c55ebbe44c949d4d2becb', 'blocks/dqi/pro-q8q9-889/source/source_sha256.json': 'dfe4ea312733267885e28bd770f1a8af56bee4f82157f1964ab33919003a1f54'}
INPUT_SOURCES = {'blocks/dqi/pro-q8q9-889/source/exp/frontier/a03/consuming_plan.json': {'sha256': '3b7a8711649b90384c5c86114ea29612c7526eed41290da36360b165cd572e6e', 'size': 40192}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a04/nonzero_plan16.json': {'sha256': 'd1d761c70a9b11c1e16709d3c55b41f074ced37ddade40303a97a9ac1d2e9684', 'size': 38647}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a08/generic19_plan16.json': {'sha256': 'f4293936179250c08b07da8e3a8e6e559c976453250519c10109576b628416c5', 'size': 33806}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a08/generic20_plan.json': {'sha256': '01024b001d85da02baa932e9d09ad0961b949bd9b531f82f2cd3fb9d1edc9f35', 'size': 25230}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a14/new19_retained2_b14_plan.json': {'sha256': 'fa357669acffedbfc6001b4455fa8f6e9d68399b3ce77f94cdbb249f27a975fe', 'size': 26688}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a14/new_consuming12_plan.json': {'sha256': '4f01021f90f32cd6f7f1f5dd4ed3b0d2febc3490f75cdb65da945f3564b14fb6', 'size': 20231}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/best_pair.json': {'sha256': 'e5dfbfc373f08c543eebac14c2efad1d0632286c5644d0987e0116628e4ff9a9', 'size': 3536}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/fused_characters.json': {'sha256': 'f839a6d3f50c920c3114239577f3639f70a5a948a63465bac0c6ea1d4f9da15f', 'size': 150358}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a16/fused_pair.json': {'sha256': '130e8d0550788c035b6a5d3a87bdfae18a20129bdb0bea1216b7e220d05b5f9c', 'size': 9876}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a17/a18_17_initialized_b16_cubic_plan.json': {'sha256': '7b8afc5934c377a16cac84143fd682bc0048363905bc22a31ad1cc4a489919c4', 'size': 25367}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a17/a18_dag17_frozen.json': {'sha256': 'cc7da224c3be6ce0ba0ac5b45dda4806f46362ecf356596584e8122416adf2cc', 'size': 1669}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a17/nonzero19_stream_plan.json': {'sha256': '1e2563a65cb287b45695cb7bd0e04ca3d56f1d254207fe5e16b0aec8bae6884d', 'size': 26937}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a17/source_preserving19_b16_result.json': {'sha256': '81e35c35cb3c2672a7bc87151a5c887d0281d557def7147c9a43a6e929a486d0', 'size': 25862}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a18/control_plan.json': {'sha256': 'ef23d37883c69c078ca78e85a78066d9012716275780f264952ae2477a483633', 'size': 1918}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a18/retire_last_plan.json': {'sha256': '69a7d6a150c6b052e35bd14c90f073991dee3a024f60f84ea68c307564307a6b', 'size': 1829}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/a20/final_cubic_search.json': {'sha256': '579beda90a72bbe80598af9a7511b52a54264e03a980049d6899c664c91125d9', 'size': 4598}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/retained_inverse_k2_b14_all.json': {'sha256': '85364be6f0c83915c9375c0c529bca9337c8eefdd5bbb660e3047e31255cdf03', 'size': 21315}, 'blocks/dqi/pro-q8q9-889/source/exp/frontier/root/search_quantum_phase6.json': {'sha256': '95661d7267c2c8dfadc4b1382575cf9fc23e0d73f0d8114123b269603274b2f5', 'size': 1249}, 'blocks/dqi/pro-q8q9-889/source/exp/r-main/narrow240/permutations3.json': {'sha256': 'e3e8be856ebd2980249c8f2ea1de625d7b829f2ff46bfa91c3780294f5781064', 'size': 5640}, 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse12_escape/zero_layout.json': {'sha256': '45d1361fcf582282d15cc6fabeab290f8e9fd9dc4c68a3e14ffff86e9f9a537f', 'size': 325}, 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse19_joint/layout.json': {'sha256': '0adee49170ae597ed5a7df4d4d0df0e495fba49013f6acd9ae6a3f06b9ad4ab4', 'size': 758}, 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_search/optimized.json': {'sha256': 'b18cfd6025658d02e38adfbe0ea7eca09960bbbdd458abe1ee1d539e4d3f6ca7', 'size': 967}, 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_search/optimized_isotopy2.json': {'sha256': '28424aa2f6b0b92b7b3b4f6bc62feda566746feb9a9f21660ed5103420eae606', 'size': 996}, 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_width/partial5.json': {'sha256': '202ddf3a0a62530ac4f4028774e87c729cba6cc85dd465570a59e882a83cf4f7', 'size': 495}, 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/inverse_width_subspaces/inverse19.json': {'sha256': 'c868b9fc56f05ec89d34375ad086490691237211ecdc99b4a0c0b6e86b468a3f', 'size': 864}, 'blocks/dqi/pro-q8q9-889/source/playbook/kit/decoder/exp/phase24_tradeoff/phase6_six_search.json': {'sha256': 'afb39dcf8748b1b1419a5e8db6b5440be74db6687464e6e0a9052a08b168fdb0', 'size': 2840}}
MEMBERS = {'invsqrt.kmb': {'sha256': '67482db6c53896348eadba126924868ed3011d7e1576a6d0abb6f7c137ae4997', 'size': 468649}, 'dqi.kmb': {'sha256': 'af180f734f787925800a500e558f54dc36f6533a6217130addcc922036f34ef8', 'size': 570831616}, 'dfthcsos/angle_k_0.kmb': {'sha256': '9c31da0c1d047e7dcac638df807d3cef96eac6f736a8838a0ae6943e9c46c74f', 'size': 400395}, 'dfthcsos/angle_k_1.kmb': {'sha256': '39b8d10098be097f75148639ee302249bdabe112a9278a48565eadb0cb86810b', 'size': 400453}, 'dfthcsos/angle_k_2.kmb': {'sha256': '2e8dfdeb69c496e9e510fe99722153663760980d45c35ad9b72e79ba799dde49', 'size': 394618}, 'dfthcsos/angle_k_3.kmb': {'sha256': '13817cf8d36ea112e62d7863edde2193aa5907dd0605612c4f0e1e54bacc173b', 'size': 396782}, 'dfthcsos/angle_k_4.kmb': {'sha256': '8dce8f4a380998d818bf8a08e4f3daa62aaeb31a25d084aa57549f931a937f3d', 'size': 356137}, 'dfthcsos/angle_k_5.kmb': {'sha256': '42cf7d76ef0f0b36640a9e65005214fdd4d9a9bb82b8cef6e658dbbdd8417099', 'size': 310562}, 'dfthcsos/angle_k_6.kmb': {'sha256': '7fd7bd333b1427ecf546b0c27717ed9428e255e5165c4db14e5f95f8af45c838', 'size': 351298}, 'dfthcsos/angle_k_7.kmb': {'sha256': '809a775b98836ee00e8783d3f616bb8517bb3a3621978bc704d20b543a8df4ec', 'size': 304732}, 'dfthcsos/angle_k_8.kmb': {'sha256': '42d0deb376981d0470bebdd370ca2adf79685b4bf3e6909be262e2beced030ec', 'size': 352967}, 'dfthcsos/angle_k_9.kmb': {'sha256': 'f1c49e7cf8e4f3a9adaffe167d2bbe21497a52eca2c7d09f765fc5a125e304c3', 'size': 307434}, 'dfthcsos/angle_k_10.kmb': {'sha256': 'e75f6b0685d754bf76f3f4d62589bf1d07b118faeeb0ab8745833733286db2fc', 'size': 352755}, 'dfthcsos/angle_k_11.kmb': {'sha256': '773e54f6f0d7c3f769db433f263d541cdcff1076fb75109740907e3249996141', 'size': 308137}, 'dfthcsos/angle_k_12.kmb': {'sha256': 'd86ca7ddd984059ded765b953cc936388517a7ca5789676ae90910a2be89769d', 'size': 352979}, 'dfthcsos/angle_k_13.kmb': {'sha256': '50d82cf65116f06c0fe469ebba312f72675b277263461ab88b4d15eca947c050', 'size': 308119}, 'dfthcsos/angle_k_14.kmb': {'sha256': 'ae2070998e31b7e05d01183914df20b6974c1760bddf90d1adac86ea75c7bf3f', 'size': 352701}, 'dfthcsos/submission.npz': {'sha256': '47427ee032d3719931abeea0ac88f5a6656971eabd492b9d499cae1b42c6b34b', 'size': 1999484}}
EXPECTED = {'invsqrt': {'status': 'PASS', 'score': 100692.58221354822, 'toffoli_equivalent': 1624.0739066701326, 'toffoli_count': 1327.4830924039607, 'max_toffoli_count': 1497, 't_count': 6.999961509104017, 'rotation_count': 41.0, 'rotation_eps': 2.434041480874084e-13, 'num_qubits': 62, 'reaction_depth': 805, 'num_ops': 47248, 'verified_shots': 200540928, 'worst_error_over_tolerance': 0.9993237694056958, 'worst_x': 161, 'circuit_sha256': '67482db6c53896348eadba126924868ed3011d7e1576a6d0abb6f7c137ae4997'}, 'dqi': {'status': 'PASS', 'score': 402702373901.5095, 'm': 4095, 'n': 70, 'degree': 12, 'r': 2016, 'verified_shots': 102400, 'value_errors': 0, 'phase_errors': 2, 'empirical_error_rate': 1.953125e-05, 'effective_error_rate': 0.0009488155716079551, 'log2_delta': -128.0, 'repetition_factor': 1.2013880602795501, 'mean_error_weight': 31.046737331746634, 'toffoli_count': 424128.4020996094, 'max_toffoli_count': 486571, 'reaction_depth': 68788, 'num_ops': 62925554, 'num_qubits': 889, 'num_bits': 103753, 'circuit_sha256': 'af180f734f787925800a500e558f54dc36f6533a6217130addcc922036f34ef8'}, 'dfthcsos': {'status': 'PASS', 'score': 337798090.4642403, 'lambda_eff': 48.29564456953158, 'toffoli_count': 16266.0, 'num_qubits': 430, 'brot': 8, 'batch_size': 5, 'num_circuits': 15, 'reaction_depth': 165, 'num_ops': 580308, 'verified_shots': 78080, 'submission_sha256': '883bf7652efa4a8b534ae0a486c7f9991a31faabc6bf66bf76bc40df96291601'}}
PARTS = {'invsqrt': '67482db6c53896348eadba126924868ed3011d7e1576a6d0abb6f7c137ae4997', 'dqi': 'af180f734f787925800a500e558f54dc36f6533a6217130addcc922036f34ef8', 'dfthcsos': '883bf7652efa4a8b534ae0a486c7f9991a31faabc6bf66bf76bc40df96291601'}
REFERENCE = {'invsqrt': 2076774.233715, 'dqi': 24439463583672.93, 'dfthcsos': 3355599588.3155375}
BLOCKS = {'invsqrt': {'id': 'yoyobar-67482db6', 'title': "yoyobar's 62-qubit circuit, with Ahmed Berat Özer's phase patch", 'part': '67482db6c53896348eadba126924868ed3011d7e1576a6d0abb6f7c137ae4997'}, 'dqi': {'id': 'pro-q8q9-889', 'title': "Pro's front half and back half, joined at 889 qubits", 'part': 'af180f734f787925800a500e558f54dc36f6533a6217130addcc922036f34ef8'}, 'dfthcsos': {'id': 'alexchilton-5a61ac4b', 'title': "Ahmed Berat Özer's weights on Atakan Aldemir's words, as Alex Chilton ships them", 'part': '883bf7652efa4a8b534ae0a486c7f9991a31faabc6bf66bf76bc40df96291601'}}
BUILDS = {'dqi': {'member': 'dqi.kmb', 'script': 'blocks/dqi/pro-q8q9-889/source/build.py'}}
NOTEBOOK = {'assembled_from_commit': '1896439b0689e9d1cf3049ad7439ef82d66e652f', 'blocks_changed_since_commit': False}

import hashlib, importlib.metadata, json, math, os, pathlib, platform, shutil, subprocess, sys, tempfile, time, zipfile
sys.dont_write_bytecode = True
START_TIME = time.monotonic()
INPUT = pathlib.Path(os.environ.get('QCOMP_INPUT', '/kaggle/input'))
WORKING = pathlib.Path(os.environ.get('QCOMP_WORKING', '/kaggle/working'))
WORKING.mkdir(parents=True, exist_ok=True)
SESSION = pathlib.Path(tempfile.mkdtemp(prefix='qcomp-'))
ROOT, OUT = SESSION / 'source', SESSION / 'build'
SUB = OUT / 'submission'
(ROOT / 'data').mkdir(parents=True)
SUB.mkdir(parents=True)
# Use at most the Kaggle CPU budget, one numerical-library thread per process.
if hasattr(os, 'sched_getaffinity'):
    affinity = sorted(os.sched_getaffinity(0))[:4]
    os.sched_setaffinity(0, affinity)
else:
    affinity = list(range(min(4, os.cpu_count() or 1)))
WORKERS = len(affinity)
for key in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[key] = '1'
TIMINGS = []


def sha256(path):
    with open(path, 'rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


folders = [p.parents[1] for p in sorted(INPUT.rglob('kickmix.so')) if p.parent.name == 'kickmix']
# A dataset may carry its own copies of Kickmix: the official inputs are the first folder whose files all match
# their pinned SHA-256.
COMP = next((folder for folder in folders if all(
    (folder / name).is_file() and sha256(folder / name) == digest for name, digest in COMPETITION_FILES.items())), None)
assert COMP, f'No folder holds the official inputs with their pinned SHA-256; Kickmix found in {folders}'
assert sys.version_info[:2] == (3, 13), 'Select the pinned Python 3.13 CPU image for kickmix.so'
(ROOT / 'data/comp').symlink_to(COMP)
for name, body in SOURCES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(body, encoding='utf-8')
    assert sha256(path) == SOURCE_HASHES[name], name
# Block files too large to embed come from an attached dataset or notebook output, public or private: each is found
# among the inputs by name, size and SHA-256.
for name, want in INPUT_SOURCES.items():
    path = ROOT / name
    found = next((p for p in sorted(INPUT.rglob(path.name)) if p.is_file() and p.stat().st_size == want['size']
                  and sha256(p) == want['sha256']), None)
    assert found, f'{name}: no input file has SHA-256 {want["sha256"]}; attach the inputs the Run section names'
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(found, path)
    print(f'{path.name} taken from {found}')
(ROOT / 'members.json').write_text(json.dumps(MEMBERS))


def run(*args, record):
    """Run one embedded script or official verifier in its own Python, keeping stdout as a record."""
    started = time.monotonic()
    result = subprocess.run([sys.executable, '-I', '-B', str(ROOT / 'notebooks/launch.py'), *map(str, args)],
                            cwd=ROOT, capture_output=True, text=True)
    TIMINGS.append({'command': list(map(str, args)), 'seconds': time.monotonic() - started,
                    'returncode': result.returncode})
    (OUT / record).write_text(result.stdout)
    (OUT / (record + '.stderr')).write_text(result.stderr)
    print(f'{pathlib.Path(str(args[0])).name}: {TIMINGS[-1]["seconds"]:.1f} s', flush=True)
    if result.returncode:
        print(result.stdout[-5000:], result.stderr[-5000:])
        raise RuntimeError(f'{args[0]} exited {result.returncode}')
    return result.stdout


BUILT = OUT / 'built'


def build(task):
    """Build a block's member, one file or a folder of them, from its embedded source, the way its authors run it,
    and check the SHA-256 of every file."""
    spec = BUILDS[task]
    target = BUILT / spec['member']
    target.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    # Our own code, run as a script: its folder leads sys.path, and Kickmix comes from the authenticated files.
    env = {**os.environ, 'PYTHONPATH': str(ROOT / 'data/comp/kickmix'), 'PYTHONNOUSERSITE': '1'}
    result = subprocess.run([sys.executable, '-B', str(ROOT / spec['script']), str(target)], cwd=SESSION, env=env,
                            capture_output=True, text=True)
    TIMINGS.append({'command': ['build', spec['script']], 'seconds': time.monotonic() - started,
                    'returncode': result.returncode})
    (OUT / f'{task}.build.txt').write_text(result.stdout)
    (OUT / f'{task}.build.stderr').write_text(result.stderr)
    if result.returncode:
        print(result.stdout[-5000:], result.stderr[-5000:])
        raise RuntimeError(f'{spec["script"]} exited {result.returncode}')
    names = [name for name in MEMBERS if name == spec['member'] or name.startswith(spec['member'] + '/')]
    assert names, spec['member']
    for name in names:
        digest = sha256(BUILT / name)
        assert digest == MEMBERS[name]['sha256'], f'{name}: built {digest}, not the verified file'
    record = json.loads(target.with_suffix('.build.json').read_text())
    if 'summary' in record:
        print(f'{spec["member"]} built in {TIMINGS[-1]["seconds"]:.0f} s: {record["summary"]}')
    else:
        print(f'{spec["member"]} built in {TIMINGS[-1]["seconds"]:.0f} s: Q {record["Q"]}, static Toffolis '
              f'{record["T_static"]:,}, SHA-256 {digest[:12]}..., the circuit the verifier passed')
    return record


VERIFIERS = {'invsqrt': ['data/comp/verifiers/invsqrt/verify_submission_invsqrt.py', 'invsqrt.kmb', '--max-workers'],
             'dqi': ['data/comp/verifiers/dqi/verify_submission_dqi.py', 'dqi.kmb', '--max-workers'],
             'dfthcsos': ['data/comp/verifiers/dfthcsos/verify.py', 'dfthcsos']}
CHECKED = ['status', 'score', 'num_qubits', 'toffoli_count', 'verified_shots', 'toffoli_equivalent', 't_count',
           'rotation_count', 'repetition_factor', 'value_errors', 'phase_errors', 'lambda_eff']


def verify(task):
    """Run the unmodified official verifier at its defaults and compare with the recorded run."""
    script, target, *workers = VERIFIERS[task]
    report = json.loads(run(script, SUB / target, *([*workers, WORKERS] if workers else []),
                            record=f'{task}.official.json'))
    expected = EXPECTED[task]
    assert report['status'] == 'PASS', report
    for key in CHECKED:
        if key in expected:
            assert math.isclose(report[key], expected[key], rel_tol=1e-9, abs_tol=1e-12) if isinstance(
                expected[key], float) else report[key] == expected[key], (task, key, report[key], expected[key])
    part = report.get('circuit_sha256', report.get('submission_sha256'))
    assert part == PARTS[task], (task, part)
    print(f'{task}: PASS | Q {report["num_qubits"]} | cost {report["score"]:,.6g} | '
          f'{REFERENCE[task] / report["score"]:.4f}x below the reference | same as the recorded run')
    return report


ENVIRONMENT = {'python': platform.python_version(), 'platform': platform.platform(),
               'packages': {name: importlib.metadata.version(name) for name in ['numpy', 'scipy']},
               'workers': WORKERS, 'affinity': affinity, 'numeric_threads': 1,
               'kickmix_sha256': sha256(COMP / 'kickmix/kickmix.so'),
               'seed': 'official default: derived from each circuit SHA-256, empty salt', **NOTEBOOK}
print(f'{len(COMPETITION_FILES)} official input files authenticated; {len(SOURCES)} embedded files written; '
      f'workers {WORKERS}; no network, installs or training')

In [ ]:
REBUILD = False  # True: build from the source here; False: take the built file from the inputs
BUILT_RECORDS = {task: build(task) for task in BUILDS} if REBUILD else {}

In [ ]:
taken = json.loads(run('notebooks/take.py', INPUT, SUB, ROOT / 'members.json', BUILT, record='take.json'))
for name, where in taken['taken'].items():
    print(f'{name:26} {MEMBERS[name]["sha256"][:12]}  from {where}')

In [ ]:
VERIFY = False  # False: report the recorded official runs instead of rerunning them
REPORTS = {}
for task in REFERENCE:
    REPORTS[task] = verify(task) if VERIFY else EXPECTED[task]

In [ ]:
print(run('notebooks/check-submission', SUB, record='check-submission.txt').splitlines()[-1])
extra = json.loads(run('notebooks/extra-format.py', SUB, record='extra-format.json'))
names = sorted(str(p.relative_to(SUB)) for p in SUB.rglob('*') if p.is_file())
assert names == sorted(MEMBERS), names
actual = {name: sha256(SUB / name) for name in names}
assert all(actual[n] == MEMBERS[n]['sha256'] for n in names), 'Output member hash mismatch'
with zipfile.ZipFile(WORKING / 'submission.zip', 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    # Fixed container metadata, so the same members always give the same archive.
    for name in names:
        info = zipfile.ZipInfo(name, date_time=(2026, 10, 27, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.create_system = 3
        info.external_attr = 0o100644 << 16
        info._compresslevel = 6
        with (SUB / name).open('rb') as source, archive.open(info, 'w', force_zip64=True) as target:
            shutil.copyfileobj(source, target, length=1 << 20)
with zipfile.ZipFile(WORKING / 'submission.zip') as archive:
    assert sorted(archive.namelist()) == names
    for name in names:
        with archive.open(name) as stream:
            assert hashlib.file_digest(stream, 'sha256').hexdigest() == actual[name], name
for name, digest in COMPETITION_FILES.items():
    assert sha256(COMP / name) == digest, f'Official input changed during the run: {name}'
gm = math.prod(REFERENCE[t] / REPORTS[t]['score'] for t in REFERENCE) ** (1 / 3)
summary = {'status': 'PASS', 'blocks': BLOCKS, 'verifiers': REPORTS, 'member_sha256': actual,
           'geometric_mean': gm, 'archive_sha256': sha256(WORKING / 'submission.zip'),
           'archive_bytes': (WORKING / 'submission.zip').stat().st_size, 'environment': ENVIRONMENT,
           'timings': TIMINGS, 'elapsed_seconds': time.monotonic() - START_TIME}
(WORKING / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')
(WORKING / 'submission-members.sha256').write_text(''.join(f'{actual[n]}  {n}\n' for n in names))
for task, r in REPORTS.items():
    print(f'{task:9} {BLOCKS[task]["title"]}: cost {r["score"]:,.6g}, ratio {REFERENCE[task] / r["score"]:.4f}')
print(f'{len(names)} members, each byte-identical to its block; format checks and ZIP read-back PASS.')
print(f'Geometric mean of the three ratios: {gm:.5f} (official verifiers at their defaults; '
      f'the leaderboard measures again)')
print(f'Elapsed {summary["elapsed_seconds"]:.0f} s | {WORKING / "submission.zip"}')

## Credits and licence

- **invsqrt:** As yoyobar's notebook states: **yoyobar**, with GPT-6 Astra: the 62-qubit circuit, by changes to the lookup, normalization and tail schedule and four nonlinear-cone rewrites. **Ahmed Berat Özer**: a compatible phase patch that saves one Toffoli, in [Quantum AI - DF-THC Group Compression](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression), distributed through **James Barlow (manwithacat)**'s [VQCC Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo). **Akmal Xodarev (busyaprime)**: the 63-qubit starting point, based on **mrmeows / team selenium**.
- **DQI:** **GPT-6 Astra Pro**, answering our two questions of 30 October: the front half, with its cached extension inverses, the joint six-output inverse, the tail, prefix and fold identities of the synthesis, the retained norm coordinates of the reverse and the trailing node's single scalar replay, and the conversion pair, with its shared parent correction, its measured quotient heads, its middle identities, its Hankel schedule and its root blocks. **Claude** joined the two and built the result. **Our build loop** of AI agents (GPT-6 Astra leading GPT-6.1 Sol builders and a GPT-6.1 Terra reviewer, directed by Claude for Agniv Banerjee, SomeJoint), in seven hours on 30 October: the retained reverse chart and core; the swap-free core and its zero-affine phase character; grouped forward flags with a scoped direct pole phase; adaptive norm coordinates in the inverse characters; streamed 18-Toffoli inverses; the converter route from level 6 with child gap flags 2 and 3; sequential low head restores with a reused inverse flag; high-level gap predicates from the top flag; no quantum work where the classical mask word is zero; and the trailing pole flag erased by a measured phase. **GPT-6 Astra Pro**: five exact substitutions in the decoder, answering our question of 29 October, and an independent observation of the odd full product fused with the CRT remainder, which the loop built. **Our third research team** of AI agents (GPT-6 Astra leading two GPT-6 Astra designers, three GPT-6.1 Sol builders and three GPT-6.1 Terra reviewers): the 889-qubit decoder the loop starts from, with the paired high-component chart of the extraction chains, the inverse19 reverse, the streamed synthesis divide and the retimed head schedule, after its aligned conversion route with measured low heads, the streamed full-head fusion with a monic root, the small truncated products and the replay erasure. **GPT-6 Astra Pro**: the 897-qubit decoder it starts from, with its homogeneous external chains and 658-Toffoli jet. **Ahmed Berat Özer**: the narrowing of our fused decoder in his version 9, from which our 883-qubit decoder came. **Alex Chilton**: the 883-qubit Schur decoder whose design it starts from, based on **jday96314**'s QuantumAI decoder and classical models (that repository states no license). **mrmeows** and **Ahmed Berat Özer**: the public mask projection of round17 and version 8, after **manwithacat**'s classical-mask projection. Earlier links of the design: **Dmitrii Gluzdov**'s round12 decoder, **Agniv Banerjee (SomeJoint)**'s Bézout–Hankel phase solve, **ViberTeam** and **Phuc Thai (DePie)**.
- **Chemistry:** As Alex Chilton's notebook states: **Ahmed Berat Özer**: the new weights, in version 16 of [Quantum AI - DF-THC Group Compression](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression). **Atakan Aldemir**: the word table of [QCC: 888Q Repair and Native Chemistry Co-design](https://www.kaggle.com/code/atakanaldemir/qcc-888q-repair-and-native-chemistry-co-design). **James Barlow (manwithacat)**: the 16,266-Toffoli per-slot loaders, assembled from those of **Dmitrii Gluzdov**, **heliosli** and Atakan Aldemir, and the refit of the weights in [VQCC Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo), who also assembled these three files first, in [The Real $150K Was the Friends We Made](https://www.kaggle.com/code/manwithacat/the-real-150k-was-the-friends-we-made). **Alex Chilton**: this file's portfolio. **Our research team** of AI agents, directed by Claude for Agniv Banerjee (SomeJoint): the full-XOR loaders and factorization this line inherits, from our public dataset [qcc-our-chemistry-b143-c60-xor2](https://www.kaggle.com/datasets/somejoint/qcc-our-chemistry-b143-c60-xor2).

**Inherited lineage**, as the fullest public chain names it: Agniv Banerjee (SomeJoint), Dmitrii Gluzdov, ViberTeam / Artem1410 / anazemcev, manwithacat, Phuc Thai (DePie), Kozykorpesh (kozykappa), Michael Rustad, Akmal Xodarev (busyaprime), marsyas, Sumit Chongder, Rudra Bhaskar, Zongao Bian, Mizero Lucky Gall, Gu et al. and Guang Hao Low.

Thanks to **Google Quantum AI** and the competition authors for the tasks, the verifiers and Kickmix, which keep their own terms. The circuits keep their authors' licences; the notebook code is Apache-2.0. If we have left anyone out of a chain, tell us in the comments and we will add them.